In [ ]:
%pip install "city2graph"
%pip install contextily
%pip install -U geovoronoi[plotting]

In [ ]:
import city2graph as c2g
import numpy as np
import osmnx as ox
import geopandas as gpd
import geopandas.plotting
from shapely.geometry import Point, MultiPolygon
def _flatten_multi_geoms(geoms, prefix=None):
    """
    Replacement for the missing GeoPandas function.
    Returns TWO lists: (flattened_geometries, original_indices)
    """
    flattened_geoms = []
    flattened_indices = []

    for i, geom in enumerate(geoms):
        if isinstance(geom, MultiPolygon):
            for g in geom.geoms:
                flattened_geoms.append(g)
                flattened_indices.append(i)
        else:
            flattened_geoms.append(geom)
            flattened_indices.append(i)

    return flattened_geoms, flattened_indices
geopandas.plotting._flatten_multi_geoms = _flatten_multi_geoms
import matplotlib.pyplot as plt
import contextily as cx
import matplotlib.lines as Line2D
import shapely.geometry as box
from shapely.geometry import LineString, Point
import pandas as pd
import networkx as nx
import folium
from geovoronoi import voronoi_regions_from_coords, points_to_coords
from geovoronoi.plotting import subplot_for_map, plot_voronoi_polys_with_points_in_area
from sklearn.cluster import KMeans
import seaborn as sns
import time

In [ ]:
# @title
city_name ="Varanasi"
G = ox.graph_from_place(city_name, network_type="drive")
admin = ox.geocode_to_gdf(city_name)
custom_data = {
    "name" : ["Lanka stand",
              "bhelupur",
              "Jangamwadi stand",
              "luxa stand",
              "Benia baag autostand",
              "Maidagin crossing",
              "nagar nigam stand",
              "Pillikothi stand",
              "Sankat mochan stand",
              "Durga kund stand",
              "Bhikharipur stand",
              "Sunderpur stand",
              "benaras railway stn",
              "Varanasi railway station (cantt)",
              "sarnath stand",
              "Kashi raiway station stand",
              "Rajghat stand",
              "Pandaypur stand",
              "Chetganj stand",
              "Varanasi city jn stand",
              "Golgadda",
              "Konia stand",
              "Sigra auto stand",
              "DLW chauraha stand",
              "Maruadih stand",
              "Kandwa auto stand",
              "Bhojubeer stand",
              "Kachehri stand",
              "Tengda mod stand",
              "Ashapur stand",
              "Pahadia stand",
              "Shivpur stand",
              "Lahtara stand",
              "Hydrabad gate stand",
              "sear gate stand",
              "VT stand",
              "Kamaccha stand",
              "Chittupur stand",
              "Belwababa stand",
              "Roadways stand",
              "Polutryfarm stand",
              "Assi mod stand",
              "Harishchandra ghat mod",
              "Mughalsarai stand",],
    "lat": [
        25.27870156,
        25.29123304,
        25.30702835,
        25.30855056,
        25.31516205,
        25.31954492,
        25.38334149,
        25.32690846,
        25.28335717,
        25.28873615,
        25.26551904,
        25.28717205,
        25.3017939,
        25.3292287,
        25.60964906,
        25.32675525,
        25.32891694,
        25.44151805,
        25.4241839,
        25.3305993,
        25.32901971,
        25.33459765,
        25.31064495,
        25.28615547,
        25.29968355,
        25.26210073,
        25.35341589,
        25.34464509,
        25.25868818,
        25.38131209,
        25.35725431,
        25.35698178,
        25.31647925,
        25.26371127,
        25.25727315,
        25.26524985,
        25.30120684,
        25.26360376,
        25.37809639,
        25.32882015,
        25.33286372,
        25.28909808,
        25.29863795,
        25.28138247,
    ],
    "lon": [
        83.00246121,
        82.99511844,
        83.00525619,
        82.99914758,
        83.00347854,
        83.01253144,
        83.02841979,
        83.01670405,
        83.00023408,
        82.99913508,
        82.83285088,
        82.9780112,
        82.96972771,
        82.96604001,
        83.03112195,
        83.0322849,
        83.03880186,
        83.13809051,
        82.98463394,
        82.98691903,
        83.02033606,
        83.03149219,
        82.98475977,
        82.96880284,
        82.97298997,
        82.96027739,
        82.97573716,
        82.98117563,
        83.03683511,
        82.92420346,
        83.00542984,
        82.95752719,
        82.97047792,
        82.9815466,
        82.99788682,
        82.98986872,
        82.9974857,
        83.00095002,
        82.99434731,
        82.99012132,
        82.9604066,
        83.0039991,
        83.0074472,
        83.12128478,
    ]
}

In [ ]:
# @title
data = ('''
82.97698411
82.97698411
82.97698411
82.97698411
82.97698411
83.03013666
83.03013666
83.03013666
83.03013666
83.03013666
83.01181023
83.01181023
83.01181023
83.01181023
83.01181023
82.99977969
82.99977969
82.99977969
82.99977969
82.99977969
83.00938199
83.00938199
83.00938199
83.00938199
83.00938199
83.01094434
83.01094434
83.01094434
83.01094434
83.01094434
83.01065314
83.01065314
83.01065314
83.01065314
83.01065314
83.00912003
83.00912003
83.00912003
83.00912003
83.00912003
82.96192736
82.96192736
82.96192736
82.96192736
82.96192736
82.94653711
82.94653711
82.94653711
82.94653711
82.94653711
82.9824534
82.9824534
82.9824534
82.9824534
82.9824534
83.00292842
83.00292842
83.00292842
83.00292842
83.00292842
83.029153
83.029153
83.029153
83.029153
83.029153
83.00503461
83.00503461
83.00503461
83.00503461
83.00503461
82.99546951
82.99546951
82.99546951
82.99546951
82.99546951
82.9415478
82.9415478
82.9415478
82.9415478
82.9415478
83.00993769
83.00993769
83.00993769
83.00993769
83.00993769
82.99368113
82.99368113
82.99368113
82.99368113
82.99368113
82.99880529
82.99880529
82.99880529
82.99880529
82.99880529
83.02218601
83.02218601
83.02218601
83.02218601
83.02218601
83.01122875
83.01122875
83.01122875
83.01122875
83.01122875
82.99666621
82.99666621
82.99666621
82.99666621
82.99666621
82.98402112
82.98402112
82.98402112
82.98402112
82.98402112
83.01774148
83.01774148
83.01774148
83.01774148
83.01774148
82.98584991
82.98584991
82.98584991
82.98584991
82.98584991
82.99697096
82.99697096
82.99697096
82.99697096
82.99697096
82.98374298
82.98374298
82.98374298
82.98374298
82.98374298
82.97872271
82.97872271
82.97872271
82.97872271
82.97872271
83.01370846
83.01370846
83.01370846
83.01370846
83.01370846
83.01576183
83.01576183
83.01576183
83.01576183
83.01576183
82.99052034
82.99052034
82.99052034
82.99052034
82.99052034
83.00267333
83.00267333
83.00267333
83.00267333
83.00267333
83.01220973
83.01220973
83.01220973
83.01220973
83.01220973
82.99334778
82.99334778
82.99334778
82.99334778
82.99334778
82.97889586
82.97889586
82.97889586
82.97889586
82.97889586
82.98830107
82.98830107
82.98830107
82.98830107
82.98830107
82.97894321
82.97894321
82.97894321
82.97894321
82.97894321
82.9781284
82.9781284
82.9781284
82.9781284
82.9781284
82.97182114
82.97182114
82.97182114
82.97182114
82.97182114
83.01966165
83.01966165
83.01966165
83.01966165
83.01966165
83.00420656
83.00420656
83.00420656
83.00420656
83.00420656
82.97677951
82.97677951
82.97677951
82.97677951
82.97677951
83.00921108
83.00921108
83.00921108
83.00921108
83.00921108
82.99533906
82.99533906
82.99533906
82.99533906
82.99533906
82.98412237
82.98412237
82.98412237
82.98412237
82.98412237
83.00110996
83.00110996
83.00110996
83.00110996
83.00110996
83.01134038
83.01134038
83.01134038
83.01134038
83.01134038
82.99589261
82.99589261
82.99589261
82.99589261
82.99589261
83.01480833
83.01480833
83.01480833
83.01480833
83.01480833
83.01375835
83.01375835
83.01375835
83.01375835
83.01375835
83.00701757
83.00701757
83.00701757
83.00701757
83.00701757
83.01818226
83.01818226
83.01818226
83.01818226
83.01818226
83.0051464
83.0051464
83.0051464
83.0051464
83.0051464
82.97488678
82.97488678
82.97488678
82.97488678
82.97488678
82.98074271
82.98074271
82.98074271
82.98074271
82.98074271
83.00665289
83.00665289
83.00665289
83.00665289
83.00665289
83.0233721
83.0233721
83.0233721
83.0233721
83.0233721
83.02638573
83.02638573
83.02638573
83.02638573
83.02638573
82.98081555
82.98081555
82.98081555
82.98081555
82.98081555
83.00811347
83.00811347
83.00811347
83.00811347
83.00811347
83.01530013
83.01530013
83.01530013
83.01530013
83.01530013
83.00564699
83.00564699
83.00564699
83.00564699
83.00564699
83.0173927
83.0173927
83.0173927
83.0173927
83.0173927
83.00922371
83.00922371
83.00922371
83.00922371
83.00922371
83.01772067
83.01772067
83.01772067
83.01772067
83.01772067
82.97767324
82.97767324
82.97767324
82.97767324
82.97767324
82.97938391
82.97938391
82.97938391
82.97938391
82.97938391
82.99761872
82.99761872
82.99761872
82.99761872
82.99761872
83.02090591
83.02090591
83.02090591
83.02090591
83.02090591
83.00634731
83.00634731
83.00634731
83.00634731
83.00634731
83.01050023
83.01050023
83.01050023
83.01050023
83.01050023
82.98883432
82.98883432
82.98883432
82.98883432
82.98883432
83.01491172
83.01491172
83.01491172
83.01491172
83.01491172
82.99900355
82.99900355
82.99900355
82.99900355
82.99900355
83.01069727
83.01069727
83.01069727
83.01069727
83.01069727
83.01151155
83.01151155
83.01151155
83.01151155
83.01151155
82.98935508
82.98935508
82.98935508
82.98935508
82.98935508
82.99985204
82.99985204
82.99985204
82.99985204
82.99985204
83.01539251
83.01539251
83.01539251
83.01539251
83.01539251
83.01192571
83.01192571
83.01192571
83.01192571
83.01192571
82.95002742
82.95002742
82.95002742
82.95002742
82.95002742
83.02840588
83.02840588
83.02840588
83.02840588
83.02840588
83.01361667
83.01361667
83.01361667
83.01361667
83.01361667
83.01098958
83.01098958
83.01098958
83.01098958
83.01098958
83.00096056
83.00096056
83.00096056
83.00096056
83.00096056
83.00862892
83.00862892
83.00862892
83.00862892
83.00862892
82.98158605
82.98158605
82.98158605
82.98158605
82.98158605
83.01796382
83.01796382
83.01796382
83.01796382
83.01796382
82.98064475
82.98064475
82.98064475
82.98064475
82.98064475
83.00407224
83.00407224
83.00407224
83.00407224
83.00407224
83.01717922
83.01717922
83.01717922
83.01717922
83.01717922
83.0111768
83.0111768
83.0111768
83.0111768
83.0111768
83.01579598
83.01579598
83.01579598
83.01579598
83.01579598
83.00000752
83.00000752
83.00000752
83.00000752
83.00000752
83.01968977
83.01968977
83.01968977
83.01968977
83.01968977
82.99392481
82.99392481
82.99392481
82.99392481
82.99392481
83.01863655
83.01863655
83.01863655
83.01863655
83.01863655
83.00308692
83.00308692
83.00308692
83.00308692
83.00308692
83.01192689
83.01192689
83.01192689
83.01192689
83.01192689
83.01132767
83.01132767
83.01132767
83.01132767
83.01132767
83.00790329
83.00790329
83.00790329
83.00790329
83.00790329
82.99206927
82.99206927
82.99206927
82.99206927
82.99206927
82.98249292
82.98249292
82.98249292
82.98249292
82.98249292
82.98795021
82.98795021
82.98795021
82.98795021
82.98795021
82.98810267
82.98810267
82.98810267
82.98810267
82.98810267
83.02230156
83.02230156
83.02230156
83.02230156
83.02230156
82.99709215
82.99709215
82.99709215
82.99709215
82.99709215
82.99604687
82.99604687
82.99604687
82.99604687
82.99604687
82.98176387
82.98176387
82.98176387
82.98176387
82.98176387
83.00420524
83.00420524
83.00420524
83.00420524
83.00420524
83.02664054
83.02664054
83.02664054
83.02664054
83.02664054
82.97721421
82.97721421
82.97721421
82.97721421
82.97721421
83.01658661
83.01658661
83.01658661
83.01658661
83.01658661
82.98351214
82.98351214
82.98351214
82.98351214
82.98351214
83.02235038
83.02235038
83.02235038
83.02235038
83.02235038
83.02272715
83.02272715
83.02272715
83.02272715
83.02272715
82.9739599
82.9739599
82.9739599
82.9739599
82.9739599
83.00388061
83.00388061
83.00388061
83.00388061
83.00388061
83.01953949
83.01953949
83.01953949
83.01953949
83.01953949
83.00346521
83.00346521
83.00346521
83.00346521
83.00346521
82.99838396
82.99838396
82.99838396
82.99838396
82.99838396
83.00514118
83.00514118
83.00514118
83.00514118
83.00514118
83.0108641
83.0108641
83.0108641
83.0108641
83.0108641
82.9783213
82.9783213
82.9783213
82.9783213
82.9783213
82.97906917
82.97906917
82.97906917
82.97906917
82.97906917
83.02953158
83.02953158
83.02953158
83.02953158
83.02953158
83.01217496
83.01217496
83.01217496
83.01217496
83.01217496
82.98941065
82.98941065
82.98941065
82.98941065
82.98941065
83.00706407
83.00706407
83.00706407
83.00706407
83.00706407
83.00422066
83.00422066
83.00422066
83.00422066
83.00422066
82.98465185
82.98465185
82.98465185
82.98465185
82.98465185
83.01155616
83.01155616
83.01155616
83.01155616
83.01155616
82.98624175
82.98624175
82.98624175
82.98624175
82.98624175
83.02555919
83.02555919
83.02555919
83.02555919
83.02555919
83.01721314
83.01721314
83.01721314
83.01721314
83.01721314
83.03389347
83.03389347
83.03389347
83.03389347
83.03389347
83.00309901
83.00309901
83.00309901
83.00309901
83.00309901
82.99765306
82.99765306
82.99765306
82.99765306
82.99765306
82.98577965
82.98577965
82.98577965
82.98577965
82.98577965
83.00549442
83.00549442
83.00549442
83.00549442
83.00549442
83.02367702
83.02367702
83.02367702
83.02367702
83.02367702
83.00022825
83.00022825
83.00022825
83.00022825
83.00022825
83.0121782
83.0121782
83.0121782
83.0121782
83.0121782
83.02606975
83.02606975
83.02606975
83.02606975
83.02606975
82.99202498
82.99202498
82.99202498
82.99202498
82.99202498
82.99429046
82.99429046
82.99429046
82.99429046
82.99429046
83.01064958
83.01064958
83.01064958
83.01064958
83.01064958
83.00645898
83.00645898
83.00645898
83.00645898
83.00645898
82.98963105
82.98963105
82.98963105
82.98963105
82.98963105
82.98170061
82.98170061
82.98170061
82.98170061
82.98170061
83.01422669
83.01422669
83.01422669
83.01422669
83.01422669
83.02480409
83.02480409
83.02480409
83.02480409
83.02480409
82.98232977
82.98232977
82.98232977
82.98232977
82.98232977
83.00708861
83.00708861
83.00708861
83.00708861
83.00708861
82.97935869
82.97935869
82.97935869
82.97935869
82.97935869
83.00913552
83.00913552
83.00913552
83.00913552
83.00913552
83.00137762
83.00137762
83.00137762
83.00137762
83.00137762
82.97622331
82.97622331
82.97622331
82.97622331
82.97622331
83.00291816
83.00291816
83.00291816
83.00291816
83.00291816
83.01185711
83.01185711
83.01185711
83.01185711
83.01185711
82.97671221
82.97671221
82.97671221
82.97671221
82.97671221
83.0074734
83.0074734
83.0074734
83.0074734
83.0074734
83.02553092
83.02553092
83.02553092
83.02553092
83.02553092
83.00035736
83.00035736
83.00035736
83.00035736
83.00035736
83.00057854
83.00057854
83.00057854
83.00057854
83.00057854
82.99096091
82.99096091
82.99096091
82.99096091
82.99096091
82.97219311
82.97219311
82.97219311
82.97219311
82.97219311
82.96864644
82.96864644
82.96864644
82.96864644
82.96864644
82.97888941
82.97888941
82.97888941
82.97888941
82.97888941
82.97873023
82.97873023
82.97873023
82.97873023
82.97873023
82.97417897
82.97417897
82.97417897
82.97417897
82.97417897
82.98336781
82.98336781
82.98336781
82.98336781
82.98336781
82.99676789
82.99676789
82.99676789
82.99676789
82.99676789
83.00662677
83.00662677
83.00662677
83.00662677
83.00662677
83.03698939
83.03698939
83.03698939
83.03698939
83.03698939
83.01817881
83.01817881
83.01817881
83.01817881
83.01817881
82.98916924
82.98916924
82.98916924
82.98916924
82.98916924
82.97176396
82.97176396
82.97176396
82.97176396
82.97176396
83.00425056
83.00425056
83.00425056
83.00425056
83.00425056
82.99253463
82.99253463
82.99253463
82.99253463
82.99253463
83.02365721
83.02365721
83.02365721
83.02365721
83.02365721
83.01306691
83.01306691
83.01306691
83.01306691
83.01306691
83.01512089
83.01512089
83.01512089
83.01512089
83.01512089
83.01202021
83.01202021
83.01202021
83.01202021
83.01202021
82.98310781
82.98310781
82.98310781
82.98310781
82.98310781
83.00600158
83.00600158
83.00600158
83.00600158
83.00600158
82.97999275
82.97999275
82.97999275
82.97999275
82.97999275
83.00444709
83.00444709
83.00444709
83.00444709
83.00444709
82.98954606
82.98954606
82.98954606
82.98954606
82.98954606
82.9966416
82.9966416
82.9966416
82.9966416
82.9966416
83.03206021
83.03206021
83.03206021
83.03206021
83.03206021
82.98827558
82.98827558
82.98827558
82.98827558
82.98827558
83.02158115
83.02158115
83.02158115
83.02158115
83.02158115
83.00993505
83.00993505
83.00993505
83.00993505
83.00993505
82.99280024
82.99280024
82.99280024
82.99280024
82.99280024
82.97189273
82.97189273
82.97189273
82.97189273
82.97189273
83.00053161
83.00053161
83.00053161
83.00053161
83.00053161
82.9886232
82.9886232
82.9886232
82.9886232
82.9886232
83.00235228
83.00235228
83.00235228
83.00235228
83.00235228
82.98825523
82.98825523
82.98825523
82.98825523
82.98825523
83.0091627
83.0091627
83.0091627
83.0091627
83.0091627
83.0246204
83.0246204
83.0246204
83.0246204
83.0246204
82.97894247
82.97894247
82.97894247
82.97894247
82.97894247
83.02424349
83.02424349
83.02424349
83.02424349
83.02424349
83.01113396
83.01113396
83.01113396
83.01113396
83.01113396
82.9689935
82.9689935
82.9689935
82.9689935
82.9689935
83.01040904
83.01040904
83.01040904
83.01040904
83.01040904
83.01531873
83.01531873
83.01531873
83.01531873
83.01531873
82.98509141
82.98509141
82.98509141
82.98509141
82.98509141
83.01275448
83.01275448
83.01275448
83.01275448
83.01275448
83.01310516
83.01310516
83.01310516
83.01310516
83.01310516
83.02233601
83.02233601
83.02233601
83.02233601
83.02233601
82.98752239
82.98752239
82.98752239
82.98752239
82.98752239
82.9969292
82.9969292
82.9969292
82.9969292
82.9969292
83.00540511
83.00540511
83.00540511
83.00540511
83.00540511
83.00247508
83.00247508
83.00247508
83.00247508
83.00247508
82.98822043
82.98822043
82.98822043
82.98822043
82.98822043
82.98224916
82.98224916
82.98224916
82.98224916
82.98224916
82.96842916
82.96842916
82.96842916
82.96842916
82.96842916
83.02891327
83.02891327
83.02891327
83.02891327
83.02891327
82.99892005
82.99892005
82.99892005
82.99892005
82.99892005
82.9463371
82.9463371
82.9463371
82.9463371
82.9463371
83.01906963
83.01906963
83.01906963
83.01906963
83.01906963
83.00503057
83.00503057
83.00503057
83.00503057
83.00503057
83.01781446
83.01781446
83.01781446
83.01781446
83.01781446
82.99858939
82.99858939
82.99858939
82.99858939
82.99858939
82.97737672
82.97737672
82.97737672
82.97737672
82.97737672
83.01692645
83.01692645
83.01692645
83.01692645
83.01692645
83.02026935
83.02026935
83.02026935
83.02026935
83.02026935
83.02375623
83.02375623
83.02375623
83.02375623
83.02375623
83.00124359
83.00124359
83.00124359
83.00124359
83.00124359
83.01814958
83.01814958
83.01814958
83.01814958
83.01814958
83.00129216
83.00129216
83.00129216
83.00129216
83.00129216
83.01179772
83.01179772
83.01179772
83.01179772
83.01179772
82.98076976
82.98076976
82.98076976
82.98076976
82.98076976
83.01449837
83.01449837
83.01449837
83.01449837
83.01449837
83.02167739
83.02167739
83.02167739
83.02167739
83.02167739
82.98862826
82.98862826
82.98862826
82.98862826
82.98862826
82.98237667
82.98237667
82.98237667
82.98237667
82.98237667
82.99662914
82.99662914
82.99662914
82.99662914
82.99662914
82.99313867
82.99313867
82.99313867
82.99313867
82.99313867
82.9781059
82.9781059
82.9781059
82.9781059
82.9781059
82.97415389
82.97415389
82.97415389
82.97415389
82.97415389
82.99668607
82.99668607
82.99668607
82.99668607
82.99668607
83.00123622
83.00123622
83.00123622
83.00123622
83.00123622
83.03743875
83.03743875
83.03743875
83.03743875
83.03743875
83.01841892
83.01841892
83.01841892
83.01841892
83.01841892
83.0068765
83.0068765
83.0068765
83.0068765
83.0068765
83.01074799
83.01074799
83.01074799
83.01074799
83.01074799
83.02183161
83.02183161
83.02183161
83.02183161
83.02183161
83.01281378
83.01281378
83.01281378
83.01281378
83.01281378
82.99686004
82.99686004
82.99686004
82.99686004
82.99686004
83.0153113
83.0153113
83.0153113
83.0153113
83.0153113
83.00648718
83.00648718
83.00648718
83.00648718
83.00648718
82.99745654
82.99745654
82.99745654
82.99745654
82.99745654
83.00577379
83.00577379
83.00577379
83.00577379
83.00577379
82.98390368
82.98390368
82.98390368
82.98390368
82.98390368
82.99938439
82.99938439
82.99938439
82.99938439
82.99938439
82.99456382
82.99456382
82.99456382
82.99456382
82.99456382
82.97650717
82.97650717
82.97650717
82.97650717
82.97650717
83.00126502
83.00126502
83.00126502
83.00126502
83.00126502
82.97965083
82.97965083
82.97965083
82.97965083
82.97965083
82.99348783
82.99348783
82.99348783
82.99348783
82.99348783
82.98264624
82.98264624
82.98264624
82.98264624
82.98264624
83.00326836
83.00326836
83.00326836
83.00326836
83.00326836
82.97856709
82.97856709
82.97856709
82.97856709
82.97856709
82.98624025
82.98624025
82.98624025
82.98624025
82.98624025
83.00263052
83.00263052
83.00263052
83.00263052
83.00263052
83.00166971
83.00166971
83.00166971
83.00166971
83.00166971
83.00192648
83.00192648
83.00192648
83.00192648
83.00192648
83.00094391
83.00094391
83.00094391
83.00094391
83.00094391
82.97417391
82.97417391
82.97417391
82.97417391
82.97417391
82.98968028
82.98968028
82.98968028
82.98968028
82.98968028
83.00292788
83.00292788
83.00292788
83.00292788
83.00292788
82.97727126
82.97727126
82.97727126
82.97727126
82.97727126
82.99379033
82.99379033
82.99379033
82.99379033
82.99379033
83.00346028
83.00346028
83.00346028
83.00346028
83.00346028
83.00904997
83.00904997
83.00904997
83.00904997
83.00904997
83.00289478
83.00289478
83.00289478
83.00289478
83.00289478
82.99021681
82.99021681
82.99021681
82.99021681
82.99021681
83.0085273
83.0085273
83.0085273
83.0085273
83.0085273
83.00111941
83.00111941
83.00111941
83.00111941
83.00111941
82.95146477
82.95146477
82.95146477
82.95146477
82.95146477
82.97535385
82.97535385
82.97535385
82.97535385
82.97535385
83.02076601
83.02076601
83.02076601
83.02076601
83.02076601
83.003226
83.003226
83.003226
83.003226
83.003226
83.00883088
83.00883088
83.00883088
83.00883088
83.00883088
83.01040108
83.01040108
83.01040108
83.01040108
83.01040108
82.97699525
82.97699525
82.97699525
82.97699525
82.97699525
82.99741522
82.99741522
82.99741522
82.99741522
82.99741522
83.0238867
83.0238867
83.0238867
83.0238867
83.0238867
83.0111657
83.0111657
83.0111657
83.0111657
83.0111657
83.00488322
83.00488322
83.00488322
83.00488322
83.00488322
83.0190364
83.0190364
83.0190364
83.0190364
83.0190364
82.99522243
82.99522243
82.99522243
82.99522243
82.99522243
82.98681838
82.98681838
82.98681838
82.98681838
82.98681838
82.99709717
82.99709717
82.99709717
82.99709717
82.99709717
82.99358014
82.99358014
82.99358014
82.99358014
82.99358014
83.01412485
83.01412485
83.01412485
83.01412485
83.01412485
83.02862381
83.02862381
83.02862381
83.02862381
83.02862381
83.01057718
83.01057718
83.01057718
83.01057718
83.01057718
82.97441093
82.97441093
82.97441093
82.97441093
82.97441093
82.97560539
82.97560539
82.97560539
82.97560539
82.97560539
82.99077515
82.99077515
82.99077515
82.99077515
82.99077515
82.98915594
82.98915594
82.98915594
82.98915594
82.98915594
83.00504593
83.00504593
83.00504593
83.00504593
83.00504593
83.01954711
83.01954711
83.01954711
83.01954711
83.01954711
83.0038173
83.0038173
83.0038173
83.0038173
83.0038173
83.00352537
83.00352537
83.00352537
83.00352537
83.00352537
82.98332487
82.98332487
82.98332487
82.98332487
82.98332487
83.011488
83.011488
83.011488
83.011488
83.011488
83.00888364
83.00888364
83.00888364
83.00888364
83.00888364
83.02764906
83.02764906
83.02764906
83.02764906
83.02764906
83.0026859
83.0026859
83.0026859
83.0026859
83.0026859
83.0181947
83.0181947
83.0181947
83.0181947
83.0181947
82.96081652
82.96081652
82.96081652
82.96081652
82.96081652
83.00248324
83.00248324
83.00248324
83.00248324
83.00248324
82.97670126
82.97670126
82.97670126
82.97670126
82.97670126
82.99139543
82.99139543
82.99139543
82.99139543
82.99139543
82.97806612
82.97806612
82.97806612
82.97806612
82.97806612
82.99059217
82.99059217
82.99059217
82.99059217
82.99059217
82.98820547
82.98820547
82.98820547
82.98820547
82.98820547
83.01524044
83.01524044
83.01524044
83.01524044
83.01524044
82.97188315
82.97188315
82.97188315
82.97188315
82.97188315
83.01021396
83.01021396
83.01021396
83.01021396
83.01021396
82.98078021
82.98078021
82.98078021
82.98078021
82.98078021
83.01153255
83.01153255
83.01153255
83.01153255
83.01153255
83.0059966
83.0059966
83.0059966
83.0059966
83.0059966
83.00609624
83.00609624
83.00609624
83.00609624
83.00609624
83.03591451
83.03591451
83.03591451
83.03591451
83.03591451
82.96267126
82.96267126
82.96267126
82.96267126
82.96267126
83.00035529
83.00035529
83.00035529
83.00035529
83.00035529
83.00360335
83.00360335
83.00360335
83.00360335
83.00360335
82.97951241
82.97951241
82.97951241
82.97951241
82.97951241
82.98647437
82.98647437
82.98647437
82.98647437
82.98647437
83.01064956
83.01064956
83.01064956
83.01064956
83.01064956
82.99397287
82.99397287
82.99397287
82.99397287
82.99397287
83.00542381
83.00542381
83.00542381
83.00542381
83.00542381
82.99355749
82.99355749
82.99355749
82.99355749
82.99355749
83.01282
83.01282
83.01282
83.01282
83.01282
82.99684395
82.99684395
82.99684395
82.99684395
82.99684395
83.00789107
83.00789107
83.00789107
83.00789107
83.00789107
83.00257381
83.00257381
83.00257381
83.00257381
83.00257381
82.99669997
82.99669997
82.99669997
82.99669997
82.99669997
83.00210969
83.00210969
83.00210969
83.00210969
83.00210969
83.00021672
83.00021672
83.00021672
83.00021672
83.00021672
82.98733854
82.98733854
82.98733854
82.98733854
82.98733854
83.00876559
83.00876559
83.00876559
83.00876559
83.00876559
83.00416503
83.00416503
83.00416503
83.00416503
83.00416503
83.00709546
83.00709546
83.00709546
83.00709546
83.00709546
82.99663445
82.99663445
82.99663445
82.99663445
82.99663445
82.98442257
82.98442257
82.98442257
82.98442257
82.98442257
83.01255242
83.01255242
83.01255242
83.01255242
83.01255242
83.00358356
83.00358356
83.00358356
83.00358356
83.00358356
83.00035996
83.00035996
83.00035996
83.00035996
83.00035996
83.00878464
83.00878464
83.00878464
83.00878464
83.00878464
83.01263451
83.01263451
83.01263451
83.01263451
83.01263451
83.01667686
83.01667686
83.01667686
83.01667686
83.01667686
83.01767495
83.01767495
83.01767495
83.01767495
83.01767495
83.02758579
83.02758579
83.02758579
83.02758579
83.02758579
83.02158168
83.02158168
83.02158168
83.02158168
83.02158168
82.98053343
82.98053343
82.98053343
82.98053343
82.98053343
83.00506604
83.00506604
83.00506604
83.00506604
83.00506604
82.99958631
82.99958631
82.99958631
82.99958631
82.99958631
82.99939586
82.99939586
82.99939586
82.99939586
82.99939586
83.00016558
83.00016558
83.00016558
83.00016558
83.00016558
83.01277485
83.01277485
83.01277485
83.01277485
83.01277485
83.02081767
83.02081767
83.02081767
83.02081767
83.02081767
82.97568762
82.97568762
82.97568762
82.97568762
82.97568762
83.00369756
83.00369756
83.00369756
83.00369756
83.00369756
83.01111767
83.01111767
83.01111767
83.01111767
83.01111767
82.99498792
82.99498792
82.99498792
82.99498792
82.99498792
82.93558473
82.93558473
82.93558473
82.93558473
82.93558473
82.97304919
82.97304919
82.97304919
82.97304919
82.97304919
83.02332642
83.02332642
83.02332642
83.02332642
83.02332642
83.02274869
83.02274869
83.02274869
83.02274869
83.02274869
83.01477735
83.01477735
83.01477735
83.01477735
83.01477735
82.98723197
82.98723197
82.98723197
82.98723197
82.98723197
82.97573655
82.97573655
82.97573655
82.97573655
82.97573655
82.98166513
82.98166513
82.98166513
82.98166513
82.98166513
83.00808846
83.00808846
83.00808846
83.00808846
83.00808846
83.00899783
83.00899783
83.00899783
83.00899783
83.00899783
83.01437987
83.01437987
83.01437987
83.01437987
83.01437987
83.01417692
83.01417692
83.01417692
83.01417692
83.01417692
83.00593936
83.00593936
83.00593936
83.00593936
83.00593936
82.98030459
82.98030459
82.98030459
82.98030459
82.98030459
83.00967444
83.00967444
83.00967444
83.00967444
83.00967444
82.99560324
82.99560324
82.99560324
82.99560324
82.99560324
82.99965402
82.99965402
82.99965402
82.99965402
82.99965402
83.00940462
83.00940462
83.00940462
83.00940462
83.00940462
83.01841364
83.01841364
83.01841364
83.01841364
83.01841364
83.00068687
83.00068687
83.00068687
83.00068687
83.00068687
83.00051507
83.00051507
83.00051507
83.00051507
83.00051507
83.0045831
83.0045831
83.0045831
83.0045831
83.0045831
83.00436294
83.00436294
83.00436294
83.00436294
83.00436294
82.98107021
82.98107021
82.98107021
82.98107021
82.98107021
83.02350545
83.02350545
83.02350545
83.02350545
83.02350545
83.02078794
83.02078794
83.02078794
83.02078794
83.02078794
82.99792052
82.99792052
82.99792052
82.99792052
82.99792052
82.98363436
82.98363436
82.98363436
82.98363436
82.98363436
82.99396964
82.99396964
82.99396964
82.99396964
82.99396964
83.00775783
83.00775783
83.00775783
83.00775783
83.00775783
83.00355346
83.00355346
83.00355346
83.00355346
83.00355346
82.99652516
82.99652516
82.99652516
82.99652516
82.99652516
83.00993227
83.00993227
83.00993227
83.00993227
83.00993227
83.02018771
83.02018771
83.02018771
83.02018771
83.02018771
82.98150192
82.98150192
82.98150192
82.98150192
82.98150192
83.00447082
83.00447082
83.00447082
83.00447082
83.00447082
83.01456754
83.01456754
83.01456754
83.01456754
83.01456754
83.02298001
83.02298001
83.02298001
83.02298001
83.02298001
83.00969807
83.00969807
83.00969807
83.00969807
83.00969807
82.94218151
82.94218151
82.94218151
82.94218151
82.94218151
82.97591916
82.97591916
82.97591916
82.97591916
82.97591916
82.97797721
82.97797721
82.97797721
82.97797721
82.97797721
83.0039817
83.0039817
83.0039817
83.0039817
83.0039817
83.01988534
83.01988534
83.01988534
83.01988534
83.01988534
82.99562058
82.99562058
82.99562058
82.99562058
82.99562058
83.02717042
83.02717042
83.02717042
83.02717042
83.02717042
83.00951565
83.00951565
83.00951565
83.00951565
83.00951565
83.00557962
83.00557962
83.00557962
83.00557962
83.00557962
83.02122518
83.02122518
83.02122518
83.02122518
83.02122518
82.96603493
82.96603493
82.96603493
82.96603493
82.96603493
83.01532459
83.01532459
83.01532459
83.01532459
83.01532459
83.0099104
83.0099104
83.0099104
83.0099104
83.0099104
82.97683407
82.97683407
82.97683407
82.97683407
82.97683407
83.00170009
83.00170009
83.00170009
83.00170009
83.00170009
83.02426817
83.02426817
83.02426817
83.02426817
83.02426817
83.01065347
83.01065347
83.01065347
83.01065347
83.01065347
83.00217244
83.00217244
83.00217244
83.00217244
83.00217244
82.9980268
82.9980268
82.9980268
82.9980268
82.9980268
82.97873068
82.97873068
82.97873068
82.97873068
82.97873068
82.99620406
82.99620406
82.99620406
82.99620406
82.99620406
82.99828195
82.99828195
82.99828195
82.99828195
82.99828195
82.99770133
82.99770133
82.99770133
82.99770133
82.99770133
83.01852822
83.01852822
83.01852822
83.01852822
83.01852822
82.99218361
82.99218361
82.99218361
82.99218361
82.99218361
83.0149606
83.0149606
83.0149606
83.0149606
83.0149606
83.01471726
83.01471726
83.01471726
83.01471726
83.01471726
82.99541972
82.99541972
82.99541972
82.99541972
82.99541972
83.02046376
83.02046376
83.02046376
83.02046376
83.02046376
83.01866428
83.01866428
83.01866428
83.01866428
83.01866428
82.9869133
82.9869133
82.9869133
82.9869133
82.9869133
82.99696692
82.99696692
82.99696692
82.99696692
82.99696692
82.99466995
82.99466995
82.99466995
82.99466995
82.99466995
83.02454868
83.02454868
83.02454868
83.02454868
83.02454868
83.01266087
83.01266087
83.01266087
83.01266087
83.01266087
83.00843336
83.00843336
83.00843336
83.00843336
83.00843336
83.00726665
83.00726665
83.00726665
83.00726665
83.00726665
83.00053298
83.00053298
83.00053298
83.00053298
83.00053298
83.0030775
83.0030775
83.0030775
83.0030775
83.0030775
82.94995962
82.94995962
82.94995962
82.94995962
82.94995962
83.00120005
83.00120005
83.00120005
83.00120005
83.00120005
83.00356103
83.00356103
83.00356103
83.00356103
83.00356103
83.00996815
83.00996815
83.00996815
83.00996815
83.00996815
83.01340528
83.01340528
83.01340528
83.01340528
83.01340528
82.98698296
82.98698296
82.98698296
82.98698296
82.98698296
83.00951351
83.00951351
83.00951351
83.00951351
83.00951351
83.02362533
83.02362533
83.02362533
83.02362533
83.02362533
83.01600984
83.01600984
83.01600984
83.01600984
83.01600984
82.98219882
82.98219882
82.98219882
82.98219882
82.98219882
83.01432001
83.01432001
83.01432001
83.01432001
83.01432001
83.01353794
83.01353794
83.01353794
83.01353794
83.01353794
82.99276401
82.99276401
82.99276401
82.99276401
82.99276401
82.95924284
82.95924284
82.95924284
82.95924284
82.95924284
82.98970791
82.98970791
82.98970791
82.98970791
82.98970791
82.96951146
82.96951146
82.96951146
82.96951146
82.96951146
82.99713688
82.99713688
82.99713688
82.99713688
82.99713688
83.01813374
83.01813374
83.01813374
83.01813374
83.01813374
83.00889857
83.00889857
83.00889857
83.00889857
83.00889857
83.01330205
83.01330205
83.01330205
83.01330205
83.01330205
83.00764294
83.00764294
83.00764294
83.00764294
83.00764294
83.02573591
83.02573591
83.02573591
83.02573591
83.02573591
83.01329615
83.01329615
83.01329615
83.01329615
83.01329615
83.01880766
83.01880766
83.01880766
83.01880766
83.01880766
82.99178065
82.99178065
82.99178065
82.99178065
82.99178065
82.99803512
82.99803512
82.99803512
82.99803512
82.99803512
83.01118897
83.01118897
83.01118897
83.01118897
83.01118897
82.97823115
82.97823115
82.97823115
82.97823115
82.97823115
83.0079935
83.0079935
83.0079935
83.0079935
83.0079935
82.97703458
82.97703458
82.97703458
82.97703458
82.97703458
82.9914671
82.9914671
82.9914671
82.9914671
82.9914671
82.98840957
82.98840957
82.98840957
82.98840957
82.98840957
82.95890808
82.95890808
82.95890808
82.95890808
82.95890808
82.99938373
82.99938373
82.99938373
82.99938373
82.99938373
83.01257377
83.01257377
83.01257377
83.01257377
83.01257377
83.00286261
83.00286261
83.00286261
83.00286261
83.00286261
83.0325398
83.0325398
83.0325398
83.0325398
83.0325398
82.9861446
82.9861446
82.9861446
82.9861446
82.9861446
83.00551212
83.00551212
83.00551212
83.00551212
83.00551212
83.02594495
83.02594495
83.02594495
83.02594495
83.02594495
82.99845376
82.99845376
82.99845376
82.99845376
82.99845376
83.00226497
83.00226497
83.00226497
83.00226497
83.00226497
83.00334863
83.00334863
83.00334863
83.00334863
83.00334863
83.01379862
83.01379862
83.01379862
83.01379862
83.01379862
83.03011196
83.03011196
83.03011196
83.03011196
83.03011196
83.00336108
83.00336108
83.00336108
83.00336108
83.00336108
82.9995674
82.9995674
82.9995674
82.9995674
82.9995674
83.01418386
83.01418386
83.01418386
83.01418386
83.01418386
83.02790634
83.02790634
83.02790634
83.02790634
83.02790634
82.99254423
82.99254423
82.99254423
82.99254423
82.99254423
83.00502073
83.00502073
83.00502073
83.00502073
83.00502073
82.99244688
82.99244688
82.99244688
82.99244688
82.99244688
83.01783524
83.01783524
83.01783524
83.01783524
83.01783524
82.98504741
82.98504741
82.98504741
82.98504741
82.98504741
83.01145244
83.01145244
83.01145244
83.01145244
83.01145244
83.01614831
83.01614831
83.01614831
83.01614831
83.01614831
82.97962355
82.97962355
82.97962355
82.97962355
82.97962355
82.96807828
82.96807828
82.96807828
82.96807828
82.96807828
82.9948883
82.9948883
82.9948883
82.9948883
82.9948883
82.98028605
82.98028605
82.98028605
82.98028605
82.98028605
83.0282825
83.0282825
83.0282825
83.0282825
83.0282825
82.99110355
82.99110355
82.99110355
82.99110355
82.99110355
82.99937413
82.99937413
82.99937413
82.99937413
82.99937413
83.02010762
83.02010762
83.02010762
83.02010762
83.02010762
82.94898078
82.94898078
82.94898078
82.94898078
82.94898078
82.9391358
82.9391358
82.9391358
82.9391358
82.9391358
82.99620078
82.99620078
82.99620078
82.99620078
82.99620078
83.02207562
83.02207562
83.02207562
83.02207562
83.02207562
83.01228735
83.01228735
83.01228735
83.01228735
83.01228735
83.00463923
83.00463923
83.00463923
83.00463923
83.00463923
83.01673361
83.01673361
83.01673361
83.01673361
83.01673361
82.97713128
82.97713128
82.97713128
82.97713128
82.97713128
82.96910223
82.96910223
82.96910223
82.96910223
82.96910223
83.00540615
83.00540615
83.00540615
83.00540615
83.00540615
82.99447239
82.99447239
82.99447239
82.99447239
82.99447239
82.9863044
82.9863044
82.9863044
82.9863044
82.9863044
83.01136303
83.01136303
83.01136303
83.01136303
83.01136303
83.01007463
83.01007463
83.01007463
83.01007463
83.01007463
82.99774569
82.99774569
82.99774569
82.99774569
82.99774569
83.00392478
83.00392478
83.00392478
83.00392478
83.00392478
83.0176175
83.0176175
83.0176175
83.0176175
83.0176175
82.99046258
82.99046258
82.99046258
82.99046258
82.99046258
82.99445489
82.99445489
82.99445489
82.99445489
82.99445489
83.02133145
83.02133145
83.02133145
83.02133145
83.02133145
82.99625615
82.99625615
82.99625615
82.99625615
82.99625615
83.01381974
83.01381974
83.01381974
83.01381974
83.01381974
82.9755526
82.9755526
82.9755526
82.9755526
82.9755526
83.01316973
83.01316973
83.01316973
83.01316973
83.01316973
83.00861907
83.00861907
83.00861907
83.00861907
83.00861907
82.97300935
82.97300935
82.97300935
82.97300935
82.97300935
83.01864886
83.01864886
83.01864886
83.01864886
83.01864886
82.99339365
82.99339365
82.99339365
82.99339365
82.99339365
83.00504154
83.00504154
83.00504154
83.00504154
83.00504154
83.02611613
83.02611613
83.02611613
83.02611613
83.02611613
83.02170771
83.02170771
83.02170771
83.02170771
83.02170771
82.99028888
82.99028888
82.99028888
82.99028888
82.99028888
82.98911151
82.98911151
82.98911151
82.98911151
82.98911151
83.0226522
83.0226522
83.0226522
83.0226522
83.0226522
83.01666375
83.01666375
83.01666375
83.01666375
83.01666375
83.0029311
83.0029311
83.0029311
83.0029311
83.0029311
82.99950227
82.99950227
82.99950227
82.99950227
82.99950227
82.99461532
82.99461532
82.99461532
82.99461532
82.99461532
82.99477102
82.99477102
82.99477102
82.99477102
82.99477102
83.01294935
83.01294935
83.01294935
83.01294935
83.01294935
83.01975363
83.01975363
83.01975363
83.01975363
83.01975363
82.97491934
82.97491934
82.97491934
82.97491934
82.97491934
82.9797579
82.9797579
82.9797579
82.9797579
82.9797579
82.99230073
82.99230073
82.99230073
82.99230073
82.99230073
83.00822409
83.00822409
83.00822409
83.00822409
83.00822409
83.01441755
83.01441755
83.01441755
83.01441755
83.01441755
82.9593207
82.9593207
82.9593207
82.9593207
82.9593207
83.0251244
83.0251244
83.0251244
83.0251244
83.0251244
83.0024301
83.0024301
83.0024301
83.0024301
83.0024301
83.00676918
83.00676918
83.00676918
83.00676918
83.00676918
82.99332352
82.99332352
82.99332352
82.99332352
82.99332352
82.94683259
82.94683259
82.94683259
82.94683259
82.94683259
83.00816877
83.00816877
83.00816877
83.00816877
83.00816877
83.01742888
83.01742888
83.01742888
83.01742888
83.01742888
83.00235758
83.00235758
83.00235758
83.00235758
83.00235758
82.97889454
82.97889454
82.97889454
82.97889454
82.97889454
83.00440801
83.00440801
83.00440801
83.00440801
83.00440801
83.00163704
83.00163704
83.00163704
83.00163704
83.00163704
83.00225057
83.00225057
83.00225057
83.00225057
83.00225057
83.0143961
83.0143961
83.0143961
83.0143961
83.0143961
83.03545519
83.03545519
83.03545519
83.03545519
83.03545519
82.98099078
82.98099078
82.98099078
82.98099078
82.98099078
82.98698007
82.98698007
82.98698007
82.98698007
82.98698007
83.00329406
83.00329406
83.00329406
83.00329406
83.00329406
82.98647666
82.98647666
82.98647666
82.98647666
82.98647666
83.00569551
83.00569551
83.00569551
83.00569551
83.00569551
83.0060089
83.0060089
83.0060089
83.0060089
83.0060089
83.00535485
83.00535485
83.00535485
83.00535485
83.00535485
82.98860847
82.98860847
82.98860847
82.98860847
82.98860847
83.03140907
83.03140907
83.03140907
83.03140907
83.03140907
83.02081467
83.02081467
83.02081467
83.02081467
83.02081467
83.02488948
83.02488948
83.02488948
83.02488948
83.02488948
82.98676755
82.98676755
82.98676755
82.98676755
82.98676755
83.01682241
83.01682241
83.01682241
83.01682241
83.01682241
83.01604067
83.01604067
83.01604067
83.01604067
83.01604067
83.00850323
83.00850323
83.00850323
83.00850323
83.00850323
82.99811705
82.99811705
82.99811705
82.99811705
82.99811705
83.01348457
83.01348457
83.01348457
83.01348457
83.01348457
82.98708361
82.98708361
82.98708361
82.98708361
82.98708361
83.03192046
83.03192046
83.03192046
83.03192046
83.03192046
82.98107043
82.98107043
82.98107043
82.98107043
82.98107043
83.00412895
83.00412895
83.00412895
83.00412895
83.00412895
83.0144457
83.0144457
83.0144457
83.0144457
83.0144457
82.99151105
82.99151105
82.99151105
82.99151105
82.99151105
83.00979343
83.00979343
83.00979343
83.00979343
83.00979343
83.02026439
83.02026439
83.02026439
83.02026439
83.02026439
83.00856439
83.00856439
83.00856439
83.00856439
83.00856439
83.00959587
83.00959587
83.00959587
83.00959587
83.00959587
83.02233693
83.02233693
83.02233693
83.02233693
83.02233693
82.99717774
82.99717774
82.99717774
82.99717774
82.99717774
82.99398922
82.99398922
82.99398922
82.99398922
82.99398922
82.99501145
82.99501145
82.99501145
82.99501145
82.99501145
83.00562818
83.00562818
83.00562818
83.00562818
83.00562818
82.99931259
82.99931259
82.99931259
82.99931259
82.99931259
82.97800184
82.97800184
82.97800184
82.97800184
82.97800184
82.9977808
82.9977808
82.9977808
82.9977808
82.9977808
83.00408093
83.00408093
83.00408093
83.00408093
83.00408093
82.98815204
82.98815204
82.98815204
82.98815204
82.98815204
83.01103369
83.01103369
83.01103369
83.01103369
83.01103369
82.99730141
82.99730141
82.99730141
82.99730141
82.99730141
83.01200047
83.01200047
83.01200047
83.01200047
83.01200047
82.99658924
82.99658924
82.99658924
82.99658924
82.99658924
83.01605371
83.01605371
83.01605371
83.01605371
83.01605371
82.99945808
82.99945808
82.99945808
82.99945808
82.99945808
83.0090399
83.0090399
83.0090399
83.0090399
83.0090399
83.01059936
83.01059936
83.01059936
83.01059936
83.01059936
82.99141481
82.99141481
82.99141481
82.99141481
82.99141481
83.01451858
83.01451858
83.01451858
83.01451858
83.01451858
83.01754666
83.01754666
83.01754666
83.01754666
83.01754666
83.01817351
83.01817351
83.01817351
83.01817351
83.01817351
82.96794709
82.96794709
82.96794709
82.96794709
82.96794709
82.99904229
82.99904229
82.99904229
82.99904229
82.99904229
82.99822055
82.99822055
82.99822055
82.99822055
82.99822055
83.01787211
83.01787211
83.01787211
83.01787211
83.01787211
83.02433871
83.02433871
83.02433871
83.02433871
83.02433871
83.00917219
83.00917219
83.00917219
83.00917219
83.00917219
83.02319535
83.02319535
83.02319535
83.02319535
83.02319535
83.00591173
83.00591173
83.00591173
83.00591173
83.00591173
82.98274782
82.98274782
82.98274782
82.98274782
82.98274782
83.01062057
83.01062057
83.01062057
83.01062057
83.01062057
83.00127116
83.00127116
83.00127116
83.00127116
83.00127116
83.01189882
83.01189882
83.01189882
83.01189882
83.01189882
83.00203362
83.00203362
83.00203362
83.00203362
83.00203362
83.01957903
83.01957903
83.01957903
83.01957903
83.01957903
83.0186789
83.0186789
83.0186789
83.0186789
83.0186789
82.99703252
82.99703252
82.99703252
82.99703252
82.99703252
83.02275971
83.02275971
83.02275971
83.02275971
83.02275971
83.03176249
83.03176249
83.03176249
83.03176249
83.03176249
83.01351802
83.01351802
83.01351802
83.01351802
83.01351802
83.00102831
83.00102831
83.00102831
83.00102831
83.00102831
83.00572174
83.00572174
83.00572174
83.00572174
83.00572174
83.01925588
83.01925588
83.01925588
83.01925588
83.01925588
83.02208625
83.02208625
83.02208625
83.02208625
83.02208625
82.99820495
82.99820495
82.99820495
82.99820495
82.99820495
82.98359302
82.98359302
82.98359302
82.98359302
82.98359302
82.98259701
82.98259701
82.98259701
82.98259701
82.98259701
83.01331476
83.01331476
83.01331476
83.01331476
83.01331476
83.01667146
83.01667146
83.01667146
83.01667146
83.01667146
82.99374702
82.99374702
82.99374702
82.99374702
82.99374702
83.00913094
83.00913094
83.00913094
83.00913094
83.00913094
83.01148962
83.01148962
83.01148962
83.01148962
83.01148962
82.99022367
82.99022367
82.99022367
82.99022367
82.99022367
82.95999724
82.95999724
82.95999724
82.95999724
82.95999724
82.99875223
82.99875223
82.99875223
82.99875223
82.99875223
83.00075174
83.00075174
83.00075174
83.00075174
83.00075174
82.97790725
82.97790725
82.97790725
82.97790725
82.97790725
82.97934427
82.97934427
82.97934427
82.97934427
82.97934427
83.00782813
83.00782813
83.00782813
83.00782813
83.00782813
83.01373884
83.01373884
83.01373884
83.01373884
83.01373884
83.02824838
83.02824838
83.02824838
83.02824838
83.02824838
83.00672459
83.00672459
83.00672459
83.00672459
83.00672459
83.01171361
83.01171361
83.01171361
83.01171361
83.01171361
83.01412574
83.01412574
83.01412574
83.01412574
83.01412574
83.00030812
83.00030812
83.00030812
83.00030812
83.00030812
83.03175805
83.03175805
83.03175805
83.03175805
83.03175805
83.02151011
83.02151011
83.02151011
83.02151011
83.02151011
83.0039397
83.0039397
83.0039397
83.0039397
83.0039397
83.00506926
83.00506926
83.00506926
83.00506926
83.00506926
82.99882429
82.99882429
82.99882429
82.99882429
82.99882429
82.96779559
82.96779559
82.96779559
82.96779559
82.96779559
82.99783769
82.99783769
82.99783769
82.99783769
82.99783769
83.00219847
83.00219847
83.00219847
83.00219847
83.00219847
83.01830805
83.01830805
83.01830805
83.01830805
83.01830805
83.01548556
83.01548556
83.01548556
83.01548556
83.01548556
82.98476598
82.98476598
82.98476598
82.98476598
82.98476598
83.01923858
83.01923858
83.01923858
83.01923858
83.01923858
82.96737086
82.96737086
82.96737086
82.96737086
82.96737086
82.98132743
82.98132743
82.98132743
82.98132743
82.98132743
82.99717092
82.99717092
82.99717092
82.99717092
82.99717092
83.00268009
83.00268009
83.00268009
83.00268009
83.00268009
83.00569376
83.00569376
83.00569376
83.00569376
83.00569376
83.00886893
83.00886893
83.00886893
83.00886893
83.00886893
83.01375596
83.01375596
83.01375596
83.01375596
83.01375596
82.98282616
82.98282616
82.98282616
82.98282616
82.98282616
83.0181011
83.0181011
83.0181011
83.0181011
83.0181011
83.0044298
83.0044298
83.0044298
83.0044298
83.0044298
83.00088251
83.00088251
83.00088251
83.00088251
83.00088251
83.01912603
83.01912603
83.01912603
83.01912603
83.01912603
83.00163023
83.00163023
83.00163023
83.00163023
83.00163023
82.99577877
82.99577877
82.99577877
82.99577877
82.99577877
83.00005395
83.00005395
83.00005395
83.00005395
83.00005395
83.02362735
83.02362735
83.02362735
83.02362735
83.02362735
83.00877436
83.00877436
83.00877436
83.00877436
83.00877436
82.96263265
82.96263265
82.96263265
82.96263265
82.96263265
82.97472463
82.97472463
82.97472463
82.97472463
82.97472463
82.95655212
82.95655212
82.95655212
82.95655212
82.95655212
83.00664977
83.00664977
83.00664977
83.00664977
83.00664977
83.00097274
83.00097274
83.00097274
83.00097274
83.00097274
82.99508699
82.99508699
82.99508699
82.99508699
82.99508699
82.9563477
82.9563477
82.9563477
82.9563477
82.9563477
82.99652242
82.99652242
82.99652242
82.99652242
82.99652242
83.00521372
83.00521372
83.00521372
83.00521372
83.00521372
82.98009113
82.98009113
82.98009113
82.98009113
82.98009113
83.00562644
83.00562644
83.00562644
83.00562644
83.00562644
83.01729686
83.01729686
83.01729686
83.01729686
83.01729686
82.99992506
82.99992506
82.99992506
82.99992506
82.99992506
82.96846408
82.96846408
82.96846408
82.96846408
82.96846408
82.95200806
82.95200806
82.95200806
82.95200806
82.95200806
83.00413262
83.00413262
83.00413262
83.00413262
83.00413262
82.98153223
82.98153223
82.98153223
82.98153223
82.98153223
83.01130592
83.01130592
83.01130592
83.01130592
83.01130592
82.99092754
82.99092754
82.99092754
82.99092754
82.99092754
83.01184492
83.01184492
83.01184492
83.01184492
83.01184492
83.01707383
83.01707383
83.01707383
83.01707383
83.01707383
82.98840003
82.98840003
82.98840003
82.98840003
82.98840003
83.01904915
83.01904915
83.01904915
83.01904915
83.01904915
83.00713376
83.00713376
83.00713376
83.00713376
83.00713376
83.00492519
83.00492519
83.00492519
83.00492519
83.00492519
83.00259441
83.00259441
83.00259441
83.00259441
83.00259441
83.00294808
83.00294808
83.00294808
83.00294808
83.00294808
82.99340759
82.99340759
82.99340759
82.99340759
82.99340759
82.99585112
82.99585112
82.99585112
82.99585112
82.99585112
83.00218674
83.00218674
83.00218674
83.00218674
83.00218674
83.00583455
83.00583455
83.00583455
83.00583455
83.00583455
83.02655035
83.02655035
83.02655035
83.02655035
83.02655035
83.0134134
83.0134134
83.0134134
83.0134134
83.0134134
82.98503315
82.98503315
82.98503315
82.98503315
82.98503315
82.99769796
82.99769796
82.99769796
82.99769796
82.99769796
82.99793357
82.99793357
82.99793357
82.99793357
82.99793357
82.97992565
82.97992565
82.97992565
82.97992565
82.97992565
82.99651817
82.99651817
82.99651817
82.99651817
82.99651817
83.031909
83.031909
83.031909
83.031909
83.031909
83.00386883
83.00386883
83.00386883
83.00386883
83.00386883
82.99926095
82.99926095
82.99926095
82.99926095
82.99926095
83.00762617
83.00762617
83.00762617
83.00762617
83.00762617
83.00800371
83.00800371
83.00800371
83.00800371
83.00800371
82.97617181
82.97617181
82.97617181
82.97617181
82.97617181
82.9987686
82.9987686
82.9987686
82.9987686
82.9987686
82.98910787
82.98910787
82.98910787
82.98910787
82.98910787
83.00997294
83.00997294
83.00997294
83.00997294
83.00997294
82.99084847
82.99084847
82.99084847
82.99084847
82.99084847
83.01780955
83.01780955
83.01780955
83.01780955
83.01780955
82.99684429
82.99684429
82.99684429
82.99684429
82.99684429
83.00800124
83.00800124
83.00800124
83.00800124
83.00800124
83.01293845
83.01293845
83.01293845
83.01293845
83.01293845
82.99870698
82.99870698
82.99870698
82.99870698
82.99870698
83.00243974
83.00243974
83.00243974
83.00243974
83.00243974
82.99982704
82.99982704
82.99982704
82.99982704
82.99982704
82.98965727
82.98965727
82.98965727
82.98965727
82.98965727
82.99965523
82.99965523
82.99965523
82.99965523
82.99965523
83.00038266
83.00038266
83.00038266
83.00038266
83.00038266
82.9929317
82.9929317
82.9929317
82.9929317
82.9929317
83.02403031
83.02403031
83.02403031
83.02403031
83.02403031
82.97892356
82.97892356
82.97892356
82.97892356
82.97892356
82.98596367
82.98596367
82.98596367
82.98596367
82.98596367
83.00517279
83.00517279
83.00517279
83.00517279
83.00517279
83.01507485
83.01507485
83.01507485
83.01507485
83.01507485
82.98184079
82.98184079
82.98184079
82.98184079
82.98184079
82.99651418
82.99651418
82.99651418
82.99651418
82.99651418
82.99481394
82.99481394
82.99481394
82.99481394
82.99481394
82.95016462
82.95016462
82.95016462
82.95016462
82.95016462
82.98505381
82.98505381
82.98505381
82.98505381
82.98505381
82.99707773
82.99707773
82.99707773
82.99707773
82.99707773
83.02419544
83.02419544
83.02419544
83.02419544
83.02419544
83.00201118
83.00201118
83.00201118
83.00201118
83.00201118
83.00831316
83.00831316
83.00831316
83.00831316
83.00831316
82.98241083
82.98241083
82.98241083
82.98241083
82.98241083
83.01070208
83.01070208
83.01070208
83.01070208
83.01070208
83.01357209
83.01357209
83.01357209
83.01357209
83.01357209
83.00786878
83.00786878
83.00786878
83.00786878
83.00786878
83.00905598
83.00905598
83.00905598
83.00905598
83.00905598
83.02856201
83.02856201
83.02856201
83.02856201
83.02856201
82.9709789
82.9709789
82.9709789
82.9709789
82.9709789
83.02131681
83.02131681
83.02131681
83.02131681
83.02131681
82.99333627
82.99333627
82.99333627
82.99333627
82.99333627
83.0250881
83.0250881
83.0250881
83.0250881
83.0250881
82.98243871
82.98243871
82.98243871
82.98243871
82.98243871
83.01593736
83.01593736
83.01593736
83.01593736
83.01593736
83.01251389
83.01251389
83.01251389
83.01251389
83.01251389
82.99117306
82.99117306
82.99117306
82.99117306
82.99117306
83.02046055
83.02046055
83.02046055
83.02046055
83.02046055
82.97314037
82.97314037
82.97314037
82.97314037
82.97314037
83.00704514
83.00704514
83.00704514
83.00704514
83.00704514
83.01884926
83.01884926
83.01884926
83.01884926
83.01884926
83.01082123
83.01082123
83.01082123
83.01082123
83.01082123
83.00547302
83.00547302
83.00547302
83.00547302
83.00547302
82.9964426
82.9964426
82.9964426
82.9964426
82.9964426
83.02405683
83.02405683
83.02405683
83.02405683
83.02405683
82.98168481
82.98168481
82.98168481
82.98168481
82.98168481
82.95094025
82.95094025
82.95094025
82.95094025
82.95094025
83.02864365
83.02864365
83.02864365
83.02864365
83.02864365
83.00121402
83.00121402
83.00121402
83.00121402
83.00121402
83.00721986
83.00721986
83.00721986
83.00721986
83.00721986
82.99779057
82.99779057
82.99779057
82.99779057
82.99779057
83.00815489
83.00815489
83.00815489
83.00815489
83.00815489
83.00252296
83.00252296
83.00252296
83.00252296
83.00252296
82.98260915
82.98260915
82.98260915
82.98260915
82.98260915
83.00683302
83.00683302
83.00683302
83.00683302
83.00683302
82.99156957
82.99156957
82.99156957
82.99156957
82.99156957
83.01936
83.01936
83.01936
83.01936
83.01936
82.98719724
82.98719724
82.98719724
82.98719724
82.98719724
83.01393376
83.01393376
83.01393376
83.01393376
83.01393376
82.98688718
82.98688718
82.98688718
82.98688718
82.98688718
82.98616159
82.98616159
82.98616159
82.98616159
82.98616159
83.00829805
83.00829805
83.00829805
83.00829805
83.00829805
82.96122296
82.96122296
82.96122296
82.96122296
82.96122296
83.02373511
83.02373511
83.02373511
83.02373511
83.02373511
83.01569438
83.01569438
83.01569438
83.01569438
83.01569438
83.00599773
83.00599773
83.00599773
83.00599773
83.00599773
82.99851797
82.99851797
82.99851797
82.99851797
82.99851797
82.98128849
82.98128849
82.98128849
82.98128849
82.98128849
82.99854586
82.99854586
82.99854586
82.99854586
82.99854586
83.00665025
83.00665025
83.00665025
83.00665025
83.00665025
82.98290268
82.98290268
82.98290268
82.98290268
82.98290268
83.01348526
83.01348526
83.01348526
83.01348526
83.01348526
83.0129537
83.0129537
83.0129537
83.0129537
83.0129537
82.97185643
82.97185643
82.97185643
82.97185643
82.97185643
82.99952484
82.99952484
82.99952484
82.99952484
82.99952484
82.99355821
82.99355821
82.99355821
82.99355821
82.99355821
82.9716938
82.9716938
82.9716938
82.9716938
82.9716938
82.96345111
82.96345111
82.96345111
82.96345111
82.96345111
83.02717134
83.02717134
83.02717134
83.02717134
83.02717134
82.97689728
82.97689728
82.97689728
82.97689728
82.97689728
83.01412214
83.01412214
83.01412214
83.01412214
83.01412214
82.98525696
82.98525696
82.98525696
82.98525696
82.98525696
82.9951496
82.9951496
82.9951496
82.9951496
82.9951496
82.95977225
82.95977225
82.95977225
82.95977225
82.95977225
82.98875729
82.98875729
82.98875729
82.98875729
82.98875729
82.99948144
82.99948144
82.99948144
82.99948144
82.99948144
83.01877741
83.01877741
83.01877741
83.01877741
83.01877741
82.97773985
82.97773985
82.97773985
82.97773985
82.97773985
83.00371404
83.00371404
83.00371404
83.00371404
83.00371404
82.98168444
82.98168444
82.98168444
82.98168444
82.98168444
82.99818048
82.99818048
82.99818048
82.99818048
82.99818048
83.00957938
83.00957938
83.00957938
83.00957938
83.00957938
82.97729307
82.97729307
82.97729307
82.97729307
82.97729307
83.02431127
83.02431127
83.02431127
83.02431127
83.02431127
82.99794911
82.99794911
82.99794911
82.99794911
82.99794911
82.95682503
82.95682503
82.95682503
82.95682503
82.95682503
82.99941143
82.99941143
82.99941143
82.99941143
82.99941143
83.00267937
83.00267937
83.00267937
83.00267937
83.00267937
82.9816314
82.9816314
82.9816314
82.9816314
82.9816314
83.01991026
83.01991026
83.01991026
83.01991026
83.01991026
82.99430366
82.99430366
82.99430366
82.99430366
82.99430366
83.03444095
83.03444095
83.03444095
83.03444095
83.03444095
83.01284093
83.01284093
83.01284093
83.01284093
83.01284093
83.0141924
83.0141924
83.0141924
83.0141924
83.0141924
83.01322298
83.01322298
83.01322298
83.01322298
83.01322298
83.01703525
83.01703525
83.01703525
83.01703525
83.01703525
83.01842991
83.01842991
83.01842991
83.01842991
83.01842991
82.97870133
82.97870133
82.97870133
82.97870133
82.97870133
83.00568181
83.00568181
83.00568181
83.00568181
83.00568181
83.00613781
83.00613781
83.00613781
83.00613781
83.00613781
83.00401139
83.00401139
83.00401139
83.00401139
83.00401139
83.00304069
83.00304069
83.00304069
83.00304069
83.00304069
83.02514701
83.02514701
83.02514701
83.02514701
83.02514701
82.97969383
82.97969383
82.97969383
82.97969383
82.97969383
82.98271199
82.98271199
82.98271199
82.98271199
82.98271199
83.01018354
83.01018354
83.01018354
83.01018354
83.01018354
82.97945352
82.97945352
82.97945352
82.97945352
82.97945352
83.01335518
83.01335518
83.01335518
83.01335518
83.01335518
83.00688204
83.00688204
83.00688204
83.00688204
83.00688204
83.01966078
83.01966078
83.01966078
83.01966078
83.01966078
83.01685259
83.01685259
83.01685259
83.01685259
83.01685259
83.01420686
83.01420686
83.01420686
83.01420686
83.01420686
83.03391483
83.03391483
83.03391483
83.03391483
83.03391483
83.01420921
83.01420921
83.01420921
83.01420921
83.01420921
83.0156375
83.0156375
83.0156375
83.0156375
83.0156375
82.97969185
82.97969185
82.97969185
82.97969185
82.97969185
83.00314553
83.00314553
83.00314553
83.00314553
83.00314553
82.9982908
82.9982908
82.9982908
82.9982908
82.9982908
83.01510989
83.01510989
83.01510989
83.01510989
83.01510989
83.00789534
83.00789534
83.00789534
83.00789534
83.00789534
82.95582843
82.95582843
82.95582843
82.95582843
82.95582843
83.00675317
83.00675317
83.00675317
83.00675317
83.00675317
83.00754592
83.00754592
83.00754592
83.00754592
83.00754592
82.96397803
82.96397803
82.96397803
82.96397803
82.96397803
83.03036913
83.03036913
83.03036913
83.03036913
83.03036913
82.99272805
82.99272805
82.99272805
82.99272805
82.99272805
83.01582214
83.01582214
83.01582214
83.01582214
83.01582214
83.01751883
83.01751883
83.01751883
83.01751883
83.01751883
82.97656892
82.97656892
82.97656892
82.97656892
82.97656892
82.97968282
82.97968282
82.97968282
82.97968282
82.97968282
83.02182565
83.02182565
83.02182565
83.02182565
83.02182565
83.00916558
83.00916558
83.00916558
83.00916558
83.00916558
83.0040404
83.0040404
83.0040404
83.0040404
83.0040404
82.99488272
82.99488272
82.99488272
82.99488272
82.99488272
83.01484026
83.01484026
83.01484026
83.01484026
83.01484026
82.98195672
82.98195672
82.98195672
82.98195672
82.98195672
82.9971674
82.9971674
82.9971674
82.9971674
82.9971674
83.00564551
83.00564551
83.00564551
83.00564551
83.00564551
82.99795656
82.99795656
82.99795656
82.99795656
82.99795656
83.00884983
83.00884983
83.00884983
83.00884983
83.00884983
82.98480717
82.98480717
82.98480717
82.98480717
82.98480717
83.00249851
83.00249851
83.00249851
83.00249851
83.00249851
82.98558971
82.98558971
82.98558971
82.98558971
82.98558971
82.98831229
82.98831229
82.98831229
82.98831229
82.98831229
82.96925376
82.96925376
82.96925376
82.96925376
82.96925376
83.0151635
83.0151635
83.0151635
83.0151635
83.0151635
82.97804872
82.97804872
82.97804872
82.97804872
82.97804872
83.03328375
83.03328375
83.03328375
83.03328375
83.03328375
83.02326776
83.02326776
83.02326776
83.02326776
83.02326776
82.99525272
82.99525272
82.99525272
82.99525272
82.99525272
82.98180663
82.98180663
82.98180663
82.98180663
82.98180663
82.96765895
82.96765895
82.96765895
82.96765895
82.96765895
82.98555656
82.98555656
82.98555656
82.98555656
82.98555656
83.01575587
83.01575587
83.01575587
83.01575587
83.01575587
82.99334188
82.99334188
82.99334188
82.99334188
82.99334188
82.96918481
82.96918481
82.96918481
82.96918481
82.96918481
82.99503438
82.99503438
82.99503438
82.99503438
82.99503438
82.97691899
82.97691899
82.97691899
82.97691899
82.97691899
83.02406718
83.02406718
83.02406718
83.02406718
83.02406718
83.02684473
83.02684473
83.02684473
83.02684473
83.02684473
82.98862399
82.98862399
82.98862399
82.98862399
82.98862399
83.01592569
83.01592569
83.01592569
83.01592569
83.01592569
83.01487894
83.01487894
83.01487894
83.01487894
83.01487894
83.00974364
83.00974364
83.00974364
83.00974364
83.00974364
82.99287312
82.99287312
82.99287312
82.99287312
82.99287312
83.00613824
83.00613824
83.00613824
83.00613824
83.00613824
82.996438
82.996438
82.996438
82.996438
82.996438
82.9736695
82.9736695
82.9736695
82.9736695
82.9736695
83.00689466
83.00689466
83.00689466
83.00689466
83.00689466
83.02511104
83.02511104
83.02511104
83.02511104
83.02511104
82.98296094
82.98296094
82.98296094
82.98296094
82.98296094
83.01861132
83.01861132
83.01861132
83.01861132
83.01861132
83.02786719
83.02786719
83.02786719
83.02786719
83.02786719
82.98560223
82.98560223
82.98560223
82.98560223
82.98560223
83.00528068
83.00528068
83.00528068
83.00528068
83.00528068
82.99694293
82.99694293
82.99694293
82.99694293
82.99694293
83.01322698
83.01322698
83.01322698
83.01322698
83.01322698
83.00615052
83.00615052
83.00615052
83.00615052
83.00615052
83.02293757
83.02293757
83.02293757
83.02293757
83.02293757
82.98758016
82.98758016
82.98758016
82.98758016
82.98758016
83.02061891
83.02061891
83.02061891
83.02061891
83.02061891
82.99587741
82.99587741
82.99587741
82.99587741
82.99587741
82.99156318
82.99156318
82.99156318
82.99156318
82.99156318
82.98933342
82.98933342
82.98933342
82.98933342
82.98933342
83.00817204
83.00817204
83.00817204
83.00817204
83.00817204
83.01202793
83.01202793
83.01202793
83.01202793
83.01202793
83.01440087
83.01440087
83.01440087
83.01440087
83.01440087
82.95845243
82.95845243
82.95845243
82.95845243
82.95845243
82.99364204
82.99364204
82.99364204
82.99364204
82.99364204
83.00736469
83.00736469
83.00736469
83.00736469
83.00736469
83.00570695
83.00570695
83.00570695
83.00570695
83.00570695
83.01686048
83.01686048
83.01686048
83.01686048
83.01686048
82.98727747
82.98727747
82.98727747
82.98727747
82.98727747
83.00497658
83.00497658
83.00497658
83.00497658
83.00497658
83.02113905
83.02113905
83.02113905
83.02113905
83.02113905
83.01517171
83.01517171
83.01517171
83.01517171
83.01517171
83.00635286
83.00635286
83.00635286
83.00635286
83.00635286
82.97982717
82.97982717
82.97982717
82.97982717
82.97982717
83.00677417
83.00677417
83.00677417
83.00677417
83.00677417
82.99799825
82.99799825
82.99799825
82.99799825
82.99799825
83.01771645
83.01771645
83.01771645
83.01771645
83.01771645
83.0242741
83.0242741
83.0242741
83.0242741
83.0242741
83.01300777
83.01300777
83.01300777
83.01300777
83.01300777
83.00219768
83.00219768
83.00219768
83.00219768
83.00219768
82.99171204
82.99171204
82.99171204
82.99171204
82.99171204
83.01101783
83.01101783
83.01101783
83.01101783
83.01101783
82.9851857
82.9851857
82.9851857
82.9851857
82.9851857
82.98421217
82.98421217
82.98421217
82.98421217
82.98421217
82.98091953
82.98091953
82.98091953
82.98091953
82.98091953
83.01885208
83.01885208
83.01885208
83.01885208
83.01885208
83.00609401
83.00609401
83.00609401
83.00609401
83.00609401
82.99188031
82.99188031
82.99188031
82.99188031
82.99188031
82.9738073
82.9738073
82.9738073
82.9738073
82.9738073
82.96882227
82.96882227
82.96882227
82.96882227
82.96882227
82.979157
82.979157
82.979157
82.979157
82.979157
83.00369171
83.00369171
83.00369171
83.00369171
83.00369171
83.01129283
83.01129283
83.01129283
83.01129283
83.01129283
83.00263308
83.00263308
83.00263308
83.00263308
83.00263308
82.98068682
82.98068682
82.98068682
82.98068682
82.98068682
82.99475299
82.99475299
82.99475299
82.99475299
82.99475299
83.02685223
83.02685223
83.02685223
83.02685223
83.02685223
83.00424323
83.00424323
83.00424323
83.00424323
83.00424323
82.98900745
82.98900745
82.98900745
82.98900745
82.98900745
82.99975451
82.99975451
82.99975451
82.99975451
82.99975451
83.01506881
83.01506881
83.01506881
83.01506881
83.01506881
82.9879798
82.9879798
82.9879798
82.9879798
82.9879798
83.00679967
83.00679967
83.00679967
83.00679967
83.00679967
82.97279981
82.97279981
82.97279981
82.97279981
82.97279981
83.0049231
83.0049231
83.0049231
83.0049231
83.0049231
83.00602816
83.00602816
83.00602816
83.00602816
83.00602816
83.02179878
83.02179878
83.02179878
83.02179878
83.02179878
83.00632112
83.00632112
83.00632112
83.00632112
83.00632112
82.95733059
82.95733059
82.95733059
82.95733059
82.95733059
82.98018538
82.98018538
82.98018538
82.98018538
82.98018538
83.00359841
83.00359841
83.00359841
83.00359841
83.00359841
82.9508149
82.9508149
82.9508149
82.9508149
82.9508149
82.9980428
82.9980428
82.9980428
82.9980428
82.9980428
83.00898985
83.00898985
83.00898985
83.00898985
83.00898985
83.00663372
83.00663372
83.00663372
83.00663372
83.00663372
82.98353206
82.98353206
82.98353206
82.98353206
82.98353206
82.98132835
82.98132835
82.98132835
82.98132835
82.98132835
82.97890305
82.97890305
82.97890305
82.97890305
82.97890305
83.00865098
83.00865098
83.00865098
83.00865098
83.00865098
82.98658922
82.98658922
82.98658922
82.98658922
82.98658922
83.027252
83.027252
83.027252
83.027252
83.027252
83.01214192
83.01214192
83.01214192
83.01214192
83.01214192
83.01044626
83.01044626
83.01044626
83.01044626
83.01044626
82.9622366
82.9622366
82.9622366
82.9622366
82.9622366
82.98879156
82.98879156
82.98879156
82.98879156
82.98879156
83.00781741
83.00781741
83.00781741
83.00781741
83.00781741
83.00635261
83.00635261
83.00635261
83.00635261
83.00635261
82.97863991
82.97863991
82.97863991
82.97863991
82.97863991
82.99875299
82.99875299
82.99875299
82.99875299
82.99875299
83.00160163
83.00160163
83.00160163
83.00160163
83.00160163
82.99692082
82.99692082
82.99692082
82.99692082
82.99692082
83.00359992
83.00359992
83.00359992
83.00359992
83.00359992
83.010452
83.010452
83.010452
83.010452
83.010452
82.97533374
82.97533374
82.97533374
82.97533374
82.97533374
82.99693038
82.99693038
82.99693038
82.99693038
82.99693038
83.00991823
83.00991823
83.00991823
83.00991823
83.00991823
82.99662428
82.99662428
82.99662428
82.99662428
82.99662428
82.99336609
82.99336609
82.99336609
82.99336609
82.99336609
83.02176655
83.02176655
83.02176655
83.02176655
83.02176655
83.0082401
83.0082401
83.0082401
83.0082401
83.0082401
83.0189002
83.0189002
83.0189002
83.0189002
83.0189002
83.0071436
83.0071436
83.0071436
83.0071436
83.0071436
83.00230838
83.00230838
83.00230838
83.00230838
83.00230838
83.0191033
83.0191033
83.0191033
83.0191033
83.0191033
82.99420757
82.99420757
82.99420757
82.99420757
82.99420757
82.96109245
82.96109245
82.96109245
82.96109245
82.96109245
82.98659938
82.98659938
82.98659938
82.98659938
82.98659938
83.01351815
83.01351815
83.01351815
83.01351815
83.01351815
83.01463938
83.01463938
83.01463938
83.01463938
83.01463938
82.98087533
82.98087533
82.98087533
82.98087533
82.98087533
82.99692752
82.99692752
82.99692752
82.99692752
82.99692752
82.98269999
82.98269999
82.98269999
82.98269999
82.98269999
83.00712656
83.00712656
83.00712656
83.00712656
83.00712656
83.01314135
83.01314135
83.01314135
83.01314135
83.01314135
82.98286768
82.98286768
82.98286768
82.98286768
82.98286768
83.01492363
83.01492363
83.01492363
83.01492363
83.01492363
82.99703867
82.99703867
82.99703867
82.99703867
82.99703867
83.00389951
83.00389951
83.00389951
83.00389951
83.00389951
82.99262768
82.99262768
82.99262768
82.99262768
82.99262768
83.0095952
83.0095952
83.0095952
83.0095952
83.0095952
82.99802999
82.99802999
82.99802999
82.99802999
82.99802999
83.00031202
83.00031202
83.00031202
83.00031202
83.00031202
82.97710184
82.97710184
82.97710184
82.97710184
82.97710184
83.00729228
83.00729228
83.00729228
83.00729228
83.00729228
83.02793894
83.02793894
83.02793894
83.02793894
83.02793894
83.01901158
83.01901158
83.01901158
83.01901158
83.01901158
83.03432612
83.03432612
83.03432612
83.03432612
83.03432612
82.99987942
82.99987942
82.99987942
82.99987942
82.99987942
83.01756831
83.01756831
83.01756831
83.01756831
83.01756831
83.00581369
83.00581369
83.00581369
83.00581369
83.00581369
83.00502467
83.00502467
83.00502467
83.00502467
83.00502467
82.98329683
82.98329683
82.98329683
82.98329683
82.98329683
83.01201442
83.01201442
83.01201442
83.01201442
83.01201442
83.01431954
83.01431954
83.01431954
83.01431954
83.01431954
82.99207669
82.99207669
82.99207669
82.99207669
82.99207669
83.00605966
83.00605966
83.00605966
83.00605966
83.00605966
82.97312813
82.97312813
82.97312813
82.97312813
82.97312813
82.98391034
82.98391034
82.98391034
82.98391034
82.98391034
83.00819568
83.00819568
83.00819568
83.00819568
83.00819568
82.98099117
82.98099117
82.98099117
82.98099117
82.98099117
82.98714573
82.98714573
82.98714573
82.98714573
82.98714573
82.9883214
82.9883214
82.9883214
82.9883214
82.9883214
83.0021637
83.0021637
83.0021637
83.0021637
83.0021637
83.007131
83.007131
83.007131
83.007131
83.007131
82.95285301
82.95285301
82.95285301
82.95285301
82.95285301
82.99487795
82.99487795
82.99487795
82.99487795
82.99487795
82.96906323
82.96906323
82.96906323
82.96906323
82.96906323
83.00547312
83.00547312
83.00547312
83.00547312
83.00547312
83.00441305
83.00441305
83.00441305
83.00441305
83.00441305
83.02604826
83.02604826
83.02604826
83.02604826
83.02604826
83.03086085
83.03086085
83.03086085
83.03086085
83.03086085
82.97270597
82.97270597
82.97270597
82.97270597
82.97270597
83.0226974
83.0226974
83.0226974
83.0226974
83.0226974
83.01463096
83.01463096
83.01463096
83.01463096
83.01463096
82.96654042
82.96654042
82.96654042
82.96654042
82.96654042
83.00835107
83.00835107
83.00835107
83.00835107
83.00835107
83.00742293
83.00742293
83.00742293
83.00742293
83.00742293
83.02329129
83.02329129
83.02329129
83.02329129
83.02329129
82.97543232
82.97543232
82.97543232
82.97543232
82.97543232
83.03391665
83.03391665
83.03391665
83.03391665
83.03391665
82.98554231
82.98554231
82.98554231
82.98554231
82.98554231
82.97826063
82.97826063
82.97826063
82.97826063
82.97826063
82.98723673
82.98723673
82.98723673
82.98723673
82.98723673
82.99920762
82.99920762
82.99920762
82.99920762
82.99920762
83.02569721
83.02569721
83.02569721
83.02569721
83.02569721
83.00656448
83.00656448
83.00656448
83.00656448
83.00656448
83.03104023
83.03104023
83.03104023
83.03104023
83.03104023
82.99329288
82.99329288
82.99329288
82.99329288
82.99329288
83.01614928
83.01614928
83.01614928
83.01614928
83.01614928
83.0082844
83.0082844
83.0082844
83.0082844
83.0082844
83.00693012
83.00693012
83.00693012
83.00693012
83.00693012
82.98638376
82.98638376
82.98638376
82.98638376
82.98638376
83.02488139
83.02488139
83.02488139
83.02488139
83.02488139
82.99610309
82.99610309
82.99610309
82.99610309
82.99610309
82.97273172
82.97273172
82.97273172
82.97273172
82.97273172
83.01701314
83.01701314
83.01701314
83.01701314
83.01701314
82.99196691
82.99196691
82.99196691
82.99196691
82.99196691
83.01669881
83.01669881
83.01669881
83.01669881
83.01669881
83.01524876
83.01524876
83.01524876
83.01524876
83.01524876
83.02579641
83.02579641
83.02579641
83.02579641
83.02579641
82.9867014
82.9867014
82.9867014
82.9867014
82.9867014
83.00194341
83.00194341
83.00194341
83.00194341
83.00194341
83.00038425
83.00038425
83.00038425
83.00038425
83.00038425
82.98251073
82.98251073
82.98251073
82.98251073
82.98251073
82.98588745
82.98588745
82.98588745
82.98588745
82.98588745
82.97573957
82.97573957
82.97573957
82.97573957
82.97573957
83.00249561
83.00249561
83.00249561
83.00249561
83.00249561
83.0097973
83.0097973
83.0097973
83.0097973
83.0097973
83.00307508
83.00307508
83.00307508
83.00307508
83.00307508
82.97956586
82.97956586
82.97956586
82.97956586
82.97956586
82.99732008
82.99732008
82.99732008
82.99732008
82.99732008
82.99882744
82.99882744
82.99882744
82.99882744
82.99882744
83.0037146
83.0037146
83.0037146
83.0037146
83.0037146
82.99367901
82.99367901
82.99367901
82.99367901
82.99367901
83.01422133
83.01422133
83.01422133
83.01422133
83.01422133
82.96795471
82.96795471
82.96795471
82.96795471
82.96795471
82.99155819
82.99155819
82.99155819
82.99155819
82.99155819
83.00337612
83.00337612
83.00337612
83.00337612
83.00337612
83.01104863
83.01104863
83.01104863
83.01104863
83.01104863
82.99795245
82.99795245
82.99795245
82.99795245
82.99795245
83.03372964
83.03372964
83.03372964
83.03372964
83.03372964
83.01696285
83.01696285
83.01696285
83.01696285
83.01696285
82.9991015
82.9991015
82.9991015
82.9991015
82.9991015
82.99608011
82.99608011
82.99608011
82.99608011
82.99608011
83.00784828
83.00784828
83.00784828
83.00784828
83.00784828
82.9832379
82.9832379
82.9832379
82.9832379
82.9832379
83.01080064
83.01080064
83.01080064
83.01080064
83.01080064
83.00539431
83.00539431
83.00539431
83.00539431
83.00539431
82.99706827
82.99706827
82.99706827
82.99706827
82.99706827
83.01773848
83.01773848
83.01773848
83.01773848
83.01773848
83.0076188
83.0076188
83.0076188
83.0076188
83.0076188
83.01763372
83.01763372
83.01763372
83.01763372
83.01763372
83.00284705
83.00284705
83.00284705
83.00284705
83.00284705
83.0025125
83.0025125
83.0025125
83.0025125
83.0025125
82.9823388
82.9823388
82.9823388
82.9823388
82.9823388
83.00833508
83.00833508
83.00833508
83.00833508
83.00833508
83.00491992
83.00491992
83.00491992
83.00491992
83.00491992
82.99773748
82.99773748
82.99773748
82.99773748
82.99773748
82.9815728
82.9815728
82.9815728
82.9815728
82.9815728
82.99616313
82.99616313
82.99616313
82.99616313
82.99616313
82.99342573
82.99342573
82.99342573
82.99342573
82.99342573
83.00234413
83.00234413
83.00234413
83.00234413
83.00234413
83.00178892
83.00178892
83.00178892
83.00178892
83.00178892
82.98109883
82.98109883
82.98109883
82.98109883
82.98109883
82.98177753
82.98177753
82.98177753
82.98177753
82.98177753
82.99689734
82.99689734
82.99689734
82.99689734
82.99689734
82.99763177
82.99763177
82.99763177
82.99763177
82.99763177
83.00114956
83.00114956
83.00114956
83.00114956
83.00114956
82.98006429
82.98006429
82.98006429
82.98006429
82.98006429
82.99262012
82.99262012
82.99262012
82.99262012
82.99262012
83.01903474
83.01903474
83.01903474
83.01903474
83.01903474
82.99940479
82.99940479
82.99940479
82.99940479
82.99940479
83.02845096
83.02845096
83.02845096
83.02845096
83.02845096
83.02131399
83.02131399
83.02131399
83.02131399
83.02131399
82.97706525
82.97706525
82.97706525
82.97706525
82.97706525
83.01015728
83.01015728
83.01015728
83.01015728
83.01015728
82.97045005
82.97045005
82.97045005
82.97045005
82.97045005
83.00218544
83.00218544
83.00218544
83.00218544
83.00218544
83.00005501
83.00005501
83.00005501
83.00005501
83.00005501
83.0255033
83.0255033
83.0255033
83.0255033
83.0255033
83.01014361
83.01014361
83.01014361
83.01014361
83.01014361
83.00354512
83.00354512
83.00354512
83.00354512
83.00354512
83.02618623
83.02618623
83.02618623
83.02618623
83.02618623
83.00996281
83.00996281
83.00996281
83.00996281
83.00996281
83.02635032
83.02635032
83.02635032
83.02635032
83.02635032
82.99583714
82.99583714
82.99583714
82.99583714
82.99583714
83.00328334
83.00328334
83.00328334
83.00328334
83.00328334
82.99571513
82.99571513
82.99571513
82.99571513
82.99571513
82.97706965
82.97706965
82.97706965
82.97706965
82.97706965
83.00067714
83.00067714
83.00067714
83.00067714
83.00067714
83.00217439
83.00217439
83.00217439
83.00217439
83.00217439
82.97785392
82.97785392
82.97785392
82.97785392
82.97785392
83.02530405
83.02530405
83.02530405
83.02530405
83.02530405
83.0100276
83.0100276
83.0100276
83.0100276
83.0100276
82.99038145
82.99038145
82.99038145
82.99038145
82.99038145
82.97074924
82.97074924
82.97074924
82.97074924
82.97074924
83.00578823
83.00578823
83.00578823
83.00578823
83.00578823
82.99857509
82.99857509
82.99857509
82.99857509
82.99857509
83.01101296
83.01101296
83.01101296
83.01101296
83.01101296
83.00765605
83.00765605
83.00765605
83.00765605
83.00765605
83.00133139
83.00133139
83.00133139
83.00133139
83.00133139
83.00988028
83.00988028
83.00988028
83.00988028
83.00988028
83.00276725
83.00276725
83.00276725
83.00276725
83.00276725
83.01704692
83.01704692
83.01704692
83.01704692
83.01704692
83.00822057
83.00822057
83.00822057
83.00822057
83.00822057
82.97749395
82.97749395
82.97749395
82.97749395
82.97749395
83.00277692
83.00277692
83.00277692
83.00277692
83.00277692
83.0270853
83.0270853
83.0270853
83.0270853
83.0270853
83.02164758
83.02164758
83.02164758
83.02164758
83.02164758
83.00634875
83.00634875
83.00634875
83.00634875
83.00634875
82.98439863
82.98439863
82.98439863
82.98439863
82.98439863
82.99137923
82.99137923
82.99137923
82.99137923
82.99137923
83.01784369
83.01784369
83.01784369
83.01784369
83.01784369
82.97976972
82.97976972
82.97976972
82.97976972
82.97976972
82.9948467
82.9948467
82.9948467
82.9948467
82.9948467
82.98264954
82.98264954
82.98264954
82.98264954
82.98264954
82.99115843
82.99115843
82.99115843
82.99115843
82.99115843
83.00310937
83.00310937
83.00310937
83.00310937
83.00310937
83.0004835
83.0004835
83.0004835
83.0004835
83.0004835
83.00014846
83.00014846
83.00014846
83.00014846
83.00014846
83.02755627
83.02755627
83.02755627
83.02755627
83.02755627
83.00160708
83.00160708
83.00160708
83.00160708
83.00160708
83.01062664
83.01062664
83.01062664
83.01062664
83.01062664
82.99606186
82.99606186
82.99606186
82.99606186
82.99606186
82.98074047
82.98074047
82.98074047
82.98074047
82.98074047
82.99026743
82.99026743
82.99026743
82.99026743
82.99026743
82.98627232
82.98627232
82.98627232
82.98627232
82.98627232
83.01197101
83.01197101
83.01197101
83.01197101
83.01197101
83.00317321
83.00317321
83.00317321
83.00317321
83.00317321
83.00322464
83.00322464
83.00322464
83.00322464
83.00322464
83.0151281
83.0151281
83.0151281
83.0151281
83.0151281
82.99913467
82.99913467
82.99913467
82.99913467
82.99913467
82.98411801
82.98411801
82.98411801
82.98411801
82.98411801
82.97468018
82.97468018
82.97468018
82.97468018
82.97468018
83.00172676
83.00172676
83.00172676
83.00172676
83.00172676
82.99414122
82.99414122
82.99414122
82.99414122
82.99414122
82.99250184
82.99250184
82.99250184
82.99250184
82.99250184
82.98310119
82.98310119
82.98310119
82.98310119
82.98310119
82.99418227
82.99418227
82.99418227
82.99418227
82.99418227
83.02823805
83.02823805
83.02823805
83.02823805
83.02823805
83.01277808
83.01277808
83.01277808
83.01277808
83.01277808
82.99745528
82.99745528
82.99745528
82.99745528
82.99745528
82.96714858
82.96714858
82.96714858
82.96714858
82.96714858
83.00926129
83.00926129
83.00926129
83.00926129
83.00926129
83.00709133
83.00709133
83.00709133
83.00709133
83.00709133
82.97873578
82.97873578
82.97873578
82.97873578
82.97873578
83.02257995
83.02257995
83.02257995
83.02257995
83.02257995
83.00365067
83.00365067
83.00365067
83.00365067
83.00365067
83.0186709
83.0186709
83.0186709
83.0186709
83.0186709
82.99932222
82.99932222
82.99932222
82.99932222
82.99932222
82.97268832
82.97268832
82.97268832
82.97268832
82.97268832
83.02825315
83.02825315
83.02825315
83.02825315
83.02825315
83.00405158
83.00405158
83.00405158
83.00405158
83.00405158
82.99015581
82.99015581
82.99015581
82.99015581
82.99015581
83.01914401
83.01914401
83.01914401
83.01914401
83.01914401
83.01797325
83.01797325
83.01797325
83.01797325
83.01797325
82.94366064
82.94366064
82.94366064
82.94366064
82.94366064
82.97996039
82.97996039
82.97996039
82.97996039
82.97996039
82.98051552
82.98051552
82.98051552
82.98051552
82.98051552
82.9809858
82.9809858
82.9809858
82.9809858
82.9809858
82.97999053
82.97999053
82.97999053
82.97999053
82.97999053
83.02398643
83.02398643
83.02398643
83.02398643
83.02398643
83.00918613
83.00918613
83.00918613
83.00918613
83.00918613
83.02851662
83.02851662
83.02851662
83.02851662
83.02851662
82.97399662
82.97399662
82.97399662
82.97399662
82.97399662
83.02188004
83.02188004
83.02188004
83.02188004
83.02188004
82.99401593
82.99401593
82.99401593
82.99401593
82.99401593
83.0028276
83.0028276
83.0028276
83.0028276
83.0028276
83.02319083
83.02319083
83.02319083
83.02319083
83.02319083
83.00666102
83.00666102
83.00666102
83.00666102
83.00666102
82.97580421
82.97580421
82.97580421
82.97580421
82.97580421
83.02177666
83.02177666
83.02177666
83.02177666
83.02177666
83.00339183
83.00339183
83.00339183
83.00339183
83.00339183
82.99794157
82.99794157
82.99794157
82.99794157
82.99794157
82.99548734
82.99548734
82.99548734
82.99548734
82.99548734
83.00287311
83.00287311
83.00287311
83.00287311
83.00287311
82.996111
82.996111
82.996111
82.996111
82.996111
83.01374487
83.01374487
83.01374487
83.01374487
83.01374487
83.02153865
83.02153865
83.02153865
83.02153865
83.02153865
83.01131488
83.01131488
83.01131488
83.01131488
83.01131488
83.00496863
83.00496863
83.00496863
83.00496863
83.00496863
83.02085799
83.02085799
83.02085799
83.02085799
83.02085799
83.01458838
83.01458838
83.01458838
83.01458838
83.01458838
82.97480602
82.97480602
82.97480602
82.97480602
82.97480602
82.97663379
82.97663379
82.97663379
82.97663379
82.97663379
83.00877846
83.00877846
83.00877846
83.00877846
83.00877846
82.99866637
82.99866637
82.99866637
82.99866637
82.99866637
83.00268908
83.00268908
83.00268908
83.00268908
83.00268908
83.01242752
83.01242752
83.01242752
83.01242752
83.01242752
83.00499756
83.00499756
83.00499756
83.00499756
83.00499756
83.00764495
83.00764495
83.00764495
83.00764495
83.00764495
82.99743799
82.99743799
82.99743799
82.99743799
82.99743799
83.00996137
83.00996137
83.00996137
83.00996137
83.00996137
83.00303951
83.00303951
83.00303951
83.00303951
83.00303951
83.00757194
83.00757194
83.00757194
83.00757194
83.00757194
83.00141989
83.00141989
83.00141989
83.00141989
83.00141989
82.97739642
82.97739642
82.97739642
82.97739642
82.97739642
82.98957253
82.98957253
82.98957253
82.98957253
82.98957253
83.00778931
83.00778931
83.00778931
83.00778931
83.00778931
83.02353185
83.02353185
83.02353185
83.02353185
83.02353185
82.99606344
82.99606344
82.99606344
82.99606344
82.99606344
83.01161689
83.01161689
83.01161689
83.01161689
83.01161689
82.94082067
82.94082067
82.94082067
82.94082067
82.94082067
83.0089404
83.0089404
83.0089404
83.0089404
83.0089404
82.99280841
82.99280841
82.99280841
82.99280841
82.99280841
83.02563095
83.02563095
83.02563095
83.02563095
83.02563095
83.03335083
83.03335083
83.03335083
83.03335083
83.03335083
83.01254294
83.01254294
83.01254294
83.01254294
83.01254294
82.9790697
82.9790697
82.9790697
82.9790697
82.9790697
83.00780327
83.00780327
83.00780327
83.00780327
83.00780327
83.01537586
83.01537586
83.01537586
83.01537586
83.01537586
83.01143862
83.01143862
83.01143862
83.01143862
83.01143862
83.00452908
83.00452908
83.00452908
83.00452908
83.00452908
83.00166926
83.00166926
83.00166926
83.00166926
83.00166926
83.00319988
83.00319988
83.00319988
83.00319988
83.00319988
83.00731306
83.00731306
83.00731306
83.00731306
83.00731306
83.00535433
83.00535433
83.00535433
83.00535433
83.00535433
82.99825949
82.99825949
82.99825949
82.99825949
82.99825949
83.00540157
83.00540157
83.00540157
83.00540157
83.00540157
83.00445754
83.00445754
83.00445754
83.00445754
83.00445754
83.00139204
83.00139204
83.00139204
83.00139204
83.00139204
82.97875915
82.97875915
82.97875915
82.97875915
82.97875915
83.01152976
83.01152976
83.01152976
83.01152976
83.01152976
82.99841339
82.99841339
82.99841339
82.99841339
82.99841339
82.99872376
82.99872376
82.99872376
82.99872376
82.99872376
83.03519454
83.03519454
83.03519454
83.03519454
83.03519454
82.97718857
82.97718857
82.97718857
82.97718857
82.97718857
82.99435629
82.99435629
82.99435629
82.99435629
82.99435629
83.00929679
83.00929679
83.00929679
83.00929679
83.00929679
82.98101136
82.98101136
82.98101136
82.98101136
82.98101136
82.99272977
82.99272977
82.99272977
82.99272977
82.99272977
83.00606151
83.00606151
83.00606151
83.00606151
83.00606151
83.01593962
83.01593962
83.01593962
83.01593962
83.01593962
83.0105035
83.0105035
83.0105035
83.0105035
83.0105035
83.00725897
83.00725897
83.00725897
83.00725897
83.00725897
82.97894498
82.97894498
82.97894498
82.97894498
82.97894498
83.01186799
83.01186799
83.01186799
83.01186799
83.01186799
83.00273449
83.00273449
83.00273449
83.00273449
83.00273449
82.98470222
82.98470222
82.98470222
82.98470222
82.98470222
83.0229525
83.0229525
83.0229525
83.0229525
83.0229525
83.01267371
83.01267371
83.01267371
83.01267371
83.01267371
82.99974394
82.99974394
82.99974394
82.99974394
82.99974394
82.99757506
82.99757506
82.99757506
82.99757506
82.99757506
82.98209831
82.98209831
82.98209831
82.98209831
82.98209831
82.97049014
82.97049014
82.97049014
82.97049014
82.97049014
83.00014104
83.00014104
83.00014104
83.00014104
83.00014104
83.00342481
83.00342481
83.00342481
83.00342481
83.00342481
82.94110109
82.94110109
82.94110109
82.94110109
82.94110109
82.9966013
82.9966013
82.9966013
82.9966013
82.9966013
83.01384585
83.01384585
83.01384585
83.01384585
83.01384585
82.95965561
82.95965561
82.95965561
82.95965561
82.95965561
82.96628823
82.96628823
82.96628823
82.96628823
82.96628823
82.98871771
82.98871771
82.98871771
82.98871771
82.98871771
83.03636158
83.03636158
83.03636158
83.03636158
83.03636158
83.00796708
83.00796708
83.00796708
83.00796708
83.00796708
83.01920264
83.01920264
83.01920264
83.01920264
83.01920264
83.01381355
83.01381355
83.01381355
83.01381355
83.01381355
83.00377873
83.00377873
83.00377873
83.00377873
83.00377873
83.0048445
83.0048445
83.0048445
83.0048445
83.0048445
83.03331416
83.03331416
83.03331416
83.03331416
83.03331416
82.99191512
82.99191512
82.99191512
82.99191512
82.99191512
82.99959411
82.99959411
82.99959411
82.99959411
82.99959411
83.0086855
83.0086855
83.0086855
83.0086855
83.0086855
83.00674724
83.00674724
83.00674724
83.00674724
83.00674724
83.01827306
83.01827306
83.01827306
83.01827306
83.01827306
82.99711412
82.99711412
82.99711412
82.99711412
82.99711412
82.9879478
82.9879478
82.9879478
82.9879478
82.9879478
82.99254825
82.99254825
82.99254825
82.99254825
82.99254825
82.96281675
82.96281675
82.96281675
82.96281675
82.96281675
82.95018748
82.95018748
82.95018748
82.95018748
82.95018748
82.98495875
82.98495875
82.98495875
82.98495875
82.98495875
82.98709785
82.98709785
82.98709785
82.98709785
82.98709785
83.00746383
83.00746383
83.00746383
83.00746383
83.00746383
83.02226013
83.02226013
83.02226013
83.02226013
83.02226013
83.00337998
83.00337998
83.00337998
83.00337998
83.00337998
82.98849162
82.98849162
82.98849162
82.98849162
82.98849162
82.99939487
82.99939487
82.99939487
82.99939487
82.99939487
83.00117128
83.00117128
83.00117128
83.00117128
83.00117128
83.00385081
83.00385081
83.00385081
83.00385081
83.00385081
82.99868999
82.99868999
82.99868999
82.99868999
82.99868999
83.02095004
83.02095004
83.02095004
83.02095004
83.02095004
83.01138887
83.01138887
83.01138887
83.01138887
83.01138887
83.0034785
83.0034785
83.0034785
83.0034785
83.0034785
82.96570111
82.96570111
82.96570111
82.96570111
82.96570111
83.01468727
83.01468727
83.01468727
83.01468727
83.01468727
82.95881225
82.95881225
82.95881225
82.95881225
82.95881225
82.96079654
82.96079654
82.96079654
82.96079654
82.96079654
82.98192192
82.98192192
82.98192192
82.98192192
82.98192192
83.01883179
83.01883179
83.01883179
83.01883179
83.01883179
83.00343001
83.00343001
83.00343001
83.00343001
83.00343001
83.00184326
83.00184326
83.00184326
83.00184326
83.00184326
83.01135484
83.01135484
83.01135484
83.01135484
83.01135484
83.02078228
83.02078228
83.02078228
83.02078228
83.02078228
82.992242
82.992242
82.992242
82.992242
82.992242
83.01165336
83.01165336
83.01165336
83.01165336
83.01165336
83.00738009
83.00738009
83.00738009
83.00738009
83.00738009
83.00786751
83.00786751
83.00786751
83.00786751
83.00786751
83.01721549
83.01721549
83.01721549
83.01721549
83.01721549
83.00022328
83.00022328
83.00022328
83.00022328
83.00022328
83.01856265
83.01856265
83.01856265
83.01856265
83.01856265
82.99228483
82.99228483
82.99228483
82.99228483
82.99228483
83.00217035
83.00217035
83.00217035
83.00217035
83.00217035
82.99218063
82.99218063
82.99218063
82.99218063
82.99218063
82.98318902
82.98318902
82.98318902
82.98318902
82.98318902
83.00430603
83.00430603
83.00430603
83.00430603
83.00430603
83.00280772
83.00280772
83.00280772
83.00280772
83.00280772
82.97384247
82.97384247
82.97384247
82.97384247
82.97384247
83.01076882
83.01076882
83.01076882
83.01076882
83.01076882
82.99489464
82.99489464
82.99489464
82.99489464
82.99489464
82.99153115
82.99153115
82.99153115
82.99153115
82.99153115
82.97366053
82.97366053
82.97366053
82.97366053
82.97366053
82.98462058
82.98462058
82.98462058
82.98462058
82.98462058
82.99681669
82.99681669
82.99681669
82.99681669
82.99681669
83.00133865
83.00133865
83.00133865
83.00133865
83.00133865
83.00462865
83.00462865
83.00462865
83.00462865
83.00462865
83.00480181
83.00480181
83.00480181
83.00480181
83.00480181
82.99692586
82.99692586
82.99692586
82.99692586
82.99692586
82.97752011
82.97752011
82.97752011
82.97752011
82.97752011
83.01149564
83.01149564
83.01149564
83.01149564
83.01149564
82.98090603
82.98090603
82.98090603
82.98090603
82.98090603
82.95110896
82.95110896
82.95110896
82.95110896
82.95110896
83.02859662
83.02859662
83.02859662
83.02859662
83.02859662
82.97953162
82.97953162
82.97953162
82.97953162
82.97953162
83.01502368
83.01502368
83.01502368
83.01502368
83.01502368
82.98904587
82.98904587
82.98904587
82.98904587
82.98904587
83.01055556
83.01055556
83.01055556
83.01055556
83.01055556
83.03054696
83.03054696
83.03054696
83.03054696
83.03054696
82.99179381
82.99179381
82.99179381
82.99179381
82.99179381
83.00626059
83.00626059
83.00626059
83.00626059
83.00626059
82.99098461
82.99098461
82.99098461
82.99098461
82.99098461
83.01157365
83.01157365
83.01157365
83.01157365
83.01157365
83.01566815
83.01566815
83.01566815
83.01566815
83.01566815
83.00213938
83.00213938
83.00213938
83.00213938
83.00213938
82.99732926
82.99732926
82.99732926
82.99732926
82.99732926
83.00671306
83.00671306
83.00671306
83.00671306
83.00671306
83.00531875
83.00531875
83.00531875
83.00531875
83.00531875
83.0212558
83.0212558
83.0212558
83.0212558
83.0212558
82.98110795
82.98110795
82.98110795
82.98110795
82.98110795
83.01632882
83.01632882
83.01632882
83.01632882
83.01632882
83.01306141
83.01306141
83.01306141
83.01306141
83.01306141
83.00399397
83.00399397
83.00399397
83.00399397
83.00399397
83.00370665
83.00370665
83.00370665
83.00370665
83.00370665
83.0196036
83.0196036
83.0196036
83.0196036
83.0196036
83.00963422
83.00963422
83.00963422
83.00963422
83.00963422
83.00782763
83.00782763
83.00782763
83.00782763
83.00782763
83.00638745
83.00638745
83.00638745
83.00638745
83.00638745
83.00543732
83.00543732
83.00543732
83.00543732
83.00543732
82.97200895
82.97200895
82.97200895
82.97200895
82.97200895
82.99693072
82.99693072
82.99693072
82.99693072
82.99693072
83.00198015
83.00198015
83.00198015
83.00198015
83.00198015
83.00619798
83.00619798
83.00619798
83.00619798
83.00619798
83.00630716
83.00630716
83.00630716
83.00630716
83.00630716
83.00962109
83.00962109
83.00962109
83.00962109
83.00962109
83.00092054
83.00092054
83.00092054
83.00092054
83.00092054
83.00312815
83.00312815
83.00312815
83.00312815
83.00312815
82.9911873
82.9911873
82.9911873
82.9911873
82.9911873
83.00413851
83.00413851
83.00413851
83.00413851
83.00413851
83.00755089
83.00755089
83.00755089
83.00755089
83.00755089
82.97956013
82.97956013
82.97956013
82.97956013
82.97956013
82.97914497
82.97914497
82.97914497
82.97914497
82.97914497
82.9801461
82.9801461
82.9801461
82.9801461
82.9801461
83.01613375
83.01613375
83.01613375
83.01613375
83.01613375
83.01202291
83.01202291
83.01202291
83.01202291
83.01202291
82.98023644
82.98023644
82.98023644
82.98023644
82.98023644
82.98004389
82.98004389
82.98004389
82.98004389
82.98004389
83.00082404
83.00082404
83.00082404
83.00082404
83.00082404
83.02910863
83.02910863
83.02910863
83.02910863
83.02910863
83.0290272
83.0290272
83.0290272
83.0290272
83.0290272
83.01135028
83.01135028
83.01135028
83.01135028
83.01135028
83.02919589
83.02919589
83.02919589
83.02919589
83.02919589
82.99261417
82.99261417
82.99261417
82.99261417
82.99261417
82.96292031
82.96292031
82.96292031
82.96292031
82.96292031
83.00598086
83.00598086
83.00598086
83.00598086
83.00598086
83.00170022
83.00170022
83.00170022
83.00170022
83.00170022
82.98917088
82.98917088
82.98917088
82.98917088
82.98917088
83.00446229
83.00446229
83.00446229
83.00446229
83.00446229
83.02106147
83.02106147
83.02106147
83.02106147
83.02106147
82.99879239
82.99879239
82.99879239
82.99879239
82.99879239
83.017413
83.017413
83.017413
83.017413
83.017413
83.01060206
83.01060206
83.01060206
83.01060206
83.01060206
82.97624887
82.97624887
82.97624887
82.97624887
82.97624887
82.98406065
82.98406065
82.98406065
82.98406065
82.98406065
82.9824869
82.9824869
82.9824869
82.9824869
82.9824869
82.95937332
82.95937332
82.95937332
82.95937332
82.95937332
82.99318626
82.99318626
82.99318626
82.99318626
82.99318626
82.98686622
82.98686622
82.98686622
82.98686622
82.98686622
83.00031107
83.00031107
83.00031107
83.00031107
83.00031107
83.01932094
83.01932094
83.01932094
83.01932094
83.01932094
83.01379548
83.01379548
83.01379548
83.01379548
83.01379548
83.00560051
83.00560051
83.00560051
83.00560051
83.00560051
82.98602072
82.98602072
82.98602072
82.98602072
82.98602072
83.01949108
83.01949108
83.01949108
83.01949108
83.01949108
83.01206091
83.01206091
83.01206091
83.01206091
83.01206091
82.93849938
82.93849938
82.93849938
82.93849938
82.93849938
83.00239353
83.00239353
83.00239353
83.00239353
83.00239353
82.97329133
82.97329133
82.97329133
82.97329133
82.97329133
82.99672428
82.99672428
82.99672428
82.99672428
82.99672428
82.98955666
82.98955666
82.98955666
82.98955666
82.98955666
82.99043493
82.99043493
82.99043493
82.99043493
82.99043493
82.98894239
82.98894239
82.98894239
82.98894239
82.98894239
83.00420893
83.00420893
83.00420893
83.00420893
83.00420893
83.00569059
83.00569059
83.00569059
83.00569059
83.00569059
83.02707772
83.02707772
83.02707772
83.02707772
83.02707772
82.95682841
82.95682841
82.95682841
82.95682841
82.95682841
82.99399095
82.99399095
82.99399095
82.99399095
82.99399095
83.01502981
83.01502981
83.01502981
83.01502981
83.01502981
82.98710038
82.98710038
82.98710038
82.98710038
82.98710038
82.9924655
82.9924655
82.9924655
82.9924655
82.9924655
82.9768258
82.9768258
82.9768258
82.9768258
82.9768258
83.00730556
83.00730556
83.00730556
83.00730556
83.00730556
83.01732521
83.01732521
83.01732521
83.01732521
83.01732521
83.01448414
83.01448414
83.01448414
83.01448414
83.01448414
83.00570056
83.00570056
83.00570056
83.00570056
83.00570056
83.0054241
83.0054241
83.0054241
83.0054241
83.0054241
83.02421949
83.02421949
83.02421949
83.02421949
83.02421949
83.00920606
83.00920606
83.00920606
83.00920606
83.00920606
83.01455188
83.01455188
83.01455188
83.01455188
83.01455188
83.01400431
83.01400431
83.01400431
83.01400431
83.01400431
82.98733467
82.98733467
82.98733467
82.98733467
82.98733467
83.00782907
83.00782907
83.00782907
83.00782907
83.00782907
83.00674575
83.00674575
83.00674575
83.00674575
83.00674575
82.99802978
82.99802978
82.99802978
82.99802978
82.99802978
82.97270533
82.97270533
82.97270533
82.97270533
82.97270533
82.99366908
82.99366908
82.99366908
82.99366908
82.99366908
83.01712901
83.01712901
83.01712901
83.01712901
83.01712901
83.01328258
83.01328258
83.01328258
83.01328258
83.01328258
83.00818492
83.00818492
83.00818492
83.00818492
83.00818492
83.01189666
83.01189666
83.01189666
83.01189666
83.01189666
82.99923396
82.99923396
82.99923396
82.99923396
82.99923396
83.00571726
83.00571726
83.00571726
83.00571726
83.00571726
83.01575813
83.01575813
83.01575813
83.01575813
83.01575813
83.02474709
83.02474709
83.02474709
83.02474709
83.02474709
82.99534792
82.99534792
82.99534792
82.99534792
82.99534792
83.00354318
83.00354318
83.00354318
83.00354318
83.00354318
83.00575894
83.00575894
83.00575894
83.00575894
83.00575894
83.00452703
83.00452703
83.00452703
83.00452703
83.00452703
83.00189354
83.00189354
83.00189354
83.00189354
83.00189354
83.00366148
83.00366148
83.00366148
83.00366148
83.00366148
83.01270621
83.01270621
83.01270621
83.01270621
83.01270621
82.9936804
82.9936804
82.9936804
82.9936804
82.9936804
82.99134471
82.99134471
82.99134471
82.99134471
82.99134471
82.98558311
82.98558311
82.98558311
82.98558311
82.98558311
83.00590626
83.00590626
83.00590626
83.00590626
83.00590626
82.99434698
82.99434698
82.99434698
82.99434698
82.99434698
83.01721986
83.01721986
83.01721986
83.01721986
83.01721986
82.9775663
82.9775663
82.9775663
82.9775663
82.9775663
83.0220874
83.0220874
83.0220874
83.0220874
83.0220874
82.97132169
82.97132169
82.97132169
82.97132169
82.97132169
83.01846386
83.01846386
83.01846386
83.01846386
83.01846386
82.97958277
82.97958277
82.97958277
82.97958277
82.97958277
82.99901062
82.99901062
82.99901062
82.99901062
82.99901062
83.00292468
83.00292468
83.00292468
83.00292468
83.00292468
83.00828587
83.00828587
83.00828587
83.00828587
83.00828587
83.01199844
83.01199844
83.01199844
83.01199844
83.01199844
82.99674061
82.99674061
82.99674061
82.99674061
82.99674061
83.01749462
83.01749462
83.01749462
83.01749462
83.01749462
82.98277407
82.98277407
82.98277407
82.98277407
82.98277407
83.01134649
83.01134649
83.01134649
83.01134649
83.01134649
82.99460042
82.99460042
82.99460042
82.99460042
82.99460042
83.00435348
83.00435348
83.00435348
83.00435348
83.00435348
83.00746081
83.00746081
83.00746081
83.00746081
83.00746081
82.97394128
82.97394128
82.97394128
82.97394128
82.97394128
83.00385648
83.00385648
83.00385648
83.00385648
83.00385648
83.01107089
83.01107089
83.01107089
83.01107089
83.01107089
83.0053644
83.0053644
83.0053644
83.0053644
83.0053644
82.9960985
82.9960985
82.9960985
82.9960985
82.9960985
83.00573298
83.00573298
83.00573298
83.00573298
83.00573298
82.99934219
82.99934219
82.99934219
82.99934219
82.99934219
83.02021029
83.02021029
83.02021029
83.02021029
83.02021029
83.01458038
83.01458038
83.01458038
83.01458038
83.01458038
83.00198587
83.00198587
83.00198587
83.00198587
83.00198587
83.02020718
83.02020718
83.02020718
83.02020718
83.02020718
83.00714521
83.00714521
83.00714521
83.00714521
83.00714521
83.00842734
83.00842734
83.00842734
83.00842734
83.00842734
83.00558765
83.00558765
83.00558765
83.00558765
83.00558765
82.97783233
82.97783233
82.97783233
82.97783233
82.97783233
83.01806107
83.01806107
83.01806107
83.01806107
83.01806107
83.00998666
83.00998666
83.00998666
83.00998666
83.00998666
83.00639578
83.00639578
83.00639578
83.00639578
83.00639578
82.99851165
82.99851165
82.99851165
82.99851165
82.99851165
83.01039341
83.01039341
83.01039341
83.01039341
83.01039341
83.00246812
83.00246812
83.00246812
83.00246812
83.00246812
83.00400915
83.00400915
83.00400915
83.00400915
83.00400915
82.9957459
82.9957459
82.9957459
82.9957459
82.9957459
83.00600743
83.00600743
83.00600743
83.00600743
83.00600743
82.9782774
82.9782774
82.9782774
82.9782774
82.9782774
82.98471994
82.98471994
82.98471994
82.98471994
82.98471994
83.02337459
83.02337459
83.02337459
83.02337459
83.02337459
82.9834973
82.9834973
82.9834973
82.9834973
82.9834973
83.00742387
83.00742387
83.00742387
83.00742387
83.00742387
83.00863647
83.00863647
83.00863647
83.00863647
83.00863647
83.00895543
83.00895543
83.00895543
83.00895543
83.00895543
83.01019384
83.01019384
83.01019384
83.01019384
83.01019384
82.99551661
82.99551661
82.99551661
82.99551661
82.99551661
82.97597716
82.97597716
82.97597716
82.97597716
82.97597716
83.01131038
83.01131038
83.01131038
83.01131038
83.01131038
82.99458614
82.99458614
82.99458614
82.99458614
82.99458614
83.01489955
83.01489955
83.01489955
83.01489955
83.01489955
83.01035384
83.01035384
83.01035384
83.01035384
83.01035384
83.00929983
83.00929983
83.00929983
83.00929983
83.00929983
82.99395929
82.99395929
82.99395929
82.99395929
82.99395929
83.0080868
83.0080868
83.0080868
83.0080868
83.0080868
83.01201724
83.01201724
83.01201724
83.01201724
83.01201724
83.02452842
83.02452842
83.02452842
83.02452842
83.02452842
82.99366545
82.99366545
82.99366545
82.99366545
82.99366545
83.02554785
83.02554785
83.02554785
83.02554785
83.02554785
82.97327911
82.97327911
82.97327911
82.97327911
82.97327911
82.99951397
82.99951397
82.99951397
82.99951397
82.99951397
83.02193748
83.02193748
83.02193748
83.02193748
83.02193748
83.00199768
83.00199768
83.00199768
83.00199768
83.00199768
83.02482217
83.02482217
83.02482217
83.02482217
83.02482217
83.00925918
83.00925918
83.00925918
83.00925918
83.00925918
83.0174879
83.0174879
83.0174879
83.0174879
83.0174879
83.0008699
83.0008699
83.0008699
83.0008699
83.0008699
83.02353144
83.02353144
83.02353144
83.02353144
83.02353144
82.99414963
82.99414963
82.99414963
82.99414963
82.99414963
83.03295786
83.03295786
83.03295786
83.03295786
83.03295786
83.01784914
83.01784914
83.01784914
83.01784914
83.01784914
83.00411347
83.00411347
83.00411347
83.00411347
83.00411347
82.98694601
82.98694601
82.98694601
82.98694601
82.98694601
82.98203128
82.98203128
82.98203128
82.98203128
82.98203128
82.99216193
82.99216193
82.99216193
82.99216193
82.99216193
83.00704754
83.00704754
83.00704754
83.00704754
83.00704754
83.00333879
83.00333879
83.00333879
83.00333879
83.00333879
83.01631398
83.01631398
83.01631398
83.01631398
83.01631398
83.01224874
83.01224874
83.01224874
83.01224874
83.01224874
83.00341629
83.00341629
83.00341629
83.00341629
83.00341629
83.02510559
83.02510559
83.02510559
83.02510559
83.02510559
82.96912746
82.96912746
82.96912746
82.96912746
82.96912746
82.99871268
82.99871268
82.99871268
82.99871268
82.99871268
83.00401515
83.00401515
83.00401515
83.00401515
83.00401515
82.98949748
82.98949748
82.98949748
82.98949748
82.98949748
82.99161713
82.99161713
82.99161713
82.99161713
82.99161713
83.00590419
83.00590419
83.00590419
83.00590419
83.00590419
82.96971653
82.96971653
82.96971653
82.96971653
82.96971653
82.95237616
82.95237616
82.95237616
82.95237616
82.95237616
83.01003495
83.01003495
83.01003495
83.01003495
83.01003495
82.99335327
82.99335327
82.99335327
82.99335327
82.99335327
83.008213
83.008213
83.008213
83.008213
83.008213
83.00468523
83.00468523
83.00468523
83.00468523
83.00468523
82.97310253
82.97310253
82.97310253
82.97310253
82.97310253
83.01749883
83.01749883
83.01749883
83.01749883
83.01749883
82.97019813
82.97019813
82.97019813
82.97019813
82.97019813
82.95735003
82.95735003
82.95735003
82.95735003
82.95735003
82.96754415
82.96754415
82.96754415
82.96754415
82.96754415
83.01060213
83.01060213
83.01060213
83.01060213
83.01060213
83.01102992
83.01102992
83.01102992
83.01102992
83.01102992
83.00686915
83.00686915
83.00686915
83.00686915
83.00686915
82.97627612
82.97627612
82.97627612
82.97627612
82.97627612
83.02334011
83.02334011
83.02334011
83.02334011
83.02334011
83.00853715
83.00853715
83.00853715
83.00853715
83.00853715
82.99807466
82.99807466
82.99807466
82.99807466
82.99807466
83.01095713
83.01095713
83.01095713
83.01095713
83.01095713
83.01007215
83.01007215
83.01007215
83.01007215
83.01007215
82.98571468
82.98571468
82.98571468
82.98571468
82.98571468
83.01317472
83.01317472
83.01317472
83.01317472
83.01317472
82.99357363
82.99357363
82.99357363
82.99357363
82.99357363
82.99988325
82.99988325
82.99988325
82.99988325
82.99988325
82.99190486
82.99190486
82.99190486
82.99190486
82.99190486
83.00123804
83.00123804
83.00123804
83.00123804
83.00123804
83.00801288
83.00801288
83.00801288
83.00801288
83.00801288
82.99026001
82.99026001
82.99026001
82.99026001
82.99026001
82.99696389
82.99696389
82.99696389
82.99696389
82.99696389
83.00263688
83.00263688
83.00263688
83.00263688
83.00263688
83.00134372
83.00134372
83.00134372
83.00134372
83.00134372
82.98235548
82.98235548
82.98235548
82.98235548
82.98235548
83.02452887
83.02452887
83.02452887
83.02452887
83.02452887
82.96746279
82.96746279
82.96746279
82.96746279
82.96746279
82.99827641
82.99827641
82.99827641
82.99827641
82.99827641
83.02590529
83.02590529
83.02590529
83.02590529
83.02590529
82.99795877
82.99795877
82.99795877
82.99795877
82.99795877
82.98885315
82.98885315
82.98885315
82.98885315
82.98885315
83.0268575
83.0268575
83.0268575
83.0268575
83.0268575
83.00892617
83.00892617
83.00892617
83.00892617
83.00892617
82.98557214
82.98557214
82.98557214
82.98557214
82.98557214
83.01578433
83.01578433
83.01578433
83.01578433
83.01578433
83.0065247
83.0065247
83.0065247
83.0065247
83.0065247
83.00979384
83.00979384
83.00979384
83.00979384
83.00979384
83.00613825
83.00613825
83.00613825
83.00613825
83.00613825
83.0138981
83.0138981
83.0138981
83.0138981
83.0138981
82.97047057
82.97047057
82.97047057
82.97047057
82.97047057
82.99348171
82.99348171
82.99348171
82.99348171
82.99348171
82.98224204
82.98224204
82.98224204
82.98224204
82.98224204
83.00234847
83.00234847
83.00234847
83.00234847
83.00234847
83.00455156
83.00455156
83.00455156
83.00455156
83.00455156
83.01711712
83.01711712
83.01711712
83.01711712
83.01711712
83.0230365
83.0230365
83.0230365
83.0230365
83.0230365
83.01361127
83.01361127
83.01361127
83.01361127
83.01361127
83.01296111
83.01296111
83.01296111
83.01296111
83.01296111
82.99384513
82.99384513
82.99384513
82.99384513
82.99384513
82.99666066
82.99666066
82.99666066
82.99666066
82.99666066
83.01190855
83.01190855
83.01190855
83.01190855
83.01190855
82.97659966
82.97659966
82.97659966
82.97659966
82.97659966
83.00453386
83.00453386
83.00453386
83.00453386
83.00453386
82.96972903
82.96972903
82.96972903
82.96972903
82.96972903
83.00713897
83.00713897
83.00713897
83.00713897
83.00713897
83.00081699
83.00081699
83.00081699
83.00081699
83.00081699
82.9979626
82.9979626
82.9979626
82.9979626
82.9979626
83.01325985
83.01325985
83.01325985
83.01325985
83.01325985
83.00578962
83.00578962
83.00578962
83.00578962
83.00578962
82.97939157
82.97939157
82.97939157
82.97939157
82.97939157
83.0085144
83.0085144
83.0085144
83.0085144
83.0085144
83.01130089
83.01130089
83.01130089
83.01130089
83.01130089
83.0114714
83.0114714
83.0114714
83.0114714
83.0114714
83.02407643
83.02407643
83.02407643
83.02407643
83.02407643
83.02683467
83.02683467
83.02683467
83.02683467
83.02683467
83.0281188
83.0281188
83.0281188
83.0281188
83.0281188
83.0180675
83.0180675
83.0180675
83.0180675
83.0180675
83.02110136
83.02110136
83.02110136
83.02110136
83.02110136
83.00648186
83.00648186
83.00648186
83.00648186
83.00648186
83.00394645
83.00394645
83.00394645
83.00394645
83.00394645
83.02134085
83.02134085
83.02134085
83.02134085
83.02134085
82.95674435
82.95674435
82.95674435
82.95674435
82.95674435
83.0152798
83.0152798
83.0152798
83.0152798
83.0152798
83.01704591
83.01704591
83.01704591
83.01704591
83.01704591
82.99739717
82.99739717
82.99739717
82.99739717
82.99739717
83.00224574
83.00224574
83.00224574
83.00224574
83.00224574
83.02654064
83.02654064
83.02654064
83.02654064
83.02654064
83.02393823
83.02393823
83.02393823
83.02393823
83.02393823
82.97596988
82.97596988
82.97596988
82.97596988
82.97596988
83.00461593
83.00461593
83.00461593
83.00461593
83.00461593
82.98676119
82.98676119
82.98676119
82.98676119
82.98676119
83.00155691
83.00155691
83.00155691
83.00155691
83.00155691
83.00757646
83.00757646
83.00757646
83.00757646
83.00757646
83.02280562
83.02280562
83.02280562
83.02280562
83.02280562
82.99281328
82.99281328
82.99281328
82.99281328
82.99281328
83.00980014
83.00980014
83.00980014
83.00980014
83.00980014
83.01104755
83.01104755
83.01104755
83.01104755
83.01104755
82.99183412
82.99183412
82.99183412
82.99183412
82.99183412
82.94421551
82.94421551
82.94421551
82.94421551
82.94421551
83.00560857
83.00560857
83.00560857
83.00560857
83.00560857
82.97118029
82.97118029
82.97118029
82.97118029
82.97118029
83.01072731
83.01072731
83.01072731
83.01072731
83.01072731
82.99218727
82.99218727
82.99218727
82.99218727
82.99218727
82.96806057
82.96806057
82.96806057
82.96806057
82.96806057
83.01776
83.01776
83.01776
83.01776
83.01776
83.01295598
83.01295598
83.01295598
83.01295598
83.01295598
82.9698514
82.9698514
82.9698514
82.9698514
82.9698514
82.96799657
82.96799657
82.96799657
82.96799657
82.96799657
83.00780777
83.00780777
83.00780777
83.00780777
83.00780777
82.97853231
82.97853231
82.97853231
82.97853231
82.97853231
83.0118739
83.0118739
83.0118739
83.0118739
83.0118739
82.99839318
82.99839318
82.99839318
82.99839318
82.99839318
83.01005505
83.01005505
83.01005505
83.01005505
83.01005505
83.01479428
83.01479428
83.01479428
83.01479428
83.01479428
83.00340756
83.00340756
83.00340756
83.00340756
83.00340756
82.98284975
82.98284975
82.98284975
82.98284975
82.98284975
82.99647933
82.99647933
82.99647933
82.99647933
82.99647933
83.00508876
83.00508876
83.00508876
83.00508876
83.00508876
82.98871686
82.98871686
82.98871686
82.98871686
82.98871686
82.99047397
82.99047397
82.99047397
82.99047397
82.99047397
83.02679113
83.02679113
83.02679113
83.02679113
83.02679113
83.02546532
83.02546532
83.02546532
83.02546532
83.02546532
83.00745909
83.00745909
83.00745909
83.00745909
83.00745909
82.97232328
82.97232328
82.97232328
82.97232328
82.97232328
82.97527168
82.97527168
82.97527168
82.97527168
82.97527168
83.00730743
83.00730743
83.00730743
83.00730743
83.00730743
82.99620955
82.99620955
82.99620955
82.99620955
82.99620955
83.00820765
83.00820765
83.00820765
83.00820765
83.00820765
83.00178825
83.00178825
83.00178825
83.00178825
83.00178825
82.99159273
82.99159273
82.99159273
82.99159273
82.99159273
83.00224928
83.00224928
83.00224928
83.00224928
83.00224928
82.99793077
82.99793077
82.99793077
82.99793077
82.99793077
83.0022343
83.0022343
83.0022343
83.0022343
83.0022343
82.99274822
82.99274822
82.99274822
82.99274822
82.99274822
83.00436448
83.00436448
83.00436448
83.00436448
83.00436448
82.98923916
82.98923916
82.98923916
82.98923916
82.98923916
82.99693429
82.99693429
82.99693429
82.99693429
82.99693429
82.97935519
82.97935519
82.97935519
82.97935519
82.97935519
83.02961206
83.02961206
83.02961206
83.02961206
83.02961206
83.01976978
83.01976978
83.01976978
83.01976978
83.01976978
83.0219673
83.0219673
83.0219673
83.0219673
83.0219673
83.02437015
83.02437015
83.02437015
83.02437015
83.02437015
83.0133373
83.0133373
83.0133373
83.0133373
83.0133373
83.030013
83.030013
83.030013
83.030013
83.030013
83.00745636
83.00745636
83.00745636
83.00745636
83.00745636
82.9668881
82.9668881
82.9668881
82.9668881
82.9668881
82.99165671
82.99165671
82.99165671
82.99165671
82.99165671
82.99780248
82.99780248
82.99780248
82.99780248
82.99780248
83.00633251
83.00633251
83.00633251
83.00633251
83.00633251
83.02262345
83.02262345
83.02262345
83.02262345
83.02262345
83.02316443
83.02316443
83.02316443
83.02316443
83.02316443
82.97835054
82.97835054
82.97835054
82.97835054
82.97835054
83.02907804
83.02907804
83.02907804
83.02907804
83.02907804
83.00563957
83.00563957
83.00563957
83.00563957
83.00563957
82.97433724
82.97433724
82.97433724
82.97433724
82.97433724
83.02303531
83.02303531
83.02303531
83.02303531
83.02303531
82.98619572
82.98619572
82.98619572
82.98619572
82.98619572
83.01175591
83.01175591
83.01175591
83.01175591
83.01175591
83.02052808
83.02052808
83.02052808
83.02052808
83.02052808
82.97384137
82.97384137
82.97384137
82.97384137
82.97384137
83.02259537
83.02259537
83.02259537
83.02259537
83.02259537
83.00294167
83.00294167
83.00294167
83.00294167
83.00294167
82.96055572
82.96055572
82.96055572
82.96055572
82.96055572
82.98773323
82.98773323
82.98773323
82.98773323
82.98773323
82.97974811
82.97974811
82.97974811
82.97974811
82.97974811
83.02500809
83.02500809
83.02500809
83.02500809
83.02500809
82.99553703
82.99553703
82.99553703
82.99553703
82.99553703
83.01103106
83.01103106
83.01103106
83.01103106
83.01103106
83.01630877
83.01630877
83.01630877
83.01630877
83.01630877
83.02437325
83.02437325
83.02437325
83.02437325
83.02437325
82.97374132
82.97374132
82.97374132
82.97374132
82.97374132
83.00885837
83.00885837
83.00885837
83.00885837
83.00885837
82.99597807
82.99597807
82.99597807
82.99597807
82.99597807
82.98251826
82.98251826
82.98251826
82.98251826
82.98251826
82.94843157
82.94843157
82.94843157
82.94843157
82.94843157
82.96759288
82.96759288
82.96759288
82.96759288
82.96759288
83.00683895
83.00683895
83.00683895
83.00683895
83.00683895
82.97810938
82.97810938
82.97810938
82.97810938
82.97810938
82.9944751
82.9944751
82.9944751
82.9944751
82.9944751
82.98783112
82.98783112
82.98783112
82.98783112
82.98783112
83.02365706
83.02365706
83.02365706
83.02365706
83.02365706
83.01775376
83.01775376
83.01775376
83.01775376
83.01775376
82.97055061
82.97055061
82.97055061
82.97055061
82.97055061
82.99393607
82.99393607
82.99393607
82.99393607
82.99393607
83.02342861
83.02342861
83.02342861
83.02342861
83.02342861
83.00489447
83.00489447
83.00489447
83.00489447
83.00489447
83.00158591
83.00158591
83.00158591
83.00158591
83.00158591
83.02449996
83.02449996
83.02449996
83.02449996
83.02449996
82.97443531
82.97443531
82.97443531
82.97443531
82.97443531
83.00177778
83.00177778
83.00177778
83.00177778
83.00177778
83.02981488
83.02981488
83.02981488
83.02981488
83.02981488
82.98213193
82.98213193
82.98213193
82.98213193
82.98213193
83.01185411
83.01185411
83.01185411
83.01185411
83.01185411
83.01148041
83.01148041
83.01148041
83.01148041
83.01148041
83.00463332
83.00463332
83.00463332
83.00463332
83.00463332
82.98039517
82.98039517
82.98039517
82.98039517
82.98039517
83.00319277
83.00319277
83.00319277
83.00319277
83.00319277
83.00326043
83.00326043
83.00326043
83.00326043
83.00326043
82.99713481
82.99713481
82.99713481
82.99713481
82.99713481
82.97117067
82.97117067
82.97117067
82.97117067
82.97117067
83.01961777
83.01961777
83.01961777
83.01961777
83.01961777
82.98454432
82.98454432
82.98454432
82.98454432
82.98454432
83.0080925
83.0080925
83.0080925
83.0080925
83.0080925
83.03721835
83.03721835
83.03721835
83.03721835
83.03721835
82.98383527
82.98383527
82.98383527
82.98383527
82.98383527
83.00946913
83.00946913
83.00946913
83.00946913
83.00946913
83.01353017
83.01353017
83.01353017
83.01353017
83.01353017
83.01602789
83.01602789
83.01602789
83.01602789
83.01602789
83.01748954
83.01748954
83.01748954
83.01748954
83.01748954
83.00211976
83.00211976
83.00211976
83.00211976
83.00211976
82.98050671
82.98050671
82.98050671
82.98050671
82.98050671
82.99319709
82.99319709
82.99319709
82.99319709
82.99319709
82.99369845
82.99369845
82.99369845
82.99369845
82.99369845
83.015155
83.015155
83.015155
83.015155
83.015155
83.00294599
83.00294599
83.00294599
83.00294599
83.00294599
82.97686033
82.97686033
82.97686033
82.97686033
82.97686033
83.01795095
83.01795095
83.01795095
83.01795095
83.01795095
83.00835223
83.00835223
83.00835223
83.00835223
83.00835223
82.97414235
82.97414235
82.97414235
82.97414235
82.97414235
83.01142496
83.01142496
83.01142496
83.01142496
83.01142496
82.97380922
82.97380922
82.97380922
82.97380922
82.97380922
83.01648003
83.01648003
83.01648003
83.01648003
83.01648003
82.98811886
82.98811886
82.98811886
82.98811886
82.98811886
82.9966818
82.9966818
82.9966818
82.9966818
82.9966818
82.96043991
82.96043991
82.96043991
82.96043991
82.96043991
83.00992453
83.00992453
83.00992453
83.00992453
83.00992453
82.98430146
82.98430146
82.98430146
82.98430146
82.98430146
82.99971368
82.99971368
82.99971368
82.99971368
82.99971368
83.02845934
83.02845934
83.02845934
83.02845934
83.02845934
83.02071941
83.02071941
83.02071941
83.02071941
83.02071941
83.02279458
83.02279458
83.02279458
83.02279458
83.02279458
83.0107517
83.0107517
83.0107517
83.0107517
83.0107517
82.98122209
82.98122209
82.98122209
82.98122209
82.98122209
83.03499424
83.03499424
83.03499424
83.03499424
83.03499424
82.97717557
82.97717557
82.97717557
82.97717557
82.97717557
82.97630982
82.97630982
82.97630982
82.97630982
82.97630982
83.00936617
83.00936617
83.00936617
83.00936617
83.00936617
82.99099104
82.99099104
82.99099104
82.99099104
82.99099104
82.99753629
82.99753629
82.99753629
82.99753629
82.99753629
83.00915303
83.00915303
83.00915303
83.00915303
83.00915303
82.95379836
82.95379836
82.95379836
82.95379836
82.95379836
83.0116243
83.0116243
83.0116243
83.0116243
83.0116243
82.98977523
82.98977523
82.98977523
82.98977523
82.98977523
83.00254484
83.00254484
83.00254484
83.00254484
83.00254484
83.00623174
83.00623174
83.00623174
83.00623174
83.00623174
82.98430645
82.98430645
82.98430645
82.98430645
82.98430645
83.00592149
83.00592149
83.00592149
83.00592149
83.00592149
83.00517387
83.00517387
83.00517387
83.00517387
83.00517387
83.01198945
83.01198945
83.01198945
83.01198945
83.01198945
83.00890894
83.00890894
83.00890894
83.00890894
83.00890894
83.01552501
83.01552501
83.01552501
83.01552501
83.01552501
82.98950133
82.98950133
82.98950133
82.98950133
82.98950133
82.96742723
82.96742723
82.96742723
82.96742723
82.96742723
82.9972995
82.9972995
82.9972995
82.9972995
82.9972995
83.00442186
83.00442186
83.00442186
83.00442186
83.00442186
82.99968512
82.99968512
82.99968512
82.99968512
82.99968512
83.0236157
83.0236157
83.0236157
83.0236157
83.0236157
83.00973409
83.00973409
83.00973409
83.00973409
83.00973409
83.03541047
83.03541047
83.03541047
83.03541047
83.03541047
82.99788767
82.99788767
82.99788767
82.99788767
82.99788767
82.96584788
82.96584788
82.96584788
82.96584788
82.96584788
83.01226819
83.01226819
83.01226819
83.01226819
83.01226819
83.00162009
83.00162009
83.00162009
83.00162009
83.00162009
83.0125179
83.0125179
83.0125179
83.0125179
83.0125179
82.98447493
82.98447493
82.98447493
82.98447493
82.98447493
82.99946134
82.99946134
82.99946134
82.99946134
82.99946134
83.00723628
83.00723628
83.00723628
83.00723628
83.00723628
83.01426289
83.01426289
83.01426289
83.01426289
83.01426289
82.96657084
82.96657084
82.96657084
82.96657084
82.96657084
83.01251803
83.01251803
83.01251803
83.01251803
83.01251803
83.00622687
83.00622687
83.00622687
83.00622687
83.00622687
83.02136296
83.02136296
83.02136296
83.02136296
83.02136296
82.9776638
82.9776638
82.9776638
82.9776638
82.9776638
83.0042767
83.0042767
83.0042767
83.0042767
83.0042767
82.97988916
82.97988916
82.97988916
82.97988916
82.97988916
83.0033188
83.0033188
83.0033188
83.0033188
83.0033188
82.97593901
82.97593901
82.97593901
82.97593901
82.97593901
83.00204813
83.00204813
83.00204813
83.00204813
83.00204813
83.00900146
83.00900146
83.00900146
83.00900146
83.00900146
82.98274481
82.98274481
82.98274481
82.98274481
82.98274481
83.01552855
83.01552855
83.01552855
83.01552855
83.01552855
82.99985832
82.99985832
82.99985832
82.99985832
82.99985832
83.00216302
83.00216302
83.00216302
83.00216302
83.00216302
82.96802474
82.96802474
82.96802474
82.96802474
82.96802474
83.0070672
83.0070672
83.0070672
83.0070672
83.0070672
82.99978286
82.99978286
82.99978286
82.99978286
82.99978286
82.97841038
82.97841038
82.97841038
82.97841038
82.97841038
82.99863291
82.99863291
82.99863291
82.99863291
82.99863291
82.97007988
82.97007988
82.97007988
82.97007988
82.97007988
82.97527274
82.97527274
82.97527274
82.97527274
82.97527274
82.97735102
82.97735102
82.97735102
82.97735102
82.97735102
82.98962351
82.98962351
82.98962351
82.98962351
82.98962351
83.02378457
83.02378457
83.02378457
83.02378457
83.02378457
82.99953658
82.99953658
82.99953658
82.99953658
82.99953658
83.00719447
83.00719447
83.00719447
83.00719447
83.00719447
83.02939749
83.02939749
83.02939749
83.02939749
83.02939749
83.00319834
83.00319834
83.00319834
83.00319834
83.00319834
83.00825922
83.00825922
83.00825922
83.00825922
83.00825922
83.00675475
83.00675475
83.00675475
83.00675475
83.00675475
83.00976293
83.00976293
83.00976293
83.00976293
83.00976293
82.98813295
82.98813295
82.98813295
82.98813295
82.98813295
82.99852392
82.99852392
82.99852392
82.99852392
82.99852392
83.00866657
83.00866657
83.00866657
83.00866657
83.00866657
82.97747047
82.97747047
82.97747047
82.97747047
82.97747047
83.02184094
83.02184094
83.02184094
83.02184094
83.02184094
83.00415528
83.00415528
83.00415528
83.00415528
83.00415528
83.02909299
83.02909299
83.02909299
83.02909299
83.02909299
83.00114582
83.00114582
83.00114582
83.00114582
83.00114582
83.00383053
83.00383053
83.00383053
83.00383053
83.00383053
82.9728608
82.9728608
82.9728608
82.9728608
82.9728608
82.9959639
82.9959639
82.9959639
82.9959639
82.9959639
83.00176171
83.00176171
83.00176171
83.00176171
83.00176171
83.01115494
83.01115494
83.01115494
83.01115494
83.01115494
83.0239934
83.0239934
83.0239934
83.0239934
83.0239934
82.96952685
82.96952685
82.96952685
82.96952685
82.96952685
82.99914045
82.99914045
82.99914045
82.99914045
82.99914045
83.00180245
83.00180245
83.00180245
83.00180245
83.00180245
83.01239484
83.01239484
83.01239484
83.01239484
83.01239484
82.98550997
82.98550997
82.98550997
82.98550997
82.98550997
82.98178783
82.98178783
82.98178783
82.98178783
82.98178783
82.99211113
82.99211113
82.99211113
82.99211113
82.99211113
83.00239822
83.00239822
83.00239822
83.00239822
83.00239822
83.02731901
83.02731901
83.02731901
83.02731901
83.02731901
82.98891928
82.98891928
82.98891928
82.98891928
82.98891928
83.01591387
83.01591387
83.01591387
83.01591387
83.01591387
83.00100521
83.00100521
83.00100521
83.00100521
83.00100521
83.01215218
83.01215218
83.01215218
83.01215218
83.01215218
83.01764701
83.01764701
83.01764701
83.01764701
83.01764701
83.02150623
83.02150623
83.02150623
83.02150623
83.02150623
83.01296621
83.01296621
83.01296621
83.01296621
83.01296621
83.00396334
83.00396334
83.00396334
83.00396334
83.00396334
82.99389206
82.99389206
82.99389206
82.99389206
82.99389206
82.99947727
82.99947727
82.99947727
82.99947727
82.99947727
83.0019335
83.0019335
83.0019335
83.0019335
83.0019335
83.01755485
83.01755485
83.01755485
83.01755485
83.01755485
83.00972409
83.00972409
83.00972409
83.00972409
83.00972409
82.99096276
82.99096276
82.99096276
82.99096276
82.99096276
82.98325677
82.98325677
82.98325677
82.98325677
82.98325677
83.00562668
83.00562668
83.00562668
83.00562668
83.00562668
82.97636356
82.97636356
82.97636356
82.97636356
82.97636356
83.00036682
83.00036682
83.00036682
83.00036682
83.00036682
83.00062471
83.00062471
83.00062471
83.00062471
83.00062471
82.97794098
82.97794098
82.97794098
82.97794098
82.97794098
83.02301465
83.02301465
83.02301465
83.02301465
83.02301465
83.00251103
83.00251103
83.00251103
83.00251103
83.00251103
82.99775293
82.99775293
82.99775293
82.99775293
82.99775293
82.9940771
82.9940771
82.9940771
82.9940771
82.9940771
82.99619329
82.99619329
82.99619329
82.99619329
82.99619329
83.00783372
83.00783372
83.00783372
83.00783372
83.00783372
83.01027258
83.01027258
83.01027258
83.01027258
83.01027258
83.02252896
83.02252896
83.02252896
83.02252896
83.02252896
82.98903367
82.98903367
82.98903367
82.98903367
82.98903367
82.98880547
82.98880547
82.98880547
82.98880547
82.98880547
83.01811012
83.01811012
83.01811012
83.01811012
83.01811012
83.01790741
83.01790741
83.01790741
83.01790741
83.01790741
83.00579591
83.00579591
83.00579591
83.00579591
83.00579591
83.02240109
83.02240109
83.02240109
83.02240109
83.02240109
83.00693707
83.00693707
83.00693707
83.00693707
83.00693707
82.99736847
82.99736847
82.99736847
82.99736847
82.99736847
83.02424832
83.02424832
83.02424832
83.02424832
83.02424832
82.98281699
82.98281699
82.98281699
82.98281699
82.98281699
83.0140333
83.0140333
83.0140333
83.0140333
83.0140333
82.98227247
82.98227247
82.98227247
82.98227247
82.98227247
82.99173932
82.99173932
82.99173932
82.99173932
82.99173932
83.01671499
83.01671499
83.01671499
83.01671499
83.01671499
82.97647511
82.97647511
82.97647511
82.97647511
82.97647511
82.99839271
82.99839271
82.99839271
82.99839271
82.99839271
82.98546655
82.98546655
82.98546655
82.98546655
82.98546655
82.9797168
82.9797168
82.9797168
82.9797168
82.9797168
83.0125391
83.0125391
83.0125391
83.0125391
83.0125391
82.97822586
82.97822586
82.97822586
82.97822586
82.97822586
82.96590816
82.96590816
82.96590816
82.96590816
82.96590816
82.98697062
82.98697062
82.98697062
82.98697062
82.98697062
82.97667129
82.97667129
82.97667129
82.97667129
82.97667129
83.0084962
83.0084962
83.0084962
83.0084962
83.0084962
83.00290447
83.00290447
83.00290447
83.00290447
83.00290447
82.99912142
82.99912142
82.99912142
82.99912142
82.99912142
83.01819101
83.01819101
83.01819101
83.01819101
83.01819101
82.99078755
82.99078755
82.99078755
82.99078755
82.99078755
83.03219697
83.03219697
83.03219697
83.03219697
83.03219697
82.99858278
82.99858278
82.99858278
82.99858278
82.99858278
83.01517758
83.01517758
83.01517758
83.01517758
83.01517758
83.00606072
83.00606072
83.00606072
83.00606072
83.00606072
83.00449553
83.00449553
83.00449553
83.00449553
83.00449553
82.99441569
82.99441569
82.99441569
82.99441569
82.99441569
83.01982177
83.01982177
83.01982177
83.01982177
83.01982177
83.01120454
83.01120454
83.01120454
83.01120454
83.01120454
83.00209923
83.00209923
83.00209923
83.00209923
83.00209923
83.00405049
83.00405049
83.00405049
83.00405049
83.00405049
83.0257203
83.0257203
83.0257203
83.0257203
83.0257203
82.99875736
82.99875736
82.99875736
82.99875736
82.99875736
83.00731889
83.00731889
83.00731889
83.00731889
83.00731889
82.99429571
82.99429571
82.99429571
82.99429571
82.99429571
82.98834981
82.98834981
82.98834981
82.98834981
82.98834981
83.01049284
83.01049284
83.01049284
83.01049284
83.01049284
83.0134959
83.0134959
83.0134959
83.0134959
83.0134959
82.97483761
82.97483761
82.97483761
82.97483761
82.97483761
83.00670592
83.00670592
83.00670592
83.00670592
83.00670592
83.02136233
83.02136233
83.02136233
83.02136233
83.02136233
82.99472296
82.99472296
82.99472296
82.99472296
82.99472296
82.99489353
82.99489353
82.99489353
82.99489353
82.99489353
82.97725879
82.97725879
82.97725879
82.97725879
82.97725879
82.98539957
82.98539957
82.98539957
82.98539957
82.98539957
83.02737601
83.02737601
83.02737601
83.02737601
83.02737601
83.00588165
83.00588165
83.00588165
83.00588165
83.00588165
83.0070141
83.0070141
83.0070141
83.0070141
83.0070141
83.00566747
83.00566747
83.00566747
83.00566747
83.00566747
83.01161903
83.01161903
83.01161903
83.01161903
83.01161903
82.98651129
82.98651129
82.98651129
82.98651129
82.98651129
83.00306429
83.00306429
83.00306429
83.00306429
83.00306429
82.99157076
82.99157076
82.99157076
82.99157076
82.99157076
83.02534122
83.02534122
83.02534122
83.02534122
83.02534122
83.01178907
83.01178907
83.01178907
83.01178907
83.01178907
82.98593306
82.98593306
82.98593306
82.98593306
82.98593306
82.99964262
82.99964262
82.99964262
82.99964262
82.99964262
82.97828696
82.97828696
82.97828696
82.97828696
82.97828696
83.00909261
83.00909261
83.00909261
83.00909261
83.00909261
83.01178442
83.01178442
83.01178442
83.01178442
83.01178442
83.02091851
83.02091851
83.02091851
83.02091851
83.02091851
82.99803715
82.99803715
82.99803715
82.99803715
82.99803715
83.01347994
83.01347994
83.01347994
83.01347994
83.01347994
83.01738603
83.01738603
83.01738603
83.01738603
83.01738603
83.01528553
83.01528553
83.01528553
83.01528553
83.01528553
83.00102637
83.00102637
83.00102637
83.00102637
83.00102637
83.00161875
83.00161875
83.00161875
83.00161875
83.00161875
83.01900513
83.01900513
83.01900513
83.01900513
83.01900513
82.98817716
82.98817716
82.98817716
82.98817716
82.98817716
83.0266779
83.0266779
83.0266779
83.0266779
83.0266779
83.01403888
83.01403888
83.01403888
83.01403888
83.01403888
83.0091888
83.0091888
83.0091888
83.0091888
83.0091888
82.9903092
82.9903092
82.9903092
82.9903092
82.9903092
82.98203647
82.98203647
82.98203647
82.98203647
82.98203647
83.02068026
83.02068026
83.02068026
83.02068026
83.02068026
83.00145986
83.00145986
83.00145986
83.00145986
83.00145986
83.00806918
83.00806918
83.00806918
83.00806918
83.00806918
82.99701024
82.99701024
82.99701024
82.99701024
82.99701024
82.98783927
82.98783927
82.98783927
82.98783927
82.98783927
82.97905917
82.97905917
82.97905917
82.97905917
82.97905917
83.01063786
83.01063786
83.01063786
83.01063786
83.01063786
82.95900545
82.95900545
82.95900545
82.95900545
82.95900545
82.9784296
82.9784296
82.9784296
82.9784296
82.9784296
82.9687291
82.9687291
82.9687291
82.9687291
82.9687291
82.97882995
82.97882995
82.97882995
82.97882995
82.97882995
83.00922433
83.00922433
83.00922433
83.00922433
83.00922433
83.00181945
83.00181945
83.00181945
83.00181945
83.00181945
82.99680235
82.99680235
82.99680235
82.99680235
82.99680235
83.01843637
83.01843637
83.01843637
83.01843637
83.01843637
82.97124011
82.97124011
82.97124011
82.97124011
82.97124011
83.00295716
83.00295716
83.00295716
83.00295716
83.00295716
82.97341879
82.97341879
82.97341879
82.97341879
82.97341879
83.02373524
83.02373524
83.02373524
83.02373524
83.02373524
83.00353479
83.00353479
83.00353479
83.00353479
83.00353479
82.99396111
82.99396111
82.99396111
82.99396111
82.99396111
82.98808142
82.98808142
82.98808142
82.98808142
82.98808142
83.01663281
83.01663281
83.01663281
83.01663281
83.01663281
82.99714032
82.99714032
82.99714032
82.99714032
82.99714032
83.00946576
83.00946576
83.00946576
83.00946576
83.00946576
83.00148723
83.00148723
83.00148723
83.00148723
83.00148723
82.980483
82.980483
82.980483
82.980483
82.980483
83.00859357
83.00859357
83.00859357
83.00859357
83.00859357
83.0020359
83.0020359
83.0020359
83.0020359
83.0020359
83.01388679
83.01388679
83.01388679
83.01388679
83.01388679
82.97925307
82.97925307
82.97925307
82.97925307
82.97925307
82.97883679
82.97883679
82.97883679
82.97883679
82.97883679
83.00813443
83.00813443
83.00813443
83.00813443
83.00813443
82.9977472
82.9977472
82.9977472
82.9977472
82.9977472
83.01505038
83.01505038
83.01505038
83.01505038
83.01505038
83.01535466
83.01535466
83.01535466
83.01535466
83.01535466
83.02551718
83.02551718
83.02551718
83.02551718
83.02551718
82.97643401
82.97643401
82.97643401
82.97643401
82.97643401
82.99314933
82.99314933
82.99314933
82.99314933
82.99314933
82.99769486
82.99769486
82.99769486
82.99769486
82.99769486
82.99791774
82.99791774
82.99791774
82.99791774
82.99791774
83.03288505
83.03288505
83.03288505
83.03288505
83.03288505
83.02740157
83.02740157
83.02740157
83.02740157
83.02740157
83.00810557
83.00810557
83.00810557
83.00810557
83.00810557
82.98312174
82.98312174
82.98312174
82.98312174
82.98312174
83.00441119
83.00441119
83.00441119
83.00441119
83.00441119
82.98554425
82.98554425
82.98554425
82.98554425
82.98554425
83.03154595
83.03154595
83.03154595
83.03154595
83.03154595
82.98537557
82.98537557
82.98537557
82.98537557
82.98537557
83.02314952
83.02314952
83.02314952
83.02314952
83.02314952
83.00482548
83.00482548
83.00482548
83.00482548
83.00482548
83.00765985
83.00765985
83.00765985
83.00765985
83.00765985
83.02941093
83.02941093
83.02941093
83.02941093
83.02941093
83.00677633
83.00677633
83.00677633
83.00677633
83.00677633
82.95230957
82.95230957
82.95230957
82.95230957
82.95230957
83.03063776
83.03063776
83.03063776
83.03063776
83.03063776
83.00320503
83.00320503
83.00320503
83.00320503
83.00320503
82.99209119
82.99209119
82.99209119
82.99209119
82.99209119
83.02201432
83.02201432
83.02201432
83.02201432
83.02201432
83.01834834
83.01834834
83.01834834
83.01834834
83.01834834
83.01520774
83.01520774
83.01520774
83.01520774
83.01520774
83.00204285
83.00204285
83.00204285
83.00204285
83.00204285
83.0046114
83.0046114
83.0046114
83.0046114
83.0046114
83.0279028
83.0279028
83.0279028
83.0279028
83.0279028
82.98030318
82.98030318
82.98030318
82.98030318
82.98030318
83.00155377
83.00155377
83.00155377
83.00155377
83.00155377
83.01242625
83.01242625
83.01242625
83.01242625
83.01242625
83.01241623
83.01241623
83.01241623
83.01241623
83.01241623
82.98802694
82.98802694
82.98802694
82.98802694
82.98802694
83.00178602
83.00178602
83.00178602
83.00178602
83.00178602
82.97088353
82.97088353
82.97088353
82.97088353
82.97088353
82.99736351
82.99736351
82.99736351
82.99736351
82.99736351
83.00479736
83.00479736
83.00479736
83.00479736
83.00479736
82.97578837
82.97578837
82.97578837
82.97578837
82.97578837
83.01349625
83.01349625
83.01349625
83.01349625
83.01349625
82.99231769
82.99231769
82.99231769
82.99231769
82.99231769
82.99882374
82.99882374
82.99882374
82.99882374
82.99882374
83.00273455
83.00273455
83.00273455
83.00273455
83.00273455
83.01288419
83.01288419
83.01288419
83.01288419
83.01288419
82.99831336
82.99831336
82.99831336
82.99831336
82.99831336
83.00002971
83.00002971
83.00002971
83.00002971
83.00002971
82.99172293
82.99172293
82.99172293
82.99172293
82.99172293
83.01162786
83.01162786
83.01162786
83.01162786
83.01162786
83.01648496
83.01648496
83.01648496
83.01648496
83.01648496
82.99407437
82.99407437
82.99407437
82.99407437
82.99407437
83.02210792
83.02210792
83.02210792
83.02210792
83.02210792
82.98426485
82.98426485
82.98426485
82.98426485
82.98426485
82.99390839
82.99390839
82.99390839
82.99390839
82.99390839
82.99606839
82.99606839
82.99606839
82.99606839
82.99606839
83.01615063
83.01615063
83.01615063
83.01615063
83.01615063
82.97605186
82.97605186
82.97605186
82.97605186
82.97605186
82.97938617
82.97938617
82.97938617
82.97938617
82.97938617
82.97893608
82.97893608
82.97893608
82.97893608
82.97893608
83.02076746
83.02076746
83.02076746
83.02076746
83.02076746
83.00390078
83.00390078
83.00390078
83.00390078
83.00390078
82.99704361
82.99704361
82.99704361
82.99704361
82.99704361
83.01334923
83.01334923
83.01334923
83.01334923
83.01334923
83.0237944
83.0237944
83.0237944
83.0237944
83.0237944
83.00748522
83.00748522
83.00748522
83.00748522
83.00748522
83.00320968
83.00320968
83.00320968
83.00320968
83.00320968
83.00208731
83.00208731
83.00208731
83.00208731
83.00208731
83.01884689
83.01884689
83.01884689
83.01884689
83.01884689
82.97927977
82.97927977
82.97927977
82.97927977
82.97927977
82.99619524
82.99619524
82.99619524
82.99619524
82.99619524
83.00196051
83.00196051
83.00196051
83.00196051
83.00196051
82.96958529
82.96958529
82.96958529
82.96958529
82.96958529
83.01271752
83.01271752
83.01271752
83.01271752
83.01271752
82.97600411
82.97600411
82.97600411
82.97600411
82.97600411
82.99231272
82.99231272
82.99231272
82.99231272
82.99231272
82.97733888
82.97733888
82.97733888
82.97733888
82.97733888
83.02630026
83.02630026
83.02630026
83.02630026
83.02630026
82.97907163
82.97907163
82.97907163
82.97907163
82.97907163
82.97762884
82.97762884
82.97762884
82.97762884
82.97762884
83.02201899
83.02201899
83.02201899
83.02201899
83.02201899
83.00253151
83.00253151
83.00253151
83.00253151
83.00253151
82.99786105
82.99786105
82.99786105
82.99786105
82.99786105
83.00996488
83.00996488
83.00996488
83.00996488
83.00996488
82.98778961
82.98778961
82.98778961
82.98778961
82.98778961
83.00450366
83.00450366
83.00450366
83.00450366
83.00450366
83.01002012
83.01002012
83.01002012
83.01002012
83.01002012
83.01614374
83.01614374
83.01614374
83.01614374
83.01614374
82.96892404
82.96892404
82.96892404
82.96892404
82.96892404
82.99637217
82.99637217
82.99637217
82.99637217
82.99637217
83.00557245
83.00557245
83.00557245
83.00557245
83.00557245
83.01313807
83.01313807
83.01313807
83.01313807
83.01313807
82.99338172
82.99338172
82.99338172
82.99338172
82.99338172
83.01849451
83.01849451
83.01849451
83.01849451
83.01849451
83.01288251
83.01288251
83.01288251
83.01288251
83.01288251
82.97866526
82.97866526
82.97866526
82.97866526
82.97866526
83.00957649
83.00957649
83.00957649
83.00957649
83.00957649
82.99422391
82.99422391
82.99422391
82.99422391
82.99422391
83.00315021
83.00315021
83.00315021
83.00315021
83.00315021
83.01540648
83.01540648
83.01540648
83.01540648
83.01540648
82.98218572
82.98218572
82.98218572
82.98218572
82.98218572
83.01295411
83.01295411
83.01295411
83.01295411
83.01295411
82.99712294
82.99712294
82.99712294
82.99712294
82.99712294
83.00869612
83.00869612
83.00869612
83.00869612
83.00869612
83.00249247
83.00249247
83.00249247
83.00249247
83.00249247
83.02337129
83.02337129
83.02337129
83.02337129
83.02337129
82.98126652
82.98126652
82.98126652
82.98126652
82.98126652
82.97742342
82.97742342
82.97742342
82.97742342
82.97742342
83.00867065
83.00867065
83.00867065
83.00867065
83.00867065
83.02598212
83.02598212
83.02598212
83.02598212
83.02598212
82.96544939
82.96544939
82.96544939
82.96544939
82.96544939
83.01924769
83.01924769
83.01924769
83.01924769
83.01924769
83.01015188
83.01015188
83.01015188
83.01015188
83.01015188
82.99714461
82.99714461
82.99714461
82.99714461
82.99714461
83.01441122
83.01441122
83.01441122
83.01441122
83.01441122
83.00506445
83.00506445
83.00506445
83.00506445
83.00506445
83.00782039
83.00782039
83.00782039
83.00782039
83.00782039
83.02417992
83.02417992
83.02417992
83.02417992
83.02417992
82.9819007
82.9819007
82.9819007
82.9819007
82.9819007
82.9786855
82.9786855
82.9786855
82.9786855
82.9786855
83.02892302
83.02892302
83.02892302
83.02892302
83.02892302
82.99185318
82.99185318
82.99185318
82.99185318
82.99185318
83.00417409
83.00417409
83.00417409
83.00417409
83.00417409
83.00224011
83.00224011
83.00224011
83.00224011
83.00224011
82.98724706
82.98724706
82.98724706
82.98724706
82.98724706
82.96989463
82.96989463
82.96989463
82.96989463
82.96989463
83.01287809
83.01287809
83.01287809
83.01287809
83.01287809
83.00157501
83.00157501
83.00157501
83.00157501
83.00157501
82.97074405
82.97074405
82.97074405
82.97074405
82.97074405
83.02605016
83.02605016
83.02605016
83.02605016
83.02605016
83.00263084
83.00263084
83.00263084
83.00263084
83.00263084
82.97967257
82.97967257
82.97967257
82.97967257
82.97967257
82.98950024
82.98950024
82.98950024
82.98950024
82.98950024
82.99647456
82.99647456
82.99647456
82.99647456
82.99647456
82.95581542
82.95581542
82.95581542
82.95581542
82.95581542
82.98970465
82.98970465
82.98970465
82.98970465
82.98970465
83.00864597
83.00864597
83.00864597
83.00864597
83.00864597
82.97126021
82.97126021
82.97126021
82.97126021
82.97126021
83.00959313
83.00959313
83.00959313
83.00959313
83.00959313
83.00725412
83.00725412
83.00725412
83.00725412
83.00725412
82.97377771
82.97377771
82.97377771
82.97377771
82.97377771
83.00704172
83.00704172
83.00704172
83.00704172
83.00704172
83.00792808
83.00792808
83.00792808
83.00792808
83.00792808
82.98433066
82.98433066
82.98433066
82.98433066
82.98433066
82.96936685
82.96936685
82.96936685
82.96936685
82.96936685
83.00317225
83.00317225
83.00317225
83.00317225
83.00317225
83.00888293
83.00888293
83.00888293
83.00888293
83.00888293
82.99784932
82.99784932
82.99784932
82.99784932
82.99784932
83.00308321
83.00308321
83.00308321
83.00308321
83.00308321
83.00484818
83.00484818
83.00484818
83.00484818
83.00484818
83.02517467
83.02517467
83.02517467
83.02517467
83.02517467
82.99330246
82.99330246
82.99330246
82.99330246
82.99330246
82.99070717
82.99070717
82.99070717
82.99070717
82.99070717
82.99811354
82.99811354
82.99811354
82.99811354
82.99811354
83.00651399
83.00651399
83.00651399
83.00651399
83.00651399
83.00166108
83.00166108
83.00166108
83.00166108
83.00166108
83.00937279
83.00937279
83.00937279
83.00937279
83.00937279
83.00844949
83.00844949
83.00844949
83.00844949
83.00844949
82.98119947
82.98119947
82.98119947
82.98119947
82.98119947
83.00300582
83.00300582
83.00300582
83.00300582
83.00300582
83.01502763
83.01502763
83.01502763
83.01502763
83.01502763
83.01609446
83.01609446
83.01609446
83.01609446
83.01609446
82.99188846
82.99188846
82.99188846
82.99188846
82.99188846
82.98317634
82.98317634
82.98317634
82.98317634
82.98317634
83.00051519
83.00051519
83.00051519
83.00051519
83.00051519
83.00738902
83.00738902
83.00738902
83.00738902
83.00738902
82.98413357
82.98413357
82.98413357
82.98413357
82.98413357
82.98350208
82.98350208
82.98350208
82.98350208
82.98350208
82.98570784
82.98570784
82.98570784
82.98570784
82.98570784
83.01720038
83.01720038
83.01720038
83.01720038
83.01720038
82.99435523
82.99435523
82.99435523
82.99435523
82.99435523
82.99074201
82.99074201
82.99074201
82.99074201
82.99074201
83.00155289
83.00155289
83.00155289
83.00155289
83.00155289
82.97080484
82.97080484
82.97080484
82.97080484
82.97080484
83.00275628
83.00275628
83.00275628
83.00275628
83.00275628
83.01648327
83.01648327
83.01648327
83.01648327
83.01648327
82.97976062
82.97976062
82.97976062
82.97976062
82.97976062
82.99621135
82.99621135
82.99621135
82.99621135
82.99621135
83.02615049
83.02615049
83.02615049
83.02615049
83.02615049
83.01970078
83.01970078
83.01970078
83.01970078
83.01970078
83.01981944
83.01981944
83.01981944
83.01981944
83.01981944
82.94998455
82.94998455
82.94998455
82.94998455
82.94998455
83.025359
83.025359
83.025359
83.025359
83.025359
83.01440756
83.01440756
83.01440756
83.01440756
83.01440756
83.01080978
83.01080978
83.01080978
83.01080978
83.01080978
83.01532692
83.01532692
83.01532692
83.01532692
83.01532692
83.01803889
83.01803889
83.01803889
83.01803889
83.01803889
83.00767711
83.00767711
83.00767711
83.00767711
83.00767711
83.00379765
83.00379765
83.00379765
83.00379765
83.00379765
83.00200948
83.00200948
83.00200948
83.00200948
83.00200948
82.96213873
82.96213873
82.96213873
82.96213873
82.96213873
83.00144786
83.00144786
83.00144786
83.00144786
83.00144786
82.99638221
82.99638221
82.99638221
82.99638221
82.99638221
82.97668355
82.97668355
82.97668355
82.97668355
82.97668355
83.00961313
83.00961313
83.00961313
83.00961313
83.00961313
82.98813973
82.98813973
82.98813973
82.98813973
82.98813973
82.9947376
82.9947376
82.9947376
82.9947376
82.9947376
83.00951513
83.00951513
83.00951513
83.00951513
83.00951513
82.96564118
82.96564118
82.96564118
82.96564118
82.96564118
82.98827442
82.98827442
82.98827442
82.98827442
82.98827442
82.9983629
82.9983629
82.9983629
82.9983629
82.9983629
83.00592548
83.00592548
83.00592548
83.00592548
83.00592548
82.96992303
82.96992303
82.96992303
82.96992303
82.96992303
83.00605313
83.00605313
83.00605313
83.00605313
83.00605313
82.96445256
82.96445256
82.96445256
82.96445256
82.96445256
82.98999102
82.98999102
82.98999102
82.98999102
82.98999102
82.98042241
82.98042241
82.98042241
82.98042241
82.98042241
82.98013113
82.98013113
82.98013113
82.98013113
82.98013113
83.01644503
83.01644503
83.01644503
83.01644503
83.01644503
83.00708735
83.00708735
83.00708735
83.00708735
83.00708735
83.00871181
83.00871181
83.00871181
83.00871181
83.00871181
82.98230347
82.98230347
82.98230347
82.98230347
82.98230347
83.00568345
83.00568345
83.00568345
83.00568345
83.00568345
83.00865807
83.00865807
83.00865807
83.00865807
83.00865807
83.00802388
83.00802388
83.00802388
83.00802388
83.00802388
83.00171669
83.00171669
83.00171669
83.00171669
83.00171669
82.95154251
82.95154251
82.95154251
82.95154251
82.95154251
82.98204965
82.98204965
82.98204965
82.98204965
82.98204965
82.95743607
82.95743607
82.95743607
82.95743607
82.95743607
83.00725572
83.00725572
83.00725572
83.00725572
83.00725572
83.00639028
83.00639028
83.00639028
83.00639028
83.00639028
82.97340558
82.97340558
82.97340558
82.97340558
82.97340558
83.00535825
83.00535825
83.00535825
83.00535825
83.00535825
83.00748486
83.00748486
83.00748486
83.00748486
83.00748486
83.00934839
83.00934839
83.00934839
83.00934839
83.00934839
82.99021824
82.99021824
82.99021824
82.99021824
82.99021824
82.99304196
82.99304196
82.99304196
82.99304196
82.99304196
83.02122345
83.02122345
83.02122345
83.02122345
83.02122345
82.99419469
82.99419469
82.99419469
82.99419469
82.99419469
82.99448901
82.99448901
82.99448901
82.99448901
82.99448901
83.02505937
83.02505937
83.02505937
83.02505937
83.02505937
83.03670478
83.03670478
83.03670478
83.03670478
83.03670478
83.03354863
83.03354863
83.03354863
83.03354863
83.03354863
83.00508202
83.00508202
83.00508202
83.00508202
83.00508202
82.97943651
82.97943651
82.97943651
82.97943651
82.97943651
83.02163367
83.02163367
83.02163367
83.02163367
83.02163367
82.9583396
82.9583396
82.9583396
82.9583396
82.9583396
82.97582983
82.97582983
82.97582983
82.97582983
82.97582983
83.01347503
83.01347503
83.01347503
83.01347503
83.01347503
82.96900355
82.96900355
82.96900355
82.96900355
82.96900355
82.98641278
82.98641278
82.98641278
82.98641278
82.98641278
82.97024714
82.97024714
82.97024714
82.97024714
82.97024714
83.01134953
83.01134953
83.01134953
83.01134953
83.01134953
82.99093511
82.99093511
82.99093511
82.99093511
82.99093511
82.98579248
82.98579248
82.98579248
82.98579248
82.98579248
83.00899056
83.00899056
83.00899056
83.00899056
83.00899056
83.00386243
83.00386243
83.00386243
83.00386243
83.00386243
83.01466212
83.01466212
83.01466212
83.01466212
83.01466212
83.02900015
83.02900015
83.02900015
83.02900015
83.02900015
83.01997583
83.01997583
83.01997583
83.01997583
83.01997583
82.96270191
82.96270191
82.96270191
82.96270191
82.96270191
83.01406443
83.01406443
83.01406443
83.01406443
83.01406443
83.0036414
83.0036414
83.0036414
83.0036414
83.0036414
82.96648829
82.96648829
82.96648829
82.96648829
82.96648829
83.00168714
83.00168714
83.00168714
83.00168714
83.00168714
83.00048381
83.00048381
83.00048381
83.00048381
83.00048381
83.00737424
83.00737424
83.00737424
83.00737424
83.00737424
82.99485712
82.99485712
82.99485712
82.99485712
82.99485712
83.00498303
83.00498303
83.00498303
83.00498303
83.00498303
83.00496706
83.00496706
83.00496706
83.00496706
83.00496706
82.98952098
82.98952098
82.98952098
82.98952098
82.98952098
83.00967201
83.00967201
83.00967201
83.00967201
83.00967201
83.01208185
83.01208185
83.01208185
83.01208185
83.01208185
83.00739388
83.00739388
83.00739388
83.00739388
83.00739388
83.02458552
83.02458552
83.02458552
83.02458552
83.02458552
83.01159628
83.01159628
83.01159628
83.01159628
83.01159628
83.00603365
83.00603365
83.00603365
83.00603365
83.00603365
82.98712132
82.98712132
82.98712132
82.98712132
82.98712132
83.0188305
83.0188305
83.0188305
83.0188305
83.0188305
83.01508206
83.01508206
83.01508206
83.01508206
83.01508206
83.00029883
83.00029883
83.00029883
83.00029883
83.00029883
83.0110334
83.0110334
83.0110334
83.0110334
83.0110334
82.98837897
82.98837897
82.98837897
82.98837897
82.98837897
82.99668834
82.99668834
82.99668834
82.99668834
82.99668834
83.00182476
83.00182476
83.00182476
83.00182476
83.00182476
83.00636865
83.00636865
83.00636865
83.00636865
83.00636865
83.01236807
83.01236807
83.01236807
83.01236807
83.01236807
82.994862
82.994862
82.994862
82.994862
82.994862
83.02510989
83.02510989
83.02510989
83.02510989
83.02510989
83.01995534
83.01995534
83.01995534
83.01995534
83.01995534
83.01134319
83.01134319
83.01134319
83.01134319
83.01134319
83.00155691
83.00155691
83.00155691
83.00155691
83.00155691
83.00613989
83.00613989
83.00613989
83.00613989
83.00613989
82.98766649
82.98766649
82.98766649
82.98766649
82.98766649
82.98169466
82.98169466
82.98169466
82.98169466
82.98169466
83.00913848
83.00913848
83.00913848
83.00913848
83.00913848
83.01468757
83.01468757
83.01468757
83.01468757
83.01468757
83.00398127
83.00398127
83.00398127
83.00398127
83.00398127
83.00173849
83.00173849
83.00173849
83.00173849
83.00173849
83.01067256
83.01067256
83.01067256
83.01067256
83.01067256
82.98836626
82.98836626
82.98836626
82.98836626
82.98836626
83.02493655
83.02493655
83.02493655
83.02493655
83.02493655
82.99729128
82.99729128
82.99729128
82.99729128
82.99729128
82.9995915
82.9995915
82.9995915
82.9995915
82.9995915
83.00203379
83.00203379
83.00203379
83.00203379
83.00203379
83.0256462
83.0256462
83.0256462
83.0256462
83.0256462
83.01019971
83.01019971
83.01019971
83.01019971
83.01019971
83.01461123
83.01461123
83.01461123
83.01461123
83.01461123
83.01370455
83.01370455
83.01370455
83.01370455
83.01370455
82.97834167
82.97834167
82.97834167
82.97834167
82.97834167
83.00796099
83.00796099
83.00796099
83.00796099
83.00796099
83.01101903
83.01101903
83.01101903
83.01101903
83.01101903
83.01094029
83.01094029
83.01094029
83.01094029
83.01094029
83.0200428
83.0200428
83.0200428
83.0200428
83.0200428
82.99008609
82.99008609
82.99008609
82.99008609
82.99008609
83.02445123
83.02445123
83.02445123
83.02445123
83.02445123
83.00492952
83.00492952
83.00492952
83.00492952
83.00492952
83.0095584
83.0095584
83.0095584
83.0095584
83.0095584
83.02165563
83.02165563
83.02165563
83.02165563
83.02165563
83.01672235
83.01672235
83.01672235
83.01672235
83.01672235
82.98527888
82.98527888
82.98527888
82.98527888
82.98527888
83.01812594
83.01812594
83.01812594
83.01812594
83.01812594
83.01579415
83.01579415
83.01579415
83.01579415
83.01579415
82.99569075
82.99569075
82.99569075
82.99569075
82.99569075
82.98302069
82.98302069
82.98302069
82.98302069
82.98302069
83.00499003
83.00499003
83.00499003
83.00499003
83.00499003
82.99966678
82.99966678
82.99966678
82.99966678
82.99966678
82.99592222
82.99592222
82.99592222
82.99592222
82.99592222
83.00345918
83.00345918
83.00345918
83.00345918
83.00345918
83.01333139
83.01333139
83.01333139
83.01333139
83.01333139
83.00842983
83.00842983
83.00842983
83.00842983
83.00842983
83.00394345
83.00394345
83.00394345
83.00394345
83.00394345
83.01076847
83.01076847
83.01076847
83.01076847
83.01076847
83.00007886
83.00007886
83.00007886
83.00007886
83.00007886
82.99526654
82.99526654
82.99526654
82.99526654
82.99526654
83.00497465
83.00497465
83.00497465
83.00497465
83.00497465
83.00454922
83.00454922
83.00454922
83.00454922
83.00454922
83.0162757
83.0162757
83.0162757
83.0162757
83.0162757
82.98868028
82.98868028
82.98868028
82.98868028
82.98868028
83.01453732
83.01453732
83.01453732
83.01453732
83.01453732
83.02565979
83.02565979
83.02565979
83.02565979
83.02565979
82.98594084
82.98594084
82.98594084
82.98594084
82.98594084
83.01507803
83.01507803
83.01507803
83.01507803
83.01507803
82.97723235
82.97723235
82.97723235
82.97723235
82.97723235
82.9909483
82.9909483
82.9909483
82.9909483
82.9909483
83.01797907
83.01797907
83.01797907
83.01797907
83.01797907
83.01317102
83.01317102
83.01317102
83.01317102
83.01317102
82.97754653
82.97754653
82.97754653
82.97754653
82.97754653
82.97952136
82.97952136
82.97952136
82.97952136
82.97952136
82.9901796
82.9901796
82.9901796
82.9901796
82.9901796
83.02794292
83.02794292
83.02794292
83.02794292
83.02794292
83.00583309
83.00583309
83.00583309
83.00583309
83.00583309
82.97335131
82.97335131
82.97335131
82.97335131
82.97335131
82.98253877
82.98253877
82.98253877
82.98253877
82.98253877
83.02398463
83.02398463
83.02398463
83.02398463
83.02398463
83.00599787
83.00599787
83.00599787
83.00599787
83.00599787
82.97847779
82.97847779
82.97847779
82.97847779
82.97847779
82.98119661
82.98119661
82.98119661
82.98119661
82.98119661
82.98776039
82.98776039
82.98776039
82.98776039
82.98776039
83.00666744
83.00666744
83.00666744
83.00666744
83.00666744
82.98109615
82.98109615
82.98109615
82.98109615
82.98109615
82.99032164
82.99032164
82.99032164
82.99032164
82.99032164
82.99630831
82.99630831
82.99630831
82.99630831
82.99630831
83.00723246
83.00723246
83.00723246
83.00723246
83.00723246
82.97940315
82.97940315
82.97940315
82.97940315
82.97940315
83.00920358
83.00920358
83.00920358
83.00920358
83.00920358
83.00063136
83.00063136
83.00063136
83.00063136
83.00063136
83.00408196
83.00408196
83.00408196
83.00408196
83.00408196
83.00850085
83.00850085
83.00850085
83.00850085
83.00850085
83.00273853
83.00273853
83.00273853
83.00273853
83.00273853
83.00484543
83.00484543
83.00484543
83.00484543
83.00484543
82.99870778
82.99870778
82.99870778
82.99870778
82.99870778
83.00241737
83.00241737
83.00241737
83.00241737
83.00241737
83.00909483
83.00909483
83.00909483
83.00909483
83.00909483
82.98322755
82.98322755
82.98322755
82.98322755
82.98322755
83.00206809
83.00206809
83.00206809
83.00206809
83.00206809
83.01724432
83.01724432
83.01724432
83.01724432
83.01724432
82.98469387
82.98469387
82.98469387
82.98469387
82.98469387
83.02121534
83.02121534
83.02121534
83.02121534
83.02121534
82.96432291
82.96432291
82.96432291
82.96432291
82.96432291
83.00532026
83.00532026
83.00532026
83.00532026
83.00532026
83.00889407
83.00889407
83.00889407
83.00889407
83.00889407
82.98044361
82.98044361
82.98044361
82.98044361
82.98044361
82.97241361
82.97241361
82.97241361
82.97241361
82.97241361
83.0070852
83.0070852
83.0070852
83.0070852
83.0070852
83.00557924
83.00557924
83.00557924
83.00557924
83.00557924
83.01576375
83.01576375
83.01576375
83.01576375
83.01576375
83.02552874
83.02552874
83.02552874
83.02552874
83.02552874
83.02541751
83.02541751
83.02541751
83.02541751
83.02541751
83.01203198
83.01203198
83.01203198
83.01203198
83.01203198
82.99830362
82.99830362
82.99830362
82.99830362
82.99830362
82.96040591
82.96040591
82.96040591
82.96040591
82.96040591
82.98375242
82.98375242
82.98375242
82.98375242
82.98375242
83.00665132
83.00665132
83.00665132
83.00665132
83.00665132
82.99303375
82.99303375
82.99303375
82.99303375
82.99303375
83.01781238
83.01781238
83.01781238
83.01781238
83.01781238
83.02266762
83.02266762
83.02266762
83.02266762
83.02266762
83.01032683
83.01032683
83.01032683
83.01032683
83.01032683
83.01044281
83.01044281
83.01044281
83.01044281
83.01044281
82.9600195
82.9600195
82.9600195
82.9600195
82.9600195
83.00027009
83.00027009
83.00027009
83.00027009
83.00027009
83.01537743
83.01537743
83.01537743
83.01537743
83.01537743
82.99460283
82.99460283
82.99460283
82.99460283
82.99460283
82.99379746
82.99379746
82.99379746
82.99379746
82.99379746
82.99762443
82.99762443
82.99762443
82.99762443
82.99762443
83.00691176
83.00691176
83.00691176
83.00691176
83.00691176
83.00911407
83.00911407
83.00911407
83.00911407
83.00911407
83.01509946
83.01509946
83.01509946
83.01509946
83.01509946
83.00417456
83.00417456
83.00417456
83.00417456
83.00417456
83.00501077
83.00501077
83.00501077
83.00501077
83.00501077
83.00525922
83.00525922
83.00525922
83.00525922
83.00525922
82.98736258
82.98736258
82.98736258
82.98736258
82.98736258
83.02632647
83.02632647
83.02632647
83.02632647
83.02632647
83.02386076
83.02386076
83.02386076
83.02386076
83.02386076
83.02360577
83.02360577
83.02360577
83.02360577
83.02360577
83.00200205
83.00200205
83.00200205
83.00200205
83.00200205
83.01589292
83.01589292
83.01589292
83.01589292
83.01589292
83.00882595
83.00882595
83.00882595
83.00882595
83.00882595
82.99157072
82.99157072
82.99157072
82.99157072
82.99157072
82.98609944
82.98609944
82.98609944
82.98609944
82.98609944
82.9802281
82.9802281
82.9802281
82.9802281
82.9802281
83.00355177
83.00355177
83.00355177
83.00355177
83.00355177
83.01901564
83.01901564
83.01901564
83.01901564
83.01901564
83.01831946
83.01831946
83.01831946
83.01831946
83.01831946
83.00222354
83.00222354
83.00222354
83.00222354
83.00222354
83.01089927
83.01089927
83.01089927
83.01089927
83.01089927
83.00280203
83.00280203
83.00280203
83.00280203
83.00280203
82.99033024
82.99033024
82.99033024
82.99033024
82.99033024
82.97866682
82.97866682
82.97866682
82.97866682
82.97866682
82.95733258
82.95733258
82.95733258
82.95733258
82.95733258
83.00776452
83.00776452
83.00776452
83.00776452
83.00776452
82.97728965
82.97728965
82.97728965
82.97728965
82.97728965
83.00047844
83.00047844
83.00047844
83.00047844
83.00047844
82.97978014
82.97978014
82.97978014
82.97978014
82.97978014
83.00060397
83.00060397
83.00060397
83.00060397
83.00060397
82.96955211
82.96955211
82.96955211
82.96955211
82.96955211
83.01705904
83.01705904
83.01705904
83.01705904
83.01705904
83.00592874
83.00592874
83.00592874
83.00592874
83.00592874
83.0289186
83.0289186
83.0289186
83.0289186
83.0289186
83.00073538
83.00073538
83.00073538
83.00073538
83.00073538
82.99876886
82.99876886
82.99876886
82.99876886
82.99876886
83.00921475
83.00921475
83.00921475
83.00921475
83.00921475
82.96595099
82.96595099
82.96595099
82.96595099
82.96595099
83.01929358
83.01929358
83.01929358
83.01929358
83.01929358
82.97231013
82.97231013
82.97231013
82.97231013
82.97231013
82.94759827
82.94759827
82.94759827
82.94759827
82.94759827
82.99385092
82.99385092
82.99385092
82.99385092
82.99385092
83.00452589
83.00452589
83.00452589
83.00452589
83.00452589
83.02534054
83.02534054
83.02534054
83.02534054
83.02534054
82.97219711
82.97219711
82.97219711
82.97219711
82.97219711
82.98763045
82.98763045
82.98763045
82.98763045
82.98763045
82.99178869
82.99178869
82.99178869
82.99178869
82.99178869
82.97210387
82.97210387
82.97210387
82.97210387
82.97210387
83.01366681
83.01366681
83.01366681
83.01366681
83.01366681
83.00355188
83.00355188
83.00355188
83.00355188
83.00355188
82.99642515
82.99642515
82.99642515
82.99642515
82.99642515
83.00697032
83.00697032
83.00697032
83.00697032
83.00697032
82.95646594
82.95646594
82.95646594
82.95646594
82.95646594
83.00056912
83.00056912
83.00056912
83.00056912
83.00056912
82.98886184
82.98886184
82.98886184
82.98886184
82.98886184
83.0214182
83.0214182
83.0214182
83.0214182
83.0214182
82.98045108
82.98045108
82.98045108
82.98045108
82.98045108
83.01337196
83.01337196
83.01337196
83.01337196
83.01337196
83.02195805
83.02195805
83.02195805
83.02195805
83.02195805
82.98065323
82.98065323
82.98065323
82.98065323
82.98065323
83.00376971
83.00376971
83.00376971
83.00376971
83.00376971
82.99886852
82.99886852
82.99886852
82.99886852
82.99886852
83.00007163
83.00007163
83.00007163
83.00007163
83.00007163
82.98363321
82.98363321
82.98363321
82.98363321
82.98363321
83.00151038
83.00151038
83.00151038
83.00151038
83.00151038
83.00866109
83.00866109
83.00866109
83.00866109
83.00866109
83.00166487
83.00166487
83.00166487
83.00166487
83.00166487
83.00669155
83.00669155
83.00669155
83.00669155
83.00669155
83.01116002
83.01116002
83.01116002
83.01116002
83.01116002
83.02077792
83.02077792
83.02077792
83.02077792
83.02077792
83.02781195
83.02781195
83.02781195
83.02781195
83.02781195
82.95630093
82.95630093
82.95630093
82.95630093
82.95630093
83.01729757
83.01729757
83.01729757
83.01729757
83.01729757
83.00850986
83.00850986
83.00850986
83.00850986
83.00850986
82.96723409
82.96723409
82.96723409
82.96723409
82.96723409
82.98209611
82.98209611
82.98209611
82.98209611
82.98209611
83.00429451
83.00429451
83.00429451
83.00429451
83.00429451
82.96094402
82.96094402
82.96094402
82.96094402
82.96094402
82.99454762
82.99454762
82.99454762
82.99454762
82.99454762
83.01110438
83.01110438
83.01110438
83.01110438
83.01110438
82.96946562
82.96946562
82.96946562
82.96946562
82.96946562
83.02668275
83.02668275
83.02668275
83.02668275
83.02668275
83.00988914
83.00988914
83.00988914
83.00988914
83.00988914
83.00068534
83.00068534
83.00068534
83.00068534
83.00068534
82.98024598
82.98024598
82.98024598
82.98024598
82.98024598
83.02724084
83.02724084
83.02724084
83.02724084
83.02724084
83.03650035
83.03650035
83.03650035
83.03650035
83.03650035
83.02007151
83.02007151
83.02007151
83.02007151
83.02007151
83.01349563
83.01349563
83.01349563
83.01349563
83.01349563
83.01473364
83.01473364
83.01473364
83.01473364
83.01473364
83.0048764
83.0048764
83.0048764
83.0048764
83.0048764
82.98957042
82.98957042
82.98957042
82.98957042
82.98957042
83.00892553
83.00892553
83.00892553
83.00892553
83.00892553
82.99881377
82.99881377
82.99881377
82.99881377
82.99881377
83.00697055
83.00697055
83.00697055
83.00697055
83.00697055
83.01580343
83.01580343
83.01580343
83.01580343
83.01580343
83.01055685
83.01055685
83.01055685
83.01055685
83.01055685
83.00694202
83.00694202
83.00694202
83.00694202
83.00694202
82.98672002
82.98672002
82.98672002
82.98672002
82.98672002
83.00684198
83.00684198
83.00684198
83.00684198
83.00684198
83.00335531
83.00335531
83.00335531
83.00335531
83.00335531
82.98928306
82.98928306
82.98928306
82.98928306
82.98928306
83.009795
83.009795
83.009795
83.009795
83.009795
83.00752956
83.00752956
83.00752956
83.00752956
83.00752956
83.0040095
83.0040095
83.0040095
83.0040095
83.0040095
82.97082059
82.97082059
82.97082059
82.97082059
82.97082059
83.01120447
83.01120447
83.01120447
83.01120447
83.01120447
82.98193555
82.98193555
82.98193555
82.98193555
82.98193555
83.01465767
83.01465767
83.01465767
83.01465767
83.01465767
83.01857379
83.01857379
83.01857379
83.01857379
83.01857379
83.02521297
83.02521297
83.02521297
83.02521297
83.02521297
83.00322691
83.00322691
83.00322691
83.00322691
83.00322691
83.00296239
83.00296239
83.00296239
83.00296239
83.00296239
83.00420684
83.00420684
83.00420684
83.00420684
83.00420684
83.01016705
83.01016705
83.01016705
83.01016705
83.01016705
83.00664083
83.00664083
83.00664083
83.00664083
83.00664083
83.00389734
83.00389734
83.00389734
83.00389734
83.00389734
82.98625913
82.98625913
82.98625913
82.98625913
82.98625913
83.01411206
83.01411206
83.01411206
83.01411206
83.01411206
82.96861782
82.96861782
82.96861782
82.96861782
82.96861782
82.97518844
82.97518844
82.97518844
82.97518844
82.97518844
82.99806218
82.99806218
82.99806218
82.99806218
82.99806218
82.98177778
82.98177778
82.98177778
82.98177778
82.98177778
83.01020743
83.01020743
83.01020743
83.01020743
83.01020743
83.01142937
83.01142937
83.01142937
83.01142937
83.01142937
82.98461784
82.98461784
82.98461784
82.98461784
82.98461784
82.98313533
82.98313533
82.98313533
82.98313533
82.98313533
82.98678268
82.98678268
82.98678268
82.98678268
82.98678268
83.01534324
83.01534324
83.01534324
83.01534324
83.01534324
83.01977466
83.01977466
83.01977466
83.01977466
83.01977466
82.97104712
82.97104712
82.97104712
82.97104712
82.97104712
82.95348482
82.95348482
82.95348482
82.95348482
82.95348482
83.01118611
83.01118611
83.01118611
83.01118611
83.01118611
82.99641444
82.99641444
82.99641444
82.99641444
82.99641444
82.94993155
82.94993155
82.94993155
82.94993155
82.94993155
83.0056097
83.0056097
83.0056097
83.0056097
83.0056097
82.99276518
82.99276518
82.99276518
82.99276518
82.99276518
83.00092265
83.00092265
83.00092265
83.00092265
83.00092265
82.99266125
82.99266125
82.99266125
82.99266125
82.99266125
83.00363104
83.00363104
83.00363104
83.00363104
83.00363104
83.00672683
83.00672683
83.00672683
83.00672683
83.00672683
83.01638178
83.01638178
83.01638178
83.01638178
83.01638178
82.99710774
82.99710774
82.99710774
82.99710774
82.99710774
83.01560738
83.01560738
83.01560738
83.01560738
83.01560738
83.01048002
83.01048002
83.01048002
83.01048002
83.01048002
83.01244791
83.01244791
83.01244791
83.01244791
83.01244791
82.9971625
82.9971625
82.9971625
82.9971625
82.9971625
83.00222651
83.00222651
83.00222651
83.00222651
83.00222651
83.00560696
83.00560696
83.00560696
83.00560696
83.00560696
82.97820654
82.97820654
82.97820654
82.97820654
82.97820654
83.01613481
83.01613481
83.01613481
83.01613481
83.01613481
82.99945644
82.99945644
82.99945644
82.99945644
82.99945644
82.99508123
82.99508123
82.99508123
82.99508123
82.99508123
83.00942852
83.00942852
83.00942852
83.00942852
83.00942852
83.00112434
83.00112434
83.00112434
83.00112434
83.00112434
83.02585003
83.02585003
83.02585003
83.02585003
83.02585003
83.01079106
83.01079106
83.01079106
83.01079106
83.01079106
82.99232434
82.99232434
82.99232434
82.99232434
82.99232434
82.96454317
82.96454317
82.96454317
82.96454317
82.96454317
83.02353285
83.02353285
83.02353285
83.02353285
83.02353285
83.01082714
83.01082714
83.01082714
83.01082714
83.01082714
82.99388482
82.99388482
82.99388482
82.99388482
82.99388482
83.00646611
83.00646611
83.00646611
83.00646611
83.00646611
83.00147371
83.00147371
83.00147371
83.00147371
83.00147371
82.95766036
82.95766036
82.95766036
82.95766036
82.95766036
83.02557981
83.02557981
83.02557981
83.02557981
83.02557981
82.96671713
82.96671713
82.96671713
82.96671713
82.96671713
83.00512535
83.00512535
83.00512535
83.00512535
83.00512535
83.00644365
83.00644365
83.00644365
83.00644365
83.00644365
83.01126677
83.01126677
83.01126677
83.01126677
83.01126677
83.00484711
83.00484711
83.00484711
83.00484711
83.00484711
82.99838855
82.99838855
82.99838855
82.99838855
82.99838855
83.00612003
83.00612003
83.00612003
83.00612003
83.00612003
82.98681957
82.98681957
82.98681957
82.98681957
82.98681957
83.00835151
83.00835151
83.00835151
83.00835151
83.00835151
82.98194375
82.98194375
82.98194375
82.98194375
82.98194375
82.9807756
82.9807756
82.9807756
82.9807756
82.9807756
83.00586287
83.00586287
83.00586287
83.00586287
83.00586287
82.97664711
82.97664711
82.97664711
82.97664711
82.97664711
83.00668635
83.00668635
83.00668635
83.00668635
83.00668635
82.99229851
82.99229851
82.99229851
82.99229851
82.99229851
82.98124467
82.98124467
82.98124467
82.98124467
82.98124467
82.96573405
82.96573405
82.96573405
82.96573405
82.96573405
82.99756508
82.99756508
82.99756508
82.99756508
82.99756508
82.99800879
82.99800879
82.99800879
82.99800879
82.99800879
83.01735048
83.01735048
83.01735048
83.01735048
83.01735048
82.99427959
82.99427959
82.99427959
82.99427959
82.99427959
82.98308542
82.98308542
82.98308542
82.98308542
82.98308542
83.00706672
83.00706672
83.00706672
83.00706672
83.00706672
83.00686962
83.00686962
83.00686962
83.00686962
83.00686962
83.00488453
83.00488453
83.00488453
83.00488453
83.00488453
83.01467899
83.01467899
83.01467899
83.01467899
83.01467899
83.02578643
83.02578643
83.02578643
83.02578643
83.02578643
83.01059264
83.01059264
83.01059264
83.01059264
83.01059264
83.01581658
83.01581658
83.01581658
83.01581658
83.01581658
82.97655825
82.97655825
82.97655825
82.97655825
82.97655825
82.99106088
82.99106088
82.99106088
82.99106088
82.99106088
82.93665209
82.93665209
82.93665209
82.93665209
82.93665209
82.99908276
82.99908276
82.99908276
82.99908276
82.99908276
83.00547904
83.00547904
83.00547904
83.00547904
83.00547904
82.9775106
82.9775106
82.9775106
82.9775106
82.9775106
82.97448463
82.97448463
82.97448463
82.97448463
82.97448463
82.99981511
82.99981511
82.99981511
82.99981511
82.99981511
82.9735108
82.9735108
82.9735108
82.9735108
82.9735108
82.9761255
82.9761255
82.9761255
82.9761255
82.9761255
82.9772387
82.9772387
82.9772387
82.9772387
82.9772387
82.97794234
82.97794234
82.97794234
82.97794234
82.97794234
83.00475224
83.00475224
83.00475224
83.00475224
83.00475224
82.98055943
82.98055943
82.98055943
82.98055943
82.98055943
82.98585305
82.98585305
82.98585305
82.98585305
82.98585305
82.99457243
82.99457243
82.99457243
82.99457243
82.99457243
83.00695955
83.00695955
83.00695955
83.00695955
83.00695955
82.99986449
82.99986449
82.99986449
82.99986449
82.99986449
82.98035548
82.98035548
82.98035548
82.98035548
82.98035548
83.01614544
83.01614544
83.01614544
83.01614544
83.01614544
82.93850549
82.93850549
82.93850549
82.93850549
82.93850549
83.00967601
83.00967601
83.00967601
83.00967601
83.00967601
83.01883378
83.01883378
83.01883378
83.01883378
83.01883378
83.01321221
83.01321221
83.01321221
83.01321221
83.01321221
82.97271705
82.97271705
82.97271705
82.97271705
82.97271705
82.99755023
82.99755023
82.99755023
82.99755023
82.99755023
83.00360034
83.00360034
83.00360034
83.00360034
83.00360034
82.9916553
82.9916553
82.9916553
82.9916553
82.9916553
83.02460547
83.02460547
83.02460547
83.02460547
83.02460547
82.98273666
82.98273666
82.98273666
82.98273666
82.98273666
82.99728159
82.99728159
82.99728159
82.99728159
82.99728159
82.99552039
82.99552039
82.99552039
82.99552039
82.99552039
82.96787967
82.96787967
82.96787967
82.96787967
82.96787967
82.97818258
82.97818258
82.97818258
82.97818258
82.97818258
83.01556144
83.01556144
83.01556144
83.01556144
83.01556144
82.99192711
82.99192711
82.99192711
82.99192711
82.99192711
82.99165229
82.99165229
82.99165229
82.99165229
82.99165229
82.99038812
82.99038812
82.99038812
82.99038812
82.99038812
82.96611234
82.96611234
82.96611234
82.96611234
82.96611234
83.0157598
83.0157598
83.0157598
83.0157598
83.0157598
83.00948075
83.00948075
83.00948075
83.00948075
83.00948075
82.9973843
82.9973843
82.9973843
82.9973843
82.9973843
83.01318089
83.01318089
83.01318089
83.01318089
83.01318089
83.00112937
83.00112937
83.00112937
83.00112937
83.00112937
82.98525802
82.98525802
82.98525802
82.98525802
82.98525802
82.99729222
82.99729222
82.99729222
82.99729222
82.99729222
83.00698519
83.00698519
83.00698519
83.00698519
83.00698519
82.98811314
82.98811314
82.98811314
82.98811314
82.98811314
82.99562982
82.99562982
82.99562982
82.99562982
82.99562982
83.00295197
83.00295197
83.00295197
83.00295197
83.00295197
82.97228049
82.97228049
82.97228049
82.97228049
82.97228049
83.03048046
83.03048046
83.03048046
83.03048046
83.03048046
83.00588615
83.00588615
83.00588615
83.00588615
83.00588615
82.99853084
82.99853084
82.99853084
82.99853084
82.99853084
83.01057761
83.01057761
83.01057761
83.01057761
83.01057761
82.97447644
82.97447644
82.97447644
82.97447644
82.97447644
83.00381846
83.00381846
83.00381846
83.00381846
83.00381846
82.99488646
82.99488646
82.99488646
82.99488646
82.99488646
82.99009599
82.99009599
82.99009599
82.99009599
82.99009599
83.00587069
83.00587069
83.00587069
83.00587069
83.00587069
82.99244439
82.99244439
82.99244439
82.99244439
82.99244439
83.00825568
83.00825568
83.00825568
83.00825568
83.00825568
83.02127704
83.02127704
83.02127704
83.02127704
83.02127704
82.99867949
82.99867949
82.99867949
82.99867949
82.99867949
83.01242842
83.01242842
83.01242842
83.01242842
83.01242842
83.01005986
83.01005986
83.01005986
83.01005986
83.01005986
82.99526138
82.99526138
82.99526138
82.99526138
82.99526138
82.99822112
82.99822112
82.99822112
82.99822112
82.99822112
83.00526386
83.00526386
83.00526386
83.00526386
83.00526386
83.00181174
83.00181174
83.00181174
83.00181174
83.00181174
82.99299481
82.99299481
82.99299481
82.99299481
82.99299481
83.00912781
83.00912781
83.00912781
83.00912781
83.00912781
83.01512649
83.01512649
83.01512649
83.01512649
83.01512649
83.00336445
83.00336445
83.00336445
83.00336445
83.00336445
83.01625866
83.01625866
83.01625866
83.01625866
83.01625866
83.00546196
83.00546196
83.00546196
83.00546196
83.00546196
83.01526315
83.01526315
83.01526315
83.01526315
83.01526315
82.98568851
82.98568851
82.98568851
82.98568851
82.98568851
82.9944725
82.9944725
82.9944725
82.9944725
82.9944725
82.98049342
82.98049342
82.98049342
82.98049342
82.98049342
83.00655729
83.00655729
83.00655729
83.00655729
83.00655729
83.008309
83.008309
83.008309
83.008309
83.008309
83.00403559
83.00403559
83.00403559
83.00403559
83.00403559
83.00736533
83.00736533
83.00736533
83.00736533
83.00736533
83.0061561
83.0061561
83.0061561
83.0061561
83.0061561
83.00682528
83.00682528
83.00682528
83.00682528
83.00682528
83.00760337
83.00760337
83.00760337
83.00760337
83.00760337
82.9984642
82.9984642
82.9984642
82.9984642
82.9984642
82.99314656
82.99314656
82.99314656
82.99314656
82.99314656
83.0111999
83.0111999
83.0111999
83.0111999
83.0111999
82.98330312
82.98330312
82.98330312
82.98330312
82.98330312
82.98940405
82.98940405
82.98940405
82.98940405
82.98940405
83.01311158
83.01311158
83.01311158
83.01311158
83.01311158
83.00503962
83.00503962
83.00503962
83.00503962
83.00503962
83.00351085
83.00351085
83.00351085
83.00351085
83.00351085
83.01451953
83.01451953
83.01451953
83.01451953
83.01451953
83.02553987
83.02553987
83.02553987
83.02553987
83.02553987
83.00369372
83.00369372
83.00369372
83.00369372
83.00369372
82.97354605
82.97354605
82.97354605
82.97354605
82.97354605
82.99273699
82.99273699
82.99273699
82.99273699
82.99273699
82.99205416
82.99205416
82.99205416
82.99205416
82.99205416
82.99388257
82.99388257
82.99388257
82.99388257
82.99388257
82.99306641
82.99306641
82.99306641
82.99306641
82.99306641
83.0129279
83.0129279
83.0129279
83.0129279
83.0129279
82.99493015
82.99493015
82.99493015
82.99493015
82.99493015
83.01424538
83.01424538
83.01424538
83.01424538
83.01424538
82.99867159
82.99867159
82.99867159
82.99867159
82.99867159
83.01911091
83.01911091
83.01911091
83.01911091
83.01911091
82.93937794
82.93937794
82.93937794
82.93937794
82.93937794
82.99498683
82.99498683
82.99498683
82.99498683
82.99498683
82.97830832
82.97830832
82.97830832
82.97830832
82.97830832
82.98895522
82.98895522
82.98895522
82.98895522
82.98895522
83.01650721
83.01650721
83.01650721
83.01650721
83.01650721
83.02623741
83.02623741
83.02623741
83.02623741
83.02623741
82.97600512
82.97600512
82.97600512
82.97600512
82.97600512
82.98467251
82.98467251
82.98467251
82.98467251
82.98467251
82.9786859
82.9786859
82.9786859
82.9786859
82.9786859
83.01151356
83.01151356
83.01151356
83.01151356
83.01151356
82.96872194
82.96872194
82.96872194
82.96872194
82.96872194
83.00276138
83.00276138
83.00276138
83.00276138
83.00276138
83.00697491
83.00697491
83.00697491
83.00697491
83.00697491
83.01262099
83.01262099
83.01262099
83.01262099
83.01262099
83.01959163
83.01959163
83.01959163
83.01959163
83.01959163
83.0105326
83.0105326
83.0105326
83.0105326
83.0105326
83.00012972
83.00012972
83.00012972
83.00012972
83.00012972
83.00045567
83.00045567
83.00045567
83.00045567
83.00045567
82.95115359
82.95115359
82.95115359
82.95115359
82.95115359
82.98038952
82.98038952
82.98038952
82.98038952
82.98038952
82.97687159
82.97687159
82.97687159
82.97687159
82.97687159
83.01671224
83.01671224
83.01671224
83.01671224
83.01671224
82.9809574
82.9809574
82.9809574
82.9809574
82.9809574
83.01510039
83.01510039
83.01510039
83.01510039
83.01510039
83.01114182
83.01114182
83.01114182
83.01114182
83.01114182
83.01098826
83.01098826
83.01098826
83.01098826
83.01098826
82.99039861
82.99039861
82.99039861
82.99039861
82.99039861
83.00306861
83.00306861
83.00306861
83.00306861
83.00306861
83.02418327
83.02418327
83.02418327
83.02418327
83.02418327
83.010777
83.010777
83.010777
83.010777
83.010777
83.00945094
83.00945094
83.00945094
83.00945094
83.00945094
83.01198749
83.01198749
83.01198749
83.01198749
83.01198749
83.02259005
83.02259005
83.02259005
83.02259005
83.02259005
83.0063973
83.0063973
83.0063973
83.0063973
83.0063973
83.01091586
83.01091586
83.01091586
83.01091586
83.01091586
83.00406765
83.00406765
83.00406765
83.00406765
83.00406765
82.9694354
82.9694354
82.9694354
82.9694354
82.9694354
82.97918623
82.97918623
82.97918623
82.97918623
82.97918623
83.01273397
83.01273397
83.01273397
83.01273397
83.01273397
83.01354441
83.01354441
83.01354441
83.01354441
83.01354441
82.99505648
82.99505648
82.99505648
82.99505648
82.99505648
83.02487603
83.02487603
83.02487603
83.02487603
83.02487603
83.00272914
83.00272914
83.00272914
83.00272914
83.00272914
83.0096395
83.0096395
83.0096395
83.0096395
83.0096395
83.00661167
83.00661167
83.00661167
83.00661167
83.00661167
83.0223057
83.0223057
83.0223057
83.0223057
83.0223057
83.00370085
83.00370085
83.00370085
83.00370085
83.00370085
83.02645651
83.02645651
83.02645651
83.02645651
83.02645651
82.99524439
82.99524439
82.99524439
82.99524439
82.99524439
83.01161057
83.01161057
83.01161057
83.01161057
83.01161057
83.00725785
83.00725785
83.00725785
83.00725785
83.00725785
83.01260771
83.01260771
83.01260771
83.01260771
83.01260771
82.98510847
82.98510847
82.98510847
82.98510847
82.98510847
82.97931738
82.97931738
82.97931738
82.97931738
82.97931738
83.00746053
83.00746053
83.00746053
83.00746053
83.00746053
83.01070603
83.01070603
83.01070603
83.01070603
83.01070603
83.00860093
83.00860093
83.00860093
83.00860093
83.00860093
82.97987525
82.97987525
82.97987525
82.97987525
82.97987525
83.02380199
83.02380199
83.02380199
83.02380199
83.02380199
83.00635978
83.00635978
83.00635978
83.00635978
83.00635978
82.96375359
82.96375359
82.96375359
82.96375359
82.96375359
82.97775072
82.97775072
82.97775072
82.97775072
82.97775072
82.97829546
82.97829546
82.97829546
82.97829546
82.97829546
83.00486854
83.00486854
83.00486854
83.00486854
83.00486854
83.02714212
83.02714212
83.02714212
83.02714212
83.02714212
82.99676773
82.99676773
82.99676773
82.99676773
82.99676773
83.00184434
83.00184434
83.00184434
83.00184434
83.00184434
83.02397037
83.02397037
83.02397037
83.02397037
83.02397037
82.9981698
82.9981698
82.9981698
82.9981698
82.9981698
83.01139243
83.01139243
83.01139243
83.01139243
83.01139243
82.99660295
82.99660295
82.99660295
82.99660295
82.99660295
82.96581761
82.96581761
82.96581761
82.96581761
82.96581761
83.00573077
83.00573077
83.00573077
83.00573077
83.00573077
83.00680531
83.00680531
83.00680531
83.00680531
83.00680531
82.993669
82.993669
82.993669
82.993669
82.993669
82.97405411
82.97405411
82.97405411
82.97405411
82.97405411
83.0223907
83.0223907
83.0223907
83.0223907
83.0223907
83.00707062
83.00707062
83.00707062
83.00707062
83.00707062
82.99731218
82.99731218
82.99731218
82.99731218
82.99731218
83.00992004
83.00992004
83.00992004
83.00992004
83.00992004
83.02312421
83.02312421
83.02312421
83.02312421
83.02312421
82.95746743
82.95746743
82.95746743
82.95746743
82.95746743
82.99540375
82.99540375
82.99540375
82.99540375
82.99540375
83.00677569
83.00677569
83.00677569
83.00677569
83.00677569
83.00699906
83.00699906
83.00699906
83.00699906
83.00699906
82.9984533
82.9984533
82.9984533
82.9984533
82.9984533
83.00590424
83.00590424
83.00590424
83.00590424
83.00590424
83.00807518
83.00807518
83.00807518
83.00807518
83.00807518
82.96995725
82.96995725
82.96995725
82.96995725
82.96995725
82.9744109
82.9744109
82.9744109
82.9744109
82.9744109
83.03544494
83.03544494
83.03544494
83.03544494
83.03544494
82.99694387
82.99694387
82.99694387
82.99694387
82.99694387
82.98405448
82.98405448
82.98405448
82.98405448
82.98405448
83.01327835
83.01327835
83.01327835
83.01327835
83.01327835
82.99888245
82.99888245
82.99888245
82.99888245
82.99888245
83.0159057
83.0159057
83.0159057
83.0159057
83.0159057
82.99168247
82.99168247
82.99168247
82.99168247
82.99168247
83.01483723
83.01483723
83.01483723
83.01483723
83.01483723
82.96359676
82.96359676
82.96359676
82.96359676
82.96359676
83.01612423
83.01612423
83.01612423
83.01612423
83.01612423
83.00204974
83.00204974
83.00204974
83.00204974
83.00204974
82.97599804
82.97599804
82.97599804
82.97599804
82.97599804
82.99145196
82.99145196
82.99145196
82.99145196
82.99145196
82.99662119
82.99662119
82.99662119
82.99662119
82.99662119
83.00820993
83.00820993
83.00820993
83.00820993
83.00820993
83.01166516
83.01166516
83.01166516
83.01166516
83.01166516
83.02047043
83.02047043
83.02047043
83.02047043
83.02047043
83.00355298
83.00355298
83.00355298
83.00355298
83.00355298
82.98513415
82.98513415
82.98513415
82.98513415
82.98513415
83.01011147
83.01011147
83.01011147
83.01011147
83.01011147
83.00181923
83.00181923
83.00181923
83.00181923
83.00181923
82.99911715
82.99911715
82.99911715
82.99911715
82.99911715
83.00489775
83.00489775
83.00489775
83.00489775
83.00489775
83.01735729
83.01735729
83.01735729
83.01735729
83.01735729
82.94000787
82.94000787
82.94000787
82.94000787
82.94000787
83.00750623
83.00750623
83.00750623
83.00750623
83.00750623
83.01667657
83.01667657
83.01667657
83.01667657
83.01667657
83.01099569
83.01099569
83.01099569
83.01099569
83.01099569
82.97916948
82.97916948
82.97916948
82.97916948
82.97916948
82.97964473
82.97964473
82.97964473
82.97964473
82.97964473
83.01780851
83.01780851
83.01780851
83.01780851
83.01780851
83.00904069
83.00904069
83.00904069
83.00904069
83.00904069
83.0038585
83.0038585
83.0038585
83.0038585
83.0038585
82.96949786
82.96949786
82.96949786
82.96949786
82.96949786
83.00492254
83.00492254
83.00492254
83.00492254
83.00492254
83.01687077
83.01687077
83.01687077
83.01687077
83.01687077
83.01195466
83.01195466
83.01195466
83.01195466
83.01195466
83.00924721
83.00924721
83.00924721
83.00924721
83.00924721
83.01157432
83.01157432
83.01157432
83.01157432
83.01157432
83.00400406
83.00400406
83.00400406
83.00400406
83.00400406
83.02546733
83.02546733
83.02546733
83.02546733
83.02546733
83.01316011
83.01316011
83.01316011
83.01316011
83.01316011
83.00691627
83.00691627
83.00691627
83.00691627
83.00691627
83.01338443
83.01338443
83.01338443
83.01338443
83.01338443
82.95926277
82.95926277
82.95926277
82.95926277
82.95926277
83.01363769
83.01363769
83.01363769
83.01363769
83.01363769
82.99390496
82.99390496
82.99390496
82.99390496
82.99390496
83.0083837
83.0083837
83.0083837
83.0083837
83.0083837
82.96227705
82.96227705
82.96227705
82.96227705
82.96227705
83.01555667
83.01555667
83.01555667
83.01555667
83.01555667
83.01189977
83.01189977
83.01189977
83.01189977
83.01189977
82.99347778
82.99347778
82.99347778
82.99347778
82.99347778
82.97323317
82.97323317
82.97323317
82.97323317
82.97323317
83.02523588
83.02523588
83.02523588
83.02523588
83.02523588
83.00426204
83.00426204
83.00426204
83.00426204
83.00426204
82.94479501
82.94479501
82.94479501
82.94479501
82.94479501
82.98596298
82.98596298
82.98596298
82.98596298
82.98596298
82.99601007
82.99601007
82.99601007
82.99601007
82.99601007
83.01043552
83.01043552
83.01043552
83.01043552
83.01043552
83.00106876
83.00106876
83.00106876
83.00106876
83.00106876
82.98310763
82.98310763
82.98310763
82.98310763
82.98310763
83.01621917
83.01621917
83.01621917
83.01621917
83.01621917
83.01471008
83.01471008
83.01471008
83.01471008
83.01471008
82.98235268
82.98235268
82.98235268
82.98235268
82.98235268
83.02302941
83.02302941
83.02302941
83.02302941
83.02302941
83.00709902
83.00709902
83.00709902
83.00709902
83.00709902
82.9863981
82.9863981
82.9863981
82.9863981
82.9863981
82.94296215
82.94296215
82.94296215
82.94296215
82.94296215
83.01622183
83.01622183
83.01622183
83.01622183
83.01622183
82.97631736
82.97631736
82.97631736
82.97631736
82.97631736
83.00423365
83.00423365
83.00423365
83.00423365
83.00423365
83.0194685
83.0194685
83.0194685
83.0194685
83.0194685
83.01233557
83.01233557
83.01233557
83.01233557
83.01233557
82.99992749
82.99992749
82.99992749
82.99992749
82.99992749
82.98404168
82.98404168
82.98404168
82.98404168
82.98404168
82.9994904
82.9994904
82.9994904
82.9994904
82.9994904
82.98689801
82.98689801
82.98689801
82.98689801
82.98689801
83.02111249
83.02111249
83.02111249
83.02111249
83.02111249
82.98723255
82.98723255
82.98723255
82.98723255
82.98723255
82.97978593
82.97978593
82.97978593
82.97978593
82.97978593
82.94825476
82.94825476
82.94825476
82.94825476
82.94825476
83.00596599
83.00596599
83.00596599
83.00596599
83.00596599
83.00412841
83.00412841
83.00412841
83.00412841
83.00412841
83.0115292
83.0115292
83.0115292
83.0115292
83.0115292
82.98181483
82.98181483
82.98181483
82.98181483
82.98181483
83.02192301
83.02192301
83.02192301
83.02192301
83.02192301
82.99843054
82.99843054
82.99843054
82.99843054
82.99843054
82.97817082
82.97817082
82.97817082
82.97817082
82.97817082
83.00302118
83.00302118
83.00302118
83.00302118
83.00302118
83.00614385
83.00614385
83.00614385
83.00614385
83.00614385
83.03274045
83.03274045
83.03274045
83.03274045
83.03274045
82.99091162
82.99091162
82.99091162
82.99091162
82.99091162
82.99763122
82.99763122
82.99763122
82.99763122
82.99763122
82.98408488
82.98408488
82.98408488
82.98408488
82.98408488
83.00031705
83.00031705
83.00031705
83.00031705
83.00031705
82.98848501
82.98848501
82.98848501
82.98848501
82.98848501
82.99697844
82.99697844
82.99697844
82.99697844
82.99697844
82.99549706
82.99549706
82.99549706
82.99549706
82.99549706
82.99234397
82.99234397
82.99234397
82.99234397
82.99234397
82.99045532
82.99045532
82.99045532
82.99045532
82.99045532
83.00266683
83.00266683
83.00266683
83.00266683
83.00266683
82.99549684
82.99549684
82.99549684
82.99549684
82.99549684
82.99916223
82.99916223
82.99916223
82.99916223
82.99916223
82.98398924
82.98398924
82.98398924
82.98398924
82.98398924
82.99778485
82.99778485
82.99778485
82.99778485
82.99778485
83.02562635
83.02562635
83.02562635
83.02562635
83.02562635
83.00805267
83.00805267
83.00805267
83.00805267
83.00805267
83.00610042
83.00610042
83.00610042
83.00610042
83.00610042
82.99975197
82.99975197
82.99975197
82.99975197
82.99975197
82.97591019
82.97591019
82.97591019
82.97591019
82.97591019
83.00306675
83.00306675
83.00306675
83.00306675
83.00306675
82.99942414
82.99942414
82.99942414
82.99942414
82.99942414
82.99730584
82.99730584
82.99730584
82.99730584
82.99730584
83.03356381
83.03356381
83.03356381
83.03356381
83.03356381
82.98396526
82.98396526
82.98396526
82.98396526
82.98396526
83.02220184
83.02220184
83.02220184
83.02220184
83.02220184
82.97909308
82.97909308
82.97909308
82.97909308
82.97909308
83.00605954
83.00605954
83.00605954
83.00605954
83.00605954
82.99178116
82.99178116
82.99178116
82.99178116
82.99178116
82.97629354
82.97629354
82.97629354
82.97629354
82.97629354
82.97515125
82.97515125
82.97515125
82.97515125
82.97515125
83.00489395
83.00489395
83.00489395
83.00489395
83.00489395
82.99932698
82.99932698
82.99932698
82.99932698
82.99932698
83.02034386
83.02034386
83.02034386
83.02034386
83.02034386
82.99559849
82.99559849
82.99559849
82.99559849
82.99559849
82.97510323
82.97510323
82.97510323
82.97510323
82.97510323
82.97629329
82.97629329
82.97629329
82.97629329
82.97629329
82.9974632
82.9974632
82.9974632
82.9974632
82.9974632
82.99972341
82.99972341
82.99972341
82.99972341
82.99972341
83.0118672
83.0118672
83.0118672
83.0118672
83.0118672
83.00462652
83.00462652
83.00462652
83.00462652
83.00462652
82.98226305
82.98226305
82.98226305
82.98226305
82.98226305
83.00472666
83.00472666
83.00472666
83.00472666
83.00472666
82.97122583
82.97122583
82.97122583
82.97122583
82.97122583
83.02642984
83.02642984
83.02642984
83.02642984
83.02642984
83.00768656
83.00768656
83.00768656
83.00768656
83.00768656
83.0032751
83.0032751
83.0032751
83.0032751
83.0032751
82.97842013
82.97842013
82.97842013
82.97842013
82.97842013
83.01481292
83.01481292
83.01481292
83.01481292
83.01481292
83.00993027
83.00993027
83.00993027
83.00993027
83.00993027
83.00327774
83.00327774
83.00327774
83.00327774
83.00327774
82.98904172
82.98904172
82.98904172
82.98904172
82.98904172
83.00297577
83.00297577
83.00297577
83.00297577
83.00297577
83.00048977
83.00048977
83.00048977
83.00048977
83.00048977
82.99401779
82.99401779
82.99401779
82.99401779
82.99401779
82.98207231
82.98207231
82.98207231
82.98207231
82.98207231
83.00548026
83.00548026
83.00548026
83.00548026
83.00548026
82.97634938
82.97634938
82.97634938
82.97634938
82.97634938
83.01727204
83.01727204
83.01727204
83.01727204
83.01727204
83.00581581
83.00581581
83.00581581
83.00581581
83.00581581
82.99810218
82.99810218
82.99810218
82.99810218
82.99810218
82.99158635
82.99158635
82.99158635
82.99158635
82.99158635
82.98325183
82.98325183
82.98325183
82.98325183
82.98325183
83.00500803
83.00500803
83.00500803
83.00500803
83.00500803
83.01890514
83.01890514
83.01890514
83.01890514
83.01890514
83.02635633
83.02635633
83.02635633
83.02635633
83.02635633
82.95782509
82.95782509
82.95782509
82.95782509
82.95782509
83.01534411
83.01534411
83.01534411
83.01534411
83.01534411
82.99842755
82.99842755
82.99842755
82.99842755
82.99842755
82.9802277
82.9802277
82.9802277
82.9802277
82.9802277
83.02043672
83.02043672
83.02043672
83.02043672
83.02043672
83.00313266
83.00313266
83.00313266
83.00313266
83.00313266
82.95091464
82.95091464
82.95091464
82.95091464
82.95091464
83.00931437
83.00931437
83.00931437
83.00931437
83.00931437
83.01415153
83.01415153
83.01415153
83.01415153
83.01415153
82.97663649
82.97663649
82.97663649
82.97663649
82.97663649
83.00801001
83.00801001
83.00801001
83.00801001
83.00801001
83.01325898
83.01325898
83.01325898
83.01325898
83.01325898
82.97709769
82.97709769
82.97709769
82.97709769
82.97709769
82.98183904
82.98183904
82.98183904
82.98183904
82.98183904
82.96168555
82.96168555
82.96168555
82.96168555
82.96168555
83.00945998
83.00945998
83.00945998
83.00945998
83.00945998
83.01252964
83.01252964
83.01252964
83.01252964
83.01252964
83.00384014
83.00384014
83.00384014
83.00384014
83.00384014
82.99005694
82.99005694
82.99005694
82.99005694
82.99005694
82.99358203
82.99358203
82.99358203
82.99358203
82.99358203
83.00523289
83.00523289
83.00523289
83.00523289
83.00523289
83.00828298
83.00828298
83.00828298
83.00828298
83.00828298
83.00315747
83.00315747
83.00315747
83.00315747
83.00315747
83.01779233
83.01779233
83.01779233
83.01779233
83.01779233
82.97954854
82.97954854
82.97954854
82.97954854
82.97954854
83.02156198
83.02156198
83.02156198
83.02156198
83.02156198
83.01794444
83.01794444
83.01794444
83.01794444
83.01794444
82.98432389
82.98432389
82.98432389
82.98432389
82.98432389
83.00066775
83.00066775
83.00066775
83.00066775
83.00066775
82.97628007
82.97628007
82.97628007
82.97628007
82.97628007
82.96207447
82.96207447
82.96207447
82.96207447
82.96207447
83.00405059
83.00405059
83.00405059
83.00405059
83.00405059
83.00514363
83.00514363
83.00514363
83.00514363
83.00514363
82.9764841
82.9764841
82.9764841
82.9764841
82.9764841
82.9956278
82.9956278
82.9956278
82.9956278
82.9956278
83.01056975
83.01056975
83.01056975
83.01056975
83.01056975
83.01434411
83.01434411
83.01434411
83.01434411
83.01434411
82.97802758
82.97802758
82.97802758
82.97802758
82.97802758
83.02363757
83.02363757
83.02363757
83.02363757
83.02363757
83.01481023
83.01481023
83.01481023
83.01481023
83.01481023
83.01327572
83.01327572
83.01327572
83.01327572
83.01327572
83.01323768
83.01323768
83.01323768
83.01323768
83.01323768
82.9848564
82.9848564
82.9848564
82.9848564
82.9848564
82.99795506
82.99795506
82.99795506
82.99795506
82.99795506
83.01498269
83.01498269
83.01498269
83.01498269
83.01498269
83.015581
83.015581
83.015581
83.015581
83.015581
83.0029012
83.0029012
83.0029012
83.0029012
83.0029012
82.95870591
82.95870591
82.95870591
82.95870591
82.95870591
83.00395787
83.00395787
83.00395787
83.00395787
83.00395787
82.9975116
82.9975116
82.9975116
82.9975116
82.9975116
82.97479935
82.97479935
82.97479935
82.97479935
82.97479935
82.98194648
82.98194648
82.98194648
82.98194648
82.98194648
83.00214027
83.00214027
83.00214027
83.00214027
83.00214027
83.01072598
83.01072598
83.01072598
83.01072598
83.01072598
83.00391914
83.00391914
83.00391914
83.00391914
83.00391914
83.01291163
83.01291163
83.01291163
83.01291163
83.01291163
83.00229888
83.00229888
83.00229888
83.00229888
83.00229888
83.0077443
83.0077443
83.0077443
83.0077443
83.0077443
83.007061
83.007061
83.007061
83.007061
83.007061
83.01914666
83.01914666
83.01914666
83.01914666
83.01914666
83.01798774
83.01798774
83.01798774
83.01798774
83.01798774
83.01412429
83.01412429
83.01412429
83.01412429
83.01412429
82.98997612
82.98997612
82.98997612
82.98997612
82.98997612
82.99872261
82.99872261
82.99872261
82.99872261
82.99872261
83.0197086
83.0197086
83.0197086
83.0197086
83.0197086
82.98232179
82.98232179
82.98232179
82.98232179
82.98232179
82.98204887
82.98204887
82.98204887
82.98204887
82.98204887
83.01746239
83.01746239
83.01746239
83.01746239
83.01746239
83.00085841
83.00085841
83.00085841
83.00085841
83.00085841
83.02867673
83.02867673
83.02867673
83.02867673
83.02867673
83.01855783
83.01855783
83.01855783
83.01855783
83.01855783
83.01596586
83.01596586
83.01596586
83.01596586
83.01596586
83.01370179
83.01370179
83.01370179
83.01370179
83.01370179
82.99492938
82.99492938
82.99492938
82.99492938
82.99492938
82.9748683
82.9748683
82.9748683
82.9748683
82.9748683
82.99523945
82.99523945
82.99523945
82.99523945
82.99523945
83.02461466
83.02461466
83.02461466
83.02461466
83.02461466
82.98206786
82.98206786
82.98206786
82.98206786
82.98206786
83.02701296
83.02701296
83.02701296
83.02701296
83.02701296
82.99261749
82.99261749
82.99261749
82.99261749
82.99261749
83.02498159
83.02498159
83.02498159
83.02498159
83.02498159
83.01392413
83.01392413
83.01392413
83.01392413
83.01392413
82.98437552
82.98437552
82.98437552
82.98437552
82.98437552
83.00838306
83.00838306
83.00838306
83.00838306
83.00838306
83.00440129
83.00440129
83.00440129
83.00440129
83.00440129
83.01565093
83.01565093
83.01565093
83.01565093
83.01565093
83.02749983
83.02749983
83.02749983
83.02749983
83.02749983
83.01988119
83.01988119
83.01988119
83.01988119
83.01988119
82.99475664
82.99475664
82.99475664
82.99475664
82.99475664
83.00168486
83.00168486
83.00168486
83.00168486
83.00168486
82.99704047
82.99704047
82.99704047
82.99704047
82.99704047
82.97617198
82.97617198
82.97617198
82.97617198
82.97617198
82.99189986
82.99189986
82.99189986
82.99189986
82.99189986
82.97875294
82.97875294
82.97875294
82.97875294
82.97875294
83.01622617
83.01622617
83.01622617
83.01622617
83.01622617
83.01573089
83.01573089
83.01573089
83.01573089
83.01573089
83.01633678
83.01633678
83.01633678
83.01633678
83.01633678
83.00766861
83.00766861
83.00766861
83.00766861
83.00766861
83.00156046
83.00156046
83.00156046
83.00156046
83.00156046
82.9854395
82.9854395
82.9854395
82.9854395
82.9854395
83.01065732
83.01065732
83.01065732
83.01065732
83.01065732
82.99963233
82.99963233
82.99963233
82.99963233
82.99963233
83.00801701
83.00801701
83.00801701
83.00801701
83.00801701
83.00894468
83.00894468
83.00894468
83.00894468
83.00894468
83.00329567
83.00329567
83.00329567
83.00329567
83.00329567
82.99501465
82.99501465
82.99501465
82.99501465
82.99501465
82.99584708
82.99584708
82.99584708
82.99584708
82.99584708
83.00603906
83.00603906
83.00603906
83.00603906
83.00603906
83.0165671
83.0165671
83.0165671
83.0165671
83.0165671
82.99647573
82.99647573
82.99647573
82.99647573
82.99647573
82.9965342
82.9965342
82.9965342
82.9965342
82.9965342
83.00430921
83.00430921
83.00430921
83.00430921
83.00430921
83.01199092
83.01199092
83.01199092
83.01199092
83.01199092
83.00381445
83.00381445
83.00381445
83.00381445
83.00381445
83.00879449
83.00879449
83.00879449
83.00879449
83.00879449
83.0206529
83.0206529
83.0206529
83.0206529
83.0206529
83.01990825
83.01990825
83.01990825
83.01990825
83.01990825
82.97570344
82.97570344
82.97570344
82.97570344
82.97570344
83.01239213
83.01239213
83.01239213
83.01239213
83.01239213
82.99613107
82.99613107
82.99613107
82.99613107
82.99613107
83.0219383
83.0219383
83.0219383
83.0219383
83.0219383
82.9976285
82.9976285
82.9976285
82.9976285
82.9976285
82.97645895
82.97645895
82.97645895
82.97645895
82.97645895
83.00166466
83.00166466
83.00166466
83.00166466
83.00166466
83.00200888
83.00200888
83.00200888
83.00200888
83.00200888
83.02588465
83.02588465
83.02588465
83.02588465
83.02588465
83.01625043
83.01625043
83.01625043
83.01625043
83.01625043
82.99954474
82.99954474
82.99954474
82.99954474
82.99954474
82.98466226
82.98466226
82.98466226
82.98466226
82.98466226
83.01697421
83.01697421
83.01697421
83.01697421
83.01697421
82.97776622
82.97776622
82.97776622
82.97776622
82.97776622
82.97432252
82.97432252
82.97432252
82.97432252
82.97432252
83.00311991
83.00311991
83.00311991
83.00311991
83.00311991
82.99834893
82.99834893
82.99834893
82.99834893
82.99834893
83.00117237
83.00117237
83.00117237
83.00117237
83.00117237
82.97979455
82.97979455
82.97979455
82.97979455
82.97979455
83.02355545
83.02355545
83.02355545
83.02355545
83.02355545
82.97770997
82.97770997
82.97770997
82.97770997
82.97770997
83.01566249
83.01566249
83.01566249
83.01566249
83.01566249
82.9786902
82.9786902
82.9786902
82.9786902
82.9786902
82.97303058
82.97303058
82.97303058
82.97303058
82.97303058
83.00958523
83.00958523
83.00958523
83.00958523
83.00958523
83.02839794
83.02839794
83.02839794
83.02839794
83.02839794
82.99966577
82.99966577
82.99966577
82.99966577
82.99966577
82.9943185
82.9943185
82.9943185
82.9943185
82.9943185
83.01246969
83.01246969
83.01246969
83.01246969
83.01246969
82.99214096
82.99214096
82.99214096
82.99214096
82.99214096
82.98008965
82.98008965
82.98008965
82.98008965
82.98008965
83.01981556
83.01981556
83.01981556
83.01981556
83.01981556
82.96800209
82.96800209
82.96800209
82.96800209
82.96800209
83.01317349
83.01317349
83.01317349
83.01317349
83.01317349
83.0032807
83.0032807
83.0032807
83.0032807
83.0032807
83.01339345
83.01339345
83.01339345
83.01339345
83.01339345
83.01038488
83.01038488
83.01038488
83.01038488
83.01038488
83.00941685
83.00941685
83.00941685
83.00941685
83.00941685
82.98822891
82.98822891
82.98822891
82.98822891
82.98822891
82.99735205
82.99735205
82.99735205
82.99735205
82.99735205
83.01227654
83.01227654
83.01227654
83.01227654
83.01227654
83.00263005
83.00263005
83.00263005
83.00263005
83.00263005
83.02562355
83.02562355
83.02562355
83.02562355
83.02562355
83.0156845
83.0156845
83.0156845
83.0156845
83.0156845
82.97185686
82.97185686
82.97185686
82.97185686
82.97185686
83.01358286
83.01358286
83.01358286
83.01358286
83.01358286
83.00082529
83.00082529
83.00082529
83.00082529
83.00082529
83.02205443
83.02205443
83.02205443
83.02205443
83.02205443
82.99837631
82.99837631
82.99837631
82.99837631
82.99837631
83.00465709
83.00465709
83.00465709
83.00465709
83.00465709
82.97980634
82.97980634
82.97980634
82.97980634
82.97980634
82.98956766
82.98956766
82.98956766
82.98956766
82.98956766
83.00402198
83.00402198
83.00402198
83.00402198
83.00402198
82.97278647
82.97278647
82.97278647
82.97278647
82.97278647
82.99822851
82.99822851
82.99822851
82.99822851
82.99822851
83.02181556
83.02181556
83.02181556
83.02181556
83.02181556
83.01836421
83.01836421
83.01836421
83.01836421
83.01836421
82.9955251
82.9955251
82.9955251
82.9955251
82.9955251
83.00645544
83.00645544
83.00645544
83.00645544
83.00645544
83.01779141
83.01779141
83.01779141
83.01779141
83.01779141
83.01983191
83.01983191
83.01983191
83.01983191
83.01983191
83.00536044
83.00536044
83.00536044
83.00536044
83.00536044
83.0174199
83.0174199
83.0174199
83.0174199
83.0174199
83.00361487
83.00361487
83.00361487
83.00361487
83.00361487
83.02335598
83.02335598
83.02335598
83.02335598
83.02335598
83.00558356
83.00558356
83.00558356
83.00558356
83.00558356
83.02281886
83.02281886
83.02281886
83.02281886
83.02281886
83.01005456
83.01005456
83.01005456
83.01005456
83.01005456
83.01524918
83.01524918
83.01524918
83.01524918
83.01524918
83.00324882
83.00324882
83.00324882
83.00324882
83.00324882
82.98352625
82.98352625
82.98352625
82.98352625
82.98352625
82.98751427
82.98751427
82.98751427
82.98751427
82.98751427
83.01527715
83.01527715
83.01527715
83.01527715
83.01527715
83.01868721
83.01868721
83.01868721
83.01868721
83.01868721
82.99609104
82.99609104
82.99609104
82.99609104
82.99609104
82.95686215
82.95686215
82.95686215
82.95686215
82.95686215
82.99765937
82.99765937
82.99765937
82.99765937
82.99765937
83.00394564
83.00394564
83.00394564
83.00394564
83.00394564
82.97257285
82.97257285
82.97257285
82.97257285
82.97257285
82.9870648
82.9870648
82.9870648
82.9870648
82.9870648
83.01449471
83.01449471
83.01449471
83.01449471
83.01449471
83.00570704
83.00570704
83.00570704
83.00570704
83.00570704
83.00416953
83.00416953
83.00416953
83.00416953
83.00416953
83.00293975
83.00293975
83.00293975
83.00293975
83.00293975
82.99807394
82.99807394
82.99807394
82.99807394
82.99807394
83.0275108
83.0275108
83.0275108
83.0275108
83.0275108
83.00115513
83.00115513
83.00115513
83.00115513
83.00115513
83.02424355
83.02424355
83.02424355
83.02424355
83.02424355
82.99599492
82.99599492
82.99599492
82.99599492
82.99599492
83.01637791
83.01637791
83.01637791
83.01637791
83.01637791
82.96856882
82.96856882
82.96856882
82.96856882
82.96856882
83.00664161
83.00664161
83.00664161
83.00664161
83.00664161
82.94458757
82.94458757
82.94458757
82.94458757
82.94458757
82.99336329
82.99336329
82.99336329
82.99336329
82.99336329
83.0255
83.0255
83.0255
83.0255
83.0255
83.0037597
83.0037597
83.0037597
83.0037597
83.0037597
82.98574452
82.98574452
82.98574452
82.98574452
82.98574452
82.9971154
82.9971154
82.9971154
82.9971154
82.9971154
82.98663655
82.98663655
82.98663655
82.98663655
82.98663655
82.99956798
82.99956798
82.99956798
82.99956798
82.99956798
83.00587918
83.00587918
83.00587918
83.00587918
83.00587918
82.99080959
82.99080959
82.99080959
82.99080959
82.99080959
83.00862558
83.00862558
83.00862558
83.00862558
83.00862558
82.97754146
82.97754146
82.97754146
82.97754146
82.97754146
82.99982521
82.99982521
82.99982521
82.99982521
82.99982521
82.9807657
82.9807657
82.9807657
82.9807657
82.9807657
83.00609232
83.00609232
83.00609232
83.00609232
83.00609232
82.99973648
82.99973648
82.99973648
82.99973648
82.99973648
''')

array_1 = data.split('\n')
origin_log_1 = []
origin_log = []
for i in range(len(array_1)-1):
  if(array_1[i] != ''):
     origin_log_1.append(float(array_1[i]))
for i in range(len(origin_log_1)-1):
  if(origin_log_1[i] != origin_log_1[i+1]):
    origin_log.append(origin_log_1[i])

print((origin_log[0]))
# origin_log.append(float(len(array_1)-1))

In [ ]:
# @title
data1 = ('''
25.30985469
25.30985469
25.30985469
25.30985469
25.30985469
25.32378553
25.32378553
25.32378553
25.32378553
25.32378553
25.36114673
25.36114673
25.36114673
25.36114673
25.36114673
25.31599308
25.31599308
25.31599308
25.31599308
25.31599308
25.32585547
25.32585547
25.32585547
25.32585547
25.32585547
25.3082424
25.3082424
25.3082424
25.3082424
25.3082424
25.35804748
25.35804748
25.35804748
25.35804748
25.35804748
25.27902872
25.27902872
25.27902872
25.27902872
25.27902872
25.35503201
25.35503201
25.35503201
25.35503201
25.35503201
25.36896631
25.36896631
25.36896631
25.36896631
25.36896631
25.34420118
25.34420118
25.34420118
25.34420118
25.34420118
25.3463697
25.3463697
25.3463697
25.3463697
25.3463697
25.37658287
25.37658287
25.37658287
25.37658287
25.37658287
25.29390593
25.29390593
25.29390593
25.29390593
25.29390593
25.30878346
25.30878346
25.30878346
25.30878346
25.30878346
25.3628417
25.3628417
25.3628417
25.3628417
25.3628417
25.33813138
25.33813138
25.33813138
25.33813138
25.33813138
25.32710803
25.32710803
25.32710803
25.32710803
25.32710803
25.32209484
25.32209484
25.32209484
25.32209484
25.32209484
25.31959297
25.31959297
25.31959297
25.31959297
25.31959297
25.31875383
25.31875383
25.31875383
25.31875383
25.31875383
25.31867032
25.31867032
25.31867032
25.31867032
25.31867032
25.30877621
25.30877621
25.30877621
25.30877621
25.30877621
25.32516897
25.32516897
25.32516897
25.32516897
25.32516897
25.32433704
25.32433704
25.32433704
25.32433704
25.32433704
25.32505542
25.32505542
25.32505542
25.32505542
25.32505542
25.32123524
25.32123524
25.32123524
25.32123524
25.32123524
25.30705377
25.30705377
25.30705377
25.30705377
25.30705377
25.31609416
25.31609416
25.31609416
25.31609416
25.31609416
25.33150378
25.33150378
25.33150378
25.33150378
25.33150378
25.32675435
25.32675435
25.32675435
25.32675435
25.32675435
25.31381636
25.31381636
25.31381636
25.31381636
25.31381636
25.33816145
25.33816145
25.33816145
25.33816145
25.33816145
25.3228057
25.3228057
25.3228057
25.3228057
25.3228057
25.37004823
25.37004823
25.37004823
25.37004823
25.37004823
25.32211911
25.32211911
25.32211911
25.32211911
25.32211911
25.35532377
25.35532377
25.35532377
25.35532377
25.35532377
25.37041474
25.37041474
25.37041474
25.37041474
25.37041474
25.28761985
25.28761985
25.28761985
25.28761985
25.28761985
25.32270474
25.32270474
25.32270474
25.32270474
25.32270474
25.31698431
25.31698431
25.31698431
25.31698431
25.31698431
25.35005162
25.35005162
25.35005162
25.35005162
25.35005162
25.31979113
25.31979113
25.31979113
25.31979113
25.31979113
25.33030354
25.33030354
25.33030354
25.33030354
25.33030354
25.3185187
25.3185187
25.3185187
25.3185187
25.3185187
25.36487555
25.36487555
25.36487555
25.36487555
25.36487555
25.33287433
25.33287433
25.33287433
25.33287433
25.33287433
25.28877049
25.28877049
25.28877049
25.28877049
25.28877049
25.32312316
25.32312316
25.32312316
25.32312316
25.32312316
25.31237769
25.31237769
25.31237769
25.31237769
25.31237769
25.307458
25.307458
25.307458
25.307458
25.307458
25.32506909
25.32506909
25.32506909
25.32506909
25.32506909
25.30501509
25.30501509
25.30501509
25.30501509
25.30501509
25.30430028
25.30430028
25.30430028
25.30430028
25.30430028
25.29324476
25.29324476
25.29324476
25.29324476
25.29324476
25.29361022
25.29361022
25.29361022
25.29361022
25.29361022
25.32760376
25.32760376
25.32760376
25.32760376
25.32760376
25.34155346
25.34155346
25.34155346
25.34155346
25.34155346
25.30284326
25.30284326
25.30284326
25.30284326
25.30284326
25.3327307
25.3327307
25.3327307
25.3327307
25.3327307
25.31380685
25.31380685
25.31380685
25.31380685
25.31380685
25.29859265
25.29859265
25.29859265
25.29859265
25.29859265
25.31580466
25.31580466
25.31580466
25.31580466
25.31580466
25.31703111
25.31703111
25.31703111
25.31703111
25.31703111
25.3147751
25.3147751
25.3147751
25.3147751
25.3147751
25.28141109
25.28141109
25.28141109
25.28141109
25.28141109
25.29165342
25.29165342
25.29165342
25.29165342
25.29165342
25.28952273
25.28952273
25.28952273
25.28952273
25.28952273
25.33419389
25.33419389
25.33419389
25.33419389
25.33419389
25.30437797
25.30437797
25.30437797
25.30437797
25.30437797
25.31167179
25.31167179
25.31167179
25.31167179
25.31167179
25.32806246
25.32806246
25.32806246
25.32806246
25.32806246
25.33728429
25.33728429
25.33728429
25.33728429
25.33728429
25.32282291
25.32282291
25.32282291
25.32282291
25.32282291
25.33265313
25.33265313
25.33265313
25.33265313
25.33265313
25.32934031
25.32934031
25.32934031
25.32934031
25.32934031
25.34359056
25.34359056
25.34359056
25.34359056
25.34359056
25.31357143
25.31357143
25.31357143
25.31357143
25.31357143
25.32989445
25.32989445
25.32989445
25.32989445
25.32989445
25.30844676
25.30844676
25.30844676
25.30844676
25.30844676
25.35749198
25.35749198
25.35749198
25.35749198
25.35749198
25.32595568
25.32595568
25.32595568
25.32595568
25.32595568
25.31115368
25.31115368
25.31115368
25.31115368
25.31115368
25.32301988
25.32301988
25.32301988
25.32301988
25.32301988
25.30674803
25.30674803
25.30674803
25.30674803
25.30674803
25.27717098
25.27717098
25.27717098
25.27717098
25.27717098
25.35407566
25.35407566
25.35407566
25.35407566
25.35407566
25.32559755
25.32559755
25.32559755
25.32559755
25.32559755
25.30654092
25.30654092
25.30654092
25.30654092
25.30654092
25.29606275
25.29606275
25.29606275
25.29606275
25.29606275
25.32546834
25.32546834
25.32546834
25.32546834
25.32546834
25.331592
25.331592
25.331592
25.331592
25.331592
25.32542617
25.32542617
25.32542617
25.32542617
25.32542617
25.30729726
25.30729726
25.30729726
25.30729726
25.30729726
25.31737221
25.31737221
25.31737221
25.31737221
25.31737221
25.32650615
25.32650615
25.32650615
25.32650615
25.32650615
25.32236654
25.32236654
25.32236654
25.32236654
25.32236654
25.33376852
25.33376852
25.33376852
25.33376852
25.33376852
25.3194702
25.3194702
25.3194702
25.3194702
25.3194702
25.32976472
25.32976472
25.32976472
25.32976472
25.32976472
25.33490107
25.33490107
25.33490107
25.33490107
25.33490107
25.31181457
25.31181457
25.31181457
25.31181457
25.31181457
25.3445018
25.3445018
25.3445018
25.3445018
25.3445018
25.35073265
25.35073265
25.35073265
25.35073265
25.35073265
25.31417561
25.31417561
25.31417561
25.31417561
25.31417561
25.32379971
25.32379971
25.32379971
25.32379971
25.32379971
25.33989594
25.33989594
25.33989594
25.33989594
25.33989594
25.28712664
25.28712664
25.28712664
25.28712664
25.28712664
25.29500477
25.29500477
25.29500477
25.29500477
25.29500477
25.27885373
25.27885373
25.27885373
25.27885373
25.27885373
25.32402176
25.32402176
25.32402176
25.32402176
25.32402176
25.29225012
25.29225012
25.29225012
25.29225012
25.29225012
25.3179574
25.3179574
25.3179574
25.3179574
25.3179574
25.35061689
25.35061689
25.35061689
25.35061689
25.35061689
25.32314698
25.32314698
25.32314698
25.32314698
25.32314698
25.31854183
25.31854183
25.31854183
25.31854183
25.31854183
25.36565848
25.36565848
25.36565848
25.36565848
25.36565848
25.2979259
25.2979259
25.2979259
25.2979259
25.2979259
25.36247895
25.36247895
25.36247895
25.36247895
25.36247895
25.36003309
25.36003309
25.36003309
25.36003309
25.36003309
25.33058742
25.33058742
25.33058742
25.33058742
25.33058742
25.32253357
25.32253357
25.32253357
25.32253357
25.32253357
25.33223059
25.33223059
25.33223059
25.33223059
25.33223059
25.29401047
25.29401047
25.29401047
25.29401047
25.29401047
25.3000805
25.3000805
25.3000805
25.3000805
25.3000805
25.32602581
25.32602581
25.32602581
25.32602581
25.32602581
25.33717254
25.33717254
25.33717254
25.33717254
25.33717254
25.31468936
25.31468936
25.31468936
25.31468936
25.31468936
25.31484637
25.31484637
25.31484637
25.31484637
25.31484637
25.29975815
25.29975815
25.29975815
25.29975815
25.29975815
25.29630832
25.29630832
25.29630832
25.29630832
25.29630832
25.33345951
25.33345951
25.33345951
25.33345951
25.33345951
25.30925194
25.30925194
25.30925194
25.30925194
25.30925194
25.36620448
25.36620448
25.36620448
25.36620448
25.36620448
25.31932441
25.31932441
25.31932441
25.31932441
25.31932441
25.32638162
25.32638162
25.32638162
25.32638162
25.32638162
25.32388386
25.32388386
25.32388386
25.32388386
25.32388386
25.31316325
25.31316325
25.31316325
25.31316325
25.31316325
25.35340333
25.35340333
25.35340333
25.35340333
25.35340333
25.31569344
25.31569344
25.31569344
25.31569344
25.31569344
25.33625221
25.33625221
25.33625221
25.33625221
25.33625221
25.29792309
25.29792309
25.29792309
25.29792309
25.29792309
25.27462945
25.27462945
25.27462945
25.27462945
25.27462945
25.33874711
25.33874711
25.33874711
25.33874711
25.33874711
25.30881764
25.30881764
25.30881764
25.30881764
25.30881764
25.31552923
25.31552923
25.31552923
25.31552923
25.31552923
25.30739393
25.30739393
25.30739393
25.30739393
25.30739393
25.3132907
25.3132907
25.3132907
25.3132907
25.3132907
25.31388972
25.31388972
25.31388972
25.31388972
25.31388972
25.3451406
25.3451406
25.3451406
25.3451406
25.3451406
25.32467199
25.32467199
25.32467199
25.32467199
25.32467199
25.32559945
25.32559945
25.32559945
25.32559945
25.32559945
25.25713618
25.25713618
25.25713618
25.25713618
25.25713618
25.31836409
25.31836409
25.31836409
25.31836409
25.31836409
25.29490934
25.29490934
25.29490934
25.29490934
25.29490934
25.30669911
25.30669911
25.30669911
25.30669911
25.30669911
25.30342088
25.30342088
25.30342088
25.30342088
25.30342088
25.34373701
25.34373701
25.34373701
25.34373701
25.34373701
25.32711563
25.32711563
25.32711563
25.32711563
25.32711563
25.32674722
25.32674722
25.32674722
25.32674722
25.32674722
25.2891453
25.2891453
25.2891453
25.2891453
25.2891453
25.29712707
25.29712707
25.29712707
25.29712707
25.29712707
25.33340619
25.33340619
25.33340619
25.33340619
25.33340619
25.31395062
25.31395062
25.31395062
25.31395062
25.31395062
25.31997245
25.31997245
25.31997245
25.31997245
25.31997245
25.33699996
25.33699996
25.33699996
25.33699996
25.33699996
25.28013199
25.28013199
25.28013199
25.28013199
25.28013199
25.31595515
25.31595515
25.31595515
25.31595515
25.31595515
25.34219917
25.34219917
25.34219917
25.34219917
25.34219917
25.33698304
25.33698304
25.33698304
25.33698304
25.33698304
25.27854921
25.27854921
25.27854921
25.27854921
25.27854921
25.30881081
25.30881081
25.30881081
25.30881081
25.30881081
25.32316998
25.32316998
25.32316998
25.32316998
25.32316998
25.33140916
25.33140916
25.33140916
25.33140916
25.33140916
25.34151357
25.34151357
25.34151357
25.34151357
25.34151357
25.31616504
25.31616504
25.31616504
25.31616504
25.31616504
25.31503846
25.31503846
25.31503846
25.31503846
25.31503846
25.3495858
25.3495858
25.3495858
25.3495858
25.3495858
25.30709681
25.30709681
25.30709681
25.30709681
25.30709681
25.29958332
25.29958332
25.29958332
25.29958332
25.29958332
25.32112175
25.32112175
25.32112175
25.32112175
25.32112175
25.31897786
25.31897786
25.31897786
25.31897786
25.31897786
25.32799185
25.32799185
25.32799185
25.32799185
25.32799185
25.32680769
25.32680769
25.32680769
25.32680769
25.32680769
25.30970087
25.30970087
25.30970087
25.30970087
25.30970087
25.34133091
25.34133091
25.34133091
25.34133091
25.34133091
25.29783477
25.29783477
25.29783477
25.29783477
25.29783477
25.33406734
25.33406734
25.33406734
25.33406734
25.33406734
25.30326571
25.30326571
25.30326571
25.30326571
25.30326571
25.30951894
25.30951894
25.30951894
25.30951894
25.30951894
25.33430659
25.33430659
25.33430659
25.33430659
25.33430659
25.31840347
25.31840347
25.31840347
25.31840347
25.31840347
25.33406532
25.33406532
25.33406532
25.33406532
25.33406532
25.33507873
25.33507873
25.33507873
25.33507873
25.33507873
25.30799451
25.30799451
25.30799451
25.30799451
25.30799451
25.34591459
25.34591459
25.34591459
25.34591459
25.34591459
25.34096971
25.34096971
25.34096971
25.34096971
25.34096971
25.29805211
25.29805211
25.29805211
25.29805211
25.29805211
25.31047681
25.31047681
25.31047681
25.31047681
25.31047681
25.26235117
25.26235117
25.26235117
25.26235117
25.26235117
25.31227896
25.31227896
25.31227896
25.31227896
25.31227896
25.32066244
25.32066244
25.32066244
25.32066244
25.32066244
25.29197077
25.29197077
25.29197077
25.29197077
25.29197077
25.32741381
25.32741381
25.32741381
25.32741381
25.32741381
25.32656185
25.32656185
25.32656185
25.32656185
25.32656185
25.31672296
25.31672296
25.31672296
25.31672296
25.31672296
25.3307915
25.3307915
25.3307915
25.3307915
25.3307915
25.31161284
25.31161284
25.31161284
25.31161284
25.31161284
25.35063265
25.35063265
25.35063265
25.35063265
25.35063265
25.31870327
25.31870327
25.31870327
25.31870327
25.31870327
25.33323837
25.33323837
25.33323837
25.33323837
25.33323837
25.32048148
25.32048148
25.32048148
25.32048148
25.32048148
25.28862273
25.28862273
25.28862273
25.28862273
25.28862273
25.31321016
25.31321016
25.31321016
25.31321016
25.31321016
25.31249578
25.31249578
25.31249578
25.31249578
25.31249578
25.30886228
25.30886228
25.30886228
25.30886228
25.30886228
25.32312202
25.32312202
25.32312202
25.32312202
25.32312202
25.31602531
25.31602531
25.31602531
25.31602531
25.31602531
25.31821516
25.31821516
25.31821516
25.31821516
25.31821516
25.32580971
25.32580971
25.32580971
25.32580971
25.32580971
25.3252934
25.3252934
25.3252934
25.3252934
25.3252934
25.35882716
25.35882716
25.35882716
25.35882716
25.35882716
25.32515224
25.32515224
25.32515224
25.32515224
25.32515224
25.31923963
25.31923963
25.31923963
25.31923963
25.31923963
25.32776439
25.32776439
25.32776439
25.32776439
25.32776439
25.31607091
25.31607091
25.31607091
25.31607091
25.31607091
25.29231732
25.29231732
25.29231732
25.29231732
25.29231732
25.32934198
25.32934198
25.32934198
25.32934198
25.32934198
25.35130672
25.35130672
25.35130672
25.35130672
25.35130672
25.32419463
25.32419463
25.32419463
25.32419463
25.32419463
25.3080171
25.3080171
25.3080171
25.3080171
25.3080171
25.31710564
25.31710564
25.31710564
25.31710564
25.31710564
25.34853277
25.34853277
25.34853277
25.34853277
25.34853277
25.32590373
25.32590373
25.32590373
25.32590373
25.32590373
25.30464626
25.30464626
25.30464626
25.30464626
25.30464626
25.31348759
25.31348759
25.31348759
25.31348759
25.31348759
25.32627427
25.32627427
25.32627427
25.32627427
25.32627427
25.34176957
25.34176957
25.34176957
25.34176957
25.34176957
25.3677464
25.3677464
25.3677464
25.3677464
25.3677464
25.3107482
25.3107482
25.3107482
25.3107482
25.3107482
25.31867934
25.31867934
25.31867934
25.31867934
25.31867934
25.29523533
25.29523533
25.29523533
25.29523533
25.29523533
25.3402845
25.3402845
25.3402845
25.3402845
25.3402845
25.31258352
25.31258352
25.31258352
25.31258352
25.31258352
25.31509448
25.31509448
25.31509448
25.31509448
25.31509448
25.33969117
25.33969117
25.33969117
25.33969117
25.33969117
25.32452029
25.32452029
25.32452029
25.32452029
25.32452029
25.33566989
25.33566989
25.33566989
25.33566989
25.33566989
25.31452023
25.31452023
25.31452023
25.31452023
25.31452023
25.35489784
25.35489784
25.35489784
25.35489784
25.35489784
25.31971071
25.31971071
25.31971071
25.31971071
25.31971071
25.31390141
25.31390141
25.31390141
25.31390141
25.31390141
25.27251228
25.27251228
25.27251228
25.27251228
25.27251228
25.30640328
25.30640328
25.30640328
25.30640328
25.30640328
25.31441465
25.31441465
25.31441465
25.31441465
25.31441465
25.32038374
25.32038374
25.32038374
25.32038374
25.32038374
25.30920201
25.30920201
25.30920201
25.30920201
25.30920201
25.30561743
25.30561743
25.30561743
25.30561743
25.30561743
25.30686331
25.30686331
25.30686331
25.30686331
25.30686331
25.31582146
25.31582146
25.31582146
25.31582146
25.31582146
25.33420558
25.33420558
25.33420558
25.33420558
25.33420558
25.29362127
25.29362127
25.29362127
25.29362127
25.29362127
25.30811842
25.30811842
25.30811842
25.30811842
25.30811842
25.30443624
25.30443624
25.30443624
25.30443624
25.30443624
25.3063366
25.3063366
25.3063366
25.3063366
25.3063366
25.2738781
25.2738781
25.2738781
25.2738781
25.2738781
25.31219568
25.31219568
25.31219568
25.31219568
25.31219568
25.36249346
25.36249346
25.36249346
25.36249346
25.36249346
25.28340447
25.28340447
25.28340447
25.28340447
25.28340447
25.3119073
25.3119073
25.3119073
25.3119073
25.3119073
25.31448963
25.31448963
25.31448963
25.31448963
25.31448963
25.35184794
25.35184794
25.35184794
25.35184794
25.35184794
25.29735734
25.29735734
25.29735734
25.29735734
25.29735734
25.30208264
25.30208264
25.30208264
25.30208264
25.30208264
25.35129366
25.35129366
25.35129366
25.35129366
25.35129366
25.33499279
25.33499279
25.33499279
25.33499279
25.33499279
25.34239752
25.34239752
25.34239752
25.34239752
25.34239752
25.31324501
25.31324501
25.31324501
25.31324501
25.31324501
25.28946164
25.28946164
25.28946164
25.28946164
25.28946164
25.26110177
25.26110177
25.26110177
25.26110177
25.26110177
25.31662497
25.31662497
25.31662497
25.31662497
25.31662497
25.3111042
25.3111042
25.3111042
25.3111042
25.3111042
25.36850162
25.36850162
25.36850162
25.36850162
25.36850162
25.34860927
25.34860927
25.34860927
25.34860927
25.34860927
25.3258451
25.3258451
25.3258451
25.3258451
25.3258451
25.31233036
25.31233036
25.31233036
25.31233036
25.31233036
25.30701421
25.30701421
25.30701421
25.30701421
25.30701421
25.36596938
25.36596938
25.36596938
25.36596938
25.36596938
25.27716537
25.27716537
25.27716537
25.27716537
25.27716537
25.34365364
25.34365364
25.34365364
25.34365364
25.34365364
25.32030108
25.32030108
25.32030108
25.32030108
25.32030108
25.27465886
25.27465886
25.27465886
25.27465886
25.27465886
25.34184199
25.34184199
25.34184199
25.34184199
25.34184199
25.31701427
25.31701427
25.31701427
25.31701427
25.31701427
25.35730777
25.35730777
25.35730777
25.35730777
25.35730777
25.32686296
25.32686296
25.32686296
25.32686296
25.32686296
25.31328067
25.31328067
25.31328067
25.31328067
25.31328067
25.3416434
25.3416434
25.3416434
25.3416434
25.3416434
25.32059718
25.32059718
25.32059718
25.32059718
25.32059718
25.32268157
25.32268157
25.32268157
25.32268157
25.32268157
25.32353759
25.32353759
25.32353759
25.32353759
25.32353759
25.36970372
25.36970372
25.36970372
25.36970372
25.36970372
25.36830996
25.36830996
25.36830996
25.36830996
25.36830996
25.33161852
25.33161852
25.33161852
25.33161852
25.33161852
25.33145883
25.33145883
25.33145883
25.33145883
25.33145883
25.30394174
25.30394174
25.30394174
25.30394174
25.30394174
25.32648945
25.32648945
25.32648945
25.32648945
25.32648945
25.29982986
25.29982986
25.29982986
25.29982986
25.29982986
25.33109677
25.33109677
25.33109677
25.33109677
25.33109677
25.34598623
25.34598623
25.34598623
25.34598623
25.34598623
25.31869265
25.31869265
25.31869265
25.31869265
25.31869265
25.3825925
25.3825925
25.3825925
25.3825925
25.3825925
25.32233841
25.32233841
25.32233841
25.32233841
25.32233841
25.35472552
25.35472552
25.35472552
25.35472552
25.35472552
25.34058316
25.34058316
25.34058316
25.34058316
25.34058316
25.35496558
25.35496558
25.35496558
25.35496558
25.35496558
25.32479667
25.32479667
25.32479667
25.32479667
25.32479667
25.28600553
25.28600553
25.28600553
25.28600553
25.28600553
25.29527238
25.29527238
25.29527238
25.29527238
25.29527238
25.29615798
25.29615798
25.29615798
25.29615798
25.29615798
25.31943368
25.31943368
25.31943368
25.31943368
25.31943368
25.29407011
25.29407011
25.29407011
25.29407011
25.29407011
25.33472978
25.33472978
25.33472978
25.33472978
25.33472978
25.35323338
25.35323338
25.35323338
25.35323338
25.35323338
25.3116028
25.3116028
25.3116028
25.3116028
25.3116028
25.29130807
25.29130807
25.29130807
25.29130807
25.29130807
25.31043109
25.31043109
25.31043109
25.31043109
25.31043109
25.31635939
25.31635939
25.31635939
25.31635939
25.31635939
25.2963502
25.2963502
25.2963502
25.2963502
25.2963502
25.33417954
25.33417954
25.33417954
25.33417954
25.33417954
25.35741176
25.35741176
25.35741176
25.35741176
25.35741176
25.30283477
25.30283477
25.30283477
25.30283477
25.30283477
25.36109907
25.36109907
25.36109907
25.36109907
25.36109907
25.32191182
25.32191182
25.32191182
25.32191182
25.32191182
25.33794628
25.33794628
25.33794628
25.33794628
25.33794628
25.32124497
25.32124497
25.32124497
25.32124497
25.32124497
25.31091621
25.31091621
25.31091621
25.31091621
25.31091621
25.29853949
25.29853949
25.29853949
25.29853949
25.29853949
25.32373533
25.32373533
25.32373533
25.32373533
25.32373533
25.31786681
25.31786681
25.31786681
25.31786681
25.31786681
25.35555725
25.35555725
25.35555725
25.35555725
25.35555725
25.30901022
25.30901022
25.30901022
25.30901022
25.30901022
25.32912287
25.32912287
25.32912287
25.32912287
25.32912287
25.33266621
25.33266621
25.33266621
25.33266621
25.33266621
25.31456274
25.31456274
25.31456274
25.31456274
25.31456274
25.31624427
25.31624427
25.31624427
25.31624427
25.31624427
25.30238536
25.30238536
25.30238536
25.30238536
25.30238536
25.31052738
25.31052738
25.31052738
25.31052738
25.31052738
25.30095264
25.30095264
25.30095264
25.30095264
25.30095264
25.29809692
25.29809692
25.29809692
25.29809692
25.29809692
25.31588619
25.31588619
25.31588619
25.31588619
25.31588619
25.31218673
25.31218673
25.31218673
25.31218673
25.31218673
25.3209392
25.3209392
25.3209392
25.3209392
25.3209392
25.28130291
25.28130291
25.28130291
25.28130291
25.28130291
25.30763865
25.30763865
25.30763865
25.30763865
25.30763865
25.3168908
25.3168908
25.3168908
25.3168908
25.3168908
25.33024891
25.33024891
25.33024891
25.33024891
25.33024891
25.31615661
25.31615661
25.31615661
25.31615661
25.31615661
25.32506445
25.32506445
25.32506445
25.32506445
25.32506445
25.38686996
25.38686996
25.38686996
25.38686996
25.38686996
25.3239225
25.3239225
25.3239225
25.3239225
25.3239225
25.29500192
25.29500192
25.29500192
25.29500192
25.29500192
25.32147437
25.32147437
25.32147437
25.32147437
25.32147437
25.31178736
25.31178736
25.31178736
25.31178736
25.31178736
25.31539628
25.31539628
25.31539628
25.31539628
25.31539628
25.333499
25.333499
25.333499
25.333499
25.333499
25.32166841
25.32166841
25.32166841
25.32166841
25.32166841
25.32565834
25.32565834
25.32565834
25.32565834
25.32565834
25.30373453
25.30373453
25.30373453
25.30373453
25.30373453
25.33151782
25.33151782
25.33151782
25.33151782
25.33151782
25.32387539
25.32387539
25.32387539
25.32387539
25.32387539
25.30706272
25.30706272
25.30706272
25.30706272
25.30706272
25.3632689
25.3632689
25.3632689
25.3632689
25.3632689
25.35329352
25.35329352
25.35329352
25.35329352
25.35329352
25.33545397
25.33545397
25.33545397
25.33545397
25.33545397
25.3409707
25.3409707
25.3409707
25.3409707
25.3409707
25.3239733
25.3239733
25.3239733
25.3239733
25.3239733
25.31500096
25.31500096
25.31500096
25.31500096
25.31500096
25.30436715
25.30436715
25.30436715
25.30436715
25.30436715
25.30647048
25.30647048
25.30647048
25.30647048
25.30647048
25.31081695
25.31081695
25.31081695
25.31081695
25.31081695
25.30607961
25.30607961
25.30607961
25.30607961
25.30607961
25.32049515
25.32049515
25.32049515
25.32049515
25.32049515
25.32264125
25.32264125
25.32264125
25.32264125
25.32264125
25.30342727
25.30342727
25.30342727
25.30342727
25.30342727
25.29186534
25.29186534
25.29186534
25.29186534
25.29186534
25.30970197
25.30970197
25.30970197
25.30970197
25.30970197
25.32775301
25.32775301
25.32775301
25.32775301
25.32775301
25.29435025
25.29435025
25.29435025
25.29435025
25.29435025
25.3313661
25.3313661
25.3313661
25.3313661
25.3313661
25.32398928
25.32398928
25.32398928
25.32398928
25.32398928
25.31258898
25.31258898
25.31258898
25.31258898
25.31258898
25.30476062
25.30476062
25.30476062
25.30476062
25.30476062
25.30590359
25.30590359
25.30590359
25.30590359
25.30590359
25.2956615
25.2956615
25.2956615
25.2956615
25.2956615
25.28621684
25.28621684
25.28621684
25.28621684
25.28621684
25.32507288
25.32507288
25.32507288
25.32507288
25.32507288
25.32810984
25.32810984
25.32810984
25.32810984
25.32810984
25.31714429
25.31714429
25.31714429
25.31714429
25.31714429
25.26084537
25.26084537
25.26084537
25.26084537
25.26084537
25.29177503
25.29177503
25.29177503
25.29177503
25.29177503
25.28608035
25.28608035
25.28608035
25.28608035
25.28608035
25.29597021
25.29597021
25.29597021
25.29597021
25.29597021
25.31512918
25.31512918
25.31512918
25.31512918
25.31512918
25.32625455
25.32625455
25.32625455
25.32625455
25.32625455
25.32950804
25.32950804
25.32950804
25.32950804
25.32950804
25.35121106
25.35121106
25.35121106
25.35121106
25.35121106
25.32211516
25.32211516
25.32211516
25.32211516
25.32211516
25.31498338
25.31498338
25.31498338
25.31498338
25.31498338
25.34797993
25.34797993
25.34797993
25.34797993
25.34797993
25.31671689
25.31671689
25.31671689
25.31671689
25.31671689
25.36678945
25.36678945
25.36678945
25.36678945
25.36678945
25.30406337
25.30406337
25.30406337
25.30406337
25.30406337
25.35144434
25.35144434
25.35144434
25.35144434
25.35144434
25.30316933
25.30316933
25.30316933
25.30316933
25.30316933
25.32066362
25.32066362
25.32066362
25.32066362
25.32066362
25.29563072
25.29563072
25.29563072
25.29563072
25.29563072
25.33427408
25.33427408
25.33427408
25.33427408
25.33427408
25.32877299
25.32877299
25.32877299
25.32877299
25.32877299
25.32704278
25.32704278
25.32704278
25.32704278
25.32704278
25.31782413
25.31782413
25.31782413
25.31782413
25.31782413
25.37046983
25.37046983
25.37046983
25.37046983
25.37046983
25.32179659
25.32179659
25.32179659
25.32179659
25.32179659
25.32410982
25.32410982
25.32410982
25.32410982
25.32410982
25.29281038
25.29281038
25.29281038
25.29281038
25.29281038
25.32432305
25.32432305
25.32432305
25.32432305
25.32432305
25.32510222
25.32510222
25.32510222
25.32510222
25.32510222
25.31116972
25.31116972
25.31116972
25.31116972
25.31116972
25.31289505
25.31289505
25.31289505
25.31289505
25.31289505
25.35873479
25.35873479
25.35873479
25.35873479
25.35873479
25.30461204
25.30461204
25.30461204
25.30461204
25.30461204
25.34973998
25.34973998
25.34973998
25.34973998
25.34973998
25.30640308
25.30640308
25.30640308
25.30640308
25.30640308
25.29218854
25.29218854
25.29218854
25.29218854
25.29218854
25.38126461
25.38126461
25.38126461
25.38126461
25.38126461
25.30045996
25.30045996
25.30045996
25.30045996
25.30045996
25.32391519
25.32391519
25.32391519
25.32391519
25.32391519
25.3175703
25.3175703
25.3175703
25.3175703
25.3175703
25.32133858
25.32133858
25.32133858
25.32133858
25.32133858
25.32296786
25.32296786
25.32296786
25.32296786
25.32296786
25.31738612
25.31738612
25.31738612
25.31738612
25.31738612
25.30427319
25.30427319
25.30427319
25.30427319
25.30427319
25.31628982
25.31628982
25.31628982
25.31628982
25.31628982
25.29639042
25.29639042
25.29639042
25.29639042
25.29639042
25.33000063
25.33000063
25.33000063
25.33000063
25.33000063
25.31139654
25.31139654
25.31139654
25.31139654
25.31139654
25.30882041
25.30882041
25.30882041
25.30882041
25.30882041
25.32817149
25.32817149
25.32817149
25.32817149
25.32817149
25.31073428
25.31073428
25.31073428
25.31073428
25.31073428
25.36018887
25.36018887
25.36018887
25.36018887
25.36018887
25.37045018
25.37045018
25.37045018
25.37045018
25.37045018
25.32289383
25.32289383
25.32289383
25.32289383
25.32289383
25.32673505
25.32673505
25.32673505
25.32673505
25.32673505
25.31378134
25.31378134
25.31378134
25.31378134
25.31378134
25.32704909
25.32704909
25.32704909
25.32704909
25.32704909
25.30387821
25.30387821
25.30387821
25.30387821
25.30387821
25.36680047
25.36680047
25.36680047
25.36680047
25.36680047
25.3727818
25.3727818
25.3727818
25.3727818
25.3727818
25.32409628
25.32409628
25.32409628
25.32409628
25.32409628
25.29417387
25.29417387
25.29417387
25.29417387
25.29417387
25.31878858
25.31878858
25.31878858
25.31878858
25.31878858
25.33394848
25.33394848
25.33394848
25.33394848
25.33394848
25.32237086
25.32237086
25.32237086
25.32237086
25.32237086
25.34524594
25.34524594
25.34524594
25.34524594
25.34524594
25.32374772
25.32374772
25.32374772
25.32374772
25.32374772
25.28157797
25.28157797
25.28157797
25.28157797
25.28157797
25.34938606
25.34938606
25.34938606
25.34938606
25.34938606
25.32774444
25.32774444
25.32774444
25.32774444
25.32774444
25.30623963
25.30623963
25.30623963
25.30623963
25.30623963
25.31302923
25.31302923
25.31302923
25.31302923
25.31302923
25.31851456
25.31851456
25.31851456
25.31851456
25.31851456
25.32297774
25.32297774
25.32297774
25.32297774
25.32297774
25.3249812
25.3249812
25.3249812
25.3249812
25.3249812
25.32625729
25.32625729
25.32625729
25.32625729
25.32625729
25.29649197
25.29649197
25.29649197
25.29649197
25.29649197
25.30048591
25.30048591
25.30048591
25.30048591
25.30048591
25.33520519
25.33520519
25.33520519
25.33520519
25.33520519
25.30570004
25.30570004
25.30570004
25.30570004
25.30570004
25.30396973
25.30396973
25.30396973
25.30396973
25.30396973
25.30985755
25.30985755
25.30985755
25.30985755
25.30985755
25.31887126
25.31887126
25.31887126
25.31887126
25.31887126
25.3160779
25.3160779
25.3160779
25.3160779
25.3160779
25.34391598
25.34391598
25.34391598
25.34391598
25.34391598
25.31099726
25.31099726
25.31099726
25.31099726
25.31099726
25.33282348
25.33282348
25.33282348
25.33282348
25.33282348
25.30779799
25.30779799
25.30779799
25.30779799
25.30779799
25.33830194
25.33830194
25.33830194
25.33830194
25.33830194
25.3463121
25.3463121
25.3463121
25.3463121
25.3463121
25.28564876
25.28564876
25.28564876
25.28564876
25.28564876
25.32365388
25.32365388
25.32365388
25.32365388
25.32365388
25.32534089
25.32534089
25.32534089
25.32534089
25.32534089
25.29221351
25.29221351
25.29221351
25.29221351
25.29221351
25.30472314
25.30472314
25.30472314
25.30472314
25.30472314
25.32802812
25.32802812
25.32802812
25.32802812
25.32802812
25.32507678
25.32507678
25.32507678
25.32507678
25.32507678
25.30492295
25.30492295
25.30492295
25.30492295
25.30492295
25.32146157
25.32146157
25.32146157
25.32146157
25.32146157
25.32415958
25.32415958
25.32415958
25.32415958
25.32415958
25.33998805
25.33998805
25.33998805
25.33998805
25.33998805
25.29422723
25.29422723
25.29422723
25.29422723
25.29422723
25.27873369
25.27873369
25.27873369
25.27873369
25.27873369
25.34474208
25.34474208
25.34474208
25.34474208
25.34474208
25.33009176
25.33009176
25.33009176
25.33009176
25.33009176
25.28459108
25.28459108
25.28459108
25.28459108
25.28459108
25.3085231
25.3085231
25.3085231
25.3085231
25.3085231
25.3217234
25.3217234
25.3217234
25.3217234
25.3217234
25.27961759
25.27961759
25.27961759
25.27961759
25.27961759
25.3502079
25.3502079
25.3502079
25.3502079
25.3502079
25.29316241
25.29316241
25.29316241
25.29316241
25.29316241
25.34675487
25.34675487
25.34675487
25.34675487
25.34675487
25.33795538
25.33795538
25.33795538
25.33795538
25.33795538
25.2887364
25.2887364
25.2887364
25.2887364
25.2887364
25.31789885
25.31789885
25.31789885
25.31789885
25.31789885
25.32586402
25.32586402
25.32586402
25.32586402
25.32586402
25.37021109
25.37021109
25.37021109
25.37021109
25.37021109
25.36737019
25.36737019
25.36737019
25.36737019
25.36737019
25.349327
25.349327
25.349327
25.349327
25.349327
25.33247643
25.33247643
25.33247643
25.33247643
25.33247643
25.33065852
25.33065852
25.33065852
25.33065852
25.33065852
25.29605567
25.29605567
25.29605567
25.29605567
25.29605567
25.32695331
25.32695331
25.32695331
25.32695331
25.32695331
25.30625048
25.30625048
25.30625048
25.30625048
25.30625048
25.28004614
25.28004614
25.28004614
25.28004614
25.28004614
25.32977405
25.32977405
25.32977405
25.32977405
25.32977405
25.29538426
25.29538426
25.29538426
25.29538426
25.29538426
25.34442874
25.34442874
25.34442874
25.34442874
25.34442874
25.33272807
25.33272807
25.33272807
25.33272807
25.33272807
25.27725672
25.27725672
25.27725672
25.27725672
25.27725672
25.30624306
25.30624306
25.30624306
25.30624306
25.30624306
25.33600197
25.33600197
25.33600197
25.33600197
25.33600197
25.3217344
25.3217344
25.3217344
25.3217344
25.3217344
25.29968279
25.29968279
25.29968279
25.29968279
25.29968279
25.35915959
25.35915959
25.35915959
25.35915959
25.35915959
25.32382415
25.32382415
25.32382415
25.32382415
25.32382415
25.26506716
25.26506716
25.26506716
25.26506716
25.26506716
25.34335215
25.34335215
25.34335215
25.34335215
25.34335215
25.30503901
25.30503901
25.30503901
25.30503901
25.30503901
25.31945593
25.31945593
25.31945593
25.31945593
25.31945593
25.36529342
25.36529342
25.36529342
25.36529342
25.36529342
25.29314498
25.29314498
25.29314498
25.29314498
25.29314498
25.33550741
25.33550741
25.33550741
25.33550741
25.33550741
25.32909823
25.32909823
25.32909823
25.32909823
25.32909823
25.29490917
25.29490917
25.29490917
25.29490917
25.29490917
25.32392301
25.32392301
25.32392301
25.32392301
25.32392301
25.33843178
25.33843178
25.33843178
25.33843178
25.33843178
25.31649368
25.31649368
25.31649368
25.31649368
25.31649368
25.29836733
25.29836733
25.29836733
25.29836733
25.29836733
25.34317252
25.34317252
25.34317252
25.34317252
25.34317252
25.31648537
25.31648537
25.31648537
25.31648537
25.31648537
25.28688791
25.28688791
25.28688791
25.28688791
25.28688791
25.31983291
25.31983291
25.31983291
25.31983291
25.31983291
25.33936377
25.33936377
25.33936377
25.33936377
25.33936377
25.32386984
25.32386984
25.32386984
25.32386984
25.32386984
25.32144053
25.32144053
25.32144053
25.32144053
25.32144053
25.33297789
25.33297789
25.33297789
25.33297789
25.33297789
25.30428845
25.30428845
25.30428845
25.30428845
25.30428845
25.31479521
25.31479521
25.31479521
25.31479521
25.31479521
25.32783979
25.32783979
25.32783979
25.32783979
25.32783979
25.3078796
25.3078796
25.3078796
25.3078796
25.3078796
25.3121488
25.3121488
25.3121488
25.3121488
25.3121488
25.34338809
25.34338809
25.34338809
25.34338809
25.34338809
25.32434386
25.32434386
25.32434386
25.32434386
25.32434386
25.29754917
25.29754917
25.29754917
25.29754917
25.29754917
25.33349611
25.33349611
25.33349611
25.33349611
25.33349611
25.33153797
25.33153797
25.33153797
25.33153797
25.33153797
25.36549559
25.36549559
25.36549559
25.36549559
25.36549559
25.28581384
25.28581384
25.28581384
25.28581384
25.28581384
25.32497699
25.32497699
25.32497699
25.32497699
25.32497699
25.29725739
25.29725739
25.29725739
25.29725739
25.29725739
25.30061043
25.30061043
25.30061043
25.30061043
25.30061043
25.36806028
25.36806028
25.36806028
25.36806028
25.36806028
25.32189089
25.32189089
25.32189089
25.32189089
25.32189089
25.30194763
25.30194763
25.30194763
25.30194763
25.30194763
25.32770233
25.32770233
25.32770233
25.32770233
25.32770233
25.34072117
25.34072117
25.34072117
25.34072117
25.34072117
25.34169646
25.34169646
25.34169646
25.34169646
25.34169646
25.3324914
25.3324914
25.3324914
25.3324914
25.3324914
25.30178492
25.30178492
25.30178492
25.30178492
25.30178492
25.32353789
25.32353789
25.32353789
25.32353789
25.32353789
25.35321688
25.35321688
25.35321688
25.35321688
25.35321688
25.31906058
25.31906058
25.31906058
25.31906058
25.31906058
25.30897
25.30897
25.30897
25.30897
25.30897
25.33505717
25.33505717
25.33505717
25.33505717
25.33505717
25.33112008
25.33112008
25.33112008
25.33112008
25.33112008
25.32470784
25.32470784
25.32470784
25.32470784
25.32470784
25.37522004
25.37522004
25.37522004
25.37522004
25.37522004
25.30058029
25.30058029
25.30058029
25.30058029
25.30058029
25.32473009
25.32473009
25.32473009
25.32473009
25.32473009
25.32669352
25.32669352
25.32669352
25.32669352
25.32669352
25.32237506
25.32237506
25.32237506
25.32237506
25.32237506
25.32188743
25.32188743
25.32188743
25.32188743
25.32188743
25.32888182
25.32888182
25.32888182
25.32888182
25.32888182
25.30195631
25.30195631
25.30195631
25.30195631
25.30195631
25.32553647
25.32553647
25.32553647
25.32553647
25.32553647
25.27653471
25.27653471
25.27653471
25.27653471
25.27653471
25.29828855
25.29828855
25.29828855
25.29828855
25.29828855
25.32326638
25.32326638
25.32326638
25.32326638
25.32326638
25.31676796
25.31676796
25.31676796
25.31676796
25.31676796
25.31934228
25.31934228
25.31934228
25.31934228
25.31934228
25.32343792
25.32343792
25.32343792
25.32343792
25.32343792
25.33769008
25.33769008
25.33769008
25.33769008
25.33769008
25.33040773
25.33040773
25.33040773
25.33040773
25.33040773
25.34976317
25.34976317
25.34976317
25.34976317
25.34976317
25.31750255
25.31750255
25.31750255
25.31750255
25.31750255
25.32742316
25.32742316
25.32742316
25.32742316
25.32742316
25.28964491
25.28964491
25.28964491
25.28964491
25.28964491
25.30219639
25.30219639
25.30219639
25.30219639
25.30219639
25.33360412
25.33360412
25.33360412
25.33360412
25.33360412
25.34642472
25.34642472
25.34642472
25.34642472
25.34642472
25.33308295
25.33308295
25.33308295
25.33308295
25.33308295
25.28333465
25.28333465
25.28333465
25.28333465
25.28333465
25.33351788
25.33351788
25.33351788
25.33351788
25.33351788
25.32455541
25.32455541
25.32455541
25.32455541
25.32455541
25.31710757
25.31710757
25.31710757
25.31710757
25.31710757
25.3114963
25.3114963
25.3114963
25.3114963
25.3114963
25.3437371
25.3437371
25.3437371
25.3437371
25.3437371
25.32110136
25.32110136
25.32110136
25.32110136
25.32110136
25.28796644
25.28796644
25.28796644
25.28796644
25.28796644
25.27998117
25.27998117
25.27998117
25.27998117
25.27998117
25.32988129
25.32988129
25.32988129
25.32988129
25.32988129
25.33072778
25.33072778
25.33072778
25.33072778
25.33072778
25.33195162
25.33195162
25.33195162
25.33195162
25.33195162
25.31650398
25.31650398
25.31650398
25.31650398
25.31650398
25.32558875
25.32558875
25.32558875
25.32558875
25.32558875
25.36779543
25.36779543
25.36779543
25.36779543
25.36779543
25.30290719
25.30290719
25.30290719
25.30290719
25.30290719
25.31502815
25.31502815
25.31502815
25.31502815
25.31502815
25.3281444
25.3281444
25.3281444
25.3281444
25.3281444
25.32286109
25.32286109
25.32286109
25.32286109
25.32286109
25.35004671
25.35004671
25.35004671
25.35004671
25.35004671
25.32338265
25.32338265
25.32338265
25.32338265
25.32338265
25.31454877
25.31454877
25.31454877
25.31454877
25.31454877
25.29042914
25.29042914
25.29042914
25.29042914
25.29042914
25.32540965
25.32540965
25.32540965
25.32540965
25.32540965
25.31718943
25.31718943
25.31718943
25.31718943
25.31718943
25.31335426
25.31335426
25.31335426
25.31335426
25.31335426
25.31468808
25.31468808
25.31468808
25.31468808
25.31468808
25.32077048
25.32077048
25.32077048
25.32077048
25.32077048
25.34164708
25.34164708
25.34164708
25.34164708
25.34164708
25.31482409
25.31482409
25.31482409
25.31482409
25.31482409
25.32338479
25.32338479
25.32338479
25.32338479
25.32338479
25.32590424
25.32590424
25.32590424
25.32590424
25.32590424
25.324627
25.324627
25.324627
25.324627
25.324627
25.30869558
25.30869558
25.30869558
25.30869558
25.30869558
25.32272066
25.32272066
25.32272066
25.32272066
25.32272066
25.34581851
25.34581851
25.34581851
25.34581851
25.34581851
25.32534934
25.32534934
25.32534934
25.32534934
25.32534934
25.32797935
25.32797935
25.32797935
25.32797935
25.32797935
25.31312973
25.31312973
25.31312973
25.31312973
25.31312973
25.30970854
25.30970854
25.30970854
25.30970854
25.30970854
25.31817216
25.31817216
25.31817216
25.31817216
25.31817216
25.32269082
25.32269082
25.32269082
25.32269082
25.32269082
25.31641872
25.31641872
25.31641872
25.31641872
25.31641872
25.31944666
25.31944666
25.31944666
25.31944666
25.31944666
25.33289302
25.33289302
25.33289302
25.33289302
25.33289302
25.25993881
25.25993881
25.25993881
25.25993881
25.25993881
25.36394264
25.36394264
25.36394264
25.36394264
25.36394264
25.30062512
25.30062512
25.30062512
25.30062512
25.30062512
25.31446226
25.31446226
25.31446226
25.31446226
25.31446226
25.29592369
25.29592369
25.29592369
25.29592369
25.29592369
25.29106526
25.29106526
25.29106526
25.29106526
25.29106526
25.30129851
25.30129851
25.30129851
25.30129851
25.30129851
25.32044376
25.32044376
25.32044376
25.32044376
25.32044376
25.3238142
25.3238142
25.3238142
25.3238142
25.3238142
25.30664782
25.30664782
25.30664782
25.30664782
25.30664782
25.32098126
25.32098126
25.32098126
25.32098126
25.32098126
25.31666565
25.31666565
25.31666565
25.31666565
25.31666565
25.33356209
25.33356209
25.33356209
25.33356209
25.33356209
25.37747862
25.37747862
25.37747862
25.37747862
25.37747862
25.32321519
25.32321519
25.32321519
25.32321519
25.32321519
25.28845011
25.28845011
25.28845011
25.28845011
25.28845011
25.30680385
25.30680385
25.30680385
25.30680385
25.30680385
25.28529792
25.28529792
25.28529792
25.28529792
25.28529792
25.2770976
25.2770976
25.2770976
25.2770976
25.2770976
25.31356543
25.31356543
25.31356543
25.31356543
25.31356543
25.32008061
25.32008061
25.32008061
25.32008061
25.32008061
25.32158344
25.32158344
25.32158344
25.32158344
25.32158344
25.31459872
25.31459872
25.31459872
25.31459872
25.31459872
25.36722112
25.36722112
25.36722112
25.36722112
25.36722112
25.32621027
25.32621027
25.32621027
25.32621027
25.32621027
25.31747224
25.31747224
25.31747224
25.31747224
25.31747224
25.29137887
25.29137887
25.29137887
25.29137887
25.29137887
25.29945433
25.29945433
25.29945433
25.29945433
25.29945433
25.33770537
25.33770537
25.33770537
25.33770537
25.33770537
25.32102117
25.32102117
25.32102117
25.32102117
25.32102117
25.30508503
25.30508503
25.30508503
25.30508503
25.30508503
25.31958281
25.31958281
25.31958281
25.31958281
25.31958281
25.3116479
25.3116479
25.3116479
25.3116479
25.3116479
25.32507599
25.32507599
25.32507599
25.32507599
25.32507599
25.31018142
25.31018142
25.31018142
25.31018142
25.31018142
25.30674733
25.30674733
25.30674733
25.30674733
25.30674733
25.33658006
25.33658006
25.33658006
25.33658006
25.33658006
25.32396749
25.32396749
25.32396749
25.32396749
25.32396749
25.31795119
25.31795119
25.31795119
25.31795119
25.31795119
25.31377736
25.31377736
25.31377736
25.31377736
25.31377736
25.33467839
25.33467839
25.33467839
25.33467839
25.33467839
25.31444222
25.31444222
25.31444222
25.31444222
25.31444222
25.31632329
25.31632329
25.31632329
25.31632329
25.31632329
25.35151048
25.35151048
25.35151048
25.35151048
25.35151048
25.34286514
25.34286514
25.34286514
25.34286514
25.34286514
25.32189189
25.32189189
25.32189189
25.32189189
25.32189189
25.31489341
25.31489341
25.31489341
25.31489341
25.31489341
25.29464065
25.29464065
25.29464065
25.29464065
25.29464065
25.35520647
25.35520647
25.35520647
25.35520647
25.35520647
25.32561776
25.32561776
25.32561776
25.32561776
25.32561776
25.30989187
25.30989187
25.30989187
25.30989187
25.30989187
25.31702191
25.31702191
25.31702191
25.31702191
25.31702191
25.30432317
25.30432317
25.30432317
25.30432317
25.30432317
25.32904336
25.32904336
25.32904336
25.32904336
25.32904336
25.3605191
25.3605191
25.3605191
25.3605191
25.3605191
25.34437253
25.34437253
25.34437253
25.34437253
25.34437253
25.35860467
25.35860467
25.35860467
25.35860467
25.35860467
25.31610583
25.31610583
25.31610583
25.31610583
25.31610583
25.30688758
25.30688758
25.30688758
25.30688758
25.30688758
25.32474668
25.32474668
25.32474668
25.32474668
25.32474668
25.2651589
25.2651589
25.2651589
25.2651589
25.2651589
25.31603881
25.31603881
25.31603881
25.31603881
25.31603881
25.33656492
25.33656492
25.33656492
25.33656492
25.33656492
25.34121317
25.34121317
25.34121317
25.34121317
25.34121317
25.35173791
25.35173791
25.35173791
25.35173791
25.35173791
25.31971124
25.31971124
25.31971124
25.31971124
25.31971124
25.33919863
25.33919863
25.33919863
25.33919863
25.33919863
25.34040722
25.34040722
25.34040722
25.34040722
25.34040722
25.33527815
25.33527815
25.33527815
25.33527815
25.33527815
25.31779101
25.31779101
25.31779101
25.31779101
25.31779101
25.33486692
25.33486692
25.33486692
25.33486692
25.33486692
25.30035634
25.30035634
25.30035634
25.30035634
25.30035634
25.31397437
25.31397437
25.31397437
25.31397437
25.31397437
25.36729153
25.36729153
25.36729153
25.36729153
25.36729153
25.31829428
25.31829428
25.31829428
25.31829428
25.31829428
25.28885978
25.28885978
25.28885978
25.28885978
25.28885978
25.31711767
25.31711767
25.31711767
25.31711767
25.31711767
25.36706816
25.36706816
25.36706816
25.36706816
25.36706816
25.29726599
25.29726599
25.29726599
25.29726599
25.29726599
25.3144307
25.3144307
25.3144307
25.3144307
25.3144307
25.33738188
25.33738188
25.33738188
25.33738188
25.33738188
25.30538984
25.30538984
25.30538984
25.30538984
25.30538984
25.30477666
25.30477666
25.30477666
25.30477666
25.30477666
25.30588849
25.30588849
25.30588849
25.30588849
25.30588849
25.32875136
25.32875136
25.32875136
25.32875136
25.32875136
25.27976282
25.27976282
25.27976282
25.27976282
25.27976282
25.33496179
25.33496179
25.33496179
25.33496179
25.33496179
25.33249418
25.33249418
25.33249418
25.33249418
25.33249418
25.32346095
25.32346095
25.32346095
25.32346095
25.32346095
25.28415578
25.28415578
25.28415578
25.28415578
25.28415578
25.34583096
25.34583096
25.34583096
25.34583096
25.34583096
25.30248664
25.30248664
25.30248664
25.30248664
25.30248664
25.33054962
25.33054962
25.33054962
25.33054962
25.33054962
25.3371644
25.3371644
25.3371644
25.3371644
25.3371644
25.31762571
25.31762571
25.31762571
25.31762571
25.31762571
25.29242799
25.29242799
25.29242799
25.29242799
25.29242799
25.3269649
25.3269649
25.3269649
25.3269649
25.3269649
25.34537774
25.34537774
25.34537774
25.34537774
25.34537774
25.32961603
25.32961603
25.32961603
25.32961603
25.32961603
25.32379881
25.32379881
25.32379881
25.32379881
25.32379881
25.28807388
25.28807388
25.28807388
25.28807388
25.28807388
25.37918482
25.37918482
25.37918482
25.37918482
25.37918482
25.2861326
25.2861326
25.2861326
25.2861326
25.2861326
25.30778692
25.30778692
25.30778692
25.30778692
25.30778692
25.32066872
25.32066872
25.32066872
25.32066872
25.32066872
25.33168657
25.33168657
25.33168657
25.33168657
25.33168657
25.29731345
25.29731345
25.29731345
25.29731345
25.29731345
25.3230986
25.3230986
25.3230986
25.3230986
25.3230986
25.31709545
25.31709545
25.31709545
25.31709545
25.31709545
25.35648708
25.35648708
25.35648708
25.35648708
25.35648708
25.29825462
25.29825462
25.29825462
25.29825462
25.29825462
25.32030651
25.32030651
25.32030651
25.32030651
25.32030651
25.32215471
25.32215471
25.32215471
25.32215471
25.32215471
25.30610039
25.30610039
25.30610039
25.30610039
25.30610039
25.30289905
25.30289905
25.30289905
25.30289905
25.30289905
25.35229393
25.35229393
25.35229393
25.35229393
25.35229393
25.33573646
25.33573646
25.33573646
25.33573646
25.33573646
25.32114425
25.32114425
25.32114425
25.32114425
25.32114425
25.32458393
25.32458393
25.32458393
25.32458393
25.32458393
25.35311934
25.35311934
25.35311934
25.35311934
25.35311934
25.32381105
25.32381105
25.32381105
25.32381105
25.32381105
25.31516453
25.31516453
25.31516453
25.31516453
25.31516453
25.32351748
25.32351748
25.32351748
25.32351748
25.32351748
25.2831898
25.2831898
25.2831898
25.2831898
25.2831898
25.32595565
25.32595565
25.32595565
25.32595565
25.32595565
25.2936181
25.2936181
25.2936181
25.2936181
25.2936181
25.31545432
25.31545432
25.31545432
25.31545432
25.31545432
25.32977486
25.32977486
25.32977486
25.32977486
25.32977486
25.29006321
25.29006321
25.29006321
25.29006321
25.29006321
25.32405468
25.32405468
25.32405468
25.32405468
25.32405468
25.27269182
25.27269182
25.27269182
25.27269182
25.27269182
25.33450127
25.33450127
25.33450127
25.33450127
25.33450127
25.32306411
25.32306411
25.32306411
25.32306411
25.32306411
25.32047946
25.32047946
25.32047946
25.32047946
25.32047946
25.32388186
25.32388186
25.32388186
25.32388186
25.32388186
25.30934826
25.30934826
25.30934826
25.30934826
25.30934826
25.32322673
25.32322673
25.32322673
25.32322673
25.32322673
25.29509384
25.29509384
25.29509384
25.29509384
25.29509384
25.37055429
25.37055429
25.37055429
25.37055429
25.37055429
25.32328244
25.32328244
25.32328244
25.32328244
25.32328244
25.30871763
25.30871763
25.30871763
25.30871763
25.30871763
25.30332601
25.30332601
25.30332601
25.30332601
25.30332601
25.30643698
25.30643698
25.30643698
25.30643698
25.30643698
25.30439556
25.30439556
25.30439556
25.30439556
25.30439556
25.31327482
25.31327482
25.31327482
25.31327482
25.31327482
25.27517767
25.27517767
25.27517767
25.27517767
25.27517767
25.34726434
25.34726434
25.34726434
25.34726434
25.34726434
25.35127617
25.35127617
25.35127617
25.35127617
25.35127617
25.32139091
25.32139091
25.32139091
25.32139091
25.32139091
25.30580287
25.30580287
25.30580287
25.30580287
25.30580287
25.32235396
25.32235396
25.32235396
25.32235396
25.32235396
25.35822435
25.35822435
25.35822435
25.35822435
25.35822435
25.28472442
25.28472442
25.28472442
25.28472442
25.28472442
25.31546441
25.31546441
25.31546441
25.31546441
25.31546441
25.3459349
25.3459349
25.3459349
25.3459349
25.3459349
25.32153642
25.32153642
25.32153642
25.32153642
25.32153642
25.33792644
25.33792644
25.33792644
25.33792644
25.33792644
25.33028822
25.33028822
25.33028822
25.33028822
25.33028822
25.30485667
25.30485667
25.30485667
25.30485667
25.30485667
25.34148896
25.34148896
25.34148896
25.34148896
25.34148896
25.31752441
25.31752441
25.31752441
25.31752441
25.31752441
25.36547353
25.36547353
25.36547353
25.36547353
25.36547353
25.32257973
25.32257973
25.32257973
25.32257973
25.32257973
25.31996403
25.31996403
25.31996403
25.31996403
25.31996403
25.31516233
25.31516233
25.31516233
25.31516233
25.31516233
25.27508775
25.27508775
25.27508775
25.27508775
25.27508775
25.34907296
25.34907296
25.34907296
25.34907296
25.34907296
25.33560221
25.33560221
25.33560221
25.33560221
25.33560221
25.34986179
25.34986179
25.34986179
25.34986179
25.34986179
25.34188991
25.34188991
25.34188991
25.34188991
25.34188991
25.3223066
25.3223066
25.3223066
25.3223066
25.3223066
25.30514091
25.30514091
25.30514091
25.30514091
25.30514091
25.32210678
25.32210678
25.32210678
25.32210678
25.32210678
25.31427039
25.31427039
25.31427039
25.31427039
25.31427039
25.34536656
25.34536656
25.34536656
25.34536656
25.34536656
25.35702852
25.35702852
25.35702852
25.35702852
25.35702852
25.32042369
25.32042369
25.32042369
25.32042369
25.32042369
25.31572981
25.31572981
25.31572981
25.31572981
25.31572981
25.32191112
25.32191112
25.32191112
25.32191112
25.32191112
25.29208841
25.29208841
25.29208841
25.29208841
25.29208841
25.31366873
25.31366873
25.31366873
25.31366873
25.31366873
25.29765657
25.29765657
25.29765657
25.29765657
25.29765657
25.33844527
25.33844527
25.33844527
25.33844527
25.33844527
25.32859486
25.32859486
25.32859486
25.32859486
25.32859486
25.34012221
25.34012221
25.34012221
25.34012221
25.34012221
25.32388052
25.32388052
25.32388052
25.32388052
25.32388052
25.30767982
25.30767982
25.30767982
25.30767982
25.30767982
25.36715333
25.36715333
25.36715333
25.36715333
25.36715333
25.30842351
25.30842351
25.30842351
25.30842351
25.30842351
25.2951983
25.2951983
25.2951983
25.2951983
25.2951983
25.31924292
25.31924292
25.31924292
25.31924292
25.31924292
25.3182146
25.3182146
25.3182146
25.3182146
25.3182146
25.29634947
25.29634947
25.29634947
25.29634947
25.29634947
25.32741864
25.32741864
25.32741864
25.32741864
25.32741864
25.3198276
25.3198276
25.3198276
25.3198276
25.3198276
25.32798867
25.32798867
25.32798867
25.32798867
25.32798867
25.32496148
25.32496148
25.32496148
25.32496148
25.32496148
25.32380543
25.32380543
25.32380543
25.32380543
25.32380543
25.32499226
25.32499226
25.32499226
25.32499226
25.32499226
25.29655017
25.29655017
25.29655017
25.29655017
25.29655017
25.3036342
25.3036342
25.3036342
25.3036342
25.3036342
25.30492138
25.30492138
25.30492138
25.30492138
25.30492138
25.313814
25.313814
25.313814
25.313814
25.313814
25.29576456
25.29576456
25.29576456
25.29576456
25.29576456
25.31993336
25.31993336
25.31993336
25.31993336
25.31993336
25.35850324
25.35850324
25.35850324
25.35850324
25.35850324
25.31453981
25.31453981
25.31453981
25.31453981
25.31453981
25.35614653
25.35614653
25.35614653
25.35614653
25.35614653
25.30696293
25.30696293
25.30696293
25.30696293
25.30696293
25.35243348
25.35243348
25.35243348
25.35243348
25.35243348
25.33143599
25.33143599
25.33143599
25.33143599
25.33143599
25.32373499
25.32373499
25.32373499
25.32373499
25.32373499
25.32048638
25.32048638
25.32048638
25.32048638
25.32048638
25.33304016
25.33304016
25.33304016
25.33304016
25.33304016
25.33882856
25.33882856
25.33882856
25.33882856
25.33882856
25.31326719
25.31326719
25.31326719
25.31326719
25.31326719
25.31492167
25.31492167
25.31492167
25.31492167
25.31492167
25.2997293
25.2997293
25.2997293
25.2997293
25.2997293
25.30496511
25.30496511
25.30496511
25.30496511
25.30496511
25.32895326
25.32895326
25.32895326
25.32895326
25.32895326
25.32771287
25.32771287
25.32771287
25.32771287
25.32771287
25.32353673
25.32353673
25.32353673
25.32353673
25.32353673
25.34374292
25.34374292
25.34374292
25.34374292
25.34374292
25.33463198
25.33463198
25.33463198
25.33463198
25.33463198
25.32956719
25.32956719
25.32956719
25.32956719
25.32956719
25.3661293
25.3661293
25.3661293
25.3661293
25.3661293
25.32771192
25.32771192
25.32771192
25.32771192
25.32771192
25.31488391
25.31488391
25.31488391
25.31488391
25.31488391
25.31309913
25.31309913
25.31309913
25.31309913
25.31309913
25.38133387
25.38133387
25.38133387
25.38133387
25.38133387
25.31496572
25.31496572
25.31496572
25.31496572
25.31496572
25.29737787
25.29737787
25.29737787
25.29737787
25.29737787
25.32194895
25.32194895
25.32194895
25.32194895
25.32194895
25.31807554
25.31807554
25.31807554
25.31807554
25.31807554
25.361599
25.361599
25.361599
25.361599
25.361599
25.3124745
25.3124745
25.3124745
25.3124745
25.3124745
25.33424584
25.33424584
25.33424584
25.33424584
25.33424584
25.29025055
25.29025055
25.29025055
25.29025055
25.29025055
25.31808603
25.31808603
25.31808603
25.31808603
25.31808603
25.30372875
25.30372875
25.30372875
25.30372875
25.30372875
25.29714578
25.29714578
25.29714578
25.29714578
25.29714578
25.32248786
25.32248786
25.32248786
25.32248786
25.32248786
25.30650406
25.30650406
25.30650406
25.30650406
25.30650406
25.3236851
25.3236851
25.3236851
25.3236851
25.3236851
25.29443678
25.29443678
25.29443678
25.29443678
25.29443678
25.25907725
25.25907725
25.25907725
25.25907725
25.25907725
25.36225806
25.36225806
25.36225806
25.36225806
25.36225806
25.3159154
25.3159154
25.3159154
25.3159154
25.3159154
25.33548251
25.33548251
25.33548251
25.33548251
25.33548251
25.33570157
25.33570157
25.33570157
25.33570157
25.33570157
25.33386689
25.33386689
25.33386689
25.33386689
25.33386689
25.31795858
25.31795858
25.31795858
25.31795858
25.31795858
25.29329224
25.29329224
25.29329224
25.29329224
25.29329224
25.28144614
25.28144614
25.28144614
25.28144614
25.28144614
25.28259734
25.28259734
25.28259734
25.28259734
25.28259734
25.32434666
25.32434666
25.32434666
25.32434666
25.32434666
25.34627653
25.34627653
25.34627653
25.34627653
25.34627653
25.34745437
25.34745437
25.34745437
25.34745437
25.34745437
25.31757879
25.31757879
25.31757879
25.31757879
25.31757879
25.29246115
25.29246115
25.29246115
25.29246115
25.29246115
25.32472298
25.32472298
25.32472298
25.32472298
25.32472298
25.37947173
25.37947173
25.37947173
25.37947173
25.37947173
25.28153771
25.28153771
25.28153771
25.28153771
25.28153771
25.324993
25.324993
25.324993
25.324993
25.324993
25.35761579
25.35761579
25.35761579
25.35761579
25.35761579
25.32185891
25.32185891
25.32185891
25.32185891
25.32185891
25.33544135
25.33544135
25.33544135
25.33544135
25.33544135
25.33284883
25.33284883
25.33284883
25.33284883
25.33284883
25.34551577
25.34551577
25.34551577
25.34551577
25.34551577
25.34655113
25.34655113
25.34655113
25.34655113
25.34655113
25.30248699
25.30248699
25.30248699
25.30248699
25.30248699
25.33989801
25.33989801
25.33989801
25.33989801
25.33989801
25.31005425
25.31005425
25.31005425
25.31005425
25.31005425
25.32547009
25.32547009
25.32547009
25.32547009
25.32547009
25.33454282
25.33454282
25.33454282
25.33454282
25.33454282
25.29549558
25.29549558
25.29549558
25.29549558
25.29549558
25.3203115
25.3203115
25.3203115
25.3203115
25.3203115
25.31786784
25.31786784
25.31786784
25.31786784
25.31786784
25.3156919
25.3156919
25.3156919
25.3156919
25.3156919
25.31836516
25.31836516
25.31836516
25.31836516
25.31836516
25.3212917
25.3212917
25.3212917
25.3212917
25.3212917
25.28442993
25.28442993
25.28442993
25.28442993
25.28442993
25.3291691
25.3291691
25.3291691
25.3291691
25.3291691
25.31313257
25.31313257
25.31313257
25.31313257
25.31313257
25.30793772
25.30793772
25.30793772
25.30793772
25.30793772
25.34408315
25.34408315
25.34408315
25.34408315
25.34408315
25.3032854
25.3032854
25.3032854
25.3032854
25.3032854
25.31357314
25.31357314
25.31357314
25.31357314
25.31357314
25.32940841
25.32940841
25.32940841
25.32940841
25.32940841
25.35482885
25.35482885
25.35482885
25.35482885
25.35482885
25.35344393
25.35344393
25.35344393
25.35344393
25.35344393
25.31699497
25.31699497
25.31699497
25.31699497
25.31699497
25.30061011
25.30061011
25.30061011
25.30061011
25.30061011
25.37435621
25.37435621
25.37435621
25.37435621
25.37435621
25.30406783
25.30406783
25.30406783
25.30406783
25.30406783
25.33011212
25.33011212
25.33011212
25.33011212
25.33011212
25.32455624
25.32455624
25.32455624
25.32455624
25.32455624
25.32819842
25.32819842
25.32819842
25.32819842
25.32819842
25.31116793
25.31116793
25.31116793
25.31116793
25.31116793
25.30023881
25.30023881
25.30023881
25.30023881
25.30023881
25.3123008
25.3123008
25.3123008
25.3123008
25.3123008
25.34082976
25.34082976
25.34082976
25.34082976
25.34082976
25.33254168
25.33254168
25.33254168
25.33254168
25.33254168
25.31992045
25.31992045
25.31992045
25.31992045
25.31992045
25.32807976
25.32807976
25.32807976
25.32807976
25.32807976
25.3318113
25.3318113
25.3318113
25.3318113
25.3318113
25.29075548
25.29075548
25.29075548
25.29075548
25.29075548
25.32543292
25.32543292
25.32543292
25.32543292
25.32543292
25.31755279
25.31755279
25.31755279
25.31755279
25.31755279
25.34160862
25.34160862
25.34160862
25.34160862
25.34160862
25.31292913
25.31292913
25.31292913
25.31292913
25.31292913
25.31744423
25.31744423
25.31744423
25.31744423
25.31744423
25.31876688
25.31876688
25.31876688
25.31876688
25.31876688
25.36194668
25.36194668
25.36194668
25.36194668
25.36194668
25.30519584
25.30519584
25.30519584
25.30519584
25.30519584
25.27841463
25.27841463
25.27841463
25.27841463
25.27841463
25.30457508
25.30457508
25.30457508
25.30457508
25.30457508
25.3154779
25.3154779
25.3154779
25.3154779
25.3154779
25.3305164
25.3305164
25.3305164
25.3305164
25.3305164
25.31634819
25.31634819
25.31634819
25.31634819
25.31634819
25.28129531
25.28129531
25.28129531
25.28129531
25.28129531
25.29385416
25.29385416
25.29385416
25.29385416
25.29385416
25.33533509
25.33533509
25.33533509
25.33533509
25.33533509
25.31492786
25.31492786
25.31492786
25.31492786
25.31492786
25.28394756
25.28394756
25.28394756
25.28394756
25.28394756
25.28604541
25.28604541
25.28604541
25.28604541
25.28604541
25.37779454
25.37779454
25.37779454
25.37779454
25.37779454
25.31294302
25.31294302
25.31294302
25.31294302
25.31294302
25.3255162
25.3255162
25.3255162
25.3255162
25.3255162
25.27935393
25.27935393
25.27935393
25.27935393
25.27935393
25.29956511
25.29956511
25.29956511
25.29956511
25.29956511
25.28900736
25.28900736
25.28900736
25.28900736
25.28900736
25.3234846
25.3234846
25.3234846
25.3234846
25.3234846
25.33066216
25.33066216
25.33066216
25.33066216
25.33066216
25.35580942
25.35580942
25.35580942
25.35580942
25.35580942
25.3170585
25.3170585
25.3170585
25.3170585
25.3170585
25.29244656
25.29244656
25.29244656
25.29244656
25.29244656
25.35184128
25.35184128
25.35184128
25.35184128
25.35184128
25.34981558
25.34981558
25.34981558
25.34981558
25.34981558
25.32428896
25.32428896
25.32428896
25.32428896
25.32428896
25.30626483
25.30626483
25.30626483
25.30626483
25.30626483
25.32467525
25.32467525
25.32467525
25.32467525
25.32467525
25.29527625
25.29527625
25.29527625
25.29527625
25.29527625
25.31780013
25.31780013
25.31780013
25.31780013
25.31780013
25.34178892
25.34178892
25.34178892
25.34178892
25.34178892
25.33584903
25.33584903
25.33584903
25.33584903
25.33584903
25.32774243
25.32774243
25.32774243
25.32774243
25.32774243
25.32068235
25.32068235
25.32068235
25.32068235
25.32068235
25.30709668
25.30709668
25.30709668
25.30709668
25.30709668
25.36883435
25.36883435
25.36883435
25.36883435
25.36883435
25.3390162
25.3390162
25.3390162
25.3390162
25.3390162
25.30260605
25.30260605
25.30260605
25.30260605
25.30260605
25.32949196
25.32949196
25.32949196
25.32949196
25.32949196
25.35500562
25.35500562
25.35500562
25.35500562
25.35500562
25.30767126
25.30767126
25.30767126
25.30767126
25.30767126
25.31438538
25.31438538
25.31438538
25.31438538
25.31438538
25.31404912
25.31404912
25.31404912
25.31404912
25.31404912
25.33573678
25.33573678
25.33573678
25.33573678
25.33573678
25.31452776
25.31452776
25.31452776
25.31452776
25.31452776
25.30466904
25.30466904
25.30466904
25.30466904
25.30466904
25.36761497
25.36761497
25.36761497
25.36761497
25.36761497
25.3305913
25.3305913
25.3305913
25.3305913
25.3305913
25.33511588
25.33511588
25.33511588
25.33511588
25.33511588
25.28645546
25.28645546
25.28645546
25.28645546
25.28645546
25.36361984
25.36361984
25.36361984
25.36361984
25.36361984
25.30934508
25.30934508
25.30934508
25.30934508
25.30934508
25.32225327
25.32225327
25.32225327
25.32225327
25.32225327
25.30147193
25.30147193
25.30147193
25.30147193
25.30147193
25.33475212
25.33475212
25.33475212
25.33475212
25.33475212
25.31765005
25.31765005
25.31765005
25.31765005
25.31765005
25.33705063
25.33705063
25.33705063
25.33705063
25.33705063
25.34039579
25.34039579
25.34039579
25.34039579
25.34039579
25.29446238
25.29446238
25.29446238
25.29446238
25.29446238
25.33378759
25.33378759
25.33378759
25.33378759
25.33378759
25.34611802
25.34611802
25.34611802
25.34611802
25.34611802
25.29376464
25.29376464
25.29376464
25.29376464
25.29376464
25.33847048
25.33847048
25.33847048
25.33847048
25.33847048
25.3378025
25.3378025
25.3378025
25.3378025
25.3378025
25.3353743
25.3353743
25.3353743
25.3353743
25.3353743
25.31369154
25.31369154
25.31369154
25.31369154
25.31369154
25.29251815
25.29251815
25.29251815
25.29251815
25.29251815
25.32984298
25.32984298
25.32984298
25.32984298
25.32984298
25.32605791
25.32605791
25.32605791
25.32605791
25.32605791
25.31687043
25.31687043
25.31687043
25.31687043
25.31687043
25.25295409
25.25295409
25.25295409
25.25295409
25.25295409
25.31601073
25.31601073
25.31601073
25.31601073
25.31601073
25.32273646
25.32273646
25.32273646
25.32273646
25.32273646
25.28195839
25.28195839
25.28195839
25.28195839
25.28195839
25.29014532
25.29014532
25.29014532
25.29014532
25.29014532
25.36354536
25.36354536
25.36354536
25.36354536
25.36354536
25.37046568
25.37046568
25.37046568
25.37046568
25.37046568
25.31890959
25.31890959
25.31890959
25.31890959
25.31890959
25.34088256
25.34088256
25.34088256
25.34088256
25.34088256
25.31058744
25.31058744
25.31058744
25.31058744
25.31058744
25.35980013
25.35980013
25.35980013
25.35980013
25.35980013
25.3144293
25.3144293
25.3144293
25.3144293
25.3144293
25.30584664
25.30584664
25.30584664
25.30584664
25.30584664
25.35483455
25.35483455
25.35483455
25.35483455
25.35483455
25.30938099
25.30938099
25.30938099
25.30938099
25.30938099
25.32520027
25.32520027
25.32520027
25.32520027
25.32520027
25.35481458
25.35481458
25.35481458
25.35481458
25.35481458
25.29640086
25.29640086
25.29640086
25.29640086
25.29640086
25.36741877
25.36741877
25.36741877
25.36741877
25.36741877
25.3467677
25.3467677
25.3467677
25.3467677
25.3467677
25.33623888
25.33623888
25.33623888
25.33623888
25.33623888
25.30708711
25.30708711
25.30708711
25.30708711
25.30708711
25.34758469
25.34758469
25.34758469
25.34758469
25.34758469
25.3397339
25.3397339
25.3397339
25.3397339
25.3397339
25.30784946
25.30784946
25.30784946
25.30784946
25.30784946
25.28107787
25.28107787
25.28107787
25.28107787
25.28107787
25.35750389
25.35750389
25.35750389
25.35750389
25.35750389
25.31830579
25.31830579
25.31830579
25.31830579
25.31830579
25.36789438
25.36789438
25.36789438
25.36789438
25.36789438
25.29463945
25.29463945
25.29463945
25.29463945
25.29463945
25.34339846
25.34339846
25.34339846
25.34339846
25.34339846
25.32849152
25.32849152
25.32849152
25.32849152
25.32849152
25.38466097
25.38466097
25.38466097
25.38466097
25.38466097
25.29102789
25.29102789
25.29102789
25.29102789
25.29102789
25.33118446
25.33118446
25.33118446
25.33118446
25.33118446
25.32000703
25.32000703
25.32000703
25.32000703
25.32000703
25.36435313
25.36435313
25.36435313
25.36435313
25.36435313
25.32734353
25.32734353
25.32734353
25.32734353
25.32734353
25.31733297
25.31733297
25.31733297
25.31733297
25.31733297
25.33047515
25.33047515
25.33047515
25.33047515
25.33047515
25.29072518
25.29072518
25.29072518
25.29072518
25.29072518
25.32976405
25.32976405
25.32976405
25.32976405
25.32976405
25.33712163
25.33712163
25.33712163
25.33712163
25.33712163
25.28752605
25.28752605
25.28752605
25.28752605
25.28752605
25.31333981
25.31333981
25.31333981
25.31333981
25.31333981
25.30790896
25.30790896
25.30790896
25.30790896
25.30790896
25.32323847
25.32323847
25.32323847
25.32323847
25.32323847
25.31990653
25.31990653
25.31990653
25.31990653
25.31990653
25.32463267
25.32463267
25.32463267
25.32463267
25.32463267
25.28459984
25.28459984
25.28459984
25.28459984
25.28459984
25.34010404
25.34010404
25.34010404
25.34010404
25.34010404
25.34134609
25.34134609
25.34134609
25.34134609
25.34134609
25.28891843
25.28891843
25.28891843
25.28891843
25.28891843
25.35231247
25.35231247
25.35231247
25.35231247
25.35231247
25.32115633
25.32115633
25.32115633
25.32115633
25.32115633
25.32122703
25.32122703
25.32122703
25.32122703
25.32122703
25.35051689
25.35051689
25.35051689
25.35051689
25.35051689
25.31701077
25.31701077
25.31701077
25.31701077
25.31701077
25.28291761
25.28291761
25.28291761
25.28291761
25.28291761
25.3189617
25.3189617
25.3189617
25.3189617
25.3189617
25.32427251
25.32427251
25.32427251
25.32427251
25.32427251
25.38592189
25.38592189
25.38592189
25.38592189
25.38592189
25.34393047
25.34393047
25.34393047
25.34393047
25.34393047
25.30603063
25.30603063
25.30603063
25.30603063
25.30603063
25.35182029
25.35182029
25.35182029
25.35182029
25.35182029
25.29939042
25.29939042
25.29939042
25.29939042
25.29939042
25.36497171
25.36497171
25.36497171
25.36497171
25.36497171
25.27530304
25.27530304
25.27530304
25.27530304
25.27530304
25.28610821
25.28610821
25.28610821
25.28610821
25.28610821
25.30774922
25.30774922
25.30774922
25.30774922
25.30774922
25.30797757
25.30797757
25.30797757
25.30797757
25.30797757
25.29041584
25.29041584
25.29041584
25.29041584
25.29041584
25.26728254
25.26728254
25.26728254
25.26728254
25.26728254
25.30571141
25.30571141
25.30571141
25.30571141
25.30571141
25.27971161
25.27971161
25.27971161
25.27971161
25.27971161
25.2957773
25.2957773
25.2957773
25.2957773
25.2957773
25.2744326
25.2744326
25.2744326
25.2744326
25.2744326
25.35947938
25.35947938
25.35947938
25.35947938
25.35947938
25.36015136
25.36015136
25.36015136
25.36015136
25.36015136
25.28633996
25.28633996
25.28633996
25.28633996
25.28633996
25.31194917
25.31194917
25.31194917
25.31194917
25.31194917
25.31660672
25.31660672
25.31660672
25.31660672
25.31660672
25.32566778
25.32566778
25.32566778
25.32566778
25.32566778
25.32031446
25.32031446
25.32031446
25.32031446
25.32031446
25.31607687
25.31607687
25.31607687
25.31607687
25.31607687
25.3048108
25.3048108
25.3048108
25.3048108
25.3048108
25.3277674
25.3277674
25.3277674
25.3277674
25.3277674
25.30898254
25.30898254
25.30898254
25.30898254
25.30898254
25.30985348
25.30985348
25.30985348
25.30985348
25.30985348
25.30662743
25.30662743
25.30662743
25.30662743
25.30662743
25.29358493
25.29358493
25.29358493
25.29358493
25.29358493
25.31491901
25.31491901
25.31491901
25.31491901
25.31491901
25.30752119
25.30752119
25.30752119
25.30752119
25.30752119
25.32891505
25.32891505
25.32891505
25.32891505
25.32891505
25.29406856
25.29406856
25.29406856
25.29406856
25.29406856
25.32961756
25.32961756
25.32961756
25.32961756
25.32961756
25.306871
25.306871
25.306871
25.306871
25.306871
25.32037606
25.32037606
25.32037606
25.32037606
25.32037606
25.3343465
25.3343465
25.3343465
25.3343465
25.3343465
25.29405873
25.29405873
25.29405873
25.29405873
25.29405873
25.294475
25.294475
25.294475
25.294475
25.294475
25.29630579
25.29630579
25.29630579
25.29630579
25.29630579
25.36251998
25.36251998
25.36251998
25.36251998
25.36251998
25.31398845
25.31398845
25.31398845
25.31398845
25.31398845
25.28868891
25.28868891
25.28868891
25.28868891
25.28868891
25.2965957
25.2965957
25.2965957
25.2965957
25.2965957
25.32375092
25.32375092
25.32375092
25.32375092
25.32375092
25.29584868
25.29584868
25.29584868
25.29584868
25.29584868
25.31505101
25.31505101
25.31505101
25.31505101
25.31505101
25.32051542
25.32051542
25.32051542
25.32051542
25.32051542
25.34216357
25.34216357
25.34216357
25.34216357
25.34216357
25.30434206
25.30434206
25.30434206
25.30434206
25.30434206
25.32551146
25.32551146
25.32551146
25.32551146
25.32551146
25.32475737
25.32475737
25.32475737
25.32475737
25.32475737
25.33414885
25.33414885
25.33414885
25.33414885
25.33414885
25.3732928
25.3732928
25.3732928
25.3732928
25.3732928
25.2927704
25.2927704
25.2927704
25.2927704
25.2927704
25.3305381
25.3305381
25.3305381
25.3305381
25.3305381
25.3585424
25.3585424
25.3585424
25.3585424
25.3585424
25.29958201
25.29958201
25.29958201
25.29958201
25.29958201
25.30442914
25.30442914
25.30442914
25.30442914
25.30442914
25.33572313
25.33572313
25.33572313
25.33572313
25.33572313
25.31429916
25.31429916
25.31429916
25.31429916
25.31429916
25.29668266
25.29668266
25.29668266
25.29668266
25.29668266
25.32342721
25.32342721
25.32342721
25.32342721
25.32342721
25.31169506
25.31169506
25.31169506
25.31169506
25.31169506
25.36447586
25.36447586
25.36447586
25.36447586
25.36447586
25.3280647
25.3280647
25.3280647
25.3280647
25.3280647
25.30492978
25.30492978
25.30492978
25.30492978
25.30492978
25.32174263
25.32174263
25.32174263
25.32174263
25.32174263
25.30381248
25.30381248
25.30381248
25.30381248
25.30381248
25.31337423
25.31337423
25.31337423
25.31337423
25.31337423
25.33426389
25.33426389
25.33426389
25.33426389
25.33426389
25.29427719
25.29427719
25.29427719
25.29427719
25.29427719
25.32273578
25.32273578
25.32273578
25.32273578
25.32273578
25.32830949
25.32830949
25.32830949
25.32830949
25.32830949
25.32840381
25.32840381
25.32840381
25.32840381
25.32840381
25.3481577
25.3481577
25.3481577
25.3481577
25.3481577
25.34032632
25.34032632
25.34032632
25.34032632
25.34032632
25.34995345
25.34995345
25.34995345
25.34995345
25.34995345
25.32320668
25.32320668
25.32320668
25.32320668
25.32320668
25.30519735
25.30519735
25.30519735
25.30519735
25.30519735
25.33099187
25.33099187
25.33099187
25.33099187
25.33099187
25.31295231
25.31295231
25.31295231
25.31295231
25.31295231
25.30930244
25.30930244
25.30930244
25.30930244
25.30930244
25.31997815
25.31997815
25.31997815
25.31997815
25.31997815
25.30843753
25.30843753
25.30843753
25.30843753
25.30843753
25.31580283
25.31580283
25.31580283
25.31580283
25.31580283
25.32832089
25.32832089
25.32832089
25.32832089
25.32832089
25.33366566
25.33366566
25.33366566
25.33366566
25.33366566
25.32067971
25.32067971
25.32067971
25.32067971
25.32067971
25.3064428
25.3064428
25.3064428
25.3064428
25.3064428
25.2985265
25.2985265
25.2985265
25.2985265
25.2985265
25.28425849
25.28425849
25.28425849
25.28425849
25.28425849
25.32593734
25.32593734
25.32593734
25.32593734
25.32593734
25.32185346
25.32185346
25.32185346
25.32185346
25.32185346
25.29272261
25.29272261
25.29272261
25.29272261
25.29272261
25.35094473
25.35094473
25.35094473
25.35094473
25.35094473
25.29678419
25.29678419
25.29678419
25.29678419
25.29678419
25.30485828
25.30485828
25.30485828
25.30485828
25.30485828
25.33343178
25.33343178
25.33343178
25.33343178
25.33343178
25.32342191
25.32342191
25.32342191
25.32342191
25.32342191
25.32646466
25.32646466
25.32646466
25.32646466
25.32646466
25.33624973
25.33624973
25.33624973
25.33624973
25.33624973
25.33349848
25.33349848
25.33349848
25.33349848
25.33349848
25.32336473
25.32336473
25.32336473
25.32336473
25.32336473
25.29706511
25.29706511
25.29706511
25.29706511
25.29706511
25.3166633
25.3166633
25.3166633
25.3166633
25.3166633
25.33302968
25.33302968
25.33302968
25.33302968
25.33302968
25.32918529
25.32918529
25.32918529
25.32918529
25.32918529
25.31781117
25.31781117
25.31781117
25.31781117
25.31781117
25.29157106
25.29157106
25.29157106
25.29157106
25.29157106
25.32126193
25.32126193
25.32126193
25.32126193
25.32126193
25.29728848
25.29728848
25.29728848
25.29728848
25.29728848
25.30841225
25.30841225
25.30841225
25.30841225
25.30841225
25.2907921
25.2907921
25.2907921
25.2907921
25.2907921
25.29510657
25.29510657
25.29510657
25.29510657
25.29510657
25.31684706
25.31684706
25.31684706
25.31684706
25.31684706
25.33306297
25.33306297
25.33306297
25.33306297
25.33306297
25.29049533
25.29049533
25.29049533
25.29049533
25.29049533
25.297986
25.297986
25.297986
25.297986
25.297986
25.32234622
25.32234622
25.32234622
25.32234622
25.32234622
25.32875248
25.32875248
25.32875248
25.32875248
25.32875248
25.34921741
25.34921741
25.34921741
25.34921741
25.34921741
25.3474416
25.3474416
25.3474416
25.3474416
25.3474416
25.31344681
25.31344681
25.31344681
25.31344681
25.31344681
25.33614826
25.33614826
25.33614826
25.33614826
25.33614826
25.34090779
25.34090779
25.34090779
25.34090779
25.34090779
25.33506708
25.33506708
25.33506708
25.33506708
25.33506708
25.32912276
25.32912276
25.32912276
25.32912276
25.32912276
25.32310956
25.32310956
25.32310956
25.32310956
25.32310956
25.31563493
25.31563493
25.31563493
25.31563493
25.31563493
25.27530322
25.27530322
25.27530322
25.27530322
25.27530322
25.34069708
25.34069708
25.34069708
25.34069708
25.34069708
25.36499478
25.36499478
25.36499478
25.36499478
25.36499478
25.32065613
25.32065613
25.32065613
25.32065613
25.32065613
25.3219486
25.3219486
25.3219486
25.3219486
25.3219486
25.32351392
25.32351392
25.32351392
25.32351392
25.32351392
25.36730903
25.36730903
25.36730903
25.36730903
25.36730903
25.35378442
25.35378442
25.35378442
25.35378442
25.35378442
25.30584985
25.30584985
25.30584985
25.30584985
25.30584985
25.31490273
25.31490273
25.31490273
25.31490273
25.31490273
25.35210436
25.35210436
25.35210436
25.35210436
25.35210436
25.32299086
25.32299086
25.32299086
25.32299086
25.32299086
25.31660512
25.31660512
25.31660512
25.31660512
25.31660512
25.33491341
25.33491341
25.33491341
25.33491341
25.33491341
25.36065127
25.36065127
25.36065127
25.36065127
25.36065127
25.3189594
25.3189594
25.3189594
25.3189594
25.3189594
25.30881199
25.30881199
25.30881199
25.30881199
25.30881199
25.29841645
25.29841645
25.29841645
25.29841645
25.29841645
25.35533731
25.35533731
25.35533731
25.35533731
25.35533731
25.30519682
25.30519682
25.30519682
25.30519682
25.30519682
25.27879739
25.27879739
25.27879739
25.27879739
25.27879739
25.32286454
25.32286454
25.32286454
25.32286454
25.32286454
25.29729867
25.29729867
25.29729867
25.29729867
25.29729867
25.34657145
25.34657145
25.34657145
25.34657145
25.34657145
25.34437937
25.34437937
25.34437937
25.34437937
25.34437937
25.29932089
25.29932089
25.29932089
25.29932089
25.29932089
25.32834013
25.32834013
25.32834013
25.32834013
25.32834013
25.32126629
25.32126629
25.32126629
25.32126629
25.32126629
25.32758853
25.32758853
25.32758853
25.32758853
25.32758853
25.30911263
25.30911263
25.30911263
25.30911263
25.30911263
25.31955043
25.31955043
25.31955043
25.31955043
25.31955043
25.35188088
25.35188088
25.35188088
25.35188088
25.35188088
25.31661733
25.31661733
25.31661733
25.31661733
25.31661733
25.34323307
25.34323307
25.34323307
25.34323307
25.34323307
25.28232845
25.28232845
25.28232845
25.28232845
25.28232845
25.31880249
25.31880249
25.31880249
25.31880249
25.31880249
25.3493838
25.3493838
25.3493838
25.3493838
25.3493838
25.32810235
25.32810235
25.32810235
25.32810235
25.32810235
25.31180368
25.31180368
25.31180368
25.31180368
25.31180368
25.32130775
25.32130775
25.32130775
25.32130775
25.32130775
25.30744467
25.30744467
25.30744467
25.30744467
25.30744467
25.31538807
25.31538807
25.31538807
25.31538807
25.31538807
25.31530781
25.31530781
25.31530781
25.31530781
25.31530781
25.31677343
25.31677343
25.31677343
25.31677343
25.31677343
25.33279456
25.33279456
25.33279456
25.33279456
25.33279456
25.32020274
25.32020274
25.32020274
25.32020274
25.32020274
25.29204005
25.29204005
25.29204005
25.29204005
25.29204005
25.29485463
25.29485463
25.29485463
25.29485463
25.29485463
25.31717523
25.31717523
25.31717523
25.31717523
25.31717523
25.33740574
25.33740574
25.33740574
25.33740574
25.33740574
25.34724747
25.34724747
25.34724747
25.34724747
25.34724747
25.3329783
25.3329783
25.3329783
25.3329783
25.3329783
25.35834331
25.35834331
25.35834331
25.35834331
25.35834331
25.33293173
25.33293173
25.33293173
25.33293173
25.33293173
25.36222837
25.36222837
25.36222837
25.36222837
25.36222837
25.37628097
25.37628097
25.37628097
25.37628097
25.37628097
25.38137492
25.38137492
25.38137492
25.38137492
25.38137492
25.31669036
25.31669036
25.31669036
25.31669036
25.31669036
25.31270694
25.31270694
25.31270694
25.31270694
25.31270694
25.31830316
25.31830316
25.31830316
25.31830316
25.31830316
25.32360864
25.32360864
25.32360864
25.32360864
25.32360864
25.32958073
25.32958073
25.32958073
25.32958073
25.32958073
25.29450943
25.29450943
25.29450943
25.29450943
25.29450943
25.30310018
25.30310018
25.30310018
25.30310018
25.30310018
25.29666056
25.29666056
25.29666056
25.29666056
25.29666056
25.30485497
25.30485497
25.30485497
25.30485497
25.30485497
25.2860981
25.2860981
25.2860981
25.2860981
25.2860981
25.31411038
25.31411038
25.31411038
25.31411038
25.31411038
25.30656824
25.30656824
25.30656824
25.30656824
25.30656824
25.30583105
25.30583105
25.30583105
25.30583105
25.30583105
25.34562456
25.34562456
25.34562456
25.34562456
25.34562456
25.27375302
25.27375302
25.27375302
25.27375302
25.27375302
25.32738191
25.32738191
25.32738191
25.32738191
25.32738191
25.31276021
25.31276021
25.31276021
25.31276021
25.31276021
25.31467122
25.31467122
25.31467122
25.31467122
25.31467122
25.34402399
25.34402399
25.34402399
25.34402399
25.34402399
25.30481546
25.30481546
25.30481546
25.30481546
25.30481546
25.29257432
25.29257432
25.29257432
25.29257432
25.29257432
25.32387723
25.32387723
25.32387723
25.32387723
25.32387723
25.34597014
25.34597014
25.34597014
25.34597014
25.34597014
25.36075804
25.36075804
25.36075804
25.36075804
25.36075804
25.30576122
25.30576122
25.30576122
25.30576122
25.30576122
25.36700291
25.36700291
25.36700291
25.36700291
25.36700291
25.31417902
25.31417902
25.31417902
25.31417902
25.31417902
25.2973032
25.2973032
25.2973032
25.2973032
25.2973032
25.31808778
25.31808778
25.31808778
25.31808778
25.31808778
25.31143115
25.31143115
25.31143115
25.31143115
25.31143115
25.29736013
25.29736013
25.29736013
25.29736013
25.29736013
25.27917793
25.27917793
25.27917793
25.27917793
25.27917793
25.32935586
25.32935586
25.32935586
25.32935586
25.32935586
25.30919275
25.30919275
25.30919275
25.30919275
25.30919275
25.29026279
25.29026279
25.29026279
25.29026279
25.29026279
25.29476615
25.29476615
25.29476615
25.29476615
25.29476615
25.29168182
25.29168182
25.29168182
25.29168182
25.29168182
25.28374053
25.28374053
25.28374053
25.28374053
25.28374053
25.30382412
25.30382412
25.30382412
25.30382412
25.30382412
25.2919719
25.2919719
25.2919719
25.2919719
25.2919719
25.35952229
25.35952229
25.35952229
25.35952229
25.35952229
25.31414383
25.31414383
25.31414383
25.31414383
25.31414383
25.3177608
25.3177608
25.3177608
25.3177608
25.3177608
25.37043401
25.37043401
25.37043401
25.37043401
25.37043401
25.35295904
25.35295904
25.35295904
25.35295904
25.35295904
25.27059947
25.27059947
25.27059947
25.27059947
25.27059947
25.34089703
25.34089703
25.34089703
25.34089703
25.34089703
25.35039022
25.35039022
25.35039022
25.35039022
25.35039022
25.33006055
25.33006055
25.33006055
25.33006055
25.33006055
25.32096197
25.32096197
25.32096197
25.32096197
25.32096197
25.30653059
25.30653059
25.30653059
25.30653059
25.30653059
25.30407613
25.30407613
25.30407613
25.30407613
25.30407613
25.32745408
25.32745408
25.32745408
25.32745408
25.32745408
25.34958587
25.34958587
25.34958587
25.34958587
25.34958587
25.34098697
25.34098697
25.34098697
25.34098697
25.34098697
25.31735214
25.31735214
25.31735214
25.31735214
25.31735214
25.36282866
25.36282866
25.36282866
25.36282866
25.36282866
25.32480727
25.32480727
25.32480727
25.32480727
25.32480727
25.31743468
25.31743468
25.31743468
25.31743468
25.31743468
25.30543291
25.30543291
25.30543291
25.30543291
25.30543291
25.31910746
25.31910746
25.31910746
25.31910746
25.31910746
25.36796778
25.36796778
25.36796778
25.36796778
25.36796778
25.36951456
25.36951456
25.36951456
25.36951456
25.36951456
25.29841451
25.29841451
25.29841451
25.29841451
25.29841451
25.35822967
25.35822967
25.35822967
25.35822967
25.35822967
25.33436002
25.33436002
25.33436002
25.33436002
25.33436002
25.33691813
25.33691813
25.33691813
25.33691813
25.33691813
25.30506557
25.30506557
25.30506557
25.30506557
25.30506557
25.35181773
25.35181773
25.35181773
25.35181773
25.35181773
25.29028843
25.29028843
25.29028843
25.29028843
25.29028843
25.32126407
25.32126407
25.32126407
25.32126407
25.32126407
25.32136411
25.32136411
25.32136411
25.32136411
25.32136411
25.31669695
25.31669695
25.31669695
25.31669695
25.31669695
25.32896123
25.32896123
25.32896123
25.32896123
25.32896123
25.310788
25.310788
25.310788
25.310788
25.310788
25.35479266
25.35479266
25.35479266
25.35479266
25.35479266
25.31487681
25.31487681
25.31487681
25.31487681
25.31487681
25.32336356
25.32336356
25.32336356
25.32336356
25.32336356
25.34278167
25.34278167
25.34278167
25.34278167
25.34278167
25.35389584
25.35389584
25.35389584
25.35389584
25.35389584
25.26208069
25.26208069
25.26208069
25.26208069
25.26208069
25.33782759
25.33782759
25.33782759
25.33782759
25.33782759
25.31334231
25.31334231
25.31334231
25.31334231
25.31334231
25.3107243
25.3107243
25.3107243
25.3107243
25.3107243
25.31199407
25.31199407
25.31199407
25.31199407
25.31199407
25.32629074
25.32629074
25.32629074
25.32629074
25.32629074
25.30824665
25.30824665
25.30824665
25.30824665
25.30824665
25.38072899
25.38072899
25.38072899
25.38072899
25.38072899
25.31401594
25.31401594
25.31401594
25.31401594
25.31401594
25.32931171
25.32931171
25.32931171
25.32931171
25.32931171
25.31446595
25.31446595
25.31446595
25.31446595
25.31446595
25.31676623
25.31676623
25.31676623
25.31676623
25.31676623
25.32514438
25.32514438
25.32514438
25.32514438
25.32514438
25.34935134
25.34935134
25.34935134
25.34935134
25.34935134
25.28932358
25.28932358
25.28932358
25.28932358
25.28932358
25.31613829
25.31613829
25.31613829
25.31613829
25.31613829
25.34997232
25.34997232
25.34997232
25.34997232
25.34997232
25.34195191
25.34195191
25.34195191
25.34195191
25.34195191
25.29693482
25.29693482
25.29693482
25.29693482
25.29693482
25.30628389
25.30628389
25.30628389
25.30628389
25.30628389
25.31495361
25.31495361
25.31495361
25.31495361
25.31495361
25.32437877
25.32437877
25.32437877
25.32437877
25.32437877
25.28747806
25.28747806
25.28747806
25.28747806
25.28747806
25.29910014
25.29910014
25.29910014
25.29910014
25.29910014
25.31192133
25.31192133
25.31192133
25.31192133
25.31192133
25.31820263
25.31820263
25.31820263
25.31820263
25.31820263
25.33926322
25.33926322
25.33926322
25.33926322
25.33926322
25.30316879
25.30316879
25.30316879
25.30316879
25.30316879
25.30452711
25.30452711
25.30452711
25.30452711
25.30452711
25.31635701
25.31635701
25.31635701
25.31635701
25.31635701
25.30627133
25.30627133
25.30627133
25.30627133
25.30627133
25.32410693
25.32410693
25.32410693
25.32410693
25.32410693
25.29490035
25.29490035
25.29490035
25.29490035
25.29490035
25.3588229
25.3588229
25.3588229
25.3588229
25.3588229
25.32435402
25.32435402
25.32435402
25.32435402
25.32435402
25.33769641
25.33769641
25.33769641
25.33769641
25.33769641
25.32591703
25.32591703
25.32591703
25.32591703
25.32591703
25.26995349
25.26995349
25.26995349
25.26995349
25.26995349
25.32126879
25.32126879
25.32126879
25.32126879
25.32126879
25.32466704
25.32466704
25.32466704
25.32466704
25.32466704
25.31686868
25.31686868
25.31686868
25.31686868
25.31686868
25.3104681
25.3104681
25.3104681
25.3104681
25.3104681
25.33501229
25.33501229
25.33501229
25.33501229
25.33501229
25.3161061
25.3161061
25.3161061
25.3161061
25.3161061
25.32672424
25.32672424
25.32672424
25.32672424
25.32672424
25.29745529
25.29745529
25.29745529
25.29745529
25.29745529
25.3182715
25.3182715
25.3182715
25.3182715
25.3182715
25.33297514
25.33297514
25.33297514
25.33297514
25.33297514
25.28544855
25.28544855
25.28544855
25.28544855
25.28544855
25.32885088
25.32885088
25.32885088
25.32885088
25.32885088
25.29590187
25.29590187
25.29590187
25.29590187
25.29590187
25.31705278
25.31705278
25.31705278
25.31705278
25.31705278
25.33070158
25.33070158
25.33070158
25.33070158
25.33070158
25.29563467
25.29563467
25.29563467
25.29563467
25.29563467
25.29613929
25.29613929
25.29613929
25.29613929
25.29613929
25.33869093
25.33869093
25.33869093
25.33869093
25.33869093
25.33296553
25.33296553
25.33296553
25.33296553
25.33296553
25.3067299
25.3067299
25.3067299
25.3067299
25.3067299
25.29896173
25.29896173
25.29896173
25.29896173
25.29896173
25.33037
25.33037
25.33037
25.33037
25.33037
25.35861205
25.35861205
25.35861205
25.35861205
25.35861205
25.31561964
25.31561964
25.31561964
25.31561964
25.31561964
25.28635312
25.28635312
25.28635312
25.28635312
25.28635312
25.30615908
25.30615908
25.30615908
25.30615908
25.30615908
25.30207504
25.30207504
25.30207504
25.30207504
25.30207504
25.27686348
25.27686348
25.27686348
25.27686348
25.27686348
25.33550478
25.33550478
25.33550478
25.33550478
25.33550478
25.31943948
25.31943948
25.31943948
25.31943948
25.31943948
25.36443228
25.36443228
25.36443228
25.36443228
25.36443228
25.31836671
25.31836671
25.31836671
25.31836671
25.31836671
25.30608176
25.30608176
25.30608176
25.30608176
25.30608176
25.31568844
25.31568844
25.31568844
25.31568844
25.31568844
25.29675234
25.29675234
25.29675234
25.29675234
25.29675234
25.27536695
25.27536695
25.27536695
25.27536695
25.27536695
25.31424096
25.31424096
25.31424096
25.31424096
25.31424096
25.32276883
25.32276883
25.32276883
25.32276883
25.32276883
25.347585
25.347585
25.347585
25.347585
25.347585
25.33981758
25.33981758
25.33981758
25.33981758
25.33981758
25.30328096
25.30328096
25.30328096
25.30328096
25.30328096
25.3809707
25.3809707
25.3809707
25.3809707
25.3809707
25.32740156
25.32740156
25.32740156
25.32740156
25.32740156
25.32101512
25.32101512
25.32101512
25.32101512
25.32101512
25.32973426
25.32973426
25.32973426
25.32973426
25.32973426
25.33608993
25.33608993
25.33608993
25.33608993
25.33608993
25.31578763
25.31578763
25.31578763
25.31578763
25.31578763
25.29305761
25.29305761
25.29305761
25.29305761
25.29305761
25.30707027
25.30707027
25.30707027
25.30707027
25.30707027
25.32459602
25.32459602
25.32459602
25.32459602
25.32459602
25.36693013
25.36693013
25.36693013
25.36693013
25.36693013
25.3205316
25.3205316
25.3205316
25.3205316
25.3205316
25.28288599
25.28288599
25.28288599
25.28288599
25.28288599
25.32869616
25.32869616
25.32869616
25.32869616
25.32869616
25.33180521
25.33180521
25.33180521
25.33180521
25.33180521
25.35555544
25.35555544
25.35555544
25.35555544
25.35555544
25.29639347
25.29639347
25.29639347
25.29639347
25.29639347
25.31305628
25.31305628
25.31305628
25.31305628
25.31305628
25.3608044
25.3608044
25.3608044
25.3608044
25.3608044
25.33974592
25.33974592
25.33974592
25.33974592
25.33974592
25.30130367
25.30130367
25.30130367
25.30130367
25.30130367
25.3124855
25.3124855
25.3124855
25.3124855
25.3124855
25.32336911
25.32336911
25.32336911
25.32336911
25.32336911
25.32003349
25.32003349
25.32003349
25.32003349
25.32003349
25.30397063
25.30397063
25.30397063
25.30397063
25.30397063
25.2963143
25.2963143
25.2963143
25.2963143
25.2963143
25.33075869
25.33075869
25.33075869
25.33075869
25.33075869
25.31441091
25.31441091
25.31441091
25.31441091
25.31441091
25.36992873
25.36992873
25.36992873
25.36992873
25.36992873
25.32157691
25.32157691
25.32157691
25.32157691
25.32157691
25.27512016
25.27512016
25.27512016
25.27512016
25.27512016
25.31937932
25.31937932
25.31937932
25.31937932
25.31937932
25.33395122
25.33395122
25.33395122
25.33395122
25.33395122
25.32915704
25.32915704
25.32915704
25.32915704
25.32915704
25.2730111
25.2730111
25.2730111
25.2730111
25.2730111
25.29897146
25.29897146
25.29897146
25.29897146
25.29897146
25.31882176
25.31882176
25.31882176
25.31882176
25.31882176
25.36349031
25.36349031
25.36349031
25.36349031
25.36349031
25.35544768
25.35544768
25.35544768
25.35544768
25.35544768
25.31513103
25.31513103
25.31513103
25.31513103
25.31513103
25.31722498
25.31722498
25.31722498
25.31722498
25.31722498
25.3482704
25.3482704
25.3482704
25.3482704
25.3482704
25.2988996
25.2988996
25.2988996
25.2988996
25.2988996
25.30674405
25.30674405
25.30674405
25.30674405
25.30674405
25.31846894
25.31846894
25.31846894
25.31846894
25.31846894
25.32974963
25.32974963
25.32974963
25.32974963
25.32974963
25.31869636
25.31869636
25.31869636
25.31869636
25.31869636
25.35187115
25.35187115
25.35187115
25.35187115
25.35187115
25.30401255
25.30401255
25.30401255
25.30401255
25.30401255
25.32583025
25.32583025
25.32583025
25.32583025
25.32583025
25.34533823
25.34533823
25.34533823
25.34533823
25.34533823
25.3322697
25.3322697
25.3322697
25.3322697
25.3322697
25.32457625
25.32457625
25.32457625
25.32457625
25.32457625
25.35669212
25.35669212
25.35669212
25.35669212
25.35669212
25.30862614
25.30862614
25.30862614
25.30862614
25.30862614
25.30559244
25.30559244
25.30559244
25.30559244
25.30559244
25.30385285
25.30385285
25.30385285
25.30385285
25.30385285
25.27575992
25.27575992
25.27575992
25.27575992
25.27575992
25.33943008
25.33943008
25.33943008
25.33943008
25.33943008
25.32641513
25.32641513
25.32641513
25.32641513
25.32641513
25.31584201
25.31584201
25.31584201
25.31584201
25.31584201
25.31051345
25.31051345
25.31051345
25.31051345
25.31051345
25.32892509
25.32892509
25.32892509
25.32892509
25.32892509
25.31008916
25.31008916
25.31008916
25.31008916
25.31008916
25.30031102
25.30031102
25.30031102
25.30031102
25.30031102
25.33148248
25.33148248
25.33148248
25.33148248
25.33148248
25.32726893
25.32726893
25.32726893
25.32726893
25.32726893
25.31786323
25.31786323
25.31786323
25.31786323
25.31786323
25.29420436
25.29420436
25.29420436
25.29420436
25.29420436
25.30285948
25.30285948
25.30285948
25.30285948
25.30285948
25.29888913
25.29888913
25.29888913
25.29888913
25.29888913
25.28841578
25.28841578
25.28841578
25.28841578
25.28841578
25.33026289
25.33026289
25.33026289
25.33026289
25.33026289
25.32259063
25.32259063
25.32259063
25.32259063
25.32259063
25.3177212
25.3177212
25.3177212
25.3177212
25.3177212
25.29404341
25.29404341
25.29404341
25.29404341
25.29404341
25.29584513
25.29584513
25.29584513
25.29584513
25.29584513
25.28306023
25.28306023
25.28306023
25.28306023
25.28306023
25.25773317
25.25773317
25.25773317
25.25773317
25.25773317
25.32838803
25.32838803
25.32838803
25.32838803
25.32838803
25.28376528
25.28376528
25.28376528
25.28376528
25.28376528
25.32574742
25.32574742
25.32574742
25.32574742
25.32574742
25.28618545
25.28618545
25.28618545
25.28618545
25.28618545
25.32074426
25.32074426
25.32074426
25.32074426
25.32074426
25.29785637
25.29785637
25.29785637
25.29785637
25.29785637
25.36730379
25.36730379
25.36730379
25.36730379
25.36730379
25.34435933
25.34435933
25.34435933
25.34435933
25.34435933
25.27759455
25.27759455
25.27759455
25.27759455
25.27759455
25.31178884
25.31178884
25.31178884
25.31178884
25.31178884
25.26788451
25.26788451
25.26788451
25.26788451
25.26788451
25.32030517
25.32030517
25.32030517
25.32030517
25.32030517
25.28707613
25.28707613
25.28707613
25.28707613
25.28707613
25.32104423
25.32104423
25.32104423
25.32104423
25.32104423
25.29492335
25.29492335
25.29492335
25.29492335
25.29492335
25.3173685
25.3173685
25.3173685
25.3173685
25.3173685
25.30013193
25.30013193
25.30013193
25.30013193
25.30013193
25.29590322
25.29590322
25.29590322
25.29590322
25.29590322
25.29358207
25.29358207
25.29358207
25.29358207
25.29358207
25.32840953
25.32840953
25.32840953
25.32840953
25.32840953
25.30434928
25.30434928
25.30434928
25.30434928
25.30434928
25.3286082
25.3286082
25.3286082
25.3286082
25.3286082
25.30786478
25.30786478
25.30786478
25.30786478
25.30786478
25.30916106
25.30916106
25.30916106
25.30916106
25.30916106
25.32440464
25.32440464
25.32440464
25.32440464
25.32440464
25.3164235
25.3164235
25.3164235
25.3164235
25.3164235
25.32723545
25.32723545
25.32723545
25.32723545
25.32723545
25.31658924
25.31658924
25.31658924
25.31658924
25.31658924
25.29934974
25.29934974
25.29934974
25.29934974
25.29934974
25.31970591
25.31970591
25.31970591
25.31970591
25.31970591
25.3083752
25.3083752
25.3083752
25.3083752
25.3083752
25.34481295
25.34481295
25.34481295
25.34481295
25.34481295
25.32918876
25.32918876
25.32918876
25.32918876
25.32918876
25.31166175
25.31166175
25.31166175
25.31166175
25.31166175
25.31993947
25.31993947
25.31993947
25.31993947
25.31993947
25.30492409
25.30492409
25.30492409
25.30492409
25.30492409
25.32805837
25.32805837
25.32805837
25.32805837
25.32805837
25.3201822
25.3201822
25.3201822
25.3201822
25.3201822
25.31169012
25.31169012
25.31169012
25.31169012
25.31169012
25.34400392
25.34400392
25.34400392
25.34400392
25.34400392
25.31760688
25.31760688
25.31760688
25.31760688
25.31760688
25.34004299
25.34004299
25.34004299
25.34004299
25.34004299
25.29455863
25.29455863
25.29455863
25.29455863
25.29455863
25.326162
25.326162
25.326162
25.326162
25.326162
25.27440857
25.27440857
25.27440857
25.27440857
25.27440857
25.28084096
25.28084096
25.28084096
25.28084096
25.28084096
25.33544672
25.33544672
25.33544672
25.33544672
25.33544672
25.34382631
25.34382631
25.34382631
25.34382631
25.34382631
25.31558531
25.31558531
25.31558531
25.31558531
25.31558531
25.32563862
25.32563862
25.32563862
25.32563862
25.32563862
25.30754464
25.30754464
25.30754464
25.30754464
25.30754464
25.3298717
25.3298717
25.3298717
25.3298717
25.3298717
25.33150925
25.33150925
25.33150925
25.33150925
25.33150925
25.31393548
25.31393548
25.31393548
25.31393548
25.31393548
25.34384896
25.34384896
25.34384896
25.34384896
25.34384896
25.31544908
25.31544908
25.31544908
25.31544908
25.31544908
25.31668623
25.31668623
25.31668623
25.31668623
25.31668623
25.30445104
25.30445104
25.30445104
25.30445104
25.30445104
25.31734432
25.31734432
25.31734432
25.31734432
25.31734432
25.32224598
25.32224598
25.32224598
25.32224598
25.32224598
25.34553239
25.34553239
25.34553239
25.34553239
25.34553239
25.32189598
25.32189598
25.32189598
25.32189598
25.32189598
25.36462746
25.36462746
25.36462746
25.36462746
25.36462746
25.31418243
25.31418243
25.31418243
25.31418243
25.31418243
25.33454206
25.33454206
25.33454206
25.33454206
25.33454206
25.28850883
25.28850883
25.28850883
25.28850883
25.28850883
25.33512815
25.33512815
25.33512815
25.33512815
25.33512815
25.33625728
25.33625728
25.33625728
25.33625728
25.33625728
25.31468051
25.31468051
25.31468051
25.31468051
25.31468051
25.28329046
25.28329046
25.28329046
25.28329046
25.28329046
25.3237475
25.3237475
25.3237475
25.3237475
25.3237475
25.34201768
25.34201768
25.34201768
25.34201768
25.34201768
25.33407753
25.33407753
25.33407753
25.33407753
25.33407753
25.32227383
25.32227383
25.32227383
25.32227383
25.32227383
25.31692324
25.31692324
25.31692324
25.31692324
25.31692324
25.3227758
25.3227758
25.3227758
25.3227758
25.3227758
25.29344954
25.29344954
25.29344954
25.29344954
25.29344954
25.3429068
25.3429068
25.3429068
25.3429068
25.3429068
25.31750423
25.31750423
25.31750423
25.31750423
25.31750423
25.30105964
25.30105964
25.30105964
25.30105964
25.30105964
25.34406136
25.34406136
25.34406136
25.34406136
25.34406136
25.31319553
25.31319553
25.31319553
25.31319553
25.31319553
25.30438165
25.30438165
25.30438165
25.30438165
25.30438165
25.3250559
25.3250559
25.3250559
25.3250559
25.3250559
25.36471627
25.36471627
25.36471627
25.36471627
25.36471627
25.30664895
25.30664895
25.30664895
25.30664895
25.30664895
25.31829165
25.31829165
25.31829165
25.31829165
25.31829165
25.31462915
25.31462915
25.31462915
25.31462915
25.31462915
25.29216708
25.29216708
25.29216708
25.29216708
25.29216708
25.33762117
25.33762117
25.33762117
25.33762117
25.33762117
25.31979707
25.31979707
25.31979707
25.31979707
25.31979707
25.3625376
25.3625376
25.3625376
25.3625376
25.3625376
25.33045739
25.33045739
25.33045739
25.33045739
25.33045739
25.29335794
25.29335794
25.29335794
25.29335794
25.29335794
25.34128873
25.34128873
25.34128873
25.34128873
25.34128873
25.29772989
25.29772989
25.29772989
25.29772989
25.29772989
25.31421904
25.31421904
25.31421904
25.31421904
25.31421904
25.3290289
25.3290289
25.3290289
25.3290289
25.3290289
25.27820107
25.27820107
25.27820107
25.27820107
25.27820107
25.3656118
25.3656118
25.3656118
25.3656118
25.3656118
25.27930046
25.27930046
25.27930046
25.27930046
25.27930046
25.34095134
25.34095134
25.34095134
25.34095134
25.34095134
25.32614178
25.32614178
25.32614178
25.32614178
25.32614178
25.30587031
25.30587031
25.30587031
25.30587031
25.30587031
25.35158268
25.35158268
25.35158268
25.35158268
25.35158268
25.33156447
25.33156447
25.33156447
25.33156447
25.33156447
25.32289557
25.32289557
25.32289557
25.32289557
25.32289557
25.30859192
25.30859192
25.30859192
25.30859192
25.30859192
25.31525903
25.31525903
25.31525903
25.31525903
25.31525903
25.33996493
25.33996493
25.33996493
25.33996493
25.33996493
25.3127035
25.3127035
25.3127035
25.3127035
25.3127035
25.32377833
25.32377833
25.32377833
25.32377833
25.32377833
25.30850964
25.30850964
25.30850964
25.30850964
25.30850964
25.31828951
25.31828951
25.31828951
25.31828951
25.31828951
25.30747964
25.30747964
25.30747964
25.30747964
25.30747964
25.31525114
25.31525114
25.31525114
25.31525114
25.31525114
25.30327284
25.30327284
25.30327284
25.30327284
25.30327284
25.34372048
25.34372048
25.34372048
25.34372048
25.34372048
25.30206761
25.30206761
25.30206761
25.30206761
25.30206761
25.30032171
25.30032171
25.30032171
25.30032171
25.30032171
25.2885296
25.2885296
25.2885296
25.2885296
25.2885296
25.29239548
25.29239548
25.29239548
25.29239548
25.29239548
25.32665415
25.32665415
25.32665415
25.32665415
25.32665415
25.34848266
25.34848266
25.34848266
25.34848266
25.34848266
25.31657694
25.31657694
25.31657694
25.31657694
25.31657694
25.32408396
25.32408396
25.32408396
25.32408396
25.32408396
25.31725237
25.31725237
25.31725237
25.31725237
25.31725237
25.28327949
25.28327949
25.28327949
25.28327949
25.28327949
25.32306209
25.32306209
25.32306209
25.32306209
25.32306209
25.33146468
25.33146468
25.33146468
25.33146468
25.33146468
25.28566316
25.28566316
25.28566316
25.28566316
25.28566316
25.3227183
25.3227183
25.3227183
25.3227183
25.3227183
25.36945448
25.36945448
25.36945448
25.36945448
25.36945448
25.31506141
25.31506141
25.31506141
25.31506141
25.31506141
25.31967816
25.31967816
25.31967816
25.31967816
25.31967816
25.31751823
25.31751823
25.31751823
25.31751823
25.31751823
25.29140693
25.29140693
25.29140693
25.29140693
25.29140693
25.29758556
25.29758556
25.29758556
25.29758556
25.29758556
25.31165399
25.31165399
25.31165399
25.31165399
25.31165399
25.32648259
25.32648259
25.32648259
25.32648259
25.32648259
25.30345697
25.30345697
25.30345697
25.30345697
25.30345697
25.32251987
25.32251987
25.32251987
25.32251987
25.32251987
25.36749957
25.36749957
25.36749957
25.36749957
25.36749957
25.3338689
25.3338689
25.3338689
25.3338689
25.3338689
25.31401737
25.31401737
25.31401737
25.31401737
25.31401737
25.35407359
25.35407359
25.35407359
25.35407359
25.35407359
25.29376724
25.29376724
25.29376724
25.29376724
25.29376724
25.31797428
25.31797428
25.31797428
25.31797428
25.31797428
25.3173049
25.3173049
25.3173049
25.3173049
25.3173049
25.36503686
25.36503686
25.36503686
25.36503686
25.36503686
25.28159681
25.28159681
25.28159681
25.28159681
25.28159681
25.2846821
25.2846821
25.2846821
25.2846821
25.2846821
25.30917433
25.30917433
25.30917433
25.30917433
25.30917433
25.31580403
25.31580403
25.31580403
25.31580403
25.31580403
25.33235353
25.33235353
25.33235353
25.33235353
25.33235353
25.33071035
25.33071035
25.33071035
25.33071035
25.33071035
25.34683159
25.34683159
25.34683159
25.34683159
25.34683159
25.33487199
25.33487199
25.33487199
25.33487199
25.33487199
25.32795313
25.32795313
25.32795313
25.32795313
25.32795313
25.3330355
25.3330355
25.3330355
25.3330355
25.3330355
25.32291976
25.32291976
25.32291976
25.32291976
25.32291976
25.32874438
25.32874438
25.32874438
25.32874438
25.32874438
25.32532183
25.32532183
25.32532183
25.32532183
25.32532183
25.31564937
25.31564937
25.31564937
25.31564937
25.31564937
25.3229432
25.3229432
25.3229432
25.3229432
25.3229432
25.35018694
25.35018694
25.35018694
25.35018694
25.35018694
25.31734845
25.31734845
25.31734845
25.31734845
25.31734845
25.3566599
25.3566599
25.3566599
25.3566599
25.3566599
25.3535088
25.3535088
25.3535088
25.3535088
25.3535088
25.32727482
25.32727482
25.32727482
25.32727482
25.32727482
25.31635703
25.31635703
25.31635703
25.31635703
25.31635703
25.33080927
25.33080927
25.33080927
25.33080927
25.33080927
25.28786985
25.28786985
25.28786985
25.28786985
25.28786985
25.38554312
25.38554312
25.38554312
25.38554312
25.38554312
25.3413458
25.3413458
25.3413458
25.3413458
25.3413458
25.28722577
25.28722577
25.28722577
25.28722577
25.28722577
25.30629617
25.30629617
25.30629617
25.30629617
25.30629617
25.30487477
25.30487477
25.30487477
25.30487477
25.30487477
25.3119851
25.3119851
25.3119851
25.3119851
25.3119851
25.30847614
25.30847614
25.30847614
25.30847614
25.30847614
25.33063584
25.33063584
25.33063584
25.33063584
25.33063584
25.33964467
25.33964467
25.33964467
25.33964467
25.33964467
25.36457069
25.36457069
25.36457069
25.36457069
25.36457069
25.31701097
25.31701097
25.31701097
25.31701097
25.31701097
25.26978078
25.26978078
25.26978078
25.26978078
25.26978078
25.35938202
25.35938202
25.35938202
25.35938202
25.35938202
25.29293681
25.29293681
25.29293681
25.29293681
25.29293681
25.27859845
25.27859845
25.27859845
25.27859845
25.27859845
25.32782114
25.32782114
25.32782114
25.32782114
25.32782114
25.29566823
25.29566823
25.29566823
25.29566823
25.29566823
25.36034285
25.36034285
25.36034285
25.36034285
25.36034285
25.3380173
25.3380173
25.3380173
25.3380173
25.3380173
25.31472525
25.31472525
25.31472525
25.31472525
25.31472525
25.36946152
25.36946152
25.36946152
25.36946152
25.36946152
25.35654366
25.35654366
25.35654366
25.35654366
25.35654366
25.30874773
25.30874773
25.30874773
25.30874773
25.30874773
25.3475867
25.3475867
25.3475867
25.3475867
25.3475867
25.37754681
25.37754681
25.37754681
25.37754681
25.37754681
25.33561089
25.33561089
25.33561089
25.33561089
25.33561089
25.35671521
25.35671521
25.35671521
25.35671521
25.35671521
25.32075646
25.32075646
25.32075646
25.32075646
25.32075646
25.31405879
25.31405879
25.31405879
25.31405879
25.31405879
25.36667788
25.36667788
25.36667788
25.36667788
25.36667788
25.31621938
25.31621938
25.31621938
25.31621938
25.31621938
25.30964143
25.30964143
25.30964143
25.30964143
25.30964143
25.31954365
25.31954365
25.31954365
25.31954365
25.31954365
25.3485759
25.3485759
25.3485759
25.3485759
25.3485759
25.33441482
25.33441482
25.33441482
25.33441482
25.33441482
25.32152678
25.32152678
25.32152678
25.32152678
25.32152678
25.31606989
25.31606989
25.31606989
25.31606989
25.31606989
25.28209977
25.28209977
25.28209977
25.28209977
25.28209977
25.35613057
25.35613057
25.35613057
25.35613057
25.35613057
25.28274186
25.28274186
25.28274186
25.28274186
25.28274186
25.31785412
25.31785412
25.31785412
25.31785412
25.31785412
25.33173902
25.33173902
25.33173902
25.33173902
25.33173902
25.30610083
25.30610083
25.30610083
25.30610083
25.30610083
25.29402821
25.29402821
25.29402821
25.29402821
25.29402821
25.30199121
25.30199121
25.30199121
25.30199121
25.30199121
25.34863649
25.34863649
25.34863649
25.34863649
25.34863649
25.2917781
25.2917781
25.2917781
25.2917781
25.2917781
25.34628662
25.34628662
25.34628662
25.34628662
25.34628662
25.28877481
25.28877481
25.28877481
25.28877481
25.28877481
25.33392316
25.33392316
25.33392316
25.33392316
25.33392316
25.30909893
25.30909893
25.30909893
25.30909893
25.30909893
25.33754466
25.33754466
25.33754466
25.33754466
25.33754466
25.32505223
25.32505223
25.32505223
25.32505223
25.32505223
25.31792663
25.31792663
25.31792663
25.31792663
25.31792663
25.33888114
25.33888114
25.33888114
25.33888114
25.33888114
25.33819363
25.33819363
25.33819363
25.33819363
25.33819363
25.33240472
25.33240472
25.33240472
25.33240472
25.33240472
25.33735095
25.33735095
25.33735095
25.33735095
25.33735095
25.3089637
25.3089637
25.3089637
25.3089637
25.3089637
25.35033239
25.35033239
25.35033239
25.35033239
25.35033239
25.29360205
25.29360205
25.29360205
25.29360205
25.29360205
25.32221319
25.32221319
25.32221319
25.32221319
25.32221319
25.30648614
25.30648614
25.30648614
25.30648614
25.30648614
25.31914147
25.31914147
25.31914147
25.31914147
25.31914147
25.32016797
25.32016797
25.32016797
25.32016797
25.32016797
25.30511053
25.30511053
25.30511053
25.30511053
25.30511053
25.33903373
25.33903373
25.33903373
25.33903373
25.33903373
25.31394398
25.31394398
25.31394398
25.31394398
25.31394398
25.36597429
25.36597429
25.36597429
25.36597429
25.36597429
25.33346228
25.33346228
25.33346228
25.33346228
25.33346228
25.33973718
25.33973718
25.33973718
25.33973718
25.33973718
25.33794522
25.33794522
25.33794522
25.33794522
25.33794522
25.33087637
25.33087637
25.33087637
25.33087637
25.33087637
25.35399466
25.35399466
25.35399466
25.35399466
25.35399466
25.33401358
25.33401358
25.33401358
25.33401358
25.33401358
25.32763516
25.32763516
25.32763516
25.32763516
25.32763516
25.35573926
25.35573926
25.35573926
25.35573926
25.35573926
25.30158398
25.30158398
25.30158398
25.30158398
25.30158398
25.29023438
25.29023438
25.29023438
25.29023438
25.29023438
25.33509954
25.33509954
25.33509954
25.33509954
25.33509954
25.31749139
25.31749139
25.31749139
25.31749139
25.31749139
25.31398894
25.31398894
25.31398894
25.31398894
25.31398894
25.38018434
25.38018434
25.38018434
25.38018434
25.38018434
25.32092023
25.32092023
25.32092023
25.32092023
25.32092023
25.3035816
25.3035816
25.3035816
25.3035816
25.3035816
25.34010616
25.34010616
25.34010616
25.34010616
25.34010616
25.2846638
25.2846638
25.2846638
25.2846638
25.2846638
25.2907314
25.2907314
25.2907314
25.2907314
25.2907314
25.35580043
25.35580043
25.35580043
25.35580043
25.35580043
25.36314453
25.36314453
25.36314453
25.36314453
25.36314453
25.33297992
25.33297992
25.33297992
25.33297992
25.33297992
25.29076152
25.29076152
25.29076152
25.29076152
25.29076152
25.29591537
25.29591537
25.29591537
25.29591537
25.29591537
25.30466652
25.30466652
25.30466652
25.30466652
25.30466652
25.31907883
25.31907883
25.31907883
25.31907883
25.31907883
25.32606223
25.32606223
25.32606223
25.32606223
25.32606223
25.36103055
25.36103055
25.36103055
25.36103055
25.36103055
25.30919853
25.30919853
25.30919853
25.30919853
25.30919853
25.32700874
25.32700874
25.32700874
25.32700874
25.32700874
25.33086602
25.33086602
25.33086602
25.33086602
25.33086602
25.30472015
25.30472015
25.30472015
25.30472015
25.30472015
25.36714292
25.36714292
25.36714292
25.36714292
25.36714292
25.27927243
25.27927243
25.27927243
25.27927243
25.27927243
25.3168594
25.3168594
25.3168594
25.3168594
25.3168594
25.32437772
25.32437772
25.32437772
25.32437772
25.32437772
25.2967187
25.2967187
25.2967187
25.2967187
25.2967187
25.31033011
25.31033011
25.31033011
25.31033011
25.31033011
25.32437021
25.32437021
25.32437021
25.32437021
25.32437021
25.28256002
25.28256002
25.28256002
25.28256002
25.28256002
25.34632485
25.34632485
25.34632485
25.34632485
25.34632485
25.30598731
25.30598731
25.30598731
25.30598731
25.30598731
25.28980363
25.28980363
25.28980363
25.28980363
25.28980363
25.31086074
25.31086074
25.31086074
25.31086074
25.31086074
25.29364529
25.29364529
25.29364529
25.29364529
25.29364529
25.32762414
25.32762414
25.32762414
25.32762414
25.32762414
25.31947035
25.31947035
25.31947035
25.31947035
25.31947035
25.33207772
25.33207772
25.33207772
25.33207772
25.33207772
25.338005
25.338005
25.338005
25.338005
25.338005
25.29407059
25.29407059
25.29407059
25.29407059
25.29407059
25.33431039
25.33431039
25.33431039
25.33431039
25.33431039
25.31545774
25.31545774
25.31545774
25.31545774
25.31545774
25.33048931
25.33048931
25.33048931
25.33048931
25.33048931
25.34537007
25.34537007
25.34537007
25.34537007
25.34537007
25.32612325
25.32612325
25.32612325
25.32612325
25.32612325
25.29476952
25.29476952
25.29476952
25.29476952
25.29476952
25.32628455
25.32628455
25.32628455
25.32628455
25.32628455
25.29634183
25.29634183
25.29634183
25.29634183
25.29634183
25.32940352
25.32940352
25.32940352
25.32940352
25.32940352
25.31568701
25.31568701
25.31568701
25.31568701
25.31568701
25.31625008
25.31625008
25.31625008
25.31625008
25.31625008
25.31637282
25.31637282
25.31637282
25.31637282
25.31637282
25.33110146
25.33110146
25.33110146
25.33110146
25.33110146
25.30519048
25.30519048
25.30519048
25.30519048
25.30519048
25.32783823
25.32783823
25.32783823
25.32783823
25.32783823
25.34863328
25.34863328
25.34863328
25.34863328
25.34863328
25.31528047
25.31528047
25.31528047
25.31528047
25.31528047
25.34301167
25.34301167
25.34301167
25.34301167
25.34301167
25.31471164
25.31471164
25.31471164
25.31471164
25.31471164
25.35151842
25.35151842
25.35151842
25.35151842
25.35151842
25.33013487
25.33013487
25.33013487
25.33013487
25.33013487
25.32467189
25.32467189
25.32467189
25.32467189
25.32467189
25.33427276
25.33427276
25.33427276
25.33427276
25.33427276
25.32248053
25.32248053
25.32248053
25.32248053
25.32248053
25.33351553
25.33351553
25.33351553
25.33351553
25.33351553
25.38041809
25.38041809
25.38041809
25.38041809
25.38041809
25.3113611
25.3113611
25.3113611
25.3113611
25.3113611
25.30072804
25.30072804
25.30072804
25.30072804
25.30072804
25.34398002
25.34398002
25.34398002
25.34398002
25.34398002
25.28966629
25.28966629
25.28966629
25.28966629
25.28966629
25.33668904
25.33668904
25.33668904
25.33668904
25.33668904
25.28156701
25.28156701
25.28156701
25.28156701
25.28156701
25.31894677
25.31894677
25.31894677
25.31894677
25.31894677
25.32318227
25.32318227
25.32318227
25.32318227
25.32318227
25.30625069
25.30625069
25.30625069
25.30625069
25.30625069
25.35202978
25.35202978
25.35202978
25.35202978
25.35202978
25.33807553
25.33807553
25.33807553
25.33807553
25.33807553
25.34730201
25.34730201
25.34730201
25.34730201
25.34730201
25.35854829
25.35854829
25.35854829
25.35854829
25.35854829
25.30605837
25.30605837
25.30605837
25.30605837
25.30605837
25.30871452
25.30871452
25.30871452
25.30871452
25.30871452
25.32917462
25.32917462
25.32917462
25.32917462
25.32917462
25.31150065
25.31150065
25.31150065
25.31150065
25.31150065
25.32030783
25.32030783
25.32030783
25.32030783
25.32030783
25.33966304
25.33966304
25.33966304
25.33966304
25.33966304
25.32408558
25.32408558
25.32408558
25.32408558
25.32408558
25.32185168
25.32185168
25.32185168
25.32185168
25.32185168
25.34819178
25.34819178
25.34819178
25.34819178
25.34819178
25.30866387
25.30866387
25.30866387
25.30866387
25.30866387
25.33080718
25.33080718
25.33080718
25.33080718
25.33080718
25.28345512
25.28345512
25.28345512
25.28345512
25.28345512
25.34275869
25.34275869
25.34275869
25.34275869
25.34275869
25.34042046
25.34042046
25.34042046
25.34042046
25.34042046
25.33640368
25.33640368
25.33640368
25.33640368
25.33640368
25.34778054
25.34778054
25.34778054
25.34778054
25.34778054
25.35812168
25.35812168
25.35812168
25.35812168
25.35812168
25.33409849
25.33409849
25.33409849
25.33409849
25.33409849
25.30685753
25.30685753
25.30685753
25.30685753
25.30685753
25.32204686
25.32204686
25.32204686
25.32204686
25.32204686
25.31316499
25.31316499
25.31316499
25.31316499
25.31316499
25.34631331
25.34631331
25.34631331
25.34631331
25.34631331
25.32064126
25.32064126
25.32064126
25.32064126
25.32064126
25.33229872
25.33229872
25.33229872
25.33229872
25.33229872
25.35904926
25.35904926
25.35904926
25.35904926
25.35904926
25.32163861
25.32163861
25.32163861
25.32163861
25.32163861
25.30113725
25.30113725
25.30113725
25.30113725
25.30113725
25.33304391
25.33304391
25.33304391
25.33304391
25.33304391
25.30567791
25.30567791
25.30567791
25.30567791
25.30567791
25.33228673
25.33228673
25.33228673
25.33228673
25.33228673
25.36559322
25.36559322
25.36559322
25.36559322
25.36559322
25.30541103
25.30541103
25.30541103
25.30541103
25.30541103
25.29209255
25.29209255
25.29209255
25.29209255
25.29209255
25.30039469
25.30039469
25.30039469
25.30039469
25.30039469
25.30710619
25.30710619
25.30710619
25.30710619
25.30710619
25.3215304
25.3215304
25.3215304
25.3215304
25.3215304
25.31350615
25.31350615
25.31350615
25.31350615
25.31350615
25.30261111
25.30261111
25.30261111
25.30261111
25.30261111
25.28697176
25.28697176
25.28697176
25.28697176
25.28697176
25.35526389
25.35526389
25.35526389
25.35526389
25.35526389
25.29947164
25.29947164
25.29947164
25.29947164
25.29947164
25.30990515
25.30990515
25.30990515
25.30990515
25.30990515
25.3048486
25.3048486
25.3048486
25.3048486
25.3048486
25.3542661
25.3542661
25.3542661
25.3542661
25.3542661
25.28284284
25.28284284
25.28284284
25.28284284
25.28284284
25.30402659
25.30402659
25.30402659
25.30402659
25.30402659
25.29253061
25.29253061
25.29253061
25.29253061
25.29253061
25.28903
25.28903
25.28903
25.28903
25.28903
25.32023418
25.32023418
25.32023418
25.32023418
25.32023418
25.30420372
25.30420372
25.30420372
25.30420372
25.30420372
25.31337489
25.31337489
25.31337489
25.31337489
25.31337489
25.32637153
25.32637153
25.32637153
25.32637153
25.32637153
25.31762162
25.31762162
25.31762162
25.31762162
25.31762162
25.32188903
25.32188903
25.32188903
25.32188903
25.32188903
25.30993489
25.30993489
25.30993489
25.30993489
25.30993489
25.32707375
25.32707375
25.32707375
25.32707375
25.32707375
25.32092956
25.32092956
25.32092956
25.32092956
25.32092956
25.33012195
25.33012195
25.33012195
25.33012195
25.33012195
25.3070141
25.3070141
25.3070141
25.3070141
25.3070141
25.29499901
25.29499901
25.29499901
25.29499901
25.29499901
25.32053105
25.32053105
25.32053105
25.32053105
25.32053105
25.31217085
25.31217085
25.31217085
25.31217085
25.31217085
25.32849329
25.32849329
25.32849329
25.32849329
25.32849329
25.29978812
25.29978812
25.29978812
25.29978812
25.29978812
25.33919219
25.33919219
25.33919219
25.33919219
25.33919219
25.29333504
25.29333504
25.29333504
25.29333504
25.29333504
25.34842477
25.34842477
25.34842477
25.34842477
25.34842477
25.28635441
25.28635441
25.28635441
25.28635441
25.28635441
25.32176591
25.32176591
25.32176591
25.32176591
25.32176591
25.33977886
25.33977886
25.33977886
25.33977886
25.33977886
25.34910168
25.34910168
25.34910168
25.34910168
25.34910168
25.31396511
25.31396511
25.31396511
25.31396511
25.31396511
25.320589
25.320589
25.320589
25.320589
25.320589
25.32876667
25.32876667
25.32876667
25.32876667
25.32876667
25.29461522
25.29461522
25.29461522
25.29461522
25.29461522
25.28954779
25.28954779
25.28954779
25.28954779
25.28954779
25.26315286
25.26315286
25.26315286
25.26315286
25.26315286
25.31453718
25.31453718
25.31453718
25.31453718
25.31453718
25.34333391
25.34333391
25.34333391
25.34333391
25.34333391
25.28303641
25.28303641
25.28303641
25.28303641
25.28303641
25.33336404
25.33336404
25.33336404
25.33336404
25.33336404
25.34291147
25.34291147
25.34291147
25.34291147
25.34291147
25.32929933
25.32929933
25.32929933
25.32929933
25.32929933
25.31542098
25.31542098
25.31542098
25.31542098
25.31542098
25.32298299
25.32298299
25.32298299
25.32298299
25.32298299
25.30967123
25.30967123
25.30967123
25.30967123
25.30967123
25.30196074
25.30196074
25.30196074
25.30196074
25.30196074
25.31597098
25.31597098
25.31597098
25.31597098
25.31597098
25.30166578
25.30166578
25.30166578
25.30166578
25.30166578
25.29940808
25.29940808
25.29940808
25.29940808
25.29940808
25.32359293
25.32359293
25.32359293
25.32359293
25.32359293
25.31611388
25.31611388
25.31611388
25.31611388
25.31611388
25.31734107
25.31734107
25.31734107
25.31734107
25.31734107
25.28205391
25.28205391
25.28205391
25.28205391
25.28205391
25.29239478
25.29239478
25.29239478
25.29239478
25.29239478
25.34469801
25.34469801
25.34469801
25.34469801
25.34469801
25.32620563
25.32620563
25.32620563
25.32620563
25.32620563
25.31201686
25.31201686
25.31201686
25.31201686
25.31201686
25.31040264
25.31040264
25.31040264
25.31040264
25.31040264
25.32430476
25.32430476
25.32430476
25.32430476
25.32430476
25.32218298
25.32218298
25.32218298
25.32218298
25.32218298
25.3475569
25.3475569
25.3475569
25.3475569
25.3475569
25.27606846
25.27606846
25.27606846
25.27606846
25.27606846
25.31644759
25.31644759
25.31644759
25.31644759
25.31644759
25.30465625
25.30465625
25.30465625
25.30465625
25.30465625
25.31481586
25.31481586
25.31481586
25.31481586
25.31481586
25.32180536
25.32180536
25.32180536
25.32180536
25.32180536
25.29663309
25.29663309
25.29663309
25.29663309
25.29663309
25.323184
25.323184
25.323184
25.323184
25.323184
25.32158039
25.32158039
25.32158039
25.32158039
25.32158039
25.31593201
25.31593201
25.31593201
25.31593201
25.31593201
25.33569666
25.33569666
25.33569666
25.33569666
25.33569666
25.32025164
25.32025164
25.32025164
25.32025164
25.32025164
25.33079772
25.33079772
25.33079772
25.33079772
25.33079772
25.32455614
25.32455614
25.32455614
25.32455614
25.32455614
25.32744858
25.32744858
25.32744858
25.32744858
25.32744858
25.3082108
25.3082108
25.3082108
25.3082108
25.3082108
25.32141257
25.32141257
25.32141257
25.32141257
25.32141257
25.28298552
25.28298552
25.28298552
25.28298552
25.28298552
25.32398962
25.32398962
25.32398962
25.32398962
25.32398962
25.32122577
25.32122577
25.32122577
25.32122577
25.32122577
25.35102068
25.35102068
25.35102068
25.35102068
25.35102068
25.32188323
25.32188323
25.32188323
25.32188323
25.32188323
25.28227344
25.28227344
25.28227344
25.28227344
25.28227344
25.34787383
25.34787383
25.34787383
25.34787383
25.34787383
25.33946723
25.33946723
25.33946723
25.33946723
25.33946723
25.27978131
25.27978131
25.27978131
25.27978131
25.27978131
25.36756059
25.36756059
25.36756059
25.36756059
25.36756059
25.36162149
25.36162149
25.36162149
25.36162149
25.36162149
25.36309285
25.36309285
25.36309285
25.36309285
25.36309285
25.30469414
25.30469414
25.30469414
25.30469414
25.30469414
25.29497663
25.29497663
25.29497663
25.29497663
25.29497663
25.30790908
25.30790908
25.30790908
25.30790908
25.30790908
25.37477518
25.37477518
25.37477518
25.37477518
25.37477518
25.32548783
25.32548783
25.32548783
25.32548783
25.32548783
25.33841184
25.33841184
25.33841184
25.33841184
25.33841184
25.33091654
25.33091654
25.33091654
25.33091654
25.33091654
25.32411446
25.32411446
25.32411446
25.32411446
25.32411446
25.36068281
25.36068281
25.36068281
25.36068281
25.36068281
25.29466083
25.29466083
25.29466083
25.29466083
25.29466083
25.34482923
25.34482923
25.34482923
25.34482923
25.34482923
25.31998051
25.31998051
25.31998051
25.31998051
25.31998051
25.32793392
25.32793392
25.32793392
25.32793392
25.32793392
25.31868355
25.31868355
25.31868355
25.31868355
25.31868355
25.28673057
25.28673057
25.28673057
25.28673057
25.28673057
25.34156186
25.34156186
25.34156186
25.34156186
25.34156186
25.33971773
25.33971773
25.33971773
25.33971773
25.33971773
25.32609079
25.32609079
25.32609079
25.32609079
25.32609079
25.28494936
25.28494936
25.28494936
25.28494936
25.28494936
25.32034664
25.32034664
25.32034664
25.32034664
25.32034664
25.32983319
25.32983319
25.32983319
25.32983319
25.32983319
25.31550178
25.31550178
25.31550178
25.31550178
25.31550178
25.27447594
25.27447594
25.27447594
25.27447594
25.27447594
25.28874348
25.28874348
25.28874348
25.28874348
25.28874348
25.32969477
25.32969477
25.32969477
25.32969477
25.32969477
25.27301912
25.27301912
25.27301912
25.27301912
25.27301912
25.34910548
25.34910548
25.34910548
25.34910548
25.34910548
25.28362348
25.28362348
25.28362348
25.28362348
25.28362348
25.33227497
25.33227497
25.33227497
25.33227497
25.33227497
25.32217724
25.32217724
25.32217724
25.32217724
25.32217724
25.32382207
25.32382207
25.32382207
25.32382207
25.32382207
25.31227277
25.31227277
25.31227277
25.31227277
25.31227277
25.31494858
25.31494858
25.31494858
25.31494858
25.31494858
25.35077176
25.35077176
25.35077176
25.35077176
25.35077176
25.30228752
25.30228752
25.30228752
25.30228752
25.30228752
25.30787579
25.30787579
25.30787579
25.30787579
25.30787579
25.2985377
25.2985377
25.2985377
25.2985377
25.2985377
25.32409671
25.32409671
25.32409671
25.32409671
25.32409671
25.32416183
25.32416183
25.32416183
25.32416183
25.32416183
25.30451643
25.30451643
25.30451643
25.30451643
25.30451643
25.30697879
25.30697879
25.30697879
25.30697879
25.30697879
25.29562914
25.29562914
25.29562914
25.29562914
25.29562914
25.28336145
25.28336145
25.28336145
25.28336145
25.28336145
25.32958346
25.32958346
25.32958346
25.32958346
25.32958346
25.33317569
25.33317569
25.33317569
25.33317569
25.33317569
25.30990014
25.30990014
25.30990014
25.30990014
25.30990014
25.32884986
25.32884986
25.32884986
25.32884986
25.32884986
25.32318672
25.32318672
25.32318672
25.32318672
25.32318672
25.32991169
25.32991169
25.32991169
25.32991169
25.32991169
25.30702206
25.30702206
25.30702206
25.30702206
25.30702206
25.29758698
25.29758698
25.29758698
25.29758698
25.29758698
25.32020452
25.32020452
25.32020452
25.32020452
25.32020452
25.32497942
25.32497942
25.32497942
25.32497942
25.32497942
25.32582465
25.32582465
25.32582465
25.32582465
25.32582465
25.33042324
25.33042324
25.33042324
25.33042324
25.33042324
25.30558477
25.30558477
25.30558477
25.30558477
25.30558477
25.30913081
25.30913081
25.30913081
25.30913081
25.30913081
25.29460501
25.29460501
25.29460501
25.29460501
25.29460501
25.32506749
25.32506749
25.32506749
25.32506749
25.32506749
25.33027707
25.33027707
25.33027707
25.33027707
25.33027707
25.30919733
25.30919733
25.30919733
25.30919733
25.30919733
25.31821917
25.31821917
25.31821917
25.31821917
25.31821917
25.33531206
25.33531206
25.33531206
25.33531206
25.33531206
25.27990647
25.27990647
25.27990647
25.27990647
25.27990647
25.32289829
25.32289829
25.32289829
25.32289829
25.32289829
25.33927258
25.33927258
25.33927258
25.33927258
25.33927258
25.29681038
25.29681038
25.29681038
25.29681038
25.29681038
25.34729403
25.34729403
25.34729403
25.34729403
25.34729403
25.30932561
25.30932561
25.30932561
25.30932561
25.30932561
25.30883669
25.30883669
25.30883669
25.30883669
25.30883669
25.3482318
25.3482318
25.3482318
25.3482318
25.3482318
25.34608539
25.34608539
25.34608539
25.34608539
25.34608539
25.32670324
25.32670324
25.32670324
25.32670324
25.32670324
25.35063287
25.35063287
25.35063287
25.35063287
25.35063287
25.30146093
25.30146093
25.30146093
25.30146093
25.30146093
25.31523051
25.31523051
25.31523051
25.31523051
25.31523051
25.34282281
25.34282281
25.34282281
25.34282281
25.34282281
25.29445054
25.29445054
25.29445054
25.29445054
25.29445054
25.3303688
25.3303688
25.3303688
25.3303688
25.3303688
25.28218043
25.28218043
25.28218043
25.28218043
25.28218043
25.3205295
25.3205295
25.3205295
25.3205295
25.3205295
25.31778018
25.31778018
25.31778018
25.31778018
25.31778018
25.33898277
25.33898277
25.33898277
25.33898277
25.33898277
25.31552456
25.31552456
25.31552456
25.31552456
25.31552456
25.28034288
25.28034288
25.28034288
25.28034288
25.28034288
25.3126086
25.3126086
25.3126086
25.3126086
25.3126086
25.29939613
25.29939613
25.29939613
25.29939613
25.29939613
25.31267767
25.31267767
25.31267767
25.31267767
25.31267767
25.29516123
25.29516123
25.29516123
25.29516123
25.29516123
25.30457035
25.30457035
25.30457035
25.30457035
25.30457035
25.33048914
25.33048914
25.33048914
25.33048914
25.33048914
25.31701642
25.31701642
25.31701642
25.31701642
25.31701642
25.32977762
25.32977762
25.32977762
25.32977762
25.32977762
25.32719596
25.32719596
25.32719596
25.32719596
25.32719596
25.32270044
25.32270044
25.32270044
25.32270044
25.32270044
25.2925714
25.2925714
25.2925714
25.2925714
25.2925714
25.29225825
25.29225825
25.29225825
25.29225825
25.29225825
25.31079229
25.31079229
25.31079229
25.31079229
25.31079229
25.30805675
25.30805675
25.30805675
25.30805675
25.30805675
25.33296568
25.33296568
25.33296568
25.33296568
25.33296568
25.32306477
25.32306477
25.32306477
25.32306477
25.32306477
25.30880014
25.30880014
25.30880014
25.30880014
25.30880014
25.31122177
25.31122177
25.31122177
25.31122177
25.31122177
25.31845685
25.31845685
25.31845685
25.31845685
25.31845685
25.36698131
25.36698131
25.36698131
25.36698131
25.36698131
25.32599936
25.32599936
25.32599936
25.32599936
25.32599936
25.35219283
25.35219283
25.35219283
25.35219283
25.35219283
25.32927748
25.32927748
25.32927748
25.32927748
25.32927748
25.33434102
25.33434102
25.33434102
25.33434102
25.33434102
25.32581383
25.32581383
25.32581383
25.32581383
25.32581383
25.32712323
25.32712323
25.32712323
25.32712323
25.32712323
25.34066612
25.34066612
25.34066612
25.34066612
25.34066612
25.36447629
25.36447629
25.36447629
25.36447629
25.36447629
25.33575161
25.33575161
25.33575161
25.33575161
25.33575161
25.30860189
25.30860189
25.30860189
25.30860189
25.30860189
25.33261895
25.33261895
25.33261895
25.33261895
25.33261895
25.32447685
25.32447685
25.32447685
25.32447685
25.32447685
25.32088157
25.32088157
25.32088157
25.32088157
25.32088157
25.33384577
25.33384577
25.33384577
25.33384577
25.33384577
25.30641882
25.30641882
25.30641882
25.30641882
25.30641882
25.35300624
25.35300624
25.35300624
25.35300624
25.35300624
25.32583192
25.32583192
25.32583192
25.32583192
25.32583192
25.30968194
25.30968194
25.30968194
25.30968194
25.30968194
25.31017935
25.31017935
25.31017935
25.31017935
25.31017935
25.34737378
25.34737378
25.34737378
25.34737378
25.34737378
25.32443143
25.32443143
25.32443143
25.32443143
25.32443143
25.29626969
25.29626969
25.29626969
25.29626969
25.29626969
25.30543983
25.30543983
25.30543983
25.30543983
25.30543983
25.31442126
25.31442126
25.31442126
25.31442126
25.31442126
25.32103344
25.32103344
25.32103344
25.32103344
25.32103344
25.31743789
25.31743789
25.31743789
25.31743789
25.31743789
25.30611029
25.30611029
25.30611029
25.30611029
25.30611029
25.34239957
25.34239957
25.34239957
25.34239957
25.34239957
25.32625583
25.32625583
25.32625583
25.32625583
25.32625583
25.28955112
25.28955112
25.28955112
25.28955112
25.28955112
25.30370756
25.30370756
25.30370756
25.30370756
25.30370756
25.32885529
25.32885529
25.32885529
25.32885529
25.32885529
25.31764039
25.31764039
25.31764039
25.31764039
25.31764039
25.30103906
25.30103906
25.30103906
25.30103906
25.30103906
25.31872372
25.31872372
25.31872372
25.31872372
25.31872372
25.31491735
25.31491735
25.31491735
25.31491735
25.31491735
25.31886451
25.31886451
25.31886451
25.31886451
25.31886451
25.31683127
25.31683127
25.31683127
25.31683127
25.31683127
25.35101615
25.35101615
25.35101615
25.35101615
25.35101615
25.31160303
25.31160303
25.31160303
25.31160303
25.31160303
25.31665676
25.31665676
25.31665676
25.31665676
25.31665676
25.32693491
25.32693491
25.32693491
25.32693491
25.32693491
25.31481732
25.31481732
25.31481732
25.31481732
25.31481732
25.34348358
25.34348358
25.34348358
25.34348358
25.34348358
25.28199434
25.28199434
25.28199434
25.28199434
25.28199434
25.27790509
25.27790509
25.27790509
25.27790509
25.27790509
25.34061087
25.34061087
25.34061087
25.34061087
25.34061087
25.31076395
25.31076395
25.31076395
25.31076395
25.31076395
25.29817458
25.29817458
25.29817458
25.29817458
25.29817458
25.33549591
25.33549591
25.33549591
25.33549591
25.33549591
25.3332215
25.3332215
25.3332215
25.3332215
25.3332215
25.30149009
25.30149009
25.30149009
25.30149009
25.30149009
25.29909672
25.29909672
25.29909672
25.29909672
25.29909672
25.29655952
25.29655952
25.29655952
25.29655952
25.29655952
25.33105995
25.33105995
25.33105995
25.33105995
25.33105995
25.30641345
25.30641345
25.30641345
25.30641345
25.30641345
25.29480073
25.29480073
25.29480073
25.29480073
25.29480073
25.3122287
25.3122287
25.3122287
25.3122287
25.3122287
25.34646439
25.34646439
25.34646439
25.34646439
25.34646439
25.33048539
25.33048539
25.33048539
25.33048539
25.33048539
25.34219286
25.34219286
25.34219286
25.34219286
25.34219286
25.29705547
25.29705547
25.29705547
25.29705547
25.29705547
25.28802723
25.28802723
25.28802723
25.28802723
25.28802723
25.34109151
25.34109151
25.34109151
25.34109151
25.34109151
25.3115465
25.3115465
25.3115465
25.3115465
25.3115465
25.28116607
25.28116607
25.28116607
25.28116607
25.28116607
25.36003566
25.36003566
25.36003566
25.36003566
25.36003566
25.35399176
25.35399176
25.35399176
25.35399176
25.35399176
25.31672295
25.31672295
25.31672295
25.31672295
25.31672295
25.33609711
25.33609711
25.33609711
25.33609711
25.33609711
25.35212053
25.35212053
25.35212053
25.35212053
25.35212053
25.28510907
25.28510907
25.28510907
25.28510907
25.28510907
25.30948187
25.30948187
25.30948187
25.30948187
25.30948187
25.32718823
25.32718823
25.32718823
25.32718823
25.32718823
25.346492
25.346492
25.346492
25.346492
25.346492
25.32034953
25.32034953
25.32034953
25.32034953
25.32034953
25.30271798
25.30271798
25.30271798
25.30271798
25.30271798
25.31852815
25.31852815
25.31852815
25.31852815
25.31852815
25.28694271
25.28694271
25.28694271
25.28694271
25.28694271
25.32355879
25.32355879
25.32355879
25.32355879
25.32355879
25.32951955
25.32951955
25.32951955
25.32951955
25.32951955
25.29406178
25.29406178
25.29406178
25.29406178
25.29406178
25.3305385
25.3305385
25.3305385
25.3305385
25.3305385
25.31706361
25.31706361
25.31706361
25.31706361
25.31706361
25.30908166
25.30908166
25.30908166
25.30908166
25.30908166
25.32841617
25.32841617
25.32841617
25.32841617
25.32841617
25.31330244
25.31330244
25.31330244
25.31330244
25.31330244
25.27568303
25.27568303
25.27568303
25.27568303
25.27568303
25.3458645
25.3458645
25.3458645
25.3458645
25.3458645
25.30695646
25.30695646
25.30695646
25.30695646
25.30695646
25.30540599
25.30540599
25.30540599
25.30540599
25.30540599
25.32723537
25.32723537
25.32723537
25.32723537
25.32723537
25.29853873
25.29853873
25.29853873
25.29853873
25.29853873
25.27463151
25.27463151
25.27463151
25.27463151
25.27463151
25.31606307
25.31606307
25.31606307
25.31606307
25.31606307
25.34678168
25.34678168
25.34678168
25.34678168
25.34678168
25.35788556
25.35788556
25.35788556
25.35788556
25.35788556
25.34256621
25.34256621
25.34256621
25.34256621
25.34256621
25.3356451
25.3356451
25.3356451
25.3356451
25.3356451
25.30886454
25.30886454
25.30886454
25.30886454
25.30886454
25.32038877
25.32038877
25.32038877
25.32038877
25.32038877
25.32959141
25.32959141
25.32959141
25.32959141
25.32959141
25.32053294
25.32053294
25.32053294
25.32053294
25.32053294
25.38434453
25.38434453
25.38434453
25.38434453
25.38434453
25.31599263
25.31599263
25.31599263
25.31599263
25.31599263
25.30286592
25.30286592
25.30286592
25.30286592
25.30286592
25.32240029
25.32240029
25.32240029
25.32240029
25.32240029
25.32839305
25.32839305
25.32839305
25.32839305
25.32839305
25.3303375
25.3303375
25.3303375
25.3303375
25.3303375
25.32588308
25.32588308
25.32588308
25.32588308
25.32588308
25.26731571
25.26731571
25.26731571
25.26731571
25.26731571
25.31550049
25.31550049
25.31550049
25.31550049
25.31550049
25.32415992
25.32415992
25.32415992
25.32415992
25.32415992
25.29938571
25.29938571
25.29938571
25.29938571
25.29938571
25.28362097
25.28362097
25.28362097
25.28362097
25.28362097
25.32292037
25.32292037
25.32292037
25.32292037
25.32292037
25.29631316
25.29631316
25.29631316
25.29631316
25.29631316
25.34161494
25.34161494
25.34161494
25.34161494
25.34161494
25.34023062
25.34023062
25.34023062
25.34023062
25.34023062
25.32040512
25.32040512
25.32040512
25.32040512
25.32040512
25.36991088
25.36991088
25.36991088
25.36991088
25.36991088
25.31709573
25.31709573
25.31709573
25.31709573
25.31709573
25.33937413
25.33937413
25.33937413
25.33937413
25.33937413
25.35481706
25.35481706
25.35481706
25.35481706
25.35481706
25.27807179
25.27807179
25.27807179
25.27807179
25.27807179
25.29436335
25.29436335
25.29436335
25.29436335
25.29436335
25.28424328
25.28424328
25.28424328
25.28424328
25.28424328
25.33762686
25.33762686
25.33762686
25.33762686
25.33762686
25.3528235
25.3528235
25.3528235
25.3528235
25.3528235
25.32441797
25.32441797
25.32441797
25.32441797
25.32441797
25.37122553
25.37122553
25.37122553
25.37122553
25.37122553
25.31633423
25.31633423
25.31633423
25.31633423
25.31633423
25.30786739
25.30786739
25.30786739
25.30786739
25.30786739
25.33234795
25.33234795
25.33234795
25.33234795
25.33234795
25.2907993
25.2907993
25.2907993
25.2907993
25.2907993
25.28081246
25.28081246
25.28081246
25.28081246
25.28081246
25.33594121
25.33594121
25.33594121
25.33594121
25.33594121
25.30517867
25.30517867
25.30517867
25.30517867
25.30517867
25.3245713
25.3245713
25.3245713
25.3245713
25.3245713
25.31522864
25.31522864
25.31522864
25.31522864
25.31522864
25.319384
25.319384
25.319384
25.319384
25.319384
25.34541729
25.34541729
25.34541729
25.34541729
25.34541729
25.33436486
25.33436486
25.33436486
25.33436486
25.33436486
25.32984375
25.32984375
25.32984375
25.32984375
25.32984375
25.29670991
25.29670991
25.29670991
25.29670991
25.29670991
25.31221935
25.31221935
25.31221935
25.31221935
25.31221935
25.31795378
25.31795378
25.31795378
25.31795378
25.31795378
25.32607395
25.32607395
25.32607395
25.32607395
25.32607395
25.33488528
25.33488528
25.33488528
25.33488528
25.33488528
25.30484393
25.30484393
25.30484393
25.30484393
25.30484393
25.33620102
25.33620102
25.33620102
25.33620102
25.33620102
25.32446119
25.32446119
25.32446119
25.32446119
25.32446119
25.34031249
25.34031249
25.34031249
25.34031249
25.34031249
25.33911462
25.33911462
25.33911462
25.33911462
25.33911462
25.31178252
25.31178252
25.31178252
25.31178252
25.31178252
25.32388154
25.32388154
25.32388154
25.32388154
25.32388154
25.29082552
25.29082552
25.29082552
25.29082552
25.29082552
25.29703338
25.29703338
25.29703338
25.29703338
25.29703338
25.3041081
25.3041081
25.3041081
25.3041081
25.3041081
25.28972805
25.28972805
25.28972805
25.28972805
25.28972805
25.34819739
25.34819739
25.34819739
25.34819739
25.34819739
25.32020918
25.32020918
25.32020918
25.32020918
25.32020918
25.31511977
25.31511977
25.31511977
25.31511977
25.31511977
25.34772788
25.34772788
25.34772788
25.34772788
25.34772788
25.32737882
25.32737882
25.32737882
25.32737882
25.32737882
25.31625617
25.31625617
25.31625617
25.31625617
25.31625617
25.32081862
25.32081862
25.32081862
25.32081862
25.32081862
25.35653117
25.35653117
25.35653117
25.35653117
25.35653117
25.32903621
25.32903621
25.32903621
25.32903621
25.32903621
25.31085527
25.31085527
25.31085527
25.31085527
25.31085527
25.32726935
25.32726935
25.32726935
25.32726935
25.32726935
25.33105151
25.33105151
25.33105151
25.33105151
25.33105151
25.32788722
25.32788722
25.32788722
25.32788722
25.32788722
25.33523564
25.33523564
25.33523564
25.33523564
25.33523564
25.28290231
25.28290231
25.28290231
25.28290231
25.28290231
25.2997168
25.2997168
25.2997168
25.2997168
25.2997168
25.36981064
25.36981064
25.36981064
25.36981064
25.36981064
25.31655191
25.31655191
25.31655191
25.31655191
25.31655191
25.2961417
25.2961417
25.2961417
25.2961417
25.2961417
25.36606785
25.36606785
25.36606785
25.36606785
25.36606785
25.31884648
25.31884648
25.31884648
25.31884648
25.31884648
25.2933681
25.2933681
25.2933681
25.2933681
25.2933681
25.339634
25.339634
25.339634
25.339634
25.339634
25.31067312
25.31067312
25.31067312
25.31067312
25.31067312
25.35987751
25.35987751
25.35987751
25.35987751
25.35987751
25.32209871
25.32209871
25.32209871
25.32209871
25.32209871
25.33802241
25.33802241
25.33802241
25.33802241
25.33802241
25.33410366
25.33410366
25.33410366
25.33410366
25.33410366
25.35590852
25.35590852
25.35590852
25.35590852
25.35590852
25.33602578
25.33602578
25.33602578
25.33602578
25.33602578
25.35727931
25.35727931
25.35727931
25.35727931
25.35727931
25.33834151
25.33834151
25.33834151
25.33834151
25.33834151
25.31244206
25.31244206
25.31244206
25.31244206
25.31244206
25.29373483
25.29373483
25.29373483
25.29373483
25.29373483
25.32805784
25.32805784
25.32805784
25.32805784
25.32805784
25.29590299
25.29590299
25.29590299
25.29590299
25.29590299
25.3285998
25.3285998
25.3285998
25.3285998
25.3285998
25.28005593
25.28005593
25.28005593
25.28005593
25.28005593
25.28557121
25.28557121
25.28557121
25.28557121
25.28557121
25.31896525
25.31896525
25.31896525
25.31896525
25.31896525
25.32156112
25.32156112
25.32156112
25.32156112
25.32156112
25.30051978
25.30051978
25.30051978
25.30051978
25.30051978
25.3694431
25.3694431
25.3694431
25.3694431
25.3694431
25.35747705
25.35747705
25.35747705
25.35747705
25.35747705
25.35187069
25.35187069
25.35187069
25.35187069
25.35187069
25.33337612
25.33337612
25.33337612
25.33337612
25.33337612
25.3549553
25.3549553
25.3549553
25.3549553
25.3549553
25.28377433
25.28377433
25.28377433
25.28377433
25.28377433
25.30482094
25.30482094
25.30482094
25.30482094
25.30482094
25.31218822
25.31218822
25.31218822
25.31218822
25.31218822
25.33330043
25.33330043
25.33330043
25.33330043
25.33330043
25.28753926
25.28753926
25.28753926
25.28753926
25.28753926
25.29591047
25.29591047
25.29591047
25.29591047
25.29591047
25.32238505
25.32238505
25.32238505
25.32238505
25.32238505
25.32319834
25.32319834
25.32319834
25.32319834
25.32319834
25.29107083
25.29107083
25.29107083
25.29107083
25.29107083
25.32316993
25.32316993
25.32316993
25.32316993
25.32316993
25.34005632
25.34005632
25.34005632
25.34005632
25.34005632
25.32751755
25.32751755
25.32751755
25.32751755
25.32751755
25.30651628
25.30651628
25.30651628
25.30651628
25.30651628
25.29361569
25.29361569
25.29361569
25.29361569
25.29361569
25.38051404
25.38051404
25.38051404
25.38051404
25.38051404
25.35583513
25.35583513
25.35583513
25.35583513
25.35583513
25.33819619
25.33819619
25.33819619
25.33819619
25.33819619
25.32765137
25.32765137
25.32765137
25.32765137
25.32765137
25.36947467
25.36947467
25.36947467
25.36947467
25.36947467
25.32447915
25.32447915
25.32447915
25.32447915
25.32447915
25.28078402
25.28078402
25.28078402
25.28078402
25.28078402
25.33333198
25.33333198
25.33333198
25.33333198
25.33333198
25.31692094
25.31692094
25.31692094
25.31692094
25.31692094
25.34097929
25.34097929
25.34097929
25.34097929
25.34097929
25.31891369
25.31891369
25.31891369
25.31891369
25.31891369
25.3127925
25.3127925
25.3127925
25.3127925
25.3127925
25.31924996
25.31924996
25.31924996
25.31924996
25.31924996
25.33078923
25.33078923
25.33078923
25.33078923
25.33078923
25.37408463
25.37408463
25.37408463
25.37408463
25.37408463
25.35305169
25.35305169
25.35305169
25.35305169
25.35305169
25.31836836
25.31836836
25.31836836
25.31836836
25.31836836
25.35374242
25.35374242
25.35374242
25.35374242
25.35374242
25.31927219
25.31927219
25.31927219
25.31927219
25.31927219
25.33068218
25.33068218
25.33068218
25.33068218
25.33068218
25.30356191
25.30356191
25.30356191
25.30356191
25.30356191
25.31613229
25.31613229
25.31613229
25.31613229
25.31613229
25.31005182
25.31005182
25.31005182
25.31005182
25.31005182
25.30307336
25.30307336
25.30307336
25.30307336
25.30307336
25.3015974
25.3015974
25.3015974
25.3015974
25.3015974
25.32353393
25.32353393
25.32353393
25.32353393
25.32353393
25.31442728
25.31442728
25.31442728
25.31442728
25.31442728
25.32784716
25.32784716
25.32784716
25.32784716
25.32784716
25.33041764
25.33041764
25.33041764
25.33041764
25.33041764
25.3385715
25.3385715
25.3385715
25.3385715
25.3385715
25.31052425
25.31052425
25.31052425
25.31052425
25.31052425
25.33353461
25.33353461
25.33353461
25.33353461
25.33353461
25.28519882
25.28519882
25.28519882
25.28519882
25.28519882
25.32888787
25.32888787
25.32888787
25.32888787
25.32888787
25.31935025
25.31935025
25.31935025
25.31935025
25.31935025
25.28255283
25.28255283
25.28255283
25.28255283
25.28255283
25.33001925
25.33001925
25.33001925
25.33001925
25.33001925
25.33946615
25.33946615
25.33946615
25.33946615
25.33946615
25.31423261
25.31423261
25.31423261
25.31423261
25.31423261
25.30629029
25.30629029
25.30629029
25.30629029
25.30629029
25.30715438
25.30715438
25.30715438
25.30715438
25.30715438
25.35655618
25.35655618
25.35655618
25.35655618
25.35655618
25.31852224
25.31852224
25.31852224
25.31852224
25.31852224
25.32413576
25.32413576
25.32413576
25.32413576
25.32413576
25.33848551
25.33848551
25.33848551
25.33848551
25.33848551
25.31153369
25.31153369
25.31153369
25.31153369
25.31153369
25.30398954
25.30398954
25.30398954
25.30398954
25.30398954
25.30397179
25.30397179
25.30397179
25.30397179
25.30397179
25.30884431
25.30884431
25.30884431
25.30884431
25.30884431
25.30979246
25.30979246
25.30979246
25.30979246
25.30979246
25.31065266
25.31065266
25.31065266
25.31065266
25.31065266
25.33806017
25.33806017
25.33806017
25.33806017
25.33806017
25.32913513
25.32913513
25.32913513
25.32913513
25.32913513
25.30366377
25.30366377
25.30366377
25.30366377
25.30366377
25.3079931
25.3079931
25.3079931
25.3079931
25.3079931
25.32284223
25.32284223
25.32284223
25.32284223
25.32284223
25.34685958
25.34685958
25.34685958
25.34685958
25.34685958
25.31927223
25.31927223
25.31927223
25.31927223
25.31927223
25.35092688
25.35092688
25.35092688
25.35092688
25.35092688
25.3293588
25.3293588
25.3293588
25.3293588
25.3293588
25.32851777
25.32851777
25.32851777
25.32851777
25.32851777
25.31779214
25.31779214
25.31779214
25.31779214
25.31779214
25.32430499
25.32430499
25.32430499
25.32430499
25.32430499
25.32991071
25.32991071
25.32991071
25.32991071
25.32991071
25.29495145
25.29495145
25.29495145
25.29495145
25.29495145
25.31323035
25.31323035
25.31323035
25.31323035
25.31323035
25.3307732
25.3307732
25.3307732
25.3307732
25.3307732
25.32979064
25.32979064
25.32979064
25.32979064
25.32979064
25.36221797
25.36221797
25.36221797
25.36221797
25.36221797
25.31378038
25.31378038
25.31378038
25.31378038
25.31378038
25.34131369
25.34131369
25.34131369
25.34131369
25.34131369
25.36414678
25.36414678
25.36414678
25.36414678
25.36414678
25.3307833
25.3307833
25.3307833
25.3307833
25.3307833
25.33587934
25.33587934
25.33587934
25.33587934
25.33587934
25.3272992
25.3272992
25.3272992
25.3272992
25.3272992
25.25914618
25.25914618
25.25914618
25.25914618
25.25914618
25.32503454
25.32503454
25.32503454
25.32503454
25.32503454
25.31804994
25.31804994
25.31804994
25.31804994
25.31804994
25.3240595
25.3240595
25.3240595
25.3240595
25.3240595
25.27398975
25.27398975
25.27398975
25.27398975
25.27398975
25.30013333
25.30013333
25.30013333
25.30013333
25.30013333
25.31251359
25.31251359
25.31251359
25.31251359
25.31251359
25.31540971
25.31540971
25.31540971
25.31540971
25.31540971
25.32885989
25.32885989
25.32885989
25.32885989
25.32885989
25.31045741
25.31045741
25.31045741
25.31045741
25.31045741
25.31443621
25.31443621
25.31443621
25.31443621
25.31443621
25.29909527
25.29909527
25.29909527
25.29909527
25.29909527
25.31479939
25.31479939
25.31479939
25.31479939
25.31479939
25.32017347
25.32017347
25.32017347
25.32017347
25.32017347
25.31667736
25.31667736
25.31667736
25.31667736
25.31667736
25.30559243
25.30559243
25.30559243
25.30559243
25.30559243
25.31768859
25.31768859
25.31768859
25.31768859
25.31768859
25.326974
25.326974
25.326974
25.326974
25.326974
25.30095062
25.30095062
25.30095062
25.30095062
25.30095062
25.33769676
25.33769676
25.33769676
25.33769676
25.33769676
25.32024653
25.32024653
25.32024653
25.32024653
25.32024653
25.35457824
25.35457824
25.35457824
25.35457824
25.35457824
25.34445002
25.34445002
25.34445002
25.34445002
25.34445002
25.30463433
25.30463433
25.30463433
25.30463433
25.30463433
25.35785973
25.35785973
25.35785973
25.35785973
25.35785973
25.32296707
25.32296707
25.32296707
25.32296707
25.32296707
25.32900114
25.32900114
25.32900114
25.32900114
25.32900114
25.3381922
25.3381922
25.3381922
25.3381922
25.3381922
25.29153065
25.29153065
25.29153065
25.29153065
25.29153065
25.3072058
25.3072058
25.3072058
25.3072058
25.3072058
25.33598899
25.33598899
25.33598899
25.33598899
25.33598899
25.31184897
25.31184897
25.31184897
25.31184897
25.31184897
25.29686743
25.29686743
25.29686743
25.29686743
25.29686743
25.32140099
25.32140099
25.32140099
25.32140099
25.32140099
25.32006455
25.32006455
25.32006455
25.32006455
25.32006455
25.30718609
25.30718609
25.30718609
25.30718609
25.30718609
25.34692136
25.34692136
25.34692136
25.34692136
25.34692136
25.34303714
25.34303714
25.34303714
25.34303714
25.34303714
25.31353261
25.31353261
25.31353261
25.31353261
25.31353261
25.34176186
25.34176186
25.34176186
25.34176186
25.34176186
25.27423406
25.27423406
25.27423406
25.27423406
25.27423406
25.2676131
25.2676131
25.2676131
25.2676131
25.2676131
25.31412698
25.31412698
25.31412698
25.31412698
25.31412698
25.32680098
25.32680098
25.32680098
25.32680098
25.32680098
25.2918826
25.2918826
25.2918826
25.2918826
25.2918826
25.31189494
25.31189494
25.31189494
25.31189494
25.31189494
25.31543003
25.31543003
25.31543003
25.31543003
25.31543003
25.33053741
25.33053741
25.33053741
25.33053741
25.33053741
25.35269894
25.35269894
25.35269894
25.35269894
25.35269894
25.30531134
25.30531134
25.30531134
25.30531134
25.30531134
25.30551967
25.30551967
25.30551967
25.30551967
25.30551967
25.32739887
25.32739887
25.32739887
25.32739887
25.32739887
25.33159799
25.33159799
25.33159799
25.33159799
25.33159799
25.32530751
25.32530751
25.32530751
25.32530751
25.32530751
25.30462308
25.30462308
25.30462308
25.30462308
25.30462308
25.28793345
25.28793345
25.28793345
25.28793345
25.28793345
25.31716588
25.31716588
25.31716588
25.31716588
25.31716588
25.31372399
25.31372399
25.31372399
25.31372399
25.31372399
25.31898453
25.31898453
25.31898453
25.31898453
25.31898453
25.34816778
25.34816778
25.34816778
25.34816778
25.34816778
25.31370772
25.31370772
25.31370772
25.31370772
25.31370772
25.32777077
25.32777077
25.32777077
25.32777077
25.32777077
25.29728695
25.29728695
25.29728695
25.29728695
25.29728695
25.2801241
25.2801241
25.2801241
25.2801241
25.2801241
25.28425293
25.28425293
25.28425293
25.28425293
25.28425293
25.34510278
25.34510278
25.34510278
25.34510278
25.34510278
25.32273547
25.32273547
25.32273547
25.32273547
25.32273547
25.34221152
25.34221152
25.34221152
25.34221152
25.34221152
25.32283649
25.32283649
25.32283649
25.32283649
25.32283649
25.31837649
25.31837649
25.31837649
25.31837649
25.31837649
25.30798272
25.30798272
25.30798272
25.30798272
25.30798272
25.34655101
25.34655101
25.34655101
25.34655101
25.34655101
25.30105915
25.30105915
25.30105915
25.30105915
25.30105915
25.2968667
25.2968667
25.2968667
25.2968667
25.2968667
25.31159589
25.31159589
25.31159589
25.31159589
25.31159589
25.31981746
25.31981746
25.31981746
25.31981746
25.31981746
25.32563751
25.32563751
25.32563751
25.32563751
25.32563751
25.33544929
25.33544929
25.33544929
25.33544929
25.33544929
25.31868324
25.31868324
25.31868324
25.31868324
25.31868324
25.35640558
25.35640558
25.35640558
25.35640558
25.35640558
25.32201732
25.32201732
25.32201732
25.32201732
25.32201732
25.3339176
25.3339176
25.3339176
25.3339176
25.3339176
25.3640829
25.3640829
25.3640829
25.3640829
25.3640829
25.34688484
25.34688484
25.34688484
25.34688484
25.34688484
25.34329414
25.34329414
25.34329414
25.34329414
25.34329414
25.27733529
25.27733529
25.27733529
25.27733529
25.27733529
25.35042726
25.35042726
25.35042726
25.35042726
25.35042726
25.33068419
25.33068419
25.33068419
25.33068419
25.33068419
25.31833213
25.31833213
25.31833213
25.31833213
25.31833213
25.29820439
25.29820439
25.29820439
25.29820439
25.29820439
25.32436087
25.32436087
25.32436087
25.32436087
25.32436087
25.25419677
25.25419677
25.25419677
25.25419677
25.25419677
25.35653375
25.35653375
25.35653375
25.35653375
25.35653375
25.33150943
25.33150943
25.33150943
25.33150943
25.33150943
25.33673913
25.33673913
25.33673913
25.33673913
25.33673913
25.34476252
25.34476252
25.34476252
25.34476252
25.34476252
25.32626586
25.32626586
25.32626586
25.32626586
25.32626586
25.33462381
25.33462381
25.33462381
25.33462381
25.33462381
25.28054868
25.28054868
25.28054868
25.28054868
25.28054868
25.35555263
25.35555263
25.35555263
25.35555263
25.35555263
25.30567306
25.30567306
25.30567306
25.30567306
25.30567306
25.30353232
25.30353232
25.30353232
25.30353232
25.30353232
25.32569448
25.32569448
25.32569448
25.32569448
25.32569448
25.32818838
25.32818838
25.32818838
25.32818838
25.32818838
25.28617804
25.28617804
25.28617804
25.28617804
25.28617804
25.31793379
25.31793379
25.31793379
25.31793379
25.31793379
25.2876699
25.2876699
25.2876699
25.2876699
25.2876699
25.33628912
25.33628912
25.33628912
25.33628912
25.33628912
25.35251723
25.35251723
25.35251723
25.35251723
25.35251723
25.34692707
25.34692707
25.34692707
25.34692707
25.34692707
25.30075958
25.30075958
25.30075958
25.30075958
25.30075958
25.31481009
25.31481009
25.31481009
25.31481009
25.31481009
25.31446637
25.31446637
25.31446637
25.31446637
25.31446637
25.30275092
25.30275092
25.30275092
25.30275092
25.30275092
25.31273583
25.31273583
25.31273583
25.31273583
25.31273583
25.34338014
25.34338014
25.34338014
25.34338014
25.34338014
25.31435716
25.31435716
25.31435716
25.31435716
25.31435716
25.28565965
25.28565965
25.28565965
25.28565965
25.28565965
25.33726637
25.33726637
25.33726637
25.33726637
25.33726637
25.28283306
25.28283306
25.28283306
25.28283306
25.28283306
25.31557532
25.31557532
25.31557532
25.31557532
25.31557532
25.30650303
25.30650303
25.30650303
25.30650303
25.30650303
25.31614479
25.31614479
25.31614479
25.31614479
25.31614479
25.31950586
25.31950586
25.31950586
25.31950586
25.31950586
25.27346394
25.27346394
25.27346394
25.27346394
25.27346394
25.37107164
25.37107164
25.37107164
25.37107164
25.37107164
25.29733871
25.29733871
25.29733871
25.29733871
25.29733871
25.31797707
25.31797707
25.31797707
25.31797707
25.31797707
25.32433395
25.32433395
25.32433395
25.32433395
25.32433395
25.30972806
25.30972806
25.30972806
25.30972806
25.30972806
25.28950286
25.28950286
25.28950286
25.28950286
25.28950286
25.31044201
25.31044201
25.31044201
25.31044201
25.31044201
25.35321464
25.35321464
25.35321464
25.35321464
25.35321464
25.31776777
25.31776777
25.31776777
25.31776777
25.31776777
25.28496379
25.28496379
25.28496379
25.28496379
25.28496379
25.29556554
25.29556554
25.29556554
25.29556554
25.29556554
25.30398383
25.30398383
25.30398383
25.30398383
25.30398383
25.3517072
25.3517072
25.3517072
25.3517072
25.3517072
25.31996923
25.31996923
25.31996923
25.31996923
25.31996923
25.30640372
25.30640372
25.30640372
25.30640372
25.30640372
25.35371695
25.35371695
25.35371695
25.35371695
25.35371695
25.3072649
25.3072649
25.3072649
25.3072649
25.3072649
25.38013712
25.38013712
25.38013712
25.38013712
25.38013712
25.32738842
25.32738842
25.32738842
25.32738842
25.32738842
25.27845379
25.27845379
25.27845379
25.27845379
25.27845379
25.31725528
25.31725528
25.31725528
25.31725528
25.31725528
25.30802383
25.30802383
25.30802383
25.30802383
25.30802383
25.29632558
25.29632558
25.29632558
25.29632558
25.29632558
25.34845833
25.34845833
25.34845833
25.34845833
25.34845833
25.3167929
25.3167929
25.3167929
25.3167929
25.3167929
25.31470851
25.31470851
25.31470851
25.31470851
25.31470851
25.31015941
25.31015941
25.31015941
25.31015941
25.31015941
25.31085646
25.31085646
25.31085646
25.31085646
25.31085646
25.32931276
25.32931276
25.32931276
25.32931276
25.32931276
25.33004722
25.33004722
25.33004722
25.33004722
25.33004722
25.38609012
25.38609012
25.38609012
25.38609012
25.38609012
25.35509716
25.35509716
25.35509716
25.35509716
25.35509716
25.32638814
25.32638814
25.32638814
25.32638814
25.32638814
25.31468896
25.31468896
25.31468896
25.31468896
25.31468896
25.35448664
25.35448664
25.35448664
25.35448664
25.35448664
25.2755193
25.2755193
25.2755193
25.2755193
25.2755193
25.28828846
25.28828846
25.28828846
25.28828846
25.28828846
25.3702528
25.3702528
25.3702528
25.3702528
25.3702528
25.30514379
25.30514379
25.30514379
25.30514379
25.30514379
25.30960622
25.30960622
25.30960622
25.30960622
25.30960622
25.35637813
25.35637813
25.35637813
25.35637813
25.35637813
25.33425956
25.33425956
25.33425956
25.33425956
25.33425956
25.30979371
25.30979371
25.30979371
25.30979371
25.30979371
25.30892025
25.30892025
25.30892025
25.30892025
25.30892025
25.33903531
25.33903531
25.33903531
25.33903531
25.33903531
25.32785041
25.32785041
25.32785041
25.32785041
25.32785041
25.34172304
25.34172304
25.34172304
25.34172304
25.34172304
25.32752907
25.32752907
25.32752907
25.32752907
25.32752907
25.3172953
25.3172953
25.3172953
25.3172953
25.3172953
25.32654556
25.32654556
25.32654556
25.32654556
25.32654556
25.30386935
25.30386935
25.30386935
25.30386935
25.30386935
25.30252356
25.30252356
25.30252356
25.30252356
25.30252356
25.30695315
25.30695315
25.30695315
25.30695315
25.30695315
25.31696239
25.31696239
25.31696239
25.31696239
25.31696239
25.31068633
25.31068633
25.31068633
25.31068633
25.31068633
25.31220279
25.31220279
25.31220279
25.31220279
25.31220279
25.32841151
25.32841151
25.32841151
25.32841151
25.32841151
25.30475298
25.30475298
25.30475298
25.30475298
25.30475298
25.35637102
25.35637102
25.35637102
25.35637102
25.35637102
25.31229603
25.31229603
25.31229603
25.31229603
25.31229603
25.30189892
25.30189892
25.30189892
25.30189892
25.30189892
25.29335441
25.29335441
25.29335441
25.29335441
25.29335441
25.33694943
25.33694943
25.33694943
25.33694943
25.33694943
25.31755388
25.31755388
25.31755388
25.31755388
25.31755388
25.33444027
25.33444027
25.33444027
25.33444027
25.33444027
25.34859392
25.34859392
25.34859392
25.34859392
25.34859392
25.30763572
25.30763572
25.30763572
25.30763572
25.30763572
25.31705958
25.31705958
25.31705958
25.31705958
25.31705958
25.31346676
25.31346676
25.31346676
25.31346676
25.31346676
25.32601094
25.32601094
25.32601094
25.32601094
25.32601094
25.34145915
25.34145915
25.34145915
25.34145915
25.34145915
25.32226481
25.32226481
25.32226481
25.32226481
25.32226481
25.30464475
25.30464475
25.30464475
25.30464475
25.30464475
25.30438207
25.30438207
25.30438207
25.30438207
25.30438207
25.31548382
25.31548382
25.31548382
25.31548382
25.31548382
25.32500456
25.32500456
25.32500456
25.32500456
25.32500456
25.3201565
25.3201565
25.3201565
25.3201565
25.3201565
25.33280876
25.33280876
25.33280876
25.33280876
25.33280876
25.31654061
25.31654061
25.31654061
25.31654061
25.31654061
25.37142367
25.37142367
25.37142367
25.37142367
25.37142367
25.34971683
25.34971683
25.34971683
25.34971683
25.34971683
25.32683831
25.32683831
25.32683831
25.32683831
25.32683831
25.2936164
25.2936164
25.2936164
25.2936164
25.2936164
25.31639568
25.31639568
25.31639568
25.31639568
25.31639568
25.32107865
25.32107865
25.32107865
25.32107865
25.32107865
25.30121296
25.30121296
25.30121296
25.30121296
25.30121296
25.33760845
25.33760845
25.33760845
25.33760845
25.33760845
25.31857622
25.31857622
25.31857622
25.31857622
25.31857622
25.32527101
25.32527101
25.32527101
25.32527101
25.32527101
25.337203
25.337203
25.337203
25.337203
25.337203
25.36634419
25.36634419
25.36634419
25.36634419
25.36634419
25.35457262
25.35457262
25.35457262
25.35457262
25.35457262
25.30930667
25.30930667
25.30930667
25.30930667
25.30930667
25.30820525
25.30820525
25.30820525
25.30820525
25.30820525
25.35441281
25.35441281
25.35441281
25.35441281
25.35441281
25.30185179
25.30185179
25.30185179
25.30185179
25.30185179
25.29262825
25.29262825
25.29262825
25.29262825
25.29262825
25.31876183
25.31876183
25.31876183
25.31876183
25.31876183
25.31991396
25.31991396
25.31991396
25.31991396
25.31991396
25.30514541
25.30514541
25.30514541
25.30514541
25.30514541
25.30296256
25.30296256
25.30296256
25.30296256
25.30296256
25.34743288
25.34743288
25.34743288
25.34743288
25.34743288
25.30842371
25.30842371
25.30842371
25.30842371
25.30842371
25.32911075
25.32911075
25.32911075
25.32911075
25.32911075
25.31285559
25.31285559
25.31285559
25.31285559
25.31285559
25.32021558
25.32021558
25.32021558
25.32021558
25.32021558
25.35263384
25.35263384
25.35263384
25.35263384
25.35263384
25.33093953
25.33093953
25.33093953
25.33093953
25.33093953
25.30196533
25.30196533
25.30196533
25.30196533
25.30196533
25.29137922
25.29137922
25.29137922
25.29137922
25.29137922
25.3270834
25.3270834
25.3270834
25.3270834
25.3270834
25.30976544
25.30976544
25.30976544
25.30976544
25.30976544
25.26436461
25.26436461
25.26436461
25.26436461
25.26436461
25.30552245
25.30552245
25.30552245
25.30552245
25.30552245
25.31368224
25.31368224
25.31368224
25.31368224
25.31368224
25.32762765
25.32762765
25.32762765
25.32762765
25.32762765
25.31882924
25.31882924
25.31882924
25.31882924
25.31882924
25.29580407
25.29580407
25.29580407
25.29580407
25.29580407
25.36698726
25.36698726
25.36698726
25.36698726
25.36698726
25.32235941
25.32235941
25.32235941
25.32235941
25.32235941
25.31677521
25.31677521
25.31677521
25.31677521
25.31677521
25.29583461
25.29583461
25.29583461
25.29583461
25.29583461
25.29384488
25.29384488
25.29384488
25.29384488
25.29384488
25.28504141
25.28504141
25.28504141
25.28504141
25.28504141
25.35567281
25.35567281
25.35567281
25.35567281
25.35567281
25.35891475
25.35891475
25.35891475
25.35891475
25.35891475
25.35416806
25.35416806
25.35416806
25.35416806
25.35416806
25.36785277
25.36785277
25.36785277
25.36785277
25.36785277
25.30531835
25.30531835
25.30531835
25.30531835
25.30531835
25.33056682
25.33056682
25.33056682
25.33056682
25.33056682
25.3696363
25.3696363
25.3696363
25.3696363
25.3696363
25.30738835
25.30738835
25.30738835
25.30738835
25.30738835
25.30204897
25.30204897
25.30204897
25.30204897
25.30204897
25.34593622
25.34593622
25.34593622
25.34593622
25.34593622
25.33410503
25.33410503
25.33410503
25.33410503
25.33410503
25.35832839
25.35832839
25.35832839
25.35832839
25.35832839
25.30520482
25.30520482
25.30520482
25.30520482
25.30520482
25.31501884
25.31501884
25.31501884
25.31501884
25.31501884
25.31802823
25.31802823
25.31802823
25.31802823
25.31802823
25.29816129
25.29816129
25.29816129
25.29816129
25.29816129
25.31057928
25.31057928
25.31057928
25.31057928
25.31057928
25.36670162
25.36670162
25.36670162
25.36670162
25.36670162
25.35528432
25.35528432
25.35528432
25.35528432
25.35528432
25.31598253
25.31598253
25.31598253
25.31598253
25.31598253
25.35694301
25.35694301
25.35694301
25.35694301
25.35694301
25.32547248
25.32547248
25.32547248
25.32547248
25.32547248
25.29533855
25.29533855
25.29533855
25.29533855
25.29533855
25.31030781
25.31030781
25.31030781
25.31030781
25.31030781
25.35143044
25.35143044
25.35143044
25.35143044
25.35143044
25.29939096
25.29939096
25.29939096
25.29939096
25.29939096
25.32258234
25.32258234
25.32258234
25.32258234
25.32258234
25.31709581
25.31709581
25.31709581
25.31709581
25.31709581
25.33363765
25.33363765
25.33363765
25.33363765
25.33363765
25.32695464
25.32695464
25.32695464
25.32695464
25.32695464
25.32040918
25.32040918
25.32040918
25.32040918
25.32040918
25.30594611
25.30594611
25.30594611
25.30594611
25.30594611
25.32339115
25.32339115
25.32339115
25.32339115
25.32339115
25.37024611
25.37024611
25.37024611
25.37024611
25.37024611
25.31629985
25.31629985
25.31629985
25.31629985
25.31629985
25.318056
25.318056
25.318056
25.318056
25.318056
25.34755788
25.34755788
25.34755788
25.34755788
25.34755788
25.34693055
25.34693055
25.34693055
25.34693055
25.34693055
25.33773493
25.33773493
25.33773493
25.33773493
25.33773493
25.3000382
25.3000382
25.3000382
25.3000382
25.3000382
25.2764514
25.2764514
25.2764514
25.2764514
25.2764514
25.29090278
25.29090278
25.29090278
25.29090278
25.29090278
25.29395063
25.29395063
25.29395063
25.29395063
25.29395063
25.29917731
25.29917731
25.29917731
25.29917731
25.29917731
25.31937805
25.31937805
25.31937805
25.31937805
25.31937805
25.34588024
25.34588024
25.34588024
25.34588024
25.34588024
25.28517504
25.28517504
25.28517504
25.28517504
25.28517504
25.28900304
25.28900304
25.28900304
25.28900304
25.28900304
25.33361372
25.33361372
25.33361372
25.33361372
25.33361372
25.27822703
25.27822703
25.27822703
25.27822703
25.27822703
25.31519327
25.31519327
25.31519327
25.31519327
25.31519327
25.36055438
25.36055438
25.36055438
25.36055438
25.36055438
25.33013301
25.33013301
25.33013301
25.33013301
25.33013301
25.32264666
25.32264666
25.32264666
25.32264666
25.32264666
25.34088657
25.34088657
25.34088657
25.34088657
25.34088657
25.27886023
25.27886023
25.27886023
25.27886023
25.27886023
25.31400418
25.31400418
25.31400418
25.31400418
25.31400418
25.322212
25.322212
25.322212
25.322212
25.322212
25.32320473
25.32320473
25.32320473
25.32320473
25.32320473
25.32456252
25.32456252
25.32456252
25.32456252
25.32456252
25.30987655
25.30987655
25.30987655
25.30987655
25.30987655
25.29244192
25.29244192
25.29244192
25.29244192
25.29244192
25.30792116
25.30792116
25.30792116
25.30792116
25.30792116
25.27166463
25.27166463
25.27166463
25.27166463
25.27166463
25.34562198
25.34562198
25.34562198
25.34562198
25.34562198
25.3373967
25.3373967
25.3373967
25.3373967
25.3373967
25.32447691
25.32447691
25.32447691
25.32447691
25.32447691
25.35681603
25.35681603
25.35681603
25.35681603
25.35681603
25.31977247
25.31977247
25.31977247
25.31977247
25.31977247
25.35089124
25.35089124
25.35089124
25.35089124
25.35089124
25.32588405
25.32588405
25.32588405
25.32588405
25.32588405
25.36317899
25.36317899
25.36317899
25.36317899
25.36317899
25.33573187
25.33573187
25.33573187
25.33573187
25.33573187
25.30981578
25.30981578
25.30981578
25.30981578
25.30981578
25.29258805
25.29258805
25.29258805
25.29258805
25.29258805
25.33240469
25.33240469
25.33240469
25.33240469
25.33240469
25.35012668
25.35012668
25.35012668
25.35012668
25.35012668
25.31926505
25.31926505
25.31926505
25.31926505
25.31926505
25.32079686
25.32079686
25.32079686
25.32079686
25.32079686
25.28979499
25.28979499
25.28979499
25.28979499
25.28979499
25.34987257
25.34987257
25.34987257
25.34987257
25.34987257
25.29463223
25.29463223
25.29463223
25.29463223
25.29463223
25.32723311
25.32723311
25.32723311
25.32723311
25.32723311
25.32310359
25.32310359
25.32310359
25.32310359
25.32310359
25.31551654
25.31551654
25.31551654
25.31551654
25.31551654
25.32917034
25.32917034
25.32917034
25.32917034
25.32917034
25.34925333
25.34925333
25.34925333
25.34925333
25.34925333
25.32588151
25.32588151
25.32588151
25.32588151
25.32588151
25.27268076
25.27268076
25.27268076
25.27268076
25.27268076
25.35465394
25.35465394
25.35465394
25.35465394
25.35465394
25.29830251
25.29830251
25.29830251
25.29830251
25.29830251
25.28988778
25.28988778
25.28988778
25.28988778
25.28988778
25.32802614
25.32802614
25.32802614
25.32802614
25.32802614
25.32647046
25.32647046
25.32647046
25.32647046
25.32647046
25.30484435
25.30484435
25.30484435
25.30484435
25.30484435
25.32755193
25.32755193
25.32755193
25.32755193
25.32755193
25.31680516
25.31680516
25.31680516
25.31680516
25.31680516
25.34657865
25.34657865
25.34657865
25.34657865
25.34657865
25.30990131
25.30990131
25.30990131
25.30990131
25.30990131
25.31195025
25.31195025
25.31195025
25.31195025
25.31195025
25.28215206
25.28215206
25.28215206
25.28215206
25.28215206
25.31933283
25.31933283
25.31933283
25.31933283
25.31933283
25.33133126
25.33133126
25.33133126
25.33133126
25.33133126
25.31936981
25.31936981
25.31936981
25.31936981
25.31936981
25.3330854
25.3330854
25.3330854
25.3330854
25.3330854
25.32112445
25.32112445
25.32112445
25.32112445
25.32112445
25.35345909
25.35345909
25.35345909
25.35345909
25.35345909
25.31629355
25.31629355
25.31629355
25.31629355
25.31629355
25.2929077
25.2929077
25.2929077
25.2929077
25.2929077
25.26540259
25.26540259
25.26540259
25.26540259
25.26540259
25.27854601
25.27854601
25.27854601
25.27854601
25.27854601
25.33157953
25.33157953
25.33157953
25.33157953
25.33157953
25.33057403
25.33057403
25.33057403
25.33057403
25.33057403
25.31632889
25.31632889
25.31632889
25.31632889
25.31632889
25.29894933
25.29894933
25.29894933
25.29894933
25.29894933
25.31477487
25.31477487
25.31477487
25.31477487
25.31477487
25.30352743
25.30352743
25.30352743
25.30352743
25.30352743
25.30584246
25.30584246
25.30584246
25.30584246
25.30584246
25.34559745
25.34559745
25.34559745
25.34559745
25.34559745
25.29278749
25.29278749
25.29278749
25.29278749
25.29278749
25.32236831
25.32236831
25.32236831
25.32236831
25.32236831
25.31135516
25.31135516
25.31135516
25.31135516
25.31135516
25.32167901
25.32167901
25.32167901
25.32167901
25.32167901
25.31284837
25.31284837
25.31284837
25.31284837
25.31284837
25.30139386
25.30139386
25.30139386
25.30139386
25.30139386
25.3012154
25.3012154
25.3012154
25.3012154
25.3012154
25.3274088
25.3274088
25.3274088
25.3274088
25.3274088
25.324564
25.324564
25.324564
25.324564
25.324564
25.33434518
25.33434518
25.33434518
25.33434518
25.33434518
25.35384109
25.35384109
25.35384109
25.35384109
25.35384109
25.29566008
25.29566008
25.29566008
25.29566008
25.29566008
25.29502187
25.29502187
25.29502187
25.29502187
25.29502187
25.33520008
25.33520008
25.33520008
25.33520008
25.33520008
25.29271937
25.29271937
25.29271937
25.29271937
25.29271937
25.32355813
25.32355813
25.32355813
25.32355813
25.32355813
25.29436097
25.29436097
25.29436097
25.29436097
25.29436097
25.31840582
25.31840582
25.31840582
25.31840582
25.31840582
25.32446229
25.32446229
25.32446229
25.32446229
25.32446229
25.32091008
25.32091008
25.32091008
25.32091008
25.32091008
25.35910045
25.35910045
25.35910045
25.35910045
25.35910045
25.28704997
25.28704997
25.28704997
25.28704997
25.28704997
25.34981888
25.34981888
25.34981888
25.34981888
25.34981888
25.32043938
25.32043938
25.32043938
25.32043938
25.32043938
25.32912231
25.32912231
25.32912231
25.32912231
25.32912231
25.33235735
25.33235735
25.33235735
25.33235735
25.33235735
25.36984741
25.36984741
25.36984741
25.36984741
25.36984741
25.3089822
25.3089822
25.3089822
25.3089822
25.3089822
25.33728833
25.33728833
25.33728833
25.33728833
25.33728833
25.32676451
25.32676451
25.32676451
25.32676451
25.32676451
25.27204882
25.27204882
25.27204882
25.27204882
25.27204882
25.30255556
25.30255556
25.30255556
25.30255556
25.30255556
25.27736932
25.27736932
25.27736932
25.27736932
25.27736932
25.3408062
25.3408062
25.3408062
25.3408062
25.3408062
25.32247319
25.32247319
25.32247319
25.32247319
25.32247319
25.32501998
25.32501998
25.32501998
25.32501998
25.32501998
25.32144056
25.32144056
25.32144056
25.32144056
25.32144056
25.32749991
25.32749991
25.32749991
25.32749991
25.32749991
25.3545209
25.3545209
25.3545209
25.3545209
25.3545209
25.34191102
25.34191102
25.34191102
25.34191102
25.34191102
25.31770995
25.31770995
25.31770995
25.31770995
25.31770995
25.3292554
25.3292554
25.3292554
25.3292554
25.3292554
25.31829557
25.31829557
25.31829557
25.31829557
25.31829557
25.32976984
25.32976984
25.32976984
25.32976984
25.32976984
25.35763347
25.35763347
25.35763347
25.35763347
25.35763347
25.33608931
25.33608931
25.33608931
25.33608931
25.33608931
25.29246236
25.29246236
25.29246236
25.29246236
25.29246236
25.3170126
25.3170126
25.3170126
25.3170126
25.3170126
25.32414933
25.32414933
25.32414933
25.32414933
25.32414933
25.33085913
25.33085913
25.33085913
25.33085913
25.33085913
25.32963075
25.32963075
25.32963075
25.32963075
25.32963075
25.31072822
25.31072822
25.31072822
25.31072822
25.31072822
25.3571219
25.3571219
25.3571219
25.3571219
25.3571219
25.32293028
25.32293028
25.32293028
25.32293028
25.32293028
25.3213241
25.3213241
25.3213241
25.3213241
25.3213241
25.29490093
25.29490093
25.29490093
25.29490093
25.29490093
25.34300167
25.34300167
25.34300167
25.34300167
25.34300167
25.30009245
25.30009245
25.30009245
25.30009245
25.30009245
25.32919349
25.32919349
25.32919349
25.32919349
25.32919349
25.33747859
25.33747859
25.33747859
25.33747859
25.33747859
25.31274374
25.31274374
25.31274374
25.31274374
25.31274374
25.32401417
25.32401417
25.32401417
25.32401417
25.32401417
25.29991238
25.29991238
25.29991238
25.29991238
25.29991238
25.3054727
25.3054727
25.3054727
25.3054727
25.3054727
25.30664731
25.30664731
25.30664731
25.30664731
25.30664731
25.32701122
25.32701122
25.32701122
25.32701122
25.32701122
25.34592814
25.34592814
25.34592814
25.34592814
25.34592814
25.37229008
25.37229008
25.37229008
25.37229008
25.37229008
25.29974104
25.29974104
25.29974104
25.29974104
25.29974104
25.35581024
25.35581024
25.35581024
25.35581024
25.35581024
25.32204332
25.32204332
25.32204332
25.32204332
25.32204332
25.32471716
25.32471716
25.32471716
25.32471716
25.32471716
25.28108799
25.28108799
25.28108799
25.28108799
25.28108799
25.3056364
25.3056364
25.3056364
25.3056364
25.3056364
25.32014427
25.32014427
25.32014427
25.32014427
25.32014427
25.32957249
25.32957249
25.32957249
25.32957249
25.32957249
25.30653253
25.30653253
25.30653253
25.30653253
25.30653253
25.29216527
25.29216527
25.29216527
25.29216527
25.29216527
25.34917777
25.34917777
25.34917777
25.34917777
25.34917777
25.30261313
25.30261313
25.30261313
25.30261313
25.30261313
25.35408106
25.35408106
25.35408106
25.35408106
25.35408106
25.29329078
25.29329078
25.29329078
25.29329078
25.29329078
25.27759784
25.27759784
25.27759784
25.27759784
25.27759784
25.30556677
25.30556677
25.30556677
25.30556677
25.30556677
25.32281463
25.32281463
25.32281463
25.32281463
25.32281463
25.35171909
25.35171909
25.35171909
25.35171909
25.35171909
25.3283423
25.3283423
25.3283423
25.3283423
25.3283423
25.32261792
25.32261792
25.32261792
25.32261792
25.32261792
25.34721567
25.34721567
25.34721567
25.34721567
25.34721567
25.32702093
25.32702093
25.32702093
25.32702093
25.32702093
25.32926783
25.32926783
25.32926783
25.32926783
25.32926783
25.3583482
25.3583482
25.3583482
25.3583482
25.3583482
25.31918981
25.31918981
25.31918981
25.31918981
25.31918981
25.31442175
25.31442175
25.31442175
25.31442175
25.31442175
25.29065171
25.29065171
25.29065171
25.29065171
25.29065171
25.34777482
25.34777482
25.34777482
25.34777482
25.34777482
25.32741866
25.32741866
25.32741866
25.32741866
25.32741866
25.32671545
25.32671545
25.32671545
25.32671545
25.32671545
25.31591099
25.31591099
25.31591099
25.31591099
25.31591099
25.28078828
25.28078828
25.28078828
25.28078828
25.28078828
25.32147449
25.32147449
25.32147449
25.32147449
25.32147449
25.34707124
25.34707124
25.34707124
25.34707124
25.34707124
25.33694822
25.33694822
25.33694822
25.33694822
25.33694822
25.31809991
25.31809991
25.31809991
25.31809991
25.31809991
25.28216857
25.28216857
25.28216857
25.28216857
25.28216857
25.3093612
25.3093612
25.3093612
25.3093612
25.3093612
25.30323212
25.30323212
25.30323212
25.30323212
25.30323212
25.31720676
25.31720676
25.31720676
25.31720676
25.31720676
25.35371757
25.35371757
25.35371757
25.35371757
25.35371757
25.28164949
25.28164949
25.28164949
25.28164949
25.28164949
25.33762114
25.33762114
25.33762114
25.33762114
25.33762114
25.31506581
25.31506581
25.31506581
25.31506581
25.31506581
25.35924042
25.35924042
25.35924042
25.35924042
25.35924042
25.32303163
25.32303163
25.32303163
25.32303163
25.32303163
25.31498997
25.31498997
25.31498997
25.31498997
25.31498997
25.33668106
25.33668106
25.33668106
25.33668106
25.33668106
25.28035288
25.28035288
25.28035288
25.28035288
25.28035288
25.32346062
25.32346062
25.32346062
25.32346062
25.32346062
25.36634821
25.36634821
25.36634821
25.36634821
25.36634821
25.32870884
25.32870884
25.32870884
25.32870884
25.32870884
25.30660417
25.30660417
25.30660417
25.30660417
25.30660417
25.35729857
25.35729857
25.35729857
25.35729857
25.35729857
25.33087245
25.33087245
25.33087245
25.33087245
25.33087245
25.2899329
25.2899329
25.2899329
25.2899329
25.2899329
25.3206774
25.3206774
25.3206774
25.3206774
25.3206774
25.33195956
25.33195956
25.33195956
25.33195956
25.33195956
25.32138837
25.32138837
25.32138837
25.32138837
25.32138837
25.3182365
25.3182365
25.3182365
25.3182365
25.3182365
25.31529989
25.31529989
25.31529989
25.31529989
25.31529989
25.33027543
25.33027543
25.33027543
25.33027543
25.33027543
25.34660894
25.34660894
25.34660894
25.34660894
25.34660894
25.30425874
25.30425874
25.30425874
25.30425874
25.30425874
25.31270605
25.31270605
25.31270605
25.31270605
25.31270605
25.32781633
25.32781633
25.32781633
25.32781633
25.32781633
25.36496588
25.36496588
25.36496588
25.36496588
25.36496588
25.3169445
25.3169445
25.3169445
25.3169445
25.3169445
25.32395845
25.32395845
25.32395845
25.32395845
25.32395845
25.32149634
25.32149634
25.32149634
25.32149634
25.32149634
25.29605377
25.29605377
25.29605377
25.29605377
25.29605377
25.29798657
25.29798657
25.29798657
25.29798657
25.29798657
25.36606479
25.36606479
25.36606479
25.36606479
25.36606479
25.3077987
25.3077987
25.3077987
25.3077987
25.3077987
25.28717681
25.28717681
25.28717681
25.28717681
25.28717681
25.28086213
25.28086213
25.28086213
25.28086213
25.28086213
25.30080253
25.30080253
25.30080253
25.30080253
25.30080253
25.33107631
25.33107631
25.33107631
25.33107631
25.33107631
25.32307765
25.32307765
25.32307765
25.32307765
25.32307765
25.35336994
25.35336994
25.35336994
25.35336994
25.35336994
25.33504487
25.33504487
25.33504487
25.33504487
25.33504487
25.30066559
25.30066559
25.30066559
25.30066559
25.30066559
25.35630238
25.35630238
25.35630238
25.35630238
25.35630238
25.31303548
25.31303548
25.31303548
25.31303548
25.31303548
25.28267798
25.28267798
25.28267798
25.28267798
25.28267798
25.327016
25.327016
25.327016
25.327016
25.327016
25.3560559
25.3560559
25.3560559
25.3560559
25.3560559
25.35102971
25.35102971
25.35102971
25.35102971
25.35102971
25.31657778
25.31657778
25.31657778
25.31657778
25.31657778
25.31520212
25.31520212
25.31520212
25.31520212
25.31520212
25.35489881
25.35489881
25.35489881
25.35489881
25.35489881
25.32945937
25.32945937
25.32945937
25.32945937
25.32945937
25.32929643
25.32929643
25.32929643
25.32929643
25.32929643
25.33734901
25.33734901
25.33734901
25.33734901
25.33734901
25.36632443
25.36632443
25.36632443
25.36632443
25.36632443
25.33658364
25.33658364
25.33658364
25.33658364
25.33658364
25.32142629
25.32142629
25.32142629
25.32142629
25.32142629
25.36434366
25.36434366
25.36434366
25.36434366
25.36434366
25.28139629
25.28139629
25.28139629
25.28139629
25.28139629
25.31404249
25.31404249
25.31404249
25.31404249
25.31404249
25.3357155
25.3357155
25.3357155
25.3357155
25.3357155
25.30405423
25.30405423
25.30405423
25.30405423
25.30405423
25.27719316
25.27719316
25.27719316
25.27719316
25.27719316
25.31560289
25.31560289
25.31560289
25.31560289
25.31560289
25.31827228
25.31827228
25.31827228
25.31827228
25.31827228
25.29109568
25.29109568
25.29109568
25.29109568
25.29109568
25.32721598
25.32721598
25.32721598
25.32721598
25.32721598
25.31050464
25.31050464
25.31050464
25.31050464
25.31050464
25.30353337
25.30353337
25.30353337
25.30353337
25.30353337
25.36895318
25.36895318
25.36895318
25.36895318
25.36895318
25.32695809
25.32695809
25.32695809
25.32695809
25.32695809
25.3478353
25.3478353
25.3478353
25.3478353
25.3478353
25.30051735
25.30051735
25.30051735
25.30051735
25.30051735
25.31608941
25.31608941
25.31608941
25.31608941
25.31608941
25.32872843
25.32872843
25.32872843
25.32872843
25.32872843
25.31686984
25.31686984
25.31686984
25.31686984
25.31686984
25.28681488
25.28681488
25.28681488
25.28681488
25.28681488
25.30420072
25.30420072
25.30420072
25.30420072
25.30420072
25.31715203
25.31715203
25.31715203
25.31715203
25.31715203
25.32458729
25.32458729
25.32458729
25.32458729
25.32458729
25.28745955
25.28745955
25.28745955
25.28745955
25.28745955
25.30533459
25.30533459
25.30533459
25.30533459
25.30533459
25.35563989
25.35563989
25.35563989
25.35563989
25.35563989
25.32063817
25.32063817
25.32063817
25.32063817
25.32063817
25.34143418
25.34143418
25.34143418
25.34143418
25.34143418
25.34979624
25.34979624
25.34979624
25.34979624
25.34979624
25.35569625
25.35569625
25.35569625
25.35569625
25.35569625
25.32421768
25.32421768
25.32421768
25.32421768
25.32421768
25.34827018
25.34827018
25.34827018
25.34827018
25.34827018
25.29428981
25.29428981
25.29428981
25.29428981
25.29428981
25.33870466
25.33870466
25.33870466
25.33870466
25.33870466
25.33157532
25.33157532
25.33157532
25.33157532
25.33157532
25.34107429
25.34107429
25.34107429
25.34107429
25.34107429
25.33581126
25.33581126
25.33581126
25.33581126
25.33581126
25.29300137
25.29300137
25.29300137
25.29300137
25.29300137
25.30605865
25.30605865
25.30605865
25.30605865
25.30605865
25.33111711
25.33111711
25.33111711
25.33111711
25.33111711
25.3389887
25.3389887
25.3389887
25.3389887
25.3389887
25.29762535
25.29762535
25.29762535
25.29762535
25.29762535
25.31614685
25.31614685
25.31614685
25.31614685
25.31614685
25.35549111
25.35549111
25.35549111
25.35549111
25.35549111
25.28037965
25.28037965
25.28037965
25.28037965
25.28037965
25.30234451
25.30234451
25.30234451
25.30234451
25.30234451
25.29097896
25.29097896
25.29097896
25.29097896
25.29097896
25.36704856
25.36704856
25.36704856
25.36704856
25.36704856
25.34703349
25.34703349
25.34703349
25.34703349
25.34703349
25.33264567
25.33264567
25.33264567
25.33264567
25.33264567
25.3223138
25.3223138
25.3223138
25.3223138
25.3223138
25.27657127
25.27657127
25.27657127
25.27657127
25.27657127
25.31499833
25.31499833
25.31499833
25.31499833
25.31499833
25.2916902
25.2916902
25.2916902
25.2916902
25.2916902
25.36288515
25.36288515
25.36288515
25.36288515
25.36288515
25.27525864
25.27525864
25.27525864
25.27525864
25.27525864
25.32415562
25.32415562
25.32415562
25.32415562
25.32415562
25.32928135
25.32928135
25.32928135
25.32928135
25.32928135
25.33377658
25.33377658
25.33377658
25.33377658
25.33377658
25.31252917
25.31252917
25.31252917
25.31252917
25.31252917
25.31853561
25.31853561
25.31853561
25.31853561
25.31853561
25.29534196
25.29534196
25.29534196
25.29534196
25.29534196
25.31192149
25.31192149
25.31192149
25.31192149
25.31192149
25.32798973
25.32798973
25.32798973
25.32798973
25.32798973
25.30565471
25.30565471
25.30565471
25.30565471
25.30565471
25.31442031
25.31442031
25.31442031
25.31442031
25.31442031
25.32599149
25.32599149
25.32599149
25.32599149
25.32599149
25.34961712
25.34961712
25.34961712
25.34961712
25.34961712
25.33158199
25.33158199
25.33158199
25.33158199
25.33158199
25.3470763
25.3470763
25.3470763
25.3470763
25.3470763
25.34962068
25.34962068
25.34962068
25.34962068
25.34962068
25.31224585
25.31224585
25.31224585
25.31224585
25.31224585
25.28232058
25.28232058
25.28232058
25.28232058
25.28232058
25.28909228
25.28909228
25.28909228
25.28909228
25.28909228
25.35452507
25.35452507
25.35452507
25.35452507
25.35452507
25.28087633
25.28087633
25.28087633
25.28087633
25.28087633
25.34745234
25.34745234
25.34745234
25.34745234
25.34745234
25.31418069
25.31418069
25.31418069
25.31418069
25.31418069
25.31559778
25.31559778
25.31559778
25.31559778
25.31559778
25.33062649
25.33062649
25.33062649
25.33062649
25.33062649
25.30958203
25.30958203
25.30958203
25.30958203
25.30958203
25.31826076
25.31826076
25.31826076
25.31826076
25.31826076
25.29364883
25.29364883
25.29364883
25.29364883
25.29364883
25.33036224
25.33036224
25.33036224
25.33036224
25.33036224
25.32330583
25.32330583
25.32330583
25.32330583
25.32330583
25.31853879
25.31853879
25.31853879
25.31853879
25.31853879
25.32698605
25.32698605
25.32698605
25.32698605
25.32698605
25.30825606
25.30825606
25.30825606
25.30825606
25.30825606
25.32759626
25.32759626
25.32759626
25.32759626
25.32759626
25.28790871
25.28790871
25.28790871
25.28790871
25.28790871
25.31820194
25.31820194
25.31820194
25.31820194
25.31820194
25.36510348
25.36510348
25.36510348
25.36510348
25.36510348
25.31590462
25.31590462
25.31590462
25.31590462
25.31590462
25.32175317
25.32175317
25.32175317
25.32175317
25.32175317
25.30595201
25.30595201
25.30595201
25.30595201
25.30595201
25.32587011
25.32587011
25.32587011
25.32587011
25.32587011
25.31814024
25.31814024
25.31814024
25.31814024
25.31814024
25.34762785
25.34762785
25.34762785
25.34762785
25.34762785
25.27742884
25.27742884
25.27742884
25.27742884
25.27742884
25.33860671
25.33860671
25.33860671
25.33860671
25.33860671
25.32447938
25.32447938
25.32447938
25.32447938
25.32447938
25.3484635
25.3484635
25.3484635
25.3484635
25.3484635
25.32246642
25.32246642
25.32246642
25.32246642
25.32246642
25.34718934
25.34718934
25.34718934
25.34718934
25.34718934
25.29101448
25.29101448
25.29101448
25.29101448
25.29101448
25.31808327
25.31808327
25.31808327
25.31808327
25.31808327
25.30554256
25.30554256
25.30554256
25.30554256
25.30554256
25.35870939
25.35870939
25.35870939
25.35870939
25.35870939
25.31499158
25.31499158
25.31499158
25.31499158
25.31499158
25.32946888
25.32946888
25.32946888
25.32946888
25.32946888
25.3193436
25.3193436
25.3193436
25.3193436
25.3193436
25.30181406
25.30181406
25.30181406
25.30181406
25.30181406
25.33487631
25.33487631
25.33487631
25.33487631
25.33487631
25.28133335
25.28133335
25.28133335
25.28133335
25.28133335
25.27804782
25.27804782
25.27804782
25.27804782
25.27804782
25.35482001
25.35482001
25.35482001
25.35482001
25.35482001
25.32869615
25.32869615
25.32869615
25.32869615
25.32869615
25.32890158
25.32890158
25.32890158
25.32890158
25.32890158
25.29566686
25.29566686
25.29566686
25.29566686
25.29566686
25.31422063
25.31422063
25.31422063
25.31422063
25.31422063
25.3511602
25.3511602
25.3511602
25.3511602
25.3511602
25.31793358
25.31793358
25.31793358
25.31793358
25.31793358
25.30580374
25.30580374
25.30580374
25.30580374
25.30580374
25.29499233
25.29499233
25.29499233
25.29499233
25.29499233
25.31783627
25.31783627
25.31783627
25.31783627
25.31783627
25.29005409
25.29005409
25.29005409
25.29005409
25.29005409
25.33035934
25.33035934
25.33035934
25.33035934
25.33035934
25.32283329
25.32283329
25.32283329
25.32283329
25.32283329
25.37041451
25.37041451
25.37041451
25.37041451
25.37041451
25.29006259
25.29006259
25.29006259
25.29006259
25.29006259
25.31897645
25.31897645
25.31897645
25.31897645
25.31897645
25.3555842
25.3555842
25.3555842
25.3555842
25.3555842
25.31598338
25.31598338
25.31598338
25.31598338
25.31598338
25.33117729
25.33117729
25.33117729
25.33117729
25.33117729
25.34069529
25.34069529
25.34069529
25.34069529
25.34069529
25.34891313
25.34891313
25.34891313
25.34891313
25.34891313
25.31676044
25.31676044
25.31676044
25.31676044
25.31676044
25.32531546
25.32531546
25.32531546
25.32531546
25.32531546
25.34189216
25.34189216
25.34189216
25.34189216
25.34189216
25.32081468
25.32081468
25.32081468
25.32081468
25.32081468
25.32945706
25.32945706
25.32945706
25.32945706
25.32945706
25.32723098
25.32723098
25.32723098
25.32723098
25.32723098
25.3265391
25.3265391
25.3265391
25.3265391
25.3265391
25.29516842
25.29516842
25.29516842
25.29516842
25.29516842
25.2699262
25.2699262
25.2699262
25.2699262
25.2699262
25.31882014
25.31882014
25.31882014
25.31882014
25.31882014
25.3203854
25.3203854
25.3203854
25.3203854
25.3203854
25.29389504
25.29389504
25.29389504
25.29389504
25.29389504
25.36859861
25.36859861
25.36859861
25.36859861
25.36859861
25.31120156
25.31120156
25.31120156
25.31120156
25.31120156
25.3485428
25.3485428
25.3485428
25.3485428
25.3485428
25.2829428
25.2829428
25.2829428
25.2829428
25.2829428
25.29394778
25.29394778
25.29394778
25.29394778
25.29394778
25.3067507
25.3067507
25.3067507
25.3067507
25.3067507
25.32514884
25.32514884
25.32514884
25.32514884
25.32514884
25.29668354
25.29668354
25.29668354
25.29668354
25.29668354
25.32999896
25.32999896
25.32999896
25.32999896
25.32999896
25.30408549
25.30408549
25.30408549
25.30408549
25.30408549
25.34588199
25.34588199
25.34588199
25.34588199
25.34588199
25.30484427
25.30484427
25.30484427
25.30484427
25.30484427
25.31655025
25.31655025
25.31655025
25.31655025
25.31655025
25.31670581
25.31670581
25.31670581
25.31670581
25.31670581
25.34126427
25.34126427
25.34126427
25.34126427
25.34126427
25.34039997
25.34039997
25.34039997
25.34039997
25.34039997
25.33465061
25.33465061
25.33465061
25.33465061
25.33465061
25.31936129
25.31936129
25.31936129
25.31936129
25.31936129
25.29073697
25.29073697
25.29073697
25.29073697
25.29073697
25.2926919
25.2926919
25.2926919
25.2926919
25.2926919
25.33102509
25.33102509
25.33102509
25.33102509
25.33102509
25.34584946
25.34584946
25.34584946
25.34584946
25.34584946
25.32537223
25.32537223
25.32537223
25.32537223
25.32537223
25.32749168
25.32749168
25.32749168
25.32749168
25.32749168
25.32947942
25.32947942
25.32947942
25.32947942
25.32947942
25.32039529
25.32039529
25.32039529
25.32039529
25.32039529
25.34616623
25.34616623
25.34616623
25.34616623
25.34616623
25.29124901
25.29124901
25.29124901
25.29124901
25.29124901
25.34447689
25.34447689
25.34447689
25.34447689
25.34447689
25.33934595
25.33934595
25.33934595
25.33934595
25.33934595
25.3007565
25.3007565
25.3007565
25.3007565
25.3007565
25.324802
25.324802
25.324802
25.324802
25.324802
25.30743073
25.30743073
25.30743073
25.30743073
25.30743073
25.33566592
25.33566592
25.33566592
25.33566592
25.33566592
25.31946479
25.31946479
25.31946479
25.31946479
25.31946479
25.3207903
25.3207903
25.3207903
25.3207903
25.3207903
25.320777
25.320777
25.320777
25.320777
25.320777
25.31483084
25.31483084
25.31483084
25.31483084
25.31483084
25.31508394
25.31508394
25.31508394
25.31508394
25.31508394
25.3393462
25.3393462
25.3393462
25.3393462
25.3393462
25.3186367
25.3186367
25.3186367
25.3186367
25.3186367
25.29133957
25.29133957
25.29133957
25.29133957
25.29133957
25.31817966
25.31817966
25.31817966
25.31817966
25.31817966
25.32073486
25.32073486
25.32073486
25.32073486
25.32073486
25.2930107
25.2930107
25.2930107
25.2930107
25.2930107
25.33808598
25.33808598
25.33808598
25.33808598
25.33808598
25.33904838
25.33904838
25.33904838
25.33904838
25.33904838
25.31930237
25.31930237
25.31930237
25.31930237
25.31930237
25.31341766
25.31341766
25.31341766
25.31341766
25.31341766
25.35291094
25.35291094
25.35291094
25.35291094
25.35291094
25.31402728
25.31402728
25.31402728
25.31402728
25.31402728
25.30202929
25.30202929
25.30202929
25.30202929
25.30202929
25.31207858
25.31207858
25.31207858
25.31207858
25.31207858
25.32233582
25.32233582
25.32233582
25.32233582
25.32233582
25.35577743
25.35577743
25.35577743
25.35577743
25.35577743
25.34776926
25.34776926
25.34776926
25.34776926
25.34776926
25.32498031
25.32498031
25.32498031
25.32498031
25.32498031
25.30487919
25.30487919
25.30487919
25.30487919
25.30487919
25.32863117
25.32863117
25.32863117
25.32863117
25.32863117
25.34638274
25.34638274
25.34638274
25.34638274
25.34638274
25.33030203
25.33030203
25.33030203
25.33030203
25.33030203
25.3216846
25.3216846
25.3216846
25.3216846
25.3216846
25.32688817
25.32688817
25.32688817
25.32688817
25.32688817
25.31359451
25.31359451
25.31359451
25.31359451
25.31359451
25.31032364
25.31032364
25.31032364
25.31032364
25.31032364
25.31720488
25.31720488
25.31720488
25.31720488
25.31720488
25.33812923
25.33812923
25.33812923
25.33812923
25.33812923
25.31615677
25.31615677
25.31615677
25.31615677
25.31615677
25.34664002
25.34664002
25.34664002
25.34664002
25.34664002
25.33688641
25.33688641
25.33688641
25.33688641
25.33688641
25.36497838
25.36497838
25.36497838
25.36497838
25.36497838
25.32639951
25.32639951
25.32639951
25.32639951
25.32639951
25.36459377
25.36459377
25.36459377
25.36459377
25.36459377
25.34757432
25.34757432
25.34757432
25.34757432
25.34757432
25.32215132
25.32215132
25.32215132
25.32215132
25.32215132
25.34621205
25.34621205
25.34621205
25.34621205
25.34621205
25.302725
25.302725
25.302725
25.302725
25.302725
25.29938029
25.29938029
25.29938029
25.29938029
25.29938029
25.33380794
25.33380794
25.33380794
25.33380794
25.33380794
25.32760898
25.32760898
25.32760898
25.32760898
25.32760898
25.33024538
25.33024538
25.33024538
25.33024538
25.33024538
25.30758552
25.30758552
25.30758552
25.30758552
25.30758552
25.32284832
25.32284832
25.32284832
25.32284832
25.32284832
25.30554155
25.30554155
25.30554155
25.30554155
25.30554155
25.35297788
25.35297788
25.35297788
25.35297788
25.35297788
25.31205264
25.31205264
25.31205264
25.31205264
25.31205264
25.31770435
25.31770435
25.31770435
25.31770435
25.31770435
25.30725378
25.30725378
25.30725378
25.30725378
25.30725378
25.3204232
25.3204232
25.3204232
25.3204232
25.3204232
25.32028129
25.32028129
25.32028129
25.32028129
25.32028129
25.33934817
25.33934817
25.33934817
25.33934817
25.33934817
25.32654613
25.32654613
25.32654613
25.32654613
25.32654613
25.29556854
25.29556854
25.29556854
25.29556854
25.29556854
25.34400935
25.34400935
25.34400935
25.34400935
25.34400935
25.36925237
25.36925237
25.36925237
25.36925237
25.36925237
25.33484168
25.33484168
25.33484168
25.33484168
25.33484168
25.30528924
25.30528924
25.30528924
25.30528924
25.30528924
25.34902589
25.34902589
25.34902589
25.34902589
25.34902589
25.32872814
25.32872814
25.32872814
25.32872814
25.32872814
25.35646569
25.35646569
25.35646569
25.35646569
25.35646569
25.27716258
25.27716258
25.27716258
25.27716258
25.27716258
25.34739794
25.34739794
25.34739794
25.34739794
25.34739794
25.35852379
25.35852379
25.35852379
25.35852379
25.35852379
25.32806371
25.32806371
25.32806371
25.32806371
25.32806371
25.28599143
25.28599143
25.28599143
25.28599143
25.28599143
25.31070317
25.31070317
25.31070317
25.31070317
25.31070317
25.27548018
25.27548018
25.27548018
25.27548018
25.27548018
25.34008403
25.34008403
25.34008403
25.34008403
25.34008403
25.3587136
25.3587136
25.3587136
25.3587136
25.3587136
25.34586347
25.34586347
25.34586347
25.34586347
25.34586347
25.31171814
25.31171814
25.31171814
25.31171814
25.31171814
25.32379329
25.32379329
25.32379329
25.32379329
25.32379329
25.33752841
25.33752841
25.33752841
25.33752841
25.33752841
25.35175955
25.35175955
25.35175955
25.35175955
25.35175955
25.34874281
25.34874281
25.34874281
25.34874281
25.34874281
25.32021424
25.32021424
25.32021424
25.32021424
25.32021424
25.30561899
25.30561899
25.30561899
25.30561899
25.30561899
25.32129655
25.32129655
25.32129655
25.32129655
25.32129655
25.30661239
25.30661239
25.30661239
25.30661239
25.30661239
25.29804411
25.29804411
25.29804411
25.29804411
25.29804411
25.27623341
25.27623341
25.27623341
25.27623341
25.27623341
25.32014855
25.32014855
25.32014855
25.32014855
25.32014855
25.33447616
25.33447616
25.33447616
25.33447616
25.33447616
25.27311423
25.27311423
25.27311423
25.27311423
25.27311423
25.30534162
25.30534162
25.30534162
25.30534162
25.30534162
25.31821285
25.31821285
25.31821285
25.31821285
25.31821285
25.34006512
25.34006512
25.34006512
25.34006512
25.34006512
25.29584728
25.29584728
25.29584728
25.29584728
25.29584728
25.3044549
25.3044549
25.3044549
25.3044549
25.3044549
25.32905418
25.32905418
25.32905418
25.32905418
25.32905418
25.32117705
25.32117705
25.32117705
25.32117705
25.32117705
25.30060157
25.30060157
25.30060157
25.30060157
25.30060157
25.31988585
25.31988585
25.31988585
25.31988585
25.31988585
25.3198083
25.3198083
25.3198083
25.3198083
25.3198083
25.32621862
25.32621862
25.32621862
25.32621862
25.32621862
25.3115697
25.3115697
25.3115697
25.3115697
25.3115697
25.34235167
25.34235167
25.34235167
25.34235167
25.34235167
25.32193522
25.32193522
25.32193522
25.32193522
25.32193522
25.33526312
25.33526312
25.33526312
25.33526312
25.33526312
25.31708141
25.31708141
25.31708141
25.31708141
25.31708141
25.30867974
25.30867974
25.30867974
25.30867974
25.30867974
25.25979482
25.25979482
25.25979482
25.25979482
25.25979482
25.31172022
25.31172022
25.31172022
25.31172022
25.31172022
25.33126627
25.33126627
25.33126627
25.33126627
25.33126627
25.26225006
25.26225006
25.26225006
25.26225006
25.26225006
25.34277132
25.34277132
25.34277132
25.34277132
25.34277132
25.31694932
25.31694932
25.31694932
25.31694932
25.31694932
25.29631406
25.29631406
25.29631406
25.29631406
25.29631406
25.35576656
25.35576656
25.35576656
25.35576656
25.35576656
25.34119999
25.34119999
25.34119999
25.34119999
25.34119999
25.32501126
25.32501126
25.32501126
25.32501126
25.32501126
25.33656628
25.33656628
25.33656628
25.33656628
25.33656628
25.30259749
25.30259749
25.30259749
25.30259749
25.30259749
25.31578645
25.31578645
25.31578645
25.31578645
25.31578645
25.31735682
25.31735682
25.31735682
25.31735682
25.31735682
25.3213579
25.3213579
25.3213579
25.3213579
25.3213579
25.30120014
25.30120014
25.30120014
25.30120014
25.30120014
25.3263592
25.3263592
25.3263592
25.3263592
25.3263592
25.34163645
25.34163645
25.34163645
25.34163645
25.34163645
25.3308901
25.3308901
25.3308901
25.3308901
25.3308901
25.3472131
25.3472131
25.3472131
25.3472131
25.3472131
25.33011151
25.33011151
25.33011151
25.33011151
25.33011151
25.36630774
25.36630774
25.36630774
25.36630774
25.36630774
25.31991137
25.31991137
25.31991137
25.31991137
25.31991137
25.32431577
25.32431577
25.32431577
25.32431577
25.32431577
25.29678894
25.29678894
25.29678894
25.29678894
25.29678894
25.31860744
25.31860744
25.31860744
25.31860744
25.31860744
25.31682109
25.31682109
25.31682109
25.31682109
25.31682109
25.35042822
25.35042822
25.35042822
25.35042822
25.35042822
25.31782267
25.31782267
25.31782267
25.31782267
25.31782267
25.32199241
25.32199241
25.32199241
25.32199241
25.32199241
25.32364744
25.32364744
25.32364744
25.32364744
25.32364744
25.27898635
25.27898635
25.27898635
25.27898635
25.27898635
25.30445301
25.30445301
25.30445301
25.30445301
25.30445301
25.30116627
25.30116627
25.30116627
25.30116627
25.30116627
25.29514386
25.29514386
25.29514386
25.29514386
25.29514386
25.29280824
25.29280824
25.29280824
25.29280824
25.29280824
25.31626265
25.31626265
25.31626265
25.31626265
25.31626265
''')
array_2 = data1.split('\n')
origin_lat_1 = []
origin_lat = []
for i in range(len(array_2)-1):
  if(array_2[i] != ''):
     origin_lat_1.append(float(array_2[i]))
for i in range(len(origin_lat_1)-1):
  if(origin_lat_1[i] != origin_lat_1[i+1]):
    origin_lat.append(origin_lat_1[i])

print((origin_lat[0]))

In [ ]:
# @title
data_sting_1 = ('''
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
Work
Shopping
Shopping
FreeTime
FreeTime
''')
data_cords_1 = ('''
25.3204
25.3208
25.3045
25.2819
25.3171
25.3204
25.3208
25.3208
25.2827
25.3088
25.348
25.3103
25.3061
25.2819
25.2827
25.3176
25.3175
25.3208
25.3088
25.3088
25.3204
25.3208
25.3175
25.2827
25.3088
25.3204
25.3175
25.3103
25.3171
25.3171
25.3204
25.3175
25.3175
25.2819
25.2819
25.2806
25.3208
25.3061
25.2827
25.2819
25.348
25.3045
25.3175
25.3792
25.3088
25.348
25.3045
25.3175
25.3088
25.3792
25.3176
25.3208
25.3045
25.3792
25.3792
25.3204
25.3045
25.3045
25.2827
25.2827
25.348
25.3175
25.3103
25.3088
25.3088
25.3176
25.3045
25.3103
25.3171
25.3088
25.3009
25.3103
25.3175
25.3088
25.3088
25.348
25.3045
25.3061
25.3792
25.3171
25.348
25.3103
25.3045
25.3792
25.3171
25.3009
25.3045
25.3045
25.3792
25.3792
25.3204
25.3045
25.3103
25.2827
25.3088
25.3204
25.3045
25.3061
25.3792
25.2819
25.348
25.3208
25.3208
25.2819
25.2819
25.3204
25.3103
25.3061
25.2827
25.3792
25.3009
25.3175
25.3061
25.3792
25.2827
25.3204
25.3045
25.3061
25.3792
25.2819
25.3176
25.3103
25.3208
25.3088
25.3792
25.3009
25.3045
25.3208
25.3088
25.2827
25.348
25.3061
25.3175
25.3792
25.3792
25.348
25.3045
25.3175
25.3088
25.2827
25.348
25.3103
25.3061
25.3088
25.3792
25.3204
25.3208
25.3175
25.3171
25.3792
25.3204
25.3061
25.3061
25.2827
25.2819
25.3009
25.3045
25.3061
25.3088
25.3171
25.3176
25.3175
25.3208
25.3088
25.3088
25.348
25.3061
25.3103
25.3792
25.2819
25.348
25.3103
25.3175
25.3088
25.3792
25.348
25.3061
25.3045
25.2819
25.3088
25.348
25.3061
25.3061
25.3792
25.3792
25.348
25.3061
25.3208
25.3171
25.3792
25.3204
25.3103
25.3175
25.2827
25.2819
25.348
25.3175
25.3045
25.2827
25.3171
25.3009
25.3103
25.3061
25.3792
25.3171
25.3204
25.3175
25.3175
25.2827
25.3171
25.3009
25.3103
25.3061
25.2819
25.2819
25.3204
25.3061
25.3061
25.2827
25.2827
25.3204
25.3175
25.3208
25.2819
25.3171
25.348
25.3061
25.3103
25.2827
25.3171
25.3204
25.3103
25.3061
25.2819
25.2827
25.3204
25.3175
25.3061
25.3171
25.2819
25.3009
25.3103
25.3045
25.3171
25.2827
25.3204
25.3103
25.3175
25.3088
25.2819
25.3176
25.3175
25.3208
25.3792
25.3171
25.3009
25.3061
25.3061
25.2819
25.2827
25.2806
25.3208
25.3061
25.3088
25.3088
25.3204
25.3103
25.3061
25.3792
25.3088
25.3176
25.3103
25.3175
25.2827
25.3088
25.3204
25.3045
25.3175
25.3088
25.3792
25.3204
25.3208
25.3045
25.3171
25.2827
25.3176
25.3061
25.3175
25.3792
25.3088
25.3009
25.3061
25.3175
25.3088
25.3171
25.348
25.3175
25.3061
25.2819
25.3792
25.2806
25.3045
25.3045
25.3171
25.3171
25.3204
25.3061
25.3175
25.3171
25.3792
25.3009
25.3175
25.3103
25.3088
25.3171
25.3009
25.3061
25.3208
25.2819
25.2827
25.348
25.3061
25.3045
25.3088
25.2819
25.3009
25.3208
25.3103
25.3171
25.3171
25.3009
25.3045
25.3103
25.3171
25.2827
25.3009
25.3208
25.3103
25.2819
25.2819
25.3204
25.3045
25.3175
25.2819
25.3171
25.3176
25.3208
25.3175
25.2819
25.2827
25.3204
25.3061
25.3061
25.3792
25.2827
25.3204
25.3045
25.3061
25.3088
25.2827
25.3176
25.3061
25.3175
25.3088
25.3088
25.3204
25.3175
25.3045
25.2819
25.3792
25.3204
25.3061
25.3061
25.3171
25.3171
25.3204
25.3208
25.3208
25.3088
25.3792
25.348
25.3103
25.3175
25.3088
25.3792
25.348
25.3208
25.3103
25.3088
25.3088
25.348
25.3045
25.3103
25.3792
25.3171
25.2806
25.3045
25.3061
25.3088
25.3171
25.348
25.3061
25.3175
25.3088
25.2827
25.3204
25.3045
25.3175
25.2827
25.3171
25.3176
25.3208
25.3061
25.3088
25.3792
25.348
25.3103
25.3175
25.3088
25.3792
25.3009
25.3061
25.3175
25.2819
25.2819
25.3176
25.3061
25.3103
25.2827
25.3088
25.3204
25.3175
25.3175
25.3792
25.3088
25.3204
25.3103
25.3208
25.3088
25.3171
25.3176
25.3103
25.3175
25.3171
25.2827
25.2806
25.3208
25.3061
25.3088
25.3088
25.3176
25.3103
25.3045
25.3792
25.3088
25.3176
25.3103
25.3175
25.2819
25.3792
25.348
25.3103
25.3061
25.3792
25.2819
25.3009
25.3103
25.3175
25.2827
25.3792
25.2806
25.3175
25.3103
25.3171
25.2827
25.348
25.3175
25.3208
25.2827
25.3088
25.3204
25.3103
25.3208
25.3171
25.2827
25.3204
25.3208
25.3175
25.2819
25.2827
25.3009
25.3103
25.3061
25.3171
25.2819
25.348
25.3061
25.3175
25.3171
25.2819
25.3176
25.3103
25.3103
25.3792
25.2827
25.3204
25.3045
25.3045
25.2827
25.3792
25.3204
25.3208
25.3175
25.3088
25.2819
25.348
25.3208
25.3103
25.3088
25.2819
25.348
25.3208
25.3061
25.3171
25.2819
25.3176
25.3208
25.3208
25.3171
25.3171
25.3009
25.3103
25.3175
25.3171
25.2819
25.3176
25.3103
25.3175
25.3088
25.3088
25.3009
25.3208
25.3175
25.2827
25.3171
25.3009
25.3208
25.3103
25.3171
25.3171
25.2806
25.3208
25.3045
25.3171
25.2819
25.3176
25.3208
25.3208
25.3088
25.3171
25.3176
25.3103
25.3103
25.3171
25.2827
25.348
25.3175
25.3175
25.2819
25.3792
25.3176
25.3061
25.3208
25.3792
25.2819
25.3009
25.3103
25.3208
25.3792
25.3088
25.348
25.3045
25.3175
25.3792
25.3171
25.2806
25.3208
25.3061
25.3171
25.2819
25.348
25.3061
25.3045
25.2819
25.2819
25.3176
25.3175
25.3061
25.3088
25.2827
25.3009
25.3045
25.3208
25.3171
25.3792
25.348
25.3061
25.3103
25.2827
25.3792
25.3204
25.3175
25.3208
25.2827
25.3088
25.3176
25.3061
25.3045
25.2827
25.3171
25.3009
25.3103
25.3061
25.2819
25.3792
25.3176
25.3175
25.3045
25.3171
25.3792
25.3009
25.3103
25.3061
25.3171
25.3171
25.3009
25.3045
25.3208
25.3792
25.2827
25.3204
25.3208
25.3208
25.2819
25.2819
25.3204
25.3208
25.3208
25.3088
25.3792
25.3176
25.3061
25.3103
25.3171
25.2819
25.3009
25.3103
25.3045
25.2827
25.3171
25.3204
25.3103
25.3208
25.2819
25.2819
25.348
25.3175
25.3175
25.3792
25.2827
25.3176
25.3045
25.3045
25.3792
25.2819
25.3009
25.3208
25.3175
25.3792
25.2819
25.3204
25.3103
25.3208
25.2819
25.2819
25.3204
25.3208
25.3103
25.3792
25.2827
25.3204
25.3103
25.3061
25.2819
25.2827
25.348
25.3045
25.3208
25.2819
25.2827
25.3204
25.3175
25.3061
25.3171
25.3171
25.3204
25.3045
25.3175
25.3171
25.3792
25.2806
25.3103
25.3175
25.2827
25.3088
25.3009
25.3061
25.3175
25.2819
25.3088
25.348
25.3061
25.3045
25.2827
25.3792
25.3176
25.3175
25.3045
25.3088
25.3171
25.3204
25.3103
25.3208
25.3171
25.3088
25.3176
25.3061
25.3103
25.2827
25.3792
25.348
25.3208
25.3208
25.2819
25.3792
25.3176
25.3208
25.3061
25.2827
25.3171
25.3009
25.3175
25.3103
25.3171
25.2819
25.348
25.3103
25.3175
25.3171
25.3171
25.3009
25.3045
25.3045
25.3088
25.3171
25.3176
25.3061
25.3175
25.3792
25.3792
25.3176
25.3045
25.3175
25.3088
25.2819
25.2806
25.3061
25.3208
25.2819
25.3171
25.3176
25.3103
25.3061
25.2827
25.3792
25.3204
25.3175
25.3061
25.3171
25.3792
25.348
25.3103
25.3045
25.2819
25.3792
25.3176
25.3175
25.3208
25.2827
25.2819
25.3204
25.3061
25.3208
25.3171
25.2819
25.3204
25.3045
25.3045
25.2827
25.3088
25.3204
25.3208
25.3061
25.2827
25.3171
25.3009
25.3208
25.3061
25.3171
25.2827
25.3204
25.3175
25.3045
25.3088
25.3088
25.3204
25.3175
25.3208
25.3088
25.3792
25.3009
25.3061
25.3045
25.2819
25.3171
25.3176
25.3103
25.3045
25.2827
25.2827
25.348
25.3061
25.3061
25.3171
25.2819
25.3176
25.3175
25.3045
25.2819
25.2819
25.3009
25.3208
25.3061
25.3171
25.2827
25.3176
25.3061
25.3208
25.3088
25.2827
25.3009
25.3208
25.3103
25.3088
25.3792
25.3204
25.3103
25.3045
25.3088
25.3088
25.348
25.3175
25.3208
25.3792
25.2827
25.3204
25.3208
25.3208
25.3171
25.2819
25.3204
25.3103
25.3103
25.2819
25.2827
25.3204
25.3061
25.3045
25.2827
25.3171
25.3176
25.3061
25.3208
25.3171
25.3171
25.2806
25.3175
25.3208
25.3088
25.2819
25.3176
25.3175
25.3061
25.3171
25.2827
25.2806
25.3175
25.3045
25.2827
25.3171
25.348
25.3103
25.3103
25.3792
25.2819
25.3009
25.3208
25.3175
25.3792
25.3171
25.3009
25.3103
25.3208
25.3792
25.2819
25.3204
25.3175
25.3208
25.3792
25.3088
25.3176
25.3208
25.3061
25.3792
25.2819
25.3176
25.3175
25.3103
25.3171
25.3088
25.3009
25.3045
25.3045
25.2827
25.3088
25.3204
25.3175
25.3208
25.2819
25.3088
25.3009
25.3175
25.3103
25.2819
25.2819
25.348
25.3103
25.3103
25.3088
25.2827
25.3204
25.3208
25.3175
25.2827
25.2819
25.3204
25.3103
25.3061
25.3792
25.3171
25.3176
25.3175
25.3045
25.3792
25.3792
25.3176
25.3103
25.3103
25.2819
25.3792
25.3204
25.3175
25.3175
25.2819
25.3088
25.3009
25.3208
25.3045
25.3792
25.3088
25.2806
25.3103
25.3061
25.3171
25.3088
25.3009
25.3175
25.3175
25.2819
25.3171
25.3176
25.3045
25.3061
25.3792
25.2827
25.3176
25.3175
25.3103
25.3171
25.2827
25.3204
25.3175
25.3103
25.2819
25.3171
25.3204
25.3208
25.3103
25.3171
25.2819
25.3204
25.3103
25.3061
25.3171
25.3088
25.3176
25.3103
25.3175
25.3171
25.2819
25.3204
25.3103
25.3208
25.3792
25.2819
25.3176
25.3103
25.3045
25.2827
25.3792
25.3176
25.3061
25.3208
25.3792
25.3171
25.3176
25.3103
25.3061
25.2827
25.3171
25.3204
25.3061
25.3208
25.2827
25.3088
25.3009
25.3045
25.3175
25.3088
25.2819
25.3176
25.3061
25.3045
25.2827
25.3088
25.3176
25.3175
25.3175
25.2819
25.3088
25.3009
25.3045
25.3061
25.3088
25.3171
25.3176
25.3103
25.3045
25.3171
25.3171
25.3176
25.3103
25.3045
25.2819
25.3088
25.3204
25.3061
25.3208
25.3792
25.2819
25.3176
25.3103
25.3103
25.2827
25.2827
25.3009
25.3208
25.3045
25.3171
25.3171
25.348
25.3103
25.3175
25.3171
25.3792
25.348
25.3103
25.3045
25.3171
25.3088
25.348
25.3045
25.3061
25.2827
25.3088
25.3009
25.3045
25.3061
25.2827
25.3792
25.3176
25.3103
25.3061
25.3171
25.3088
25.3176
25.3208
25.3061
25.2819
25.3792
25.3204
25.3103
25.3208
25.3171
25.2819
25.348
25.3045
25.3045
25.2827
25.2827
25.3204
25.3061
25.3208
25.3792
25.2827
25.2845
25.3045
25.3103
25.3171
25.3088
25.348
25.3208
25.3175
25.2827
25.2819
25.3009
25.3103
25.3061
25.2827
25.3792
25.3204
25.3045
25.3061
25.3792
25.2827
25.3009
25.3045
25.3061
25.3088
25.3088
25.3176
25.3103
25.3061
25.3171
25.3171
25.3204
25.3103
25.3175
25.2827
25.2819
25.3204
25.3045
25.3045
25.3088
25.2827
25.3176
25.3103
25.3175
25.3088
25.3088
25.348
25.3208
25.3045
25.3171
25.3171
25.3204
25.3175
25.3045
25.3171
25.2827
25.3009
25.3175
25.3103
25.2819
25.3171
25.3204
25.3208
25.3175
25.3088
25.2827
25.3176
25.3045
25.3208
25.2827
25.3792
25.3176
25.3103
25.3103
25.2827
25.3088
25.3009
25.3061
25.3061
25.3088
25.3171
25.348
25.3061
25.3045
25.3088
25.3171
25.3204
25.3045
25.3061
25.3088
25.3171
25.3009
25.3103
25.3103
25.3088
25.3088
25.2806
25.3061
25.3045
25.3171
25.3792
25.348
25.3175
25.3208
25.3792
25.2827
25.3176
25.3208
25.3061
25.3171
25.2827
25.3176
25.3208
25.3103
25.2819
25.3171
25.2806
25.3175
25.3061
25.3088
25.2827
25.2806
25.3175
25.3045
25.3171
25.3088
25.3176
25.3175
25.3061
25.3792
25.2827
25.348
25.3061
25.3103
25.3171
25.2827
25.3204
25.3175
25.3045
25.3171
25.3171
25.3009
25.3208
25.3208
25.3171
25.3792
25.3176
25.3208
25.3175
25.3792
25.3088
25.3009
25.3208
25.3175
25.3088
25.3171
25.348
25.3045
25.3061
25.3088
25.3171
25.3009
25.3175
25.3208
25.3088
25.3171
25.3009
25.3175
25.3045
25.2827
25.2827
25.348
25.3208
25.3175
25.3171
25.2827
25.3176
25.3103
25.3208
25.2819
25.3792
25.3176
25.3103
25.3103
25.3171
25.2827
25.3009
25.3175
25.3103
25.2827
25.3792
25.3204
25.3175
25.3175
25.3792
25.3088
25.3009
25.3061
25.3175
25.2819
25.3088
25.3176
25.3208
25.3208
25.2827
25.3088
25.3176
25.3061
25.3175
25.3171
25.3792
25.348
25.3208
25.3061
25.3792
25.2827
25.3176
25.3061
25.3061
25.3792
25.3171
25.3009
25.3103
25.3045
25.3792
25.3171
25.348
25.3208
25.3208
25.2819
25.2827
25.3176
25.3208
25.3175
25.2827
25.2819
25.348
25.3175
25.3045
25.2827
25.2819
25.348
25.3061
25.3175
25.3088
25.2819
25.3176
25.3045
25.3103
25.2827
25.2827
25.3009
25.3175
25.3061
25.3171
25.3171
25.3204
25.3061
25.3208
25.3792
25.2819
25.3176
25.3175
25.3208
25.2827
25.2819
25.348
25.3103
25.3061
25.3792
25.3171
25.3176
25.3045
25.3045
25.3171
25.2827
25.3176
25.3208
25.3103
25.3792
25.2827
25.3009
25.3061
25.3208
25.3792
25.3088
25.2806
25.3103
25.3061
25.3792
25.2827
25.348
25.3045
25.3175
25.3171
25.2827
25.3176
25.3103
25.3061
25.2827
25.2827
25.3204
25.3175
25.3045
25.2819
25.2819
25.3204
25.3045
25.3045
25.2827
25.3088
25.3009
25.3103
25.3103
25.2819
25.2827
25.3204
25.3045
25.3103
25.2827
25.3171
25.3204
25.3175
25.3175
25.2819
25.3171
25.3204
25.3103
25.3175
25.2827
25.3792
25.3009
25.3208
25.3103
25.3792
25.2827
25.348
25.3045
25.3175
25.3171
25.2819
25.3204
25.3045
25.3208
25.2827
25.2819
25.3204
25.3045
25.3045
25.3792
25.3088
25.2806
25.3208
25.3208
25.2819
25.3088
25.348
25.3045
25.3061
25.2827
25.3088
25.348
25.3045
25.3208
25.3088
25.3792
25.348
25.3208
25.3061
25.3792
25.3088
25.3204
25.3208
25.3175
25.3171
25.3088
25.3009
25.3061
25.3045
25.2819
25.3792
25.3176
25.3103
25.3103
25.3088
25.3088
25.348
25.3103
25.3061
25.3088
25.2827
25.3009
25.3045
25.3175
25.3088
25.2827
25.3204
25.3208
25.3175
25.3792
25.2819
25.3176
25.3045
25.3208
25.2819
25.3792
25.348
25.3208
25.3061
25.2819
25.3171
25.348
25.3208
25.3045
25.3171
25.3088
25.3009
25.3208
25.3175
25.3792
25.2827
25.348
25.3061
25.3045
25.2827
25.3792
25.4302
25.3061
25.3208
25.3088
25.2819
25.3176
25.3103
25.3103
25.3171
25.3088
25.3204
25.3208
25.3175
25.2827
25.3792
25.3009
25.3045
25.3103
25.3171
25.2819
25.2806
25.3175
25.3045
25.2819
25.3088
25.3204
25.3103
25.3045
25.3088
25.2827
25.3176
25.3103
25.3045
25.2819
25.3792
25.4302
25.3208
25.3175
25.2819
25.3088
25.3009
25.3045
25.3045
25.2819
25.3088
25.3204
25.3045
25.3061
25.3792
25.3171
25.3204
25.3045
25.3175
25.3171
25.2827
25.3204
25.3208
25.3208
25.2827
25.3171
25.3176
25.3208
25.3103
25.3171
25.2827
25.348
25.3045
25.3103
25.3792
25.3792
25.3204
25.3175
25.3103
25.3088
25.3088
25.3009
25.3208
25.3175
25.2827
25.3792
25.348
25.3061
25.3103
25.3171
25.3088
25.2845
25.3208
25.3045
25.2819
25.3088
25.3204
25.3103
25.3061
25.3088
25.3088
25.3176
25.3175
25.3208
25.2827
25.3792
25.3204
25.3045
25.3045
25.3171
25.2827
25.3176
25.3103
25.3103
25.3171
25.2819
25.3009
25.3061
25.3103
25.3171
25.2827
25.3204
25.3045
25.3045
25.2819
25.3171
25.3176
25.3175
25.3208
25.3171
25.2827
25.348
25.3103
25.3175
25.2827
25.2819
25.348
25.3103
25.3103
25.3088
25.2827
25.2806
25.3103
25.3175
25.3792
25.3792
25.3176
25.3208
25.3103
25.2819
25.3088
25.3009
25.3061
25.3045
25.3088
25.3088
25.348
25.3103
25.3103
25.3171
25.3792
25.3009
25.3175
25.3045
25.2827
25.3171
25.3176
25.3103
25.3175
25.3792
25.2819
25.3176
25.3061
25.3061
25.3171
25.3088
25.2806
25.3103
25.3208
25.2827
25.2827
25.2806
25.3208
25.3061
25.3171
25.3792
25.3176
25.3103
25.3061
25.3792
25.3088
25.3009
25.3175
25.3208
25.3171
25.3792
25.3009
25.3208
25.3045
25.2819
25.2819
25.3009
25.3208
25.3061
25.3088
25.2819
25.348
25.3045
25.3208
25.3088
25.2819
25.2806
25.3061
25.3103
25.3088
25.2819
25.3204
25.3208
25.3208
25.3088
25.2819
25.3176
25.3061
25.3208
25.3088
25.3792
25.3009
25.3103
25.3061
25.2819
25.3171
25.348
25.3061
25.3208
25.3792
25.3792
25.3176
25.3061
25.3103
25.2819
25.3088
25.3204
25.3208
25.3208
25.2819
25.2819
25.3176
25.3175
25.3103
25.3171
25.2827
25.348
25.3061
25.3103
25.2819
25.3171
25.3009
25.3061
25.3208
25.3792
25.2819
25.3009
25.3103
25.3061
25.3792
25.3088
25.348
25.3175
25.3208
25.3171
25.2819
25.3009
25.3103
25.3045
25.3171
25.2819
25.3204
25.3208
25.3175
25.3171
25.2819
25.348
25.3175
25.3045
25.3171
25.3088
25.3176
25.3061
25.3061
25.2827
25.2827
25.348
25.3208
25.3208
25.3088
25.2827
25.348
25.3175
25.3208
25.3171
25.3792
25.348
25.3208
25.3175
25.3171
25.3792
25.3176
25.3061
25.3103
25.2827
25.3792
25.3176
25.3061
25.3061
25.3088
25.3171
25.3204
25.3103
25.3208
25.3792
25.3171
25.348
25.3208
25.3103
25.3792
25.2819
25.3204
25.3208
25.3208
25.3088
25.3792
25.3009
25.3175
25.3061
25.3792
25.3792
25.3204
25.3208
25.3061
25.3171
25.2819
25.3176
25.3175
25.3061
25.2819
25.3792
25.3009
25.3103
25.3103
25.3088
25.3171
25.3176
25.3208
25.3061
25.2819
25.3088
25.3204
25.3103
25.3103
25.3171
25.3792
25.3176
25.3045
25.3208
25.2827
25.3088
25.3009
25.3175
25.3103
25.3792
25.3792
25.3176
25.3208
25.3045
25.2827
25.2827
25.3204
25.3045
25.3045
25.2827
25.3792
25.3009
25.3061
25.3175
25.3792
25.3792
25.3176
25.3175
25.3103
25.2827
25.2819
25.3009
25.3175
25.3208
25.2827
25.3088
25.3204
25.3208
25.3103
25.3171
25.2827
25.3009
25.3103
25.3175
25.3088
25.3792
25.3204
25.3045
25.3103
25.3088
25.3171
25.3009
25.3061
25.3103
25.3088
25.2827
25.3009
25.3103
25.3175
25.3088
25.2827
25.3204
25.3175
25.3208
25.3088
25.3792
25.3204
25.3208
25.3208
25.3088
25.3088
25.3009
25.3208
25.3175
25.2819
25.3088
25.2806
25.3175
25.3208
25.2827
25.2827
25.3009
25.3208
25.3045
25.3088
25.2819
25.3176
25.3045
25.3175
25.2819
25.3088
25.3176
25.3208
25.3175
25.2827
25.3171
25.3176
25.3103
25.3045
25.3171
25.2827
25.348
25.3208
25.3061
25.3171
25.2827
25.3176
25.3045
25.3175
25.3171
25.3171
25.3176
25.3045
25.3061
25.3088
25.3792
25.348
25.3061
25.3175
25.2819
25.3171
25.3176
25.3061
25.3208
25.3088
25.3171
25.3176
25.3103
25.3045
25.3171
25.3088
25.348
25.3175
25.3208
25.3088
25.3792
25.348
25.3175
25.3061
25.3171
25.3171
25.3176
25.3208
25.3175
25.3171
25.3792
25.3176
25.3175
25.3045
25.2827
25.3792
25.3009
25.3103
25.3061
25.2827
25.2827
25.3204
25.3061
25.3061
25.2827
25.3088
25.3009
25.3208
25.3175
25.3171
25.3792
25.3009
25.3061
25.3103
25.3171
25.2827
25.348
25.3061
25.3208
25.3088
25.3088
25.3176
25.3061
25.3208
25.3792
25.3088
25.348
25.3045
25.3045
25.3171
25.3171
25.348
25.3103
25.3103
25.2819
25.2827
25.3009
25.3175
25.3103
25.3088
25.3088
25.3009
25.3175
25.3103
25.3171
25.2819
25.3176
25.3175
25.3061
25.2819
25.3792
25.3176
25.3061
25.3175
25.2827
25.3171
25.2806
25.3061
25.3061
25.3792
25.3171
25.348
25.3208
25.3175
25.3171
25.3171
25.348
25.3103
25.3208
25.3171
25.2827
25.348
25.3061
25.3103
25.3088
25.2819
25.348
25.3175
25.3045
25.3088
25.2819
25.3204
25.3208
25.3103
25.3792
25.3088
25.3009
25.3175
25.3045
25.3792
25.2819
25.348
25.3061
25.3103
25.3792
25.3171
25.3204
25.3175
25.3103
25.3088
25.3171
25.3009
25.3045
25.3045
25.3088
25.3792
25.348
25.3045
25.3061
25.2819
25.2827
25.348
25.3061
25.3045
25.3792
25.3088
25.348
25.3045
25.3208
25.3088
25.3792
25.3176
25.3061
25.3061
25.3171
25.3171
25.3176
25.3061
25.3208
25.2827
25.2827
25.3204
25.3061
25.3061
25.3171
25.2819
25.3204
25.3045
25.3061
25.2827
25.3792
25.3009
25.3103
25.3208
25.2827
25.2827
25.3176
25.3103
25.3061
25.3171
25.3171
25.348
25.3103
25.3175
25.2827
25.3088
25.3204
25.3175
25.3208
25.3088
25.3792
25.3009
25.3208
25.3045
25.3088
25.3792
25.348
25.3208
25.3061
25.3792
25.3171
25.348
25.3208
25.3208
25.3171
25.3088
25.3009
25.3208
25.3175
25.2819
25.3792
25.3204
25.3175
25.3045
25.2819
25.3088
25.348
25.3175
25.3208
25.2827
25.3792
25.3204
25.3061
25.3103
25.3088
25.3171
25.3176
25.3061
25.3208
25.3088
25.3088
25.348
25.3045
25.3061
25.3792
25.2819
25.348
25.3103
25.3175
25.3171
25.3792
25.3204
25.3045
25.3045
25.2819
25.2819
25.3009
25.3103
25.3103
25.3171
25.2819
25.3009
25.3045
25.3208
25.3792
25.3171
25.348
25.3061
25.3208
25.3171
25.2827
25.3009
25.3045
25.3103
25.3792
25.2819
25.2845
25.3175
25.3175
25.3088
25.3171
25.3009
25.3045
25.3045
25.2819
25.2819
25.3204
25.3045
25.3045
25.2819
25.3088
25.3176
25.3045
25.3061
25.3088
25.3088
25.3204
25.3175
25.3208
25.2827
25.2819
25.3176
25.3045
25.3175
25.3792
25.3792
25.3009
25.3208
25.3208
25.3088
25.3088
25.3009
25.3208
25.3175
25.2827
25.3792
25.348
25.3045
25.3103
25.3792
25.2819
25.3009
25.3175
25.3208
25.3792
25.3171
25.348
25.3061
25.3045
25.2827
25.3171
25.3204
25.3045
25.3175
25.2827
25.3088
25.3176
25.3208
25.3045
25.3171
25.3171
25.3009
25.3208
25.3103
25.3088
25.3088
25.3009
25.3175
25.3045
25.3171
25.2819
25.3009
25.3045
25.3061
25.3792
25.3792
25.2845
25.3175
25.3061
25.2819
25.3088
25.3009
25.3045
25.3103
25.2819
25.3088
25.348
25.3061
25.3061
25.3171
25.3792
25.3204
25.3103
25.3045
25.3171
25.3088
25.348
25.3208
25.3061
25.2827
25.2827
25.3009
25.3045
25.3175
25.3792
25.3792
25.2845
25.3045
25.3061
25.3088
25.3171
25.348
25.3045
25.3208
25.3792
25.3088
25.348
25.3045
25.3208
25.2819
25.2827
25.3176
25.3208
25.3103
25.3171
25.2819
25.3176
25.3045
25.3175
25.3088
25.3792
25.3204
25.3175
25.3061
25.2827
25.3171
25.3204
25.3045
25.3045
25.2819
25.3171
25.3204
25.3045
25.3061
25.3792
25.3171
25.3176
25.3103
25.3208
25.3171
25.2827
25.3009
25.3103
25.3045
25.3088
25.3171
25.3009
25.3175
25.3175
25.3171
25.2819
25.3204
25.3061
25.3061
25.2827
25.3088
25.3009
25.3208
25.3061
25.3088
25.3171
25.3204
25.3103
25.3175
25.3792
25.2819
25.2806
25.3175
25.3061
25.2827
25.3171
25.3176
25.3208
25.3175
25.3171
25.3088
25.3009
25.3208
25.3208
25.3792
25.2819
25.348
25.3045
25.3175
25.3088
25.3088
25.3176
25.3175
25.3208
25.3088
25.3171
25.2806
25.3208
25.3208
25.3171
25.2819
25.3009
25.3208
25.3208
25.3088
25.3792
25.3176
25.3208
25.3208
25.3171
25.3171
25.3204
25.3175
25.3175
25.3792
25.3171
25.3009
25.3103
25.3208
25.3171
25.2827
25.3204
25.3045
25.3061
25.3792
25.3792
25.348
25.3103
25.3061
25.3792
25.3171
25.3009
25.3045
25.3208
25.2827
25.2819
25.3009
25.3208
25.3175
25.3792
25.3792
25.3204
25.3208
25.3103
25.3792
25.3792
25.348
25.3208
25.3175
25.3792
25.3792
25.348
25.3103
25.3045
25.3792
25.3171
25.3204
25.3103
25.3175
25.3171
25.3792
25.3204
25.3103
25.3175
25.2819
25.3792
25.3176
25.3045
25.3061
25.2819
25.2827
25.3009
25.3175
25.3175
25.2819
25.3171
25.3204
25.3175
25.3175
25.3792
25.2827
25.348
25.3208
25.3061
25.3171
25.3171
25.3009
25.3208
25.3061
25.2827
25.3171
25.348
25.3045
25.3208
25.3088
25.3171
25.3176
25.3175
25.3045
25.3088
25.3088
25.3176
25.3208
25.3175
25.3088
25.2827
25.3204
25.3061
25.3045
25.2819
25.3171
25.2806
25.3103
25.3103
25.3088
25.3088
25.2806
25.3103
25.3061
25.3792
25.3088
25.3009
25.3103
25.3045
25.2819
25.3088
25.348
25.3175
25.3061
25.3792
25.3792
25.3009
25.3208
25.3175
25.2819
25.3171
25.348
25.3175
25.3175
25.3171
25.3088
25.3176
25.3103
25.3045
25.3088
25.2827
25.3009
25.3175
25.3175
25.2827
25.3088
25.3204
25.3045
25.3045
25.3088
25.2819
25.3009
25.3208
25.3061
25.3171
25.2827
25.2806
25.3208
25.3103
25.2827
25.3171
25.348
25.3175
25.3103
25.2827
25.2827
25.3176
25.3061
25.3103
25.2827
25.2819
25.3176
25.3061
25.3045
25.3171
25.2819
25.3009
25.3175
25.3175
25.3171
25.3792
25.3204
25.3175
25.3103
25.2819
25.2819
25.2806
25.3175
25.3175
25.3171
25.2827
25.348
25.3208
25.3061
25.2819
25.2819
25.3009
25.3061
25.3103
25.3088
25.3792
25.3176
25.3061
25.3061
25.2827
25.2819
25.348
25.3045
25.3061
25.3171
25.3792
25.3176
25.3208
25.3208
25.3088
25.2827
25.3204
25.3103
25.3175
25.3171
25.2819
25.348
25.3175
25.3208
25.3171
25.3171
25.3176
25.3061
25.3103
25.3088
25.3171
25.3009
25.3208
25.3061
25.2827
25.2819
25.3009
25.3208
25.3061
25.3088
25.2827
25.3009
25.3208
25.3103
25.3088
25.2819
25.3009
25.3175
25.3103
25.3088
25.2827
25.348
25.3045
25.3061
25.3792
25.2827
25.3176
25.3175
25.3208
25.2827
25.2819
25.3176
25.3061
25.3175
25.2827
25.2819
25.3176
25.3175
25.3175
25.3088
25.2827
25.3176
25.3208
25.3045
25.3171
25.2827
25.2806
25.3061
25.3061
25.3792
25.2827
25.3204
25.3175
25.3103
25.3171
25.2819
25.3009
25.3045
25.3045
25.3792
25.3171
25.3176
25.3103
25.3103
25.3792
25.3792
25.348
25.3208
25.3061
25.3088
25.3171
25.3204
25.3061
25.3175
25.3171
25.3088
25.3176
25.3045
25.3208
25.3171
25.3171
25.3009
25.3175
25.3061
25.2819
25.2827
25.3204
25.3061
25.3103
25.3088
25.3792
25.348
25.3045
25.3103
25.3792
25.3171
25.3009
25.3061
25.3175
25.2819
25.3088
25.3009
25.3103
25.3103
25.3171
25.2827
25.3009
25.3061
25.3208
25.3088
25.3792
25.348
25.3103
25.3175
25.2819
25.3792
25.348
25.3103
25.3061
25.3792
25.3171
25.348
25.3045
25.3208
25.2819
25.2827
25.2806
25.3103
25.3175
25.3792
25.3792
25.3204
25.3175
25.3208
25.3088
25.3792
25.3204
25.3045
25.3208
25.3088
25.2819
25.3009
25.3103
25.3175
25.3792
25.3171
25.3204
25.3103
25.3061
25.3088
25.3171
25.348
25.3175
25.3175
25.3792
25.3792
25.3009
25.3045
25.3103
25.3792
25.3088
25.3204
25.3061
25.3061
25.3088
25.2819
25.348
25.3208
25.3103
25.3088
25.3792
25.3176
25.3061
25.3208
25.3792
25.3088
25.3204
25.3103
25.3103
25.2819
25.3088
25.3176
25.3061
25.3175
25.2827
25.3171
25.348
25.3208
25.3208
25.3088
25.2827
25.3009
25.3103
25.3208
25.3792
25.2827
25.3009
25.3045
25.3045
25.3088
25.2819
25.3176
25.3175
25.3045
25.3088
25.3792
25.3176
25.3061
25.3061
25.3792
25.3088
25.3009
25.3045
25.3045
25.2827
25.2827
25.3176
25.3045
25.3061
25.2819
25.3088
25.3009
25.3208
25.3208
25.3171
25.2819
25.3009
25.3175
25.3103
25.3792
25.2827
25.348
25.3103
25.3208
25.3792
25.2819
25.348
25.3175
25.3208
25.2827
25.3792
25.3204
25.3208
25.3175
25.3792
25.3171
25.3176
25.3103
25.3045
25.3792
25.3792
25.348
25.3208
25.3103
25.3792
25.2827
25.3204
25.3061
25.3208
25.3171
25.3792
25.348
25.3175
25.3103
25.3088
25.3171
25.3009
25.3061
25.3103
25.2827
25.2827
25.3009
25.3103
25.3061
25.3088
25.2827
25.348
25.3103
25.3045
25.3171
25.2827
25.3176
25.3208
25.3208
25.3792
25.3171
25.3204
25.3175
25.3103
25.3088
25.2819
25.3009
25.3061
25.3175
25.2827
25.3171
25.3204
25.3103
25.3175
25.2819
25.3792
25.3009
25.3208
25.3175
25.2827
25.3171
25.348
25.3045
25.3061
25.2827
25.2819
25.3204
25.3061
25.3045
25.3792
25.2819
25.3176
25.3045
25.3103
25.3088
25.2827
25.3176
25.3175
25.3208
25.2827
25.3792
25.3176
25.3045
25.3175
25.3171
25.3171
25.3176
25.3175
25.3045
25.2827
25.2819
25.348
25.3175
25.3103
25.2827
25.3792
25.3176
25.3045
25.3103
25.3171
25.3088
25.348
25.3208
25.3103
25.3171
25.2827
25.3009
25.3061
25.3061
25.2827
25.3792
25.348
25.3045
25.3045
25.3792
25.2819
25.348
25.3061
25.3045
25.3792
25.3792
25.2806
25.3061
25.3208
25.2827
25.3171
25.3009
25.3045
25.3175
25.3088
25.2819
25.3009
25.3061
25.3045
25.2819
25.3171
25.3204
25.3045
25.3208
25.3088
25.2819
25.3204
25.3175
25.3045
25.2819
25.2827
25.3176
25.3103
25.3103
25.2827
25.3792
25.348
25.3103
25.3103
25.3792
25.3088
25.3204
25.3045
25.3103
25.3171
25.2819
25.348
25.3175
25.3175
25.3171
25.3088
25.3204
25.3175
25.3208
25.2827
25.2819
25.2806
25.3175
25.3061
25.3792
25.2827
25.348
25.3045
25.3208
25.2827
25.3088
25.3009
25.3103
25.3061
25.3792
25.2819
25.3009
25.3103
25.3045
25.3088
25.3171
25.348
25.3061
25.3175
25.2819
25.3171
25.2806
25.3103
25.3175
25.3171
25.3088
25.3009
25.3103
25.3175
25.3171
25.3792
25.3176
25.3045
25.3045
25.2827
25.3088
25.2806
25.3175
25.3103
25.3088
25.3792
25.348
25.3103
25.3175
25.3792
25.3088
25.3204
25.3103
25.3175
25.2819
25.3088
25.3204
25.3208
25.3175
25.2827
25.3171
25.3204
25.3208
25.3045
25.3088
25.3792
25.3204
25.3045
25.3208
25.3171
25.2819
25.348
25.3045
25.3175
25.2827
25.3088
25.3176
25.3103
25.3208
25.3171
25.3088
25.348
25.3208
25.3103
25.2819
25.3171
25.348
25.3061
25.3208
25.3792
25.3171
25.3176
25.3103
25.3208
25.3792
25.3171
25.348
25.3208
25.3103
25.2819
25.3792
25.3009
25.3061
25.3208
25.3171
25.2819
25.348
25.3045
25.3208
25.3171
25.3171
25.3009
25.3061
25.3103
25.3171
25.3088
25.3176
25.3175
25.3061
25.3171
25.3088
25.3204
25.3103
25.3208
25.2819
25.3171
25.3176
25.3045
25.3045
25.3171
25.3088
25.3204
25.3045
25.3045
25.3792
25.3171
25.3204
25.3208
25.3175
25.2819
25.2819
25.3176
25.3061
25.3045
25.3171
25.2819
25.3176
25.3061
25.3208
25.3792
25.2827
25.3204
25.3208
25.3045
25.2819
25.3088
25.3204
25.3045
25.3103
25.3792
25.3088
25.348
25.3208
25.3175
25.3171
25.3792
25.348
25.3103
25.3061
25.3792
25.3088
25.3204
25.3103
25.3208
25.3171
25.3088
25.2806
25.3061
25.3175
25.2827
25.3171
25.3204
25.3208
25.3175
25.3088
25.3088
25.3204
25.3208
25.3208
25.2819
25.3088
25.3176
25.3208
25.3208
25.3171
25.2819
25.348
25.3208
25.3175
25.3171
25.3792
25.3204
25.3045
25.3061
25.2819
25.3171
25.3204
25.3175
25.3103
25.3171
25.3792
25.3204
25.3103
25.3103
25.2819
25.3171
25.348
25.3103
25.3103
25.2827
25.3088
25.3204
25.3061
25.3045
25.2827
25.3088
25.3204
25.3103
25.3208
25.2827
25.3171
25.3176
25.3208
25.3175
25.2827
25.3171
25.3176
25.3045
25.3208
25.3792
25.3088
25.3009
25.3208
25.3175
25.2819
25.2827
25.3204
25.3045
25.3103
25.3088
25.3088
25.3009
25.3208
25.3045
25.2819
25.2827
25.3176
25.3061
25.3061
25.3088
25.3792
25.3204
25.3061
25.3208
25.2819
25.3171
25.3176
25.3045
25.3175
25.3088
25.3088
25.3176
25.3175
25.3103
25.2819
25.2827
25.3009
25.3103
25.3045
25.3088
25.2819
25.3009
25.3061
25.3208
25.3792
25.3088
25.3009
25.3208
25.3103
25.3171
25.2827
25.348
25.3061
25.3175
25.3792
25.3088
25.3204
25.3045
25.3175
25.2827
25.3171
25.3176
25.3061
25.3061
25.2819
25.2827
25.3176
25.3208
25.3208
25.3171
25.3088
25.348
25.3175
25.3061
25.3792
25.2819
25.348
25.3045
25.3175
25.3088
25.2827
25.3204
25.3045
25.3045
25.3088
25.3171
25.3204
25.3103
25.3103
25.2827
25.3792
25.3204
25.3175
25.3103
25.2819
25.3792
25.3204
25.3061
25.3208
25.3088
25.3088
25.3204
25.3061
25.3175
25.3088
25.3088
25.348
25.3208
25.3208
25.2819
25.2819
25.2806
25.3061
25.3061
25.2827
25.2819
25.3176
25.3175
25.3061
25.2819
25.3171
25.3009
25.3103
25.3061
25.3792
25.3792
25.348
25.3061
25.3175
25.3171
25.3088
25.3204
25.3045
25.3103
25.2827
25.3792
25.348
25.3208
25.3208
25.2819
25.3088
25.348
25.3175
25.3175
25.2827
25.3792
25.3009
25.3061
25.3103
25.2827
25.2827
25.3176
25.3061
25.3045
25.3088
25.3792
25.3009
25.3175
25.3045
25.2819
25.3792
25.3009
25.3103
25.3061
25.2827
25.2819
25.3009
25.3061
25.3061
25.3171
25.2827
25.3204
25.3103
25.3175
25.3088
25.3088
25.348
25.3103
25.3103
25.3792
25.3171
25.3176
25.3208
25.3208
25.3171
25.3792
25.3176
25.3061
25.3208
25.3088
25.2827
25.3204
25.3208
25.3175
25.2819
25.3088
25.3009
25.3208
25.3175
25.3171
25.3792
25.3204
25.3103
25.3208
25.3088
25.3088
25.3176
25.3045
25.3175
25.2819
25.3171
25.3176
25.3061
25.3103
25.2819
25.3792
25.3176
25.3208
25.3175
25.3171
25.2827
25.348
25.3175
25.3061
25.3171
25.3088
25.348
25.3045
25.3103
25.3792
25.3088
25.3176
25.3045
25.3045
25.3792
25.3171
25.3176
25.3061
25.3061
25.2827
25.2819
25.3176
25.3208
25.3061
25.2827
25.2827
25.348
25.3208
25.3061
25.3171
25.2827
25.3176
25.3175
25.3061
25.3171
25.3171
25.3009
25.3175
25.3045
25.3171
25.3171
25.348
25.3045
25.3061
25.3171
25.2819
25.3204
25.3045
25.3045
25.2819
25.2827
25.2806
25.3061
25.3045
25.2827
25.3171
25.2806
25.3208
25.3103
25.2819
25.2819
25.3176
25.3103
25.3175
25.2819
25.2819
25.3176
25.3103
25.3045
25.3171
25.3088
25.3204
25.3103
25.3045
25.3088
25.2827
25.3204
25.3208
25.3061
25.3088
25.3088
25.3009
25.3175
25.3045
25.2827
25.3792
25.3204
25.3045
25.3061
25.2819
25.2827
25.3176
25.3175
25.3175
25.3088
25.3792
25.3009
25.3103
25.3103
25.3088
25.3792
25.3204
25.3175
25.3103
25.3792
25.2819
25.3204
25.3061
25.3175
25.2819
25.3171
25.3009
25.3175
25.3208
25.3792
25.2827
25.3009
25.3208
25.3061
25.3171
25.2827
25.348
25.3061
25.3045
25.3171
25.3171
25.3176
25.3103
25.3175
25.3088
25.3088
25.348
25.3045
25.3061
25.3171
25.2827
25.3176
25.3103
25.3061
25.2819
25.2827
25.3176
25.3045
25.3103
25.3171
25.3171
25.348
25.3103
25.3045
25.3171
25.3792
25.3176
25.3175
25.3045
25.3171
25.2819
25.3009
25.3175
25.3208
25.2827
25.3088
25.3176
25.3061
25.3061
25.2827
25.2819
25.3009
25.3061
25.3045
25.3088
25.3792
25.4302
25.3208
25.3061
25.3088
25.3171
25.3204
25.3175
25.3061
25.3171
25.2819
25.3176
25.3175
25.3208
25.3088
25.2819
25.348
25.3061
25.3061
25.3171
25.2827
25.3009
25.3061
25.3103
25.2819
25.2827
25.3204
25.3175
25.3103
25.3171
25.2819
25.3176
25.3208
25.3208
25.3171
25.3792
25.2806
25.3061
25.3103
25.3088
25.3171
25.3204
25.3045
25.3061
25.2819
25.2827
25.3204
25.3061
25.3045
25.3792
25.2819
25.3009
25.3103
25.3175
25.3088
25.2819
25.3009
25.3175
25.3061
25.2827
25.2819
25.348
25.3103
25.3061
25.2827
25.2827
25.3176
25.3061
25.3061
25.3088
25.2819
25.2806
25.3208
25.3208
25.2819
25.3171
25.3204
25.3045
25.3103
25.2827
25.3792
25.3176
25.3045
25.3103
25.3171
25.3171
25.3009
25.3208
25.3175
25.2819
25.2819
25.3204
25.3045
25.3045
25.2827
25.2819
25.3204
25.3061
25.3061
25.2819
25.2827
25.3176
25.3175
25.3045
25.2827
25.3088
25.348
25.3045
25.3061
25.3088
25.2819
25.3009
25.3175
25.3061
25.2827
25.3088
25.3176
25.3045
25.3061
25.3088
25.2827
25.3009
25.3061
25.3175
25.2827
25.2819
25.3176
25.3061
25.3208
25.3171
25.3088
25.3176
25.3045
25.3045
25.3792
25.3792
25.3009
25.3045
25.3103
25.3088
25.3088
25.3009
25.3175
25.3061
25.3088
25.2819
25.348
25.3061
25.3103
25.2819
25.3792
25.3176
25.3208
25.3103
25.3171
25.3792
25.3176
25.3175
25.3061
25.3088
25.3171
25.348
25.3208
25.3045
25.3171
25.3792
25.3204
25.3103
25.3103
25.2827
25.3792
25.2806
25.3061
25.3103
25.2819
25.3171
25.3204
25.3045
25.3061
25.3792
25.3088
25.3009
25.3175
25.3045
25.3088
25.3171
25.2806
25.3045
25.3045
25.3171
25.3171
25.3176
25.3061
25.3208
25.3792
25.3792
25.3176
25.3208
25.3103
25.2819
25.3088
25.3204
25.3175
25.3045
25.3792
25.2819
25.3204
25.3208
25.3103
25.3088
25.2827
25.2806
25.3061
25.3061
25.3088
25.3171
25.3176
25.3208
25.3175
25.2819
25.3171
25.3176
25.3175
25.3061
25.3792
25.2819
25.3176
25.3045
25.3061
25.2827
25.3171
25.3204
25.3103
25.3061
25.3088
25.3171
25.348
25.3175
25.3208
25.2827
25.2827
25.3204
25.3175
25.3175
25.2827
25.3088
25.3009
25.3208
25.3045
25.3171
25.3171
25.348
25.3061
25.3103
25.3792
25.3792
25.3204
25.3208
25.3045
25.3171
25.3792
25.348
25.3208
25.3175
25.3171
25.2819
25.3176
25.3061
25.3175
25.2827
25.3171
25.3009
25.3045
25.3208
25.3792
25.3088
25.348
25.3045
25.3045
25.3792
25.3088
25.348
25.3045
25.3045
25.2827
25.3171
25.3009
25.3175
25.3208
25.2819
25.2819
25.348
25.3103
25.3208
25.3088
25.3792
25.3009
25.3103
25.3045
25.3088
25.3171
25.3204
25.3175
25.3208
25.3171
25.2827
25.348
25.3208
25.3103
25.2827
25.3792
25.3204
25.3208
25.3208
25.3088
25.2827
25.348
25.3175
25.3175
25.3171
25.2827
25.3176
25.3045
25.3208
25.2827
25.3088
25.3009
25.3175
25.3175
25.2827
25.2819
25.3204
25.3103
25.3103
25.3171
25.3088
25.3176
25.3061
25.3045
25.2819
25.3792
25.3009
25.3208
25.3175
25.3088
25.3792
25.348
25.3103
25.3208
25.2827
25.2827
25.3176
25.3103
25.3061
25.3792
25.3792
25.348
25.3045
25.3103
25.3088
25.3792
25.2806
25.3208
25.3175
25.3171
25.3088
25.3009
25.3103
25.3175
25.2819
25.3171
25.3009
25.3103
25.3103
25.3088
25.3792
25.3176
25.3208
25.3045
25.3792
25.2819
25.3204
25.3061
25.3103
25.2827
25.3792
25.3176
25.3061
25.3061
25.3088
25.2827
25.3176
25.3045
25.3175
25.3171
25.2827
25.3176
25.3103
25.3103
25.3171
25.3792
25.3176
25.3208
25.3103
25.3792
25.2819
25.348
25.3208
25.3208
25.3088
25.3088
25.3009
25.3208
25.3208
25.3171
25.3088
25.3204
25.3208
25.3103
25.3171
25.3171
25.3009
25.3103
25.3045
25.2827
25.3171
25.2806
25.3061
25.3061
25.3171
25.2819
25.3176
25.3208
25.3175
25.2819
25.3171
25.3009
25.3175
25.3208
25.2827
25.3792
25.3176
25.3045
25.3045
25.3792
25.3792
25.3204
25.3045
25.3208
25.3088
25.3171
25.3204
25.3103
25.3061
25.2827
25.2827
25.3176
25.3103
25.3208
25.2827
25.2819
25.348
25.3045
25.3103
25.2819
25.3171
25.3204
25.3061
25.3175
25.2819
25.3088
25.2806
25.3175
25.3103
25.2819
25.2819
25.2806
25.3208
25.3103
25.2827
25.3171
25.2806
25.3103
25.3103
25.2819
25.3792
25.3009
25.3175
25.3045
25.3792
25.2827
25.2806
25.3208
25.3175
25.3088
25.3792
25.3204
25.3061
25.3175
25.3792
25.3171
25.3176
25.3103
25.3061
25.3088
25.3088
25.348
25.3061
25.3061
25.2827
25.3792
25.3176
25.3175
25.3208
25.2819
25.2819
25.3176
25.3103
25.3061
25.2827
25.3088
25.3009
25.3208
25.3103
25.3088
25.2819
25.3176
25.3208
25.3061
25.3088
25.3792
25.348
25.3061
25.3175
25.3088
25.3171
25.3009
25.3208
25.3061
25.3171
25.3792
25.348
25.3045
25.3103
25.3792
25.3171
25.2806
25.3208
25.3061
25.2827
25.2827
25.3204
25.3103
25.3061
25.3088
25.2827
25.3204
25.3045
25.3103
25.2819
25.3088
25.3204
25.3061
25.3045
25.3088
25.3171
25.3204
25.3103
25.3103
25.2827
25.3171
25.3204
25.3208
25.3175
25.3792
25.3171
25.348
25.3045
25.3061
25.3088
25.2819
25.3204
25.3103
25.3208
25.3792
25.3171
25.348
25.3045
25.3103
25.3792
25.3088
25.3204
25.3208
25.3045
25.2819
25.3088
25.348
25.3103
25.3045
25.3088
25.3171
25.3009
25.3175
25.3175
25.3088
25.2819
25.3204
25.3061
25.3061
25.3088
25.3088
25.3009
25.3061
25.3103
25.2827
25.2827
25.348
25.3103
25.3175
25.3792
25.3171
25.348
25.3103
25.3045
25.3792
25.2819
25.3009
25.3061
25.3045
25.2827
25.2819
25.3009
25.3061
25.3061
25.3171
25.2819
25.3009
25.3103
25.3061
25.3088
25.2827
25.3204
25.3175
25.3208
25.2819
25.3088
25.3204
25.3061
25.3175
25.3088
25.3792
25.348
25.3103
25.3208
25.3792
25.2819
25.3204
25.3045
25.3061
25.2827
25.3171
25.348
25.3175
25.3061
25.2827
25.2827
25.2806
25.3061
25.3045
25.3171
25.3171
25.3009
25.3175
25.3103
25.3088
25.3171
25.3204
25.3175
25.3175
25.2819
25.3792
25.3176
25.3208
25.3103
25.2819
25.2819
25.3204
25.3208
25.3061
25.3792
25.2819
25.3009
25.3061
25.3175
25.2819
25.3088
25.3009
25.3061
25.3175
25.3088
25.3171
25.348
25.3103
25.3208
25.2827
25.2827
25.2806
25.3208
25.3175
25.3171
25.3171
25.3009
25.3208
25.3175
25.3171
25.2819
25.3176
25.3175
25.3103
25.3171
25.3171
25.3009
25.3175
25.3103
25.3171
25.3792
25.3009
25.3061
25.3103
25.2819
25.2827
25.3204
25.3175
25.3045
25.3171
25.2827
25.3176
25.3208
25.3208
25.3171
25.3171
25.3204
25.3061
25.3175
25.2819
25.2819
25.348
25.3061
25.3103
25.3171
25.2819
25.348
25.3208
25.3061
25.3792
25.2827
25.348
25.3103
25.3045
25.3792
25.3792
25.3009
25.3045
25.3061
25.3088
25.2827
25.3176
25.3175
25.3061
25.3171
25.2819
25.3204
25.3103
25.3103
25.3171
25.2819
25.348
25.3103
25.3045
25.3088
25.3088
25.3204
25.3175
25.3103
25.2827
25.2827
25.3176
25.3208
25.3103
25.3171
25.3171
25.348
25.3045
25.3061
25.2827
25.3088
25.3204
25.3061
25.3045
25.3792
25.2827
25.3204
25.3103
25.3175
25.3088
25.2827
25.3009
25.3045
25.3045
25.2827
25.3792
25.3176
25.3045
25.3103
25.3792
25.3088
25.3176
25.3208
25.3175
25.3088
25.2827
25.3204
25.3175
25.3061
25.3792
25.2819
25.348
25.3103
25.3103
25.3792
25.3171
25.348
25.3061
25.3208
25.2819
25.3171
25.348
25.3208
25.3103
25.2827
25.3088
25.3204
25.3045
25.3045
25.2819
25.2827
25.3009
25.3103
25.3103
25.3171
25.3171
25.3176
25.3175
25.3175
25.3171
25.2827
25.3204
25.3103
25.3045
25.3171
25.2827
25.3204
25.3175
25.3061
25.3171
25.2819
25.3204
25.3175
25.3208
25.3792
25.2827
25.3176
25.3103
25.3175
25.3792
25.3171
25.3204
25.3061
25.3103
25.3088
25.2819
25.348
25.3061
25.3103
25.3088
25.2819
25.3176
25.3045
25.3045
25.3171
25.2819
25.3176
25.3103
25.3103
25.3792
25.2819
25.348
25.3061
25.3061
25.3088
25.2819
25.3176
25.3061
25.3208
25.3171
25.3171
25.2806
25.3045
25.3045
25.3171
25.3088
25.3204
25.3061
25.3208
25.3171
25.3792
25.348
25.3103
25.3045
25.3792
25.3792
25.3176
25.3045
25.3045
25.3088
25.2827
25.3204
25.3045
25.3045
25.2819
25.2827
25.3176
25.3208
25.3103
25.3088
25.2819
25.348
25.3208
25.3061
25.3088
25.3792
25.348
25.3208
25.3208
25.3792
25.3792
25.3176
25.3045
25.3045
25.3088
25.3792
25.3176
25.3061
25.3045
25.3171
25.3171
25.3204
25.3061
25.3061
25.2819
25.2827
25.348
25.3045
25.3175
25.3792
25.2819
25.3204
25.3103
25.3175
25.2819
25.2827
25.348
25.3061
25.3175
25.3792
25.2819
25.348
25.3175
25.3045
25.3171
25.2819
25.3204
25.3208
25.3208
25.3792
25.3171
25.3009
25.3061
25.3175
25.3792
25.2819
25.3204
25.3103
25.3061
25.3171
25.3171
25.3176
25.3061
25.3103
25.3792
25.3088
25.3176
25.3045
25.3175
25.2819
25.3171
25.3009
25.3175
25.3175
25.3171
25.3171
25.3176
25.3045
25.3175
25.2819
25.3792
25.348
25.3061
25.3045
25.2819
25.2827
25.3204
25.3208
25.3061
25.3792
25.3088
25.3204
25.3208
25.3175
25.3792
25.3088
25.348
25.3045
25.3061
25.3088
25.2819
25.348
25.3208
25.3175
25.3088
25.3171
25.348
25.3045
25.3208
25.2827
25.3792
25.3204
25.3103
25.3103
25.3088
25.2827
25.3204
25.3061
25.3175
25.3171
25.2827
25.348
25.3045
25.3103
25.2819
25.3792
25.348
25.3045
25.3045
25.2827
25.3171
25.3009
25.3208
25.3175
25.2819
25.3171
25.3009
25.3045
25.3045
25.2827
25.2827
25.3009
25.3208
25.3175
25.3171
25.3088
25.3009
25.3045
25.3045
25.2827
25.2827
25.348
25.3175
25.3061
25.3171
25.2819
25.2806
25.3103
25.3045
25.3088
25.2827
25.3009
25.3175
25.3061
25.2819
25.2819
25.348
25.3103
25.3103
25.3792
25.3792
25.3204
25.3103
25.3175
25.3088
25.3088
25.348
25.3045
25.3175
25.3171
25.3088
25.3009
25.3045
25.3061
25.2827
25.2819
25.2806
25.3061
25.3061
25.3792
25.2827
25.3176
25.3208
25.3175
25.2827
25.3171
25.348
25.3175
25.3175
25.3088
25.3088
25.3204
25.3208
25.3045
25.3088
25.3088
25.3204
25.3175
25.3045
25.2827
25.3792
25.348
25.3208
25.3061
25.3171
25.3088
25.2806
25.3175
25.3208
25.3088
25.3171
25.3204
25.3045
25.3208
25.3171
25.3792
25.3176
25.3175
25.3103
25.2827
25.2827
25.3176
25.3103
25.3045
25.2819
25.3171
25.2806
25.3208
25.3175
25.3171
25.3792
25.3009
25.3045
25.3061
25.2819
25.2819
25.3176
25.3045
25.3103
25.2819
25.2819
25.3176
25.3175
25.3208
25.2827
25.2827
25.348
25.3175
25.3045
25.2827
25.3088
25.3009
25.3103
25.3045
25.2827
25.2827
25.3009
25.3175
25.3045
25.3088
25.3171
25.3009
25.3103
25.3175
25.3088
25.2827
25.2845
25.3208
25.3175
25.3171
25.3088
25.348
25.3061
25.3045
25.3171
25.3792
25.3176
25.3103
25.3045
25.2819
25.2819
25.3176
25.3061
25.3061
25.2827
25.2827
25.3204
25.3103
25.3103
25.3792
25.2819
25.348
25.3045
25.3045
25.3171
25.3088
25.3204
25.3045
25.3045
25.3792
25.3171
25.3204
25.3103
25.3061
25.3792
25.3792
25.348
25.3208
25.3175
25.3792
25.3088
25.3009
25.3103
25.3061
25.3088
25.3088
25.2806
25.3175
25.3208
25.3792
25.2827
25.3009
25.3103
25.3208
25.3792
25.2819
25.348
25.3208
25.3061
25.3792
25.3171
25.348
25.3061
25.3045
25.3792
25.3792
25.3009
25.3045
25.3061
25.2819
25.2827
25.3176
25.3045
25.3045
25.2819
25.2819
25.348
25.3175
25.3208
25.3792
25.2827
25.3176
25.3061
25.3208
25.2819
25.3088
25.348
25.3061
25.3208
25.3792
25.2819
25.3176
25.3045
25.3061
25.2819
25.2819
25.3009
25.3175
25.3045
25.3171
25.3792
25.3009
25.3175
25.3061
25.3171
25.3171
25.3176
25.3103
25.3103
25.2819
25.3792
25.3204
25.3061
25.3045
25.3792
25.2827
25.3176
25.3061
25.3045
25.3171
25.2827
25.3204
25.3208
25.3061
25.3792
25.2819
25.348
25.3175
25.3208
25.3792
25.3088
25.3204
25.3103
25.3045
25.2819
25.3171
25.348
25.3208
25.3175
25.3171
25.2819
25.3204
25.3175
25.3208
25.2819
25.3171
25.3176
25.3175
25.3103
25.3171
25.2819
25.2806
25.3045
25.3061
25.2819
25.2827
25.3176
25.3103
25.3208
25.2819
25.2819
25.348
25.3103
25.3103
25.3088
25.3171
25.348
25.3103
25.3045
25.3171
25.2827
25.3204
25.3103
25.3045
25.3792
25.2827
25.3009
25.3208
25.3175
25.2827
25.2827
25.3204
25.3208
25.3045
25.2819
25.3171
25.348
25.3208
25.3208
25.3088
25.3792
25.3009
25.3175
25.3103
25.2819
25.3171
25.3176
25.3045
25.3208
25.3088
25.3088
25.348
25.3208
25.3061
25.3792
25.3171
25.348
25.3175
25.3175
25.2819
25.3088
25.3176
25.3061
25.3175
25.2819
25.2819
25.348
25.3103
25.3061
25.3171
25.3088
25.2806
25.3175
25.3045
25.2827
25.3792
25.348
25.3103
25.3175
25.2827
25.2819
25.3009
25.3175
25.3208
25.2827
25.3171
25.3204
25.3175
25.3045
25.3792
25.3171
25.3176
25.3045
25.3208
25.3088
25.3088
25.3176
25.3175
25.3103
25.2819
25.3792
25.3009
25.3103
25.3061
25.3088
25.2827
25.3176
25.3175
25.3103
25.3171
25.2819
25.3176
25.3045
25.3103
25.3792
25.3171
25.2845
25.3061
25.3208
25.2827
25.3171
25.348
25.3061
25.3208
25.2827
25.3088
25.3176
25.3103
25.3175
25.3171
25.3792
25.3009
25.3175
25.3208
25.2827
25.3792
25.3176
25.3061
25.3175
25.3792
25.3088
25.3009
25.3208
25.3208
25.2819
25.2827
25.348
25.3208
25.3045
25.2819
25.3088
25.4302
25.3045
25.3061
25.3171
25.3088
25.348
25.3103
25.3175
25.3171
25.3088
25.3176
25.3061
25.3103
25.3792
25.2819
25.348
25.3103
25.3061
25.3171
25.3088
25.3204
25.3061
25.3045
25.2827
25.3088
25.348
25.3103
25.3103
25.3171
25.3088
25.3204
25.3175
25.3061
25.3088
25.3088
25.348
25.3208
25.3103
25.3792
25.3792
25.3204
25.3103
25.3061
25.3088
25.3171
25.348
25.3045
25.3103
25.3088
25.2827
25.348
25.3061
25.3061
25.2819
25.3792
25.348
25.3061
25.3103
25.3171
25.2827
25.3176
25.3045
25.3208
25.3171
25.3792
25.348
25.3103
25.3208
25.2827
25.2827
25.3204
25.3061
25.3045
25.2819
25.3171
25.3204
25.3045
25.3175
25.3171
25.3088
25.4302
25.3103
25.3208
25.3171
25.2819
25.3204
25.3103
25.3103
25.3171
25.2819
25.3204
25.3061
25.3103
25.2819
25.2819
25.3204
25.3175
25.3175
25.2827
25.3088
25.3009
25.3175
25.3061
25.3792
25.3792
25.2806
25.3045
25.3045
25.2827
25.2827
25.3009
25.3061
25.3061
25.2827
25.3171
25.2806
25.3208
25.3061
25.3792
25.2827
25.2806
25.3103
25.3103
25.2827
25.2827
25.3176
25.3208
25.3045
25.2827
25.3792
25.348
25.3175
25.3045
25.2819
25.2827
25.2806
25.3175
25.3103
25.2819
25.2827
25.3176
25.3208
25.3175
25.2819
25.3171
25.3176
25.3061
25.3175
25.2827
25.3171
25.3176
25.3103
25.3175
25.3171
25.3088
25.3204
25.3208
25.3045
25.2827
25.2827
25.348
25.3061
25.3208
25.3088
25.2819
25.3204
25.3061
25.3061
25.2827
25.3171
25.3204
25.3208
25.3061
25.3792
25.2827
25.3009
25.3208
25.3045
25.3171
25.2819
25.348
25.3061
25.3045
25.3792
25.3171
25.3009
25.3103
25.3175
25.2827
25.2819
25.3009
25.3175
25.3045
25.2819
25.2827
25.3204
25.3061
25.3045
25.2827
25.2827
25.3176
25.3175
25.3103
25.3171
25.2819
25.348
25.3208
25.3045
25.2827
25.3088
25.3009
25.3061
25.3175
25.2827
25.2827
25.3204
25.3061
25.3175
25.2819
25.3088
25.3176
25.3175
25.3208
25.2827
25.3171
25.348
25.3208
25.3061
25.3792
25.3171
25.3176
25.3103
25.3208
25.2827
25.2819
25.3009
25.3061
25.3103
25.3171
25.2819
25.3204
25.3175
25.3208
25.2827
25.3792
25.3176
25.3103
25.3208
25.3171
25.3171
25.3204
25.3061
25.3208
25.3088
25.2819
25.2806
25.3175
25.3175
25.3088
25.2819
25.348
25.3045
25.3061
25.2827
25.3088
25.3176
25.3045
25.3175
25.3792
25.3171
25.3176
25.3045
25.3208
25.2819
25.2827
25.2806
25.3045
25.3208
25.3171
25.2827
25.3176
25.3175
25.3045
25.2819
25.2819
25.3176
25.3175
25.3208
25.2827
25.3088
25.3204
25.3208
25.3045
25.2819
25.3088
25.3009
25.3208
25.3045
25.3088
25.2827
25.4302
25.3175
25.3045
25.2827
25.3171
25.348
25.3061
25.3061
25.3088
25.3792
25.3176
25.3208
25.3103
25.3088
25.3792
25.348
25.3045
25.3103
25.2819
25.2827
25.2806
25.3103
25.3045
25.2827
25.2819
25.3009
25.3208
25.3045
25.3088
25.3088
25.3009
25.3208
25.3103
25.3792
25.2819
25.3204
25.3045
25.3045
25.3171
25.2819
25.348
25.3175
25.3061
25.2827
25.3792
25.3204
25.3175
25.3061
25.3088
25.2827
25.3009
25.3061
25.3175
25.3171
25.2827
25.3204
25.3175
25.3061
25.3171
25.2827
25.348
25.3103
25.3208
25.2819
25.3171
25.3204
25.3208
25.3103
25.2819
25.3792
25.3176
25.3061
25.3045
25.3792
25.2827
25.3204
25.3045
25.3061
25.2827
25.3171
25.3009
25.3175
25.3045
25.2819
25.3792
25.3204
25.3103
25.3208
25.3792
25.3088
25.3204
25.3208
25.3103
25.3792
25.2819
25.3009
25.3045
25.3103
25.2827
25.2827
25.3204
25.3208
25.3103
25.2819
25.3088
25.3009
25.3061
25.3175
25.2819
25.3792
25.3009
25.3208
25.3061
25.2827
25.2827
25.348
25.3061
25.3061
25.2819
25.2819
25.3176
25.3045
25.3208
25.2819
25.3171
25.3204
25.3208
25.3208
25.3792
25.3088
25.3009
25.3061
25.3103
25.3792
25.2827
25.348
25.3208
25.3061
25.3171
25.2827
25.348
25.3208
25.3208
25.3171
25.2819
25.3204
25.3061
25.3045
25.3792
25.2827
25.348
25.3061
25.3175
25.3088
25.3792
25.3176
25.3175
25.3061
25.3088
25.3171
25.3009
25.3208
25.3045
25.2819
25.3171
25.348
25.3175
25.3061
25.3088
25.3171
25.3176
25.3045
25.3103
25.3792
25.3088
25.3204
25.3175
25.3103
25.3088
25.3792
25.3009
25.3103
25.3175
25.2819
25.3171
25.3009
25.3045
25.3175
25.3171
25.3792
25.2806
25.3103
25.3103
25.3171
25.3792
25.3204
25.3175
25.3103
25.2819
25.2827
25.3009
25.3175
25.3045
25.2819
25.2819
25.348
25.3175
25.3175
25.2827
25.2827
25.348
25.3175
25.3208
25.3792
25.2819
25.3176
25.3175
25.3045
25.2827
25.3171
25.3176
25.3045
25.3175
25.2827
25.2819
25.3176
25.3175
25.3175
25.2827
25.3171
25.348
25.3061
25.3208
25.3171
25.2827
25.3009
25.3103
25.3175
25.2827
25.2819
25.3176
25.3045
25.3103
25.2819
25.2819
25.3204
25.3208
25.3103
25.3088
25.2819
25.3009
25.3045
25.3045
25.2827
25.2819
25.3009
25.3208
25.3103
25.3171
25.3792
25.3176
25.3175
25.3045
25.3171
25.3792
25.3204
25.3103
25.3175
25.2827
25.2827
25.3204
25.3045
25.3045
25.2819
25.3792
25.348
25.3045
25.3175
25.2827
25.3171
25.3204
25.3103
25.3208
25.2819
25.2827
25.348
25.3045
25.3208
25.3171
25.3792
25.348
25.3175
25.3045
25.3792
25.3171
25.348
25.3103
25.3045
25.2819
25.3792
25.2806
25.3175
25.3175
25.3792
25.2827
25.348
25.3045
25.3208
25.2827
25.3792
25.3009
25.3208
25.3208
25.3792
25.2827
25.348
25.3061
25.3045
25.2819
25.3088
25.3009
25.3061
25.3061
25.3088
25.3088
25.3204
25.3045
25.3208
25.3088
25.2827
25.3176
25.3103
25.3175
25.3088
25.2819
25.3176
25.3208
25.3045
25.3792
25.2827
25.3176
25.3175
25.3103
25.3088
25.3171
25.3204
25.3061
25.3103
25.2827
25.2819
25.348
25.3175
25.3061
25.3171
25.3088
25.3176
25.3175
25.3045
25.3171
25.3171
25.348
25.3045
25.3061
25.2827
25.3171
25.3176
25.3103
25.3045
25.2827
25.3792
25.3009
25.3045
25.3208
25.2819
25.3792
25.3204
25.3208
25.3103
25.3792
25.3792
25.348
25.3061
25.3208
25.3792
25.2827
25.3176
25.3061
25.3061
25.3792
25.3088
25.3009
25.3045
25.3061
25.3171
25.2819
25.3176
25.3045
25.3103
25.2819
25.2819
25.3204
25.3061
25.3103
25.3171
25.3171
25.3204
25.3045
25.3061
25.2819
25.2819
25.348
25.3045
25.3208
25.2819
25.3171
25.3176
25.3103
25.3208
25.2819
25.2827
25.348
25.3061
25.3045
25.3171
25.2827
25.348
25.3103
25.3103
25.2819
25.3171
25.3009
25.3103
25.3045
25.2827
25.3792
25.4302
25.3175
25.3175
25.2819
25.3171
25.3176
25.3061
25.3103
25.2819
25.3088
25.3204
25.3103
25.3061
25.2819
25.3171
25.3176
25.3175
25.3175
25.3088
25.2827
25.3176
25.3045
25.3045
25.3171
25.3792
25.3176
25.3208
25.3208
25.2819
25.3171
25.3009
25.3208
25.3175
25.3088
25.3171
25.3176
25.3103
25.3208
25.3171
25.3088
25.3176
25.3045
25.3208
25.3088
25.3088
25.3176
25.3045
25.3208
25.2819
25.2827
25.3204
25.3061
25.3175
25.2819
25.3792
25.348
25.3061
25.3208
25.3088
25.3792
25.348
25.3103
25.3175
25.3088
25.3088
25.3204
25.3208
25.3208
25.3792
25.2827
25.3009
25.3103
25.3103
25.2819
25.3792
25.3176
25.3208
25.3208
25.3792
25.2819
25.348
25.3061
25.3103
25.2827
25.3792
25.348
25.3045
25.3103
25.2819
25.2827
25.3009
25.3208
25.3061
25.3792
25.2819
25.3176
25.3061
25.3045
25.3088
25.3792
25.3176
25.3061
25.3175
25.2827
25.2827
25.3176
25.3175
25.3175
25.3088
25.3088
25.348
25.3061
25.3061
25.2827
25.2827
25.348
25.3208
25.3175
25.3792
25.3171
25.3176
25.3045
25.3175
25.2819
25.3088
25.3176
25.3175
25.3103
25.3171
25.3088
25.348
25.3208
25.3103
25.3792
25.3088
25.348
25.3175
25.3103
25.3792
25.2827
25.3204
25.3045
25.3061
25.3792
25.2819
25.3204
25.3208
25.3061
25.3088
25.2819
25.3204
25.3103
25.3061
25.3088
25.2827
25.2806
25.3208
25.3045
25.3088
25.3171
25.3204
25.3208
25.3208
25.2819
25.3088
25.348
25.3208
25.3208
25.3792
25.3792
25.3176
25.3175
25.3061
25.3792
25.3792
25.348
25.3061
25.3045
25.2827
25.3171
25.3009
25.3061
25.3208
25.3792
25.3171
25.2806
25.3045
25.3045
25.3792
25.2827
25.348
25.3175
25.3175
25.3171
25.2819
25.3204
25.3175
25.3103
25.2819
25.3171
25.3009
25.3061
25.3175
25.2827
25.3171
25.2806
25.3208
25.3208
25.3088
25.3792
25.3204
25.3061
25.3208
25.3171
25.2819
25.3176
25.3103
25.3103
25.2819
25.3088
25.348
25.3045
25.3061
25.3792
25.3792
25.3176
25.3103
25.3061
25.3171
25.2827
25.3204
25.3103
25.3045
25.3792
25.2827
25.3204
25.3061
25.3045
25.2827
25.3088
25.3204
25.3045
25.3103
25.2827
25.3171
25.3204
25.3061
25.3045
25.3171
25.3792
25.3009
25.3175
25.3103
25.3792
25.3171
25.348
25.3061
25.3103
25.3792
25.3792
25.3204
25.3175
25.3045
25.2819
25.3088
25.3176
25.3061
25.3175
25.3792
25.2819
25.3204
25.3103
25.3061
25.3171
25.3088
25.3176
25.3208
25.3103
25.2827
25.3088
25.348
25.3208
25.3103
25.3792
25.2819
25.3204
25.3103
25.3208
25.3088
25.2827
25.2806
25.3175
25.3175
25.3088
25.3088
25.3204
25.3061
25.3103
25.3171
25.3088
25.2806
25.3045
25.3208
25.3171
25.3171
25.3176
25.3103
25.3061
25.2827
25.3088
25.348
25.3061
25.3103
25.3171
25.2827
25.348
25.3045
25.3061
25.2819
25.2819
25.3204
25.3208
25.3045
25.3171
25.2819
25.3009
25.3175
25.3045
25.3792
25.2819
25.3204
25.3061
25.3045
25.2819
25.3171
25.3009
25.3175
25.3045
25.2827
25.3088
25.3204
25.3061
25.3208
25.3088
25.2819
25.348
25.3208
25.3045
25.3088
25.3088
25.3176
25.3061
25.3103
25.3171
25.3792
25.3009
25.3061
25.3061
25.3088
25.3088
25.348
25.3103
25.3061
25.3088
25.3792
25.348
25.3208
25.3061
25.3171
25.3792
25.3204
25.3103
25.3103
25.3792
25.2827
25.348
25.3175
25.3208
25.3171
25.3171
25.348
25.3045
25.3061
25.3088
25.3088
25.348
25.3061
25.3208
25.3171
25.3088
25.3204
25.3103
25.3208
25.3792
25.2827
25.3204
25.3045
25.3061
25.3792
25.3792
25.3009
25.3208
25.3061
25.3171
25.3088
25.3204
25.3045
25.3045
25.3171
25.2819
25.3176
25.3175
25.3175
25.3171
25.3171
25.3204
25.3061
25.3061
25.3171
25.2827
25.3176
25.3208
25.3175
25.2827
25.3088
25.3176
25.3103
25.3208
25.2827
25.3171
25.3204
25.3103
25.3061
25.3171
25.2827
25.3176
25.3175
25.3103
25.2819
25.3792
25.2806
25.3103
25.3208
25.3171
25.3171
25.3176
25.3208
25.3061
25.2827
25.3171
25.3204
25.3208
25.3061
25.3792
25.3171
25.3009
25.3103
25.3208
25.2819
25.2819
25.3204
25.3208
25.3103
25.3171
25.2819
25.3009
25.3045
25.3061
25.2827
25.3792
25.3176
25.3175
25.3061
25.2819
25.2827
25.348
25.3045
25.3175
25.3171
25.3171
25.3176
25.3103
25.3103
25.3171
25.3088
25.2806
25.3045
25.3175
25.2819
25.3171
25.3176
25.3103
25.3208
25.3088
25.2819
25.3204
25.3103
25.3103
25.2819
25.2819
25.348
25.3175
25.3175
25.3088
25.3088
25.3176
25.3061
25.3103
25.3171
25.2827
25.348
25.3061
25.3103
25.2827
25.3171
25.3009
25.3103
25.3208
25.3088
25.3088
25.3009
25.3061
25.3061
25.3792
25.3088
25.348
25.3103
25.3208
25.2819
25.3171
25.3009
25.3045
25.3061
25.2819
25.3171
25.3176
25.3208
25.3061
25.2827
25.2827
25.3204
25.3175
25.3103
25.3088
25.2819
25.3204
25.3061
25.3208
25.2827
25.2827
25.3009
25.3208
25.3175
25.3792
25.3171
25.3009
25.3103
25.3208
25.3088
25.2819
25.2806
25.3175
25.3175
25.3088
25.3088
25.3176
25.3061
25.3061
25.3088
25.2827
25.3176
25.3103
25.3175
25.3088
25.2819
25.3176
25.3103
25.3208
25.3171
25.2827
25.3204
25.3208
25.3045
25.3792
25.2819
25.348
25.3061
25.3045
25.3792
25.3792
25.3176
25.3061
25.3103
25.3088
25.3088
25.3009
25.3208
25.3061
25.2827
25.3792
25.348
25.3045
25.3103
25.3792
25.3088
25.3176
25.3061
25.3103
25.3792
25.2827
25.2806
25.3208
25.3208
25.3171
25.2819
25.348
25.3103
25.3061
25.3792
25.3171
25.348
25.3208
25.3175
25.3088
25.3088
25.3009
25.3175
25.3103
25.2827
25.3171
25.348
25.3103
25.3061
25.3088
25.3088
25.3009
25.3208
25.3045
25.3088
25.3792
25.3009
25.3045
25.3045
25.3792
25.2827
25.3009
25.3208
25.3045
25.3792
25.2819
25.348
25.3175
25.3045
25.2827
25.3088
25.3009
25.3045
25.3175
25.3171
25.3088
25.2806
25.3103
25.3175
25.2827
25.3171
25.348
25.3103
25.3045
25.3171
25.3171
25.3176
25.3045
25.3103
25.3792
25.3792
25.3204
25.3175
25.3045
25.3171
25.2827
25.348
25.3103
25.3208
25.3088
25.2827
25.3176
25.3061
25.3061
25.3088
25.3171
25.348
25.3103
25.3103
25.3088
25.3792
25.348
25.3175
25.3175
25.3171
25.3088
25.3204
25.3208
25.3045
25.3171
25.3171
25.3176
25.3175
25.3045
25.3171
25.3088
25.348
25.3103
25.3103
25.2827
25.2827
25.348
25.3061
25.3175
25.3792
25.3171
25.3204
25.3061
25.3208
25.3792
25.3088
25.348
25.3061
25.3045
25.3792
25.3792
25.3176
25.3103
25.3061
25.3088
25.2819
25.3176
25.3103
25.3175
25.3171
25.3088
25.3204
25.3045
25.3175
25.2827
25.3088
25.3176
25.3103
25.3103
25.3088
25.3792
25.3204
25.3103
25.3103
25.3088
25.3792
25.2806
25.3175
25.3103
25.3792
25.2827
25.348
25.3208
25.3103
25.2827
25.3171
25.4302
25.3061
25.3103
25.2819
25.2827
25.348
25.3045
25.3175
25.3792
25.3171
25.348
25.3061
25.3045
25.3088
25.3792
25.348
25.3208
25.3175
25.3088
25.3088
25.3009
25.3103
25.3208
25.3171
25.2819
25.3204
25.3045
25.3045
25.2827
25.3088
25.2806
25.3208
25.3103
25.3171
25.2819
25.3009
25.3208
25.3208
25.3088
25.3088
25.2806
25.3061
25.3045
25.3088
25.2827
25.3176
25.3045
25.3208
25.2827
25.2819
25.3204
25.3045
25.3061
25.3171
25.3088
25.348
25.3061
25.3208
25.3171
25.3088
25.3204
25.3045
25.3208
25.3792
25.3088
25.3009
25.3045
25.3061
25.3792
25.3088
25.3176
25.3103
25.3175
25.3792
25.3088
25.3009
25.3103
25.3045
25.2819
25.3088
25.3204
25.3103
25.3103
25.3088
25.3171
25.3176
25.3061
25.3103
25.2819
25.2827
25.2806
25.3103
25.3061
25.3171
25.2819
25.348
25.3061
25.3208
25.3171
25.3171
25.3204
25.3175
25.3103
25.2819
25.2827
25.348
25.3175
25.3045
25.2819
25.2819
25.3176
25.3061
25.3045
25.3792
25.3171
25.3176
25.3061
25.3103
25.3792
25.3171
25.3176
25.3045
25.3103
25.2827
25.2827
25.3009
25.3103
25.3103
25.2819
25.3792
25.3204
25.3103
25.3045
25.3171
25.2827
25.3176
25.3175
25.3045
25.3088
25.3088
25.348
25.3103
25.3208
25.2819
25.2819
25.3009
25.3061
25.3061
25.2827
25.3088
25.3176
25.3103
25.3045
25.3792
25.3171
25.2806
25.3175
25.3175
25.2827
25.3171
25.3176
25.3061
25.3208
25.3792
25.3088
25.348
25.3103
25.3208
25.3171
25.2827
25.348
25.3061
25.3175
25.2819
25.3088
25.348
25.3175
25.3103
25.3792
25.3171
25.3176
25.3061
25.3045
25.2819
25.2827
25.348
25.3103
25.3103
25.3171
25.3088
25.2806
25.3103
25.3061
25.2827
25.3792
25.3204
25.3208
25.3061
25.3088
25.3792
25.3204
25.3208
25.3103
25.3171
25.3171
25.3009
25.3045
25.3208
25.2819
25.2827
25.348
25.3175
25.3045
25.2819
25.3171
25.4302
25.3045
25.3175
25.3171
25.3792
25.3204
25.3045
25.3208
25.2819
25.2819
25.3009
25.3103
25.3045
25.3792
25.3088
25.3009
25.3175
25.3208
25.3088
25.3792
25.3009
25.3103
25.3045
25.2827
25.2819
25.3204
25.3175
25.3208
25.2819
25.3171
25.2806
25.3061
25.3208
25.3792
25.2827
25.3204
25.3175
25.3061
25.2827
25.3792
25.348
25.3175
25.3175
25.3792
25.3088
25.3176
25.3208
25.3103
25.2819
25.2827
25.348
25.3103
25.3208
25.2827
25.3088
25.3176
25.3175
25.3208
25.2819
25.3088
25.348
25.3061
25.3175
25.2819
25.3171
25.3176
25.3061
25.3175
25.2827
25.3792
25.3204
25.3061
25.3061
25.2827
25.2827
25.3009
25.3061
25.3103
25.3792
25.3171
25.348
25.3061
25.3103
25.3792
25.3171
25.348
25.3208
25.3175
25.3171
25.2827
25.3176
25.3175
25.3061
25.2827
25.2819
25.3204
25.3045
25.3103
25.3792
25.2819
25.3176
25.3045
25.3061
25.2827
25.2827
25.3204
25.3061
25.3208
25.3171
25.3171
25.348
25.3061
25.3208
25.3171
25.2819
25.3176
25.3045
25.3175
25.3171
25.2827
25.2806
25.3103
25.3175
25.3792
25.2819
25.3204
25.3208
25.3175
25.3171
25.2827
25.3009
25.3061
25.3061
25.2819
25.2819
25.3009
25.3208
25.3175
25.2819
25.2819
25.3009
25.3045
25.3175
25.3088
25.2827
25.348
25.3103
25.3061
25.2827
25.2827
25.348
25.3061
25.3061
25.3088
25.3171
25.3204
25.3061
25.3208
25.3792
25.3171
25.348
25.3045
25.3045
25.2827
25.2827
25.3009
25.3208
25.3208
25.2827
25.2827
25.3009
25.3175
25.3045
25.3171
25.2827
25.3204
25.3208
25.3061
25.3171
25.3792
25.3176
25.3045
25.3103
25.3088
25.2819
25.3176
25.3103
25.3061
25.3171
25.3088
25.3204
25.3208
25.3061
25.2819
25.3792
25.3204
25.3208
25.3103
25.3171
25.3171
25.348
25.3103
25.3045
25.3792
25.3792
25.3176
25.3103
25.3208
25.2819
25.3171
25.3204
25.3061
25.3061
25.3792
25.2819
25.3009
25.3103
25.3061
25.2819
25.3792
25.348
25.3045
25.3175
25.3792
25.2827
25.3204
25.3175
25.3103
25.3088
25.3088
25.3009
25.3061
25.3175
25.3088
25.3171
25.2806
25.3045
25.3208
25.3171
25.2827
25.348
25.3045
25.3208
25.3088
25.2819
25.348
25.3208
25.3208
25.3088
25.3088
25.3176
25.3045
25.3045
25.3171
25.2819
25.3009
25.3103
25.3175
25.2827
25.3088
25.348
25.3061
25.3103
25.2827
25.3171
25.3009
25.3175
25.3175
25.3171
25.3171
25.3204
25.3208
25.3208
25.3088
25.2827
25.3009
25.3103
25.3103
25.2827
25.3088
25.3009
25.3045
25.3175
25.2819
25.3088
25.348
25.3045
25.3061
25.2827
25.3792
25.3204
25.3175
25.3175
25.3171
25.2819
25.3009
25.3208
25.3175
25.3171
25.3792
25.3176
25.3175
25.3045
25.3171
25.2819
25.3204
25.3208
25.3061
25.2819
25.3792
25.348
25.3061
25.3208
25.2819
25.2827
25.2806
25.3103
25.3103
25.3792
25.3088
25.3204
25.3103
25.3061
25.2827
25.3792
25.3009
25.3045
25.3175
25.3088
25.2819
25.348
25.3208
25.3061
25.3171
25.3088
25.348
25.3208
25.3103
25.3792
25.3088
25.348
25.3208
25.3045
25.3171
25.2827
25.3176
25.3208
25.3045
25.2819
25.3088
25.3009
25.3103
25.3061
25.3088
25.3171
25.3009
25.3045
25.3103
25.3171
25.2819
25.348
25.3061
25.3208
25.3088
25.3171
25.3009
25.3175
25.3208
25.3171
25.3171
25.3176
25.3045
25.3045
25.2819
25.2819
25.3176
25.3175
25.3103
25.2827
25.2819
25.348
25.3045
25.3045
25.2819
25.2819
25.348
25.3103
25.3045
25.3088
25.3792
25.3204
25.3045
25.3175
25.3792
25.2827
25.2806
25.3061
25.3175
25.2819
25.2827
25.3204
25.3045
25.3208
25.3792
25.3792
25.3009
25.3175
25.3045
25.2827
25.2819
25.3009
25.3061
25.3103
25.2827
25.3792
25.348
25.3208
25.3175
25.2827
25.3792
25.3204
25.3045
25.3061
25.3088
25.2827
25.348
25.3175
25.3208
25.2827
25.3171
25.3204
25.3175
25.3103
25.3088
25.2827
25.2806
25.3175
25.3061
25.2827
25.3088
25.3009
25.3103
25.3103
25.2819
25.2819
25.3176
25.3175
25.3208
25.2827
25.2827
25.3009
25.3175
25.3103
25.3792
25.3088
25.3204
25.3208
25.3103
25.3088
25.3171
25.3176
25.3208
25.3175
25.2819
25.2827
25.3204
25.3045
25.3061
25.3792
25.3792
25.3204
25.3045
25.3061
25.3171
25.3088
25.3176
25.3045
25.3045
25.3792
25.3792
25.3009
25.3208
25.3208
25.2827
25.3088
25.3204
25.3061
25.3208
25.2827
25.3792
25.3204
25.3103
25.3103
25.3088
25.2819
25.348
25.3208
25.3061
25.2819
25.3088
25.348
25.3175
25.3045
25.3088
25.3171
25.348
25.3103
25.3103
25.2819
25.2827
25.3204
25.3045
25.3208
25.3792
25.2827
25.2806
25.3103
25.3061
25.3792
25.3792
25.348
25.3175
25.3103
25.3171
25.3088
25.3009
25.3103
25.3045
25.3171
25.3171
25.348
25.3061
25.3103
25.3171
25.3088
25.3204
25.3103
25.3045
25.3171
25.3171
25.2806
25.3208
25.3061
25.2819
25.3171
25.3176
25.3061
25.3045
25.3088
25.3792
25.2806
25.3045
25.3061
25.3171
25.2827
25.3204
25.3045
25.3208
25.2819
25.2827
25.3204
25.3175
25.3208
25.2827
25.2819
25.3009
25.3103
25.3175
25.2819
25.2819
25.3009
25.3103
25.3103
25.3088
25.3792
25.3009
25.3175
25.3061
25.2819
25.3088
25.3009
25.3175
25.3061
25.3088
25.2819
25.3009
25.3045
25.3045
25.2827
25.3088
25.3204
25.3175
25.3208
25.3088
25.2819
25.3009
25.3208
25.3061
25.2827
25.2827
25.348
25.3175
25.3103
25.3088
25.3792
25.3176
25.3045
25.3175
25.2827
25.2827
25.3176
25.3103
25.3103
25.2819
25.3792
25.3009
25.3103
25.3208
25.3088
25.2827
25.348
25.3175
25.3208
25.3171
25.2827
25.3009
25.3103
25.3045
25.3792
25.2819
25.3009
25.3061
25.3103
25.2827
25.2819
25.3009
25.3208
25.3175
25.3088
25.2827
25.2806
25.3103
25.3061
25.2819
25.3171
25.2806
25.3175
25.3103
25.3792
25.2827
25.3009
25.3045
25.3103
25.2819
25.3088
25.3176
25.3208
25.3045
25.2819
25.3171
25.3009
25.3045
25.3103
25.2819
25.3792
25.3204
25.3045
25.3208
25.2819
25.2819
25.3009
25.3208
25.3103
25.3171
25.2827
25.3009
25.3208
25.3103
25.3171
25.3088
25.3176
25.3061
25.3103
25.3088
25.2819
25.3009
25.3208
25.3045
25.2819
25.3088
25.3176
25.3045
25.3175
25.3088
25.3088
25.3204
25.3103
25.3045
25.3792
25.2827
25.3204
25.3208
25.3175
25.3792
25.2827
25.3009
25.3103
25.3208
25.3792
25.3171
25.348
25.3175
25.3208
25.3088
25.3171
25.3176
25.3208
25.3045
25.3792
25.3792
25.3176
25.3175
25.3061
25.2827
25.2827
25.3204
25.3175
25.3103
25.3171
25.3792
25.3204
25.3061
25.3045
25.3171
25.3088
25.3204
25.3208
25.3061
25.2819
25.3171
25.3204
25.3061
25.3208
25.3171
25.3171
25.3204
25.3208
25.3061
25.3792
25.3088
25.348
25.3208
25.3103
25.3088
25.3792
25.3176
25.3175
25.3045
25.3171
25.2819
25.3204
25.3175
25.3208
25.3171
25.2819
25.3204
25.3175
25.3103
25.3792
25.2819
25.3176
25.3045
25.3175
25.3171
25.3171
25.3009
25.3061
25.3103
25.3171
25.3088
25.3176
25.3208
25.3103
25.3088
25.2827
25.3176
25.3103
25.3175
25.3792
25.3088
25.3176
25.3061
25.3208
25.2819
25.3088
25.3009
25.3208
25.3103
25.2827
25.2819
25.3204
25.3103
25.3061
25.3171
25.2827
25.3009
25.3175
25.3103
25.3171
25.3088
25.3009
25.3103
25.3045
25.2819
25.3171
25.3009
25.3175
25.3208
25.3088
25.2827
25.3176
25.3045
25.3061
25.3171
25.3171
25.348
25.3045
25.3045
25.2819
25.2827
25.348
25.3208
25.3208
25.3088
25.3792
25.3009
25.3061
25.3045
25.2819
25.3171
25.3009
25.3061
25.3045
25.2819
25.2819
25.3204
25.3175
25.3103
25.3088
25.3792
25.3009
25.3103
25.3175
25.3171
25.3792
25.2806
25.3208
25.3045
25.3171
25.3792
25.3176
25.3103
25.3208
25.2827
25.2819
25.348
25.3061
25.3103
25.3171
25.3088
25.3009
25.3208
25.3175
25.3171
25.3171
25.3204
25.3175
25.3045
25.3088
25.2819
25.3009
25.3175
25.3061
25.2819
25.3088
25.3009
25.3103
25.3103
25.3792
25.2819
25.3204
25.3208
25.3103
25.3792
25.3088
25.3009
25.3175
25.3061
25.3171
25.3792
25.348
25.3175
25.3208
25.3171
25.2827
25.3204
25.3175
25.3103
25.3792
25.3792
25.348
25.3208
25.3061
25.2827
25.3088
25.3009
25.3175
25.3061
25.2827
25.2827
25.3176
25.3045
25.3103
25.3792
25.3171
25.348
25.3045
25.3103
25.3171
25.3088
25.348
25.3103
25.3045
25.3792
25.3171
25.3176
25.3045
25.3208
25.2819
25.3171
25.3176
25.3208
25.3045
25.3088
25.2819
25.348
25.3045
25.3061
25.2819
25.2827
25.3176
25.3175
25.3103
25.3792
25.2819
25.3009
25.3208
25.3208
25.3792
25.2819
25.3176
25.3061
25.3208
25.3088
25.2819
25.348
25.3061
25.3061
25.3792
25.2819
25.3009
25.3103
25.3208
25.3171
25.2819
25.3176
25.3175
25.3061
25.2819
25.3171
25.3009
25.3061
25.3061
25.3792
25.2827
25.2806
25.3061
25.3061
25.2819
25.3171
25.348
25.3045
25.3208
25.3088
25.2827
25.3204
25.3208
25.3045
25.2827
25.3792
25.3204
25.3061
25.3045
25.3088
25.2819
25.3009
25.3208
25.3061
25.3792
25.3792
25.348
25.3045
25.3103
25.3088
25.3792
25.348
25.3061
25.3175
25.3088
25.2827
25.3176
25.3103
25.3208
25.3792
25.3792
25.3176
25.3045
25.3175
25.2827
25.2827
25.3176
25.3175
25.3175
25.2827
25.3171
25.3176
25.3175
25.3061
25.2819
25.2819
25.3204
25.3175
25.3061
25.2819
25.3792
25.348
25.3045
25.3103
25.3792
25.3171
25.3176
25.3045
25.3045
25.2819
25.3088
25.3009
25.3208
25.3061
25.2819
25.3088
25.3204
25.3208
25.3175
25.3171
25.2819
25.3204
25.3103
25.3208
25.3792
25.3171
25.348
25.3061
25.3208
25.2827
25.3792
25.3204
25.3175
25.3045
25.2819
25.2819
25.3176
25.3103
25.3061
25.3171
25.3171
25.348
25.3061
25.3175
25.3088
25.3088
25.3204
25.3208
25.3103
25.2819
25.2819
25.3009
25.3208
25.3208
25.3171
25.3171
25.3204
25.3061
25.3103
25.3088
25.3171
25.3176
25.3208
25.3208
25.2819
25.3088
25.348
25.3045
25.3045
25.3171
25.2827
25.3204
25.3208
25.3103
25.3088
25.2827
25.3204
25.3103
25.3061
25.3171
25.2827
25.3204
25.3208
25.3103
25.3792
25.2819
25.3009
25.3061
25.3103
25.2827
25.3171
25.348
25.3045
25.3103
25.3792
25.3088
25.3176
25.3045
25.3103
25.2827
25.2819
25.3009
25.3045
25.3061
25.2827
25.3171
25.348
25.3103
25.3045
25.3792
25.3792
25.3176
25.3208
25.3045
25.2819
25.3171
25.3204
25.3175
25.3175
25.2827
25.3088
25.3204
25.3175
25.3045
25.2827
25.2819
25.2806
25.3175
25.3061
25.2819
25.2827
25.3176
25.3061
25.3103
25.3088
25.2827
25.3176
25.3103
25.3103
25.2827
25.3792
25.3176
25.3103
25.3208
25.3792
25.3792
25.3176
25.3208
25.3208
25.3171
25.3171
25.3176
25.3061
25.3103
25.2827
25.3088
25.3009
25.3208
25.3175
25.3792
25.3792
25.348
25.3103
25.3175
25.3088
25.3792
25.348
25.3061
25.3208
25.3171
25.2819
25.348
25.3061
25.3103
25.3792
25.2827
25.348
25.3175
25.3103
25.3171
25.2827
25.2806
25.3061
25.3061
25.3088
25.2819
25.3009
25.3175
25.3208
25.3088
25.3171
25.3176
25.3061
25.3045
25.3171
25.2819
25.3204
25.3045
25.3175
25.2819
25.3088
25.3204
25.3175
25.3045
25.3171
25.3171
25.348
25.3208
25.3175
25.3792
25.3088
25.3009
25.3103
25.3175
25.3088
25.3171
25.3204
25.3045
25.3103
25.2819
25.2827
25.3176
25.3061
25.3208
25.2819
25.2827
25.3176
25.3045
25.3103
25.2819
25.2819
25.2806
25.3045
25.3208
25.2819
25.3171
25.3176
25.3208
25.3061
25.2819
25.3171
25.3176
25.3103
25.3175
25.3792
25.2819
25.3204
25.3208
25.3061
25.2827
25.3792
25.3009
25.3208
25.3175
25.2819
25.3792
25.348
25.3208
25.3208
25.2827
25.3171
25.348
25.3103
25.3061
25.2827
25.2819
25.3204
25.3061
25.3061
25.3792
25.3171
25.348
25.3061
25.3103
25.2827
25.3792
25.3204
25.3045
25.3045
25.2827
25.3792
25.3009
25.3045
25.3061
25.3088
25.2827
25.3204
25.3103
25.3103
25.2819
25.3088
25.348
25.3103
25.3061
25.3171
25.3171
25.3204
25.3103
25.3208
25.2827
25.2827
25.2806
25.3208
25.3208
25.3171
25.3171
25.2806
25.3061
25.3208
25.3792
25.3088
25.3204
25.3103
25.3061
25.3792
25.3088
25.348
25.3103
25.3175
25.3088
25.3792
25.3009
25.3045
25.3045
25.2819
25.2819
25.348
25.3103
25.3045
25.3171
25.2827
25.3176
25.3175
25.3175
25.3088
25.2819
25.3204
25.3061
25.3175
25.2827
25.3171
25.3204
25.3045
25.3208
25.3171
25.3088
25.3204
25.3208
25.3175
25.3171
25.3171
25.348
25.3208
25.3061
25.3088
25.2827
25.3176
25.3175
25.3045
25.3792
25.2827
25.3176
25.3208
25.3103
25.2827
25.2819
25.348
25.3061
25.3103
25.3792
25.3792
25.3176
25.3061
25.3061
25.3792
25.3792
25.3009
25.3103
25.3175
25.3088
25.3088
25.348
25.3045
25.3208
25.2827
25.2827
25.348
25.3061
25.3061
25.3792
25.2827
25.348
25.3208
25.3045
25.3792
25.2819
25.3009
25.3175
25.3061
25.3792
25.3088
25.3176
25.3045
25.3103
25.3792
25.3171
25.3176
25.3103
25.3208
25.2827
25.3171
25.348
25.3045
25.3208
25.3088
25.3088
25.348
25.3208
25.3045
25.2819
25.3792
25.3176
25.3208
25.3175
25.2827
25.2827
25.2806
25.3175
25.3045
25.3088
25.3792
25.3176
25.3061
25.3045
25.3792
25.3088
25.3176
25.3061
25.3061
25.2819
25.3792
25.3176
25.3061
25.3208
25.3171
25.2827
25.348
25.3061
25.3208
25.3088
25.3171
25.3009
25.3061
25.3103
25.3171
25.3088
25.348
25.3208
25.3045
25.3088
25.2819
25.3204
25.3208
25.3175
25.2827
25.3792
25.2806
25.3061
25.3175
25.2819
25.2827
25.348
25.3103
25.3208
25.3792
25.3088
25.3009
25.3061
25.3045
25.2827
25.2819
25.3009
25.3208
25.3103
25.3088
25.3088
25.3009
25.3103
25.3175
25.3171
25.2819
25.3009
25.3175
25.3103
25.3088
25.2827
25.3204
25.3061
25.3045
25.3171
25.2827
25.348
25.3061
25.3103
25.2819
25.2827
25.3204
25.3061
25.3061
25.3171
25.3171
25.348
25.3045
25.3045
25.3088
25.3792
25.348
25.3103
25.3175
25.3088
25.3088
25.3009
25.3103
25.3208
25.3792
25.2819
25.348
25.3045
25.3103
25.3792
25.2827
25.348
25.3103
25.3175
25.3792
25.3088
25.3204
25.3045
25.3175
25.3088
25.2819
25.3176
25.3045
25.3061
25.3171
25.3088
25.3009
25.3175
25.3175
25.3171
25.3171
25.348
25.3175
25.3103
25.3088
25.3088
25.348
25.3061
25.3045
25.3792
25.2827
25.4302
25.3061
25.3103
25.3792
25.3171
25.348
25.3061
25.3103
25.3171
25.3171
25.3204
25.3045
25.3175
25.2819
25.3171
25.348
25.3045
25.3045
25.3171
25.2827
25.3176
25.3175
25.3045
25.3792
25.2819
25.3009
25.3103
25.3103
25.3171
25.2819
25.3009
25.3061
25.3103
25.3792
25.2827
25.3176
25.3103
25.3103
25.2819
25.3171
25.3176
25.3061
25.3103
25.3171
25.3171
25.3176
25.3208
25.3103
25.3088
25.3088
25.348
25.3103
25.3175
25.3088
25.2819
25.348
25.3175
25.3103
25.3088
25.3792
25.3204
25.3061
25.3175
25.2827
25.3171
25.3176
25.3208
25.3045
25.3792
25.3088
25.2806
25.3045
25.3045
25.3171
25.2827
25.3176
25.3061
25.3103
25.3792
25.3088
25.2806
25.3061
25.3045
25.3171
25.3792
25.3204
25.3208
25.3045
25.3171
25.2819
25.3009
25.3175
25.3061
25.2819
25.3088
25.3009
25.3061
25.3061
25.3792
25.3792
25.3176
25.3208
25.3208
25.3171
25.3792
25.3204
25.3061
25.3045
25.3171
25.3792
25.3176
25.3208
25.3045
25.3088
25.2827
25.3176
25.3103
25.3061
25.3088
25.2827
25.3009
25.3045
25.3103
25.3088
25.3088
25.3009
25.3175
25.3103
25.2819
25.2827
25.3009
25.3103
25.3208
25.3792
25.2819
25.3176
25.3208
25.3061
25.3171
25.3792
25.2806
25.3175
25.3045
25.3792
25.3171
25.3204
25.3208
25.3208
25.2827
25.3792
25.3176
25.3103
25.3103
25.2827
25.3792
25.3204
25.3103
25.3208
25.2819
25.2827
25.2806
25.3175
25.3103
25.2827
25.2827
25.2806
25.3208
25.3103
25.3171
25.2819
25.3204
25.3103
25.3208
25.2819
25.3171
25.3009
25.3103
25.3208
25.2819
25.3171
25.3176
25.3061
25.3208
25.3088
25.2819
25.3176
25.3045
25.3061
25.3088
25.3792
25.348
25.3061
25.3103
25.2827
25.3792
25.4302
25.3175
25.3061
25.3088
25.2827
25.3176
25.3208
25.3061
25.2819
25.3088
25.348
25.3045
25.3103
25.3171
25.2827
25.348
25.3175
25.3175
25.2827
25.3792
25.3176
25.3103
25.3103
25.3171
25.3171
25.348
25.3103
25.3175
25.2827
25.3171
25.3009
25.3208
25.3103
25.3171
25.3088
25.348
25.3208
25.3061
25.3171
25.3792
25.3204
25.3208
25.3208
25.3171
25.3171
25.3176
25.3208
25.3061
25.2827
25.3171
25.3204
25.3103
25.3208
25.3171
25.3171
25.3204
25.3061
25.3045
25.3171
25.2827
25.3176
25.3103
25.3175
25.3792
25.2827
25.348
25.3175
25.3175
25.3088
25.3171
25.3009
25.3175
25.3061
25.3792
25.2819
25.3009
25.3061
25.3208
25.3088
25.3792
25.3009
25.3045
25.3175
25.3792
25.2819
25.3176
25.3175
25.3208
25.3088
25.2827
25.3204
25.3208
25.3061
25.2827
25.3171
25.348
25.3045
25.3208
25.3088
25.3792
25.348
25.3175
25.3061
25.3792
25.3088
25.3009
25.3103
25.3175
25.3171
25.3088
25.3204
25.3208
25.3175
25.2827
25.3171
25.3176
25.3208
25.3175
25.3792
25.2819
25.3204
25.3103
25.3175
25.3171
25.2819
25.3176
25.3061
25.3045
25.2827
25.2819
25.3204
25.3103
25.3175
25.3792
25.3088
25.348
25.3061
25.3175
25.3171
25.3171
25.2845
25.3103
25.3061
25.2827
25.3088
25.3176
25.3061
25.3045
25.2819
25.3792
25.3204
25.3061
25.3208
25.2819
25.2827
25.3009
25.3208
25.3103
25.3171
25.3088
25.348
25.3103
25.3061
25.2827
25.3792
25.3204
25.3061
25.3103
25.2827
25.3088
25.3009
25.3103
25.3208
25.3088
25.3792
25.3009
25.3175
25.3061
25.3088
25.3088
25.3009
25.3061
25.3045
25.3171
25.3088
25.3009
25.3103
25.3061
25.3792
25.3088
25.3176
25.3175
25.3045
25.3088
25.3792
25.3204
25.3208
25.3061
25.3088
25.2819
25.348
25.3103
25.3045
25.2827
25.3088
25.3176
25.3208
25.3103
25.3792
25.3088
25.2806
25.3103
25.3061
25.2819
25.3088
25.3009
25.3045
25.3045
25.3088
25.3171
25.3176
25.3045
25.3175
25.3088
25.3088
25.3176
25.3061
25.3103
25.3792
25.3792
25.348
25.3208
25.3045
25.3171
25.3792
25.3009
25.3045
25.3103
25.2827
25.3171
25.348
25.3045
25.3175
25.3088
25.2827
25.3176
25.3061
25.3175
25.3088
25.2827
25.348
25.3175
25.3061
25.3792
25.3171
25.3176
25.3103
25.3061
25.2819
25.3088
25.3009
25.3175
25.3175
25.3171
25.3792
25.3176
25.3061
25.3045
25.2827
25.2819
25.3176
25.3045
25.3061
25.3088
25.3171
25.3176
25.3061
25.3045
25.3088
25.2827
25.348
25.3208
25.3208
25.3088
25.2819
25.3176
25.3103
25.3061
25.3171
25.2819
25.348
25.3061
25.3045
25.3792
25.2819
25.3176
25.3208
25.3061
25.3088
25.3088
25.3009
25.3103
25.3045
25.2827
25.3088
25.3009
25.3175
25.3061
25.3171
25.3088
25.3176
25.3175
25.3175
25.2827
25.3088
25.3176
25.3045
25.3045
25.2819
25.3792
25.3204
25.3103
25.3175
25.2827
25.2819
25.348
25.3208
25.3175
25.3088
25.2827
25.348
25.3103
25.3103
25.2827
25.2819
25.3176
25.3208
25.3208
25.3792
25.2827
25.3009
25.3208
25.3208
25.2827
25.3792
25.348
25.3175
25.3175
25.2827
25.3792
25.348
25.3175
25.3175
25.3171
25.2827
25.3009
25.3045
25.3103
25.3088
25.2819
25.348
25.3175
25.3045
25.3792
25.2827
25.3176
25.3175
25.3175
25.2827
25.3088
25.348
25.3061
25.3208
25.2827
25.2827
25.348
25.3061
25.3175
25.3171
25.3088
25.3204
25.3208
25.3208
25.2827
25.3088
25.3176
25.3208
25.3045
25.3792
25.2827
25.348
25.3103
25.3045
25.3792
25.3171
25.3009
25.3103
25.3175
25.2827
25.2819
25.3204
25.3045
25.3061
25.3088
25.3792
25.3204
25.3061
25.3103
25.2827
25.3171
25.3204
25.3061
25.3175
25.3171
25.3792
25.3009
25.3061
25.3208
25.2819
25.2827
25.3204
25.3061
25.3061
25.3792
25.3088
25.348
25.3175
25.3175
25.3792
25.2827
25.348
25.3175
25.3061
25.2827
25.3792
25.348
25.3103
25.3175
25.3088
25.2827
25.3176
25.3103
25.3208
25.3088
25.3088
25.3009
25.3045
25.3045
25.3088
25.3171
25.3204
25.3175
25.3208
25.2827
25.2827
25.3204
25.3208
25.3208
25.3088
25.3792
25.3009
25.3061
25.3061
25.3171
25.3792
25.3204
25.3175
25.3045
25.3088
25.2819
25.3176
25.3045
25.3208
25.3088
25.3792
25.348
25.3061
25.3061
25.2819
25.3792
25.3176
25.3061
25.3061
25.3792
25.2827
25.3176
25.3045
25.3208
25.3088
25.2827
25.3176
25.3208
25.3045
25.2819
25.3088
25.3204
25.3175
25.3175
25.2819
25.2827
25.3009
25.3103
25.3175
25.3171
25.2819
25.3204
25.3061
25.3175
25.3171
25.3171
25.3009
25.3061
25.3045
25.3792
25.2827
25.348
25.3175
25.3045
25.2819
25.3792
25.348
25.3208
25.3045
25.3171
25.3792
25.3204
25.3208
25.3208
25.2827
25.3088
25.3176
25.3061
25.3208
25.3171
25.2819
25.2806
25.3045
25.3045
25.3171
25.3171
25.3204
25.3061
25.3045
25.2827
25.2827
25.3009
25.3208
25.3208
25.2819
25.3171
25.3204
25.3175
25.3061
25.3792
25.2827
25.3176
25.3208
25.3208
25.2819
25.2827
25.3009
25.3175
25.3045
25.3088
25.3088
25.3204
25.3175
25.3061
25.3792
25.3088
25.3009
25.3103
25.3175
25.3171
25.3171
25.3009
25.3103
25.3045
25.2819
25.2827
25.3009
25.3045
25.3103
25.3792
25.3088
25.3176
25.3175
25.3103
25.3088
25.2819
25.3176
25.3061
25.3045
25.3088
25.2819
25.348
25.3175
25.3175
25.3792
25.3792
25.3176
25.3175
25.3208
25.2819
25.2819
25.3176
25.3045
25.3045
25.3088
25.3088
25.3176
25.3061
25.3061
25.3792
25.3792
25.3204
25.3045
25.3045
25.3088
25.3088
25.348
25.3103
25.3208
25.3171
25.2819
25.3176
25.3045
25.3208
25.2827
25.3088
25.2806
25.3208
25.3045
25.2827
25.3171
25.3176
25.3103
25.3175
25.3088
25.3088
25.348
25.3061
25.3061
25.2827
25.3171
25.3176
25.3103
25.3175
25.2827
25.3792
25.3009
25.3175
25.3103
25.2827
25.2819
25.3009
25.3045
25.3061
25.2819
25.2827
25.3204
25.3208
25.3061
25.3088
25.3088
25.3204
25.3175
25.3208
25.2819
25.3171
25.3009
25.3045
25.3103
25.2827
25.2827
25.3204
25.3045
25.3175
25.2827
25.2827
25.3204
25.3103
25.3061
25.3171
25.2819
25.3009
25.3208
25.3061
25.3792
25.3171
25.3009
25.3103
25.3061
25.3088
25.3171
25.348
25.3061
25.3045
25.3088
25.2827
25.3204
25.3103
25.3103
25.2827
25.2827
25.3009
25.3045
25.3061
25.3088
25.3088
25.3204
25.3045
25.3045
25.3171
25.2827
25.3009
25.3208
25.3045
25.3088
25.2827
25.3204
25.3061
25.3061
25.3792
25.2827
25.2845
25.3175
25.3208
25.2827
25.2827
25.348
25.3103
25.3175
25.2827
25.2827
25.2806
25.3103
25.3061
25.3171
25.3088
25.3176
25.3175
25.3061
25.2827
25.3171
25.2806
25.3045
25.3208
25.3792
25.3088
25.3009
25.3103
25.3175
25.2827
25.2827
25.348
25.3103
25.3208
25.3792
25.3088
25.2806
25.3103
25.3061
25.3088
25.2827
25.348
25.3103
25.3175
25.3792
25.2827
25.3009
25.3103
25.3045
25.2827
25.2827
25.348
25.3103
25.3061
25.2819
25.3171
25.3009
25.3061
25.3103
25.3088
25.2827
25.3176
25.3061
25.3208
25.3792
25.2827
25.3176
25.3175
25.3045
25.3792
25.2827
25.3176
25.3175
25.3045
25.3792
25.3171
25.3204
25.3061
25.3208
25.2827
25.3088
25.3204
25.3061
25.3045
25.3792
25.2827
25.3176
25.3061
25.3208
25.2819
25.3171
25.3009
25.3061
25.3061
25.3088
25.2827
25.3176
25.3175
25.3103
25.3171
25.2819
25.3009
25.3103
25.3061
25.3171
25.3088
25.3176
25.3208
25.3175
25.3088
25.2819
25.348
25.3045
25.3103
25.2827
25.3792
25.3204
25.3061
25.3061
25.3088
25.2827
25.348
25.3045
25.3045
25.2819
25.2827
25.3009
25.3103
25.3208
25.3088
25.3792
25.348
25.3045
25.3175
25.3088
25.3088
25.2806
25.3175
25.3208
25.3171
25.3088
25.2806
25.3208
25.3175
25.3088
25.3171
25.3176
25.3103
25.3208
25.3088
25.3792
25.3009
25.3208
25.3061
25.3088
25.2827
25.348
25.3175
25.3208
25.2819
25.2827
25.3009
25.3208
25.3045
25.3088
25.2827
25.3009
25.3061
25.3103
25.3088
25.3171
25.3009
25.3103
25.3045
25.2819
25.3792
25.348
25.3103
25.3061
25.3792
25.3088
25.3009
25.3175
25.3103
25.2819
25.2819
25.3009
25.3208
25.3045
25.2827
25.2827
25.3009
25.3208
25.3175
25.3792
25.3171
25.3176
25.3208
25.3175
25.2827
25.3792
25.3009
25.3045
25.3208
25.3088
25.3088
25.3176
25.3045
25.3061
25.2819
25.3088
25.3176
25.3175
25.3061
25.2819
25.3171
25.348
25.3208
25.3061
25.2819
25.3171
25.3176
25.3175
25.3061
25.3171
25.3171
25.3204
25.3045
25.3208
25.3088
25.3088
25.3176
25.3061
25.3175
25.2827
25.3171
25.348
25.3061
25.3061
25.3792
25.3792
25.3176
25.3061
25.3175
25.3088
25.3171
25.348
25.3175
25.3061
25.3088
25.2819
25.348
25.3175
25.3045
25.2827
25.3171
25.3009
25.3103
25.3045
25.3088
25.3792
25.348
25.3061
25.3045
25.3171
25.3171
25.3204
25.3175
25.3175
25.2827
25.2827
25.3204
25.3061
25.3208
25.2819
25.2827
25.3204
25.3045
25.3061
25.3171
25.3792
25.2806
25.3175
25.3061
25.2827
25.2819
25.3176
25.3045
25.3045
25.2819
25.3792
25.3009
25.3061
25.3061
25.3792
25.3792
25.3009
25.3045
25.3103
25.3171
25.3088
25.3204
25.3061
25.3103
25.3088
25.3088
25.348
25.3045
25.3061
25.3792
25.3088
25.3176
25.3045
25.3103
25.2819
25.3088
25.3204
25.3103
25.3103
25.2819
25.3792
25.348
25.3208
25.3208
25.3171
25.2827
25.3176
25.3208
25.3045
25.2827
25.3792
25.348
25.3045
25.3103
25.2827
25.3792
25.3009
25.3175
25.3208
25.2827
25.2827
25.3176
25.3175
25.3175
25.2819
25.3171
25.3204
25.3175
25.3045
25.3088
25.2819
25.3009
25.3103
25.3208
25.2819
25.2827
25.348
25.3175
25.3208
25.3792
25.3792
25.3204
25.3208
25.3061
25.3088
25.3171
25.3204
25.3103
25.3061
25.2827
25.3171
25.3204
25.3061
25.3175
25.2827
25.3792
25.3009
25.3061
25.3061
25.2819
25.2819
25.3204
25.3175
25.3045
25.2819
25.3088
25.348
25.3103
25.3175
25.3792
25.3792
25.3176
25.3045
25.3208
25.3088
25.3792
25.348
25.3175
25.3208
25.3792
25.3171
25.3176
25.3103
25.3103
25.3792
25.2819
25.3009
25.3175
25.3175
25.3171
25.3171
25.3204
25.3061
25.3175
25.3088
25.2827
25.2806
25.3045
25.3045
25.3792
25.3088
25.3204
25.3103
25.3061
25.3088
25.3088
25.3009
25.3175
25.3045
25.3088
25.3088
25.3176
25.3103
25.3045
25.2827
25.2819
25.3176
25.3208
25.3175
25.2819
25.2819
25.3176
25.3208
25.3045
25.3088
25.2819
25.3204
25.3175
25.3061
25.2827
25.3171
25.3009
25.3061
25.3175
25.3171
25.3792
25.348
25.3045
25.3175
25.3171
25.3171
25.3009
25.3208
25.3175
25.3088
25.2827
25.348
25.3061
25.3175
25.3171
25.3792
25.348
25.3045
25.3175
25.3792
25.3171
25.3176
25.3045
25.3175
25.3088
25.3792
25.3176
25.3208
25.3045
25.2819
25.2827
25.3204
25.3175
25.3208
25.3171
25.2827
25.3204
25.3208
25.3208
25.3171
25.2819
25.3009
25.3175
25.3045
25.3088
25.2827
25.348
25.3045
25.3045
25.2819
25.3792
25.3176
25.3061
25.3175
25.3088
25.3088
25.348
25.3208
25.3061
25.3171
25.2827
25.3204
25.3103
25.3103
25.3792
25.3088
25.3176
25.3103
25.3061
25.3792
25.2827
25.348
25.3061
25.3103
25.2827
25.2819
25.3176
25.3208
25.3175
25.2819
25.3088
25.3204
25.3045
25.3175
25.3792
25.3171
25.3204
25.3208
25.3061
25.3792
25.3792
25.3176
25.3045
25.3061
25.3171
25.2827
25.2806
25.3103
25.3208
25.3171
25.2819
25.3204
25.3061
25.3103
25.3171
25.3792
25.3176
25.3208
25.3061
25.3792
25.3171
25.348
25.3045
25.3061
25.3171
25.2827
25.3204
25.3061
25.3175
25.2827
25.2819
25.3176
25.3103
25.3045
25.3171
25.3088
25.3204
25.3103
25.3061
25.2827
25.2819
25.348
25.3061
25.3208
25.3792
25.2827
25.3176
25.3103
25.3208
25.2819
25.2827
25.3009
25.3103
25.3208
25.2827
25.2827
25.3204
25.3103
25.3208
25.3792
25.2819
25.3176
25.3045
25.3175
25.3792
25.2827
25.3204
25.3103
25.3103
25.3792
25.2827
25.2806
25.3208
25.3045
25.2827
25.3171
25.3176
25.3103
25.3045
25.3171
25.3171
25.348
25.3103
25.3061
25.3792
25.3792
25.3204
25.3061
25.3045
25.3792
25.3171
25.3176
25.3061
25.3045
25.3171
25.3171
25.3204
25.3175
25.3175
25.2819
25.2827
25.3176
25.3061
25.3175
25.2819
25.3792
25.3009
25.3045
25.3103
25.3088
25.2819
25.3009
25.3045
25.3061
25.3171
25.3171
25.348
25.3103
25.3061
25.2819
25.2827
25.3176
25.3208
25.3061
25.3792
25.3792
25.3176
25.3208
25.3175
25.2819
25.2827
25.3176
25.3103
25.3103
25.3088
25.3792
25.3204
25.3208
25.3175
25.2819
25.3792
25.3176
25.3103
25.3045
25.3171
25.2819
25.348
25.3061
25.3061
25.3792
25.2819
25.348
25.3175
25.3208
25.3171
25.3171
25.3009
25.3061
25.3061
25.3088
25.2819
25.3204
25.3103
25.3175
25.3088
25.3088
25.3176
25.3175
25.3045
25.3171
25.3171
25.2806
25.3103
25.3175
25.2819
25.3088
25.3204
25.3175
25.3103
25.2827
25.3792
25.348
25.3103
25.3045
25.3171
25.3088
25.3176
25.3045
25.3045
25.2819
25.3088
25.3176
25.3175
25.3175
25.3792
25.3792
25.3009
25.3175
25.3045
25.2819
25.2819
25.348
25.3045
25.3061
25.3171
25.2819
25.348
25.3175
25.3045
25.3088
25.3792
25.3009
25.3045
25.3045
25.3171
25.2819
25.3204
25.3208
25.3103
25.3171
25.3088
25.3204
25.3061
25.3045
25.2819
25.3088
25.348
25.3208
25.3103
25.3171
25.2827
25.3204
25.3061
25.3045
25.3088
25.3171
25.3204
25.3103
25.3103
25.3171
25.3088
25.3009
25.3103
25.3175
25.2819
25.3792
25.3009
25.3208
25.3061
25.3088
25.3171
25.348
25.3061
25.3103
25.3088
25.3088
25.3204
25.3045
25.3061
25.2819
25.2819
25.3204
25.3208
25.3061
25.2827
25.2819
25.3176
25.3103
25.3061
25.2827
25.3088
25.3009
25.3208
25.3103
25.2827
25.3792
25.3009
25.3175
25.3208
25.3792
25.3088
25.3204
25.3045
25.3175
25.3088
25.2827
25.3009
25.3103
25.3061
25.2819
25.3088
25.348
25.3175
25.3175
25.2819
25.3171
25.3204
25.3045
25.3045
25.3792
25.2827
25.3204
25.3175
25.3061
25.2827
25.2819
25.3009
25.3208
25.3045
25.3088
25.2827
25.3204
25.3175
25.3103
25.3088
25.3792
25.3176
25.3208
25.3061
25.2819
25.2819
25.3176
25.3103
25.3045
25.2827
25.3792
25.3176
25.3045
25.3045
25.3088
25.2827
25.3009
25.3175
25.3103
25.3171
25.2827
25.3009
25.3061
25.3175
25.2819
25.2819
25.3204
25.3103
25.3103
25.2827
25.3171
25.2845
25.3045
25.3103
25.3171
25.3171
25.3204
25.3208
25.3103
25.2819
25.3088
25.348
25.3103
25.3061
25.2827
25.2827
25.348
25.3103
25.3208
25.2819
25.3171
25.348
25.3208
25.3045
25.2819
25.3792
25.348
25.3103
25.3175
25.3792
25.2827
25.3009
25.3103
25.3175
25.2819
25.2819
25.3176
25.3208
25.3103
25.2819
25.3171
25.348
25.3045
25.3045
25.3088
25.3792
25.3009
25.3061
25.3103
25.2827
25.3171
25.3009
25.3045
25.3061
25.3088
25.3171
25.3204
25.3208
25.3103
25.2827
25.2819
25.3204
25.3061
25.3103
25.3171
25.3792
25.2806
25.3061
25.3208
25.3171
25.2827
25.348
25.3061
25.3175
25.3088
25.3171
25.3176
25.3045
25.3061
25.2827
25.3088
25.3204
25.3175
25.3061
25.3792
25.2819
25.3204
25.3208
25.3045
25.2819
25.3792
25.3009
25.3045
25.3208
25.2819
25.3171
25.348
25.3061
25.3175
25.3792
25.2819
25.3176
25.3103
25.3061
25.2819
25.2827
25.348
25.3061
25.3061
25.2819
25.3792
25.3204
25.3061
25.3103
25.3792
25.2819
25.2806
25.3061
25.3208
25.3171
25.3088
25.3009
25.3045
25.3103
25.2819
25.3171
25.3009
25.3208
25.3103
25.3171
25.2819
25.3009
25.3175
25.3208
25.3792
25.2819
25.3204
25.3103
25.3045
25.3088
25.2819
25.348
25.3045
25.3061
25.3088
25.3171
25.3204
25.3103
25.3061
25.3171
25.3792
25.2806
25.3061
25.3175
25.3171
25.3171
25.3009
25.3045
25.3045
25.3088
25.3088
25.348
25.3103
25.3061
25.2819
25.2827
25.3009
25.3103
25.3208
25.3088
25.3088
25.3176
25.3175
25.3061
25.3088
25.3171
25.3176
25.3208
25.3061
25.3171
25.3792
25.3176
25.3061
25.3103
25.2819
25.3171
25.348
25.3175
25.3045
25.3171
25.3171
25.3176
25.3045
25.3208
25.2827
25.3792
25.348
25.3175
25.3045
25.3792
25.3088
25.3176
25.3208
25.3045
25.3792
25.2819
25.3009
25.3103
25.3175
25.3792
25.2819
25.3204
25.3208
25.3208
25.2819
25.2819
25.3176
25.3208
25.3061
25.3088
25.3088
25.3204
25.3103
25.3061
25.2819
25.3171
25.3009
25.3045
25.3061
25.3171
25.2819
25.3204
25.3045
25.3175
25.2819
25.2827
25.348
25.3208
25.3175
25.3088
25.2827
25.3204
25.3208
25.3045
25.2819
25.2819
25.3009
25.3103
25.3061
25.2827
25.2827
25.3204
25.3061
25.3061
25.3171
25.3088
25.3204
25.3061
25.3103
25.3792
25.2827
25.2806
25.3103
25.3175
25.2819
25.2827
25.3176
25.3045
25.3208
25.3088
25.3088
25.2845
25.3175
25.3103
25.3088
25.3088
25.3176
25.3103
25.3061
25.3171
25.2819
25.348
25.3045
25.3175
25.3088
25.2819
25.3009
25.3208
25.3208
25.3088
25.2827
25.3176
25.3061
25.3208
25.2827
25.3171
25.348
25.3061
25.3175
25.3171
25.2827
25.3204
25.3175
25.3175
25.3171
25.3792
25.348
25.3045
25.3045
25.3171
25.3171
25.2806
25.3208
25.3175
25.3171
25.3088
25.3176
25.3175
25.3103
25.2819
25.3171
25.3176
25.3103
25.3103
25.3171
25.3088
25.3204
25.3175
25.3103
25.3171
25.2819
25.3204
25.3175
25.3103
25.2819
25.2819
25.348
25.3045
25.3103
25.2827
25.3171
25.348
25.3061
25.3061
25.3171
25.3171
25.348
25.3061
25.3045
25.2819
25.3171
25.348
25.3103
25.3103
25.3088
25.3088
25.348
25.3103
25.3175
25.3792
25.3088
25.3176
25.3208
25.3045
25.3088
25.2827
25.2806
25.3175
25.3208
25.2819
25.2819
25.3176
25.3061
25.3175
25.2827
25.3792
25.348
25.3045
25.3061
25.3088
25.2827
25.3009
25.3208
25.3103
25.2827
25.3171
25.2806
25.3045
25.3045
25.3171
25.3171
25.3176
25.3208
25.3045
25.3171
25.2819
25.3009
25.3103
25.3208
25.2819
25.3171
25.3009
25.3103
25.3045
25.2827
25.2819
25.3009
25.3175
25.3208
25.3088
25.2819
25.3009
25.3208
25.3103
25.3088
25.3088
25.348
25.3103
25.3061
25.2827
25.3171
25.3009
25.3208
25.3175
25.3171
25.3171
25.3204
25.3061
25.3103
25.2819
25.2827
25.3009
25.3061
25.3175
25.3792
25.2827
25.3009
25.3175
25.3045
25.3171
25.2827
25.2806
25.3103
25.3061
25.3088
25.3171
25.3176
25.3045
25.3061
25.3171
25.2827
25.3176
25.3061
25.3103
25.3088
25.2827
25.2806
25.3175
25.3061
25.3171
25.3171
25.348
25.3045
25.3175
25.2819
25.3792
25.3204
25.3103
25.3175
25.2819
25.3088
25.3009
25.3061
25.3175
25.2819
25.3088
25.3009
25.3208
25.3103
25.2827
25.3792
25.348
25.3045
25.3103
25.2827
25.2827
25.3204
25.3061
25.3061
25.3171
25.3088
25.3204
25.3045
25.3103
25.3088
25.2819
25.3009
25.3061
25.3045
25.3088
25.2827
25.3204
25.3208
25.3103
25.2827
25.3088
25.2806
25.3061
25.3175
25.2827
25.3088
25.3176
25.3208
25.3045
25.3088
25.3088
25.3176
25.3103
25.3175
25.2819
25.3171
25.348
25.3175
25.3045
25.3088
25.3088
25.3009
25.3103
25.3045
25.3792
25.3792
25.3009
25.3045
25.3103
25.3792
25.2819
25.2806
25.3045
25.3103
25.3171
25.3171
25.3009
25.3061
25.3045
25.3792
25.3171
25.3176
25.3103
25.3208
25.2827
25.3088
25.348
25.3103
25.3045
25.2819
25.2827
25.3009
25.3208
25.3208
25.3088
25.3171
25.3204
25.3061
25.3061
25.2827
25.3171
25.3009
25.3061
25.3103
25.2819
25.3171
25.2806
25.3103
25.3045
25.2819
25.2819
25.3204
25.3208
25.3103
25.2819
25.3088
25.2806
25.3061
25.3045
25.3792
25.3088
25.3009
25.3175
25.3103
25.2819
25.3088
25.348
25.3103
25.3103
25.3792
25.3171
25.348
25.3208
25.3045
25.3171
25.3088
25.3009
25.3208
25.3045
25.2819
25.2819
25.348
25.3175
25.3208
25.3088
25.3792
25.348
25.3061
25.3175
25.3171
25.2819
25.3176
25.3175
25.3208
25.3171
25.2819
25.348
25.3208
25.3103
25.3171
25.2827
25.2806
25.3061
25.3045
25.3088
25.2819
25.3009
25.3103
25.3208
25.3088
25.3792
25.3204
25.3061
25.3175
25.3088
25.3171
25.3009
25.3103
25.3103
25.3792
25.2819
25.3204
25.3045
25.3045
25.2827
25.3088
25.3176
25.3061
25.3103
25.3088
25.3171
25.348
25.3061
25.3103
25.3088
25.3088
25.3009
25.3103
25.3208
25.3792
25.3792
25.3204
25.3061
25.3103
25.3171
25.3171
25.3176
25.3175
25.3045
25.2819
25.3088
25.3204
25.3208
25.3103
25.2827
25.2827
25.3009
25.3103
25.3175
25.3088
25.3088
25.348
25.3103
25.3061
25.2819
25.3171
25.348
25.3175
25.3208
25.3792
25.3088
25.3204
25.3045
25.3061
25.2827
25.3088
25.348
25.3103
25.3045
25.2827
25.3088
25.3176
25.3061
25.3208
25.3792
25.2827
25.348
25.3045
25.3045
25.2827
25.2827
25.348
25.3103
25.3175
25.2827
25.3171
25.3204
25.3045
25.3208
25.3792
25.3088
25.3009
25.3208
25.3061
25.3171
25.2819
25.3009
25.3045
25.3103
25.3792
25.3792
25.348
25.3061
25.3103
25.3088
25.2819
25.3204
25.3045
25.3208
25.2819
25.3792
25.3009
25.3103
25.3061
25.3088
25.3088
25.348
25.3208
25.3045
25.3792
25.2827
25.3009
25.3061
25.3045
25.2819
25.2819
25.3176
25.3061
25.3061
25.2819
25.2827
25.3176
25.3045
25.3175
25.2819
25.3088
25.3176
25.3061
25.3175
25.2819
25.3171
25.3204
25.3175
25.3045
25.2827
25.3088
25.3204
25.3045
25.3208
25.3792
25.2819
25.3176
25.3103
25.3208
25.3171
25.3088
25.2806
25.3175
25.3103
25.2827
25.2819
25.2806
25.3175
25.3208
25.3171
25.3088
25.3009
25.3103
25.3208
25.3088
25.2819
25.3204
25.3103
25.3208
25.2827
25.2819
25.348
25.3061
25.3103
25.2827
25.2827
25.3204
25.3061
25.3208
25.2827
25.3792
25.348
25.3175
25.3103
25.3792
25.2827
25.3009
25.3103
25.3208
25.2819
25.2819
25.3176
25.3061
25.3103
25.2827
25.3088
25.348
25.3175
25.3045
25.3171
25.3088
25.348
25.3061
25.3175
25.3088
25.3792
25.348
25.3208
25.3045
25.3088
25.3088
25.3176
25.3103
25.3045
25.3792
25.2827
25.348
25.3175
25.3045
25.3171
25.2827
25.3176
25.3208
25.3175
25.3088
25.3171
25.3176
25.3103
25.3061
25.2827
25.2819
25.348
25.3175
25.3045
25.3171
25.2819
25.2806
25.3208
25.3061
25.2827
25.3088
25.2806
25.3103
25.3045
25.3171
25.2819
25.348
25.3208
25.3061
25.3171
25.3088
25.3176
25.3061
25.3208
25.3792
25.2827
25.2806
25.3061
25.3208
25.2827
25.3088
25.348
25.3103
25.3175
25.3171
25.2827
25.3176
25.3103
25.3061
25.3088
25.3792
25.3204
25.3208
25.3045
25.3088
25.3088
25.3176
25.3208
25.3061
25.3792
25.2819
25.2806
25.3045
25.3045
25.3088
25.2819
25.3204
25.3061
25.3208
25.3088
25.2827
25.3204
25.3175
25.3175
25.3171
25.3088
25.3176
25.3103
25.3103
25.3171
25.3088
25.3176
25.3208
25.3103
25.2819
25.3792
25.3204
25.3045
25.3208
25.3792
25.3088
25.3204
25.3103
25.3208
25.3792
25.2827
25.3204
25.3175
25.3175
25.3792
25.3171
25.3176
25.3208
25.3175
25.2827
25.3171
25.3176
25.3045
25.3175
25.2827
25.3171
25.3204
25.3175
25.3045
25.3171
25.3171
25.3009
25.3208
25.3061
25.2819
25.3171
25.3176
25.3103
25.3061
25.2819
25.3792
25.348
25.3061
25.3061
25.2819
25.3171
25.2845
25.3208
25.3103
25.3088
25.2819
25.3009
25.3103
25.3208
25.2827
25.2827
25.348
25.3045
25.3103
25.3792
25.3792
25.3176
25.3103
25.3045
25.3792
25.3088
25.3176
25.3208
25.3208
25.2819
25.2827
25.348
25.3061
25.3175
25.3792
25.3171
25.3176
25.3061
25.3045
25.3792
25.3088
25.4302
25.3175
25.3045
25.3792
25.3171
25.348
25.3208
25.3045
25.2827
25.3792
25.3204
25.3208
25.3208
25.3171
25.3088
25.3176
25.3175
25.3103
25.3088
25.2827
25.3204
25.3175
25.3208
25.3792
25.3088
25.3009
25.3175
25.3208
25.2827
25.2827
25.3204
25.3045
25.3175
25.3792
25.3171
25.3204
25.3103
25.3045
25.2819
25.3171
25.2806
25.3208
25.3103
25.2827
25.2819
25.2806
25.3061
25.3208
25.3792
25.2827
25.348
25.3061
25.3208
25.3088
25.2819
25.2806
25.3061
25.3045
25.2819
25.2827
25.3204
25.3175
25.3103
25.3171
25.2819
25.348
25.3045
25.3175
25.2819
25.2827
25.3009
25.3045
25.3045
25.2827
25.2819
25.3176
25.3208
25.3175
25.3171
25.3792
25.3204
25.3103
25.3208
25.3088
25.3171
25.348
25.3208
25.3175
25.3171
25.3792
25.348
25.3208
25.3103
25.3171
25.3088
25.3204
25.3175
25.3103
25.3792
25.2827
25.348
25.3175
25.3208
25.3171
25.3792
25.348
25.3175
25.3208
25.3792
25.3792
25.3176
25.3103
25.3061
25.2819
25.3088
25.3009
25.3175
25.3045
25.3171
25.3171
25.348
25.3045
25.3061
25.2819
25.3792
25.3009
25.3103
25.3103
25.3792
25.3171
25.3009
25.3061
25.3208
25.2819
25.3171
25.3204
25.3045
25.3175
25.3792
25.3792
25.348
25.3045
25.3061
25.3792
25.3792
25.3009
25.3045
25.3103
25.3792
25.2819
25.3009
25.3175
25.3103
25.3088
25.2827
25.348
25.3175
25.3208
25.3088
25.3088
25.348
25.3103
25.3061
25.3088
25.3088
25.3176
25.3061
25.3045
25.3088
25.3088
25.3204
25.3208
25.3208
25.2819
25.2819
25.3009
25.3103
25.3175
25.2819
25.3792
25.3009
25.3045
25.3061
25.3792
25.3171
25.3009
25.3045
25.3045
25.2827
25.2827
25.3176
25.3175
25.3061
25.2827
25.3088
25.3204
25.3103
25.3208
25.2827
25.3792
25.2806
25.3175
25.3061
25.3792
25.2819
25.2806
25.3208
25.3175
25.3171
25.3171
25.3204
25.3103
25.3208
25.2827
25.2819
25.3204
25.3045
25.3208
25.3792
25.2827
25.3009
25.3103
25.3103
25.2827
25.2827
25.3204
25.3208
25.3208
25.3792
25.3792
25.3009
25.3208
25.3045
25.3171
25.2819
25.3204
25.3045
25.3045
25.2827
25.3171
25.3176
25.3175
25.3175
25.2819
25.3792
25.3204
25.3103
25.3208
25.2819
25.3088
25.348
25.3208
25.3103
25.3792
25.2827
25.3204
25.3045
25.3208
25.3171
25.3171
25.3176
25.3045
25.3061
25.2819
25.2819
25.3009
25.3103
25.3061
25.3088
25.2827
25.3204
25.3061
25.3208
25.3171
25.2827
25.348
25.3208
25.3103
25.2819
25.2819
25.2806
25.3103
25.3061
25.3171
25.3792
25.3176
25.3045
25.3175
25.3792
25.3171
25.3204
25.3061
25.3061
25.2827
25.3792
25.3176
25.3208
25.3103
25.3171
25.2819
25.3204
25.3061
25.3045
25.3088
25.2827
25.3009
25.3208
25.3208
25.3171
25.3171
25.3176
25.3103
25.3175
25.3088
25.3792
25.2806
25.3103
25.3175
25.2827
25.2819
25.3009
25.3208
25.3208
25.3792
25.3171
25.348
25.3175
25.3103
25.2819
25.3171
25.3176
25.3045
25.3061
25.3088
25.3792
25.3204
25.3103
25.3103
25.3171
25.3171
25.348
25.3045
25.3208
25.3088
25.3088
25.348
25.3175
25.3208
25.3171
25.2819
25.3176
25.3103
25.3061
25.3792
25.3171
25.2806
25.3103
25.3103
25.2827
25.2819
25.3009
25.3045
25.3103
25.3792
25.3088
25.348
25.3045
25.3061
25.3792
25.3792
25.3176
25.3208
25.3175
25.2819
25.2827
25.3176
25.3208
25.3175
25.3171
25.3171
25.348
25.3103
25.3045
25.2827
25.2827
25.3009
25.3061
25.3208
25.3171
25.3088
25.3009
25.3045
25.3061
25.2827
25.3088
25.3009
25.3208
25.3208
25.2819
25.3088
25.3176
25.3045
25.3103
25.3792
25.2819
25.3009
25.3061
25.3175
25.3088
25.3088
25.348
25.3103
25.3045
25.3088
25.2819
25.3204
25.3045
25.3208
25.2819
25.2819
25.3176
25.3045
25.3175
25.2819
25.2819
25.348
25.3045
25.3103
25.2827
25.3792
25.3176
25.3208
25.3208
25.3171
25.3792
25.3176
25.3208
25.3061
25.2819
25.3171
25.348
25.3175
25.3045
25.3171
25.2819
25.3009
25.3061
25.3208
25.3171
25.3792
25.3176
25.3208
25.3175
25.3171
25.3792
25.3176
25.3208
25.3208
25.3171
25.2819
25.3009
25.3175
25.3061
25.2819
25.2819
25.3176
25.3045
25.3175
25.2827
25.3171
25.3204
25.3045
25.3045
25.3171
25.2819
25.3204
25.3061
25.3103
25.3088
25.3088
25.3009
25.3175
25.3208
25.2827
25.3088
25.3204
25.3103
25.3061
25.3088
25.3171
25.348
25.3045
25.3175
25.2827
25.3088
25.3009
25.3208
25.3175
25.3088
25.2827
25.348
25.3175
25.3045
25.2827
25.3171
25.3176
25.3208
25.3208
25.3088
25.3088
25.348
25.3208
25.3061
25.2827
25.3088
25.3176
25.3061
25.3208
25.3792
25.3792
25.3204
25.3045
25.3045
25.2819
25.2819
25.3009
25.3045
25.3061
25.3088
25.2827
25.3204
25.3045
25.3061
25.2827
25.3171
25.3204
25.3103
25.3175
25.3088
25.2819
25.348
25.3103
25.3061
25.3792
25.3171
25.3204
25.3045
25.3061
25.3171
25.2819
25.3204
25.3208
25.3103
25.3792
25.3088
25.3176
25.3045
25.3045
25.3088
25.2819
25.3204
25.3175
25.3208
25.2819
25.2819
25.3176
25.3103
25.3103
25.3171
25.3792
25.3204
25.3045
25.3045
25.3792
25.3792
25.3204
25.3061
25.3208
25.3792
25.3792
25.348
25.3061
25.3103
25.2827
25.3088
25.3176
25.3175
25.3103
25.3088
25.2827
25.348
25.3103
25.3103
25.3088
25.2819
25.3009
25.3045
25.3103
25.3171
25.3792
25.3204
25.3045
25.3061
25.2819
25.2819
25.3204
25.3103
25.3061
25.2827
25.3792
25.3176
25.3175
25.3103
25.3792
25.3792
25.3204
25.3208
25.3208
25.2827
25.3088
25.3176
25.3208
25.3045
25.3088
25.2819
25.3204
25.3175
25.3061
25.2827
25.2827
25.3176
25.3208
25.3103
25.3088
25.3171
25.3009
25.3103
25.3208
25.2819
25.3792
25.348
25.3061
25.3175
25.3792
25.2819
25.348
25.3061
25.3103
25.3088
25.3792
25.348
25.3061
25.3103
25.3088
25.2827
25.3009
25.3045
25.3103
25.2827
25.2827
25.3176
25.3061
25.3061
25.2819
25.2819
25.3009
25.3103
25.3175
25.2827
25.2827
25.348
25.3103
25.3208
25.2819
25.2827
25.348
25.3175
25.3103
25.3792
25.3792
25.3176
25.3061
25.3208
25.3088
25.2819
25.3009
25.3061
25.3175
25.3792
25.2827
25.348
25.3061
25.3103
25.2827
25.2819
25.3009
25.3103
25.3208
25.3171
25.3171
25.348
25.3103
25.3061
25.2819
25.3792
25.2806
25.3045
25.3103
25.3171
25.2827
25.3009
25.3175
25.3103
25.3171
25.3171
25.3204
25.3175
25.3045
25.3088
25.3088
25.3009
25.3103
25.3175
25.2827
25.3171
25.3176
25.3061
25.3045
25.3171
25.3088
25.3176
25.3061
25.3103
25.2827
25.3088
25.3176
25.3103
25.3175
25.2819
25.2819
25.348
25.3208
25.3045
25.3792
25.3088
25.3176
25.3175
25.3175
25.2819
25.2827
25.3204
25.3061
25.3061
25.3171
25.3792
25.2845
25.3208
25.3103
25.3792
25.3088
25.2806
25.3175
25.3061
25.3088
25.3088
25.348
25.3175
25.3061
25.2819
25.3792
25.3176
25.3103
25.3103
25.3171
25.2819
25.3009
25.3045
25.3175
25.2819
25.2827
25.3009
25.3208
25.3208
25.3171
25.3088
25.348
25.3175
25.3103
25.3171
25.2827
25.3176
25.3045
25.3208
25.2827
25.3792
25.3176
25.3103
25.3103
25.2819
25.2827
25.2806
25.3045
25.3061
25.3171
25.2827
25.3009
25.3208
25.3061
25.3792
25.3792
25.3009
25.3103
25.3103
25.3171
25.3088
25.3176
25.3103
25.3175
25.2827
25.3088
25.3009
25.3103
25.3208
25.3792
25.3171
25.3009
25.3103
25.3061
25.3171
25.2819
25.3176
25.3103
25.3175
25.3171
25.3171
25.348
25.3045
25.3061
25.2819
25.3088
25.3204
25.3175
25.3175
25.3792
25.2819
25.348
25.3208
25.3045
25.3171
25.3792
25.3204
25.3061
25.3103
25.2819
25.3792
25.3204
25.3103
25.3208
25.2827
25.2827
25.3204
25.3175
25.3045
25.3171
25.3171
25.3176
25.3208
25.3045
25.3088
25.2819
25.3204
25.3061
25.3175
25.2827
25.2827
25.348
25.3208
25.3103
25.3171
25.3171
25.3204
25.3175
25.3061
25.3792
25.2827
25.3009
25.3103
25.3103
25.3171
25.3088
25.3204
25.3045
25.3045
25.3088
25.2819
25.348
25.3175
25.3103
25.3088
25.3088
25.3204
25.3045
25.3045
25.3171
25.2827
25.3204
25.3175
25.3208
25.3792
25.2819
25.3176
25.3045
25.3175
25.3792
25.2827
25.3009
25.3061
25.3208
25.3088
25.2819
25.348
25.3045
25.3208
25.3088
25.3088
25.3176
25.3103
25.3061
25.2827
25.3171
25.348
25.3208
25.3045
25.3792
25.3171
25.3204
25.3061
25.3208
25.3088
25.3088
25.348
25.3208
25.3045
25.2827
25.3088
25.348
25.3208
25.3061
25.2819
25.3088
25.2806
25.3061
25.3103
25.2827
25.3088
25.348
25.3045
25.3208
25.3171
25.2827
25.3009
25.3103
25.3103
25.3792
25.3171
25.3176
25.3045
25.3061
25.2819
25.2827
25.348
25.3175
25.3103
25.2819
25.3171
25.3204
25.3045
25.3175
25.3088
25.2819
25.3009
25.3175
25.3061
25.2819
25.3088
25.2806
25.3103
25.3045
25.2827
25.3792
25.3204
25.3175
25.3061
25.2827
25.3088
25.3176
25.3045
25.3061
25.2819
25.2819
25.348
25.3103
25.3103
25.3088
25.3088
25.348
25.3045
25.3175
25.3792
25.2827
25.3204
25.3061
25.3103
25.3171
25.3088
25.3176
25.3208
25.3208
25.2819
25.2827
25.3009
25.3061
25.3175
25.3088
25.2819
25.3176
25.3103
25.3045
25.2819
25.2819
25.2845
25.3103
25.3103
25.3792
25.3088
25.3176
25.3175
25.3103
25.3792
25.3792
25.348
25.3045
25.3045
25.3792
25.3171
25.3009
25.3175
25.3045
25.2827
25.3088
25.348
25.3103
25.3045
25.3171
25.3171
25.3204
25.3061
25.3103
25.2827
25.3088
25.3176
25.3103
25.3061
25.2827
25.3792
25.3009
25.3175
25.3103
25.3088
25.3088
25.3204
25.3061
25.3061
25.2827
25.3792
25.3176
25.3208
25.3045
25.3171
25.3171
25.3009
25.3061
25.3208
25.2827
25.2827
25.348
25.3175
25.3175
25.3088
25.3171
25.3009
25.3175
25.3208
25.3088
25.3171
25.3204
25.3103
25.3208
25.2819
25.2827
25.3176
25.3175
25.3175
25.3792
25.3171
25.3009
25.3175
25.3103
25.3171
25.3171
25.3204
25.3061
25.3061
25.2819
25.2827
25.3204
25.3175
25.3045
25.2819
25.2819
25.3204
25.3061
25.3208
25.3088
25.3792
25.3204
25.3208
25.3175
25.2819
25.3088
25.3009
25.3103
25.3045
25.3171
25.3088
25.2806
25.3103
25.3175
25.2819
25.3171
25.3009
25.3103
25.3175
25.2819
25.2819
25.3204
25.3103
25.3175
25.3171
25.3792
25.348
25.3061
25.3045
25.2827
25.3792
25.3009
25.3061
25.3175
25.2827
25.3088
25.348
25.3208
25.3045
25.3088
25.3088
25.3176
25.3103
25.3061
25.2819
25.2827
25.348
25.3045
25.3061
25.3792
25.3088
25.348
25.3208
25.3208
25.2827
25.3171
25.3176
25.3103
25.3061
25.3171
25.3088
25.3204
25.3045
25.3103
25.3171
25.3088
25.348
25.3175
25.3208
25.3792
25.2827
25.2806
25.3175
25.3061
25.3088
25.3792
25.3176
25.3208
25.3045
25.2819
25.2819
25.3176
25.3103
25.3208
25.3088
25.3171
25.3009
25.3103
25.3061
25.2827
25.3088
25.348
25.3045
25.3208
25.3792
25.3792
25.3009
25.3175
25.3061
25.3171
25.2827
25.3176
25.3061
25.3208
25.3088
25.3171
25.3009
25.3175
25.3061
25.2827
25.2827
25.348
25.3045
25.3103
25.3171
25.3792
25.3204
25.3103
25.3045
25.3171
25.3088
25.3176
25.3061
25.3103
25.3088
25.3088
25.3009
25.3061
25.3061
25.2819
25.3088
25.348
25.3175
25.3103
25.2827
25.3088
25.3009
25.3045
25.3103
25.3171
25.2819
25.3204
25.3103
25.3045
25.3088
25.3171
25.348
25.3208
25.3103
25.3171
25.3088
25.3204
25.3045
25.3208
25.2819
25.2827
25.2806
25.3103
25.3045
25.2819
25.3792
25.3204
25.3175
25.3045
25.3792
25.2827
25.3009
25.3045
25.3208
25.3792
25.2827
25.2806
25.3208
25.3208
25.2827
25.3088
25.3009
25.3103
25.3061
25.3171
25.2827
25.3009
25.3175
25.3208
25.2827
25.2819
25.3204
25.3208
25.3045
25.2819
25.3792
25.348
25.3175
25.3061
25.2827
25.2827
25.3204
25.3045
25.3103
25.2827
25.3792
25.3009
25.3061
25.3208
25.3088
25.2819
25.3204
25.3045
25.3103
25.3088
25.3171
25.3009
25.3061
25.3045
25.2819
25.2819
25.3176
25.3103
25.3103
25.2827
25.2819
25.3204
25.3208
25.3045
25.2819
25.3171
25.3009
25.3103
25.3175
25.2819
25.3792
25.348
25.3175
25.3103
25.3792
25.2827
25.2806
25.3103
25.3175
25.3792
25.2819
25.3204
25.3045
25.3175
25.3088
25.2827
25.3204
25.3061
25.3045
25.3088
25.2819
25.3176
25.3208
25.3175
25.3171
25.3088
25.3009
25.3045
25.3103
25.2819
25.2819
25.3009
25.3175
25.3103
25.3088
25.3171
25.2845
25.3103
25.3208
25.3088
25.2827
25.3204
25.3208
25.3061
25.3792
25.3171
25.3176
25.3208
25.3061
25.3792
25.3792
25.3176
25.3061
25.3103
25.2827
25.3088
25.3204
25.3061
25.3045
25.3792
25.3171
25.3176
25.3175
25.3208
25.2819
25.2819
25.3204
25.3061
25.3208
25.3171
25.3171
25.3176
25.3061
25.3175
25.3792
25.2819
25.3204
25.3208
25.3208
25.2819
25.2819
25.3176
25.3061
25.3208
25.2827
25.2819
25.3009
25.3061
25.3061
25.3088
25.2819
25.3204
25.3045
25.3061
25.3171
25.3171
25.3176
25.3208
25.3208
25.3088
25.2819
25.3009
25.3103
25.3061
25.2827
25.3171
25.4302
25.3208
25.3061
25.2827
25.3088
25.3204
25.3208
25.3103
25.3171
25.2827
25.3176
25.3208
25.3208
25.2819
25.3088
25.2806
25.3045
25.3175
25.2827
25.3171
25.348
25.3061
25.3208
25.3088
25.3088
25.3204
25.3175
25.3045
25.3171
25.2827
25.3009
25.3208
25.3208
25.3792
25.3088
25.3009
25.3061
25.3103
25.3088
25.3088
25.3176
25.3208
25.3045
25.3088
25.3088
25.3204
25.3103
25.3103
25.3088
25.2827
25.3176
25.3061
25.3175
25.3171
25.2819
25.3204
25.3061
25.3045
25.3088
25.3088
25.3009
25.3045
25.3208
25.3792
25.3088
25.3176
25.3103
25.3208
25.3171
25.3088
25.348
25.3045
25.3061
25.3088
25.3792
25.348
25.3045
25.3175
25.3171
25.2827
25.348
25.3061
25.3045
25.3171
25.2827
25.3176
25.3061
25.3103
25.2827
25.3171
25.3176
25.3103
25.3175
25.2827
25.3088
25.348
25.3045
25.3208
25.3792
25.3088
25.2806
25.3045
25.3061
25.3171
25.3171
25.3204
25.3175
25.3103
25.2827
25.2819
25.348
25.3103
25.3175
25.2827
25.3792
25.3204
25.3175
25.3175
25.3088
25.2827
25.3009
25.3175
25.3061
25.2827
25.3792
25.3204
25.3061
25.3061
25.2827
25.3171
25.3176
25.3175
25.3045
25.2819
25.3792
25.3176
25.3045
25.3045
25.3792
25.3171
25.3009
25.3103
25.3208
25.3171
25.2827
25.3009
25.3103
25.3045
25.2819
25.3088
25.348
25.3103
25.3208
25.3792
25.3088
25.348
25.3208
25.3208
25.3088
25.3792
25.348
25.3208
25.3103
25.3088
25.2819
25.3176
25.3208
25.3175
25.2819
25.3088
25.3009
25.3045
25.3103
25.2819
25.3171
25.348
25.3045
25.3103
25.2827
25.3792
25.3204
25.3061
25.3061
25.2827
25.3088
25.3009
25.3208
25.3175
25.2819
25.3171
25.3176
25.3175
25.3103
25.3088
25.2819
25.348
25.3045
25.3061
25.2819
25.3088
25.348
25.3061
25.3208
25.3088
25.3792
25.3009
25.3208
25.3175
25.3088
25.3792
25.3176
25.3208
25.3175
25.3792
25.3088
25.348
25.3045
25.3208
25.3171
25.3792
25.3204
25.3045
25.3061
25.3088
25.3088
25.3204
25.3208
25.3208
25.3792
25.2827
25.3009
25.3103
25.3103
25.3792
25.3792
25.3009
25.3208
25.3175
25.3171
25.3792
25.2806
25.3208
25.3061
25.3171
25.3088
25.348
25.3045
25.3061
25.2827
25.3792
25.348
25.3045
25.3045
25.3088
25.3088
25.348
25.3045
25.3103
25.2827
25.3792
25.348
25.3103
25.3103
25.3171
25.3792
25.3009
25.3175
25.3175
25.2827
25.2819
25.3176
25.3103
25.3103
25.2827
25.2827
25.348
25.3061
25.3045
25.3792
25.3088
25.3009
25.3045
25.3103
25.3088
25.2827
25.3176
25.3208
25.3208
25.2819
25.2819
25.3204
25.3208
25.3208
25.3088
25.3792
25.348
25.3208
25.3175
25.2827
25.2819
25.3176
25.3208
25.3175
25.2827
25.3171
25.348
25.3061
25.3045
25.2827
25.3171
25.2806
25.3045
25.3175
25.3792
25.3088
25.3204
25.3045
25.3208
25.2819
25.2819
25.2806
25.3103
25.3061
25.3088
25.3792
25.3176
25.3103
25.3208
25.3088
25.3792
25.348
25.3103
25.3061
25.3088
25.2827
25.348
25.3061
25.3208
25.3792
25.3088
25.348
25.3208
25.3045
25.2827
25.3088
25.348
25.3061
25.3175
25.3088
25.2827
25.3009
25.3045
25.3045
25.3171
25.3171
25.3176
25.3045
25.3208
25.3171
25.2819
25.3176
25.3208
25.3175
25.3792
25.3792
25.348
25.3208
25.3103
25.2827
25.2819
25.3204
25.3045
25.3103
25.2819
25.3088
25.3176
25.3103
25.3175
25.3088
25.2827
25.2806
25.3103
25.3175
25.3792
25.2827
25.3009
25.3045
25.3175
25.3792
25.2819
25.3176
25.3045
25.3045
25.2819
25.3792
25.3009
25.3208
25.3208
25.2819
25.3792
25.3176
25.3061
25.3208
25.2827
25.2827
25.3176
25.3208
25.3061
25.3792
25.3088
25.348
25.3103
25.3175
25.3792
25.3792
25.3176
25.3208
25.3045
25.3088
25.2819
25.3009
25.3208
25.3208
25.3088
25.3088
25.3176
25.3103
25.3045
25.3792
25.2827
25.348
25.3103
25.3175
25.3792
25.3088
25.348
25.3045
25.3061
25.2819
25.3171
25.3204
25.3208
25.3103
25.3171
25.2827
25.3009
25.3103
25.3045
25.3171
25.2819
25.3009
25.3061
25.3045
25.3088
25.2827
25.3009
25.3103
25.3045
25.2827
25.2819
25.3009
25.3061
25.3175
25.2819
25.3792
25.348
25.3175
25.3103
25.2819
25.2819
25.3176
25.3175
25.3103
25.2827
25.2827
25.3176
25.3045
25.3175
25.3088
25.2827
25.3176
25.3103
25.3175
25.2819
25.3171
25.3204
25.3061
25.3175
25.2827
25.3171
25.3009
25.3045
25.3103
25.3088
25.3171
25.3204
25.3208
25.3175
25.3088
25.3792
25.348
25.3208
25.3045
25.3171
25.3792
25.348
25.3045
25.3103
25.2827
25.3171
25.3204
25.3175
25.3208
25.3088
25.2827
25.3176
25.3061
25.3061
25.2819
25.3171
25.3009
25.3061
25.3208
25.2819
25.3088
25.3176
25.3045
25.3061
25.2827
25.2819
25.3176
25.3045
25.3175
25.3171
25.3171
25.3204
25.3045
25.3045
25.3171
25.3792
25.3204
25.3061
25.3061
25.3792
25.3792
25.348
25.3208
25.3103
25.3171
25.2827
25.3176
25.3208
25.3103
25.2819
25.3171
25.3204
25.3175
25.3208
25.3792
25.3792
25.3009
25.3045
25.3208
25.3171
25.2819
25.3204
25.3175
25.3208
25.2819
25.3171
25.3204
25.3045
25.3103
25.2819
25.2819
25.348
25.3103
25.3061
25.2827
25.3171
25.3204
25.3045
25.3061
25.3171
25.2827
25.3176
25.3208
25.3045
25.3088
25.3088
25.348
25.3103
25.3061
25.3088
25.2827
25.3009
25.3208
25.3061
25.3088
25.2819
25.348
25.3061
25.3175
25.3088
25.2819
25.3009
25.3061
25.3175
25.3792
25.3792
25.3204
25.3045
25.3208
25.3171
25.3792
25.3176
25.3208
25.3045
25.3792
25.3792
25.3176
25.3061
25.3061
25.3088
25.2819
25.3204
25.3045
25.3175
25.2819
25.2819
25.3009
25.3061
25.3061
25.2819
25.3792
25.3009
25.3061
25.3175
25.3171
25.2827
25.2806
25.3175
25.3061
25.3088
25.3088
25.348
25.3175
25.3208
25.3792
25.3088
25.3204
25.3103
25.3103
25.3171
25.3171
25.3009
25.3208
25.3208
25.3171
25.3792
25.348
25.3208
25.3061
25.3171
25.2819
25.3009
25.3061
25.3208
25.3088
25.3088
25.3204
25.3208
25.3175
25.3088
25.2827
25.348
25.3208
25.3208
25.3088
25.3792
25.348
25.3045
25.3175
25.2819
25.3088
25.3009
25.3208
25.3208
25.3088
25.3088
25.3176
25.3103
25.3061
25.3792
25.3171
25.3204
25.3045
25.3045
25.3792
25.3171
25.2806
25.3208
25.3103
25.2819
25.3088
25.348
25.3208
25.3208
25.3171
25.2827
25.3204
25.3208
25.3103
25.2819
25.3088
25.348
25.3045
25.3103
25.3088
25.2827
25.3204
25.3103
25.3045
25.3088
25.2827
25.3176
25.3103
25.3103
25.2827
25.2827
25.3204
25.3103
25.3045
25.2827
25.3088
25.3009
25.3208
25.3061
25.3171
25.3088
25.348
25.3175
25.3061
25.2827
25.2819
25.3204
25.3103
25.3061
25.3088
25.3171
25.3009
25.3208
25.3208
25.3792
25.3792
25.3009
25.3103
25.3045
25.3088
25.3171
25.3176
25.3175
25.3061
25.2819
25.3088
25.3009
25.3045
25.3045
25.3171
25.2827
25.3204
25.3045
25.3045
25.3171
25.3171
25.348
25.3175
25.3061
25.3792
25.3088
25.3204
25.3103
25.3045
25.3171
25.3792
25.3176
25.3061
25.3175
25.3088
25.2827
25.2806
25.3045
25.3103
25.3171
25.2819
25.3009
25.3175
25.3175
25.2827
25.2827
25.3176
25.3175
25.3045
25.2819
25.2827
25.3176
25.3061
25.3208
25.2827
25.2827
25.348
25.3061
25.3175
25.3792
25.3088
25.3009
25.3175
25.3208
25.3088
25.3171
25.3204
25.3045
25.3045
25.3088
25.2819
25.2806
25.3061
25.3103
25.3088
25.2819
25.2806
25.3208
25.3103
25.2819
25.3088
25.3009
25.3103
25.3103
25.2827
25.2827
25.2806
25.3175
25.3103
25.2827
25.3088
25.3204
25.3045
25.3103
25.3792
25.3171
25.3204
25.3175
25.3103
25.2827
25.3171
25.348
25.3061
25.3208
25.3171
25.3088
25.3204
25.3061
25.3175
25.3171
25.3088
25.3009
25.3103
25.3103
25.3792
25.2827
25.3009
25.3061
25.3061
25.2819
25.3088
25.3176
25.3045
25.3208
25.3088
25.3171
25.3009
25.3175
25.3175
25.2819
25.2827
25.3009
25.3061
25.3061
25.2827
25.2819
25.348
25.3208
25.3103
25.3792
25.3792
25.3204
25.3175
25.3208
25.3088
25.3088
25.2806
25.3208
25.3175
25.2827
25.3088
25.3176
25.3045
25.3045
25.2827
25.3088
25.3176
25.3103
25.3045
25.2827
25.3171
25.3176
25.3103
25.3175
25.2819
25.2827
25.3204
25.3208
25.3061
25.3792
25.3088
25.3176
25.3208
25.3103
25.3088
25.3171
25.3204
25.3103
25.3103
25.3792
25.2827
25.3204
25.3061
25.3061
25.2819
25.3088
25.348
25.3208
25.3103
25.3088
25.3792
25.3009
25.3175
25.3045
25.3171
25.3088
25.3204
25.3103
25.3175
25.3171
25.3088
25.3204
25.3175
25.3061
25.3792
25.2819
25.3176
25.3175
25.3061
25.2819
25.2819
25.3009
25.3208
25.3103
25.3792
25.3171
25.348
25.3061
25.3208
25.3088
25.3171
25.3204
25.3208
25.3061
25.2827
25.3792
25.3176
25.3045
25.3208
25.3088
25.3171
25.348
25.3103
25.3045
25.2827
25.3792
25.3009
25.3045
25.3175
25.2827
25.2819
25.3009
25.3045
25.3103
25.3088
25.2819
25.3009
25.3175
25.3103
25.3088
25.3171
25.3176
25.3061
25.3061
25.3171
25.3088
25.3204
25.3061
25.3061
25.3171
25.3088
25.3204
25.3208
25.3103
25.3792
25.3088
25.3009
25.3175
25.3175
25.3792
25.2827
25.3009
25.3175
25.3061
25.3088
25.2819
25.348
25.3208
25.3103
25.2827
25.3171
25.3009
25.3208
25.3103
25.3792
25.3792
25.3176
25.3208
25.3103
25.3088
25.3171
25.3009
25.3103
25.3061
25.3088
25.2827
25.3204
25.3175
25.3103
25.2819
25.2827
25.3009
25.3045
25.3208
25.3792
25.3792
25.348
25.3061
25.3061
25.3088
25.2827
25.3009
25.3103
25.3061
25.3792
25.3792
25.3176
25.3175
25.3175
25.2819
25.2819
25.3204
25.3061
25.3045
25.3088
25.2819
25.348
25.3045
25.3103
25.3088
25.3088
25.3176
25.3175
25.3103
25.3171
25.3088
25.348
25.3045
25.3061
25.2827
25.2827
25.3176
25.3175
25.3208
25.2819
25.2827
25.348
25.3103
25.3061
25.3088
25.2827
25.3176
25.3175
25.3175
25.2819
25.2819
25.3009
25.3061
25.3103
25.3088
25.3088
25.2806
25.3061
25.3061
25.3171
25.3171
25.3204
25.3103
25.3045
25.3088
25.2827
25.3204
25.3208
25.3175
25.2827
25.2819
25.3204
25.3045
25.3103
25.2827
25.2827
25.348
25.3103
25.3175
25.3792
25.2827
25.348
25.3175
25.3103
25.3792
25.2827
25.348
25.3061
25.3061
25.2819
25.2827
25.3176
25.3045
25.3103
25.3792
25.3088
25.2806
25.3045
25.3208
25.3088
25.3792
25.3009
25.3045
25.3061
25.2819
25.3171
25.3009
25.3045
25.3208
25.3792
25.3171
25.3176
25.3175
25.3045
25.2819
25.3792
25.348
25.3061
25.3061
25.3171
25.3088
25.3204
25.3103
25.3103
25.3171
25.2819
25.3204
25.3061
25.3045
25.2819
25.2819
25.3176
25.3103
25.3061
25.3171
25.3792
25.3009
25.3045
25.3045
25.2827
25.3171
25.3204
25.3045
25.3103
25.3171
25.2819
25.348
25.3103
25.3103
25.3171
25.2827
25.3204
25.3103
25.3061
25.3792
25.2819
25.3176
25.3045
25.3103
25.3171
25.2827
25.3009
25.3208
25.3045
25.2819
25.3088
25.3176
25.3045
25.3045
25.3088
25.2827
25.348
25.3061
25.3045
25.2827
25.2819
25.3204
25.3045
25.3061
25.3792
25.2827
25.3204
25.3103
25.3061
25.3088
25.3792
25.3176
25.3103
25.3045
25.2827
25.3171
25.3176
25.3061
25.3103
25.3171
25.3088
25.2806
25.3045
25.3175
25.3792
25.3171
25.3204
25.3175
25.3175
25.2827
25.2827
25.3204
25.3175
25.3061
25.3171
25.3171
25.348
25.3175
25.3103
25.2819
25.3171
25.2806
25.3061
25.3045
25.3171
25.2827
25.3204
25.3103
25.3061
25.3171
25.3088
25.3009
25.3208
25.3045
25.2819
25.3088
25.3009
25.3061
25.3061
25.3792
25.2819
25.4302
25.3208
25.3103
25.3088
25.3088
25.3176
25.3061
25.3103
25.3792
25.3171
25.348
25.3103
25.3208
25.3171
25.2827
25.3009
25.3045
25.3175
25.2819
25.3088
25.348
25.3045
25.3103
25.3792
25.3171
25.348
25.3061
25.3175
25.2827
25.2827
25.348
25.3103
25.3103
25.2827
25.3088
25.348
25.3175
25.3045
25.3088
25.3792
25.3009
25.3175
25.3061
25.2827
25.3171
25.348
25.3208
25.3175
25.2819
25.3088
25.348
25.3208
25.3175
25.2827
25.2827
25.348
25.3103
25.3061
25.2819
25.3171
25.3009
25.3045
25.3208
25.3792
25.2819
25.2806
25.3045
25.3103
25.3088
25.2827
25.2806
25.3061
25.3175
25.3088
25.3088
25.3009
25.3175
25.3208
25.3792
25.3088
25.348
25.3045
25.3045
25.3088
25.3792
25.3176
25.3061
25.3045
25.2827
25.3792
25.3009
25.3061
25.3208
25.3171
25.2819
25.348
25.3045
25.3045
25.2819
25.2827
25.3176
25.3045
25.3061
25.3171
25.2827
25.3176
25.3208
25.3208
25.2827
25.3792
25.3176
25.3175
25.3208
25.3792
25.3171
25.3009
25.3175
25.3061
25.2827
25.3088
25.3009
25.3208
25.3061
25.3792
25.3171
25.3204
25.3061
25.3175
25.3171
25.3171
25.3176
25.3208
25.3061
25.3088
25.2827
25.348
25.3175
25.3045
25.3171
25.3171
25.348
25.3103
25.3045
25.3171
25.3171
25.2806
25.3175
25.3061
25.2827
25.2827
25.3204
25.3208
25.3208
25.3792
25.2827
25.3204
25.3175
25.3103
25.3792
25.2827
25.3009
25.3175
25.3045
25.2819
25.2819
25.3176
25.3103
25.3045
25.3171
25.2819
25.3176
25.3061
25.3045
25.2819
25.3792
25.348
25.3175
25.3208
25.3171
25.3792
25.3204
25.3208
25.3103
25.2827
25.3792
25.348
25.3103
25.3103
25.2819
25.2819
25.348
25.3175
25.3175
25.3792
25.2819
25.3204
25.3045
25.3061
25.2827
25.2827
25.3176
25.3175
25.3175
25.2827
25.2827
25.3009
25.3208
25.3061
25.2827
25.3088
25.3009
25.3045
25.3175
25.2819
25.3088
25.348
25.3175
25.3103
25.3088
25.3171
25.3204
25.3208
25.3208
25.3792
25.2827
25.3009
25.3061
25.3175
25.3171
25.3088
25.348
25.3103
25.3103
25.2827
25.2819
25.3176
25.3103
25.3061
25.2827
25.3088
25.3176
25.3208
25.3103
25.2827
25.3792
25.348
25.3045
25.3175
25.3171
25.3171
25.3204
25.3103
25.3208
25.3088
25.3792
25.3204
25.3045
25.3175
25.2827
25.2827
25.3204
25.3208
25.3061
25.3171
25.3088
25.3204
25.3103
25.3045
25.3088
25.3088
25.348
25.3175
25.3103
25.3792
25.2827
25.3204
25.3208
25.3061
25.3171
25.3171
25.348
25.3045
25.3175
25.2819
25.3792
25.348
25.3208
25.3175
25.3171
25.2827
25.2806
25.3103
25.3061
25.3171
25.3171
25.348
25.3103
25.3061
25.3171
25.3088
25.3204
25.3061
25.3045
25.3792
25.3792
25.3204
25.3103
25.3061
25.3171
25.3171
25.3009
25.3175
25.3175
25.2827
25.3792
25.3204
25.3208
25.3175
25.3171
25.2827
25.348
25.3061
25.3061
25.3171
25.3171
25.348
25.3208
25.3208
25.2827
25.3171
25.348
25.3103
25.3208
25.3088
25.3088
25.348
25.3175
25.3103
25.3792
25.3088
25.3176
25.3061
25.3103
25.3088
25.2827
25.3009
25.3061
25.3208
25.3792
25.3171
25.3204
25.3103
25.3061
25.3171
25.3792
25.348
25.3061
25.3045
25.2827
25.2827
25.3204
25.3103
25.3045
25.3792
25.2827
25.348
25.3103
25.3175
25.3171
25.3792
25.348
25.3061
25.3045
25.3792
25.3088
25.3204
25.3175
25.3045
25.3088
25.3171
25.348
25.3208
25.3175
25.3088
25.2819
25.3204
25.3208
25.3061
25.3171
25.3171
25.3204
25.3061
25.3175
25.3792
25.3171
25.3009
25.3208
25.3175
25.2827
25.3088
25.3009
25.3175
25.3208
25.2827
25.2819
25.3009
25.3208
25.3175
25.2827
25.3088
25.3204
25.3103
25.3103
25.3088
25.3088
25.3176
25.3061
25.3103
25.3792
25.2819
25.3176
25.3103
25.3061
25.3171
25.3792
25.3204
25.3103
25.3208
25.2827
25.3792
25.348
25.3103
25.3208
25.3792
25.3792
25.348
25.3103
25.3208
25.2819
25.3171
25.3204
25.3045
25.3175
25.3088
25.3792
25.2806
25.3061
25.3103
25.3088
25.2827
25.348
25.3045
25.3175
25.3171
25.3088
25.3176
25.3208
25.3045
25.3171
25.2827
25.3009
25.3061
25.3208
25.3792
25.3171
25.3176
25.3045
25.3175
25.2827
25.2827
25.3204
25.3175
25.3061
25.2819
25.3088
25.3009
25.3175
25.3103
25.2827
25.2819
25.3009
25.3061
25.3208
25.2827
25.2827
25.3204
25.3208
25.3175
25.3088
25.3088
25.3176
25.3061
25.3175
25.3792
25.3792
25.348
25.3175
25.3175
25.3171
25.2827
25.348
25.3103
25.3175
25.2827
25.3088
25.3176
25.3103
25.3175
25.3088
25.2819
25.3176
25.3175
25.3045
25.3792
25.3792
25.3204
25.3061
25.3208
25.3088
25.2827
25.3176
25.3061
25.3103
25.3171
25.3088
25.348
25.3061
25.3208
25.3171
25.3171
25.3204
25.3045
25.3208
25.2827
25.2827
25.3176
25.3175
25.3208
25.3171
25.3088
25.3176
25.3061
25.3208
25.3792
25.2827
25.348
25.3061
25.3103
25.3792
25.3171
25.3204
25.3208
25.3208
25.2827
25.2827
25.3176
25.3175
25.3175
25.2819
25.3792
25.3176
25.3103
25.3208
25.2819
25.2827
25.3176
25.3208
25.3061
25.3088
25.2819
25.3176
25.3208
25.3061
25.2819
25.3792
25.3176
25.3175
25.3103
25.3792
25.2827
25.348
25.3175
25.3175
25.3171
25.3792
25.3176
25.3208
25.3175
25.2819
25.3171
25.3009
25.3103
25.3208
25.3171
25.3088
25.3176
25.3103
25.3208
25.2827
25.3088
25.3204
25.3103
25.3061
25.3088
25.2827
25.348
25.3103
25.3045
25.3171
25.3171
25.3204
25.3061
25.3103
25.3171
25.3088
25.3204
25.3103
25.3045
25.2819
25.3792
25.3204
25.3103
25.3208
25.2819
25.3792
25.3009
25.3103
25.3208
25.3088
25.3088
25.3204
25.3061
25.3061
25.2827
25.3792
25.2806
25.3175
25.3103
25.2819
25.2819
25.3204
25.3061
25.3175
25.3171
25.3088
25.3176
25.3045
25.3175
25.2827
25.2827
25.3204
25.3045
25.3103
25.3792
25.3088
25.3176
25.3208
25.3103
25.2827
25.2827
25.3009
25.3175
25.3208
25.3792
25.3088
25.348
25.3045
25.3061
25.3792
25.3088
25.3204
25.3045
25.3103
25.3088
25.2827
25.3009
25.3103
25.3103
25.3171
25.2819
25.3009
25.3208
25.3103
25.3171
25.2827
25.3009
25.3061
25.3208
25.3088
25.3171
25.3204
25.3061
25.3208
25.3171
25.3792
25.348
25.3103
25.3061
25.2819
25.3792
25.3009
25.3045
25.3175
25.3171
25.3792
25.3204
25.3045
25.3061
25.3792
25.2827
25.3204
25.3175
25.3045
25.3171
25.3171
25.348
25.3103
25.3061
25.3088
25.3792
25.3204
25.3045
25.3175
25.3171
25.3088
25.3009
25.3045
25.3103
25.2819
25.2819
25.3009
25.3175
25.3208
25.2819
25.3088
25.4302
25.3045
25.3061
25.3171
25.2827
25.348
25.3208
25.3045
25.2819
25.3792
25.3176
25.3208
25.3061
25.2827
25.2827
25.3176
25.3061
25.3061
25.3171
25.2819
25.348
25.3175
25.3061
25.2827
25.2819
25.3176
25.3045
25.3061
25.3171
25.2819
25.3176
25.3175
25.3045
25.3171
25.2819
25.348
25.3103
25.3045
25.3088
25.3088
25.3176
25.3208
25.3103
25.3171
25.3792
25.3009
25.3175
25.3103
25.3171
25.3088
25.3204
25.3103
25.3175
25.2827
25.3171
25.3204
25.3103
25.3103
25.3792
25.3792
25.3204
25.3175
25.3103
25.3171
25.3171
25.348
25.3045
25.3061
25.3792
25.2819
25.3204
25.3045
25.3175
25.3792
25.2819
25.348
25.3175
25.3208
25.3171
25.3792
25.3009
25.3208
25.3103
25.2827
25.3171
25.3009
25.3175
25.3208
25.3171
25.2827
25.348
25.3103
25.3061
25.2819
25.2827
25.348
25.3208
25.3175
25.3171
25.2827
25.3176
25.3061
25.3061
25.3171
25.2827
25.2806
25.3175
25.3175
25.3792
25.3171
25.3176
25.3208
25.3045
25.2827
25.3088
25.2845
25.3045
25.3061
25.3171
25.3792
25.3204
25.3208
25.3208
25.2827
25.2827
25.2806
25.3208
25.3175
25.3171
25.3088
25.3176
25.3061
25.3103
25.2819
25.3792
25.3204
25.3045
25.3175
25.3088
25.2827
25.348
25.3103
25.3045
25.3088
25.3088
25.348
25.3175
25.3175
25.3088
25.2827
25.3204
25.3103
25.3208
25.2827
25.3171
25.3009
25.3061
25.3175
25.3088
25.2827
25.348
25.3175
25.3175
25.2819
25.2827
25.2806
25.3175
25.3208
25.3171
25.3171
25.348
25.3175
25.3175
25.3088
25.3088
25.3009
25.3061
25.3061
25.2819
25.3088
25.3009
25.3061
25.3045
25.2827
25.3171
25.3204
25.3045
25.3103
25.3171
25.3088
25.3204
25.3208
25.3045
25.3171
25.3171
25.348
25.3175
25.3061
25.3171
25.3088
25.3176
25.3103
25.3045
25.3088
25.3088
25.3176
25.3061
25.3208
25.3171
25.2819
25.3204
25.3061
25.3045
25.2827
25.3088
25.3204
25.3208
25.3208
25.2827
25.2819
25.3204
25.3175
25.3045
25.3792
25.3792
25.3009
25.3061
25.3208
25.3088
25.2819
25.348
25.3208
25.3061
25.3792
25.2827
25.348
25.3045
25.3061
25.3171
25.2819
25.3176
25.3208
25.3045
25.3171
25.2827
25.3176
25.3103
25.3103
25.3088
25.2819
25.3204
25.3045
25.3045
25.3171
25.2827
25.348
25.3208
25.3061
25.3088
25.3792
25.3009
25.3061
25.3103
25.2819
25.2827
25.348
25.3103
25.3208
25.2827
25.3088
25.3009
25.3103
25.3045
25.3171
25.2819
25.348
25.3103
25.3103
25.3171
25.3088
25.3009
25.3175
25.3061
25.3171
25.2827
25.3176
25.3208
25.3103
25.3088
25.3171
25.3204
25.3061
25.3061
25.3171
25.3792
25.3009
25.3208
25.3045
25.2819
25.2827
25.3009
25.3045
25.3045
25.2819
25.3088
25.3204
25.3045
25.3045
25.3792
25.3792
25.3176
25.3175
25.3208
25.3792
25.3171
25.3204
25.3208
25.3103
25.3171
25.3792
25.3009
25.3175
25.3045
25.3792
25.3171
25.3176
25.3061
25.3061
25.2827
25.2819
25.3176
25.3208
25.3208
25.2819
25.2819
25.3204
25.3208
25.3103
25.3088
25.3171
25.3176
25.3208
25.3045
25.2819
25.2819
25.3204
25.3103
25.3045
25.3171
25.3171
25.3204
25.3175
25.3208
25.2827
25.2819
25.3009
25.3045
25.3061
25.2819
25.3171
25.348
25.3175
25.3061
25.3088
25.2819
25.3009
25.3103
25.3208
25.3792
25.3171
25.3176
25.3208
25.3208
25.3088
25.3792
25.348
25.3175
25.3045
25.3792
25.3171
25.3204
25.3208
25.3208
25.3171
25.3088
25.3204
25.3208
25.3103
25.3088
25.3088
25.3176
25.3103
25.3175
25.3088
25.3171
25.3176
25.3208
25.3061
25.3171
25.3171
25.348
25.3045
25.3103
25.2827
25.3171
25.3204
25.3061
25.3045
25.3088
25.2827
25.2806
25.3045
25.3208
25.2827
25.2819
25.3204
25.3208
25.3175
25.3792
25.3088
25.3176
25.3045
25.3061
25.2827
25.2819
25.3176
25.3061
25.3061
25.3088
25.2819
25.3204
25.3103
25.3103
25.2827
25.2819
25.2806
25.3061
25.3208
25.3088
25.2819
25.2845
25.3045
25.3061
25.2827
25.2827
25.348
25.3175
25.3045
25.2819
25.3088
25.348
25.3061
25.3175
25.3171
25.3088
25.348
25.3175
25.3045
25.3088
25.3171
25.3204
25.3045
25.3061
25.3171
25.2819
25.3204
25.3045
25.3208
25.3088
25.3171
25.3204
25.3103
25.3103
25.3088
25.3088
25.3176
25.3045
25.3103
25.3171
25.3171
25.3009
25.3175
25.3061
25.3171
25.3171
25.3204
25.3061
25.3175
25.3171
25.2819
25.3176
25.3103
25.3061
25.3171
25.3088
25.3176
25.3208
25.3061
25.3171
25.3171
25.3009
25.3045
25.3208
25.3792
25.3792
25.3204
25.3175
25.3208
25.3171
25.3792
25.3204
25.3208
25.3175
25.3171
25.3088
25.3009
25.3045
25.3175
25.2819
25.2827
25.3204
25.3208
25.3208
25.2827
25.2819
25.3176
25.3103
25.3103
25.3171
25.2827
25.3009
25.3045
25.3103
25.3792
25.2819
25.348
25.3208
25.3045
25.3792
25.3171
25.3009
25.3045
25.3103
25.3171
25.3171
25.3204
25.3103
25.3208
25.3171
25.3792
25.3009
25.3208
25.3103
25.2819
25.3171
25.3009
25.3061
25.3045
25.2827
25.2827
25.3204
25.3045
25.3208
25.3088
25.2819
25.2806
25.3208
25.3208
25.3088
25.2819
25.348
25.3061
25.3045
25.2827
25.3088
25.2806
25.3103
25.3045
25.3088
25.3792
25.3176
25.3175
25.3061
25.2819
25.2827
25.348
25.3208
25.3103
25.3088
25.3088
25.348
25.3061
25.3208
25.3088
25.3792
25.3176
25.3175
25.3061
25.2819
25.3171
25.3176
25.3103
25.3208
25.2827
25.2827
25.348
25.3061
25.3175
25.3088
25.3088
25.2806
25.3208
25.3061
25.2819
25.2827
25.2806
25.3175
25.3208
25.3088
25.2827
25.3204
25.3175
25.3103
25.2827
25.2819
25.3204
25.3045
25.3103
25.2827
25.2819
25.3204
25.3045
25.3061
25.3171
25.2819
25.3176
25.3103
25.3175
25.2819
25.3792
25.348
25.3103
25.3045
25.3171
25.2827
25.3176
25.3175
25.3061
25.3088
25.3171
25.348
25.3103
25.3061
25.2819
25.3792
25.3176
25.3103
25.3061
25.2819
25.3792
25.3176
25.3103
25.3061
25.3792
25.3088
25.3009
25.3045
25.3103
25.2819
25.3088
25.3204
25.3103
25.3045
25.2827
25.3171
25.348
25.3045
25.3103
25.2819
25.3792
25.3009
25.3045
25.3175
25.3088
25.3792
25.3009
25.3103
25.3175
25.3171
25.3171
25.3204
25.3103
25.3208
25.2827
25.2827
25.2806
25.3208
25.3103
25.3171
25.3792
25.2806
25.3061
25.3061
25.2819
25.2827
25.348
25.3208
25.3061
25.2819
25.3171
25.3204
25.3045
25.3175
25.3792
25.2819
25.3176
25.3061
25.3045
25.3792
25.3171
25.3009
25.3175
25.3045
25.3171
25.3171
25.348
25.3103
25.3175
25.3088
25.3088
25.3009
25.3061
25.3175
25.2819
25.2819
25.348
25.3045
25.3103
25.3792
25.3792
25.3176
25.3175
25.3045
25.2819
25.3792
25.3009
25.3103
25.3103
25.3171
25.2819
25.348
25.3208
25.3103
25.2827
25.3792
25.2806
25.3208
25.3208
25.2819
25.3792
25.3176
25.3175
25.3208
25.2827
25.2819
25.3204
25.3045
25.3208
25.2827
25.2819
25.3009
25.3175
25.3061
25.3171
25.3088
25.3009
25.3208
25.3045
25.3171
25.2819
25.348
25.3045
25.3208
25.3088
25.2827
25.3176
25.3061
25.3208
25.3171
25.2819
25.348
25.3061
25.3103
25.2819
25.3088
25.3176
25.3045
25.3208
25.3792
25.2827
25.3176
25.3045
25.3103
25.3088
25.3088
25.348
25.3061
25.3208
25.3792
25.3088
25.348
25.3208
25.3045
25.2827
25.3088
25.3176
25.3103
25.3175
25.3171
25.3088
25.3176
25.3175
25.3045
25.3088
25.2827
25.348
25.3175
25.3045
25.3088
25.3171
25.3009
25.3045
25.3045
25.3171
25.3171
25.3204
25.3045
25.3175
25.3171
25.2827
25.3204
25.3175
25.3103
25.3792
25.3792
25.348
25.3061
25.3208
25.3171
25.2827
25.348
25.3175
25.3061
25.2827
25.3171
25.348
25.3208
25.3103
25.2827
25.3088
25.348
25.3045
25.3045
25.3171
25.2819
25.3176
25.3208
25.3208
25.3171
25.2827
25.348
25.3208
25.3175
25.3088
25.2819
25.3204
25.3061
25.3061
25.2827
25.3171
25.3204
25.3175
25.3208
25.3792
25.2827
25.348
25.3175
25.3061
25.2827
25.2827
25.3176
25.3175
25.3103
25.3171
25.3171
25.3176
25.3208
25.3045
25.3171
25.3171
25.3009
25.3061
25.3175
25.3792
25.2827
25.2806
25.3208
25.3061
25.3088
25.2827
25.348
25.3103
25.3061
25.2819
25.3088
25.3176
25.3175
25.3208
25.3171
25.3792
25.3176
25.3103
25.3103
25.3171
25.2827
25.3176
25.3061
25.3045
25.3792
25.2819
25.348
25.3045
25.3175
25.3088
25.3171
25.3204
25.3061
25.3103
25.3792
25.2819
25.3176
25.3045
25.3045
25.3088
25.3088
25.3176
25.3061
25.3061
25.3792
25.3792
25.3204
25.3045
25.3103
25.3171
25.3792
25.3176
25.3045
25.3061
25.2819
25.3088
25.348
25.3045
25.3208
25.2827
25.2819
25.3009
25.3045
25.3175
25.3171
25.3792
25.2806
25.3175
25.3103
25.3171
25.3792
25.3176
25.3103
25.3061
25.2819
25.2819
25.348
25.3045
25.3045
25.3171
25.2819
25.3009
25.3208
25.3045
25.2827
25.3171
25.3009
25.3175
25.3175
25.3171
25.2827
25.3009
25.3175
25.3103
25.3088
25.2819
25.3204
25.3103
25.3061
25.3088
25.2819
25.3009
25.3045
25.3103
25.2827
25.3088
25.3009
25.3061
25.3175
25.3171
25.3792
25.3204
25.3208
25.3045
25.2819
25.3171
25.3204
25.3103
25.3061
25.3171
25.3088
25.3009
25.3045
25.3103
25.2827
25.3088
25.2806
25.3208
25.3103
25.2819
25.2819
25.2806
25.3208
25.3061
25.3792
25.2827
25.348
25.3061
25.3175
25.3088
25.2819
25.3204
25.3175
25.3045
25.3088
25.2819
25.348
25.3061
25.3208
25.3792
25.3792
25.3204
25.3061
25.3175
25.2827
25.3088
25.3176
25.3061
25.3103
25.2819
25.3792
25.3176
25.3175
25.3103
25.3171
25.3792
25.3009
25.3045
25.3061
25.2819
25.3792
25.3176
25.3175
25.3045
25.2819
25.3792
25.3009
25.3208
25.3208
25.3171
25.2819
25.3204
25.3103
25.3045
25.3171
25.3171
25.3176
25.3208
25.3175
25.2819
25.2819
25.3009
25.3061
25.3045
25.3088
25.2827
25.3176
25.3103
25.3061
25.3792
25.2827
25.3009
25.3045
25.3175
25.3171
25.2827
25.3176
25.3208
25.3208
25.3792
25.3792
25.348
25.3103
25.3045
25.2827
25.2827
25.3204
25.3045
25.3061
25.3171
25.3088
25.348
25.3175
25.3061
25.3088
25.2827
25.3176
25.3175
25.3208
25.3171
25.3088
25.2806
25.3061
25.3061
25.3088
25.3792
25.348
25.3208
25.3175
25.2819
25.3088
25.3009
25.3208
25.3061
25.3792
25.2819
25.3204
25.3045
25.3061
25.2819
25.3088
25.2806
25.3045
25.3061
25.3171
25.2827
25.3176
25.3061
25.3175
25.3171
25.3792
25.3176
25.3045
25.3045
25.3792
25.3792
25.3176
25.3208
25.3061
25.3171
25.2819
25.348
25.3045
25.3103
25.3088
25.3171
25.3204
25.3208
25.3045
25.2827
25.2827
25.348
25.3175
25.3061
25.3171
25.3171
25.3176
25.3045
25.3061
25.3171
25.2819
25.3009
25.3208
25.3208
25.3088
25.3088
25.3176
25.3103
25.3045
25.2827
25.2827
25.3204
25.3103
25.3103
25.3792
25.2819
25.3204
25.3175
25.3103
25.3088
25.2819
25.3204
25.3103
25.3208
25.3792
25.2827
25.3204
25.3061
25.3045
25.2827
25.3088
25.3009
25.3103
25.3175
25.3088
25.3792
25.348
25.3208
25.3103
25.3792
25.3088
25.3176
25.3061
25.3045
25.2827
25.3171
25.348
25.3103
25.3045
25.2819
25.3171
25.2806
25.3175
25.3061
25.3792
25.2827
25.3204
25.3061
25.3208
25.2819
25.3171
25.3176
25.3103
25.3175
25.2819
25.3171
25.3204
25.3175
25.3061
25.3171
25.3171
''')
array_string_1 = data_sting_1.split('\n')
array_cords_1 = data_cords_1.split('\n')
origin_lat_2 = []
origin_lat1 = []
for i in range(len(array_cords_1)-1):
  if(array_cords_1[i] != ''):
     origin_lat_2.append(float(array_cords_1[i]))
array_string_1.pop(0)
for i in range(len(array_string_1)-1):
  if(array_string_1[i] == 'Work'):
    origin_lat1.append(origin_lat_2[i])

In [ ]:
# @title
data_cords_2 =('''
82.9853
82.9974
83.0019
83.0396
83.0063
82.9853
82.9974
82.9974
83.0045
83.0094
82.9848
83.0091
83.0107
83.0396
83.0045
82.9871
83.0004
82.9974
83.0094
83.0094
82.9853
82.9974
83.0004
83.0045
83.0094
82.9853
83.0004
83.0091
83.0063
83.0063
82.9853
83.0004
83.0004
83.0396
83.0396
83.0366
82.9974
83.0107
83.0045
83.0396
82.9848
83.0019
83.0004
83.0218
83.0094
82.9848
83.0019
83.0004
83.0094
83.0218
82.9871
82.9974
83.0019
83.0218
83.0218
82.9853
83.0019
83.0019
83.0045
83.0045
82.9848
83.0004
83.0091
83.0094
83.0094
82.9871
83.0019
83.0091
83.0063
83.0094
82.9997
83.0091
83.0004
83.0094
83.0094
82.9848
83.0019
83.0107
83.0218
83.0063
82.9848
83.0091
83.0019
83.0218
83.0063
82.9997
83.0019
83.0019
83.0218
83.0218
82.9853
83.0019
83.0091
83.0045
83.0094
82.9853
83.0019
83.0107
83.0218
83.0396
82.9848
82.9974
82.9974
83.0396
83.0396
82.9853
83.0091
83.0107
83.0045
83.0218
82.9997
83.0004
83.0107
83.0218
83.0045
82.9853
83.0019
83.0107
83.0218
83.0396
82.9871
83.0091
82.9974
83.0094
83.0218
82.9997
83.0019
82.9974
83.0094
83.0045
82.9848
83.0107
83.0004
83.0218
83.0218
82.9848
83.0019
83.0004
83.0094
83.0045
82.9848
83.0091
83.0107
83.0094
83.0218
82.9853
82.9974
83.0004
83.0063
83.0218
82.9853
83.0107
83.0107
83.0045
83.0396
82.9997
83.0019
83.0107
83.0094
83.0063
82.9871
83.0004
82.9974
83.0094
83.0094
82.9848
83.0107
83.0091
83.0218
83.0396
82.9848
83.0091
83.0004
83.0094
83.0218
82.9848
83.0107
83.0019
83.0396
83.0094
82.9848
83.0107
83.0107
83.0218
83.0218
82.9848
83.0107
82.9974
83.0063
83.0218
82.9853
83.0091
83.0004
83.0045
83.0396
82.9848
83.0004
83.0019
83.0045
83.0063
82.9997
83.0091
83.0107
83.0218
83.0063
82.9853
83.0004
83.0004
83.0045
83.0063
82.9997
83.0091
83.0107
83.0396
83.0396
82.9853
83.0107
83.0107
83.0045
83.0045
82.9853
83.0004
82.9974
83.0396
83.0063
82.9848
83.0107
83.0091
83.0045
83.0063
82.9853
83.0091
83.0107
83.0396
83.0045
82.9853
83.0004
83.0107
83.0063
83.0396
82.9997
83.0091
83.0019
83.0063
83.0045
82.9853
83.0091
83.0004
83.0094
83.0396
82.9871
83.0004
82.9974
83.0218
83.0063
82.9997
83.0107
83.0107
83.0396
83.0045
83.0366
82.9974
83.0107
83.0094
83.0094
82.9853
83.0091
83.0107
83.0218
83.0094
82.9871
83.0091
83.0004
83.0045
83.0094
82.9853
83.0019
83.0004
83.0094
83.0218
82.9853
82.9974
83.0019
83.0063
83.0045
82.9871
83.0107
83.0004
83.0218
83.0094
82.9997
83.0107
83.0004
83.0094
83.0063
82.9848
83.0004
83.0107
83.0396
83.0218
83.0366
83.0019
83.0019
83.0063
83.0063
82.9853
83.0107
83.0004
83.0063
83.0218
82.9997
83.0004
83.0091
83.0094
83.0063
82.9997
83.0107
82.9974
83.0396
83.0045
82.9848
83.0107
83.0019
83.0094
83.0396
82.9997
82.9974
83.0091
83.0063
83.0063
82.9997
83.0019
83.0091
83.0063
83.0045
82.9997
82.9974
83.0091
83.0396
83.0396
82.9853
83.0019
83.0004
83.0396
83.0063
82.9871
82.9974
83.0004
83.0396
83.0045
82.9853
83.0107
83.0107
83.0218
83.0045
82.9853
83.0019
83.0107
83.0094
83.0045
82.9871
83.0107
83.0004
83.0094
83.0094
82.9853
83.0004
83.0019
83.0396
83.0218
82.9853
83.0107
83.0107
83.0063
83.0063
82.9853
82.9974
82.9974
83.0094
83.0218
82.9848
83.0091
83.0004
83.0094
83.0218
82.9848
82.9974
83.0091
83.0094
83.0094
82.9848
83.0019
83.0091
83.0218
83.0063
83.0366
83.0019
83.0107
83.0094
83.0063
82.9848
83.0107
83.0004
83.0094
83.0045
82.9853
83.0019
83.0004
83.0045
83.0063
82.9871
82.9974
83.0107
83.0094
83.0218
82.9848
83.0091
83.0004
83.0094
83.0218
82.9997
83.0107
83.0004
83.0396
83.0396
82.9871
83.0107
83.0091
83.0045
83.0094
82.9853
83.0004
83.0004
83.0218
83.0094
82.9853
83.0091
82.9974
83.0094
83.0063
82.9871
83.0091
83.0004
83.0063
83.0045
83.0366
82.9974
83.0107
83.0094
83.0094
82.9871
83.0091
83.0019
83.0218
83.0094
82.9871
83.0091
83.0004
83.0396
83.0218
82.9848
83.0091
83.0107
83.0218
83.0396
82.9997
83.0091
83.0004
83.0045
83.0218
83.0366
83.0004
83.0091
83.0063
83.0045
82.9848
83.0004
82.9974
83.0045
83.0094
82.9853
83.0091
82.9974
83.0063
83.0045
82.9853
82.9974
83.0004
83.0396
83.0045
82.9997
83.0091
83.0107
83.0063
83.0396
82.9848
83.0107
83.0004
83.0063
83.0396
82.9871
83.0091
83.0091
83.0218
83.0045
82.9853
83.0019
83.0019
83.0045
83.0218
82.9853
82.9974
83.0004
83.0094
83.0396
82.9848
82.9974
83.0091
83.0094
83.0396
82.9848
82.9974
83.0107
83.0063
83.0396
82.9871
82.9974
82.9974
83.0063
83.0063
82.9997
83.0091
83.0004
83.0063
83.0396
82.9871
83.0091
83.0004
83.0094
83.0094
82.9997
82.9974
83.0004
83.0045
83.0063
82.9997
82.9974
83.0091
83.0063
83.0063
83.0366
82.9974
83.0019
83.0063
83.0396
82.9871
82.9974
82.9974
83.0094
83.0063
82.9871
83.0091
83.0091
83.0063
83.0045
82.9848
83.0004
83.0004
83.0396
83.0218
82.9871
83.0107
82.9974
83.0218
83.0396
82.9997
83.0091
82.9974
83.0218
83.0094
82.9848
83.0019
83.0004
83.0218
83.0063
83.0366
82.9974
83.0107
83.0063
83.0396
82.9848
83.0107
83.0019
83.0396
83.0396
82.9871
83.0004
83.0107
83.0094
83.0045
82.9997
83.0019
82.9974
83.0063
83.0218
82.9848
83.0107
83.0091
83.0045
83.0218
82.9853
83.0004
82.9974
83.0045
83.0094
82.9871
83.0107
83.0019
83.0045
83.0063
82.9997
83.0091
83.0107
83.0396
83.0218
82.9871
83.0004
83.0019
83.0063
83.0218
82.9997
83.0091
83.0107
83.0063
83.0063
82.9997
83.0019
82.9974
83.0218
83.0045
82.9853
82.9974
82.9974
83.0396
83.0396
82.9853
82.9974
82.9974
83.0094
83.0218
82.9871
83.0107
83.0091
83.0063
83.0396
82.9997
83.0091
83.0019
83.0045
83.0063
82.9853
83.0091
82.9974
83.0396
83.0396
82.9848
83.0004
83.0004
83.0218
83.0045
82.9871
83.0019
83.0019
83.0218
83.0396
82.9997
82.9974
83.0004
83.0218
83.0396
82.9853
83.0091
82.9974
83.0396
83.0396
82.9853
82.9974
83.0091
83.0218
83.0045
82.9853
83.0091
83.0107
83.0396
83.0045
82.9848
83.0019
82.9974
83.0396
83.0045
82.9853
83.0004
83.0107
83.0063
83.0063
82.9853
83.0019
83.0004
83.0063
83.0218
83.0366
83.0091
83.0004
83.0045
83.0094
82.9997
83.0107
83.0004
83.0396
83.0094
82.9848
83.0107
83.0019
83.0045
83.0218
82.9871
83.0004
83.0019
83.0094
83.0063
82.9853
83.0091
82.9974
83.0063
83.0094
82.9871
83.0107
83.0091
83.0045
83.0218
82.9848
82.9974
82.9974
83.0396
83.0218
82.9871
82.9974
83.0107
83.0045
83.0063
82.9997
83.0004
83.0091
83.0063
83.0396
82.9848
83.0091
83.0004
83.0063
83.0063
82.9997
83.0019
83.0019
83.0094
83.0063
82.9871
83.0107
83.0004
83.0218
83.0218
82.9871
83.0019
83.0004
83.0094
83.0396
83.0366
83.0107
82.9974
83.0396
83.0063
82.9871
83.0091
83.0107
83.0045
83.0218
82.9853
83.0004
83.0107
83.0063
83.0218
82.9848
83.0091
83.0019
83.0396
83.0218
82.9871
83.0004
82.9974
83.0045
83.0396
82.9853
83.0107
82.9974
83.0063
83.0396
82.9853
83.0019
83.0019
83.0045
83.0094
82.9853
82.9974
83.0107
83.0045
83.0063
82.9997
82.9974
83.0107
83.0063
83.0045
82.9853
83.0004
83.0019
83.0094
83.0094
82.9853
83.0004
82.9974
83.0094
83.0218
82.9997
83.0107
83.0019
83.0396
83.0063
82.9871
83.0091
83.0019
83.0045
83.0045
82.9848
83.0107
83.0107
83.0063
83.0396
82.9871
83.0004
83.0019
83.0396
83.0396
82.9997
82.9974
83.0107
83.0063
83.0045
82.9871
83.0107
82.9974
83.0094
83.0045
82.9997
82.9974
83.0091
83.0094
83.0218
82.9853
83.0091
83.0019
83.0094
83.0094
82.9848
83.0004
82.9974
83.0218
83.0045
82.9853
82.9974
82.9974
83.0063
83.0396
82.9853
83.0091
83.0091
83.0396
83.0045
82.9853
83.0107
83.0019
83.0045
83.0063
82.9871
83.0107
82.9974
83.0063
83.0063
83.0366
83.0004
82.9974
83.0094
83.0396
82.9871
83.0004
83.0107
83.0063
83.0045
83.0366
83.0004
83.0019
83.0045
83.0063
82.9848
83.0091
83.0091
83.0218
83.0396
82.9997
82.9974
83.0004
83.0218
83.0063
82.9997
83.0091
82.9974
83.0218
83.0396
82.9853
83.0004
82.9974
83.0218
83.0094
82.9871
82.9974
83.0107
83.0218
83.0396
82.9871
83.0004
83.0091
83.0063
83.0094
82.9997
83.0019
83.0019
83.0045
83.0094
82.9853
83.0004
82.9974
83.0396
83.0094
82.9997
83.0004
83.0091
83.0396
83.0396
82.9848
83.0091
83.0091
83.0094
83.0045
82.9853
82.9974
83.0004
83.0045
83.0396
82.9853
83.0091
83.0107
83.0218
83.0063
82.9871
83.0004
83.0019
83.0218
83.0218
82.9871
83.0091
83.0091
83.0396
83.0218
82.9853
83.0004
83.0004
83.0396
83.0094
82.9997
82.9974
83.0019
83.0218
83.0094
83.0366
83.0091
83.0107
83.0063
83.0094
82.9997
83.0004
83.0004
83.0396
83.0063
82.9871
83.0019
83.0107
83.0218
83.0045
82.9871
83.0004
83.0091
83.0063
83.0045
82.9853
83.0004
83.0091
83.0396
83.0063
82.9853
82.9974
83.0091
83.0063
83.0396
82.9853
83.0091
83.0107
83.0063
83.0094
82.9871
83.0091
83.0004
83.0063
83.0396
82.9853
83.0091
82.9974
83.0218
83.0396
82.9871
83.0091
83.0019
83.0045
83.0218
82.9871
83.0107
82.9974
83.0218
83.0063
82.9871
83.0091
83.0107
83.0045
83.0063
82.9853
83.0107
82.9974
83.0045
83.0094
82.9997
83.0019
83.0004
83.0094
83.0396
82.9871
83.0107
83.0019
83.0045
83.0094
82.9871
83.0004
83.0004
83.0396
83.0094
82.9997
83.0019
83.0107
83.0094
83.0063
82.9871
83.0091
83.0019
83.0063
83.0063
82.9871
83.0091
83.0019
83.0396
83.0094
82.9853
83.0107
82.9974
83.0218
83.0396
82.9871
83.0091
83.0091
83.0045
83.0045
82.9997
82.9974
83.0019
83.0063
83.0063
82.9848
83.0091
83.0004
83.0063
83.0218
82.9848
83.0091
83.0019
83.0063
83.0094
82.9848
83.0019
83.0107
83.0045
83.0094
82.9997
83.0019
83.0107
83.0045
83.0218
82.9871
83.0091
83.0107
83.0063
83.0094
82.9871
82.9974
83.0107
83.0396
83.0218
82.9853
83.0091
82.9974
83.0063
83.0396
82.9848
83.0019
83.0019
83.0045
83.0045
82.9853
83.0107
82.9974
83.0218
83.0045
82.9612
83.0019
83.0091
83.0063
83.0094
82.9848
82.9974
83.0004
83.0045
83.0396
82.9997
83.0091
83.0107
83.0045
83.0218
82.9853
83.0019
83.0107
83.0218
83.0045
82.9997
83.0019
83.0107
83.0094
83.0094
82.9871
83.0091
83.0107
83.0063
83.0063
82.9853
83.0091
83.0004
83.0045
83.0396
82.9853
83.0019
83.0019
83.0094
83.0045
82.9871
83.0091
83.0004
83.0094
83.0094
82.9848
82.9974
83.0019
83.0063
83.0063
82.9853
83.0004
83.0019
83.0063
83.0045
82.9997
83.0004
83.0091
83.0396
83.0063
82.9853
82.9974
83.0004
83.0094
83.0045
82.9871
83.0019
82.9974
83.0045
83.0218
82.9871
83.0091
83.0091
83.0045
83.0094
82.9997
83.0107
83.0107
83.0094
83.0063
82.9848
83.0107
83.0019
83.0094
83.0063
82.9853
83.0019
83.0107
83.0094
83.0063
82.9997
83.0091
83.0091
83.0094
83.0094
83.0366
83.0107
83.0019
83.0063
83.0218
82.9848
83.0004
82.9974
83.0218
83.0045
82.9871
82.9974
83.0107
83.0063
83.0045
82.9871
82.9974
83.0091
83.0396
83.0063
83.0366
83.0004
83.0107
83.0094
83.0045
83.0366
83.0004
83.0019
83.0063
83.0094
82.9871
83.0004
83.0107
83.0218
83.0045
82.9848
83.0107
83.0091
83.0063
83.0045
82.9853
83.0004
83.0019
83.0063
83.0063
82.9997
82.9974
82.9974
83.0063
83.0218
82.9871
82.9974
83.0004
83.0218
83.0094
82.9997
82.9974
83.0004
83.0094
83.0063
82.9848
83.0019
83.0107
83.0094
83.0063
82.9997
83.0004
82.9974
83.0094
83.0063
82.9997
83.0004
83.0019
83.0045
83.0045
82.9848
82.9974
83.0004
83.0063
83.0045
82.9871
83.0091
82.9974
83.0396
83.0218
82.9871
83.0091
83.0091
83.0063
83.0045
82.9997
83.0004
83.0091
83.0045
83.0218
82.9853
83.0004
83.0004
83.0218
83.0094
82.9997
83.0107
83.0004
83.0396
83.0094
82.9871
82.9974
82.9974
83.0045
83.0094
82.9871
83.0107
83.0004
83.0063
83.0218
82.9848
82.9974
83.0107
83.0218
83.0045
82.9871
83.0107
83.0107
83.0218
83.0063
82.9997
83.0091
83.0019
83.0218
83.0063
82.9848
82.9974
82.9974
83.0396
83.0045
82.9871
82.9974
83.0004
83.0045
83.0396
82.9848
83.0004
83.0019
83.0045
83.0396
82.9848
83.0107
83.0004
83.0094
83.0396
82.9871
83.0019
83.0091
83.0045
83.0045
82.9997
83.0004
83.0107
83.0063
83.0063
82.9853
83.0107
82.9974
83.0218
83.0396
82.9871
83.0004
82.9974
83.0045
83.0396
82.9848
83.0091
83.0107
83.0218
83.0063
82.9871
83.0019
83.0019
83.0063
83.0045
82.9871
82.9974
83.0091
83.0218
83.0045
82.9997
83.0107
82.9974
83.0218
83.0094
83.0366
83.0091
83.0107
83.0218
83.0045
82.9848
83.0019
83.0004
83.0063
83.0045
82.9871
83.0091
83.0107
83.0045
83.0045
82.9853
83.0004
83.0019
83.0396
83.0396
82.9853
83.0019
83.0019
83.0045
83.0094
82.9997
83.0091
83.0091
83.0396
83.0045
82.9853
83.0019
83.0091
83.0045
83.0063
82.9853
83.0004
83.0004
83.0396
83.0063
82.9853
83.0091
83.0004
83.0045
83.0218
82.9997
82.9974
83.0091
83.0218
83.0045
82.9848
83.0019
83.0004
83.0063
83.0396
82.9853
83.0019
82.9974
83.0045
83.0396
82.9853
83.0019
83.0019
83.0218
83.0094
83.0366
82.9974
82.9974
83.0396
83.0094
82.9848
83.0019
83.0107
83.0045
83.0094
82.9848
83.0019
82.9974
83.0094
83.0218
82.9848
82.9974
83.0107
83.0218
83.0094
82.9853
82.9974
83.0004
83.0063
83.0094
82.9997
83.0107
83.0019
83.0396
83.0218
82.9871
83.0091
83.0091
83.0094
83.0094
82.9848
83.0091
83.0107
83.0094
83.0045
82.9997
83.0019
83.0004
83.0094
83.0045
82.9853
82.9974
83.0004
83.0218
83.0396
82.9871
83.0019
82.9974
83.0396
83.0218
82.9848
82.9974
83.0107
83.0396
83.0063
82.9848
82.9974
83.0019
83.0063
83.0094
82.9997
82.9974
83.0004
83.0218
83.0045
82.9848
83.0107
83.0019
83.0045
83.0218
83.031
83.0107
82.9974
83.0094
83.0396
82.9871
83.0091
83.0091
83.0063
83.0094
82.9853
82.9974
83.0004
83.0045
83.0218
82.9997
83.0019
83.0091
83.0063
83.0396
83.0366
83.0004
83.0019
83.0396
83.0094
82.9853
83.0091
83.0019
83.0094
83.0045
82.9871
83.0091
83.0019
83.0396
83.0218
83.031
82.9974
83.0004
83.0396
83.0094
82.9997
83.0019
83.0019
83.0396
83.0094
82.9853
83.0019
83.0107
83.0218
83.0063
82.9853
83.0019
83.0004
83.0063
83.0045
82.9853
82.9974
82.9974
83.0045
83.0063
82.9871
82.9974
83.0091
83.0063
83.0045
82.9848
83.0019
83.0091
83.0218
83.0218
82.9853
83.0004
83.0091
83.0094
83.0094
82.9997
82.9974
83.0004
83.0045
83.0218
82.9848
83.0107
83.0091
83.0063
83.0094
82.9612
82.9974
83.0019
83.0396
83.0094
82.9853
83.0091
83.0107
83.0094
83.0094
82.9871
83.0004
82.9974
83.0045
83.0218
82.9853
83.0019
83.0019
83.0063
83.0045
82.9871
83.0091
83.0091
83.0063
83.0396
82.9997
83.0107
83.0091
83.0063
83.0045
82.9853
83.0019
83.0019
83.0396
83.0063
82.9871
83.0004
82.9974
83.0063
83.0045
82.9848
83.0091
83.0004
83.0045
83.0396
82.9848
83.0091
83.0091
83.0094
83.0045
83.0366
83.0091
83.0004
83.0218
83.0218
82.9871
82.9974
83.0091
83.0396
83.0094
82.9997
83.0107
83.0019
83.0094
83.0094
82.9848
83.0091
83.0091
83.0063
83.0218
82.9997
83.0004
83.0019
83.0045
83.0063
82.9871
83.0091
83.0004
83.0218
83.0396
82.9871
83.0107
83.0107
83.0063
83.0094
83.0366
83.0091
82.9974
83.0045
83.0045
83.0366
82.9974
83.0107
83.0063
83.0218
82.9871
83.0091
83.0107
83.0218
83.0094
82.9997
83.0004
82.9974
83.0063
83.0218
82.9997
82.9974
83.0019
83.0396
83.0396
82.9997
82.9974
83.0107
83.0094
83.0396
82.9848
83.0019
82.9974
83.0094
83.0396
83.0366
83.0107
83.0091
83.0094
83.0396
82.9853
82.9974
82.9974
83.0094
83.0396
82.9871
83.0107
82.9974
83.0094
83.0218
82.9997
83.0091
83.0107
83.0396
83.0063
82.9848
83.0107
82.9974
83.0218
83.0218
82.9871
83.0107
83.0091
83.0396
83.0094
82.9853
82.9974
82.9974
83.0396
83.0396
82.9871
83.0004
83.0091
83.0063
83.0045
82.9848
83.0107
83.0091
83.0396
83.0063
82.9997
83.0107
82.9974
83.0218
83.0396
82.9997
83.0091
83.0107
83.0218
83.0094
82.9848
83.0004
82.9974
83.0063
83.0396
82.9997
83.0091
83.0019
83.0063
83.0396
82.9853
82.9974
83.0004
83.0063
83.0396
82.9848
83.0004
83.0019
83.0063
83.0094
82.9871
83.0107
83.0107
83.0045
83.0045
82.9848
82.9974
82.9974
83.0094
83.0045
82.9848
83.0004
82.9974
83.0063
83.0218
82.9848
82.9974
83.0004
83.0063
83.0218
82.9871
83.0107
83.0091
83.0045
83.0218
82.9871
83.0107
83.0107
83.0094
83.0063
82.9853
83.0091
82.9974
83.0218
83.0063
82.9848
82.9974
83.0091
83.0218
83.0396
82.9853
82.9974
82.9974
83.0094
83.0218
82.9997
83.0004
83.0107
83.0218
83.0218
82.9853
82.9974
83.0107
83.0063
83.0396
82.9871
83.0004
83.0107
83.0396
83.0218
82.9997
83.0091
83.0091
83.0094
83.0063
82.9871
82.9974
83.0107
83.0396
83.0094
82.9853
83.0091
83.0091
83.0063
83.0218
82.9871
83.0019
82.9974
83.0045
83.0094
82.9997
83.0004
83.0091
83.0218
83.0218
82.9871
82.9974
83.0019
83.0045
83.0045
82.9853
83.0019
83.0019
83.0045
83.0218
82.9997
83.0107
83.0004
83.0218
83.0218
82.9871
83.0004
83.0091
83.0045
83.0396
82.9997
83.0004
82.9974
83.0045
83.0094
82.9853
82.9974
83.0091
83.0063
83.0045
82.9997
83.0091
83.0004
83.0094
83.0218
82.9853
83.0019
83.0091
83.0094
83.0063
82.9997
83.0107
83.0091
83.0094
83.0045
82.9997
83.0091
83.0004
83.0094
83.0045
82.9853
83.0004
82.9974
83.0094
83.0218
82.9853
82.9974
82.9974
83.0094
83.0094
82.9997
82.9974
83.0004
83.0396
83.0094
83.0366
83.0004
82.9974
83.0045
83.0045
82.9997
82.9974
83.0019
83.0094
83.0396
82.9871
83.0019
83.0004
83.0396
83.0094
82.9871
82.9974
83.0004
83.0045
83.0063
82.9871
83.0091
83.0019
83.0063
83.0045
82.9848
82.9974
83.0107
83.0063
83.0045
82.9871
83.0019
83.0004
83.0063
83.0063
82.9871
83.0019
83.0107
83.0094
83.0218
82.9848
83.0107
83.0004
83.0396
83.0063
82.9871
83.0107
82.9974
83.0094
83.0063
82.9871
83.0091
83.0019
83.0063
83.0094
82.9848
83.0004
82.9974
83.0094
83.0218
82.9848
83.0004
83.0107
83.0063
83.0063
82.9871
82.9974
83.0004
83.0063
83.0218
82.9871
83.0004
83.0019
83.0045
83.0218
82.9997
83.0091
83.0107
83.0045
83.0045
82.9853
83.0107
83.0107
83.0045
83.0094
82.9997
82.9974
83.0004
83.0063
83.0218
82.9997
83.0107
83.0091
83.0063
83.0045
82.9848
83.0107
82.9974
83.0094
83.0094
82.9871
83.0107
82.9974
83.0218
83.0094
82.9848
83.0019
83.0019
83.0063
83.0063
82.9848
83.0091
83.0091
83.0396
83.0045
82.9997
83.0004
83.0091
83.0094
83.0094
82.9997
83.0004
83.0091
83.0063
83.0396
82.9871
83.0004
83.0107
83.0396
83.0218
82.9871
83.0107
83.0004
83.0045
83.0063
83.0366
83.0107
83.0107
83.0218
83.0063
82.9848
82.9974
83.0004
83.0063
83.0063
82.9848
83.0091
82.9974
83.0063
83.0045
82.9848
83.0107
83.0091
83.0094
83.0396
82.9848
83.0004
83.0019
83.0094
83.0396
82.9853
82.9974
83.0091
83.0218
83.0094
82.9997
83.0004
83.0019
83.0218
83.0396
82.9848
83.0107
83.0091
83.0218
83.0063
82.9853
83.0004
83.0091
83.0094
83.0063
82.9997
83.0019
83.0019
83.0094
83.0218
82.9848
83.0019
83.0107
83.0396
83.0045
82.9848
83.0107
83.0019
83.0218
83.0094
82.9848
83.0019
82.9974
83.0094
83.0218
82.9871
83.0107
83.0107
83.0063
83.0063
82.9871
83.0107
82.9974
83.0045
83.0045
82.9853
83.0107
83.0107
83.0063
83.0396
82.9853
83.0019
83.0107
83.0045
83.0218
82.9997
83.0091
82.9974
83.0045
83.0045
82.9871
83.0091
83.0107
83.0063
83.0063
82.9848
83.0091
83.0004
83.0045
83.0094
82.9853
83.0004
82.9974
83.0094
83.0218
82.9997
82.9974
83.0019
83.0094
83.0218
82.9848
82.9974
83.0107
83.0218
83.0063
82.9848
82.9974
82.9974
83.0063
83.0094
82.9997
82.9974
83.0004
83.0396
83.0218
82.9853
83.0004
83.0019
83.0396
83.0094
82.9848
83.0004
82.9974
83.0045
83.0218
82.9853
83.0107
83.0091
83.0094
83.0063
82.9871
83.0107
82.9974
83.0094
83.0094
82.9848
83.0019
83.0107
83.0218
83.0396
82.9848
83.0091
83.0004
83.0063
83.0218
82.9853
83.0019
83.0019
83.0396
83.0396
82.9997
83.0091
83.0091
83.0063
83.0396
82.9997
83.0019
82.9974
83.0218
83.0063
82.9848
83.0107
82.9974
83.0063
83.0045
82.9997
83.0019
83.0091
83.0218
83.0396
82.9612
83.0004
83.0004
83.0094
83.0063
82.9997
83.0019
83.0019
83.0396
83.0396
82.9853
83.0019
83.0019
83.0396
83.0094
82.9871
83.0019
83.0107
83.0094
83.0094
82.9853
83.0004
82.9974
83.0045
83.0396
82.9871
83.0019
83.0004
83.0218
83.0218
82.9997
82.9974
82.9974
83.0094
83.0094
82.9997
82.9974
83.0004
83.0045
83.0218
82.9848
83.0019
83.0091
83.0218
83.0396
82.9997
83.0004
82.9974
83.0218
83.0063
82.9848
83.0107
83.0019
83.0045
83.0063
82.9853
83.0019
83.0004
83.0045
83.0094
82.9871
82.9974
83.0019
83.0063
83.0063
82.9997
82.9974
83.0091
83.0094
83.0094
82.9997
83.0004
83.0019
83.0063
83.0396
82.9997
83.0019
83.0107
83.0218
83.0218
82.9612
83.0004
83.0107
83.0396
83.0094
82.9997
83.0019
83.0091
83.0396
83.0094
82.9848
83.0107
83.0107
83.0063
83.0218
82.9853
83.0091
83.0019
83.0063
83.0094
82.9848
82.9974
83.0107
83.0045
83.0045
82.9997
83.0019
83.0004
83.0218
83.0218
82.9612
83.0019
83.0107
83.0094
83.0063
82.9848
83.0019
82.9974
83.0218
83.0094
82.9848
83.0019
82.9974
83.0396
83.0045
82.9871
82.9974
83.0091
83.0063
83.0396
82.9871
83.0019
83.0004
83.0094
83.0218
82.9853
83.0004
83.0107
83.0045
83.0063
82.9853
83.0019
83.0019
83.0396
83.0063
82.9853
83.0019
83.0107
83.0218
83.0063
82.9871
83.0091
82.9974
83.0063
83.0045
82.9997
83.0091
83.0019
83.0094
83.0063
82.9997
83.0004
83.0004
83.0063
83.0396
82.9853
83.0107
83.0107
83.0045
83.0094
82.9997
82.9974
83.0107
83.0094
83.0063
82.9853
83.0091
83.0004
83.0218
83.0396
83.0366
83.0004
83.0107
83.0045
83.0063
82.9871
82.9974
83.0004
83.0063
83.0094
82.9997
82.9974
82.9974
83.0218
83.0396
82.9848
83.0019
83.0004
83.0094
83.0094
82.9871
83.0004
82.9974
83.0094
83.0063
83.0366
82.9974
82.9974
83.0063
83.0396
82.9997
82.9974
82.9974
83.0094
83.0218
82.9871
82.9974
82.9974
83.0063
83.0063
82.9853
83.0004
83.0004
83.0218
83.0063
82.9997
83.0091
82.9974
83.0063
83.0045
82.9853
83.0019
83.0107
83.0218
83.0218
82.9848
83.0091
83.0107
83.0218
83.0063
82.9997
83.0019
82.9974
83.0045
83.0396
82.9997
82.9974
83.0004
83.0218
83.0218
82.9853
82.9974
83.0091
83.0218
83.0218
82.9848
82.9974
83.0004
83.0218
83.0218
82.9848
83.0091
83.0019
83.0218
83.0063
82.9853
83.0091
83.0004
83.0063
83.0218
82.9853
83.0091
83.0004
83.0396
83.0218
82.9871
83.0019
83.0107
83.0396
83.0045
82.9997
83.0004
83.0004
83.0396
83.0063
82.9853
83.0004
83.0004
83.0218
83.0045
82.9848
82.9974
83.0107
83.0063
83.0063
82.9997
82.9974
83.0107
83.0045
83.0063
82.9848
83.0019
82.9974
83.0094
83.0063
82.9871
83.0004
83.0019
83.0094
83.0094
82.9871
82.9974
83.0004
83.0094
83.0045
82.9853
83.0107
83.0019
83.0396
83.0063
83.0366
83.0091
83.0091
83.0094
83.0094
83.0366
83.0091
83.0107
83.0218
83.0094
82.9997
83.0091
83.0019
83.0396
83.0094
82.9848
83.0004
83.0107
83.0218
83.0218
82.9997
82.9974
83.0004
83.0396
83.0063
82.9848
83.0004
83.0004
83.0063
83.0094
82.9871
83.0091
83.0019
83.0094
83.0045
82.9997
83.0004
83.0004
83.0045
83.0094
82.9853
83.0019
83.0019
83.0094
83.0396
82.9997
82.9974
83.0107
83.0063
83.0045
83.0366
82.9974
83.0091
83.0045
83.0063
82.9848
83.0004
83.0091
83.0045
83.0045
82.9871
83.0107
83.0091
83.0045
83.0396
82.9871
83.0107
83.0019
83.0063
83.0396
82.9997
83.0004
83.0004
83.0063
83.0218
82.9853
83.0004
83.0091
83.0396
83.0396
83.0366
83.0004
83.0004
83.0063
83.0045
82.9848
82.9974
83.0107
83.0396
83.0396
82.9997
83.0107
83.0091
83.0094
83.0218
82.9871
83.0107
83.0107
83.0045
83.0396
82.9848
83.0019
83.0107
83.0063
83.0218
82.9871
82.9974
82.9974
83.0094
83.0045
82.9853
83.0091
83.0004
83.0063
83.0396
82.9848
83.0004
82.9974
83.0063
83.0063
82.9871
83.0107
83.0091
83.0094
83.0063
82.9997
82.9974
83.0107
83.0045
83.0396
82.9997
82.9974
83.0107
83.0094
83.0045
82.9997
82.9974
83.0091
83.0094
83.0396
82.9997
83.0004
83.0091
83.0094
83.0045
82.9848
83.0019
83.0107
83.0218
83.0045
82.9871
83.0004
82.9974
83.0045
83.0396
82.9871
83.0107
83.0004
83.0045
83.0396
82.9871
83.0004
83.0004
83.0094
83.0045
82.9871
82.9974
83.0019
83.0063
83.0045
83.0366
83.0107
83.0107
83.0218
83.0045
82.9853
83.0004
83.0091
83.0063
83.0396
82.9997
83.0019
83.0019
83.0218
83.0063
82.9871
83.0091
83.0091
83.0218
83.0218
82.9848
82.9974
83.0107
83.0094
83.0063
82.9853
83.0107
83.0004
83.0063
83.0094
82.9871
83.0019
82.9974
83.0063
83.0063
82.9997
83.0004
83.0107
83.0396
83.0045
82.9853
83.0107
83.0091
83.0094
83.0218
82.9848
83.0019
83.0091
83.0218
83.0063
82.9997
83.0107
83.0004
83.0396
83.0094
82.9997
83.0091
83.0091
83.0063
83.0045
82.9997
83.0107
82.9974
83.0094
83.0218
82.9848
83.0091
83.0004
83.0396
83.0218
82.9848
83.0091
83.0107
83.0218
83.0063
82.9848
83.0019
82.9974
83.0396
83.0045
83.0366
83.0091
83.0004
83.0218
83.0218
82.9853
83.0004
82.9974
83.0094
83.0218
82.9853
83.0019
82.9974
83.0094
83.0396
82.9997
83.0091
83.0004
83.0218
83.0063
82.9853
83.0091
83.0107
83.0094
83.0063
82.9848
83.0004
83.0004
83.0218
83.0218
82.9997
83.0019
83.0091
83.0218
83.0094
82.9853
83.0107
83.0107
83.0094
83.0396
82.9848
82.9974
83.0091
83.0094
83.0218
82.9871
83.0107
82.9974
83.0218
83.0094
82.9853
83.0091
83.0091
83.0396
83.0094
82.9871
83.0107
83.0004
83.0045
83.0063
82.9848
82.9974
82.9974
83.0094
83.0045
82.9997
83.0091
82.9974
83.0218
83.0045
82.9997
83.0019
83.0019
83.0094
83.0396
82.9871
83.0004
83.0019
83.0094
83.0218
82.9871
83.0107
83.0107
83.0218
83.0094
82.9997
83.0019
83.0019
83.0045
83.0045
82.9871
83.0019
83.0107
83.0396
83.0094
82.9997
82.9974
82.9974
83.0063
83.0396
82.9997
83.0004
83.0091
83.0218
83.0045
82.9848
83.0091
82.9974
83.0218
83.0396
82.9848
83.0004
82.9974
83.0045
83.0218
82.9853
82.9974
83.0004
83.0218
83.0063
82.9871
83.0091
83.0019
83.0218
83.0218
82.9848
82.9974
83.0091
83.0218
83.0045
82.9853
83.0107
82.9974
83.0063
83.0218
82.9848
83.0004
83.0091
83.0094
83.0063
82.9997
83.0107
83.0091
83.0045
83.0045
82.9997
83.0091
83.0107
83.0094
83.0045
82.9848
83.0091
83.0019
83.0063
83.0045
82.9871
82.9974
82.9974
83.0218
83.0063
82.9853
83.0004
83.0091
83.0094
83.0396
82.9997
83.0107
83.0004
83.0045
83.0063
82.9853
83.0091
83.0004
83.0396
83.0218
82.9997
82.9974
83.0004
83.0045
83.0063
82.9848
83.0019
83.0107
83.0045
83.0396
82.9853
83.0107
83.0019
83.0218
83.0396
82.9871
83.0019
83.0091
83.0094
83.0045
82.9871
83.0004
82.9974
83.0045
83.0218
82.9871
83.0019
83.0004
83.0063
83.0063
82.9871
83.0004
83.0019
83.0045
83.0396
82.9848
83.0004
83.0091
83.0045
83.0218
82.9871
83.0019
83.0091
83.0063
83.0094
82.9848
82.9974
83.0091
83.0063
83.0045
82.9997
83.0107
83.0107
83.0045
83.0218
82.9848
83.0019
83.0019
83.0218
83.0396
82.9848
83.0107
83.0019
83.0218
83.0218
83.0366
83.0107
82.9974
83.0045
83.0063
82.9997
83.0019
83.0004
83.0094
83.0396
82.9997
83.0107
83.0019
83.0396
83.0063
82.9853
83.0019
82.9974
83.0094
83.0396
82.9853
83.0004
83.0019
83.0396
83.0045
82.9871
83.0091
83.0091
83.0045
83.0218
82.9848
83.0091
83.0091
83.0218
83.0094
82.9853
83.0019
83.0091
83.0063
83.0396
82.9848
83.0004
83.0004
83.0063
83.0094
82.9853
83.0004
82.9974
83.0045
83.0396
83.0366
83.0004
83.0107
83.0218
83.0045
82.9848
83.0019
82.9974
83.0045
83.0094
82.9997
83.0091
83.0107
83.0218
83.0396
82.9997
83.0091
83.0019
83.0094
83.0063
82.9848
83.0107
83.0004
83.0396
83.0063
83.0366
83.0091
83.0004
83.0063
83.0094
82.9997
83.0091
83.0004
83.0063
83.0218
82.9871
83.0019
83.0019
83.0045
83.0094
83.0366
83.0004
83.0091
83.0094
83.0218
82.9848
83.0091
83.0004
83.0218
83.0094
82.9853
83.0091
83.0004
83.0396
83.0094
82.9853
82.9974
83.0004
83.0045
83.0063
82.9853
82.9974
83.0019
83.0094
83.0218
82.9853
83.0019
82.9974
83.0063
83.0396
82.9848
83.0019
83.0004
83.0045
83.0094
82.9871
83.0091
82.9974
83.0063
83.0094
82.9848
82.9974
83.0091
83.0396
83.0063
82.9848
83.0107
82.9974
83.0218
83.0063
82.9871
83.0091
82.9974
83.0218
83.0063
82.9848
82.9974
83.0091
83.0396
83.0218
82.9997
83.0107
82.9974
83.0063
83.0396
82.9848
83.0019
82.9974
83.0063
83.0063
82.9997
83.0107
83.0091
83.0063
83.0094
82.9871
83.0004
83.0107
83.0063
83.0094
82.9853
83.0091
82.9974
83.0396
83.0063
82.9871
83.0019
83.0019
83.0063
83.0094
82.9853
83.0019
83.0019
83.0218
83.0063
82.9853
82.9974
83.0004
83.0396
83.0396
82.9871
83.0107
83.0019
83.0063
83.0396
82.9871
83.0107
82.9974
83.0218
83.0045
82.9853
82.9974
83.0019
83.0396
83.0094
82.9853
83.0019
83.0091
83.0218
83.0094
82.9848
82.9974
83.0004
83.0063
83.0218
82.9848
83.0091
83.0107
83.0218
83.0094
82.9853
83.0091
82.9974
83.0063
83.0094
83.0366
83.0107
83.0004
83.0045
83.0063
82.9853
82.9974
83.0004
83.0094
83.0094
82.9853
82.9974
82.9974
83.0396
83.0094
82.9871
82.9974
82.9974
83.0063
83.0396
82.9848
82.9974
83.0004
83.0063
83.0218
82.9853
83.0019
83.0107
83.0396
83.0063
82.9853
83.0004
83.0091
83.0063
83.0218
82.9853
83.0091
83.0091
83.0396
83.0063
82.9848
83.0091
83.0091
83.0045
83.0094
82.9853
83.0107
83.0019
83.0045
83.0094
82.9853
83.0091
82.9974
83.0045
83.0063
82.9871
82.9974
83.0004
83.0045
83.0063
82.9871
83.0019
82.9974
83.0218
83.0094
82.9997
82.9974
83.0004
83.0396
83.0045
82.9853
83.0019
83.0091
83.0094
83.0094
82.9997
82.9974
83.0019
83.0396
83.0045
82.9871
83.0107
83.0107
83.0094
83.0218
82.9853
83.0107
82.9974
83.0396
83.0063
82.9871
83.0019
83.0004
83.0094
83.0094
82.9871
83.0004
83.0091
83.0396
83.0045
82.9997
83.0091
83.0019
83.0094
83.0396
82.9997
83.0107
82.9974
83.0218
83.0094
82.9997
82.9974
83.0091
83.0063
83.0045
82.9848
83.0107
83.0004
83.0218
83.0094
82.9853
83.0019
83.0004
83.0045
83.0063
82.9871
83.0107
83.0107
83.0396
83.0045
82.9871
82.9974
82.9974
83.0063
83.0094
82.9848
83.0004
83.0107
83.0218
83.0396
82.9848
83.0019
83.0004
83.0094
83.0045
82.9853
83.0019
83.0019
83.0094
83.0063
82.9853
83.0091
83.0091
83.0045
83.0218
82.9853
83.0004
83.0091
83.0396
83.0218
82.9853
83.0107
82.9974
83.0094
83.0094
82.9853
83.0107
83.0004
83.0094
83.0094
82.9848
82.9974
82.9974
83.0396
83.0396
83.0366
83.0107
83.0107
83.0045
83.0396
82.9871
83.0004
83.0107
83.0396
83.0063
82.9997
83.0091
83.0107
83.0218
83.0218
82.9848
83.0107
83.0004
83.0063
83.0094
82.9853
83.0019
83.0091
83.0045
83.0218
82.9848
82.9974
82.9974
83.0396
83.0094
82.9848
83.0004
83.0004
83.0045
83.0218
82.9997
83.0107
83.0091
83.0045
83.0045
82.9871
83.0107
83.0019
83.0094
83.0218
82.9997
83.0004
83.0019
83.0396
83.0218
82.9997
83.0091
83.0107
83.0045
83.0396
82.9997
83.0107
83.0107
83.0063
83.0045
82.9853
83.0091
83.0004
83.0094
83.0094
82.9848
83.0091
83.0091
83.0218
83.0063
82.9871
82.9974
82.9974
83.0063
83.0218
82.9871
83.0107
82.9974
83.0094
83.0045
82.9853
82.9974
83.0004
83.0396
83.0094
82.9997
82.9974
83.0004
83.0063
83.0218
82.9853
83.0091
82.9974
83.0094
83.0094
82.9871
83.0019
83.0004
83.0396
83.0063
82.9871
83.0107
83.0091
83.0396
83.0218
82.9871
82.9974
83.0004
83.0063
83.0045
82.9848
83.0004
83.0107
83.0063
83.0094
82.9848
83.0019
83.0091
83.0218
83.0094
82.9871
83.0019
83.0019
83.0218
83.0063
82.9871
83.0107
83.0107
83.0045
83.0396
82.9871
82.9974
83.0107
83.0045
83.0045
82.9848
82.9974
83.0107
83.0063
83.0045
82.9871
83.0004
83.0107
83.0063
83.0063
82.9997
83.0004
83.0019
83.0063
83.0063
82.9848
83.0019
83.0107
83.0063
83.0396
82.9853
83.0019
83.0019
83.0396
83.0045
83.0366
83.0107
83.0019
83.0045
83.0063
83.0366
82.9974
83.0091
83.0396
83.0396
82.9871
83.0091
83.0004
83.0396
83.0396
82.9871
83.0091
83.0019
83.0063
83.0094
82.9853
83.0091
83.0019
83.0094
83.0045
82.9853
82.9974
83.0107
83.0094
83.0094
82.9997
83.0004
83.0019
83.0045
83.0218
82.9853
83.0019
83.0107
83.0396
83.0045
82.9871
83.0004
83.0004
83.0094
83.0218
82.9997
83.0091
83.0091
83.0094
83.0218
82.9853
83.0004
83.0091
83.0218
83.0396
82.9853
83.0107
83.0004
83.0396
83.0063
82.9997
83.0004
82.9974
83.0218
83.0045
82.9997
82.9974
83.0107
83.0063
83.0045
82.9848
83.0107
83.0019
83.0063
83.0063
82.9871
83.0091
83.0004
83.0094
83.0094
82.9848
83.0019
83.0107
83.0063
83.0045
82.9871
83.0091
83.0107
83.0396
83.0045
82.9871
83.0019
83.0091
83.0063
83.0063
82.9848
83.0091
83.0019
83.0063
83.0218
82.9871
83.0004
83.0019
83.0063
83.0396
82.9997
83.0004
82.9974
83.0045
83.0094
82.9871
83.0107
83.0107
83.0045
83.0396
82.9997
83.0107
83.0019
83.0094
83.0218
83.031
82.9974
83.0107
83.0094
83.0063
82.9853
83.0004
83.0107
83.0063
83.0396
82.9871
83.0004
82.9974
83.0094
83.0396
82.9848
83.0107
83.0107
83.0063
83.0045
82.9997
83.0107
83.0091
83.0396
83.0045
82.9853
83.0004
83.0091
83.0063
83.0396
82.9871
82.9974
82.9974
83.0063
83.0218
83.0366
83.0107
83.0091
83.0094
83.0063
82.9853
83.0019
83.0107
83.0396
83.0045
82.9853
83.0107
83.0019
83.0218
83.0396
82.9997
83.0091
83.0004
83.0094
83.0396
82.9997
83.0004
83.0107
83.0045
83.0396
82.9848
83.0091
83.0107
83.0045
83.0045
82.9871
83.0107
83.0107
83.0094
83.0396
83.0366
82.9974
82.9974
83.0396
83.0063
82.9853
83.0019
83.0091
83.0045
83.0218
82.9871
83.0019
83.0091
83.0063
83.0063
82.9997
82.9974
83.0004
83.0396
83.0396
82.9853
83.0019
83.0019
83.0045
83.0396
82.9853
83.0107
83.0107
83.0396
83.0045
82.9871
83.0004
83.0019
83.0045
83.0094
82.9848
83.0019
83.0107
83.0094
83.0396
82.9997
83.0004
83.0107
83.0045
83.0094
82.9871
83.0019
83.0107
83.0094
83.0045
82.9997
83.0107
83.0004
83.0045
83.0396
82.9871
83.0107
82.9974
83.0063
83.0094
82.9871
83.0019
83.0019
83.0218
83.0218
82.9997
83.0019
83.0091
83.0094
83.0094
82.9997
83.0004
83.0107
83.0094
83.0396
82.9848
83.0107
83.0091
83.0396
83.0218
82.9871
82.9974
83.0091
83.0063
83.0218
82.9871
83.0004
83.0107
83.0094
83.0063
82.9848
82.9974
83.0019
83.0063
83.0218
82.9853
83.0091
83.0091
83.0045
83.0218
83.0366
83.0107
83.0091
83.0396
83.0063
82.9853
83.0019
83.0107
83.0218
83.0094
82.9997
83.0004
83.0019
83.0094
83.0063
83.0366
83.0019
83.0019
83.0063
83.0063
82.9871
83.0107
82.9974
83.0218
83.0218
82.9871
82.9974
83.0091
83.0396
83.0094
82.9853
83.0004
83.0019
83.0218
83.0396
82.9853
82.9974
83.0091
83.0094
83.0045
83.0366
83.0107
83.0107
83.0094
83.0063
82.9871
82.9974
83.0004
83.0396
83.0063
82.9871
83.0004
83.0107
83.0218
83.0396
82.9871
83.0019
83.0107
83.0045
83.0063
82.9853
83.0091
83.0107
83.0094
83.0063
82.9848
83.0004
82.9974
83.0045
83.0045
82.9853
83.0004
83.0004
83.0045
83.0094
82.9997
82.9974
83.0019
83.0063
83.0063
82.9848
83.0107
83.0091
83.0218
83.0218
82.9853
82.9974
83.0019
83.0063
83.0218
82.9848
82.9974
83.0004
83.0063
83.0396
82.9871
83.0107
83.0004
83.0045
83.0063
82.9997
83.0019
82.9974
83.0218
83.0094
82.9848
83.0019
83.0019
83.0218
83.0094
82.9848
83.0019
83.0019
83.0045
83.0063
82.9997
83.0004
82.9974
83.0396
83.0396
82.9848
83.0091
82.9974
83.0094
83.0218
82.9997
83.0091
83.0019
83.0094
83.0063
82.9853
83.0004
82.9974
83.0063
83.0045
82.9848
82.9974
83.0091
83.0045
83.0218
82.9853
82.9974
82.9974
83.0094
83.0045
82.9848
83.0004
83.0004
83.0063
83.0045
82.9871
83.0019
82.9974
83.0045
83.0094
82.9997
83.0004
83.0004
83.0045
83.0396
82.9853
83.0091
83.0091
83.0063
83.0094
82.9871
83.0107
83.0019
83.0396
83.0218
82.9997
82.9974
83.0004
83.0094
83.0218
82.9848
83.0091
82.9974
83.0045
83.0045
82.9871
83.0091
83.0107
83.0218
83.0218
82.9848
83.0019
83.0091
83.0094
83.0218
83.0366
82.9974
83.0004
83.0063
83.0094
82.9997
83.0091
83.0004
83.0396
83.0063
82.9997
83.0091
83.0091
83.0094
83.0218
82.9871
82.9974
83.0019
83.0218
83.0396
82.9853
83.0107
83.0091
83.0045
83.0218
82.9871
83.0107
83.0107
83.0094
83.0045
82.9871
83.0019
83.0004
83.0063
83.0045
82.9871
83.0091
83.0091
83.0063
83.0218
82.9871
82.9974
83.0091
83.0218
83.0396
82.9848
82.9974
82.9974
83.0094
83.0094
82.9997
82.9974
82.9974
83.0063
83.0094
82.9853
82.9974
83.0091
83.0063
83.0063
82.9997
83.0091
83.0019
83.0045
83.0063
83.0366
83.0107
83.0107
83.0063
83.0396
82.9871
82.9974
83.0004
83.0396
83.0063
82.9997
83.0004
82.9974
83.0045
83.0218
82.9871
83.0019
83.0019
83.0218
83.0218
82.9853
83.0019
82.9974
83.0094
83.0063
82.9853
83.0091
83.0107
83.0045
83.0045
82.9871
83.0091
82.9974
83.0045
83.0396
82.9848
83.0019
83.0091
83.0396
83.0063
82.9853
83.0107
83.0004
83.0396
83.0094
83.0366
83.0004
83.0091
83.0396
83.0396
83.0366
82.9974
83.0091
83.0045
83.0063
83.0366
83.0091
83.0091
83.0396
83.0218
82.9997
83.0004
83.0019
83.0218
83.0045
83.0366
82.9974
83.0004
83.0094
83.0218
82.9853
83.0107
83.0004
83.0218
83.0063
82.9871
83.0091
83.0107
83.0094
83.0094
82.9848
83.0107
83.0107
83.0045
83.0218
82.9871
83.0004
82.9974
83.0396
83.0396
82.9871
83.0091
83.0107
83.0045
83.0094
82.9997
82.9974
83.0091
83.0094
83.0396
82.9871
82.9974
83.0107
83.0094
83.0218
82.9848
83.0107
83.0004
83.0094
83.0063
82.9997
82.9974
83.0107
83.0063
83.0218
82.9848
83.0019
83.0091
83.0218
83.0063
83.0366
82.9974
83.0107
83.0045
83.0045
82.9853
83.0091
83.0107
83.0094
83.0045
82.9853
83.0019
83.0091
83.0396
83.0094
82.9853
83.0107
83.0019
83.0094
83.0063
82.9853
83.0091
83.0091
83.0045
83.0063
82.9853
82.9974
83.0004
83.0218
83.0063
82.9848
83.0019
83.0107
83.0094
83.0396
82.9853
83.0091
82.9974
83.0218
83.0063
82.9848
83.0019
83.0091
83.0218
83.0094
82.9853
82.9974
83.0019
83.0396
83.0094
82.9848
83.0091
83.0019
83.0094
83.0063
82.9997
83.0004
83.0004
83.0094
83.0396
82.9853
83.0107
83.0107
83.0094
83.0094
82.9997
83.0107
83.0091
83.0045
83.0045
82.9848
83.0091
83.0004
83.0218
83.0063
82.9848
83.0091
83.0019
83.0218
83.0396
82.9997
83.0107
83.0019
83.0045
83.0396
82.9997
83.0107
83.0107
83.0063
83.0396
82.9997
83.0091
83.0107
83.0094
83.0045
82.9853
83.0004
82.9974
83.0396
83.0094
82.9853
83.0107
83.0004
83.0094
83.0218
82.9848
83.0091
82.9974
83.0218
83.0396
82.9853
83.0019
83.0107
83.0045
83.0063
82.9848
83.0004
83.0107
83.0045
83.0045
83.0366
83.0107
83.0019
83.0063
83.0063
82.9997
83.0004
83.0091
83.0094
83.0063
82.9853
83.0004
83.0004
83.0396
83.0218
82.9871
82.9974
83.0091
83.0396
83.0396
82.9853
82.9974
83.0107
83.0218
83.0396
82.9997
83.0107
83.0004
83.0396
83.0094
82.9997
83.0107
83.0004
83.0094
83.0063
82.9848
83.0091
82.9974
83.0045
83.0045
83.0366
82.9974
83.0004
83.0063
83.0063
82.9997
82.9974
83.0004
83.0063
83.0396
82.9871
83.0004
83.0091
83.0063
83.0063
82.9997
83.0004
83.0091
83.0063
83.0218
82.9997
83.0107
83.0091
83.0396
83.0045
82.9853
83.0004
83.0019
83.0063
83.0045
82.9871
82.9974
82.9974
83.0063
83.0063
82.9853
83.0107
83.0004
83.0396
83.0396
82.9848
83.0107
83.0091
83.0063
83.0396
82.9848
82.9974
83.0107
83.0218
83.0045
82.9848
83.0091
83.0019
83.0218
83.0218
82.9997
83.0019
83.0107
83.0094
83.0045
82.9871
83.0004
83.0107
83.0063
83.0396
82.9853
83.0091
83.0091
83.0063
83.0396
82.9848
83.0091
83.0019
83.0094
83.0094
82.9853
83.0004
83.0091
83.0045
83.0045
82.9871
82.9974
83.0091
83.0063
83.0063
82.9848
83.0019
83.0107
83.0045
83.0094
82.9853
83.0107
83.0019
83.0218
83.0045
82.9853
83.0091
83.0004
83.0094
83.0045
82.9997
83.0019
83.0019
83.0045
83.0218
82.9871
83.0019
83.0091
83.0218
83.0094
82.9871
82.9974
83.0004
83.0094
83.0045
82.9853
83.0004
83.0107
83.0218
83.0396
82.9848
83.0091
83.0091
83.0218
83.0063
82.9848
83.0107
82.9974
83.0396
83.0063
82.9848
82.9974
83.0091
83.0045
83.0094
82.9853
83.0019
83.0019
83.0396
83.0045
82.9997
83.0091
83.0091
83.0063
83.0063
82.9871
83.0004
83.0004
83.0063
83.0045
82.9853
83.0091
83.0019
83.0063
83.0045
82.9853
83.0004
83.0107
83.0063
83.0396
82.9853
83.0004
82.9974
83.0218
83.0045
82.9871
83.0091
83.0004
83.0218
83.0063
82.9853
83.0107
83.0091
83.0094
83.0396
82.9848
83.0107
83.0091
83.0094
83.0396
82.9871
83.0019
83.0019
83.0063
83.0396
82.9871
83.0091
83.0091
83.0218
83.0396
82.9848
83.0107
83.0107
83.0094
83.0396
82.9871
83.0107
82.9974
83.0063
83.0063
83.0366
83.0019
83.0019
83.0063
83.0094
82.9853
83.0107
82.9974
83.0063
83.0218
82.9848
83.0091
83.0019
83.0218
83.0218
82.9871
83.0019
83.0019
83.0094
83.0045
82.9853
83.0019
83.0019
83.0396
83.0045
82.9871
82.9974
83.0091
83.0094
83.0396
82.9848
82.9974
83.0107
83.0094
83.0218
82.9848
82.9974
82.9974
83.0218
83.0218
82.9871
83.0019
83.0019
83.0094
83.0218
82.9871
83.0107
83.0019
83.0063
83.0063
82.9853
83.0107
83.0107
83.0396
83.0045
82.9848
83.0019
83.0004
83.0218
83.0396
82.9853
83.0091
83.0004
83.0396
83.0045
82.9848
83.0107
83.0004
83.0218
83.0396
82.9848
83.0004
83.0019
83.0063
83.0396
82.9853
82.9974
82.9974
83.0218
83.0063
82.9997
83.0107
83.0004
83.0218
83.0396
82.9853
83.0091
83.0107
83.0063
83.0063
82.9871
83.0107
83.0091
83.0218
83.0094
82.9871
83.0019
83.0004
83.0396
83.0063
82.9997
83.0004
83.0004
83.0063
83.0063
82.9871
83.0019
83.0004
83.0396
83.0218
82.9848
83.0107
83.0019
83.0396
83.0045
82.9853
82.9974
83.0107
83.0218
83.0094
82.9853
82.9974
83.0004
83.0218
83.0094
82.9848
83.0019
83.0107
83.0094
83.0396
82.9848
82.9974
83.0004
83.0094
83.0063
82.9848
83.0019
82.9974
83.0045
83.0218
82.9853
83.0091
83.0091
83.0094
83.0045
82.9853
83.0107
83.0004
83.0063
83.0045
82.9848
83.0019
83.0091
83.0396
83.0218
82.9848
83.0019
83.0019
83.0045
83.0063
82.9997
82.9974
83.0004
83.0396
83.0063
82.9997
83.0019
83.0019
83.0045
83.0045
82.9997
82.9974
83.0004
83.0063
83.0094
82.9997
83.0019
83.0019
83.0045
83.0045
82.9848
83.0004
83.0107
83.0063
83.0396
83.0366
83.0091
83.0019
83.0094
83.0045
82.9997
83.0004
83.0107
83.0396
83.0396
82.9848
83.0091
83.0091
83.0218
83.0218
82.9853
83.0091
83.0004
83.0094
83.0094
82.9848
83.0019
83.0004
83.0063
83.0094
82.9997
83.0019
83.0107
83.0045
83.0396
83.0366
83.0107
83.0107
83.0218
83.0045
82.9871
82.9974
83.0004
83.0045
83.0063
82.9848
83.0004
83.0004
83.0094
83.0094
82.9853
82.9974
83.0019
83.0094
83.0094
82.9853
83.0004
83.0019
83.0045
83.0218
82.9848
82.9974
83.0107
83.0063
83.0094
83.0366
83.0004
82.9974
83.0094
83.0063
82.9853
83.0019
82.9974
83.0063
83.0218
82.9871
83.0004
83.0091
83.0045
83.0045
82.9871
83.0091
83.0019
83.0396
83.0063
83.0366
82.9974
83.0004
83.0063
83.0218
82.9997
83.0019
83.0107
83.0396
83.0396
82.9871
83.0019
83.0091
83.0396
83.0396
82.9871
83.0004
82.9974
83.0045
83.0045
82.9848
83.0004
83.0019
83.0045
83.0094
82.9997
83.0091
83.0019
83.0045
83.0045
82.9997
83.0004
83.0019
83.0094
83.0063
82.9997
83.0091
83.0004
83.0094
83.0045
82.9612
82.9974
83.0004
83.0063
83.0094
82.9848
83.0107
83.0019
83.0063
83.0218
82.9871
83.0091
83.0019
83.0396
83.0396
82.9871
83.0107
83.0107
83.0045
83.0045
82.9853
83.0091
83.0091
83.0218
83.0396
82.9848
83.0019
83.0019
83.0063
83.0094
82.9853
83.0019
83.0019
83.0218
83.0063
82.9853
83.0091
83.0107
83.0218
83.0218
82.9848
82.9974
83.0004
83.0218
83.0094
82.9997
83.0091
83.0107
83.0094
83.0094
83.0366
83.0004
82.9974
83.0218
83.0045
82.9997
83.0091
82.9974
83.0218
83.0396
82.9848
82.9974
83.0107
83.0218
83.0063
82.9848
83.0107
83.0019
83.0218
83.0218
82.9997
83.0019
83.0107
83.0396
83.0045
82.9871
83.0019
83.0019
83.0396
83.0396
82.9848
83.0004
82.9974
83.0218
83.0045
82.9871
83.0107
82.9974
83.0396
83.0094
82.9848
83.0107
82.9974
83.0218
83.0396
82.9871
83.0019
83.0107
83.0396
83.0396
82.9997
83.0004
83.0019
83.0063
83.0218
82.9997
83.0004
83.0107
83.0063
83.0063
82.9871
83.0091
83.0091
83.0396
83.0218
82.9853
83.0107
83.0019
83.0218
83.0045
82.9871
83.0107
83.0019
83.0063
83.0045
82.9853
82.9974
83.0107
83.0218
83.0396
82.9848
83.0004
82.9974
83.0218
83.0094
82.9853
83.0091
83.0019
83.0396
83.0063
82.9848
82.9974
83.0004
83.0063
83.0396
82.9853
83.0004
82.9974
83.0396
83.0063
82.9871
83.0004
83.0091
83.0063
83.0396
83.0366
83.0019
83.0107
83.0396
83.0045
82.9871
83.0091
82.9974
83.0396
83.0396
82.9848
83.0091
83.0091
83.0094
83.0063
82.9848
83.0091
83.0019
83.0063
83.0045
82.9853
83.0091
83.0019
83.0218
83.0045
82.9997
82.9974
83.0004
83.0045
83.0045
82.9853
82.9974
83.0019
83.0396
83.0063
82.9848
82.9974
82.9974
83.0094
83.0218
82.9997
83.0004
83.0091
83.0396
83.0063
82.9871
83.0019
82.9974
83.0094
83.0094
82.9848
82.9974
83.0107
83.0218
83.0063
82.9848
83.0004
83.0004
83.0396
83.0094
82.9871
83.0107
83.0004
83.0396
83.0396
82.9848
83.0091
83.0107
83.0063
83.0094
83.0366
83.0004
83.0019
83.0045
83.0218
82.9848
83.0091
83.0004
83.0045
83.0396
82.9997
83.0004
82.9974
83.0045
83.0063
82.9853
83.0004
83.0019
83.0218
83.0063
82.9871
83.0019
82.9974
83.0094
83.0094
82.9871
83.0004
83.0091
83.0396
83.0218
82.9997
83.0091
83.0107
83.0094
83.0045
82.9871
83.0004
83.0091
83.0063
83.0396
82.9871
83.0019
83.0091
83.0218
83.0063
82.9612
83.0107
82.9974
83.0045
83.0063
82.9848
83.0107
82.9974
83.0045
83.0094
82.9871
83.0091
83.0004
83.0063
83.0218
82.9997
83.0004
82.9974
83.0045
83.0218
82.9871
83.0107
83.0004
83.0218
83.0094
82.9997
82.9974
82.9974
83.0396
83.0045
82.9848
82.9974
83.0019
83.0396
83.0094
83.031
83.0019
83.0107
83.0063
83.0094
82.9848
83.0091
83.0004
83.0063
83.0094
82.9871
83.0107
83.0091
83.0218
83.0396
82.9848
83.0091
83.0107
83.0063
83.0094
82.9853
83.0107
83.0019
83.0045
83.0094
82.9848
83.0091
83.0091
83.0063
83.0094
82.9853
83.0004
83.0107
83.0094
83.0094
82.9848
82.9974
83.0091
83.0218
83.0218
82.9853
83.0091
83.0107
83.0094
83.0063
82.9848
83.0019
83.0091
83.0094
83.0045
82.9848
83.0107
83.0107
83.0396
83.0218
82.9848
83.0107
83.0091
83.0063
83.0045
82.9871
83.0019
82.9974
83.0063
83.0218
82.9848
83.0091
82.9974
83.0045
83.0045
82.9853
83.0107
83.0019
83.0396
83.0063
82.9853
83.0019
83.0004
83.0063
83.0094
83.031
83.0091
82.9974
83.0063
83.0396
82.9853
83.0091
83.0091
83.0063
83.0396
82.9853
83.0107
83.0091
83.0396
83.0396
82.9853
83.0004
83.0004
83.0045
83.0094
82.9997
83.0004
83.0107
83.0218
83.0218
83.0366
83.0019
83.0019
83.0045
83.0045
82.9997
83.0107
83.0107
83.0045
83.0063
83.0366
82.9974
83.0107
83.0218
83.0045
83.0366
83.0091
83.0091
83.0045
83.0045
82.9871
82.9974
83.0019
83.0045
83.0218
82.9848
83.0004
83.0019
83.0396
83.0045
83.0366
83.0004
83.0091
83.0396
83.0045
82.9871
82.9974
83.0004
83.0396
83.0063
82.9871
83.0107
83.0004
83.0045
83.0063
82.9871
83.0091
83.0004
83.0063
83.0094
82.9853
82.9974
83.0019
83.0045
83.0045
82.9848
83.0107
82.9974
83.0094
83.0396
82.9853
83.0107
83.0107
83.0045
83.0063
82.9853
82.9974
83.0107
83.0218
83.0045
82.9997
82.9974
83.0019
83.0063
83.0396
82.9848
83.0107
83.0019
83.0218
83.0063
82.9997
83.0091
83.0004
83.0045
83.0396
82.9997
83.0004
83.0019
83.0396
83.0045
82.9853
83.0107
83.0019
83.0045
83.0045
82.9871
83.0004
83.0091
83.0063
83.0396
82.9848
82.9974
83.0019
83.0045
83.0094
82.9997
83.0107
83.0004
83.0045
83.0045
82.9853
83.0107
83.0004
83.0396
83.0094
82.9871
83.0004
82.9974
83.0045
83.0063
82.9848
82.9974
83.0107
83.0218
83.0063
82.9871
83.0091
82.9974
83.0045
83.0396
82.9997
83.0107
83.0091
83.0063
83.0396
82.9853
83.0004
82.9974
83.0045
83.0218
82.9871
83.0091
82.9974
83.0063
83.0063
82.9853
83.0107
82.9974
83.0094
83.0396
83.0366
83.0004
83.0004
83.0094
83.0396
82.9848
83.0019
83.0107
83.0045
83.0094
82.9871
83.0019
83.0004
83.0218
83.0063
82.9871
83.0019
82.9974
83.0396
83.0045
83.0366
83.0019
82.9974
83.0063
83.0045
82.9871
83.0004
83.0019
83.0396
83.0396
82.9871
83.0004
82.9974
83.0045
83.0094
82.9853
82.9974
83.0019
83.0396
83.0094
82.9997
82.9974
83.0019
83.0094
83.0045
83.031
83.0004
83.0019
83.0045
83.0063
82.9848
83.0107
83.0107
83.0094
83.0218
82.9871
82.9974
83.0091
83.0094
83.0218
82.9848
83.0019
83.0091
83.0396
83.0045
83.0366
83.0091
83.0019
83.0045
83.0396
82.9997
82.9974
83.0019
83.0094
83.0094
82.9997
82.9974
83.0091
83.0218
83.0396
82.9853
83.0019
83.0019
83.0063
83.0396
82.9848
83.0004
83.0107
83.0045
83.0218
82.9853
83.0004
83.0107
83.0094
83.0045
82.9997
83.0107
83.0004
83.0063
83.0045
82.9853
83.0004
83.0107
83.0063
83.0045
82.9848
83.0091
82.9974
83.0396
83.0063
82.9853
82.9974
83.0091
83.0396
83.0218
82.9871
83.0107
83.0019
83.0218
83.0045
82.9853
83.0019
83.0107
83.0045
83.0063
82.9997
83.0004
83.0019
83.0396
83.0218
82.9853
83.0091
82.9974
83.0218
83.0094
82.9853
82.9974
83.0091
83.0218
83.0396
82.9997
83.0019
83.0091
83.0045
83.0045
82.9853
82.9974
83.0091
83.0396
83.0094
82.9997
83.0107
83.0004
83.0396
83.0218
82.9997
82.9974
83.0107
83.0045
83.0045
82.9848
83.0107
83.0107
83.0396
83.0396
82.9871
83.0019
82.9974
83.0396
83.0063
82.9853
82.9974
82.9974
83.0218
83.0094
82.9997
83.0107
83.0091
83.0218
83.0045
82.9848
82.9974
83.0107
83.0063
83.0045
82.9848
82.9974
82.9974
83.0063
83.0396
82.9853
83.0107
83.0019
83.0218
83.0045
82.9848
83.0107
83.0004
83.0094
83.0218
82.9871
83.0004
83.0107
83.0094
83.0063
82.9997
82.9974
83.0019
83.0396
83.0063
82.9848
83.0004
83.0107
83.0094
83.0063
82.9871
83.0019
83.0091
83.0218
83.0094
82.9853
83.0004
83.0091
83.0094
83.0218
82.9997
83.0091
83.0004
83.0396
83.0063
82.9997
83.0019
83.0004
83.0063
83.0218
83.0366
83.0091
83.0091
83.0063
83.0218
82.9853
83.0004
83.0091
83.0396
83.0045
82.9997
83.0004
83.0019
83.0396
83.0396
82.9848
83.0004
83.0004
83.0045
83.0045
82.9848
83.0004
82.9974
83.0218
83.0396
82.9871
83.0004
83.0019
83.0045
83.0063
82.9871
83.0019
83.0004
83.0045
83.0396
82.9871
83.0004
83.0004
83.0045
83.0063
82.9848
83.0107
82.9974
83.0063
83.0045
82.9997
83.0091
83.0004
83.0045
83.0396
82.9871
83.0019
83.0091
83.0396
83.0396
82.9853
82.9974
83.0091
83.0094
83.0396
82.9997
83.0019
83.0019
83.0045
83.0396
82.9997
82.9974
83.0091
83.0063
83.0218
82.9871
83.0004
83.0019
83.0063
83.0218
82.9853
83.0091
83.0004
83.0045
83.0045
82.9853
83.0019
83.0019
83.0396
83.0218
82.9848
83.0019
83.0004
83.0045
83.0063
82.9853
83.0091
82.9974
83.0396
83.0045
82.9848
83.0019
82.9974
83.0063
83.0218
82.9848
83.0004
83.0019
83.0218
83.0063
82.9848
83.0091
83.0019
83.0396
83.0218
83.0366
83.0004
83.0004
83.0218
83.0045
82.9848
83.0019
82.9974
83.0045
83.0218
82.9997
82.9974
82.9974
83.0218
83.0045
82.9848
83.0107
83.0019
83.0396
83.0094
82.9997
83.0107
83.0107
83.0094
83.0094
82.9853
83.0019
82.9974
83.0094
83.0045
82.9871
83.0091
83.0004
83.0094
83.0396
82.9871
82.9974
83.0019
83.0218
83.0045
82.9871
83.0004
83.0091
83.0094
83.0063
82.9853
83.0107
83.0091
83.0045
83.0396
82.9848
83.0004
83.0107
83.0063
83.0094
82.9871
83.0004
83.0019
83.0063
83.0063
82.9848
83.0019
83.0107
83.0045
83.0063
82.9871
83.0091
83.0019
83.0045
83.0218
82.9997
83.0019
82.9974
83.0396
83.0218
82.9853
82.9974
83.0091
83.0218
83.0218
82.9848
83.0107
82.9974
83.0218
83.0045
82.9871
83.0107
83.0107
83.0218
83.0094
82.9997
83.0019
83.0107
83.0063
83.0396
82.9871
83.0019
83.0091
83.0396
83.0396
82.9853
83.0107
83.0091
83.0063
83.0063
82.9853
83.0019
83.0107
83.0396
83.0396
82.9848
83.0019
82.9974
83.0396
83.0063
82.9871
83.0091
82.9974
83.0396
83.0045
82.9848
83.0107
83.0019
83.0063
83.0045
82.9848
83.0091
83.0091
83.0396
83.0063
82.9997
83.0091
83.0019
83.0045
83.0218
83.031
83.0004
83.0004
83.0396
83.0063
82.9871
83.0107
83.0091
83.0396
83.0094
82.9853
83.0091
83.0107
83.0396
83.0063
82.9871
83.0004
83.0004
83.0094
83.0045
82.9871
83.0019
83.0019
83.0063
83.0218
82.9871
82.9974
82.9974
83.0396
83.0063
82.9997
82.9974
83.0004
83.0094
83.0063
82.9871
83.0091
82.9974
83.0063
83.0094
82.9871
83.0019
82.9974
83.0094
83.0094
82.9871
83.0019
82.9974
83.0396
83.0045
82.9853
83.0107
83.0004
83.0396
83.0218
82.9848
83.0107
82.9974
83.0094
83.0218
82.9848
83.0091
83.0004
83.0094
83.0094
82.9853
82.9974
82.9974
83.0218
83.0045
82.9997
83.0091
83.0091
83.0396
83.0218
82.9871
82.9974
82.9974
83.0218
83.0396
82.9848
83.0107
83.0091
83.0045
83.0218
82.9848
83.0019
83.0091
83.0396
83.0045
82.9997
82.9974
83.0107
83.0218
83.0396
82.9871
83.0107
83.0019
83.0094
83.0218
82.9871
83.0107
83.0004
83.0045
83.0045
82.9871
83.0004
83.0004
83.0094
83.0094
82.9848
83.0107
83.0107
83.0045
83.0045
82.9848
82.9974
83.0004
83.0218
83.0063
82.9871
83.0019
83.0004
83.0396
83.0094
82.9871
83.0004
83.0091
83.0063
83.0094
82.9848
82.9974
83.0091
83.0218
83.0094
82.9848
83.0004
83.0091
83.0218
83.0045
82.9853
83.0019
83.0107
83.0218
83.0396
82.9853
82.9974
83.0107
83.0094
83.0396
82.9853
83.0091
83.0107
83.0094
83.0045
83.0366
82.9974
83.0019
83.0094
83.0063
82.9853
82.9974
82.9974
83.0396
83.0094
82.9848
82.9974
82.9974
83.0218
83.0218
82.9871
83.0004
83.0107
83.0218
83.0218
82.9848
83.0107
83.0019
83.0045
83.0063
82.9997
83.0107
82.9974
83.0218
83.0063
83.0366
83.0019
83.0019
83.0218
83.0045
82.9848
83.0004
83.0004
83.0063
83.0396
82.9853
83.0004
83.0091
83.0396
83.0063
82.9997
83.0107
83.0004
83.0045
83.0063
83.0366
82.9974
82.9974
83.0094
83.0218
82.9853
83.0107
82.9974
83.0063
83.0396
82.9871
83.0091
83.0091
83.0396
83.0094
82.9848
83.0019
83.0107
83.0218
83.0218
82.9871
83.0091
83.0107
83.0063
83.0045
82.9853
83.0091
83.0019
83.0218
83.0045
82.9853
83.0107
83.0019
83.0045
83.0094
82.9853
83.0019
83.0091
83.0045
83.0063
82.9853
83.0107
83.0019
83.0063
83.0218
82.9997
83.0004
83.0091
83.0218
83.0063
82.9848
83.0107
83.0091
83.0218
83.0218
82.9853
83.0004
83.0019
83.0396
83.0094
82.9871
83.0107
83.0004
83.0218
83.0396
82.9853
83.0091
83.0107
83.0063
83.0094
82.9871
82.9974
83.0091
83.0045
83.0094
82.9848
82.9974
83.0091
83.0218
83.0396
82.9853
83.0091
82.9974
83.0094
83.0045
83.0366
83.0004
83.0004
83.0094
83.0094
82.9853
83.0107
83.0091
83.0063
83.0094
83.0366
83.0019
82.9974
83.0063
83.0063
82.9871
83.0091
83.0107
83.0045
83.0094
82.9848
83.0107
83.0091
83.0063
83.0045
82.9848
83.0019
83.0107
83.0396
83.0396
82.9853
82.9974
83.0019
83.0063
83.0396
82.9997
83.0004
83.0019
83.0218
83.0396
82.9853
83.0107
83.0019
83.0396
83.0063
82.9997
83.0004
83.0019
83.0045
83.0094
82.9853
83.0107
82.9974
83.0094
83.0396
82.9848
82.9974
83.0019
83.0094
83.0094
82.9871
83.0107
83.0091
83.0063
83.0218
82.9997
83.0107
83.0107
83.0094
83.0094
82.9848
83.0091
83.0107
83.0094
83.0218
82.9848
82.9974
83.0107
83.0063
83.0218
82.9853
83.0091
83.0091
83.0218
83.0045
82.9848
83.0004
82.9974
83.0063
83.0063
82.9848
83.0019
83.0107
83.0094
83.0094
82.9848
83.0107
82.9974
83.0063
83.0094
82.9853
83.0091
82.9974
83.0218
83.0045
82.9853
83.0019
83.0107
83.0218
83.0218
82.9997
82.9974
83.0107
83.0063
83.0094
82.9853
83.0019
83.0019
83.0063
83.0396
82.9871
83.0004
83.0004
83.0063
83.0063
82.9853
83.0107
83.0107
83.0063
83.0045
82.9871
82.9974
83.0004
83.0045
83.0094
82.9871
83.0091
82.9974
83.0045
83.0063
82.9853
83.0091
83.0107
83.0063
83.0045
82.9871
83.0004
83.0091
83.0396
83.0218
83.0366
83.0091
82.9974
83.0063
83.0063
82.9871
82.9974
83.0107
83.0045
83.0063
82.9853
82.9974
83.0107
83.0218
83.0063
82.9997
83.0091
82.9974
83.0396
83.0396
82.9853
82.9974
83.0091
83.0063
83.0396
82.9997
83.0019
83.0107
83.0045
83.0218
82.9871
83.0004
83.0107
83.0396
83.0045
82.9848
83.0019
83.0004
83.0063
83.0063
82.9871
83.0091
83.0091
83.0063
83.0094
83.0366
83.0019
83.0004
83.0396
83.0063
82.9871
83.0091
82.9974
83.0094
83.0396
82.9853
83.0091
83.0091
83.0396
83.0396
82.9848
83.0004
83.0004
83.0094
83.0094
82.9871
83.0107
83.0091
83.0063
83.0045
82.9848
83.0107
83.0091
83.0045
83.0063
82.9997
83.0091
82.9974
83.0094
83.0094
82.9997
83.0107
83.0107
83.0218
83.0094
82.9848
83.0091
82.9974
83.0396
83.0063
82.9997
83.0019
83.0107
83.0396
83.0063
82.9871
82.9974
83.0107
83.0045
83.0045
82.9853
83.0004
83.0091
83.0094
83.0396
82.9853
83.0107
82.9974
83.0045
83.0045
82.9997
82.9974
83.0004
83.0218
83.0063
82.9997
83.0091
82.9974
83.0094
83.0396
83.0366
83.0004
83.0004
83.0094
83.0094
82.9871
83.0107
83.0107
83.0094
83.0045
82.9871
83.0091
83.0004
83.0094
83.0396
82.9871
83.0091
82.9974
83.0063
83.0045
82.9853
82.9974
83.0019
83.0218
83.0396
82.9848
83.0107
83.0019
83.0218
83.0218
82.9871
83.0107
83.0091
83.0094
83.0094
82.9997
82.9974
83.0107
83.0045
83.0218
82.9848
83.0019
83.0091
83.0218
83.0094
82.9871
83.0107
83.0091
83.0218
83.0045
83.0366
82.9974
82.9974
83.0063
83.0396
82.9848
83.0091
83.0107
83.0218
83.0063
82.9848
82.9974
83.0004
83.0094
83.0094
82.9997
83.0004
83.0091
83.0045
83.0063
82.9848
83.0091
83.0107
83.0094
83.0094
82.9997
82.9974
83.0019
83.0094
83.0218
82.9997
83.0019
83.0019
83.0218
83.0045
82.9997
82.9974
83.0019
83.0218
83.0396
82.9848
83.0004
83.0019
83.0045
83.0094
82.9997
83.0019
83.0004
83.0063
83.0094
83.0366
83.0091
83.0004
83.0045
83.0063
82.9848
83.0091
83.0019
83.0063
83.0063
82.9871
83.0019
83.0091
83.0218
83.0218
82.9853
83.0004
83.0019
83.0063
83.0045
82.9848
83.0091
82.9974
83.0094
83.0045
82.9871
83.0107
83.0107
83.0094
83.0063
82.9848
83.0091
83.0091
83.0094
83.0218
82.9848
83.0004
83.0004
83.0063
83.0094
82.9853
82.9974
83.0019
83.0063
83.0063
82.9871
83.0004
83.0019
83.0063
83.0094
82.9848
83.0091
83.0091
83.0045
83.0045
82.9848
83.0107
83.0004
83.0218
83.0063
82.9853
83.0107
82.9974
83.0218
83.0094
82.9848
83.0107
83.0019
83.0218
83.0218
82.9871
83.0091
83.0107
83.0094
83.0396
82.9871
83.0091
83.0004
83.0063
83.0094
82.9853
83.0019
83.0004
83.0045
83.0094
82.9871
83.0091
83.0091
83.0094
83.0218
82.9853
83.0091
83.0091
83.0094
83.0218
83.0366
83.0004
83.0091
83.0218
83.0045
82.9848
82.9974
83.0091
83.0045
83.0063
83.031
83.0107
83.0091
83.0396
83.0045
82.9848
83.0019
83.0004
83.0218
83.0063
82.9848
83.0107
83.0019
83.0094
83.0218
82.9848
82.9974
83.0004
83.0094
83.0094
82.9997
83.0091
82.9974
83.0063
83.0396
82.9853
83.0019
83.0019
83.0045
83.0094
83.0366
82.9974
83.0091
83.0063
83.0396
82.9997
82.9974
82.9974
83.0094
83.0094
83.0366
83.0107
83.0019
83.0094
83.0045
82.9871
83.0019
82.9974
83.0045
83.0396
82.9853
83.0019
83.0107
83.0063
83.0094
82.9848
83.0107
82.9974
83.0063
83.0094
82.9853
83.0019
82.9974
83.0218
83.0094
82.9997
83.0019
83.0107
83.0218
83.0094
82.9871
83.0091
83.0004
83.0218
83.0094
82.9997
83.0091
83.0019
83.0396
83.0094
82.9853
83.0091
83.0091
83.0094
83.0063
82.9871
83.0107
83.0091
83.0396
83.0045
83.0366
83.0091
83.0107
83.0063
83.0396
82.9848
83.0107
82.9974
83.0063
83.0063
82.9853
83.0004
83.0091
83.0396
83.0045
82.9848
83.0004
83.0019
83.0396
83.0396
82.9871
83.0107
83.0019
83.0218
83.0063
82.9871
83.0107
83.0091
83.0218
83.0063
82.9871
83.0019
83.0091
83.0045
83.0045
82.9997
83.0091
83.0091
83.0396
83.0218
82.9853
83.0091
83.0019
83.0063
83.0045
82.9871
83.0004
83.0019
83.0094
83.0094
82.9848
83.0091
82.9974
83.0396
83.0396
82.9997
83.0107
83.0107
83.0045
83.0094
82.9871
83.0091
83.0019
83.0218
83.0063
83.0366
83.0004
83.0004
83.0045
83.0063
82.9871
83.0107
82.9974
83.0218
83.0094
82.9848
83.0091
82.9974
83.0063
83.0045
82.9848
83.0107
83.0004
83.0396
83.0094
82.9848
83.0004
83.0091
83.0218
83.0063
82.9871
83.0107
83.0019
83.0396
83.0045
82.9848
83.0091
83.0091
83.0063
83.0094
83.0366
83.0091
83.0107
83.0045
83.0218
82.9853
82.9974
83.0107
83.0094
83.0218
82.9853
82.9974
83.0091
83.0063
83.0063
82.9997
83.0019
82.9974
83.0396
83.0045
82.9848
83.0004
83.0019
83.0396
83.0063
83.031
83.0019
83.0004
83.0063
83.0218
82.9853
83.0019
82.9974
83.0396
83.0396
82.9997
83.0091
83.0019
83.0218
83.0094
82.9997
83.0004
82.9974
83.0094
83.0218
82.9997
83.0091
83.0019
83.0045
83.0396
82.9853
83.0004
82.9974
83.0396
83.0063
83.0366
83.0107
82.9974
83.0218
83.0045
82.9853
83.0004
83.0107
83.0045
83.0218
82.9848
83.0004
83.0004
83.0218
83.0094
82.9871
82.9974
83.0091
83.0396
83.0045
82.9848
83.0091
82.9974
83.0045
83.0094
82.9871
83.0004
82.9974
83.0396
83.0094
82.9848
83.0107
83.0004
83.0396
83.0063
82.9871
83.0107
83.0004
83.0045
83.0218
82.9853
83.0107
83.0107
83.0045
83.0045
82.9997
83.0107
83.0091
83.0218
83.0063
82.9848
83.0107
83.0091
83.0218
83.0063
82.9848
82.9974
83.0004
83.0063
83.0045
82.9871
83.0004
83.0107
83.0045
83.0396
82.9853
83.0019
83.0091
83.0218
83.0396
82.9871
83.0019
83.0107
83.0045
83.0045
82.9853
83.0107
82.9974
83.0063
83.0063
82.9848
83.0107
82.9974
83.0063
83.0396
82.9871
83.0019
83.0004
83.0063
83.0045
83.0366
83.0091
83.0004
83.0218
83.0396
82.9853
82.9974
83.0004
83.0063
83.0045
82.9997
83.0107
83.0107
83.0396
83.0396
82.9997
82.9974
83.0004
83.0396
83.0396
82.9997
83.0019
83.0004
83.0094
83.0045
82.9848
83.0091
83.0107
83.0045
83.0045
82.9848
83.0107
83.0107
83.0094
83.0063
82.9853
83.0107
82.9974
83.0218
83.0063
82.9848
83.0019
83.0019
83.0045
83.0045
82.9997
82.9974
82.9974
83.0045
83.0045
82.9997
83.0004
83.0019
83.0063
83.0045
82.9853
82.9974
83.0107
83.0063
83.0218
82.9871
83.0019
83.0091
83.0094
83.0396
82.9871
83.0091
83.0107
83.0063
83.0094
82.9853
82.9974
83.0107
83.0396
83.0218
82.9853
82.9974
83.0091
83.0063
83.0063
82.9848
83.0091
83.0019
83.0218
83.0218
82.9871
83.0091
82.9974
83.0396
83.0063
82.9853
83.0107
83.0107
83.0218
83.0396
82.9997
83.0091
83.0107
83.0396
83.0218
82.9848
83.0019
83.0004
83.0218
83.0045
82.9853
83.0004
83.0091
83.0094
83.0094
82.9997
83.0107
83.0004
83.0094
83.0063
83.0366
83.0019
82.9974
83.0063
83.0045
82.9848
83.0019
82.9974
83.0094
83.0396
82.9848
82.9974
82.9974
83.0094
83.0094
82.9871
83.0019
83.0019
83.0063
83.0396
82.9997
83.0091
83.0004
83.0045
83.0094
82.9848
83.0107
83.0091
83.0045
83.0063
82.9997
83.0004
83.0004
83.0063
83.0063
82.9853
82.9974
82.9974
83.0094
83.0045
82.9997
83.0091
83.0091
83.0045
83.0094
82.9997
83.0019
83.0004
83.0396
83.0094
82.9848
83.0019
83.0107
83.0045
83.0218
82.9853
83.0004
83.0004
83.0063
83.0396
82.9997
82.9974
83.0004
83.0063
83.0218
82.9871
83.0004
83.0019
83.0063
83.0396
82.9853
82.9974
83.0107
83.0396
83.0218
82.9848
83.0107
82.9974
83.0396
83.0045
83.0366
83.0091
83.0091
83.0218
83.0094
82.9853
83.0091
83.0107
83.0045
83.0218
82.9997
83.0019
83.0004
83.0094
83.0396
82.9848
82.9974
83.0107
83.0063
83.0094
82.9848
82.9974
83.0091
83.0218
83.0094
82.9848
82.9974
83.0019
83.0063
83.0045
82.9871
82.9974
83.0019
83.0396
83.0094
82.9997
83.0091
83.0107
83.0094
83.0063
82.9997
83.0019
83.0091
83.0063
83.0396
82.9848
83.0107
82.9974
83.0094
83.0063
82.9997
83.0004
82.9974
83.0063
83.0063
82.9871
83.0019
83.0019
83.0396
83.0396
82.9871
83.0004
83.0091
83.0045
83.0396
82.9848
83.0019
83.0019
83.0396
83.0396
82.9848
83.0091
83.0019
83.0094
83.0218
82.9853
83.0019
83.0004
83.0218
83.0045
83.0366
83.0107
83.0004
83.0396
83.0045
82.9853
83.0019
82.9974
83.0218
83.0218
82.9997
83.0004
83.0019
83.0045
83.0396
82.9997
83.0107
83.0091
83.0045
83.0218
82.9848
82.9974
83.0004
83.0045
83.0218
82.9853
83.0019
83.0107
83.0094
83.0045
82.9848
83.0004
82.9974
83.0045
83.0063
82.9853
83.0004
83.0091
83.0094
83.0045
83.0366
83.0004
83.0107
83.0045
83.0094
82.9997
83.0091
83.0091
83.0396
83.0396
82.9871
83.0004
82.9974
83.0045
83.0045
82.9997
83.0004
83.0091
83.0218
83.0094
82.9853
82.9974
83.0091
83.0094
83.0063
82.9871
82.9974
83.0004
83.0396
83.0045
82.9853
83.0019
83.0107
83.0218
83.0218
82.9853
83.0019
83.0107
83.0063
83.0094
82.9871
83.0019
83.0019
83.0218
83.0218
82.9997
82.9974
82.9974
83.0045
83.0094
82.9853
83.0107
82.9974
83.0045
83.0218
82.9853
83.0091
83.0091
83.0094
83.0396
82.9848
82.9974
83.0107
83.0396
83.0094
82.9848
83.0004
83.0019
83.0094
83.0063
82.9848
83.0091
83.0091
83.0396
83.0045
82.9853
83.0019
82.9974
83.0218
83.0045
83.0366
83.0091
83.0107
83.0218
83.0218
82.9848
83.0004
83.0091
83.0063
83.0094
82.9997
83.0091
83.0019
83.0063
83.0063
82.9848
83.0107
83.0091
83.0063
83.0094
82.9853
83.0091
83.0019
83.0063
83.0063
83.0366
82.9974
83.0107
83.0396
83.0063
82.9871
83.0107
83.0019
83.0094
83.0218
83.0366
83.0019
83.0107
83.0063
83.0045
82.9853
83.0019
82.9974
83.0396
83.0045
82.9853
83.0004
82.9974
83.0045
83.0396
82.9997
83.0091
83.0004
83.0396
83.0396
82.9997
83.0091
83.0091
83.0094
83.0218
82.9997
83.0004
83.0107
83.0396
83.0094
82.9997
83.0004
83.0107
83.0094
83.0396
82.9997
83.0019
83.0019
83.0045
83.0094
82.9853
83.0004
82.9974
83.0094
83.0396
82.9997
82.9974
83.0107
83.0045
83.0045
82.9848
83.0004
83.0091
83.0094
83.0218
82.9871
83.0019
83.0004
83.0045
83.0045
82.9871
83.0091
83.0091
83.0396
83.0218
82.9997
83.0091
82.9974
83.0094
83.0045
82.9848
83.0004
82.9974
83.0063
83.0045
82.9997
83.0091
83.0019
83.0218
83.0396
82.9997
83.0107
83.0091
83.0045
83.0396
82.9997
82.9974
83.0004
83.0094
83.0045
83.0366
83.0091
83.0107
83.0396
83.0063
83.0366
83.0004
83.0091
83.0218
83.0045
82.9997
83.0019
83.0091
83.0396
83.0094
82.9871
82.9974
83.0019
83.0396
83.0063
82.9997
83.0019
83.0091
83.0396
83.0218
82.9853
83.0019
82.9974
83.0396
83.0396
82.9997
82.9974
83.0091
83.0063
83.0045
82.9997
82.9974
83.0091
83.0063
83.0094
82.9871
83.0107
83.0091
83.0094
83.0396
82.9997
82.9974
83.0019
83.0396
83.0094
82.9871
83.0019
83.0004
83.0094
83.0094
82.9853
83.0091
83.0019
83.0218
83.0045
82.9853
82.9974
83.0004
83.0218
83.0045
82.9997
83.0091
82.9974
83.0218
83.0063
82.9848
83.0004
82.9974
83.0094
83.0063
82.9871
82.9974
83.0019
83.0218
83.0218
82.9871
83.0004
83.0107
83.0045
83.0045
82.9853
83.0004
83.0091
83.0063
83.0218
82.9853
83.0107
83.0019
83.0063
83.0094
82.9853
82.9974
83.0107
83.0396
83.0063
82.9853
83.0107
82.9974
83.0063
83.0063
82.9853
82.9974
83.0107
83.0218
83.0094
82.9848
82.9974
83.0091
83.0094
83.0218
82.9871
83.0004
83.0019
83.0063
83.0396
82.9853
83.0004
82.9974
83.0063
83.0396
82.9853
83.0004
83.0091
83.0218
83.0396
82.9871
83.0019
83.0004
83.0063
83.0063
82.9997
83.0107
83.0091
83.0063
83.0094
82.9871
82.9974
83.0091
83.0094
83.0045
82.9871
83.0091
83.0004
83.0218
83.0094
82.9871
83.0107
82.9974
83.0396
83.0094
82.9997
82.9974
83.0091
83.0045
83.0396
82.9853
83.0091
83.0107
83.0063
83.0045
82.9997
83.0004
83.0091
83.0063
83.0094
82.9997
83.0091
83.0019
83.0396
83.0063
82.9997
83.0004
82.9974
83.0094
83.0045
82.9871
83.0019
83.0107
83.0063
83.0063
82.9848
83.0019
83.0019
83.0396
83.0045
82.9848
82.9974
82.9974
83.0094
83.0218
82.9997
83.0107
83.0019
83.0396
83.0063
82.9997
83.0107
83.0019
83.0396
83.0396
82.9853
83.0004
83.0091
83.0094
83.0218
82.9997
83.0091
83.0004
83.0063
83.0218
83.0366
82.9974
83.0019
83.0063
83.0218
82.9871
83.0091
82.9974
83.0045
83.0396
82.9848
83.0107
83.0091
83.0063
83.0094
82.9997
82.9974
83.0004
83.0063
83.0063
82.9853
83.0004
83.0019
83.0094
83.0396
82.9997
83.0004
83.0107
83.0396
83.0094
82.9997
83.0091
83.0091
83.0218
83.0396
82.9853
82.9974
83.0091
83.0218
83.0094
82.9997
83.0004
83.0107
83.0063
83.0218
82.9848
83.0004
82.9974
83.0063
83.0045
82.9853
83.0004
83.0091
83.0218
83.0218
82.9848
82.9974
83.0107
83.0045
83.0094
82.9997
83.0004
83.0107
83.0045
83.0045
82.9871
83.0019
83.0091
83.0218
83.0063
82.9848
83.0019
83.0091
83.0063
83.0094
82.9848
83.0091
83.0019
83.0218
83.0063
82.9871
83.0019
82.9974
83.0396
83.0063
82.9871
82.9974
83.0019
83.0094
83.0396
82.9848
83.0019
83.0107
83.0396
83.0045
82.9871
83.0004
83.0091
83.0218
83.0396
82.9997
82.9974
82.9974
83.0218
83.0396
82.9871
83.0107
82.9974
83.0094
83.0396
82.9848
83.0107
83.0107
83.0218
83.0396
82.9997
83.0091
82.9974
83.0063
83.0396
82.9871
83.0004
83.0107
83.0396
83.0063
82.9997
83.0107
83.0107
83.0218
83.0045
83.0366
83.0107
83.0107
83.0396
83.0063
82.9848
83.0019
82.9974
83.0094
83.0045
82.9853
82.9974
83.0019
83.0045
83.0218
82.9853
83.0107
83.0019
83.0094
83.0396
82.9997
82.9974
83.0107
83.0218
83.0218
82.9848
83.0019
83.0091
83.0094
83.0218
82.9848
83.0107
83.0004
83.0094
83.0045
82.9871
83.0091
82.9974
83.0218
83.0218
82.9871
83.0019
83.0004
83.0045
83.0045
82.9871
83.0004
83.0004
83.0045
83.0063
82.9871
83.0004
83.0107
83.0396
83.0396
82.9853
83.0004
83.0107
83.0396
83.0218
82.9848
83.0019
83.0091
83.0218
83.0063
82.9871
83.0019
83.0019
83.0396
83.0094
82.9997
82.9974
83.0107
83.0396
83.0094
82.9853
82.9974
83.0004
83.0063
83.0396
82.9853
83.0091
82.9974
83.0218
83.0063
82.9848
83.0107
82.9974
83.0045
83.0218
82.9853
83.0004
83.0019
83.0396
83.0396
82.9871
83.0091
83.0107
83.0063
83.0063
82.9848
83.0107
83.0004
83.0094
83.0094
82.9853
82.9974
83.0091
83.0396
83.0396
82.9997
82.9974
82.9974
83.0063
83.0063
82.9853
83.0107
83.0091
83.0094
83.0063
82.9871
82.9974
82.9974
83.0396
83.0094
82.9848
83.0019
83.0019
83.0063
83.0045
82.9853
82.9974
83.0091
83.0094
83.0045
82.9853
83.0091
83.0107
83.0063
83.0045
82.9853
82.9974
83.0091
83.0218
83.0396
82.9997
83.0107
83.0091
83.0045
83.0063
82.9848
83.0019
83.0091
83.0218
83.0094
82.9871
83.0019
83.0091
83.0045
83.0396
82.9997
83.0019
83.0107
83.0045
83.0063
82.9848
83.0091
83.0019
83.0218
83.0218
82.9871
82.9974
83.0019
83.0396
83.0063
82.9853
83.0004
83.0004
83.0045
83.0094
82.9853
83.0004
83.0019
83.0045
83.0396
83.0366
83.0004
83.0107
83.0396
83.0045
82.9871
83.0107
83.0091
83.0094
83.0045
82.9871
83.0091
83.0091
83.0045
83.0218
82.9871
83.0091
82.9974
83.0218
83.0218
82.9871
82.9974
82.9974
83.0063
83.0063
82.9871
83.0107
83.0091
83.0045
83.0094
82.9997
82.9974
83.0004
83.0218
83.0218
82.9848
83.0091
83.0004
83.0094
83.0218
82.9848
83.0107
82.9974
83.0063
83.0396
82.9848
83.0107
83.0091
83.0218
83.0045
82.9848
83.0004
83.0091
83.0063
83.0045
83.0366
83.0107
83.0107
83.0094
83.0396
82.9997
83.0004
82.9974
83.0094
83.0063
82.9871
83.0107
83.0019
83.0063
83.0396
82.9853
83.0019
83.0004
83.0396
83.0094
82.9853
83.0004
83.0019
83.0063
83.0063
82.9848
82.9974
83.0004
83.0218
83.0094
82.9997
83.0091
83.0004
83.0094
83.0063
82.9853
83.0019
83.0091
83.0396
83.0045
82.9871
83.0107
82.9974
83.0396
83.0045
82.9871
83.0019
83.0091
83.0396
83.0396
83.0366
83.0019
82.9974
83.0396
83.0063
82.9871
82.9974
83.0107
83.0396
83.0063
82.9871
83.0091
83.0004
83.0218
83.0396
82.9853
82.9974
83.0107
83.0045
83.0218
82.9997
82.9974
83.0004
83.0396
83.0218
82.9848
82.9974
82.9974
83.0045
83.0063
82.9848
83.0091
83.0107
83.0045
83.0396
82.9853
83.0107
83.0107
83.0218
83.0063
82.9848
83.0107
83.0091
83.0045
83.0218
82.9853
83.0019
83.0019
83.0045
83.0218
82.9997
83.0019
83.0107
83.0094
83.0045
82.9853
83.0091
83.0091
83.0396
83.0094
82.9848
83.0091
83.0107
83.0063
83.0063
82.9853
83.0091
82.9974
83.0045
83.0045
83.0366
82.9974
82.9974
83.0063
83.0063
83.0366
83.0107
82.9974
83.0218
83.0094
82.9853
83.0091
83.0107
83.0218
83.0094
82.9848
83.0091
83.0004
83.0094
83.0218
82.9997
83.0019
83.0019
83.0396
83.0396
82.9848
83.0091
83.0019
83.0063
83.0045
82.9871
83.0004
83.0004
83.0094
83.0396
82.9853
83.0107
83.0004
83.0045
83.0063
82.9853
83.0019
82.9974
83.0063
83.0094
82.9853
82.9974
83.0004
83.0063
83.0063
82.9848
82.9974
83.0107
83.0094
83.0045
82.9871
83.0004
83.0019
83.0218
83.0045
82.9871
82.9974
83.0091
83.0045
83.0396
82.9848
83.0107
83.0091
83.0218
83.0218
82.9871
83.0107
83.0107
83.0218
83.0218
82.9997
83.0091
83.0004
83.0094
83.0094
82.9848
83.0019
82.9974
83.0045
83.0045
82.9848
83.0107
83.0107
83.0218
83.0045
82.9848
82.9974
83.0019
83.0218
83.0396
82.9997
83.0004
83.0107
83.0218
83.0094
82.9871
83.0019
83.0091
83.0218
83.0063
82.9871
83.0091
82.9974
83.0045
83.0063
82.9848
83.0019
82.9974
83.0094
83.0094
82.9848
82.9974
83.0019
83.0396
83.0218
82.9871
82.9974
83.0004
83.0045
83.0045
83.0366
83.0004
83.0019
83.0094
83.0218
82.9871
83.0107
83.0019
83.0218
83.0094
82.9871
83.0107
83.0107
83.0396
83.0218
82.9871
83.0107
82.9974
83.0063
83.0045
82.9848
83.0107
82.9974
83.0094
83.0063
82.9997
83.0107
83.0091
83.0063
83.0094
82.9848
82.9974
83.0019
83.0094
83.0396
82.9853
82.9974
83.0004
83.0045
83.0218
83.0366
83.0107
83.0004
83.0396
83.0045
82.9848
83.0091
82.9974
83.0218
83.0094
82.9997
83.0107
83.0019
83.0045
83.0396
82.9997
82.9974
83.0091
83.0094
83.0094
82.9997
83.0091
83.0004
83.0063
83.0396
82.9997
83.0004
83.0091
83.0094
83.0045
82.9853
83.0107
83.0019
83.0063
83.0045
82.9848
83.0107
83.0091
83.0396
83.0045
82.9853
83.0107
83.0107
83.0063
83.0063
82.9848
83.0019
83.0019
83.0094
83.0218
82.9848
83.0091
83.0004
83.0094
83.0094
82.9997
83.0091
82.9974
83.0218
83.0396
82.9848
83.0019
83.0091
83.0218
83.0045
82.9848
83.0091
83.0004
83.0218
83.0094
82.9853
83.0019
83.0004
83.0094
83.0396
82.9871
83.0019
83.0107
83.0063
83.0094
82.9997
83.0004
83.0004
83.0063
83.0063
82.9848
83.0004
83.0091
83.0094
83.0094
82.9848
83.0107
83.0019
83.0218
83.0045
83.031
83.0107
83.0091
83.0218
83.0063
82.9848
83.0107
83.0091
83.0063
83.0063
82.9853
83.0019
83.0004
83.0396
83.0063
82.9848
83.0019
83.0019
83.0063
83.0045
82.9871
83.0004
83.0019
83.0218
83.0396
82.9997
83.0091
83.0091
83.0063
83.0396
82.9997
83.0107
83.0091
83.0218
83.0045
82.9871
83.0091
83.0091
83.0396
83.0063
82.9871
83.0107
83.0091
83.0063
83.0063
82.9871
82.9974
83.0091
83.0094
83.0094
82.9848
83.0091
83.0004
83.0094
83.0396
82.9848
83.0004
83.0091
83.0094
83.0218
82.9853
83.0107
83.0004
83.0045
83.0063
82.9871
82.9974
83.0019
83.0218
83.0094
83.0366
83.0019
83.0019
83.0063
83.0045
82.9871
83.0107
83.0091
83.0218
83.0094
83.0366
83.0107
83.0019
83.0063
83.0218
82.9853
82.9974
83.0019
83.0063
83.0396
82.9997
83.0004
83.0107
83.0396
83.0094
82.9997
83.0107
83.0107
83.0218
83.0218
82.9871
82.9974
82.9974
83.0063
83.0218
82.9853
83.0107
83.0019
83.0063
83.0218
82.9871
82.9974
83.0019
83.0094
83.0045
82.9871
83.0091
83.0107
83.0094
83.0045
82.9997
83.0019
83.0091
83.0094
83.0094
82.9997
83.0004
83.0091
83.0396
83.0045
82.9997
83.0091
82.9974
83.0218
83.0396
82.9871
82.9974
83.0107
83.0063
83.0218
83.0366
83.0004
83.0019
83.0218
83.0063
82.9853
82.9974
82.9974
83.0045
83.0218
82.9871
83.0091
83.0091
83.0045
83.0218
82.9853
83.0091
82.9974
83.0396
83.0045
83.0366
83.0004
83.0091
83.0045
83.0045
83.0366
82.9974
83.0091
83.0063
83.0396
82.9853
83.0091
82.9974
83.0396
83.0063
82.9997
83.0091
82.9974
83.0396
83.0063
82.9871
83.0107
82.9974
83.0094
83.0396
82.9871
83.0019
83.0107
83.0094
83.0218
82.9848
83.0107
83.0091
83.0045
83.0218
83.031
83.0004
83.0107
83.0094
83.0045
82.9871
82.9974
83.0107
83.0396
83.0094
82.9848
83.0019
83.0091
83.0063
83.0045
82.9848
83.0004
83.0004
83.0045
83.0218
82.9871
83.0091
83.0091
83.0063
83.0063
82.9848
83.0091
83.0004
83.0045
83.0063
82.9997
82.9974
83.0091
83.0063
83.0094
82.9848
82.9974
83.0107
83.0063
83.0218
82.9853
82.9974
82.9974
83.0063
83.0063
82.9871
82.9974
83.0107
83.0045
83.0063
82.9853
83.0091
82.9974
83.0063
83.0063
82.9853
83.0107
83.0019
83.0063
83.0045
82.9871
83.0091
83.0004
83.0218
83.0045
82.9848
83.0004
83.0004
83.0094
83.0063
82.9997
83.0004
83.0107
83.0218
83.0396
82.9997
83.0107
82.9974
83.0094
83.0218
82.9997
83.0019
83.0004
83.0218
83.0396
82.9871
83.0004
82.9974
83.0094
83.0045
82.9853
82.9974
83.0107
83.0045
83.0063
82.9848
83.0019
82.9974
83.0094
83.0218
82.9848
83.0004
83.0107
83.0218
83.0094
82.9997
83.0091
83.0004
83.0063
83.0094
82.9853
82.9974
83.0004
83.0045
83.0063
82.9871
82.9974
83.0004
83.0218
83.0396
82.9853
83.0091
83.0004
83.0063
83.0396
82.9871
83.0107
83.0019
83.0045
83.0396
82.9853
83.0091
83.0004
83.0218
83.0094
82.9848
83.0107
83.0004
83.0063
83.0063
82.9612
83.0091
83.0107
83.0045
83.0094
82.9871
83.0107
83.0019
83.0396
83.0218
82.9853
83.0107
82.9974
83.0396
83.0045
82.9997
82.9974
83.0091
83.0063
83.0094
82.9848
83.0091
83.0107
83.0045
83.0218
82.9853
83.0107
83.0091
83.0045
83.0094
82.9997
83.0091
82.9974
83.0094
83.0218
82.9997
83.0004
83.0107
83.0094
83.0094
82.9997
83.0107
83.0019
83.0063
83.0094
82.9997
83.0091
83.0107
83.0218
83.0094
82.9871
83.0004
83.0019
83.0094
83.0218
82.9853
82.9974
83.0107
83.0094
83.0396
82.9848
83.0091
83.0019
83.0045
83.0094
82.9871
82.9974
83.0091
83.0218
83.0094
83.0366
83.0091
83.0107
83.0396
83.0094
82.9997
83.0019
83.0019
83.0094
83.0063
82.9871
83.0019
83.0004
83.0094
83.0094
82.9871
83.0107
83.0091
83.0218
83.0218
82.9848
82.9974
83.0019
83.0063
83.0218
82.9997
83.0019
83.0091
83.0045
83.0063
82.9848
83.0019
83.0004
83.0094
83.0045
82.9871
83.0107
83.0004
83.0094
83.0045
82.9848
83.0004
83.0107
83.0218
83.0063
82.9871
83.0091
83.0107
83.0396
83.0094
82.9997
83.0004
83.0004
83.0063
83.0218
82.9871
83.0107
83.0019
83.0045
83.0396
82.9871
83.0019
83.0107
83.0094
83.0063
82.9871
83.0107
83.0019
83.0094
83.0045
82.9848
82.9974
82.9974
83.0094
83.0396
82.9871
83.0091
83.0107
83.0063
83.0396
82.9848
83.0107
83.0019
83.0218
83.0396
82.9871
82.9974
83.0107
83.0094
83.0094
82.9997
83.0091
83.0019
83.0045
83.0094
82.9997
83.0004
83.0107
83.0063
83.0094
82.9871
83.0004
83.0004
83.0045
83.0094
82.9871
83.0019
83.0019
83.0396
83.0218
82.9853
83.0091
83.0004
83.0045
83.0396
82.9848
82.9974
83.0004
83.0094
83.0045
82.9848
83.0091
83.0091
83.0045
83.0396
82.9871
82.9974
82.9974
83.0218
83.0045
82.9997
82.9974
82.9974
83.0045
83.0218
82.9848
83.0004
83.0004
83.0045
83.0218
82.9848
83.0004
83.0004
83.0063
83.0045
82.9997
83.0019
83.0091
83.0094
83.0396
82.9848
83.0004
83.0019
83.0218
83.0045
82.9871
83.0004
83.0004
83.0045
83.0094
82.9848
83.0107
82.9974
83.0045
83.0045
82.9848
83.0107
83.0004
83.0063
83.0094
82.9853
82.9974
82.9974
83.0045
83.0094
82.9871
82.9974
83.0019
83.0218
83.0045
82.9848
83.0091
83.0019
83.0218
83.0063
82.9997
83.0091
83.0004
83.0045
83.0396
82.9853
83.0019
83.0107
83.0094
83.0218
82.9853
83.0107
83.0091
83.0045
83.0063
82.9853
83.0107
83.0004
83.0063
83.0218
82.9997
83.0107
82.9974
83.0396
83.0045
82.9853
83.0107
83.0107
83.0218
83.0094
82.9848
83.0004
83.0004
83.0218
83.0045
82.9848
83.0004
83.0107
83.0045
83.0218
82.9848
83.0091
83.0004
83.0094
83.0045
82.9871
83.0091
82.9974
83.0094
83.0094
82.9997
83.0019
83.0019
83.0094
83.0063
82.9853
83.0004
82.9974
83.0045
83.0045
82.9853
82.9974
82.9974
83.0094
83.0218
82.9997
83.0107
83.0107
83.0063
83.0218
82.9853
83.0004
83.0019
83.0094
83.0396
82.9871
83.0019
82.9974
83.0094
83.0218
82.9848
83.0107
83.0107
83.0396
83.0218
82.9871
83.0107
83.0107
83.0218
83.0045
82.9871
83.0019
82.9974
83.0094
83.0045
82.9871
82.9974
83.0019
83.0396
83.0094
82.9853
83.0004
83.0004
83.0396
83.0045
82.9997
83.0091
83.0004
83.0063
83.0396
82.9853
83.0107
83.0004
83.0063
83.0063
82.9997
83.0107
83.0019
83.0218
83.0045
82.9848
83.0004
83.0019
83.0396
83.0218
82.9848
82.9974
83.0019
83.0063
83.0218
82.9853
82.9974
82.9974
83.0045
83.0094
82.9871
83.0107
82.9974
83.0063
83.0396
83.0366
83.0019
83.0019
83.0063
83.0063
82.9853
83.0107
83.0019
83.0045
83.0045
82.9997
82.9974
82.9974
83.0396
83.0063
82.9853
83.0004
83.0107
83.0218
83.0045
82.9871
82.9974
82.9974
83.0396
83.0045
82.9997
83.0004
83.0019
83.0094
83.0094
82.9853
83.0004
83.0107
83.0218
83.0094
82.9997
83.0091
83.0004
83.0063
83.0063
82.9997
83.0091
83.0019
83.0396
83.0045
82.9997
83.0019
83.0091
83.0218
83.0094
82.9871
83.0004
83.0091
83.0094
83.0396
82.9871
83.0107
83.0019
83.0094
83.0396
82.9848
83.0004
83.0004
83.0218
83.0218
82.9871
83.0004
82.9974
83.0396
83.0396
82.9871
83.0019
83.0019
83.0094
83.0094
82.9871
83.0107
83.0107
83.0218
83.0218
82.9853
83.0019
83.0019
83.0094
83.0094
82.9848
83.0091
82.9974
83.0063
83.0396
82.9871
83.0019
82.9974
83.0045
83.0094
83.0366
82.9974
83.0019
83.0045
83.0063
82.9871
83.0091
83.0004
83.0094
83.0094
82.9848
83.0107
83.0107
83.0045
83.0063
82.9871
83.0091
83.0004
83.0045
83.0218
82.9997
83.0004
83.0091
83.0045
83.0396
82.9997
83.0019
83.0107
83.0396
83.0045
82.9853
82.9974
83.0107
83.0094
83.0094
82.9853
83.0004
82.9974
83.0396
83.0063
82.9997
83.0019
83.0091
83.0045
83.0045
82.9853
83.0019
83.0004
83.0045
83.0045
82.9853
83.0091
83.0107
83.0063
83.0396
82.9997
82.9974
83.0107
83.0218
83.0063
82.9997
83.0091
83.0107
83.0094
83.0063
82.9848
83.0107
83.0019
83.0094
83.0045
82.9853
83.0091
83.0091
83.0045
83.0045
82.9997
83.0019
83.0107
83.0094
83.0094
82.9853
83.0019
83.0019
83.0063
83.0045
82.9997
82.9974
83.0019
83.0094
83.0045
82.9853
83.0107
83.0107
83.0218
83.0045
82.9612
83.0004
82.9974
83.0045
83.0045
82.9848
83.0091
83.0004
83.0045
83.0045
83.0366
83.0091
83.0107
83.0063
83.0094
82.9871
83.0004
83.0107
83.0045
83.0063
83.0366
83.0019
82.9974
83.0218
83.0094
82.9997
83.0091
83.0004
83.0045
83.0045
82.9848
83.0091
82.9974
83.0218
83.0094
83.0366
83.0091
83.0107
83.0094
83.0045
82.9848
83.0091
83.0004
83.0218
83.0045
82.9997
83.0091
83.0019
83.0045
83.0045
82.9848
83.0091
83.0107
83.0396
83.0063
82.9997
83.0107
83.0091
83.0094
83.0045
82.9871
83.0107
82.9974
83.0218
83.0045
82.9871
83.0004
83.0019
83.0218
83.0045
82.9871
83.0004
83.0019
83.0218
83.0063
82.9853
83.0107
82.9974
83.0045
83.0094
82.9853
83.0107
83.0019
83.0218
83.0045
82.9871
83.0107
82.9974
83.0396
83.0063
82.9997
83.0107
83.0107
83.0094
83.0045
82.9871
83.0004
83.0091
83.0063
83.0396
82.9997
83.0091
83.0107
83.0063
83.0094
82.9871
82.9974
83.0004
83.0094
83.0396
82.9848
83.0019
83.0091
83.0045
83.0218
82.9853
83.0107
83.0107
83.0094
83.0045
82.9848
83.0019
83.0019
83.0396
83.0045
82.9997
83.0091
82.9974
83.0094
83.0218
82.9848
83.0019
83.0004
83.0094
83.0094
83.0366
83.0004
82.9974
83.0063
83.0094
83.0366
82.9974
83.0004
83.0094
83.0063
82.9871
83.0091
82.9974
83.0094
83.0218
82.9997
82.9974
83.0107
83.0094
83.0045
82.9848
83.0004
82.9974
83.0396
83.0045
82.9997
82.9974
83.0019
83.0094
83.0045
82.9997
83.0107
83.0091
83.0094
83.0063
82.9997
83.0091
83.0019
83.0396
83.0218
82.9848
83.0091
83.0107
83.0218
83.0094
82.9997
83.0004
83.0091
83.0396
83.0396
82.9997
82.9974
83.0019
83.0045
83.0045
82.9997
82.9974
83.0004
83.0218
83.0063
82.9871
82.9974
83.0004
83.0045
83.0218
82.9997
83.0019
82.9974
83.0094
83.0094
82.9871
83.0019
83.0107
83.0396
83.0094
82.9871
83.0004
83.0107
83.0396
83.0063
82.9848
82.9974
83.0107
83.0396
83.0063
82.9871
83.0004
83.0107
83.0063
83.0063
82.9853
83.0019
82.9974
83.0094
83.0094
82.9871
83.0107
83.0004
83.0045
83.0063
82.9848
83.0107
83.0107
83.0218
83.0218
82.9871
83.0107
83.0004
83.0094
83.0063
82.9848
83.0004
83.0107
83.0094
83.0396
82.9848
83.0004
83.0019
83.0045
83.0063
82.9997
83.0091
83.0019
83.0094
83.0218
82.9848
83.0107
83.0019
83.0063
83.0063
82.9853
83.0004
83.0004
83.0045
83.0045
82.9853
83.0107
82.9974
83.0396
83.0045
82.9853
83.0019
83.0107
83.0063
83.0218
83.0366
83.0004
83.0107
83.0045
83.0396
82.9871
83.0019
83.0019
83.0396
83.0218
82.9997
83.0107
83.0107
83.0218
83.0218
82.9997
83.0019
83.0091
83.0063
83.0094
82.9853
83.0107
83.0091
83.0094
83.0094
82.9848
83.0019
83.0107
83.0218
83.0094
82.9871
83.0019
83.0091
83.0396
83.0094
82.9853
83.0091
83.0091
83.0396
83.0218
82.9848
82.9974
82.9974
83.0063
83.0045
82.9871
82.9974
83.0019
83.0045
83.0218
82.9848
83.0019
83.0091
83.0045
83.0218
82.9997
83.0004
82.9974
83.0045
83.0045
82.9871
83.0004
83.0004
83.0396
83.0063
82.9853
83.0004
83.0019
83.0094
83.0396
82.9997
83.0091
82.9974
83.0396
83.0045
82.9848
83.0004
82.9974
83.0218
83.0218
82.9853
82.9974
83.0107
83.0094
83.0063
82.9853
83.0091
83.0107
83.0045
83.0063
82.9853
83.0107
83.0004
83.0045
83.0218
82.9997
83.0107
83.0107
83.0396
83.0396
82.9853
83.0004
83.0019
83.0396
83.0094
82.9848
83.0091
83.0004
83.0218
83.0218
82.9871
83.0019
82.9974
83.0094
83.0218
82.9848
83.0004
82.9974
83.0218
83.0063
82.9871
83.0091
83.0091
83.0218
83.0396
82.9997
83.0004
83.0004
83.0063
83.0063
82.9853
83.0107
83.0004
83.0094
83.0045
83.0366
83.0019
83.0019
83.0218
83.0094
82.9853
83.0091
83.0107
83.0094
83.0094
82.9997
83.0004
83.0019
83.0094
83.0094
82.9871
83.0091
83.0019
83.0045
83.0396
82.9871
82.9974
83.0004
83.0396
83.0396
82.9871
82.9974
83.0019
83.0094
83.0396
82.9853
83.0004
83.0107
83.0045
83.0063
82.9997
83.0107
83.0004
83.0063
83.0218
82.9848
83.0019
83.0004
83.0063
83.0063
82.9997
82.9974
83.0004
83.0094
83.0045
82.9848
83.0107
83.0004
83.0063
83.0218
82.9848
83.0019
83.0004
83.0218
83.0063
82.9871
83.0019
83.0004
83.0094
83.0218
82.9871
82.9974
83.0019
83.0396
83.0045
82.9853
83.0004
82.9974
83.0063
83.0045
82.9853
82.9974
82.9974
83.0063
83.0396
82.9997
83.0004
83.0019
83.0094
83.0045
82.9848
83.0019
83.0019
83.0396
83.0218
82.9871
83.0107
83.0004
83.0094
83.0094
82.9848
82.9974
83.0107
83.0063
83.0045
82.9853
83.0091
83.0091
83.0218
83.0094
82.9871
83.0091
83.0107
83.0218
83.0045
82.9848
83.0107
83.0091
83.0045
83.0396
82.9871
82.9974
83.0004
83.0396
83.0094
82.9853
83.0019
83.0004
83.0218
83.0063
82.9853
82.9974
83.0107
83.0218
83.0218
82.9871
83.0019
83.0107
83.0063
83.0045
83.0366
83.0091
82.9974
83.0063
83.0396
82.9853
83.0107
83.0091
83.0063
83.0218
82.9871
82.9974
83.0107
83.0218
83.0063
82.9848
83.0019
83.0107
83.0063
83.0045
82.9853
83.0107
83.0004
83.0045
83.0396
82.9871
83.0091
83.0019
83.0063
83.0094
82.9853
83.0091
83.0107
83.0045
83.0396
82.9848
83.0107
82.9974
83.0218
83.0045
82.9871
83.0091
82.9974
83.0396
83.0045
82.9997
83.0091
82.9974
83.0045
83.0045
82.9853
83.0091
82.9974
83.0218
83.0396
82.9871
83.0019
83.0004
83.0218
83.0045
82.9853
83.0091
83.0091
83.0218
83.0045
83.0366
82.9974
83.0019
83.0045
83.0063
82.9871
83.0091
83.0019
83.0063
83.0063
82.9848
83.0091
83.0107
83.0218
83.0218
82.9853
83.0107
83.0019
83.0218
83.0063
82.9871
83.0107
83.0019
83.0063
83.0063
82.9853
83.0004
83.0004
83.0396
83.0045
82.9871
83.0107
83.0004
83.0396
83.0218
82.9997
83.0019
83.0091
83.0094
83.0396
82.9997
83.0019
83.0107
83.0063
83.0063
82.9848
83.0091
83.0107
83.0396
83.0045
82.9871
82.9974
83.0107
83.0218
83.0218
82.9871
82.9974
83.0004
83.0396
83.0045
82.9871
83.0091
83.0091
83.0094
83.0218
82.9853
82.9974
83.0004
83.0396
83.0218
82.9871
83.0091
83.0019
83.0063
83.0396
82.9848
83.0107
83.0107
83.0218
83.0396
82.9848
83.0004
82.9974
83.0063
83.0063
82.9997
83.0107
83.0107
83.0094
83.0396
82.9853
83.0091
83.0004
83.0094
83.0094
82.9871
83.0004
83.0019
83.0063
83.0063
83.0366
83.0091
83.0004
83.0396
83.0094
82.9853
83.0004
83.0091
83.0045
83.0218
82.9848
83.0091
83.0019
83.0063
83.0094
82.9871
83.0019
83.0019
83.0396
83.0094
82.9871
83.0004
83.0004
83.0218
83.0218
82.9997
83.0004
83.0019
83.0396
83.0396
82.9848
83.0019
83.0107
83.0063
83.0396
82.9848
83.0004
83.0019
83.0094
83.0218
82.9997
83.0019
83.0019
83.0063
83.0396
82.9853
82.9974
83.0091
83.0063
83.0094
82.9853
83.0107
83.0019
83.0396
83.0094
82.9848
82.9974
83.0091
83.0063
83.0045
82.9853
83.0107
83.0019
83.0094
83.0063
82.9853
83.0091
83.0091
83.0063
83.0094
82.9997
83.0091
83.0004
83.0396
83.0218
82.9997
82.9974
83.0107
83.0094
83.0063
82.9848
83.0107
83.0091
83.0094
83.0094
82.9853
83.0019
83.0107
83.0396
83.0396
82.9853
82.9974
83.0107
83.0045
83.0396
82.9871
83.0091
83.0107
83.0045
83.0094
82.9997
82.9974
83.0091
83.0045
83.0218
82.9997
83.0004
82.9974
83.0218
83.0094
82.9853
83.0019
83.0004
83.0094
83.0045
82.9997
83.0091
83.0107
83.0396
83.0094
82.9848
83.0004
83.0004
83.0396
83.0063
82.9853
83.0019
83.0019
83.0218
83.0045
82.9853
83.0004
83.0107
83.0045
83.0396
82.9997
82.9974
83.0019
83.0094
83.0045
82.9853
83.0004
83.0091
83.0094
83.0218
82.9871
82.9974
83.0107
83.0396
83.0396
82.9871
83.0091
83.0019
83.0045
83.0218
82.9871
83.0019
83.0019
83.0094
83.0045
82.9997
83.0004
83.0091
83.0063
83.0045
82.9997
83.0107
83.0004
83.0396
83.0396
82.9853
83.0091
83.0091
83.0045
83.0063
82.9612
83.0019
83.0091
83.0063
83.0063
82.9853
82.9974
83.0091
83.0396
83.0094
82.9848
83.0091
83.0107
83.0045
83.0045
82.9848
83.0091
82.9974
83.0396
83.0063
82.9848
82.9974
83.0019
83.0396
83.0218
82.9848
83.0091
83.0004
83.0218
83.0045
82.9997
83.0091
83.0004
83.0396
83.0396
82.9871
82.9974
83.0091
83.0396
83.0063
82.9848
83.0019
83.0019
83.0094
83.0218
82.9997
83.0107
83.0091
83.0045
83.0063
82.9997
83.0019
83.0107
83.0094
83.0063
82.9853
82.9974
83.0091
83.0045
83.0396
82.9853
83.0107
83.0091
83.0063
83.0218
83.0366
83.0107
82.9974
83.0063
83.0045
82.9848
83.0107
83.0004
83.0094
83.0063
82.9871
83.0019
83.0107
83.0045
83.0094
82.9853
83.0004
83.0107
83.0218
83.0396
82.9853
82.9974
83.0019
83.0396
83.0218
82.9997
83.0019
82.9974
83.0396
83.0063
82.9848
83.0107
83.0004
83.0218
83.0396
82.9871
83.0091
83.0107
83.0396
83.0045
82.9848
83.0107
83.0107
83.0396
83.0218
82.9853
83.0107
83.0091
83.0218
83.0396
83.0366
83.0107
82.9974
83.0063
83.0094
82.9997
83.0019
83.0091
83.0396
83.0063
82.9997
82.9974
83.0091
83.0063
83.0396
82.9997
83.0004
82.9974
83.0218
83.0396
82.9853
83.0091
83.0019
83.0094
83.0396
82.9848
83.0019
83.0107
83.0094
83.0063
82.9853
83.0091
83.0107
83.0063
83.0218
83.0366
83.0107
83.0004
83.0063
83.0063
82.9997
83.0019
83.0019
83.0094
83.0094
82.9848
83.0091
83.0107
83.0396
83.0045
82.9997
83.0091
82.9974
83.0094
83.0094
82.9871
83.0004
83.0107
83.0094
83.0063
82.9871
82.9974
83.0107
83.0063
83.0218
82.9871
83.0107
83.0091
83.0396
83.0063
82.9848
83.0004
83.0019
83.0063
83.0063
82.9871
83.0019
82.9974
83.0045
83.0218
82.9848
83.0004
83.0019
83.0218
83.0094
82.9871
82.9974
83.0019
83.0218
83.0396
82.9997
83.0091
83.0004
83.0218
83.0396
82.9853
82.9974
82.9974
83.0396
83.0396
82.9871
82.9974
83.0107
83.0094
83.0094
82.9853
83.0091
83.0107
83.0396
83.0063
82.9997
83.0019
83.0107
83.0063
83.0396
82.9853
83.0019
83.0004
83.0396
83.0045
82.9848
82.9974
83.0004
83.0094
83.0045
82.9853
82.9974
83.0019
83.0396
83.0396
82.9997
83.0091
83.0107
83.0045
83.0045
82.9853
83.0107
83.0107
83.0063
83.0094
82.9853
83.0107
83.0091
83.0218
83.0045
83.0366
83.0091
83.0004
83.0396
83.0045
82.9871
83.0019
82.9974
83.0094
83.0094
82.9612
83.0004
83.0091
83.0094
83.0094
82.9871
83.0091
83.0107
83.0063
83.0396
82.9848
83.0019
83.0004
83.0094
83.0396
82.9997
82.9974
82.9974
83.0094
83.0045
82.9871
83.0107
82.9974
83.0045
83.0063
82.9848
83.0107
83.0004
83.0063
83.0045
82.9853
83.0004
83.0004
83.0063
83.0218
82.9848
83.0019
83.0019
83.0063
83.0063
83.0366
82.9974
83.0004
83.0063
83.0094
82.9871
83.0004
83.0091
83.0396
83.0063
82.9871
83.0091
83.0091
83.0063
83.0094
82.9853
83.0004
83.0091
83.0063
83.0396
82.9853
83.0004
83.0091
83.0396
83.0396
82.9848
83.0019
83.0091
83.0045
83.0063
82.9848
83.0107
83.0107
83.0063
83.0063
82.9848
83.0107
83.0019
83.0396
83.0063
82.9848
83.0091
83.0091
83.0094
83.0094
82.9848
83.0091
83.0004
83.0218
83.0094
82.9871
82.9974
83.0019
83.0094
83.0045
83.0366
83.0004
82.9974
83.0396
83.0396
82.9871
83.0107
83.0004
83.0045
83.0218
82.9848
83.0019
83.0107
83.0094
83.0045
82.9997
82.9974
83.0091
83.0045
83.0063
83.0366
83.0019
83.0019
83.0063
83.0063
82.9871
82.9974
83.0019
83.0063
83.0396
82.9997
83.0091
82.9974
83.0396
83.0063
82.9997
83.0091
83.0019
83.0045
83.0396
82.9997
83.0004
82.9974
83.0094
83.0396
82.9997
82.9974
83.0091
83.0094
83.0094
82.9848
83.0091
83.0107
83.0045
83.0063
82.9997
82.9974
83.0004
83.0063
83.0063
82.9853
83.0107
83.0091
83.0396
83.0045
82.9997
83.0107
83.0004
83.0218
83.0045
82.9997
83.0004
83.0019
83.0063
83.0045
83.0366
83.0091
83.0107
83.0094
83.0063
82.9871
83.0019
83.0107
83.0063
83.0045
82.9871
83.0107
83.0091
83.0094
83.0045
83.0366
83.0004
83.0107
83.0063
83.0063
82.9848
83.0019
83.0004
83.0396
83.0218
82.9853
83.0091
83.0004
83.0396
83.0094
82.9997
83.0107
83.0004
83.0396
83.0094
82.9997
82.9974
83.0091
83.0045
83.0218
82.9848
83.0019
83.0091
83.0045
83.0045
82.9853
83.0107
83.0107
83.0063
83.0094
82.9853
83.0019
83.0091
83.0094
83.0396
82.9997
83.0107
83.0019
83.0094
83.0045
82.9853
82.9974
83.0091
83.0045
83.0094
83.0366
83.0107
83.0004
83.0045
83.0094
82.9871
82.9974
83.0019
83.0094
83.0094
82.9871
83.0091
83.0004
83.0396
83.0063
82.9848
83.0004
83.0019
83.0094
83.0094
82.9997
83.0091
83.0019
83.0218
83.0218
82.9997
83.0019
83.0091
83.0218
83.0396
83.0366
83.0019
83.0091
83.0063
83.0063
82.9997
83.0107
83.0019
83.0218
83.0063
82.9871
83.0091
82.9974
83.0045
83.0094
82.9848
83.0091
83.0019
83.0396
83.0045
82.9997
82.9974
82.9974
83.0094
83.0063
82.9853
83.0107
83.0107
83.0045
83.0063
82.9997
83.0107
83.0091
83.0396
83.0063
83.0366
83.0091
83.0019
83.0396
83.0396
82.9853
82.9974
83.0091
83.0396
83.0094
83.0366
83.0107
83.0019
83.0218
83.0094
82.9997
83.0004
83.0091
83.0396
83.0094
82.9848
83.0091
83.0091
83.0218
83.0063
82.9848
82.9974
83.0019
83.0063
83.0094
82.9997
82.9974
83.0019
83.0396
83.0396
82.9848
83.0004
82.9974
83.0094
83.0218
82.9848
83.0107
83.0004
83.0063
83.0396
82.9871
83.0004
82.9974
83.0063
83.0396
82.9848
82.9974
83.0091
83.0063
83.0045
83.0366
83.0107
83.0019
83.0094
83.0396
82.9997
83.0091
82.9974
83.0094
83.0218
82.9853
83.0107
83.0004
83.0094
83.0063
82.9997
83.0091
83.0091
83.0218
83.0396
82.9853
83.0019
83.0019
83.0045
83.0094
82.9871
83.0107
83.0091
83.0094
83.0063
82.9848
83.0107
83.0091
83.0094
83.0094
82.9997
83.0091
82.9974
83.0218
83.0218
82.9853
83.0107
83.0091
83.0063
83.0063
82.9871
83.0004
83.0019
83.0396
83.0094
82.9853
82.9974
83.0091
83.0045
83.0045
82.9997
83.0091
83.0004
83.0094
83.0094
82.9848
83.0091
83.0107
83.0396
83.0063
82.9848
83.0004
82.9974
83.0218
83.0094
82.9853
83.0019
83.0107
83.0045
83.0094
82.9848
83.0091
83.0019
83.0045
83.0094
82.9871
83.0107
82.9974
83.0218
83.0045
82.9848
83.0019
83.0019
83.0045
83.0045
82.9848
83.0091
83.0004
83.0045
83.0063
82.9853
83.0019
82.9974
83.0218
83.0094
82.9997
82.9974
83.0107
83.0063
83.0396
82.9997
83.0019
83.0091
83.0218
83.0218
82.9848
83.0107
83.0091
83.0094
83.0396
82.9853
83.0019
82.9974
83.0396
83.0218
82.9997
83.0091
83.0107
83.0094
83.0094
82.9848
82.9974
83.0019
83.0218
83.0045
82.9997
83.0107
83.0019
83.0396
83.0396
82.9871
83.0107
83.0107
83.0396
83.0045
82.9871
83.0019
83.0004
83.0396
83.0094
82.9871
83.0107
83.0004
83.0396
83.0063
82.9853
83.0004
83.0019
83.0045
83.0094
82.9853
83.0019
82.9974
83.0218
83.0396
82.9871
83.0091
82.9974
83.0063
83.0094
83.0366
83.0004
83.0091
83.0045
83.0396
83.0366
83.0004
82.9974
83.0063
83.0094
82.9997
83.0091
82.9974
83.0094
83.0396
82.9853
83.0091
82.9974
83.0045
83.0396
82.9848
83.0107
83.0091
83.0045
83.0045
82.9853
83.0107
82.9974
83.0045
83.0218
82.9848
83.0004
83.0091
83.0218
83.0045
82.9997
83.0091
82.9974
83.0396
83.0396
82.9871
83.0107
83.0091
83.0045
83.0094
82.9848
83.0004
83.0019
83.0063
83.0094
82.9848
83.0107
83.0004
83.0094
83.0218
82.9848
82.9974
83.0019
83.0094
83.0094
82.9871
83.0091
83.0019
83.0218
83.0045
82.9848
83.0004
83.0019
83.0063
83.0045
82.9871
82.9974
83.0004
83.0094
83.0063
82.9871
83.0091
83.0107
83.0045
83.0396
82.9848
83.0004
83.0019
83.0063
83.0396
83.0366
82.9974
83.0107
83.0045
83.0094
83.0366
83.0091
83.0019
83.0063
83.0396
82.9848
82.9974
83.0107
83.0063
83.0094
82.9871
83.0107
82.9974
83.0218
83.0045
83.0366
83.0107
82.9974
83.0045
83.0094
82.9848
83.0091
83.0004
83.0063
83.0045
82.9871
83.0091
83.0107
83.0094
83.0218
82.9853
82.9974
83.0019
83.0094
83.0094
82.9871
82.9974
83.0107
83.0218
83.0396
83.0366
83.0019
83.0019
83.0094
83.0396
82.9853
83.0107
82.9974
83.0094
83.0045
82.9853
83.0004
83.0004
83.0063
83.0094
82.9871
83.0091
83.0091
83.0063
83.0094
82.9871
82.9974
83.0091
83.0396
83.0218
82.9853
83.0019
82.9974
83.0218
83.0094
82.9853
83.0091
82.9974
83.0218
83.0045
82.9853
83.0004
83.0004
83.0218
83.0063
82.9871
82.9974
83.0004
83.0045
83.0063
82.9871
83.0019
83.0004
83.0045
83.0063
82.9853
83.0004
83.0019
83.0063
83.0063
82.9997
82.9974
83.0107
83.0396
83.0063
82.9871
83.0091
83.0107
83.0396
83.0218
82.9848
83.0107
83.0107
83.0396
83.0063
82.9612
82.9974
83.0091
83.0094
83.0396
82.9997
83.0091
82.9974
83.0045
83.0045
82.9848
83.0019
83.0091
83.0218
83.0218
82.9871
83.0091
83.0019
83.0218
83.0094
82.9871
82.9974
82.9974
83.0396
83.0045
82.9848
83.0107
83.0004
83.0218
83.0063
82.9871
83.0107
83.0019
83.0218
83.0094
83.031
83.0004
83.0019
83.0218
83.0063
82.9848
82.9974
83.0019
83.0045
83.0218
82.9853
82.9974
82.9974
83.0063
83.0094
82.9871
83.0004
83.0091
83.0094
83.0045
82.9853
83.0004
82.9974
83.0218
83.0094
82.9997
83.0004
82.9974
83.0045
83.0045
82.9853
83.0019
83.0004
83.0218
83.0063
82.9853
83.0091
83.0019
83.0396
83.0063
83.0366
82.9974
83.0091
83.0045
83.0396
83.0366
83.0107
82.9974
83.0218
83.0045
82.9848
83.0107
82.9974
83.0094
83.0396
83.0366
83.0107
83.0019
83.0396
83.0045
82.9853
83.0004
83.0091
83.0063
83.0396
82.9848
83.0019
83.0004
83.0396
83.0045
82.9997
83.0019
83.0019
83.0045
83.0396
82.9871
82.9974
83.0004
83.0063
83.0218
82.9853
83.0091
82.9974
83.0094
83.0063
82.9848
82.9974
83.0004
83.0063
83.0218
82.9848
82.9974
83.0091
83.0063
83.0094
82.9853
83.0004
83.0091
83.0218
83.0045
82.9848
83.0004
82.9974
83.0063
83.0218
82.9848
83.0004
82.9974
83.0218
83.0218
82.9871
83.0091
83.0107
83.0396
83.0094
82.9997
83.0004
83.0019
83.0063
83.0063
82.9848
83.0019
83.0107
83.0396
83.0218
82.9997
83.0091
83.0091
83.0218
83.0063
82.9997
83.0107
82.9974
83.0396
83.0063
82.9853
83.0019
83.0004
83.0218
83.0218
82.9848
83.0019
83.0107
83.0218
83.0218
82.9997
83.0019
83.0091
83.0218
83.0396
82.9997
83.0004
83.0091
83.0094
83.0045
82.9848
83.0004
82.9974
83.0094
83.0094
82.9848
83.0091
83.0107
83.0094
83.0094
82.9871
83.0107
83.0019
83.0094
83.0094
82.9853
82.9974
82.9974
83.0396
83.0396
82.9997
83.0091
83.0004
83.0396
83.0218
82.9997
83.0019
83.0107
83.0218
83.0063
82.9997
83.0019
83.0019
83.0045
83.0045
82.9871
83.0004
83.0107
83.0045
83.0094
82.9853
83.0091
82.9974
83.0045
83.0218
83.0366
83.0004
83.0107
83.0218
83.0396
83.0366
82.9974
83.0004
83.0063
83.0063
82.9853
83.0091
82.9974
83.0045
83.0396
82.9853
83.0019
82.9974
83.0218
83.0045
82.9997
83.0091
83.0091
83.0045
83.0045
82.9853
82.9974
82.9974
83.0218
83.0218
82.9997
82.9974
83.0019
83.0063
83.0396
82.9853
83.0019
83.0019
83.0045
83.0063
82.9871
83.0004
83.0004
83.0396
83.0218
82.9853
83.0091
82.9974
83.0396
83.0094
82.9848
82.9974
83.0091
83.0218
83.0045
82.9853
83.0019
82.9974
83.0063
83.0063
82.9871
83.0019
83.0107
83.0396
83.0396
82.9997
83.0091
83.0107
83.0094
83.0045
82.9853
83.0107
82.9974
83.0063
83.0045
82.9848
82.9974
83.0091
83.0396
83.0396
83.0366
83.0091
83.0107
83.0063
83.0218
82.9871
83.0019
83.0004
83.0218
83.0063
82.9853
83.0107
83.0107
83.0045
83.0218
82.9871
82.9974
83.0091
83.0063
83.0396
82.9853
83.0107
83.0019
83.0094
83.0045
82.9997
82.9974
82.9974
83.0063
83.0063
82.9871
83.0091
83.0004
83.0094
83.0218
83.0366
83.0091
83.0004
83.0045
83.0396
82.9997
82.9974
82.9974
83.0218
83.0063
82.9848
83.0004
83.0091
83.0396
83.0063
82.9871
83.0019
83.0107
83.0094
83.0218
82.9853
83.0091
83.0091
83.0063
83.0063
82.9848
83.0019
82.9974
83.0094
83.0094
82.9848
83.0004
82.9974
83.0063
83.0396
82.9871
83.0091
83.0107
83.0218
83.0063
83.0366
83.0091
83.0091
83.0045
83.0396
82.9997
83.0019
83.0091
83.0218
83.0094
82.9848
83.0019
83.0107
83.0218
83.0218
82.9871
82.9974
83.0004
83.0396
83.0045
82.9871
82.9974
83.0004
83.0063
83.0063
82.9848
83.0091
83.0019
83.0045
83.0045
82.9997
83.0107
82.9974
83.0063
83.0094
82.9997
83.0019
83.0107
83.0045
83.0094
82.9997
82.9974
82.9974
83.0396
83.0094
82.9871
83.0019
83.0091
83.0218
83.0396
82.9997
83.0107
83.0004
83.0094
83.0094
82.9848
83.0091
83.0019
83.0094
83.0396
82.9853
83.0019
82.9974
83.0396
83.0396
82.9871
83.0019
83.0004
83.0396
83.0396
82.9848
83.0019
83.0091
83.0045
83.0218
82.9871
82.9974
82.9974
83.0063
83.0218
82.9871
82.9974
83.0107
83.0396
83.0063
82.9848
83.0004
83.0019
83.0063
83.0396
82.9997
83.0107
82.9974
83.0063
83.0218
82.9871
82.9974
83.0004
83.0063
83.0218
82.9871
82.9974
82.9974
83.0063
83.0396
82.9997
83.0004
83.0107
83.0396
83.0396
82.9871
83.0019
83.0004
83.0045
83.0063
82.9853
83.0019
83.0019
83.0063
83.0396
82.9853
83.0107
83.0091
83.0094
83.0094
82.9997
83.0004
82.9974
83.0045
83.0094
82.9853
83.0091
83.0107
83.0094
83.0063
82.9848
83.0019
83.0004
83.0045
83.0094
82.9997
82.9974
83.0004
83.0094
83.0045
82.9848
83.0004
83.0019
83.0045
83.0063
82.9871
82.9974
82.9974
83.0094
83.0094
82.9848
82.9974
83.0107
83.0045
83.0094
82.9871
83.0107
82.9974
83.0218
83.0218
82.9853
83.0019
83.0019
83.0396
83.0396
82.9997
83.0019
83.0107
83.0094
83.0045
82.9853
83.0019
83.0107
83.0045
83.0063
82.9853
83.0091
83.0004
83.0094
83.0396
82.9848
83.0091
83.0107
83.0218
83.0063
82.9853
83.0019
83.0107
83.0063
83.0396
82.9853
82.9974
83.0091
83.0218
83.0094
82.9871
83.0019
83.0019
83.0094
83.0396
82.9853
83.0004
82.9974
83.0396
83.0396
82.9871
83.0091
83.0091
83.0063
83.0218
82.9853
83.0019
83.0019
83.0218
83.0218
82.9853
83.0107
82.9974
83.0218
83.0218
82.9848
83.0107
83.0091
83.0045
83.0094
82.9871
83.0004
83.0091
83.0094
83.0045
82.9848
83.0091
83.0091
83.0094
83.0396
82.9997
83.0019
83.0091
83.0063
83.0218
82.9853
83.0019
83.0107
83.0396
83.0396
82.9853
83.0091
83.0107
83.0045
83.0218
82.9871
83.0004
83.0091
83.0218
83.0218
82.9853
82.9974
82.9974
83.0045
83.0094
82.9871
82.9974
83.0019
83.0094
83.0396
82.9853
83.0004
83.0107
83.0045
83.0045
82.9871
82.9974
83.0091
83.0094
83.0063
82.9997
83.0091
82.9974
83.0396
83.0218
82.9848
83.0107
83.0004
83.0218
83.0396
82.9848
83.0107
83.0091
83.0094
83.0218
82.9848
83.0107
83.0091
83.0094
83.0045
82.9997
83.0019
83.0091
83.0045
83.0045
82.9871
83.0107
83.0107
83.0396
83.0396
82.9997
83.0091
83.0004
83.0045
83.0045
82.9848
83.0091
82.9974
83.0396
83.0045
82.9848
83.0004
83.0091
83.0218
83.0218
82.9871
83.0107
82.9974
83.0094
83.0396
82.9997
83.0107
83.0004
83.0218
83.0045
82.9848
83.0107
83.0091
83.0045
83.0396
82.9997
83.0091
82.9974
83.0063
83.0063
82.9848
83.0091
83.0107
83.0396
83.0218
83.0366
83.0019
83.0091
83.0063
83.0045
82.9997
83.0004
83.0091
83.0063
83.0063
82.9853
83.0004
83.0019
83.0094
83.0094
82.9997
83.0091
83.0004
83.0045
83.0063
82.9871
83.0107
83.0019
83.0063
83.0094
82.9871
83.0107
83.0091
83.0045
83.0094
82.9871
83.0091
83.0004
83.0396
83.0396
82.9848
82.9974
83.0019
83.0218
83.0094
82.9871
83.0004
83.0004
83.0396
83.0045
82.9853
83.0107
83.0107
83.0063
83.0218
82.9612
82.9974
83.0091
83.0218
83.0094
83.0366
83.0004
83.0107
83.0094
83.0094
82.9848
83.0004
83.0107
83.0396
83.0218
82.9871
83.0091
83.0091
83.0063
83.0396
82.9997
83.0019
83.0004
83.0396
83.0045
82.9997
82.9974
82.9974
83.0063
83.0094
82.9848
83.0004
83.0091
83.0063
83.0045
82.9871
83.0019
82.9974
83.0045
83.0218
82.9871
83.0091
83.0091
83.0396
83.0045
83.0366
83.0019
83.0107
83.0063
83.0045
82.9997
82.9974
83.0107
83.0218
83.0218
82.9997
83.0091
83.0091
83.0063
83.0094
82.9871
83.0091
83.0004
83.0045
83.0094
82.9997
83.0091
82.9974
83.0218
83.0063
82.9997
83.0091
83.0107
83.0063
83.0396
82.9871
83.0091
83.0004
83.0063
83.0063
82.9848
83.0019
83.0107
83.0396
83.0094
82.9853
83.0004
83.0004
83.0218
83.0396
82.9848
82.9974
83.0019
83.0063
83.0218
82.9853
83.0107
83.0091
83.0396
83.0218
82.9853
83.0091
82.9974
83.0045
83.0045
82.9853
83.0004
83.0019
83.0063
83.0063
82.9871
82.9974
83.0019
83.0094
83.0396
82.9853
83.0107
83.0004
83.0045
83.0045
82.9848
82.9974
83.0091
83.0063
83.0063
82.9853
83.0004
83.0107
83.0218
83.0045
82.9997
83.0091
83.0091
83.0063
83.0094
82.9853
83.0019
83.0019
83.0094
83.0396
82.9848
83.0004
83.0091
83.0094
83.0094
82.9853
83.0019
83.0019
83.0063
83.0045
82.9853
83.0004
82.9974
83.0218
83.0396
82.9871
83.0019
83.0004
83.0218
83.0045
82.9997
83.0107
82.9974
83.0094
83.0396
82.9848
83.0019
82.9974
83.0094
83.0094
82.9871
83.0091
83.0107
83.0045
83.0063
82.9848
82.9974
83.0019
83.0218
83.0063
82.9853
83.0107
82.9974
83.0094
83.0094
82.9848
82.9974
83.0019
83.0045
83.0094
82.9848
82.9974
83.0107
83.0396
83.0094
83.0366
83.0107
83.0091
83.0045
83.0094
82.9848
83.0019
82.9974
83.0063
83.0045
82.9997
83.0091
83.0091
83.0218
83.0063
82.9871
83.0019
83.0107
83.0396
83.0045
82.9848
83.0004
83.0091
83.0396
83.0063
82.9853
83.0019
83.0004
83.0094
83.0396
82.9997
83.0004
83.0107
83.0396
83.0094
83.0366
83.0091
83.0019
83.0045
83.0218
82.9853
83.0004
83.0107
83.0045
83.0094
82.9871
83.0019
83.0107
83.0396
83.0396
82.9848
83.0091
83.0091
83.0094
83.0094
82.9848
83.0019
83.0004
83.0218
83.0045
82.9853
83.0107
83.0091
83.0063
83.0094
82.9871
82.9974
82.9974
83.0396
83.0045
82.9997
83.0107
83.0004
83.0094
83.0396
82.9871
83.0091
83.0019
83.0396
83.0396
82.9612
83.0091
83.0091
83.0218
83.0094
82.9871
83.0004
83.0091
83.0218
83.0218
82.9848
83.0019
83.0019
83.0218
83.0063
82.9997
83.0004
83.0019
83.0045
83.0094
82.9848
83.0091
83.0019
83.0063
83.0063
82.9853
83.0107
83.0091
83.0045
83.0094
82.9871
83.0091
83.0107
83.0045
83.0218
82.9997
83.0004
83.0091
83.0094
83.0094
82.9853
83.0107
83.0107
83.0045
83.0218
82.9871
82.9974
83.0019
83.0063
83.0063
82.9997
83.0107
82.9974
83.0045
83.0045
82.9848
83.0004
83.0004
83.0094
83.0063
82.9997
83.0004
82.9974
83.0094
83.0063
82.9853
83.0091
82.9974
83.0396
83.0045
82.9871
83.0004
83.0004
83.0218
83.0063
82.9997
83.0004
83.0091
83.0063
83.0063
82.9853
83.0107
83.0107
83.0396
83.0045
82.9853
83.0004
83.0019
83.0396
83.0396
82.9853
83.0107
82.9974
83.0094
83.0218
82.9853
82.9974
83.0004
83.0396
83.0094
82.9997
83.0091
83.0019
83.0063
83.0094
83.0366
83.0091
83.0004
83.0396
83.0063
82.9997
83.0091
83.0004
83.0396
83.0396
82.9853
83.0091
83.0004
83.0063
83.0218
82.9848
83.0107
83.0019
83.0045
83.0218
82.9997
83.0107
83.0004
83.0045
83.0094
82.9848
82.9974
83.0019
83.0094
83.0094
82.9871
83.0091
83.0107
83.0396
83.0045
82.9848
83.0019
83.0107
83.0218
83.0094
82.9848
82.9974
82.9974
83.0045
83.0063
82.9871
83.0091
83.0107
83.0063
83.0094
82.9853
83.0019
83.0091
83.0063
83.0094
82.9848
83.0004
82.9974
83.0218
83.0045
83.0366
83.0004
83.0107
83.0094
83.0218
82.9871
82.9974
83.0019
83.0396
83.0396
82.9871
83.0091
82.9974
83.0094
83.0063
82.9997
83.0091
83.0107
83.0045
83.0094
82.9848
83.0019
82.9974
83.0218
83.0218
82.9997
83.0004
83.0107
83.0063
83.0045
82.9871
83.0107
82.9974
83.0094
83.0063
82.9997
83.0004
83.0107
83.0045
83.0045
82.9848
83.0019
83.0091
83.0063
83.0218
82.9853
83.0091
83.0019
83.0063
83.0094
82.9871
83.0107
83.0091
83.0094
83.0094
82.9997
83.0107
83.0107
83.0396
83.0094
82.9848
83.0004
83.0091
83.0045
83.0094
82.9997
83.0019
83.0091
83.0063
83.0396
82.9853
83.0091
83.0019
83.0094
83.0063
82.9848
82.9974
83.0091
83.0063
83.0094
82.9853
83.0019
82.9974
83.0396
83.0045
83.0366
83.0091
83.0019
83.0396
83.0218
82.9853
83.0004
83.0019
83.0218
83.0045
82.9997
83.0019
82.9974
83.0218
83.0045
83.0366
82.9974
82.9974
83.0045
83.0094
82.9997
83.0091
83.0107
83.0063
83.0045
82.9997
83.0004
82.9974
83.0045
83.0396
82.9853
82.9974
83.0019
83.0396
83.0218
82.9848
83.0004
83.0107
83.0045
83.0045
82.9853
83.0019
83.0091
83.0045
83.0218
82.9997
83.0107
82.9974
83.0094
83.0396
82.9853
83.0019
83.0091
83.0094
83.0063
82.9997
83.0107
83.0019
83.0396
83.0396
82.9871
83.0091
83.0091
83.0045
83.0396
82.9853
82.9974
83.0019
83.0396
83.0063
82.9997
83.0091
83.0004
83.0396
83.0218
82.9848
83.0004
83.0091
83.0218
83.0045
83.0366
83.0091
83.0004
83.0218
83.0396
82.9853
83.0019
83.0004
83.0094
83.0045
82.9853
83.0107
83.0019
83.0094
83.0396
82.9871
82.9974
83.0004
83.0063
83.0094
82.9997
83.0019
83.0091
83.0396
83.0396
82.9997
83.0004
83.0091
83.0094
83.0063
82.9612
83.0091
82.9974
83.0094
83.0045
82.9853
82.9974
83.0107
83.0218
83.0063
82.9871
82.9974
83.0107
83.0218
83.0218
82.9871
83.0107
83.0091
83.0045
83.0094
82.9853
83.0107
83.0019
83.0218
83.0063
82.9871
83.0004
82.9974
83.0396
83.0396
82.9853
83.0107
82.9974
83.0063
83.0063
82.9871
83.0107
83.0004
83.0218
83.0396
82.9853
82.9974
82.9974
83.0396
83.0396
82.9871
83.0107
82.9974
83.0045
83.0396
82.9997
83.0107
83.0107
83.0094
83.0396
82.9853
83.0019
83.0107
83.0063
83.0063
82.9871
82.9974
82.9974
83.0094
83.0396
82.9997
83.0091
83.0107
83.0045
83.0063
83.031
82.9974
83.0107
83.0045
83.0094
82.9853
82.9974
83.0091
83.0063
83.0045
82.9871
82.9974
82.9974
83.0396
83.0094
83.0366
83.0019
83.0004
83.0045
83.0063
82.9848
83.0107
82.9974
83.0094
83.0094
82.9853
83.0004
83.0019
83.0063
83.0045
82.9997
82.9974
82.9974
83.0218
83.0094
82.9997
83.0107
83.0091
83.0094
83.0094
82.9871
82.9974
83.0019
83.0094
83.0094
82.9853
83.0091
83.0091
83.0094
83.0045
82.9871
83.0107
83.0004
83.0063
83.0396
82.9853
83.0107
83.0019
83.0094
83.0094
82.9997
83.0019
82.9974
83.0218
83.0094
82.9871
83.0091
82.9974
83.0063
83.0094
82.9848
83.0019
83.0107
83.0094
83.0218
82.9848
83.0019
83.0004
83.0063
83.0045
82.9848
83.0107
83.0019
83.0063
83.0045
82.9871
83.0107
83.0091
83.0045
83.0063
82.9871
83.0091
83.0004
83.0045
83.0094
82.9848
83.0019
82.9974
83.0218
83.0094
83.0366
83.0019
83.0107
83.0063
83.0063
82.9853
83.0004
83.0091
83.0045
83.0396
82.9848
83.0091
83.0004
83.0045
83.0218
82.9853
83.0004
83.0004
83.0094
83.0045
82.9997
83.0004
83.0107
83.0045
83.0218
82.9853
83.0107
83.0107
83.0045
83.0063
82.9871
83.0004
83.0019
83.0396
83.0218
82.9871
83.0019
83.0019
83.0218
83.0063
82.9997
83.0091
82.9974
83.0063
83.0045
82.9997
83.0091
83.0019
83.0396
83.0094
82.9848
83.0091
82.9974
83.0218
83.0094
82.9848
82.9974
82.9974
83.0094
83.0218
82.9848
82.9974
83.0091
83.0094
83.0396
82.9871
82.9974
83.0004
83.0396
83.0094
82.9997
83.0019
83.0091
83.0396
83.0063
82.9848
83.0019
83.0091
83.0045
83.0218
82.9853
83.0107
83.0107
83.0045
83.0094
82.9997
82.9974
83.0004
83.0396
83.0063
82.9871
83.0004
83.0091
83.0094
83.0396
82.9848
83.0019
83.0107
83.0396
83.0094
82.9848
83.0107
82.9974
83.0094
83.0218
82.9997
82.9974
83.0004
83.0094
83.0218
82.9871
82.9974
83.0004
83.0218
83.0094
82.9848
83.0019
82.9974
83.0063
83.0218
82.9853
83.0019
83.0107
83.0094
83.0094
82.9853
82.9974
82.9974
83.0218
83.0045
82.9997
83.0091
83.0091
83.0218
83.0218
82.9997
82.9974
83.0004
83.0063
83.0218
83.0366
82.9974
83.0107
83.0063
83.0094
82.9848
83.0019
83.0107
83.0045
83.0218
82.9848
83.0019
83.0019
83.0094
83.0094
82.9848
83.0019
83.0091
83.0045
83.0218
82.9848
83.0091
83.0091
83.0063
83.0218
82.9997
83.0004
83.0004
83.0045
83.0396
82.9871
83.0091
83.0091
83.0045
83.0045
82.9848
83.0107
83.0019
83.0218
83.0094
82.9997
83.0019
83.0091
83.0094
83.0045
82.9871
82.9974
82.9974
83.0396
83.0396
82.9853
82.9974
82.9974
83.0094
83.0218
82.9848
82.9974
83.0004
83.0045
83.0396
82.9871
82.9974
83.0004
83.0045
83.0063
82.9848
83.0107
83.0019
83.0045
83.0063
83.0366
83.0019
83.0004
83.0218
83.0094
82.9853
83.0019
82.9974
83.0396
83.0396
83.0366
83.0091
83.0107
83.0094
83.0218
82.9871
83.0091
82.9974
83.0094
83.0218
82.9848
83.0091
83.0107
83.0094
83.0045
82.9848
83.0107
82.9974
83.0218
83.0094
82.9848
82.9974
83.0019
83.0045
83.0094
82.9848
83.0107
83.0004
83.0094
83.0045
82.9997
83.0019
83.0019
83.0063
83.0063
82.9871
83.0019
82.9974
83.0063
83.0396
82.9871
82.9974
83.0004
83.0218
83.0218
82.9848
82.9974
83.0091
83.0045
83.0396
82.9853
83.0019
83.0091
83.0396
83.0094
82.9871
83.0091
83.0004
83.0094
83.0045
83.0366
83.0091
83.0004
83.0218
83.0045
82.9997
83.0019
83.0004
83.0218
83.0396
82.9871
83.0019
83.0019
83.0396
83.0218
82.9997
82.9974
82.9974
83.0396
83.0218
82.9871
83.0107
82.9974
83.0045
83.0045
82.9871
82.9974
83.0107
83.0218
83.0094
82.9848
83.0091
83.0004
83.0218
83.0218
82.9871
82.9974
83.0019
83.0094
83.0396
82.9997
82.9974
82.9974
83.0094
83.0094
82.9871
83.0091
83.0019
83.0218
83.0045
82.9848
83.0091
83.0004
83.0218
83.0094
82.9848
83.0019
83.0107
83.0396
83.0063
82.9853
82.9974
83.0091
83.0063
83.0045
82.9997
83.0091
83.0019
83.0063
83.0396
82.9997
83.0107
83.0019
83.0094
83.0045
82.9997
83.0091
83.0019
83.0045
83.0396
82.9997
83.0107
83.0004
83.0396
83.0218
82.9848
83.0004
83.0091
83.0396
83.0396
82.9871
83.0004
83.0091
83.0045
83.0045
82.9871
83.0019
83.0004
83.0094
83.0045
82.9871
83.0091
83.0004
83.0396
83.0063
82.9853
83.0107
83.0004
83.0045
83.0063
82.9997
83.0019
83.0091
83.0094
83.0063
82.9853
82.9974
83.0004
83.0094
83.0218
82.9848
82.9974
83.0019
83.0063
83.0218
82.9848
83.0019
83.0091
83.0045
83.0063
82.9853
83.0004
82.9974
83.0094
83.0045
82.9871
83.0107
83.0107
83.0396
83.0063
82.9997
83.0107
82.9974
83.0396
83.0094
82.9871
83.0019
83.0107
83.0045
83.0396
82.9871
83.0019
83.0004
83.0063
83.0063
82.9853
83.0019
83.0019
83.0063
83.0218
82.9853
83.0107
83.0107
83.0218
83.0218
82.9848
82.9974
83.0091
83.0063
83.0045
82.9871
82.9974
83.0091
83.0396
83.0063
82.9853
83.0004
82.9974
83.0218
83.0218
82.9997
83.0019
82.9974
83.0063
83.0396
82.9853
83.0004
82.9974
83.0396
83.0063
82.9853
83.0019
83.0091
83.0396
83.0396
82.9848
83.0091
83.0107
83.0045
83.0063
82.9853
83.0019
83.0107
83.0063
83.0045
82.9871
82.9974
83.0019
83.0094
83.0094
82.9848
83.0091
83.0107
83.0094
83.0045
82.9997
82.9974
83.0107
83.0094
83.0396
82.9848
83.0107
83.0004
83.0094
83.0396
82.9997
83.0107
83.0004
83.0218
83.0218
82.9853
83.0019
82.9974
83.0063
83.0218
82.9871
82.9974
83.0019
83.0218
83.0218
82.9871
83.0107
83.0107
83.0094
83.0396
82.9853
83.0019
83.0004
83.0396
83.0396
82.9997
83.0107
83.0107
83.0396
83.0218
82.9997
83.0107
83.0004
83.0063
83.0045
83.0366
83.0004
83.0107
83.0094
83.0094
82.9848
83.0004
82.9974
83.0218
83.0094
82.9853
83.0091
83.0091
83.0063
83.0063
82.9997
82.9974
82.9974
83.0063
83.0218
82.9848
82.9974
83.0107
83.0063
83.0396
82.9997
83.0107
82.9974
83.0094
83.0094
82.9853
82.9974
83.0004
83.0094
83.0045
82.9848
82.9974
82.9974
83.0094
83.0218
82.9848
83.0019
83.0004
83.0396
83.0094
82.9997
82.9974
82.9974
83.0094
83.0094
82.9871
83.0091
83.0107
83.0218
83.0063
82.9853
83.0019
83.0019
83.0218
83.0063
83.0366
82.9974
83.0091
83.0396
83.0094
82.9848
82.9974
82.9974
83.0063
83.0045
82.9853
82.9974
83.0091
83.0396
83.0094
82.9848
83.0019
83.0091
83.0094
83.0045
82.9853
83.0091
83.0019
83.0094
83.0045
82.9871
83.0091
83.0091
83.0045
83.0045
82.9853
83.0091
83.0019
83.0045
83.0094
82.9997
82.9974
83.0107
83.0063
83.0094
82.9848
83.0004
83.0107
83.0045
83.0396
82.9853
83.0091
83.0107
83.0094
83.0063
82.9997
82.9974
82.9974
83.0218
83.0218
82.9997
83.0091
83.0019
83.0094
83.0063
82.9871
83.0004
83.0107
83.0396
83.0094
82.9997
83.0019
83.0019
83.0063
83.0045
82.9853
83.0019
83.0019
83.0063
83.0063
82.9848
83.0004
83.0107
83.0218
83.0094
82.9853
83.0091
83.0019
83.0063
83.0218
82.9871
83.0107
83.0004
83.0094
83.0045
83.0366
83.0019
83.0091
83.0063
83.0396
82.9997
83.0004
83.0004
83.0045
83.0045
82.9871
83.0004
83.0019
83.0396
83.0045
82.9871
83.0107
82.9974
83.0045
83.0045
82.9848
83.0107
83.0004
83.0218
83.0094
82.9997
83.0004
82.9974
83.0094
83.0063
82.9853
83.0019
83.0019
83.0094
83.0396
83.0366
83.0107
83.0091
83.0094
83.0396
83.0366
82.9974
83.0091
83.0396
83.0094
82.9997
83.0091
83.0091
83.0045
83.0045
83.0366
83.0004
83.0091
83.0045
83.0094
82.9853
83.0019
83.0091
83.0218
83.0063
82.9853
83.0004
83.0091
83.0045
83.0063
82.9848
83.0107
82.9974
83.0063
83.0094
82.9853
83.0107
83.0004
83.0063
83.0094
82.9997
83.0091
83.0091
83.0218
83.0045
82.9997
83.0107
83.0107
83.0396
83.0094
82.9871
83.0019
82.9974
83.0094
83.0063
82.9997
83.0004
83.0004
83.0396
83.0045
82.9997
83.0107
83.0107
83.0045
83.0396
82.9848
82.9974
83.0091
83.0218
83.0218
82.9853
83.0004
82.9974
83.0094
83.0094
83.0366
82.9974
83.0004
83.0045
83.0094
82.9871
83.0019
83.0019
83.0045
83.0094
82.9871
83.0091
83.0019
83.0045
83.0063
82.9871
83.0091
83.0004
83.0396
83.0045
82.9853
82.9974
83.0107
83.0218
83.0094
82.9871
82.9974
83.0091
83.0094
83.0063
82.9853
83.0091
83.0091
83.0218
83.0045
82.9853
83.0107
83.0107
83.0396
83.0094
82.9848
82.9974
83.0091
83.0094
83.0218
82.9997
83.0004
83.0019
83.0063
83.0094
82.9853
83.0091
83.0004
83.0063
83.0094
82.9853
83.0004
83.0107
83.0218
83.0396
82.9871
83.0004
83.0107
83.0396
83.0396
82.9997
82.9974
83.0091
83.0218
83.0063
82.9848
83.0107
82.9974
83.0094
83.0063
82.9853
82.9974
83.0107
83.0045
83.0218
82.9871
83.0019
82.9974
83.0094
83.0063
82.9848
83.0091
83.0019
83.0045
83.0218
82.9997
83.0019
83.0004
83.0045
83.0396
82.9997
83.0019
83.0091
83.0094
83.0396
82.9997
83.0004
83.0091
83.0094
83.0063
82.9871
83.0107
83.0107
83.0063
83.0094
82.9853
83.0107
83.0107
83.0063
83.0094
82.9853
82.9974
83.0091
83.0218
83.0094
82.9997
83.0004
83.0004
83.0218
83.0045
82.9997
83.0004
83.0107
83.0094
83.0396
82.9848
82.9974
83.0091
83.0045
83.0063
82.9997
82.9974
83.0091
83.0218
83.0218
82.9871
82.9974
83.0091
83.0094
83.0063
82.9997
83.0091
83.0107
83.0094
83.0045
82.9853
83.0004
83.0091
83.0396
83.0045
82.9997
83.0019
82.9974
83.0218
83.0218
82.9848
83.0107
83.0107
83.0094
83.0045
82.9997
83.0091
83.0107
83.0218
83.0218
82.9871
83.0004
83.0004
83.0396
83.0396
82.9853
83.0107
83.0019
83.0094
83.0396
82.9848
83.0019
83.0091
83.0094
83.0094
82.9871
83.0004
83.0091
83.0063
83.0094
82.9848
83.0019
83.0107
83.0045
83.0045
82.9871
83.0004
82.9974
83.0396
83.0045
82.9848
83.0091
83.0107
83.0094
83.0045
82.9871
83.0004
83.0004
83.0396
83.0396
82.9997
83.0107
83.0091
83.0094
83.0094
83.0366
83.0107
83.0107
83.0063
83.0063
82.9853
83.0091
83.0019
83.0094
83.0045
82.9853
82.9974
83.0004
83.0045
83.0396
82.9853
83.0019
83.0091
83.0045
83.0045
82.9848
83.0091
83.0004
83.0218
83.0045
82.9848
83.0004
83.0091
83.0218
83.0045
82.9848
83.0107
83.0107
83.0396
83.0045
82.9871
83.0019
83.0091
83.0218
83.0094
83.0366
83.0019
82.9974
83.0094
83.0218
82.9997
83.0019
83.0107
83.0396
83.0063
82.9997
83.0019
82.9974
83.0218
83.0063
82.9871
83.0004
83.0019
83.0396
83.0218
82.9848
83.0107
83.0107
83.0063
83.0094
82.9853
83.0091
83.0091
83.0063
83.0396
82.9853
83.0107
83.0019
83.0396
83.0396
82.9871
83.0091
83.0107
83.0063
83.0218
82.9997
83.0019
83.0019
83.0045
83.0063
82.9853
83.0019
83.0091
83.0063
83.0396
82.9848
83.0091
83.0091
83.0063
83.0045
82.9853
83.0091
83.0107
83.0218
83.0396
82.9871
83.0019
83.0091
83.0063
83.0045
82.9997
82.9974
83.0019
83.0396
83.0094
82.9871
83.0019
83.0019
83.0094
83.0045
82.9848
83.0107
83.0019
83.0045
83.0396
82.9853
83.0019
83.0107
83.0218
83.0045
82.9853
83.0091
83.0107
83.0094
83.0218
82.9871
83.0091
83.0019
83.0045
83.0063
82.9871
83.0107
83.0091
83.0063
83.0094
83.0366
83.0019
83.0004
83.0218
83.0063
82.9853
83.0004
83.0004
83.0045
83.0045
82.9853
83.0004
83.0107
83.0063
83.0063
82.9848
83.0004
83.0091
83.0396
83.0063
83.0366
83.0107
83.0019
83.0063
83.0045
82.9853
83.0091
83.0107
83.0063
83.0094
82.9997
82.9974
83.0019
83.0396
83.0094
82.9997
83.0107
83.0107
83.0218
83.0396
83.031
82.9974
83.0091
83.0094
83.0094
82.9871
83.0107
83.0091
83.0218
83.0063
82.9848
83.0091
82.9974
83.0063
83.0045
82.9997
83.0019
83.0004
83.0396
83.0094
82.9848
83.0019
83.0091
83.0218
83.0063
82.9848
83.0107
83.0004
83.0045
83.0045
82.9848
83.0091
83.0091
83.0045
83.0094
82.9848
83.0004
83.0019
83.0094
83.0218
82.9997
83.0004
83.0107
83.0045
83.0063
82.9848
82.9974
83.0004
83.0396
83.0094
82.9848
82.9974
83.0004
83.0045
83.0045
82.9848
83.0091
83.0107
83.0396
83.0063
82.9997
83.0019
82.9974
83.0218
83.0396
83.0366
83.0019
83.0091
83.0094
83.0045
83.0366
83.0107
83.0004
83.0094
83.0094
82.9997
83.0004
82.9974
83.0218
83.0094
82.9848
83.0019
83.0019
83.0094
83.0218
82.9871
83.0107
83.0019
83.0045
83.0218
82.9997
83.0107
82.9974
83.0063
83.0396
82.9848
83.0019
83.0019
83.0396
83.0045
82.9871
83.0019
83.0107
83.0063
83.0045
82.9871
82.9974
82.9974
83.0045
83.0218
82.9871
83.0004
82.9974
83.0218
83.0063
82.9997
83.0004
83.0107
83.0045
83.0094
82.9997
82.9974
83.0107
83.0218
83.0063
82.9853
83.0107
83.0004
83.0063
83.0063
82.9871
82.9974
83.0107
83.0094
83.0045
82.9848
83.0004
83.0019
83.0063
83.0063
82.9848
83.0091
83.0019
83.0063
83.0063
83.0366
83.0004
83.0107
83.0045
83.0045
82.9853
82.9974
82.9974
83.0218
83.0045
82.9853
83.0004
83.0091
83.0218
83.0045
82.9997
83.0004
83.0019
83.0396
83.0396
82.9871
83.0091
83.0019
83.0063
83.0396
82.9871
83.0107
83.0019
83.0396
83.0218
82.9848
83.0004
82.9974
83.0063
83.0218
82.9853
82.9974
83.0091
83.0045
83.0218
82.9848
83.0091
83.0091
83.0396
83.0396
82.9848
83.0004
83.0004
83.0218
83.0396
82.9853
83.0019
83.0107
83.0045
83.0045
82.9871
83.0004
83.0004
83.0045
83.0045
82.9997
82.9974
83.0107
83.0045
83.0094
82.9997
83.0019
83.0004
83.0396
83.0094
82.9848
83.0004
83.0091
83.0094
83.0063
82.9853
82.9974
82.9974
83.0218
83.0045
82.9997
83.0107
83.0004
83.0063
83.0094
82.9848
83.0091
83.0091
83.0045
83.0396
82.9871
83.0091
83.0107
83.0045
83.0094
82.9871
82.9974
83.0091
83.0045
83.0218
82.9848
83.0019
83.0004
83.0063
83.0063
82.9853
83.0091
82.9974
83.0094
83.0218
82.9853
83.0019
83.0004
83.0045
83.0045
82.9853
82.9974
83.0107
83.0063
83.0094
82.9853
83.0091
83.0019
83.0094
83.0094
82.9848
83.0004
83.0091
83.0218
83.0045
82.9853
82.9974
83.0107
83.0063
83.0063
82.9848
83.0019
83.0004
83.0396
83.0218
82.9848
82.9974
83.0004
83.0063
83.0045
83.0366
83.0091
83.0107
83.0063
83.0063
82.9848
83.0091
83.0107
83.0063
83.0094
82.9853
83.0107
83.0019
83.0218
83.0218
82.9853
83.0091
83.0107
83.0063
83.0063
82.9997
83.0004
83.0004
83.0045
83.0218
82.9853
82.9974
83.0004
83.0063
83.0045
82.9848
83.0107
83.0107
83.0063
83.0063
82.9848
82.9974
82.9974
83.0045
83.0063
82.9848
83.0091
82.9974
83.0094
83.0094
82.9848
83.0004
83.0091
83.0218
83.0094
82.9871
83.0107
83.0091
83.0094
83.0045
82.9997
83.0107
82.9974
83.0218
83.0063
82.9853
83.0091
83.0107
83.0063
83.0218
82.9848
83.0107
83.0019
83.0045
83.0045
82.9853
83.0091
83.0019
83.0218
83.0045
82.9848
83.0091
83.0004
83.0063
83.0218
82.9848
83.0107
83.0019
83.0218
83.0094
82.9853
83.0004
83.0019
83.0094
83.0063
82.9848
82.9974
83.0004
83.0094
83.0396
82.9853
82.9974
83.0107
83.0063
83.0063
82.9853
83.0107
83.0004
83.0218
83.0063
82.9997
82.9974
83.0004
83.0045
83.0094
82.9997
83.0004
82.9974
83.0045
83.0396
82.9997
82.9974
83.0004
83.0045
83.0094
82.9853
83.0091
83.0091
83.0094
83.0094
82.9871
83.0107
83.0091
83.0218
83.0396
82.9871
83.0091
83.0107
83.0063
83.0218
82.9853
83.0091
82.9974
83.0045
83.0218
82.9848
83.0091
82.9974
83.0218
83.0218
82.9848
83.0091
82.9974
83.0396
83.0063
82.9853
83.0019
83.0004
83.0094
83.0218
83.0366
83.0107
83.0091
83.0094
83.0045
82.9848
83.0019
83.0004
83.0063
83.0094
82.9871
82.9974
83.0019
83.0063
83.0045
82.9997
83.0107
82.9974
83.0218
83.0063
82.9871
83.0019
83.0004
83.0045
83.0045
82.9853
83.0004
83.0107
83.0396
83.0094
82.9997
83.0004
83.0091
83.0045
83.0396
82.9997
83.0107
82.9974
83.0045
83.0045
82.9853
82.9974
83.0004
83.0094
83.0094
82.9871
83.0107
83.0004
83.0218
83.0218
82.9848
83.0004
83.0004
83.0063
83.0045
82.9848
83.0091
83.0004
83.0045
83.0094
82.9871
83.0091
83.0004
83.0094
83.0396
82.9871
83.0004
83.0019
83.0218
83.0218
82.9853
83.0107
82.9974
83.0094
83.0045
82.9871
83.0107
83.0091
83.0063
83.0094
82.9848
83.0107
82.9974
83.0063
83.0063
82.9853
83.0019
82.9974
83.0045
83.0045
82.9871
83.0004
82.9974
83.0063
83.0094
82.9871
83.0107
82.9974
83.0218
83.0045
82.9848
83.0107
83.0091
83.0218
83.0063
82.9853
82.9974
82.9974
83.0045
83.0045
82.9871
83.0004
83.0004
83.0396
83.0218
82.9871
83.0091
82.9974
83.0396
83.0045
82.9871
82.9974
83.0107
83.0094
83.0396
82.9871
82.9974
83.0107
83.0396
83.0218
82.9871
83.0004
83.0091
83.0218
83.0045
82.9848
83.0004
83.0004
83.0063
83.0218
82.9871
82.9974
83.0004
83.0396
83.0063
82.9997
83.0091
82.9974
83.0063
83.0094
82.9871
83.0091
82.9974
83.0045
83.0094
82.9853
83.0091
83.0107
83.0094
83.0045
82.9848
83.0091
83.0019
83.0063
83.0063
82.9853
83.0107
83.0091
83.0063
83.0094
82.9853
83.0091
83.0019
83.0396
83.0218
82.9853
83.0091
82.9974
83.0396
83.0218
82.9997
83.0091
82.9974
83.0094
83.0094
82.9853
83.0107
83.0107
83.0045
83.0218
83.0366
83.0004
83.0091
83.0396
83.0396
82.9853
83.0107
83.0004
83.0063
83.0094
82.9871
83.0019
83.0004
83.0045
83.0045
82.9853
83.0019
83.0091
83.0218
83.0094
82.9871
82.9974
83.0091
83.0045
83.0045
82.9997
83.0004
82.9974
83.0218
83.0094
82.9848
83.0019
83.0107
83.0218
83.0094
82.9853
83.0019
83.0091
83.0094
83.0045
82.9997
83.0091
83.0091
83.0063
83.0396
82.9997
82.9974
83.0091
83.0063
83.0045
82.9997
83.0107
82.9974
83.0094
83.0063
82.9853
83.0107
82.9974
83.0063
83.0218
82.9848
83.0091
83.0107
83.0396
83.0218
82.9997
83.0019
83.0004
83.0063
83.0218
82.9853
83.0019
83.0107
83.0218
83.0045
82.9853
83.0004
83.0019
83.0063
83.0063
82.9848
83.0091
83.0107
83.0094
83.0218
82.9853
83.0019
83.0004
83.0063
83.0094
82.9997
83.0019
83.0091
83.0396
83.0396
82.9997
83.0004
82.9974
83.0396
83.0094
83.031
83.0019
83.0107
83.0063
83.0045
82.9848
82.9974
83.0019
83.0396
83.0218
82.9871
82.9974
83.0107
83.0045
83.0045
82.9871
83.0107
83.0107
83.0063
83.0396
82.9848
83.0004
83.0107
83.0045
83.0396
82.9871
83.0019
83.0107
83.0063
83.0396
82.9871
83.0004
83.0019
83.0063
83.0396
82.9848
83.0091
83.0019
83.0094
83.0094
82.9871
82.9974
83.0091
83.0063
83.0218
82.9997
83.0004
83.0091
83.0063
83.0094
82.9853
83.0091
83.0004
83.0045
83.0063
82.9853
83.0091
83.0091
83.0218
83.0218
82.9853
83.0004
83.0091
83.0063
83.0063
82.9848
83.0019
83.0107
83.0218
83.0396
82.9853
83.0019
83.0004
83.0218
83.0396
82.9848
83.0004
82.9974
83.0063
83.0218
82.9997
82.9974
83.0091
83.0045
83.0063
82.9997
83.0004
82.9974
83.0063
83.0045
82.9848
83.0091
83.0107
83.0396
83.0045
82.9848
82.9974
83.0004
83.0063
83.0045
82.9871
83.0107
83.0107
83.0063
83.0045
83.0366
83.0004
83.0004
83.0218
83.0063
82.9871
82.9974
83.0019
83.0045
83.0094
82.9612
83.0019
83.0107
83.0063
83.0218
82.9853
82.9974
82.9974
83.0045
83.0045
83.0366
82.9974
83.0004
83.0063
83.0094
82.9871
83.0107
83.0091
83.0396
83.0218
82.9853
83.0019
83.0004
83.0094
83.0045
82.9848
83.0091
83.0019
83.0094
83.0094
82.9848
83.0004
83.0004
83.0094
83.0045
82.9853
83.0091
82.9974
83.0045
83.0063
82.9997
83.0107
83.0004
83.0094
83.0045
82.9848
83.0004
83.0004
83.0396
83.0045
83.0366
83.0004
82.9974
83.0063
83.0063
82.9848
83.0004
83.0004
83.0094
83.0094
82.9997
83.0107
83.0107
83.0396
83.0094
82.9997
83.0107
83.0019
83.0045
83.0063
82.9853
83.0019
83.0091
83.0063
83.0094
82.9853
82.9974
83.0019
83.0063
83.0063
82.9848
83.0004
83.0107
83.0063
83.0094
82.9871
83.0091
83.0019
83.0094
83.0094
82.9871
83.0107
82.9974
83.0063
83.0396
82.9853
83.0107
83.0019
83.0045
83.0094
82.9853
82.9974
82.9974
83.0045
83.0396
82.9853
83.0004
83.0019
83.0218
83.0218
82.9997
83.0107
82.9974
83.0094
83.0396
82.9848
82.9974
83.0107
83.0218
83.0045
82.9848
83.0019
83.0107
83.0063
83.0396
82.9871
82.9974
83.0019
83.0063
83.0045
82.9871
83.0091
83.0091
83.0094
83.0396
82.9853
83.0019
83.0019
83.0063
83.0045
82.9848
82.9974
83.0107
83.0094
83.0218
82.9997
83.0107
83.0091
83.0396
83.0045
82.9848
83.0091
82.9974
83.0045
83.0094
82.9997
83.0091
83.0019
83.0063
83.0396
82.9848
83.0091
83.0091
83.0063
83.0094
82.9997
83.0004
83.0107
83.0063
83.0045
82.9871
82.9974
83.0091
83.0094
83.0063
82.9853
83.0107
83.0107
83.0063
83.0218
82.9997
82.9974
83.0019
83.0396
83.0045
82.9997
83.0019
83.0019
83.0396
83.0094
82.9853
83.0019
83.0019
83.0218
83.0218
82.9871
83.0004
82.9974
83.0218
83.0063
82.9853
82.9974
83.0091
83.0063
83.0218
82.9997
83.0004
83.0019
83.0218
83.0063
82.9871
83.0107
83.0107
83.0045
83.0396
82.9871
82.9974
82.9974
83.0396
83.0396
82.9853
82.9974
83.0091
83.0094
83.0063
82.9871
82.9974
83.0019
83.0396
83.0396
82.9853
83.0091
83.0019
83.0063
83.0063
82.9853
83.0004
82.9974
83.0045
83.0396
82.9997
83.0019
83.0107
83.0396
83.0063
82.9848
83.0004
83.0107
83.0094
83.0396
82.9997
83.0091
82.9974
83.0218
83.0063
82.9871
82.9974
82.9974
83.0094
83.0218
82.9848
83.0004
83.0019
83.0218
83.0063
82.9853
82.9974
82.9974
83.0063
83.0094
82.9853
82.9974
83.0091
83.0094
83.0094
82.9871
83.0091
83.0004
83.0094
83.0063
82.9871
82.9974
83.0107
83.0063
83.0063
82.9848
83.0019
83.0091
83.0045
83.0063
82.9853
83.0107
83.0019
83.0094
83.0045
83.0366
83.0019
82.9974
83.0045
83.0396
82.9853
82.9974
83.0004
83.0218
83.0094
82.9871
83.0019
83.0107
83.0045
83.0396
82.9871
83.0107
83.0107
83.0094
83.0396
82.9853
83.0091
83.0091
83.0045
83.0396
83.0366
83.0107
82.9974
83.0094
83.0396
82.9612
83.0019
83.0107
83.0045
83.0045
82.9848
83.0004
83.0019
83.0396
83.0094
82.9848
83.0107
83.0004
83.0063
83.0094
82.9848
83.0004
83.0019
83.0094
83.0063
82.9853
83.0019
83.0107
83.0063
83.0396
82.9853
83.0019
82.9974
83.0094
83.0063
82.9853
83.0091
83.0091
83.0094
83.0094
82.9871
83.0019
83.0091
83.0063
83.0063
82.9997
83.0004
83.0107
83.0063
83.0063
82.9853
83.0107
83.0004
83.0063
83.0396
82.9871
83.0091
83.0107
83.0063
83.0094
82.9871
82.9974
83.0107
83.0063
83.0063
82.9997
83.0019
82.9974
83.0218
83.0218
82.9853
83.0004
82.9974
83.0063
83.0218
82.9853
82.9974
83.0004
83.0063
83.0094
82.9997
83.0019
83.0004
83.0396
83.0045
82.9853
82.9974
82.9974
83.0045
83.0396
82.9871
83.0091
83.0091
83.0063
83.0045
82.9997
83.0019
83.0091
83.0218
83.0396
82.9848
82.9974
83.0019
83.0218
83.0063
82.9997
83.0019
83.0091
83.0063
83.0063
82.9853
83.0091
82.9974
83.0063
83.0218
82.9997
82.9974
83.0091
83.0396
83.0063
82.9997
83.0107
83.0019
83.0045
83.0045
82.9853
83.0019
82.9974
83.0094
83.0396
83.0366
82.9974
82.9974
83.0094
83.0396
82.9848
83.0107
83.0019
83.0045
83.0094
83.0366
83.0091
83.0019
83.0094
83.0218
82.9871
83.0004
83.0107
83.0396
83.0045
82.9848
82.9974
83.0091
83.0094
83.0094
82.9848
83.0107
82.9974
83.0094
83.0218
82.9871
83.0004
83.0107
83.0396
83.0063
82.9871
83.0091
82.9974
83.0045
83.0045
82.9848
83.0107
83.0004
83.0094
83.0094
83.0366
82.9974
83.0107
83.0396
83.0045
83.0366
83.0004
82.9974
83.0094
83.0045
82.9853
83.0004
83.0091
83.0045
83.0396
82.9853
83.0019
83.0091
83.0045
83.0396
82.9853
83.0019
83.0107
83.0063
83.0396
82.9871
83.0091
83.0004
83.0396
83.0218
82.9848
83.0091
83.0019
83.0063
83.0045
82.9871
83.0004
83.0107
83.0094
83.0063
82.9848
83.0091
83.0107
83.0396
83.0218
82.9871
83.0091
83.0107
83.0396
83.0218
82.9871
83.0091
83.0107
83.0218
83.0094
82.9997
83.0019
83.0091
83.0396
83.0094
82.9853
83.0091
83.0019
83.0045
83.0063
82.9848
83.0019
83.0091
83.0396
83.0218
82.9997
83.0019
83.0004
83.0094
83.0218
82.9997
83.0091
83.0004
83.0063
83.0063
82.9853
83.0091
82.9974
83.0045
83.0045
83.0366
82.9974
83.0091
83.0063
83.0218
83.0366
83.0107
83.0107
83.0396
83.0045
82.9848
82.9974
83.0107
83.0396
83.0063
82.9853
83.0019
83.0004
83.0218
83.0396
82.9871
83.0107
83.0019
83.0218
83.0063
82.9997
83.0004
83.0019
83.0063
83.0063
82.9848
83.0091
83.0004
83.0094
83.0094
82.9997
83.0107
83.0004
83.0396
83.0396
82.9848
83.0019
83.0091
83.0218
83.0218
82.9871
83.0004
83.0019
83.0396
83.0218
82.9997
83.0091
83.0091
83.0063
83.0396
82.9848
82.9974
83.0091
83.0045
83.0218
83.0366
82.9974
82.9974
83.0396
83.0218
82.9871
83.0004
82.9974
83.0045
83.0396
82.9853
83.0019
82.9974
83.0045
83.0396
82.9997
83.0004
83.0107
83.0063
83.0094
82.9997
82.9974
83.0019
83.0063
83.0396
82.9848
83.0019
82.9974
83.0094
83.0045
82.9871
83.0107
82.9974
83.0063
83.0396
82.9848
83.0107
83.0091
83.0396
83.0094
82.9871
83.0019
82.9974
83.0218
83.0045
82.9871
83.0019
83.0091
83.0094
83.0094
82.9848
83.0107
82.9974
83.0218
83.0094
82.9848
82.9974
83.0019
83.0045
83.0094
82.9871
83.0091
83.0004
83.0063
83.0094
82.9871
83.0004
83.0019
83.0094
83.0045
82.9848
83.0004
83.0019
83.0094
83.0063
82.9997
83.0019
83.0019
83.0063
83.0063
82.9853
83.0019
83.0004
83.0063
83.0045
82.9853
83.0004
83.0091
83.0218
83.0218
82.9848
83.0107
82.9974
83.0063
83.0045
82.9848
83.0004
83.0107
83.0045
83.0063
82.9848
82.9974
83.0091
83.0045
83.0094
82.9848
83.0019
83.0019
83.0063
83.0396
82.9871
82.9974
82.9974
83.0063
83.0045
82.9848
82.9974
83.0004
83.0094
83.0396
82.9853
83.0107
83.0107
83.0045
83.0063
82.9853
83.0004
82.9974
83.0218
83.0045
82.9848
83.0004
83.0107
83.0045
83.0045
82.9871
83.0004
83.0091
83.0063
83.0063
82.9871
82.9974
83.0019
83.0063
83.0063
82.9997
83.0107
83.0004
83.0218
83.0045
83.0366
82.9974
83.0107
83.0094
83.0045
82.9848
83.0091
83.0107
83.0396
83.0094
82.9871
83.0004
82.9974
83.0063
83.0218
82.9871
83.0091
83.0091
83.0063
83.0045
82.9871
83.0107
83.0019
83.0218
83.0396
82.9848
83.0019
83.0004
83.0094
83.0063
82.9853
83.0107
83.0091
83.0218
83.0396
82.9871
83.0019
83.0019
83.0094
83.0094
82.9871
83.0107
83.0107
83.0218
83.0218
82.9853
83.0019
83.0091
83.0063
83.0218
82.9871
83.0019
83.0107
83.0396
83.0094
82.9848
83.0019
82.9974
83.0045
83.0396
82.9997
83.0019
83.0004
83.0063
83.0218
83.0366
83.0004
83.0091
83.0063
83.0218
82.9871
83.0091
83.0107
83.0396
83.0396
82.9848
83.0019
83.0019
83.0063
83.0396
82.9997
82.9974
83.0019
83.0045
83.0063
82.9997
83.0004
83.0004
83.0063
83.0045
82.9997
83.0004
83.0091
83.0094
83.0396
82.9853
83.0091
83.0107
83.0094
83.0396
82.9997
83.0019
83.0091
83.0045
83.0094
82.9997
83.0107
83.0004
83.0063
83.0218
82.9853
82.9974
83.0019
83.0396
83.0063
82.9853
83.0091
83.0107
83.0063
83.0094
82.9997
83.0019
83.0091
83.0045
83.0094
83.0366
82.9974
83.0091
83.0396
83.0396
83.0366
82.9974
83.0107
83.0218
83.0045
82.9848
83.0107
83.0004
83.0094
83.0396
82.9853
83.0004
83.0019
83.0094
83.0396
82.9848
83.0107
82.9974
83.0218
83.0218
82.9853
83.0107
83.0004
83.0045
83.0094
82.9871
83.0107
83.0091
83.0396
83.0218
82.9871
83.0004
83.0091
83.0063
83.0218
82.9997
83.0019
83.0107
83.0396
83.0218
82.9871
83.0004
83.0019
83.0396
83.0218
82.9997
82.9974
82.9974
83.0063
83.0396
82.9853
83.0091
83.0019
83.0063
83.0063
82.9871
82.9974
83.0004
83.0396
83.0396
82.9997
83.0107
83.0019
83.0094
83.0045
82.9871
83.0091
83.0107
83.0218
83.0045
82.9997
83.0019
83.0004
83.0063
83.0045
82.9871
82.9974
82.9974
83.0218
83.0218
82.9848
83.0091
83.0019
83.0045
83.0045
82.9853
83.0019
83.0107
83.0063
83.0094
82.9848
83.0004
83.0107
83.0094
83.0045
82.9871
83.0004
82.9974
83.0063
83.0094
83.0366
83.0107
83.0107
83.0094
83.0218
82.9848
82.9974
83.0004
83.0396
83.0094
82.9997
82.9974
83.0107
83.0218
83.0396
82.9853
83.0019
83.0107
83.0396
83.0094
83.0366
83.0019
83.0107
83.0063
83.0045
82.9871
83.0107
83.0004
83.0063
83.0218
82.9871
83.0019
83.0019
83.0218
83.0218
82.9871
82.9974
83.0107
83.0063
83.0396
82.9848
83.0019
83.0091
83.0094
83.0063
82.9853
82.9974
83.0019
83.0045
83.0045
82.9848
83.0004
83.0107
83.0063
83.0063
82.9871
83.0019
83.0107
83.0063
83.0396
82.9997
82.9974
82.9974
83.0094
83.0094
82.9871
83.0091
83.0019
83.0045
83.0045
82.9853
83.0091
83.0091
83.0218
83.0396
82.9853
83.0004
83.0091
83.0094
83.0396
82.9853
83.0091
82.9974
83.0218
83.0045
82.9853
83.0107
83.0019
83.0045
83.0094
82.9997
83.0091
83.0004
83.0094
83.0218
82.9848
82.9974
83.0091
83.0218
83.0094
82.9871
83.0107
83.0019
83.0045
83.0063
82.9848
83.0091
83.0019
83.0396
83.0063
83.0366
83.0004
83.0107
83.0218
83.0045
82.9853
83.0107
82.9974
83.0396
83.0063
82.9871
83.0091
83.0004
83.0396
83.0063
82.9853
83.0004
83.0107
83.0063
83.0063
''')
array_string_1 = data_sting_1.split('\n')
array_cords_2 = data_cords_2.split('\n')
origin_log_2 = []
origin_log1 = []
for i in range(len(array_cords_2)-1):
  if(array_cords_2[i] != ''):
     origin_log_2.append(float(array_cords_2[i]))
array_string_1.pop(0)
for i in range(len(origin_log_2)-1):
  if(array_string_1[i] == 'Work'):
    origin_log1.append(origin_log_2[i])

print(array_string_1[0])
print(origin_log1[3])

In [ ]:
# @title
data_string_2 = ('''
Company bus/ Cab
Two wheeler
Car
Individual Auto
Car
Car
Individual Auto
Car
Two wheeler
Car
Car
Two wheeler
Car
Car
Car
Two wheeler
Public transport
Car
Two wheeler
Two wheeler
Car
Two wheeler
Car
Public transport
Car
Car
Car
Car
Two wheeler
Two wheeler
Car
Individual Auto
Car
Two wheeler
Car
Car
Car
Two wheeler
Company bus/ Cab
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Individual Auto
Public transport
Car
Two wheeler
Car
Individual Auto
Car
Car
Individual Auto
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Car
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
Individual Auto
Walk
Public transport
Individual Auto
Public transport
Public transport
Individual Auto
Car
Individual Auto
Car
Public transport
Two wheeler
Two wheeler
Two wheeler
Walk
Public transport
Public transport
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Public transport
Individual Auto
Public transport
Public transport
Public transport
Cycle
Cycle
Cycle
Cycle
Cycle
Two wheeler
Two wheeler
Individual Auto
Public transport
Public transport
Public transport
Company bus/ Cab
Public transport
Individual Auto
Share Auto
Cycle
Two wheeler
Two wheeler
Public transport
Company bus/ Cab
Car
Two wheeler
Two wheeler
Individual Auto
Car
Two wheeler
Company bus/ Cab
Public transport
Public transport
Company bus/ Cab
Car
Walk
Public transport
Two wheeler
Public transport
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
Car
Public transport
Two wheeler
Public transport
Company bus/ Cab
Public transport
Car
Public transport
Public transport
Two wheeler
Public transport
Individual Auto
Car
Public transport
Public transport
Public transport
Two wheeler
Public transport
Share Auto
Public transport
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Cycle
Individual Auto
Public transport
Share Auto
Individual Auto
Company bus/ Cab
Two wheeler
Public transport
Individual Auto
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Public transport
Public transport
Company bus/ Cab
Company bus/ Cab
Individual Auto
Public transport
Company bus/ Cab
Share Auto
Car
Cycle
Public transport
Individual Auto
Public transport
Company bus/ Cab
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Company bus/ Cab
Public transport
Public transport
Public transport
Public transport
Company bus/ Cab
Individual Auto
Two wheeler
Individual Auto
Public transport
Public transport
Public transport
Two wheeler
Public transport
Company bus/ Cab
Walk
Car
Individual Auto
Individual Auto
Car
Individual Auto
Public transport
Public transport
Two wheeler
Car
Public transport
Two wheeler
Public transport
Public transport
Company bus/ Cab
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Public transport
Two wheeler
Public transport
Company bus/ Cab
Public transport
Public transport
Public transport
Company bus/ Cab
Company bus/ Cab
Two wheeler
Public transport
Public transport
Car
Public transport
Public transport
Cycle
Individual Auto
Two wheeler
Car
Share Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Company bus/ Cab
Company bus/ Cab
Two wheeler
Company bus/ Cab
Company bus/ Cab
Company bus/ Cab
Public transport
Public transport
Two wheeler
Public transport
Public transport
Car
Two wheeler
Share Auto
Two wheeler
Share Auto
Share Auto
Two wheeler
Car
Two wheeler
Company bus/ Cab
Public transport
Share Auto
Public transport
Company bus/ Cab
Share Auto
Company bus/ Cab
Company bus/ Cab
Public transport
Individual Auto
Car
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Public transport
Car
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Cycle
Individual Auto
Two wheeler
Two wheeler
Public transport
Individual Auto
Car
Car
Individual Auto
Two wheeler
Car
Public transport
Public transport
Public transport
Public transport
Individual Auto
Two wheeler
Public transport
Individual Auto
Individual Auto
Two wheeler
Public transport
Public transport
Public transport
Cycle
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Individual Auto
Company bus/ Cab
Public transport
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Car
Car
Two wheeler
Individual Auto
Car
Car
Car
Individual Auto
Car
Public transport
Two wheeler
Individual Auto
Two wheeler
Public transport
Individual Auto
Public transport
Car
Public transport
Car
Share Auto
Cycle
Car
Public transport
Car
Public transport
Car
Two wheeler
Public transport
Company bus/ Cab
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Cycle
Two wheeler
Two wheeler
Public transport
Car
Car
Walk
Two wheeler
Two wheeler
Share Auto
Two wheeler
Car
Car
Cycle
Car
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Cycle
Two wheeler
Two wheeler
Car
Car
Car
Company bus/ Cab
Public transport
Two wheeler
Share Auto
Public transport
Share Auto
Two wheeler
Public transport
Cycle
Individual Auto
Individual Auto
Individual Auto
Public transport
Public transport
Company bus/ Cab
Public transport
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Car
Two wheeler
Public transport
Two wheeler
Walk
Public transport
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Car
Walk
Car
Public transport
Two wheeler
Individual Auto
Company bus/ Cab
Public transport
Company bus/ Cab
Walk
Two wheeler
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
Two wheeler
Cycle
Individual Auto
Two wheeler
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Company bus/ Cab
Two wheeler
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Company bus/ Cab
Individual Auto
Individual Auto
Public transport
Public transport
Individual Auto
Two wheeler
Two wheeler
Company bus/ Cab
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Public transport
Public transport
Individual Auto
Individual Auto
Car
Two wheeler
Share Auto
Share Auto
Public transport
Car
Public transport
Public transport
Share Auto
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Share Auto
Share Auto
Two wheeler
Public transport
Two wheeler
Two wheeler
Share Auto
Public transport
Two wheeler
Car
Two wheeler
Individual Auto
Individual Auto
Car
Individual Auto
Two wheeler
Public transport
Public transport
Public transport
Company bus/ Cab
Individual Auto
Car
Individual Auto
Public transport
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
Two wheeler
Walk
Car
Public transport
Two wheeler
Two wheeler
Individual Auto
Cycle
Two wheeler
Two wheeler
Public transport
Car
Public transport
Two wheeler
Car
Two wheeler
Two wheeler
Car
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Cycle
Car
Public transport
Public transport
Public transport
Two wheeler
Public transport
Car
Individual Auto
Car
Two wheeler
Two wheeler
Car
Car
Walk
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Cycle
Public transport
Public transport
Two wheeler
Cycle
Two wheeler
Two wheeler
Public transport
Public transport
Car
Car
Public transport
Company bus/ Cab
Individual Auto
Two wheeler
Car
Car
Individual Auto
Two wheeler
Public transport
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Car
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Car
Car
Two wheeler
Car
Individual Auto
Two wheeler
Public transport
Car
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Car
Individual Auto
Two wheeler
Two wheeler
Public transport
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
Car
Public transport
Public transport
Car
Two wheeler
Individual Auto
Cycle
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Public transport
Two wheeler
Cycle
Two wheeler
Car
Individual Auto
Two wheeler
Public transport
Car
Individual Auto
Company bus/ Cab
Car
Company bus/ Cab
Two wheeler
Two wheeler
Public transport
Individual Auto
Company bus/ Cab
Car
Car
Individual Auto
Car
Car
Individual Auto
Two wheeler
Public transport
Car
Individual Auto
Individual Auto
Public transport
Company bus/ Cab
Two wheeler
Public transport
Car
Two wheeler
Car
Car
Car
Car
Car
Two wheeler
Cycle
Cycle
Car
Car
Car
Car
Car
Car
Public transport
Public transport
Public transport
Individual Auto
Company bus/ Cab
Individual Auto
Public transport
Two wheeler
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Public transport
Two wheeler
Two wheeler
Public transport
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Individual Auto
Car
Individual Auto
Car
Individual Auto
Car
Car
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
Two wheeler
Car
Car
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Company bus/ Cab
Car
Individual Auto
Car
Car
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Company bus/ Cab
Two wheeler
Individual Auto
Two wheeler
Car
Two wheeler
Company bus/ Cab
Car
Public transport
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Car
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Public transport
Individual Auto
Individual Auto
Two wheeler
Public transport
Company bus/ Cab
Individual Auto
Individual Auto
Public transport
Public transport
Two wheeler
Two wheeler
Car
Public transport
Individual Auto
Individual Auto
Company bus/ Cab
Individual Auto
Public transport
Individual Auto
Public transport
Cycle
Two wheeler
Public transport
Individual Auto
Individual Auto
Company bus/ Cab
Public transport
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
Company bus/ Cab
Two wheeler
Two wheeler
Public transport
Car
Individual Auto
Individual Auto
Individual Auto
Walk
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Company bus/ Cab
Individual Auto
Public transport
Car
Public transport
Individual Auto
Two wheeler
Individual Auto
Public transport
Two wheeler
Two wheeler
Individual Auto
Public transport
Individual Auto
Individual Auto
Public transport
Two wheeler
Individual Auto
Walk
Individual Auto
Cycle
Public transport
Two wheeler
Cycle
Car
Public transport
Individual Auto
Two wheeler
Two wheeler
Car
Public transport
Car
Public transport
Car
Public transport
Individual Auto
Public transport
Individual Auto
Two wheeler
Two wheeler
Public transport
Public transport
Public transport
Two wheeler
Walk
Public transport
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Individual Auto
Two wheeler
Car
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Two wheeler
Individual Auto
Walk
Walk
Car
Car
Two wheeler
Walk
Two wheeler
Car
Two wheeler
Individual Auto
Car
Two wheeler
Two wheeler
Public transport
Car
Car
Car
Walk
Two wheeler
Individual Auto
Cycle
Two wheeler
Two wheeler
Public transport
Public transport
Car
Public transport
Individual Auto
Two wheeler
Public transport
Public transport
Two wheeler
Individual Auto
Car
Two wheeler
Individual Auto
Public transport
Cycle
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
Two wheeler
Two wheeler
Car
Car
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Individual Auto
Two wheeler
Individual Auto
Public transport
Two wheeler
Individual Auto
Public transport
Two wheeler
Individual Auto
Company bus/ Cab
Car
Car
Walk
Car
Public transport
Public transport
Public transport
Two wheeler
Car
Two wheeler
Two wheeler
Public transport
Individual Auto
Car
Car
Public transport
Cycle
Company bus/ Cab
Two wheeler
Company bus/ Cab
Two wheeler
Car
Car
Public transport
Two wheeler
Two wheeler
Cycle
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
Public transport
Two wheeler
Two wheeler
Car
Car
Two wheeler
Car
Public transport
Public transport
Company bus/ Cab
Company bus/ Cab
Car
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Individual Auto
Individual Auto
Car
Individual Auto
Two wheeler
Cycle
Two wheeler
Two wheeler
Car
Public transport
Two wheeler
Individual Auto
Car
Car
Public transport
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Car
Walk
Cycle
Two wheeler
Car
Individual Auto
Public transport
Individual Auto
Public transport
Car
Two wheeler
Two wheeler
Car
Public transport
Car
Public transport
Individual Auto
Car
Cycle
Two wheeler
Car
Cycle
Two wheeler
Public transport
Public transport
Car
Two wheeler
Cycle
Car
Car
Two wheeler
Two wheeler
Two wheeler
Walk
Cycle
Two wheeler
Car
Cycle
Cycle
Car
Car
Car
Two wheeler
Car
Car
Two wheeler
Two wheeler
Car
Public transport
Car
Individual Auto
Car
Individual Auto
Two wheeler
Individual Auto
Public transport
Public transport
Public transport
Car
Individual Auto
Individual Auto
Car
Public transport
Public transport
Individual Auto
Public transport
Public transport
Company bus/ Cab
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Car
Individual Auto
Car
Two wheeler
Two wheeler
Public transport
Car
Car
Car
Two wheeler
Public transport
Car
Two wheeler
Car
Car
Cycle
Car
Car
Cycle
Car
Public transport
Car
Public transport
Car
Car
Two wheeler
Two wheeler
Car
Public transport
Individual Auto
Public transport
Public transport
Public transport
Company bus/ Cab
Public transport
Public transport
Two wheeler
Individual Auto
Two wheeler
Public transport
Company bus/ Cab
Public transport
Individual Auto
Car
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Cycle
Two wheeler
Car
Public transport
Two wheeler
Walk
Car
Walk
Two wheeler
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
Car
Walk
Cycle
Public transport
Car
Walk
Car
Individual Auto
Two wheeler
Public transport
Walk
Public transport
Car
Public transport
Two wheeler
Walk
Two wheeler
Car
Two wheeler
Car
Two wheeler
Company bus/ Cab
Public transport
Public transport
Individual Auto
Two wheeler
Cycle
Car
Individual Auto
Car
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Car
Public transport
Individual Auto
Company bus/ Cab
Public transport
Car
Individual Auto
Public transport
Public transport
Public transport
Car
Public transport
Individual Auto
Public transport
Public transport
Two wheeler
Public transport
Car
Two wheeler
Share Auto
Share Auto
Public transport
Public transport
Public transport
Public transport
Company bus/ Cab
Two wheeler
Share Auto
Company bus/ Cab
Share Auto
Individual Auto
Company bus/ Cab
Public transport
Public transport
Public transport
Individual Auto
Car
Car
Public transport
Individual Auto
Public transport
Public transport
Cycle
Two wheeler
Individual Auto
Company bus/ Cab
Individual Auto
Two wheeler
Company bus/ Cab
Individual Auto
Public transport
Two wheeler
Company bus/ Cab
Company bus/ Cab
Company bus/ Cab
Company bus/ Cab
Individual Auto
Company bus/ Cab
Two wheeler
Public transport
Individual Auto
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
Public transport
Company bus/ Cab
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Public transport
Two wheeler
Company bus/ Cab
Company bus/ Cab
Two wheeler
Public transport
Two wheeler
Two wheeler
Public transport
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Public transport
Public transport
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Two wheeler
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Car
Two wheeler
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Public transport
Two wheeler
Car
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
Car
Public transport
Public transport
Public transport
Company bus/ Cab
Individual Auto
Public transport
Individual Auto
Company bus/ Cab
Company bus/ Cab
Car
Public transport
Two wheeler
Individual Auto
Public transport
Individual Auto
Car
Public transport
Company bus/ Cab
Public transport
Individual Auto
Public transport
Car
Individual Auto
Public transport
Company bus/ Cab
Car
Company bus/ Cab
Company bus/ Cab
Company bus/ Cab
Car
Public transport
Public transport
Company bus/ Cab
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Company bus/ Cab
Individual Auto
Car
Public transport
Company bus/ Cab
Public transport
Two wheeler
Two wheeler
Public transport
Company bus/ Cab
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Company bus/ Cab
Public transport
Public transport
Car
Public transport
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Walk
Public transport
Car
Walk
Public transport
Public transport
Public transport
Public transport
Individual Auto
Two wheeler
Public transport
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Car
Public transport
Car
Public transport
Public transport
Car
Individual Auto
Individual Auto
Car
Two wheeler
Two wheeler
Individual Auto
Public transport
Individual Auto
Individual Auto
Public transport
Individual Auto
Two wheeler
Two wheeler
Public transport
Public transport
Individual Auto
Car
Two wheeler
Two wheeler
Public transport
Public transport
Individual Auto
Car
Car
Public transport
Public transport
Car
Two wheeler
Company bus/ Cab
Public transport
Two wheeler
Individual Auto
Two wheeler
Public transport
Public transport
Public transport
Public transport
Two wheeler
Public transport
Car
Public transport
Public transport
Public transport
Company bus/ Cab
Company bus/ Cab
Public transport
Individual Auto
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Public transport
Two wheeler
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Public transport
Individual Auto
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Public transport
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Two wheeler
Walk
Company bus/ Cab
Two wheeler
Company bus/ Cab
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Public transport
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Public transport
Individual Auto
Two wheeler
Public transport
Two wheeler
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Cycle
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Car
Company bus/ Cab
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Individual Auto
Company bus/ Cab
Public transport
Company bus/ Cab
Individual Auto
Company bus/ Cab
Public transport
Two wheeler
Company bus/ Cab
Company bus/ Cab
Company bus/ Cab
Two wheeler
Individual Auto
Public transport
Two wheeler
Cycle
Public transport
Two wheeler
Company bus/ Cab
Two wheeler
Car
Public transport
Individual Auto
Car
Public transport
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Car
Two wheeler
Car
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Car
Two wheeler
Cycle
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Car
Public transport
Share Auto
Share Auto
Individual Auto
Individual Auto
Car
Two wheeler
Car
Cycle
Car
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Public transport
Individual Auto
Walk
Car
Individual Auto
Car
Individual Auto
Car
Car
Public transport
Car
Company bus/ Cab
Public transport
Car
Two wheeler
Individual Auto
Two wheeler
Car
Public transport
Company bus/ Cab
Company bus/ Cab
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Car
Individual Auto
Public transport
Individual Auto
Walk
Individual Auto
Individual Auto
Public transport
Individual Auto
Two wheeler
Two wheeler
Car
Car
Car
Individual Auto
Individual Auto
Individual Auto
Car
Individual Auto
Individual Auto
Public transport
Car
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Company bus/ Cab
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Individual Auto
Two wheeler
Two wheeler
Public transport
Individual Auto
Two wheeler
Car
Two wheeler
Company bus/ Cab
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Company bus/ Cab
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Car
Car
Two wheeler
Public transport
Two wheeler
Car
Public transport
Two wheeler
Two wheeler
Individual Auto
Walk
Individual Auto
Public transport
Public transport
Individual Auto
Two wheeler
Public transport
Share Auto
Public transport
Company bus/ Cab
Share Auto
Public transport
Two wheeler
Cycle
Share Auto
Car
Two wheeler
Two wheeler
Two wheeler
Public transport
Individual Auto
Individual Auto
Company bus/ Cab
Public transport
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Company bus/ Cab
Public transport
Company bus/ Cab
Walk
Two wheeler
Individual Auto
Car
Walk
Individual Auto
Walk
Company bus/ Cab
Car
Individual Auto
Public transport
Car
Public transport
Cycle
Car
Walk
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Car
Individual Auto
Car
Cycle
Individual Auto
Car
Individual Auto
Car
Public transport
Car
Individual Auto
Two wheeler
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Car
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Cycle
Walk
Walk
Two wheeler
Two wheeler
Two wheeler
Car
Walk
Public transport
Two wheeler
Car
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Public transport
Car
Individual Auto
Individual Auto
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Company bus/ Cab
Two wheeler
Public transport
Two wheeler
Public transport
Walk
Two wheeler
Public transport
Public transport
Two wheeler
Individual Auto
Public transport
Walk
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Public transport
Individual Auto
Car
Two wheeler
Company bus/ Cab
Two wheeler
Walk
Two wheeler
Walk
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Car
Two wheeler
Individual Auto
Car
Two wheeler
Individual Auto
Public transport
Two wheeler
Two wheeler
Car
Individual Auto
Cycle
Cycle
Car
Individual Auto
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Company bus/ Cab
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Walk
Two wheeler
Car
Car
Individual Auto
Car
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Walk
Two wheeler
Two wheeler
Individual Auto
Car
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Car
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Public transport
Car
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Walk
Cycle
Car
Walk
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Car
Two wheeler
Car
Cycle
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Car
Two wheeler
Individual Auto
Car
Two wheeler
Two wheeler
Cycle
Two wheeler
Walk
Two wheeler
Car
Car
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Walk
Two wheeler
Two wheeler
Car
Individual Auto
Individual Auto
Two wheeler
Car
Car
Individual Auto
Two wheeler
Public transport
Car
Public transport
Two wheeler
Two wheeler
Public transport
Car
Company bus/ Cab
Two wheeler
Two wheeler
Public transport
Car
Public transport
Public transport
Walk
Car
Two wheeler
Individual Auto
Two wheeler
Public transport
Two wheeler
Public transport
Car
Two wheeler
Two wheeler
Car
Two wheeler
Company bus/ Cab
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Public transport
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Car
Individual Auto
Two wheeler
Individual Auto
Car
Walk
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Car
Company bus/ Cab
Car
Car
Two wheeler
Two wheeler
Individual Auto
Company bus/ Cab
Individual Auto
Two wheeler
Two wheeler
Car
Car
Two wheeler
Public transport
Two wheeler
Company bus/ Cab
Two wheeler
Cycle
Two wheeler
Car
Individual Auto
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Public transport
Public transport
Individual Auto
Public transport
Public transport
Individual Auto
Car
Public transport
Individual Auto
Car
Car
Car
Walk
Two wheeler
Car
Public transport
Two wheeler
Car
Company bus/ Cab
Public transport
Walk
Cycle
Company bus/ Cab
Walk
Walk
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Public transport
Car
Cycle
Cycle
Public transport
Company bus/ Cab
Car
Car
Car
Walk
Cycle
Cycle
Car
Car
Public transport
Walk
Public transport
Car
Two wheeler
Public transport
Car
Car
Car
Walk
Two wheeler
Two wheeler
Individual Auto
Walk
Car
Two wheeler
Public transport
Car
Public transport
Public transport
Public transport
Two wheeler
Cycle
Individual Auto
Public transport
Walk
Individual Auto
Car
Public transport
Individual Auto
Individual Auto
Two wheeler
Public transport
Two wheeler
Two wheeler
Public transport
Public transport
Walk
Two wheeler
Car
Two wheeler
Public transport
Walk
Public transport
Individual Auto
Company bus/ Cab
Two wheeler
Car
Two wheeler
Walk
Two wheeler
Public transport
Company bus/ Cab
Public transport
Individual Auto
Two wheeler
Two wheeler
Public transport
Public transport
Individual Auto
Company bus/ Cab
Two wheeler
Individual Auto
Public transport
Two wheeler
Two wheeler
Car
Public transport
Two wheeler
Car
Car
Two wheeler
Two wheeler
Two wheeler
Car
Car
Car
Two wheeler
Car
Car
Car
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Car
Two wheeler
Two wheeler
Cycle
Cycle
Two wheeler
Cycle
Two wheeler
Two wheeler
Two wheeler
Car
Company bus/ Cab
Individual Auto
Individual Auto
Company bus/ Cab
Public transport
Two wheeler
Individual Auto
Two wheeler
Car
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
Public transport
Company bus/ Cab
Two wheeler
Car
Public transport
Public transport
Individual Auto
Two wheeler
Car
Public transport
Two wheeler
Car
Walk
Car
Two wheeler
Car
Public transport
Car
Two wheeler
Two wheeler
Cycle
Car
Car
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Company bus/ Cab
Two wheeler
Two wheeler
Car
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Company bus/ Cab
Car
Two wheeler
Individual Auto
Public transport
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Two wheeler
Individual Auto
Public transport
Public transport
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Cycle
Public transport
Two wheeler
Public transport
Car
Company bus/ Cab
Two wheeler
Car
Public transport
Two wheeler
Public transport
Two wheeler
Individual Auto
Individual Auto
Public transport
Two wheeler
Car
Individual Auto
Car
Walk
Individual Auto
Individual Auto
Two wheeler
Car
Public transport
Individual Auto
Two wheeler
Public transport
Public transport
Company bus/ Cab
Individual Auto
Car
Car
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Car
Individual Auto
Individual Auto
Car
Car
Walk
Cycle
Car
Two wheeler
Car
Two wheeler
Car
Individual Auto
Two wheeler
Car
Public transport
Two wheeler
Individual Auto
Cycle
Two wheeler
Two wheeler
Public transport
Car
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Individual Auto
Two wheeler
Two wheeler
Walk
Public transport
Individual Auto
Two wheeler
Individual Auto
Public transport
Car
Two wheeler
Walk
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Car
Car
Car
Two wheeler
Two wheeler
Public transport
Two wheeler
Car
Two wheeler
Car
Individual Auto
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Car
Two wheeler
Walk
Car
Cycle
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Walk
Public transport
Two wheeler
Company bus/ Cab
Two wheeler
Car
Cycle
Public transport
Company bus/ Cab
Two wheeler
Individual Auto
Two wheeler
Public transport
Walk
Two wheeler
Individual Auto
Walk
Car
Two wheeler
Two wheeler
Public transport
Individual Auto
Two wheeler
Car
Car
Car
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Public transport
Two wheeler
Car
Two wheeler
Car
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Public transport
Public transport
Public transport
Company bus/ Cab
Public transport
Public transport
Cycle
Two wheeler
Car
Two wheeler
Car
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Cycle
Two wheeler
Two wheeler
Public transport
Individual Auto
Public transport
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Car
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Walk
Two wheeler
Individual Auto
Cycle
Two wheeler
Two wheeler
Walk
Company bus/ Cab
Two wheeler
Two wheeler
Two wheeler
Public transport
Car
Two wheeler
Company bus/ Cab
Two wheeler
Walk
Public transport
Two wheeler
Two wheeler
Public transport
Public transport
Public transport
Public transport
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Company bus/ Cab
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Company bus/ Cab
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
Public transport
Public transport
Cycle
Car
Two wheeler
Public transport
Individual Auto
Public transport
Company bus/ Cab
Company bus/ Cab
Public transport
Public transport
Individual Auto
Individual Auto
Company bus/ Cab
Cycle
Public transport
Public transport
Individual Auto
Public transport
Walk
Individual Auto
Cycle
Individual Auto
Public transport
Individual Auto
Company bus/ Cab
Car
Company bus/ Cab
Individual Auto
Walk
Two wheeler
Public transport
Car
Public transport
Public transport
Cycle
Two wheeler
Walk
Individual Auto
Public transport
Individual Auto
Public transport
Public transport
Car
Cycle
Public transport
Public transport
Car
Public transport
Two wheeler
Public transport
Public transport
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Individual Auto
Public transport
Public transport
Two wheeler
Company bus/ Cab
Individual Auto
Public transport
Individual Auto
Individual Auto
Company bus/ Cab
Public transport
Two wheeler
Individual Auto
Company bus/ Cab
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Public transport
Car
Public transport
Two wheeler
Public transport
Public transport
Public transport
Cycle
Car
Car
Individual Auto
Car
Public transport
Cycle
Public transport
Company bus/ Cab
Public transport
Two wheeler
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Individual Auto
Public transport
Company bus/ Cab
Public transport
Public transport
Individual Auto
Public transport
Car
Individual Auto
Individual Auto
Public transport
Car
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Cycle
Public transport
Individual Auto
Walk
Public transport
Public transport
Public transport
Two wheeler
Cycle
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Two wheeler
Public transport
Public transport
Two wheeler
Two wheeler
Public transport
Public transport
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Public transport
Individual Auto
Company bus/ Cab
Two wheeler
Two wheeler
Individual Auto
Company bus/ Cab
Two wheeler
Individual Auto
Car
Public transport
Individual Auto
Individual Auto
Company bus/ Cab
Car
Individual Auto
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Public transport
Two wheeler
Individual Auto
Individual Auto
Public transport
Company bus/ Cab
Individual Auto
Public transport
Walk
Cycle
Car
Individual Auto
Public transport
Car
Company bus/ Cab
Two wheeler
Two wheeler
Public transport
Two wheeler
Cycle
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Car
Walk
Two wheeler
Public transport
Company bus/ Cab
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Car
Company bus/ Cab
Two wheeler
Public transport
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Cycle
Public transport
Public transport
Public transport
Company bus/ Cab
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Cycle
Car
Two wheeler
Two wheeler
Individual Auto
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Cycle
Public transport
Public transport
Car
Public transport
Two wheeler
Public transport
Two wheeler
Cycle
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Two wheeler
Individual Auto
Company bus/ Cab
Public transport
Car
Public transport
Car
Public transport
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Two wheeler
Car
Car
Car
Walk
Car
Individual Auto
Public transport
Company bus/ Cab
Public transport
Public transport
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
Car
Car
Walk
Car
Public transport
Public transport
Company bus/ Cab
Public transport
Public transport
Individual Auto
Car
Public transport
Individual Auto
Public transport
Car
Individual Auto
Public transport
Cycle
Company bus/ Cab
Public transport
Public transport
Individual Auto
Car
Public transport
Public transport
Individual Auto
Car
Public transport
Public transport
Two wheeler
Public transport
Public transport
Car
Public transport
Car
Car
Car
Car
Car
Two wheeler
Public transport
Car
Individual Auto
Car
Two wheeler
Car
Two wheeler
Two wheeler
Public transport
Individual Auto
Two wheeler
Car
Public transport
Individual Auto
Two wheeler
Public transport
Individual Auto
Public transport
Individual Auto
Individual Auto
Car
Car
Two wheeler
Two wheeler
Car
Two wheeler
''')

In [ ]:
# @title
data_string_shop_mode = ('''
Car
Two wheeler
Car
Public transport
Car
Individual Auto
Public transport
Car
Two wheeler
Car
Car
Public transport
Car
TNC or Cabs
Car
Two wheeler
Public transport
Car
TNC or Cabs
Car
Car
Two wheeler
Car
Public transport
Car
Car
Car
Car
Car
Individual Auto
Car
Public transport
TNC or Cabs
Public transport
Car
Car
Car
Public transport
Public transport
Individual Auto
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Car
Public transport
Car
Two wheeler
Car
Public transport
Car
Individual Auto
Public transport
Car
TNC or Cabs
Two wheeler
Car
Car
Two wheeler
Car
Public transport
Two wheeler
Public transport
Two wheeler
Individual Auto
Car
Two wheeler
Public transport
Public transport
Public transport
Two wheeler
Public transport
Individual Auto
TNC or Cabs
Public transport
Car
Public transport
Individual Auto
Public transport
Car
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Individual Auto
Car
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Public transport
Public transport
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Share Auto
Public transport
Public transport
Public transport
Public transport
Public transport
Car
Two wheeler
Two wheeler
Public transport
Car
Two wheeler
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Two wheeler
Public transport
Car
Public transport
Public transport
Car
Public transport
Car
Public transport
Two wheeler
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Two wheeler
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Two wheeler
Public transport
Share Auto
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
Share Auto
Car
Public transport
Car
Public transport
Car
Public transport
Public transport
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
Public transport
Public transport
Share Auto
Car
Cycle
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Car
Car
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Public transport
Car
Public transport
Two wheeler
Car
Individual Auto
Individual Auto
Public transport
Public transport
Two wheeler
Public transport
Public transport
Car
Individual Auto
Car
Car
Public transport
Public transport
Individual Auto
Car
Individual Auto
Public transport
Two wheeler
Public transport
Public transport
Public transport
Car
Public transport
Car
Public transport
Car
Public transport
Two wheeler
Public transport
Two wheeler
Car
Public transport
Public transport
Public transport
Car
Two wheeler
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Cycle
Public transport
Car
Car
Share Auto
Car
Two wheeler
Car
Car
Car
Individual Auto
Car
Public transport
Individual Auto
Car
Public transport
Car
Two wheeler
Public transport
Public transport
Individual Auto
Car
Public transport
Two wheeler
Public transport
Car
Two wheeler
Car
Two wheeler
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Car
Car
Two wheeler
TNC or Cabs
Car
Public transport
Car
Two wheeler
Car
Public transport
Two wheeler
Car
Car
TNC or Cabs
TNC or Cabs
Public transport
TNC or Cabs
Car
Two wheeler
Public transport
Public transport
Cycle
Individual Auto
Two wheeler
Car
Two wheeler
Individual Auto
Car
Car
Individual Auto
Two wheeler
Car
Public transport
Individual Auto
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Public transport
Two wheeler
Cycle
Two wheeler
Public transport
Public transport
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
Individual Auto
Public transport
Individual Auto
Car
Car
Two wheeler
Two wheeler
Car
Individual Auto
Car
Car
Car
Two wheeler
Individual Auto
Car
TNC or Cabs
Car
Individual Auto
Car
Public transport
Two wheeler
Car
Car
Individual Auto
Individual Auto
Individual Auto
Car
Individual Auto
Car
Car
Cycle
Car
Public transport
Two wheeler
Two wheeler
Car
Two wheeler
Individual Auto
Public transport
Two wheeler
Public transport
Public transport
Individual Auto
Public transport
Two wheeler
Cycle
Two wheeler
Two wheeler
Public transport
Car
Car
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Car
Cycle
Car
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Car
Car
Car
Public transport
Public transport
Public transport
Individual Auto
Public transport
Public transport
Two wheeler
Public transport
Cycle
Car
Car
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Car
Two wheeler
Individual Auto
Two wheeler
Public transport
Public transport
Individual Auto
Two wheeler
Public transport
Public transport
Car
Public transport
Car
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Two wheeler
Public transport
Individual Auto
Two wheeler
Public transport
Individual Auto
Two wheeler
TNC or Cabs
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
Two wheeler
Public transport
Public transport
Individual Auto
Public transport
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Public transport
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Public transport
Car
Public transport
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Share Auto
Share Auto
Two wheeler
Public transport
Two wheeler
Individual Auto
Share Auto
Public transport
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Car
Individual Auto
Two wheeler
Public transport
Public transport
Individual Auto
Public transport
Individual Auto
Car
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
TNC or Cabs
Public transport
Two wheeler
TNC or Cabs
Public transport
Car
Public transport
TNC or Cabs
Two wheeler
Car
Public transport
Two wheeler
Two wheeler
Individual Auto
Car
Public transport
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Cycle
Car
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Car
Car
Two wheeler
Two wheeler
TNC or Cabs
Car
Car
Individual Auto
Public transport
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Individual Auto
Public transport
TNC or Cabs
Car
Public transport
Public transport
Individual Auto
TNC or Cabs
Two wheeler
Car
Individual Auto
Car
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
TNC or Cabs
Car
Two wheeler
Car
Individual Auto
Two wheeler
Individual Auto
Car
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Car
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Car
Car
Two wheeler
Two wheeler
Car
Individual Auto
Individual Auto
Car
Two wheeler
Public transport
Cycle
Individual Auto
Car
Two wheeler
Individual Auto
Individual Auto
Public transport
Public transport
Two wheeler
Public transport
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Public transport
Car
Public transport
Public transport
Car
Car
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Public transport
Two wheeler
Car
TNC or Cabs
Car
Car
Individual Auto
Two wheeler
Public transport
Car
Public transport
Individual Auto
Individual Auto
Public transport
Two wheeler
Public transport
Car
Two wheeler
Car
Car
Individual Auto
Car
Car
Public transport
Public transport
Public transport
Car
Car
Car
Car
Car
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Two wheeler
Individual Auto
Individual Auto
Public transport
Car
Public transport
Two wheeler
Car
Car
Car
Two wheeler
Individual Auto
Public transport
Individual Auto
Two wheeler
Car
Individual Auto
Car
Car
Two wheeler
Two wheeler
Car
Public transport
Car
Public transport
Car
Public transport
Car
Individual Auto
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Two wheeler
Public transport
Two wheeler
Car
Two wheeler
Public transport
Individual Auto
Public transport
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Car
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Two wheeler
Public transport
Public transport
Public transport
Two wheeler
Public transport
Public transport
Individual Auto
Public transport
Public transport
Public transport
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Individual Auto
Two wheeler
Public transport
Individual Auto
Public transport
Individual Auto
Car
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
TNC or Cabs
Individual Auto
Individual Auto
TNC or Cabs
Public transport
Individual Auto
TNC or Cabs
Public transport
Car
Two wheeler
Car
Public transport
Car
Public transport
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
TNC or Cabs
Car
Two wheeler
Individual Auto
Two wheeler
Car
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Car
Individual Auto
Public transport
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Car
Car
Public transport
Two wheeler
Public transport
Car
Two wheeler
Individual Auto
Car
Individual Auto
Public transport
Public transport
Car
Car
Car
Two wheeler
Individual Auto
Public transport
Cycle
Public transport
Two wheeler
Individual Auto
TNC or Cabs
Car
Two wheeler
Individual Auto
Public transport
TNC or Cabs
Individual Auto
Individual Auto
Two wheeler
Car
Two wheeler
Individual Auto
Individual Auto
Cycle
Individual Auto
TNC or Cabs
Individual Auto
Public transport
Car
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Public transport
Public transport
Two wheeler
Two wheeler
Car
Car
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
TNC or Cabs
Individual Auto
Public transport
Individual Auto
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Individual Auto
Public transport
Two wheeler
Public transport
Individual Auto
TNC or Cabs
Car
Public transport
Public transport
Public transport
TNC or Cabs
Public transport
TNC or Cabs
Car
Two wheeler
Public transport
TNC or Cabs
Two wheeler
Cycle
Two wheeler
TNC or Cabs
Two wheeler
Public transport
TNC or Cabs
Public transport
TNC or Cabs
Two wheeler
Car
Car
Two wheeler
TNC or Cabs
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Car
Public transport
Public transport
Two wheeler
Individual Auto
Individual Auto
Car
Individual Auto
Two wheeler
Cycle
Public transport
TNC or Cabs
Two wheeler
Public transport
Public transport
Public transport
Car
Car
Public transport
Car
Car
TNC or Cabs
Two wheeler
Car
Individual Auto
TNC or Cabs
Car
Individual Auto
Cycle
TNC or Cabs
Car
Individual Auto
Public transport
Car
TNC or Cabs
Car
TNC or Cabs
Two wheeler
TNC or Cabs
TNC or Cabs
Car
Two wheeler
Two wheeler
Car
Cycle
TNC or Cabs
Car
Cycle
Two wheeler
Public transport
Two wheeler
Car
TNC or Cabs
Cycle
Car
Car
Two wheeler
TNC or Cabs
Public transport
Two wheeler
Cycle
Two wheeler
Car
Cycle
Public transport
Car
Car
Car
Two wheeler
Car
Car
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Car
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Car
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Individual Auto
Individual Auto
Individual Auto
Public transport
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Car
Two wheeler
Car
Two wheeler
Two wheeler
Individual Auto
Car
Car
Car
Public transport
Public transport
Car
Public transport
Car
Car
Public transport
Car
Car
Public transport
Car
Public transport
Car
Public transport
Car
Car
Public transport
Public transport
Car
Public transport
Individual Auto
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Car
Individual Auto
Two wheeler
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Car
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Cycle
Public transport
Car
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
TNC or Cabs
Public transport
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Individual Auto
Car
Two wheeler
Car
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Car
Public transport
Two wheeler
Two wheeler
Two wheeler
Car
Public transport
Car
Public transport
Public transport
Public transport
Public transport
Individual Auto
Public transport
Public transport
Car
Individual Auto
Car
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Car
Public transport
Individual Auto
Public transport
Public transport
Car
Individual Auto
Public transport
Public transport
Public transport
Car
Public transport
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
Car
Individual Auto
Car
Individual Auto
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Individual Auto
Public transport
Individual Auto
Car
Car
Public transport
Two wheeler
Public transport
Public transport
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
Public transport
Public transport
Public transport
Public transport
Two wheeler
Individual Auto
Public transport
Individual Auto
Public transport
Public transport
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Public transport
Individual Auto
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Car
TNC or Cabs
Individual Auto
Individual Auto
Public transport
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Public transport
Individual Auto
Car
Two wheeler
Individual Auto
Individual Auto
Public transport
Individual Auto
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Car
Individual Auto
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Car
Individual Auto
Public transport
Individual Auto
Two wheeler
Individual Auto
Car
Individual Auto
Public transport
Public transport
Public transport
Public transport
Car
Individual Auto
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Car
Public transport
Car
Car
Car
Car
Car
Car
Individual Auto
Car
Public transport
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Car
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
Public transport
Car
Public transport
Individual Auto
Public transport
Public transport
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Car
Car
Car
Car
Public transport
Car
Public transport
Public transport
Car
Individual Auto
Individual Auto
Car
Individual Auto
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Public transport
Individual Auto
Two wheeler
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Individual Auto
Individual Auto
Public transport
Public transport
Car
Public transport
Individual Auto
Public transport
Two wheeler
Individual Auto
Two wheeler
Public transport
Public transport
Public transport
Public transport
Two wheeler
Public transport
Car
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Two wheeler
Public transport
Public transport
Public transport
Individual Auto
Public transport
Individual Auto
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Individual Auto
Public transport
Individual Auto
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Public transport
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Public transport
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Public transport
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Public transport
Individual Auto
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Public transport
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Car
Individual Auto
Public transport
Individual Auto
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Public transport
Two wheeler
Car
Public transport
Public transport
Car
Public transport
Car
Individual Auto
Car
Public transport
Individual Auto
TNC or Cabs
Public transport
Car
Public transport
Car
TNC or Cabs
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Public transport
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Public transport
Share Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Cycle
Individual Auto
Individual Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Car
TNC or Cabs
Car
Public transport
Individual Auto
Public transport
Car
Public transport
Car
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Public transport
Public transport
Car
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Two wheeler
Car
Individual Auto
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Car
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Public transport
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Public transport
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
Public transport
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
Individual Auto
Public transport
Two wheeler
Individual Auto
Public transport
Two wheeler
TNC or Cabs
Two wheeler
Public transport
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Public transport
TNC or Cabs
TNC or Cabs
Public transport
Individual Auto
Individual Auto
Public transport
Car
Car
Public transport
Individual Auto
Individual Auto
Individual Auto
Public transport
Share Auto
Public transport
Public transport
Share Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Public transport
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Public transport
Individual Auto
Public transport
Public transport
Car
Public transport
Public transport
Car
Public transport
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Car
Cycle
Individual Auto
Car
Individual Auto
Individual Auto
Public transport
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Individual Auto
Car
Car
Individual Auto
Individual Auto
Public transport
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Public transport
Car
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Public transport
Two wheeler
Public transport
Car
Public transport
Public transport
Individual Auto
Public transport
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Car
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Two wheeler
Car
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Public transport
Public transport
Car
Public transport
Car
TNC or Cabs
Individual Auto
TNC or Cabs
Two wheeler
Car
Public transport
Two wheeler
Two wheeler
TNC or Cabs
Individual Auto
Cycle
Public transport
Individual Auto
Car
Individual Auto
Public transport
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
Two wheeler
Individual Auto
Car
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Car
Individual Auto
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
Public transport
TNC or Cabs
TNC or Cabs
Car
Car
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
Car
TNC or Cabs
TNC or Cabs
Individual Auto
Car
Individual Auto
Car
TNC or Cabs
TNC or Cabs
Car
Two wheeler
Public transport
Individual Auto
Car
Individual Auto
TNC or Cabs
Car
Two wheeler
TNC or Cabs
Car
Individual Auto
TNC or Cabs
Public transport
Car
Individual Auto
Cycle
Car
Public transport
Two wheeler
Car
Individual Auto
Car
TNC or Cabs
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
TNC or Cabs
Car
Individual Auto
TNC or Cabs
Cycle
Two wheeler
TNC or Cabs
Car
Two wheeler
TNC or Cabs
Car
TNC or Cabs
Car
TNC or Cabs
Two wheeler
Two wheeler
Cycle
TNC or Cabs
Individual Auto
Individual Auto
TNC or Cabs
Car
Two wheeler
Individual Auto
Individual Auto
TNC or Cabs
Car
Individual Auto
Two wheeler
TNC or Cabs
Car
Car
Individual Auto
TNC or Cabs
Car
TNC or Cabs
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Car
TNC or Cabs
Individual Auto
Car
Car
TNC or Cabs
Two wheeler
Public transport
TNC or Cabs
Individual Auto
Individual Auto
Two wheeler
Car
TNC or Cabs
Two wheeler
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Car
Two wheeler
Car
TNC or Cabs
Public transport
Car
Individual Auto
Individual Auto
TNC or Cabs
Public transport
Car
Individual Auto
TNC or Cabs
Individual Auto
Car
Individual Auto
Individual Auto
Car
Car
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Car
Public transport
Two wheeler
Public transport
TNC or Cabs
Public transport
Two wheeler
Car
TNC or Cabs
Public transport
Two wheeler
Car
TNC or Cabs
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Car
Car
Two wheeler
TNC or Cabs
Public transport
Two wheeler
Car
TNC or Cabs
Public transport
Two wheeler
TNC or Cabs
Car
Public transport
TNC or Cabs
Public transport
Public transport
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Car
Two wheeler
Two wheeler
Individual Auto
Public transport
Public transport
Individual Auto
Car
Public transport
Individual Auto
Car
Car
Car
Individual Auto
Two wheeler
Car
Car
Public transport
Car
Public transport
Public transport
Two wheeler
Individual Auto
Public transport
Individual Auto
Individual Auto
Public transport
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Car
Public transport
Cycle
Public transport
Individual Auto
Car
Car
Car
Individual Auto
Cycle
Cycle
Car
Car
Public transport
Two wheeler
Public transport
Car
Public transport
Public transport
Car
Car
Car
Public transport
Car
Individual Auto
Public transport
Public transport
Car
TNC or Cabs
Public transport
Car
TNC or Cabs
Individual Auto
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Public transport
Car
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Public transport
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Individual Auto
Public transport
TNC or Cabs
Individual Auto
Public transport
Two wheeler
Two wheeler
TNC or Cabs
Public transport
TNC or Cabs
Car
Car
TNC or Cabs
Car
TNC or Cabs
Car
Car
Car
TNC or Cabs
Car
Car
Car
Car
Car
Car
Car
Two wheeler
Car
Public transport
Car
Public transport
Two wheeler
Public transport
Public transport
TNC or Cabs
Cycle
Two wheeler
TNC or Cabs
Car
TNC or Cabs
Public transport
Car
Public transport
Public transport
Public transport
TNC or Cabs
Public transport
TNC or Cabs
Car
TNC or Cabs
Car
Public transport
Two wheeler
Car
TNC or Cabs
Public transport
TNC or Cabs
Two wheeler
Car
TNC or Cabs
Car
Individual Auto
Two wheeler
Individual Auto
Public transport
Public transport
Two wheeler
Car
Public transport
Public transport
Public transport
TNC or Cabs
Car
Public transport
TNC or Cabs
Car
Individual Auto
Car
TNC or Cabs
Car
Public transport
Car
TNC or Cabs
Individual Auto
Cycle
Car
Car
Car
Car
Two wheeler
Individual Auto
Two wheeler
Car
Car
Public transport
TNC or Cabs
Public transport
Car
TNC or Cabs
Car
Public transport
Car
Two wheeler
Car
Two wheeler
TNC or Cabs
Car
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Two wheeler
TNC or Cabs
Public transport
TNC or Cabs
Two wheeler
Individual Auto
Public transport
TNC or Cabs
Car
TNC or Cabs
Public transport
Two wheeler
TNC or Cabs
Individual Auto
TNC or Cabs
Public transport
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
Cycle
Individual Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Car
TNC or Cabs
Car
Public transport
TNC or Cabs
Public transport
TNC or Cabs
Individual Auto
Individual Auto
Public transport
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Individual Auto
Car
Public transport
Public transport
Car
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Car
Individual Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
Car
Individual Auto
Individual Auto
Car
Car
Individual Auto
Car
Individual Auto
Individual Auto
Cycle
Individual Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Car
Car
Individual Auto
Individual Auto
TNC or Cabs
TNC or Cabs
Car
Cycle
Individual Auto
TNC or Cabs
Individual Auto
TNC or Cabs
TNC or Cabs
Public transport
TNC or Cabs
Public transport
TNC or Cabs
TNC or Cabs
Two wheeler
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Individual Auto
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Car
Public transport
Two wheeler
Individual Auto
Car
Car
Car
Car
Car
Two wheeler
Car
TNC or Cabs
Two wheeler
Individual Auto
TNC or Cabs
Car
TNC or Cabs
Two wheeler
Car
TNC or Cabs
Two wheeler
TNC or Cabs
Car
TNC or Cabs
Car
TNC or Cabs
Public transport
TNC or Cabs
Cycle
Two wheeler
TNC or Cabs
TNC or Cabs
Car
Individual Auto
Two wheeler
TNC or Cabs
Individual Auto
TNC or Cabs
Car
Cycle
Public transport
Two wheeler
TNC or Cabs
Car
Two wheeler
Two wheeler
Public transport
Two wheeler
Car
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Public transport
Car
Two wheeler
Car
Car
Car
Car
Car
TNC or Cabs
Car
Car
Car
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Public transport
Two wheeler
Two wheeler
Public transport
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Cycle
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
Car
Individual Auto
TNC or Cabs
TNC or Cabs
Individual Auto
Cycle
Two wheeler
Two wheeler
Public transport
Car
Public transport
Two wheeler
Car
Car
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Cycle
Two wheeler
Public transport
Two wheeler
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
Two wheeler
Two wheeler
Public transport
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Car
Car
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Car
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Car
Two wheeler
Car
Car
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
Public transport
Individual Auto
Car
Car
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Individual Auto
TNC or Cabs
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
TNC or Cabs
Public transport
Cycle
Public transport
TNC or Cabs
Car
Public transport
Car
Two wheeler
Public transport
Two wheeler
Car
Two wheeler
Car
Public transport
Two wheeler
Public transport
Public transport
Two wheeler
Public transport
Public transport
Car
Public transport
Public transport
Car
Cycle
Public transport
Public transport
Car
Public transport
Individual Auto
Public transport
Two wheeler
Two wheeler
Two wheeler
Public transport
Car
Public transport
Public transport
Car
Public transport
Public transport
Car
Public transport
Car
Public transport
Car
Car
Public transport
Public transport
Two wheeler
Car
Car
Public transport
Car
Car
Public transport
Public transport
Car
Public transport
Individual Auto
Public transport
Public transport
Public transport
Public transport
Car
Car
Public transport
Individual Auto
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Individual Auto
Public transport
Car
Public transport
Public transport
Public transport
TNC or Cabs
Public transport
Public transport
TNC or Cabs
Public transport
Public transport
Car
Public transport
Public transport
Car
Public transport
Car
Car
Public transport
Public transport
Car
TNC or Cabs
Car
Car
Car
Car
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Public transport
Two wheeler
Cycle
Two wheeler
Public transport
Individual Auto
Public transport
Public transport
Public transport
Two wheeler
Public transport
Public transport
Individual Auto
Two wheeler
Public transport
Public transport
Individual Auto
Individual Auto
Public transport
Public transport
Public transport
Individual Auto
Public transport
Public transport
Car
Car
Car
Individual Auto
Car
TNC or Cabs
Car
Car
Car
TNC or Cabs
Car
Car
Car
Car
Car
Car
Car
Car
Individual Auto
Car
Car
Public transport
Public transport
Public transport
Car
Public transport
Car
Car
Car
Car
Two wheeler
Car
Two wheeler
Individual Auto
Public transport
Public transport
TNC or Cabs
Two wheeler
Individual Auto
Cycle
Individual Auto
Individual Auto
Public transport
Individual Auto
Public transport
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Cycle
Public transport
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Public transport
Public transport
TNC or Cabs
Car
Two wheeler
TNC or Cabs
Individual Auto
Two wheeler
TNC or Cabs
TNC or Cabs
Public transport
Two wheeler
Individual Auto
Car
Two wheeler
Car
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Cycle
TNC or Cabs
Two wheeler
Individual Auto
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Cycle
Car
Two wheeler
Two wheeler
Car
Car
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Car
Cycle
Individual Auto
TNC or Cabs
TNC or Cabs
Car
Two wheeler
Car
Individual Auto
Cycle
Individual Auto
Two wheeler
Car
Individual Auto
Car
Car
TNC or Cabs
TNC or Cabs
Public transport
Individual Auto
TNC or Cabs
Car
Individual Auto
Two wheeler
TNC or Cabs
Public transport
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Two wheeler
Car
Car
Car
Public transport
TNC or Cabs
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Individual Auto
Car
Public transport
Public transport
Car
Car
Public transport
TNC or Cabs
Car
Individual Auto
Public transport
Public transport
Public transport
Public transport
Car
Public transport
Public transport
Public transport
Car
Individual Auto
Public transport
Cycle
Public transport
Public transport
Public transport
Car
Car
Public transport
Car
TNC or Cabs
Car
Individual Auto
Public transport
Two wheeler
Public transport
Public transport
Individual Auto
Public transport
Car
Car
Car
Car
Car
Car
Public transport
Car
Public transport
TNC or Cabs
Public transport
Two wheeler
TNC or Cabs
Two wheeler
Public transport
Car
Two wheeler
Car
Public transport
Public transport
Two wheeler
Public transport
Public transport
Public transport
Public transport
Public transport
Car
Car
Car
TNC or Cabs
Car
Car
''')


In [ ]:
# @title
data_string_free_mode = ('''
Share Auto
Two wheeler
Car
Share Auto
TNC or Cabs
Individual Auto
Share Auto
TNC or Cabs
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
Car
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
Car
Two wheeler
Car
Car
Two wheeler
TNC or Cabs
Share Auto
Car
TNC or Cabs
TNC or Cabs
Share Auto
Car
Individual Auto
TNC or Cabs
TNC or Cabs
Two wheeler
TNC or Cabs
Share Auto
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
TNC or Cabs
Share Auto
Two wheeler
Car
Share Auto
TNC or Cabs
Individual Auto
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
Share Auto
Car
Two wheeler
Car
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Car
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
Two wheeler
TNC or Cabs
Individual Auto
Two wheeler
TNC or Cabs
Car
TNC or Cabs
Individual Auto
Share Auto
Car
Share Auto
Two wheeler
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Individual Auto
Car
Share Auto
Share Auto
Car
Share Auto
TNC or Cabs
Share Auto
Cycle
Cycle
Cycle
Cycle
Cycle
Two wheeler
Share Auto
Share Auto
TNC or Cabs
Two wheeler
Share Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Cycle
TNC or Cabs
TNC or Cabs
Share Auto
Share Auto
Share Auto
Two wheeler
Two wheeler
Share Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Car
TNC or Cabs
Share Auto
Car
Share Auto
Share Auto
Share Auto
Two wheeler
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Share Auto
Share Auto
Two wheeler
TNC or Cabs
Share Auto
Share Auto
Share Auto
TNC or Cabs
TNC or Cabs
Two wheeler
TNC or Cabs
Car
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
TNC or Cabs
Cycle
Share Auto
Individual Auto
Share Auto
Car
Share Auto
Car
Share Auto
Car
Share Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
TNC or Cabs
Share Auto
Individual Auto
Share Auto
Share Auto
Share Auto
Share Auto
Cycle
Share Auto
Share Auto
Share Auto
Car
Share Auto
Share Auto
Car
Car
Share Auto
Share Auto
Share Auto
TNC or Cabs
Share Auto
Two wheeler
TNC or Cabs
Car
Share Auto
Two wheeler
Car
Individual Auto
Individual Auto
TNC or Cabs
Share Auto
Two wheeler
TNC or Cabs
Share Auto
Car
Individual Auto
Car
TNC or Cabs
Share Auto
Share Auto
Individual Auto
Car
Individual Auto
Share Auto
Two wheeler
Share Auto
Share Auto
TNC or Cabs
Car
Share Auto
Car
Share Auto
Car
Share Auto
Two wheeler
Share Auto
Two wheeler
Car
Share Auto
Share Auto
Share Auto
Car
Two wheeler
Two wheeler
Share Auto
Two wheeler
Share Auto
Share Auto
Cycle
Share Auto
Car
Car
Share Auto
Car
Two wheeler
Car
Car
Car
Individual Auto
Car
Share Auto
Individual Auto
Car
Share Auto
Car
Two wheeler
Share Auto
TNC or Cabs
Individual Auto
Car
Share Auto
Two wheeler
Share Auto
Car
Two wheeler
Car
Two wheeler
Share Auto
Share Auto
Car
Share Auto
Share Auto
Share Auto
Car
Car
Two wheeler
Two wheeler
Car
Share Auto
Car
Two wheeler
Car
TNC or Cabs
Two wheeler
Car
Car
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Car
Two wheeler
TNC or Cabs
Share Auto
Cycle
Individual Auto
Two wheeler
Car
Two wheeler
Individual Auto
Share Auto
TNC or Cabs
Individual Auto
Two wheeler
Share Auto
Share Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Share Auto
Two wheeler
Cycle
Two wheeler
Share Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
Individual Auto
TNC or Cabs
Individual Auto
Car
Car
Two wheeler
Two wheeler
Car
Individual Auto
Car
Share Auto
Car
Two wheeler
Individual Auto
Car
Two wheeler
Share Auto
Individual Auto
Share Auto
Share Auto
Two wheeler
Car
Car
Individual Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Share Auto
Car
Cycle
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
Share Auto
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Share Auto
Share Auto
Individual Auto
Share Auto
Two wheeler
Cycle
Two wheeler
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Two wheeler
Two wheeler
Share Auto
Two wheeler
Two wheeler
Share Auto
Cycle
Share Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Cycle
Two wheeler
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Individual Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Cycle
Car
Car
Share Auto
Individual Auto
Individual Auto
Share Auto
Share Auto
Share Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Share Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Share Auto
Two wheeler
Individual Auto
Two wheeler
Share Auto
Share Auto
Individual Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Car
Share Auto
Two wheeler
Share Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Two wheeler
Two wheeler
Share Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Cycle
Individual Auto
Two wheeler
Share Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Individual Auto
Two wheeler
Share Auto
Share Auto
Individual Auto
Share Auto
Individual Auto
Share Auto
Share Auto
Individual Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Share Auto
Share Auto
Share Auto
TNC or Cabs
Share Auto
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Individual Auto
Share Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Share Auto
Individual Auto
Two wheeler
Share Auto
Share Auto
Individual Auto
Share Auto
Individual Auto
Share Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Two wheeler
TNC or Cabs
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
Car
Cycle
Two wheeler
Two wheeler
Individual Auto
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Cycle
Share Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Share Auto
Car
Two wheeler
Two wheeler
Two wheeler
Share Auto
Share Auto
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Share Auto
Cycle
Individual Auto
TNC or Cabs
Two wheeler
Cycle
Two wheeler
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Share Auto
TNC or Cabs
TNC or Cabs
Individual Auto
Two wheeler
Two wheeler
Share Auto
Individual Auto
Car
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Share Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
TNC or Cabs
Two wheeler
Share Auto
Individual Auto
Two wheeler
Individual Auto
Share Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Car
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Share Auto
Car
Two wheeler
Two wheeler
TNC or Cabs
Individual Auto
Individual Auto
Share Auto
Two wheeler
TNC or Cabs
Cycle
Individual Auto
Car
Two wheeler
Individual Auto
Individual Auto
TNC or Cabs
TNC or Cabs
Two wheeler
Cycle
Individual Auto
Individual Auto
Individual Auto
Two wheeler
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Share Auto
Car
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Share Auto
Two wheeler
Share Auto
Two wheeler
Share Auto
TNC or Cabs
Individual Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Share Auto
Two wheeler
TNC or Cabs
Share Auto
Two wheeler
Share Auto
Share Auto
Individual Auto
Share Auto
Share Auto
TNC or Cabs
Cycle
Cycle
Share Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Share Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Individual Auto
Share Auto
Car
Share Auto
Two wheeler
Car
Car
Car
Two wheeler
Individual Auto
Share Auto
Individual Auto
Two wheeler
Car
Individual Auto
Car
Car
Two wheeler
Two wheeler
Car
Share Auto
Car
Share Auto
Car
Share Auto
Car
Individual Auto
Share Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Share Auto
Two wheeler
Share Auto
Two wheeler
Share Auto
Two wheeler
Share Auto
Individual Auto
Share Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Share Auto
Share Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Share Auto
Share Auto
Share Auto
Two wheeler
TNC or Cabs
Share Auto
Individual Auto
Share Auto
Share Auto
Share Auto
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
TNC or Cabs
Share Auto
TNC or Cabs
Cycle
Two wheeler
TNC or Cabs
Share Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Share Auto
Share Auto
Individual Auto
Individual Auto
Two wheeler
Share Auto
Individual Auto
Share Auto
Individual Auto
Car
Individual Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Cycle
Two wheeler
Individual Auto
Individual Auto
Two wheeler
TNC or Cabs
Individual Auto
Two wheeler
TNC or Cabs
TNC or Cabs
Two wheeler
Share Auto
TNC or Cabs
Share Auto
TNC or Cabs
Individual Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Car
Two wheeler
Individual Auto
Two wheeler
Share Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Share Auto
Individual Auto
Share Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Share Auto
Share Auto
TNC or Cabs
Two wheeler
Share Auto
Share Auto
Two wheeler
Individual Auto
Share Auto
Individual Auto
Share Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Two wheeler
Individual Auto
TNC or Cabs
Cycle
Share Auto
Two wheeler
Individual Auto
Two wheeler
Share Auto
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Share Auto
Two wheeler
Individual Auto
Individual Auto
Cycle
Individual Auto
Two wheeler
Individual Auto
Share Auto
Share Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
Car
Car
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Two wheeler
Individual Auto
Share Auto
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Share Auto
Two wheeler
TNC or Cabs
Individual Auto
Two wheeler
Share Auto
TNC or Cabs
Cycle
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Share Auto
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Cycle
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Share Auto
Share Auto
Two wheeler
Two wheeler
Share Auto
Share Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Two wheeler
Individual Auto
Individual Auto
Share Auto
Individual Auto
Two wheeler
Cycle
Share Auto
Two wheeler
Two wheeler
Share Auto
TNC or Cabs
Share Auto
Share Auto
Car
TNC or Cabs
Share Auto
Car
Two wheeler
Two wheeler
Car
Individual Auto
Two wheeler
TNC or Cabs
Individual Auto
Cycle
Two wheeler
Share Auto
Individual Auto
TNC or Cabs
Car
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Two wheeler
Share Auto
Cycle
Two wheeler
Share Auto
Cycle
Two wheeler
Share Auto
Two wheeler
Share Auto
Two wheeler
Cycle
Share Auto
Share Auto
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Cycle
Two wheeler
Share Auto
Cycle
Cycle
Share Auto
TNC or Cabs
Share Auto
Two wheeler
TNC or Cabs
Share Auto
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Share Auto
TNC or Cabs
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Car
Share Auto
Share Auto
Share Auto
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
Share Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Share Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Individual Auto
Share Auto
Share Auto
TNC or Cabs
Share Auto
TNC or Cabs
Share Auto
TNC or Cabs
Share Auto
TNC or Cabs
Cycle
TNC or Cabs
Share Auto
Cycle
TNC or Cabs
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Share Auto
Share Auto
Share Auto
TNC or Cabs
TNC or Cabs
Individual Auto
Share Auto
Share Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Car
Individual Auto
Two wheeler
TNC or Cabs
Share Auto
Share Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Share Auto
TNC or Cabs
Share Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Cycle
Share Auto
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Share Auto
Share Auto
Cycle
Individual Auto
Share Auto
Two wheeler
Share Auto
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Share Auto
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Share Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Cycle
TNC or Cabs
Individual Auto
Share Auto
Share Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Share Auto
Individual Auto
TNC or Cabs
Individual Auto
Share Auto
Individual Auto
Share Auto
Individual Auto
Share Auto
Share Auto
Individual Auto
Share Auto
Share Auto
TNC or Cabs
Individual Auto
TNC or Cabs
Share Auto
Share Auto
Car
Share Auto
Individual Auto
Share Auto
Share Auto
Individual Auto
Share Auto
Car
Individual Auto
Car
Individual Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Two wheeler
Individual Auto
Share Auto
Individual Auto
Car
Car
Share Auto
Two wheeler
Share Auto
Share Auto
Cycle
Two wheeler
Share Auto
Share Auto
TNC or Cabs
TNC or Cabs
Share Auto
TNC or Cabs
Share Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Individual Auto
Share Auto
TNC or Cabs
Share Auto
Share Auto
Two wheeler
Individual Auto
Share Auto
Individual Auto
Share Auto
Share Auto
Two wheeler
Individual Auto
Share Auto
Share Auto
Individual Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
TNC or Cabs
Individual Auto
Two wheeler
Car
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
Two wheeler
Two wheeler
Share Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Car
Two wheeler
Individual Auto
Individual Auto
Share Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Share Auto
Individual Auto
TNC or Cabs
Two wheeler
Individual Auto
Individual Auto
Share Auto
Individual Auto
Two wheeler
Individual Auto
Share Auto
Share Auto
Individual Auto
TNC or Cabs
Individual Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Car
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Car
Share Auto
Share Auto
Car
Individual Auto
TNC or Cabs
Individual Auto
Two wheeler
Individual Auto
Car
Individual Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Share Auto
Individual Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Car
Share Auto
Car
Car
Car
Car
Car
Car
Individual Auto
Car
Share Auto
Individual Auto
Share Auto
Share Auto
TNC or Cabs
Individual Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
TNC or Cabs
TNC or Cabs
Individual Auto
Individual Auto
Share Auto
TNC or Cabs
Individual Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
Share Auto
TNC or Cabs
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
Car
Car
Share Auto
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Share Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
Share Auto
Individual Auto
TNC or Cabs
Individual Auto
Two wheeler
Individual Auto
Share Auto
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
TNC or Cabs
Share Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Two wheeler
Individual Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Share Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Individual Auto
Share Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Two wheeler
Two wheeler
Share Auto
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Individual Auto
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Individual Auto
Share Auto
Individual Auto
TNC or Cabs
Two wheeler
Share Auto
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Two wheeler
Share Auto
Share Auto
Two wheeler
Two wheeler
Share Auto
Two wheeler
Share Auto
Two wheeler
Share Auto
Two wheeler
Share Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Share Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
Share Auto
TNC or Cabs
Individual Auto
Share Auto
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Car
Individual Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Two wheeler
Cycle
TNC or Cabs
Two wheeler
Share Auto
Two wheeler
Car
Share Auto
Share Auto
Car
Share Auto
Car
Individual Auto
Car
Share Auto
Individual Auto
Two wheeler
Share Auto
Car
Share Auto
Car
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Share Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Share Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Cycle
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Share Auto
TNC or Cabs
Individual Auto
TNC or Cabs
Share Auto
TNC or Cabs
Share Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
TNC or Cabs
TNC or Cabs
Share Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Two wheeler
Car
Individual Auto
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Individual Auto
Share Auto
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
TNC or Cabs
Individual Auto
Individual Auto
Share Auto
Car
Car
Share Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
Individual Auto
Share Auto
Individual Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Share Auto
Share Auto
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Share Auto
Individual Auto
TNC or Cabs
Share Auto
Share Auto
TNC or Cabs
Cycle
Share Auto
TNC or Cabs
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Share Auto
Cycle
Individual Auto
Share Auto
Individual Auto
Individual Auto
TNC or Cabs
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Car
Share Auto
Individual Auto
Individual Auto
Share Auto
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Share Auto
Share Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Share Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
Car
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Two wheeler
Individual Auto
Individual Auto
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
TNC or Cabs
Car
TNC or Cabs
Share Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Car
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Cycle
Cycle
Individual Auto
Car
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Share Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Car
Car
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Car
Two wheeler
Two wheeler
Individual Auto
Car
Individual Auto
Car
Two wheeler
Two wheeler
Car
Two wheeler
Share Auto
Individual Auto
Car
Individual Auto
Two wheeler
Car
Two wheeler
Two wheeler
Car
Individual Auto
Two wheeler
Share Auto
Car
Individual Auto
Cycle
Car
Share Auto
Two wheeler
Car
Individual Auto
Car
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Share Auto
Individual Auto
Two wheeler
Cycle
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Cycle
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Car
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
TNC or Cabs
Car
Individual Auto
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Car
Two wheeler
Individual Auto
Share Auto
Car
Two wheeler
Two wheeler
Share Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Car
Two wheeler
Share Auto
Two wheeler
TNC or Cabs
Car
Individual Auto
Individual Auto
Two wheeler
TNC or Cabs
Car
Individual Auto
Two wheeler
Individual Auto
Car
Individual Auto
Individual Auto
Car
Share Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Car
Two wheeler
Share Auto
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Car
Two wheeler
TNC or Cabs
Two wheeler
Car
Share Auto
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Car
Two wheeler
Cycle
Two wheeler
Two wheeler
Car
TNC or Cabs
Two wheeler
TNC or Cabs
TNC or Cabs
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Individual Auto
TNC or Cabs
Share Auto
Individual Auto
TNC or Cabs
TNC or Cabs
Individual Auto
Share Auto
Share Auto
TNC or Cabs
Individual Auto
Two wheeler
TNC or Cabs
Car
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Two wheeler
Individual Auto
Share Auto
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Cycle
Cycle
Share Auto
Individual Auto
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
Cycle
Cycle
Share Auto
Car
TNC or Cabs
Two wheeler
TNC or Cabs
Share Auto
TNC or Cabs
TNC or Cabs
Share Auto
Share Auto
TNC or Cabs
TNC or Cabs
Car
Individual Auto
Share Auto
TNC or Cabs
Car
Two wheeler
TNC or Cabs
TNC or Cabs
Two wheeler
Individual Auto
TNC or Cabs
TNC or Cabs
Cycle
Share Auto
TNC or Cabs
TNC or Cabs
Individual Auto
TNC or Cabs
Individual Auto
Individual Auto
Individual Auto
Individual Auto
TNC or Cabs
Car
Two wheeler
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Car
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Two wheeler
Car
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Share Auto
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Car
Car
Two wheeler
Car
Two wheeler
Car
Car
Car
Two wheeler
Car
Share Auto
Car
Car
Car
Car
Car
Two wheeler
Car
TNC or Cabs
Car
TNC or Cabs
Two wheeler
Cycle
Cycle
Two wheeler
Cycle
Two wheeler
Two wheeler
Car
Two wheeler
TNC or Cabs
Car
TNC or Cabs
TNC or Cabs
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Car
Two wheeler
Car
TNC or Cabs
Two wheeler
Car
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
Individual Auto
Two wheeler
Individual Auto
TNC or Cabs
TNC or Cabs
Two wheeler
Car
TNC or Cabs
TNC or Cabs
TNC or Cabs
Two wheeler
Car
TNC or Cabs
Two wheeler
TNC or Cabs
Individual Auto
Car
Two wheeler
Car
TNC or Cabs
Car
Two wheeler
Individual Auto
Cycle
TNC or Cabs
Share Auto
Car
Car
Two wheeler
Individual Auto
Two wheeler
Car
Share Auto
TNC or Cabs
Two wheeler
TNC or Cabs
Car
Two wheeler
Car
Share Auto
Car
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Car
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Individual Auto
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Cycle
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
TNC or Cabs
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Car
TNC or Cabs
TNC or Cabs
TNC or Cabs
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Individual Auto
Car
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Individual Auto
Individual Auto
Car
Car
Individual Auto
Car
Individual Auto
Individual Auto
Cycle
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Car
Car
Individual Auto
Individual Auto
Two wheeler
Two wheeler
Car
Cycle
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
TNC or Cabs
TNC or Cabs
Two wheeler
Individual Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Car
Share Auto
Two wheeler
Individual Auto
TNC or Cabs
Car
Car
Car
Share Auto
Two wheeler
Car
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Car
Two wheeler
Share Auto
Two wheeler
Cycle
Two wheeler
Two wheeler
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Share Auto
Cycle
TNC or Cabs
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Car
Two wheeler
Share Auto
Car
TNC or Cabs
Car
Car
Two wheeler
Car
Car
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
TNC or Cabs
Two wheeler
Individual Auto
Two wheeler
Two wheeler
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Cycle
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Individual Auto
Two wheeler
Two wheeler
Individual Auto
Cycle
Two wheeler
Two wheeler
TNC or Cabs
Car
TNC or Cabs
Two wheeler
TNC or Cabs
Car
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Cycle
Two wheeler
Share Auto
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Share Auto
TNC or Cabs
Two wheeler
Two wheeler
TNC or Cabs
Share Auto
Two wheeler
TNC or Cabs
Share Auto
Share Auto
TNC or Cabs
Share Auto
Car
Car
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Car
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Car
Two wheeler
Car
Car
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
TNC or Cabs
TNC or Cabs
Individual Auto
Car
Car
Share Auto
Share Auto
TNC or Cabs
Share Auto
Share Auto
Share Auto
Individual Auto
Two wheeler
Share Auto
Share Auto
Cycle
Share Auto
Share Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Cycle
Share Auto
Two wheeler
Car
TNC or Cabs
Car
Two wheeler
Share Auto
Two wheeler
Car
Two wheeler
Share Auto
Share Auto
Two wheeler
Cycle
TNC or Cabs
Two wheeler
Share Auto
TNC or Cabs
Car
TNC or Cabs
TNC or Cabs
Car
Cycle
Share Auto
TNC or Cabs
Car
Share Auto
Individual Auto
Share Auto
Two wheeler
Two wheeler
Two wheeler
Share Auto
Car
Share Auto
Share Auto
Car
Share Auto
TNC or Cabs
Car
Share Auto
Car
Share Auto
Car
Car
Share Auto
Share Auto
Two wheeler
Car
Car
Share Auto
Car
Car
Share Auto
Share Auto
Car
Share Auto
Individual Auto
TNC or Cabs
Share Auto
Share Auto
Cycle
Car
Share Auto
Share Auto
Individual Auto
Share Auto
Cycle
Car
Share Auto
Share Auto
Share Auto
Individual Auto
Share Auto
Car
Share Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Car
Share Auto
TNC or Cabs
Car
Share Auto
Car
Car
Share Auto
TNC or Cabs
TNC or Cabs
Two wheeler
Car
Car
Car
Car
TNC or Cabs
Share Auto
Cycle
Share Auto
Share Auto
Share Auto
TNC or Cabs
Share Auto
TNC or Cabs
Two wheeler
Cycle
Two wheeler
TNC or Cabs
Individual Auto
TNC or Cabs
TNC or Cabs
Share Auto
Two wheeler
TNC or Cabs
TNC or Cabs
Individual Auto
Two wheeler
TNC or Cabs
TNC or Cabs
Individual Auto
Individual Auto
Share Auto
TNC or Cabs
Share Auto
Individual Auto
TNC or Cabs
TNC or Cabs
Car
Car
Car
Individual Auto
Car
Two wheeler
Car
Car
Car
Two wheeler
Car
Car
Car
Car
Car
Car
Car
Car
Individual Auto
Car
Car
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
TNC or Cabs
Car
Car
Car
Car
Two wheeler
Car
Two wheeler
Individual Auto
TNC or Cabs
TNC or Cabs
Two wheeler
Two wheeler
Individual Auto
Cycle
Individual Auto
Individual Auto
TNC or Cabs
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Cycle
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Individual Auto
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
TNC or Cabs
Two wheeler
Car
Two wheeler
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Share Auto
Two wheeler
Individual Auto
Car
Two wheeler
Car
Two wheeler
Individual Auto
Two wheeler
Individual Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Share Auto
Cycle
Two wheeler
Two wheeler
Individual Auto
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Cycle
TNC or Cabs
Two wheeler
Two wheeler
Car
Car
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Car
Cycle
Individual Auto
Two wheeler
Two wheeler
Car
Two wheeler
Car
Individual Auto
Cycle
Individual Auto
Two wheeler
TNC or Cabs
Individual Auto
Car
Car
Two wheeler
Two wheeler
TNC or Cabs
Individual Auto
Two wheeler
Car
Individual Auto
Two wheeler
Two wheeler
Share Auto
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Two wheeler
Car
Car
Individual Auto
Individual Auto
Individual Auto
Individual Auto
Two wheeler
Individual Auto
Two wheeler
TNC or Cabs
Share Auto
Share Auto
TNC or Cabs
Two wheeler
Share Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Individual Auto
Car
Share Auto
Share Auto
Share Auto
Share Auto
TNC or Cabs
Two wheeler
Car
Individual Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
Individual Auto
Share Auto
Cycle
Share Auto
Share Auto
Share Auto
Car
TNC or Cabs
Share Auto
Car
Two wheeler
Car
Individual Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Individual Auto
Share Auto
Car
Car
TNC or Cabs
Car
Share Auto
Car
Share Auto
TNC or Cabs
TNC or Cabs
Two wheeler
TNC or Cabs
Two wheeler
Two wheeler
Two wheeler
TNC or Cabs
Car
Two wheeler
Share Auto
Share Auto
Share Auto
Two wheeler
Share Auto
Share Auto
Share Auto
Share Auto
Share Auto
TNC or Cabs
Car
Car
Two wheeler
Car
Car
''')

In [ ]:
# @title
time_based_constraint = ('''
4.3
14.3
15.4
35.1
15.3
13.6
19.8
19.8
26.2
13.3
9.2
34
36.7
46.2
43.8
3.9
1.1
3.5
6.3
6.3
7.5
8
7.8
24.1
9.5
8.7
8.9
1.8
5.4
5.4
14.7
27.7
27.7
44.8
44.8
8.3
28.8
18.1
3.1
15.4
7.3
41.4
34.1
32.9
35.1
13.5
54.4
47.3
46
38.2
9
18
29
27.7
27.7
10.2
27.9
27.9
35.4
35.4
16.4
43.1
45.8
39
39
9.6
7.3
11.2
12.9
8.6
2.9
8.3
6.5
7
7
13.9
53.3
56.3
41.3
41.3
8.3
18.6
23
23.6
11.8
8.9
15.9
15.9
32.2
32.2
4.1
11.9
10
22.1
9.1
11.1
15.8
11.4
33.1
22.7
12.6
8.5
8.5
25
25
3.5
9.3
11.9
20.4
35.9
5.4
11.5
16.2
43.5
17.8
9.9
16.8
13.4
30.1
26.4
2.3
16.9
7.4
14.7
35.4
8.1
14
2.9
11
23.9
8.9
19.1
10.4
37.5
37.5
13.8
14.1
14.8
15.4
18.7
13.8
4.8
6.9
4.6
35.3
9.9
13.2
13.2
9.3
26.7
2.6
18.4
18.4
25.5
35.1
4.4
6.2
7.1
4.4
2.6
10.2
15.5
14.6
16.4
16.4
8.8
15.3
12.6
34.5
32.5
7.6
43.8
37.4
37.3
22.1
8.7
17.2
14.3
34.1
12.9
3
38
38
25.3
25.3
7.7
47.2
35.1
32.8
22.5
11.7
27.1
26.4
16.7
34.2
13.5
12.1
16.2
23.5
7.4
5.5
5.3
8.3
35.7
1.1
10.2
26
26
40
23.6
6.9
6.3
9.2
26
26
4.5
18.6
18.6
26.9
26.9
0.7
9.8
8.2
34.5
11.2
7.5
39.6
36.7
45.7
26.7
8.9
15.1
17.9
31.7
28.1
11
19.4
14.6
16.6
22.3
8.7
9.2
14.7
5.4
23.1
9
3.1
8.8
3
21.4
6.9
7.8
10.6
40.6
5.4
9.8
13.4
13.4
26.3
24.5
12.5
11.5
3.4
3
3
6.2
21
21.6
47.9
17.5
8.3
20.5
20.1
13.3
16.8
11
7.8
16.4
8.6
48.2
11.7
16.3
20.1
10.4
26.7
14.3
25.5
22.4
21.1
20.1
5.7
18.2
15.3
14.7
15.1
8.7
11.2
17.8
32.4
26.7
12.8
10.2
10.2
4.9
4.9
9.5
5.9
13
10.3
45.5
7.3
10.3
6.2
5.6
5.6
6.1
7.3
7.6
24.8
19.2
14.9
7.2
11.7
5.3
21.3
9.3
28.8
27
24.5
24.5
6.9
16.1
21.8
19.6
13.6
3.8
20.9
15.5
21.5
21.5
11.7
22.9
16.6
30.6
12
7.3
12.2
9.5
20.9
12.1
8.1
3.7
3.7
38
16.4
2.8
17.6
19.7
14.9
26.4
10.7
21
15.8
16.1
16.1
4.2
3.6
12.3
30.6
33.4
8.7
17.7
17.7
8.9
8.9
8.4
10.2
10.2
11.5
28.2
2
25.2
18.6
21.8
25.6
12.3
5
6
5.5
5.5
11
18.8
13.6
27.6
8.5
11.9
6.6
1.7
1.3
5.6
11
50.1
40.4
40.3
49.8
13.1
21.5
17.8
26.9
12.2
8.3
11.7
3.8
2.5
38.1
11.5
8.6
7.4
7.9
31.7
2
5.9
7.2
23.8
23.8
15
19.3
22.1
3.7
17.6
11.3
26.9
26.9
24.6
28.8
10
11.5
12.8
10.3
7.5
4.2
17.3
14
14.2
17.9
11.1
17
7.8
7.6
7.6
9.4
11.2
16.8
30
10.1
8.6
14.3
11.4
31.1
27
12
10.9
13.3
30
27
2.1
5.8
6.8
13.9
41.4
13.3
11.6
7.9
6.7
20.7
7.7
7.2
4.3
24.9
12.5
10.1
9.9
12.8
6.9
23.2
7
9.3
11
34.2
28.4
7.2
6.4
9
3.1
25.1
10
15.8
10.5
7.5
30.2
8.5
16.4
16.4
25.6
29.1
3.5
7.7
7.7
17.4
40.3
8.1
18.2
21
24
45.1
1.3
20.8
29.8
25.7
46.2
11.3
7.1
14.7
9.3
31.5
10.8
15.2
15.2
8.9
8.9
13
21
15.1
13.5
38.7
10.5
17.3
20.4
13.8
13.8
5.8
19.6
18.8
13.3
17.4
7.5
28.3
21.2
21.3
21.3
14.8
17.8
19.8
10.9
24.3
9
22.6
22.6
18.6
20.1
8.9
6.8
6.8
5.2
20.5
1
24.3
24.3
47.5
25
10.8
13.4
15.1
31.2
24.5
9.1
9.9
15.3
33.7
8.6
6.7
44.1
35.9
25.2
31.5
11.4
15.8
6.8
10.7
20
11.5
38
40.1
45.9
45.9
15
28.4
36.2
28.6
43
9.9
17.5
6.6
8.5
29.5
10.5
11.5
8.5
22.1
32.6
8.7
11.7
11.1
27.7
13
8.3
21.1
15.9
14.6
19
6.2
19.4
19.5
32.1
48.9
13.1
18.5
22
12.7
29.8
12.7
18
20.7
11.5
11.5
5.5
10.1
6.3
39.4
19.3
6.8
7.1
7.1
24.6
24.6
8.9
14.6
14.6
5.7
45
7.1
17
17.5
15.9
28.8
11.4
15.5
20.2
28.4
9.5
3.7
13.8
10.2
30.8
30.8
13.7
35.9
35.9
7.5
47.6
9.1
13.5
13.5
33.4
23.7
13.4
22.3
21.1
30
24.9
5.5
9.8
4
29.7
29.7
4.4
5.1
7.2
38.7
17.3
11
32
35
48.1
40.4
12.5
7.8
6
25.4
18.4
12.7
18.8
21.6
13.8
13.8
8.7
4.5
13.1
11.1
46.5
7.6
23.9
29.5
5.9
19
14.9
23.7
21
32.3
18.6
13.3
11.4
6.6
15.8
41.9
2.3
3.9
8.7
8.5
6.1
8.8
2.2
12
5.8
1
6
5.4
2.6
17
37.4
11.5
6.6
6.6
30.8
39.7
9.3
18.8
31.4
36.6
19.9
9.1
9.6
10.1
5.8
27
14.2
13.9
15.7
10.4
10.4
15.5
33.7
33.7
31.8
35.4
6
8.5
4.1
34.6
34.6
7.9
15
19.7
17
31.1
12
1
11.8
20.6
6
6.4
6.5
5.9
11.6
43.4
8.3
22.8
32.6
21.2
30.2
8.9
11.8
15.1
31.2
30.5
8.1
9.3
9.6
24.8
28.6
10.7
23.4
24.5
21.5
31.9
10.2
6
6
8.2
6.6
12.9
18.9
20.3
30.1
13.3
4.4
4.9
8.1
3.5
17.5
4.6
1.7
10.4
7.6
7.6
5.8
14.2
11.5
18.2
28.1
10.8
29
24.2
33.9
26.8
5.6
24.7
21.5
25.8
25.8
2.6
30.8
30.8
19.6
45.3
6.9
18.4
25.8
43.3
43.3
10.7
31.5
28.7
26.8
15.4
3.1
16.6
11.6
13.1
18
7.5
1.6
11.4
10.2
33.6
7.4
14.2
18.2
12.6
12.6
15.9
27.3
27.6
22.3
36.5
10
12.9
12.9
6
21.9
2.1
12.4
12.4
31.3
19.6
10.6
37.3
35.1
40.7
25
6.2
3.9
10
5.7
5.7
14.7
12.9
14.5
9.9
25.6
11.1
14.2
12.7
9
23.4
14.6
7.7
11.8
20.6
3.6
11.3
12.3
12.3
28.7
28.4
9.4
9.7
9.4
29.5
6.1
5.8
15.7
11.4
43.3
32.3
9.4
16.3
14.6
22.5
18.2
6.9
18.6
19.3
49.9
31.2
7.6
11.3
16.1
9.5
14.3
3.2
7.5
7.5
13.7
10.4
5
5.8
7.5
26.5
6.4
14.8
22.1
21.2
29.4
29.4
9.9
13.7
13.7
11.9
21.5
11.9
17.1
16.9
29.8
30.4
8.9
16.5
19.3
25.2
10.2
3.6
7.8
6
42.2
42.2
10.5
32.7
32.7
49.2
31.2
8.3
15.7
15.7
38.3
18.4
3.5
16.1
9.1
48.1
12
14.4
4.1
5.8
4.2
3.7
13.3
37.5
37.5
28
31.8
6.9
6.8
4.2
37.7
16.6
11.4
14.8
11.6
9.4
23.4
9.7
21.4
21.9
31
19.6
12
16.8
14.6
10.7
26.5
8.1
10.9
13.7
5.8
9.9
5.5
24.6
18.9
18.8
40.4
8.3
13.7
10.3
27.5
30.9
8.7
3.9
9.4
17
37.7
11
33.5
21.2
24.3
21.5
7.7
6
8.5
20.4
3.4
9.4
18.2
12.6
28.4
13.7
9.4
16.3
13.4
9.2
23.1
9.7
18.2
13.7
9.1
15.7
3.3
3.5
3.5
27.6
6.7
4.2
5.7
5.3
2.9
2.6
5.5
4.1
2.9
5
5
1.9
15.2
14.9
34.5
13.3
1.7
18.4
9.7
40.4
34.5
5.6
25.1
25.1
26.8
26.8
12.1
19.3
21.6
12.4
12.4
8.7
11.7
5.3
5.9
32.1
12.1
49.8
49.4
38
42.2
12.8
17.2
13.7
24.7
10.3
6.3
10
9.4
20.3
34.4
9.9
12.8
15.1
8.3
11.4
3.5
3.2
9.9
28
37
9.7
22.6
22.5
20
31.8
11.5
18.9
18.9
26.7
26.7
14.7
30.7
24.6
15.5
39
23
18.6
12.8
9.6
11.2
14.2
8.8
6.3
14.2
24.1
7.8
7.1
8.6
20.3
34.6
10.6
29.4
28.9
19.9
36.6
9.1
15.5
13.2
9.6
9.6
4.7
17.5
18.1
14.6
14.6
9.1
3.9
8.9
17.8
21.6
11.1
18.8
18.8
11.5
25.7
8.1
24.4
17.7
21.1
21.1
6.6
32.6
43.8
30.6
30.6
4.7
5
5.2
6
16.1
6.3
4.4
11.1
31
6.7
8.7
20.6
20
17.4
15
8.5
29.2
19.1
35.5
32.3
3.3
7.6
7.6
17.1
6.7
4.8
8.3
8.3
5.4
2.8
16.1
27.6
31.8
22.2
20.1
10.1
16.7
13.1
9.8
7.4
11.8
17
17
15
15
13.7
5.6
8.6
2.7
36.4
11.4
28.1
27.1
13.5
41.1
7.8
9.3
9.2
3.6
21
3.2
4.6
7.8
27.9
5.1
7
31.3
22.6
20.4
7.9
12.5
8.3
3
5.9
2
3.3
2.7
9.7
38
18
11.2
10
7
1.8
21
3.8
11.4
11.3
12.1
12.1
1.6
10.2
10.2
7.3
42.4
4.2
9.5
7.9
42.5
7.5
8.6
13
14.5
17
15
6.8
19.8
19.6
14.7
9.8
6.5
20.3
21.1
17.2
18.7
3.1
7.5
5.6
15.2
15.2
14.5
14.1
13.8
13.8
16.3
6.2
4.4
10.3
22.8
41.6
14.8
30.5
30.5
27.8
13.9
5.5
9.2
13.8
18.8
41.3
15
30
30
13.4
30
5.9
16.1
22.8
19.1
14.6
4.9
6.5
6.5
16.3
4.1
4.3
8.1
2
3.1
37.5
3.5
25
37.6
28.3
41.4
6.8
14
14
48.3
13.8
1.1
6.6
1.7
43.9
8.5
2.5
23.7
23.7
49.7
40.5
6.1
9.7
12.3
29.6
37.4
5.9
16.7
25.3
33.2
38.2
13.7
4.9
5.9
2.5
23.2
10.5
10.1
14.4
3.8
3.8
13.6
38.1
32.5
32.2
32.2
7.1
7.1
7.3
35.4
24.8
4.8
4.3
6.8
15.9
25.3
12.2
52.1
54.9
35.8
39.7
10.9
33.5
33.5
23.4
39.5
10.5
14.5
12.5
29.7
25.3
4
6.1
6.7
38.3
3.7
12.2
2.2
1.3
40.7
13.7
9.8
41.3
32.9
27.2
46.4
13.8
29.4
28
14.2
14.2
8.6
17.5
26.3
40.4
40.4
11.6
16.9
16.9
23.1
9.7
9.4
23.8
23.8
14.9
5.6
9.3
25
21.2
32.9
13.8
10.2
11.2
11.2
22.1
6.4
12.7
32.5
26.7
41.7
18.1
9.5
7.6
17.4
34
26.1
12.2
6.5
3.4
5.1
27.6
7.5
25.3
14.1
33.2
40.5
8.7
13
13
32.8
7
14.2
18.9
18.9
23.3
12.4
11.3
13.7
11.6
22.9
8.2
7.9
46.6
35.4
38.2
24.4
7.3
34.3
46.6
24
37.2
4.1
8.2
11.1
11.2
15.8
10.7
21.3
19.6
37.4
31.2
7.1
4.9
4.9
3.5
3.5
12.7
12.5
14.6
11.1
25.5
1.3
3.3
12
5.7
9.5
6.6
7.8
9.3
28.3
32.8
9.5
29.9
18.8
45.5
26.7
12.7
8.6
8.4
24.9
2.8
13.6
41.8
52.3
36.4
41
11.1
18.3
16.7
31.7
24.9
5.8
32.8
33.5
40.1
16.7
30.1
23.4
18.2
18.2
34.4
14.8
41.7
41.7
31.1
35.4
5.4
4.1
5
23.4
31.8
8.5
19.6
25.4
22.8
31.7
14.5
15.8
8.8
25.3
11.8
8.4
21
15.4
17.2
15.2
1.2
12.7
12.1
32.3
36.7
47.2
18.7
17.3
26.7
13.4
12.2
21.7
21.7
31.8
14.7
11.7
37.2
39.2
28.9
26.5
8.1
6.9
7.1
3.6
16.3
9.8
22.1
22.1
12.8
19.2
7.7
11
1.5
4.5
15.8
12.3
8.3
4.5
35.8
35.8
10.2
14.5
9.5
7.1
7.1
15.6
24.9
24.1
32.7
26
7.4
44.8
42.1
31.4
35.8
13.3
12.1
1.4
22.9
5.6
14.7
34.1
36.9
29.2
29.2
2.7
12.9
10.8
25.2
38.3
5.9
24.2
24.2
15.3
32
7.2
7.4
7.4
3.2
26.3
3.8
10.6
9.1
7.1
16.6
9.5
4.5
4.5
19.5
10.3
2.8
5.9
3
7.4
23.5
13.1
5.5
7.5
20
24.1
4.4
31.1
31.1
26.8
40.7
12.8
1.1
7.2
39.6
39.6
6
6.4
13.2
32.2
11.8
10.6
19.6
19.1
14.7
14.7
12.3
5.1
5.1
2.5
37.3
5.1
0.8
7.9
18.8
3.1
5.1
14.1
12.8
46.1
28.6
6.9
3.2
3.2
3.9
1
11.9
6.9
13.9
10.1
10.1
10.6
16.2
5.8
10.6
45.7
2.9
8.4
10.7
37.4
7.5
6
10.3
9.7
11.3
41.7
7.7
9.1
12.7
25.6
25.6
6.6
26.6
17.1
15.6
18.1
14.3
2.3
9
4.6
24.4
14.7
7.3
4.4
4.5
24.9
8.9
11.1
11.1
12
30.1
8.9
7.6
12
5.5
35.1
9.7
11.1
13.3
26.4
7.2
18.3
54.8
47.7
5.2
5.2
10.6
13.6
11.8
25.1
10.4
8.6
20
20
30.6
30.6
5.6
3.9
7.8
2.5
21.6
12.9
7.7
5.8
26.1
4.5
4.8
9.2
3.8
37.2
27.5
10.9
16.4
19.4
27.6
14.5
12.2
8
9.3
4.1
25.9
10.4
12.4
18.2
8.7
26.1
6.3
17.4
17.5
17.1
34.3
7.9
9.6
18.1
8.1
13
7.5
11.9
11.9
23.1
23.1
14
9.3
9.3
7.3
14.4
15.7
49.6
46.8
43.8
44.2
4
26.2
29
26.2
28.4
12.4
21
18.8
30.8
24.3
13.3
24.4
24.4
19.1
15.6
9
9.7
10.7
30.9
5.7
11
7.3
13.6
39.7
32.1
6.1
17.1
17.1
17.1
47.6
5.7
13.5
17.5
45.2
45.2
7.6
9.3
3.5
3.6
22.6
7.6
9.2
1
20.4
41.2
7.9
7.5
7.5
7
4.5
8.3
10.2
11.2
26
8.1
8.4
5
5
7.6
42.9
8.8
15.5
21.9
13.2
17.4
4.2
7.6
0.5
39.1
39.1
4.2
4.8
16
25.4
25.4
9.7
6.9
6.9
6.9
48.5
10.6
16.9
10.7
27.3
27.3
9.7
11.7
10.7
24
25.7
3.9
3.3
5.8
16.7
4.9
6.9
10.9
6.4
7.5
12.4
2.2
4
8.1
2.9
41.7
10.1
6.1
10.2
7.7
12
7.5
22.3
23.3
19
11.9
10.8
13.1
14.8
11.5
25.4
11
14.2
14.9
12.2
28.4
4
2.5
2.5
7.4
7.4
14.2
40.9
39.1
30.5
29.6
13.4
17.6
19.5
7.3
7.3
5.5
24
12.8
12.7
16.2
8.8
5.8
14.5
19.7
7.7
3
3.8
2.8
18.5
5
7.5
10.7
15.3
5.4
24.4
12.3
14.9
16.6
9.8
27.2
11.3
33.5
25.2
22.7
22.7
5.5
11.9
11.3
7.8
32.9
14.2
6.4
8.7
22.3
4.3
14.8
28.9
23.8
22.8
19.1
6.8
4.3
9.4
1.7
4.4
14.3
48.1
45.3
46.7
40.6
14.9
17.3
21
16.9
16.9
11.6
23.6
26.4
23.8
26.9
7
9.8
1.5
11.4
43.2
9
9.5
11.2
22.5
22.5
8.8
11.5
11.5
8.5
10.1
13.9
20.1
19.6
14.2
25.1
9.8
15.1
12.3
6.7
25.7
9.4
14.3
6.5
10.3
10.3
10.3
10.1
14.5
34.1
7.8
9.4
49
49
35.9
35.9
12.7
8.5
8.5
25.3
22.4
8.3
7.2
9.2
8.5
8.5
7.4
21.8
22.7
20
32.1
4.9
4.6
13.3
30.3
32.1
11.5
15.1
15.3
25.6
10.1
12.9
3.4
3.4
38.2
4
12.8
6
3.3
3.1
3.1
5.4
33
25.3
23.5
42.4
14.6
19.3
18.7
15.6
33.1
3.5
21.7
30.4
23.7
43.6
6.1
9.6
7
42.2
5.7
3
17
8.6
49.9
21.8
15
50.4
47.7
2
36.2
7
12.4
12.1
9.8
11.7
9
15.2
15.2
8.9
30.9
13.6
11.7
8
23.4
20.1
9.5
13.7
11.9
34.8
9.9
13.6
16.6
14
9.6
31.3
9.5
8.9
8.9
6.2
6.2
4.4
14.4
12.7
14.9
14.9
3.8
10.7
10.7
4.7
28.7
8.5
6.9
11.6
9.1
48
12.3
16.1
17.5
28.2
28.2
8
2.3
3.7
4.5
4.5
14.9
1.1
7.6
14.7
0.5
7.1
8.2
7.7
10.8
29.3
3.3
7
4.2
4.6
39.5
6.8
26.5
36.4
14.1
24
12.9
43.8
43.8
41
45.5
7.4
2.7
3.6
29.9
33
5.9
6.4
14.9
30.8
10.4
13.7
6.3
8.9
17.5
36.9
8.8
14.1
11.5
10.3
6.6
4.6
14.4
12.9
11.6
11.6
9.7
41.8
40.5
9.3
49.6
14.3
42.6
39.5
32.2
3.7
9.3
15.6
15.6
26.3
26.3
5.7
19.5
19.5
17.6
29.7
7.4
12.1
10.3
33.8
4.1
9.9
18.7
13.1
10
28.9
7.5
13.1
12.7
34.8
32.6
20.3
31
31
32.3
28.4
8.2
14.8
14.8
34.2
34.2
13.8
24.8
24.8
35.2
25.1
11
30.1
30
23.4
23.4
10.2
12.7
13.3
26
27.7
7.6
4.4
9.1
41.1
41.1
5.8
10.9
10.9
3.1
3.1
6.3
6.4
4.4
20
34.5
14.9
18.9
13.1
31.3
23.9
9
9.2
10
30.4
5.6
12.6
14.3
17.7
25.3
8.1
8.2
8.1
14.9
10
11.2
6.6
13.6
3.6
10.1
10.1
12
12.7
16.7
14.7
14.7
6.7
15.5
14.3
15.5
33.6
2.7
3.7
2.2
42.4
42.4
9.7
15
20.5
35.1
16.3
6.5
11.5
12.1
31.7
10.6
10.7
15
15
9
38.9
11.2
37.7
36.9
28.1
32
13.1
6.6
7.6
15.9
15.9
11.3
20
12.6
26.2
26.2
14.8
2.3
4.9
3.3
5.5
14.8
29.2
24.2
23.4
20.1
0.7
29.5
18.3
44.8
36.6
12
24
16.6
17.5
17.3
11.9
19.3
15.9
11.7
31
4.3
5.4
14.8
23.9
6
10.7
8.2
8.2
19.6
14
7.5
0.9
4.5
42.4
7
8.8
12.2
11
7.1
25.6
12.2
16.1
21.9
13.8
12.8
1.7
8.6
8.6
6.9
22.3
4.3
12.3
12.3
21.7
8.6
8.9
10.4
12.2
8.9
5.6
14.4
22.8
22.4
22
32.8
14
16.2
13.5
8.8
14.5
14
28.4
26
21.3
16.9
14.8
16.3
16.3
24.2
42.2
11.6
19.6
13.5
12.6
12.6
11
23.8
25.3
18.2
21
12
11.8
11.8
5.4
20.5
8.5
11.3
11.3
7.9
32.1
12.9
29.5
29.5
24.8
24.8
11.2
29.3
29.3
31.4
26.6
3
14.3
18.5
14.5
7.6
8.9
31.1
32.7
27.6
27.6
13.5
21.8
23.8
23.2
16
4.8
12.4
21.7
7.5
24.7
5.7
2.3
0.7
35.9
35.9
10.7
14.1
12.3
29.7
29.7
13.1
44
46.9
36.9
36.9
15.2
56.8
56.5
42
43.8
10.2
27.2
21.4
18.6
21
11.8
16.7
16.5
29.5
26
8.8
18.5
16.4
30.4
26.9
2.2
14.5
14.5
19.3
11.7
9.7
11.7
11.7
29.2
25.4
14.1
15.6
20.2
15.9
15.9
11.6
32.1
30.5
17.9
27.8
8.7
17
7.7
11.8
7.1
7.7
15.2
7.6
10.6
10.6
9
17.1
19.9
23
35.5
8.9
17.8
19.7
31.6
9.1
8.1
22.1
22.1
17.5
17.5
14.5
7.4
7.8
42.3
6
11.8
17.4
21.1
35
15.4
13.2
10.8
11.2
32
32
2.8
14.7
13.3
26.6
12.5
4.7
28
28
24.1
29
10.5
11.7
17.4
10.3
24.4
12
35.1
35.1
10.6
25.2
11.5
26.9
26.9
19.3
36.5
7.4
16.9
21.2
16.8
19.1
14.8
9.6
6.6
20.9
3.7
9.2
32.3
36.7
46
46
9.2
24.3
24.6
16.9
34.1
11.2
20.2
23
12
31.6
9.6
8.8
8.8
9.3
31.3
10.4
15.3
10.6
18.8
18.8
14.8
16.1
16.1
10.7
25.4
11.6
18.8
22.6
32.7
32.7
5.9
14.1
12.1
10.5
38.3
6.4
14
14
11.6
27
11.5
28.7
25.8
16.7
20
8.9
12
12
5.6
19.8
12.4
16.1
20.5
16.9
18.6
10.4
1.6
1.4
3.7
3.7
7.6
24.2
21.3
18.5
13.7
7.8
2.6
15.3
23.4
32.4
7.9
9.4
10.3
7.3
22
12.3
15.7
16.4
14.4
30.1
7.6
17.7
21
17.5
19.1
11.2
15
19.5
41.6
21.8
3.8
8.5
5.6
25.8
34.9
7.1
1.9
8
14.1
21.4
8.4
9.2
9.2
3.1
17.1
12
27.5
36.5
27.8
40.7
15
15
15
30.5
25.4
9.2
13.4
9.4
11
20.6
11.1
19.6
19.6
26.5
9.1
5
17.1
17.1
30.1
30.1
12.8
42.6
55.3
44.5
40.2
13.4
13.6
21.7
17.4
12.8
9.5
16.6
12.4
7.1
7.1
1.5
13.6
7.8
20.6
8.2
6.9
19.5
19.3
16
48.7
8.9
42.4
38.6
10.7
28.3
7
11.9
3
29.3
8.3
0.8
6.9
6.9
8.7
10.8
10
14.6
11.2
10.8
28.9
15.5
25.8
26.2
32.8
22.5
2.4
26.9
29.7
29.2
18.7
5.2
20.7
10
38.6
29
12.3
6.7
10.6
44
44
1.1
9.3
6.8
14.1
35.7
12.6
32.6
22.2
24.8
43.2
6.3
6.1
3.5
34.4
1.1
7.1
2.4
3.7
2
4.5
4.5
13.7
13.7
29.7
29.7
13.9
25.1
19.3
27.2
16.6
10.8
13.8
13.8
10.5
25.6
15.1
39.9
44.3
37.7
2.7
5.7
14.9
14.9
47.1
12.3
9.6
10.7
10.7
26.4
9.6
9.2
14.1
11.3
25.1
7.2
11.1
6.8
6.8
7.6
22.1
7
10.2
0.8
34
22
10.2
17.7
17.7
11.4
29.2
5.2
13.1
9.1
11.8
46.4
13.8
18.2
18.2
30.3
14.7
9.9
22.5
22.5
12.3
12.3
8.2
4.4
6.5
20
6.4
8.7
10.4
10.4
5.3
26.2
5.8
5.4
11.5
37.9
20
12.2
6
7.5
33.8
25.6
13.5
12.6
13.9
24
31
9.1
13.1
14.3
24
11.5
8
13.4
17.9
27.8
27.8
11.3
24.5
27.5
16.4
38.3
3.7
11.2
2.2
4.6
36.5
7.4
7.7
14.6
12.9
8.4
4
14.5
16.2
6.1
6.1
1.8
5.8
4
4.1
10.9
6.5
16.6
19.5
9.8
28.4
10
20.7
20.7
28.6
21.6
5.7
10.5
16.7
14.7
35.4
6
15.7
22.9
0.4
18.8
4.5
20
13
38.6
30.5
8.6
8.6
8
23.5
4.8
11
8.9
10.9
19.5
28.9
8.6
3.7
7.7
38
21.5
9.2
26.4
23.5
20.5
34.2
8.8
9.7
11.3
22.1
32.4
10.6
11.1
19.7
16.6
16.6
14.2
25.6
16.9
2.7
15.4
9.9
10.3
13.1
26.4
28
4.6
18.6
17.3
10.6
15.2
10.4
12.7
14.8
9.2
27.8
7.5
8.1
8.1
19.9
34.9
12.5
17.2
17.2
29.9
26.6
8.3
48.6
46.9
27.8
27.8
13.6
7.3
12
11.6
8.7
4.7
7.4
2.1
6.6
27.8
10.6
15.3
18.5
27.9
8.5
11.8
18.3
16.3
10.8
24
12.2
22.3
30.7
40.9
37.5
11.1
12.2
12.2
24.5
31
12.8
3.4
3.4
36.8
3.6
10
14.9
20.7
19
29
10.8
8.1
8.1
5.1
9.3
4.9
0.6
3.4
19.2
27.5
13.2
7.5
4.9
36.9
17.4
12.3
6.8
4.9
17.8
4.9
8.9
9.4
11.2
32.5
23.8
14.7
21.7
26.8
18.8
15
11.7
10.1
2.7
28.2
4.8
14.9
12
14
9
10.5
12.8
17.2
19.7
13.7
30.1
8.3
15.1
15.1
23.7
9
14.2
5.9
5
4.2
40.6
10.5
8.5
4.7
32.4
8
13.3
24.5
22.1
37
21.2
11.2
15.2
14.1
25.3
9.2
4.6
4.8
15.8
12.1
30.8
2.5
12.5
9.8
11.6
33.1
12.8
12.1
11.9
18.6
13.5
7.9
5.8
9.8
3.6
5.6
12.8
11.7
9.4
25.4
6.1
10.9
12.3
3.7
37.6
6.3
6.7
6.1
7.1
33.8
1.9
9.5
11.7
15.1
31.7
26.3
14
33.2
40.8
32.8
27.7
9.2
47
36.6
34.9
34.9
0.3
8.1
9
9.9
7
4.2
2
8.2
3.2
5.4
8.5
21.1
20.3
32
18.5
9.2
16.3
16.3
19.8
18
9.3
4.2
4.2
43.9
8.8
8.6
9.9
8.3
25.1
25.1
12.6
15.9
20.5
11.6
24
7
2.4
11
41
13.4
8
8.6
12.5
25.9
6.9
8.8
11
5.2
35
5
6.7
8.7
10.7
9.6
27.6
17.2
46.9
49.3
5.1
39.8
11
11.4
14.6
8.4
10.1
10.2
12.5
19.5
3.2
16
7.5
10.4
7.7
2.4
2.4
12.4
23.7
23.7
20.6
14.1
14.7
34.2
34.2
29.5
36.2
12.1
4.8
3
4.7
38.4
5.1
10.4
10.6
28.4
2.6
10
11.1
9.4
6.5
32.1
9.3
4.8
4.8
21.9
4.8
6.4
40.7
40.7
48
34.8
10.4
14.4
17.9
25.3
10.9
5.5
25.6
18.2
26.9
19.6
8.9
21.9
20.9
12.6
19
6.8
4.4
14.2
46
8
12.3
11.7
13.6
36.2
30.6
6.2
11.3
7.4
7
7
3.1
12.6
4.2
20.1
12.6
8.1
9.2
9.2
6.4
33.4
3
17.2
10.7
33
12.2
9.7
16.8
11.8
10.1
10.1
5.8
5.5
2.8
23.7
15.3
2
5.5
1.6
4.4
23.9
13.3
21
16.8
23.7
16.2
7.7
3.3
10.2
4.5
23
10.6
12
2.8
36.5
8.5
5
6.3
2.5
17.4
3.6
12.4
20.6
20.6
30.4
30.5
6.6
8.1
8.1
1.9
3.2
12.5
22.8
29.8
45.9
43.2
3.3
35.4
27.5
29.5
41.1
11.5
37.5
37.5
32.6
28.8
6.5
7.9
7.9
21.8
32.8
5.1
1.8
5.8
26.7
37.3
9.1
12.1
17.5
10.7
10.7
14.5
46.3
36.6
37.1
37.1
8.3
3.3
3.3
32.6
32.6
13.6
4.2
4.2
15.1
23.3
2.1
12.3
19.8
35.7
13.2
2.1
4.5
3.3
42.4
42.4
11.7
15.8
12.8
8.6
11.9
14.1
37.4
34
43.3
15.1
5.1
23.5
23.5
49.8
28.5
10.5
40
40
49.8
36.9
5.2
7.8
4.9
18.6
18.6
3.9
17.6
12.4
14
45
8.7
8.2
14.7
27.7
30.7
12.2
32
29.8
11.9
26.2
6.2
6.7
6.7
2.8
18.9
11
18.2
16.2
15.9
15.9
2.5
24.1
24.1
27
16.1
14.9
24.4
24.4
20.3
15.3
6.1
9.3
5.9
6.2
20.6
8.6
13.1
14.7
36.3
17
13.2
13.5
15.3
13.1
23.6
7.3
17.1
10.2
15.1
15.1
1.9
10.2
4.2
30.6
6.5
6.3
21.2
18.2
36.8
27.9
7.3
13.9
11.5
9.5
9.9
13
4
6
1.8
3.4
14.1
44.5
39.5
7
33.6
7.9
11.5
11.5
34.1
3.6
9.6
19.3
19.3
10.4
27.7
3.2
2.5
10.7
19.4
19.4
7.5
30.9
41.4
28.1
47
7.1
18.3
19.5
17.2
17.2
4.6
3.1
7.4
5.1
5.1
14.6
28.4
24.5
17.1
31.1
7.5
1.3
1.3
22.2
12.6
13.9
7
1.6
12.6
7.7
12.1
11.7
3.1
20.9
20.9
7.3
12.3
8.8
30.5
30.5
13
28.5
22.7
25.7
23.2
6.3
17.6
20.4
15.5
29.2
4.2
9.3
21.9
16.7
16.7
8.1
7
13.6
22.8
31.6
12.2
15.1
18.9
24.5
6.9
13.2
21.6
21.6
21
18.7
1
9
9
7.2
44.5
7.6
9.8
13.5
27.9
31.4
10
20.8
15.1
33.5
11.6
5.6
1
2.3
36.1
19.6
2.9
19.2
10.4
13.9
5.5
8.4
15.4
15
6.4
6.4
9.3
26.2
19.7
22.6
22.6
7.6
16.8
17
7.7
26.2
4.5
10.4
13.3
30.5
22.9
10
12.2
17.7
17.5
17.5
15.7
46.8
51.6
35.6
1.1
10.8
24.6
18.5
22
30.6
4.7
10.9
11.1
16.8
11.8
5.5
10.3
10.3
21.1
27.6
11.3
17.3
19.8
13
26.6
46.7
18.3
18.4
15.3
16.5
3.5
4.4
14.2
5.9
31.5
2.3
3.4
2.9
8.7
29.8
10.8
49.6
49.6
35.7
49.3
4.5
16.3
16.6
28.9
13.1
3.6
2.7
9.9
5
30.2
11.3
16.2
16.2
9.4
31.7
13.5
5.2
5.1
4
6.5
9.1
4
2.6
19.6
11.4
10.7
35.2
34
24.8
48.6
12.1
17
13.7
15
33.3
7.9
8.3
10.2
21.9
25.4
10.5
9.6
12.4
23.3
23.3
13.6
31.4
31.4
24.6
42.5
14.6
18.9
18.9
24
11.8
4.7
20
23.2
24.7
43.8
10.5
17.3
11.5
8.3
8.3
6.2
25.2
23.3
23.3
23.3
12.1
20
20
26.2
25.6
9
19
19
29.5
12.6
8.7
9.5
11.2
19.1
4.9
10.3
18
15.8
11.8
29.9
4.4
19.1
15.9
7.9
13.9
10.3
17.2
13.3
10.1
24.4
12.4
31.8
34.1
16.7
33.8
8.2
19.1
10.8
9.7
14.3
9.7
16.1
16.1
31.2
31.2
7.3
11.9
6.9
6.5
6.5
7.9
5.2
12.3
8.6
29
13.4
8.9
7.7
26.5
40.9
11.3
16.2
12.5
9.6
31.1
7.7
18.7
19
15.9
17.4
12.7
43.4
53.7
40.7
35.9
13.1
14.6
14.6
25.6
31.3
14.2
6
4.9
24.4
5.3
8.7
3.3
2.8
42.8
3.2
1.9
7.5
2.8
6
7.3
11.7
3.8
3.8
7.1
7.1
4.9
6.9
5.9
37.9
37.9
14.2
31.7
28.4
28.9
23
11.1
20.2
28.7
19.3
39.9
10.5
20.6
29.3
25.3
38.7
14.6
11.5
11.5
8.6
7
3.9
11.7
11.1
29.5
11.5
8.2
8.8
11
31.9
25.9
13.6
37
37.6
42.9
24.9
11.9
22
20.6
17.8
20.7
13
5
7.5
18.3
18.3
11.2
30.3
30.3
41.3
31.8
10
15.9
17.4
9.1
9.1
9.9
21.4
18.9
23.2
23.2
7.1
8.2
17.4
7.3
28.3
15
10.7
8.5
7.9
24.3
8.2
29.5
19.7
34.7
18.5
5.6
8.9
2.3
36.2
7.3
8.8
40.8
40.8
10.8
31.5
8.5
16.6
16.6
24.7
12.1
7.6
8.1
9.7
24.9
24.9
13.9
4
10.1
4
35.9
12
32.5
26.7
26.6
29.1
10.5
21.1
18.9
18.1
37
4.9
10.1
19.3
29.9
28.1
10.6
24.8
24.8
29.7
40.8
6.8
27.6
27.6
25.6
38.8
12.2
19.3
18
24.8
11.7
7
16.4
16.4
18.7
34.1
8.7
8.4
8.4
4.8
7.8
1.2
16.3
12
32.7
40.5
14.9
16.4
18.9
21.6
23.1
8.1
43.1
33.2
47
47
1.1
14
16.3
36.7
36.7
11.6
7.6
6.8
6.3
37
14.8
12.9
11.5
6.8
8.7
7.2
22.5
21.8
31.6
20
4.4
4
4
3.9
37.5
6.9
18.1
13
49.6
30.4
7.2
22.9
19.9
31.2
25.6
7.7
15
15
11
25.6
8.1
28
20.5
19.4
34.7
11.4
12.9
12.9
9.8
30.8
4.7
8.8
6.9
41.5
25.4
10.6
39.4
39.4
41.8
41.8
2.5
8.3
8.3
5.9
5
9.9
17.4
10.8
12.3
12.3
8.2
17.6
15.7
23.3
12.5
13.5
9.8
9.8
6.9
22.5
7.4
16.4
14.6
24.1
13
13.7
21.6
22.8
29.1
29.5
7.8
12.2
12.2
33.3
33.3
9.1
17.3
11.2
10.9
7.2
8.6
10.1
12.7
23.9
23.9
9.3
10.2
12
23.7
25.9
12.7
16.9
11.3
26.2
7.5
8.2
20.3
19.1
31.7
16.9
12.1
9.8
4.9
20.9
20.9
12.3
11.8
4
12.4
6.8
14.8
3.9
3.9
25.2
37.4
2
14.6
5.9
47.3
7.3
13.6
16.7
15
10
33
12.8
39.6
30.1
24.1
26.6
1.7
16.2
17.8
13.8
13.8
8.1
33.4
33.4
40.9
14.1
4.2
14.5
14.2
33.3
33.3
14.1
28.2
31
39
24.3
10.4
9.1
14.2
12.6
32.1
10
13.6
12.9
9.8
30.9
13.3
10.3
10.1
7.5
5.6
11.6
13
18.1
9.7
25.9
15.1
30
24.2
23.3
18.4
12.8
11.3
5.2
17.7
17.7
9.3
5
6.6
4.6
18.8
7.1
13.8
19.1
31.7
15.8
7.4
4.6
0.8
3.8
6.9
4.8
14
14
25.9
7.7
9.3
11.6
11.2
28.8
7.4
10.7
13.2
11.8
8.2
28.1
11.8
39.1
29.4
38.6
29.4
8
20.3
16.3
25.9
14.4
7.4
8.5
17.1
31
11.6
8.7
46.1
47
39.2
34.6
12.9
19.3
19.3
14.9
25.9
2.9
12.3
12.3
9
9
6.3
5.6
4.5
17.8
17.8
14.9
47.7
43.8
2.5
36.2
11.3
19.9
16.8
42.3
36.6
6.2
19.6
14.2
14.9
31.3
9.7
12.5
12.5
8.3
24
6.4
5.2
8
5.2
19.8
14.9
29.5
27.5
47.8
29.5
3.9
10.4
4.7
7.6
39.5
10.2
16.3
13.8
25.2
31.7
10.1
15.3
20.3
12.1
19.3
10.7
2
11.4
20
20
12.1
3.4
2.3
7.4
7.4
1.4
13.7
11.1
8.7
11.9
7.1
6.1
6.1
27.4
32.2
3.8
12.2
14.9
30.8
30.8
5.3
3.6
12.7
32.4
29.8
4.8
17
17.8
28
14.4
14.4
34.2
39.7
29.6
33.5
6.7
42.2
32.5
47.6
47.6
13.4
11.2
9
4.5
4.5
13.3
15.2
18
17.5
42.9
15.2
23.3
22.4
17.1
17.1
13.1
17.6
17.9
12.6
25.2
5.8
12.2
9.8
30
20.1
9.1
19.7
14.2
18.1
12.8
13.4
31.8
31.8
27.8
27.8
12.6
21.8
24.9
27.2
27.2
12.2
12.5
10.2
6.2
26.5
2.6
17.2
28.8
23.2
35.8
4.7
34.6
34.8
31.8
31.8
5.7
9.7
12.2
8.7
20
8.9
21.9
22.3
20.1
32.1
11.8
13.2
13.2
9.9
25.1
16.5
47.4
52.2
40.3
40.3
13
25
22.8
8
8
9
11.5
10.6
6.5
6.5
9.6
36.3
34.5
42
27.3
7.4
10.5
12.5
32.5
21.9
5.5
19.4
12.8
17
29.9
10.8
19.1
19.1
27.9
26.9
9.7
27.6
24.7
22.7
21.4
10.5
22.4
25.2
27.6
38.7
8.8
10.8
3.3
43.3
20
12.4
22
22
21.9
15.8
12.7
16.9
11.3
32.5
12.4
12.7
13.2
11.6
24.8
10.4
13.7
25.4
25.4
29.9
31.1
4.6
17.3
17.3
15.9
15.9
5.6
3.5
3.5
1.9
20.9
3.6
8.9
9.4
4.7
19.9
8.6
7.8
6.6
3.6
23
6.3
3.5
5.5
34.7
19.8
10.9
11.1
13.8
32.2
8.7
12
20.1
21.6
17.4
26.2
12.5
16.5
14.4
12.7
28
3
6.8
6.8
5.7
28
3.5
10.7
10.7
42.4
28.2
1.9
28.4
28.4
22.1
42.8
7.9
2.4
13.4
7.7
7.7
13.3
8.6
8.6
3.5
3
9.3
15.7
11.8
8
27.9
8.3
42.6
42.6
34.6
34.6
12.1
33
33
26.1
39.7
6.7
9
9
25.4
19.1
8
14.4
6.8
4.9
20
13.1
37.6
45.7
36.6
3.7
14.7
12.7
12.7
45.2
45.2
6.8
17.2
17.2
12.1
28.6
10.5
13.8
17.7
8.5
8.5
9.4
15
15
28.5
25.9
13.9
5.2
5.5
38.6
23.3
6.9
18.9
16.9
31.7
15.8
13.6
4.8
5.2
38
23.6
4.6
15.6
24.4
13.8
38.9
10.6
14.5
14.5
26
10.3
9.8
12.3
14.5
33
22.5
8.7
12.1
14.7
7
7
6.6
17.9
14.9
28.1
13.3
9.1
11
18.6
24.6
16.4
8.9
8.3
8.3
5.2
5.2
0.6
13.3
9.2
33.8
38.9
2.1
28.6
27
43.3
34.3
2.8
11.2
18.5
42.2
14.5
10.2
13.1
11.1
34.4
6.8
11.7
9.8
8.9
5.8
26.5
5.1
27.7
30.1
30.8
26
14.7
17
17.6
19.9
47.7
14.9
32.3
32.3
26.5
18.1
5.6
19.1
15.4
15.3
17.6
12.3
7.4
4.8
26
36.6
9.9
18.3
18.3
26.8
7.9
5.2
4.3
1.6
26.7
1.9
8.7
20.1
20.1
12
12
2.8
18
16.1
14.2
11.1
14.1
25.5
25.5
31.3
31.3
12.5
2.9
7.1
1.6
25.6
14.4
21.4
15.8
17.2
7.8
5
21
14.9
20.2
20.2
13.5
45.2
45.2
3.5
3.5
2.6
12.9
8.1
11
11
10
14.3
6.6
4.7
9.4
10.8
24.3
29
16
33.6
11.5
5.6
5.6
45.1
9.4
11.1
21.8
19.3
3.6
15.6
13.8
13.5
13.5
10.3
10.3
7.2
8.5
17.7
12.3
12.3
14.5
36.4
43.5
47.1
34.9
10.4
10.7
19.8
13.1
15.4
10.7
16.8
19.3
9.5
13.8
14.8
44.1
34.9
33.9
38.8
11.2
21.6
27.2
37.5
37.5
7
9.3
13.9
28.1
4.2
12.4
11.2
8.4
6
41.3
9.3
17.4
20.5
36.9
36.9
7.6
13.9
19.5
30.2
30.2
2.5
13
11.3
23.4
23.4
7.5
17
25.2
32.9
18.3
12.3
21.8
22.9
30.9
30.9
12.2
17.6
21.8
13.8
12.1
7.6
7.2
7.4
6.7
21.5
16.7
12.1
9.2
5.9
1.1
9.7
51
49.1
36.3
30.5
7.2
22.7
24.3
40.7
40.7
8
2.9
2.9
11.2
11.2
7
12.9
12.9
28.7
31.3
3
36.5
36.5
25.2
30
5.9
2.8
2.8
41.4
6.5
5.3
5.3
7.8
37.4
37.4
11.9
4.5
3.1
38.3
6.9
11.7
17.3
20.2
15.3
15.3
13.8
6.4
8.9
36.4
17.9
7.5
20.7
17.1
47.6
34.7
7.5
31.2
41.9
14.1
28.5
9.5
16.3
18.1
27.7
27.7
11.5
20.7
21.1
36.6
29.4
10.6
13.1
13.1
23.4
23.4
12.3
33.4
32.1
8.7
45.8
6.9
2.6
10.1
21.9
0.7
13.4
11.9
13
31.7
24.7
8.1
3.8
3.8
19.6
19.6
11.3
11.6
20.2
10
26.6
8.1
11.3
9.2
6.4
6.4
6.8
20
20
38.2
27.2
9.9
37.7
34.3
37.4
38.8
7.7
16.5
11.4
16
11.1
9.6
13
18.6
25.6
31.7
9
20.9
19.8
18.7
20.9
9
20.3
14.6
30.2
18.2
4.8
11.8
14.1
12.8
38.1
5.9
17.2
14.4
42.2
16.5
8.5
12.6
16.8
10.2
33.9
13.1
9.1
5.3
22.1
17.8
8.5
19.8
20.8
29.1
29.1
10.9
13.5
13.5
12
8.3
8.2
12.8
14.7
6.8
24.4
5.7
5.4
8.3
35.8
19
16.1
45.4
43.3
17.6
17.6
7.5
8
9
24.2
1.8
9.3
1.3
1.3
9.6
33.6
6.3
23.7
19.6
19.8
19.8
9.6
17.8
23.8
19.3
19.3
8.5
29.1
38.4
11.4
25.8
15
39
39
49.6
35.5
9.6
9.9
11.3
23
23
15.1
25.5
27.2
19.3
21.8
14.9
4.6
4.2
15.7
39.7
10.6
33.4
30.1
43.4
44.7
4.9
3.9
6.6
17.7
1.5
7.7
8.3
2.1
41.6
6.3
12.5
35.4
24.2
28.8
28.8
8
8.9
1.9
20.6
39.1
9.2
10.4
12.9
9.4
24.1
12.5
25.4
31.4
22.2
47.1
9.1
6
9.5
46.7
11.5
27.9
46.8
34.4
49.7
32.6
0.5
31.6
19.1
37.1
24.7
8.9
17.3
13.4
10.7
24.8
6
13.6
13.5
18
45
10
31.1
21.6
24.7
24.3
13.4
13.8
13.8
41.2
32.7
14.4
9.1
2.2
23.7
3.7
50.3
15.9
16.8
20
15.5
10.1
46.3
39.2
35
39.3
2.4
12.5
10.1
36.5
30.2
8.2
45.4
48.3
33.9
38.6
10.5
8.3
6.9
6.7
8.1
6.1
22.3
22.3
14.7
19.4
12.6
17.1
17.6
13.8
13.8
18.5
47.1
51.3
5.5
5.5
10.5
25.4
25
20.9
22.3
12.7
21.8
16.2
14.1
28.5
13
9.6
9.6
24.6
33.1
7.7
47.1
44.3
33
49.2
7.2
15.7
7.9
5.8
29.6
12.3
4.8
6.5
19.3
19.3
11.9
17.9
21.6
28.2
11.3
10.3
18.4
23.4
21.3
19.8
33.5
19.8
22.8
15.6
26.8
5.6
22.8
22.8
15.3
41
11.2
23.2
24
31
31
2.4
8.4
8.4
19.1
11.4
2.3
6.4
7
41.2
41.2
14.6
19
19
24.9
24.9
6.7
9.5
9.5
20.7
1.6
14.8
20.4
17.4
30.7
26.9
13.1
19.6
19.6
5.7
5.7
11.5
17.1
25.3
32.4
21.9
7.4
16.6
24.9
36.6
32.7
9.4
19.5
14.3
16.9
3.7
11.6
22
24.7
47.4
22
11.5
13.2
15
23.7
9.6
3
10.7
3.6
5.6
9.6
10.7
24.8
35.4
40.9
40.9
14.2
8.2
12.1
6
22.6
12.7
19.2
19.2
6.3
20.3
9.5
11.7
9.3
33.6
21.1
9.1
11
15.5
6
26.5
17.7
54
56.2
4.2
39.5
14.9
26.2
19.5
35.2
43.5
1.8
7.7
1
23.2
13
11.4
31.1
31.6
38.5
38.5
6.2
16.2
17.6
15.5
30.3
5.7
30.3
41.5
46.7
33.4
11.2
29.4
31.8
15
15
12.6
14.2
21
18.8
13.1
7.6
8.6
11.5
14.2
5.5
14.4
9.2
4.8
40.7
5.3
9.3
22.2
22.9
13.2
30.5
11.2
27.1
29.6
28.1
22.8
6.4
7.9
10.1
13.1
42.5
13.6
20.7
27.7
20.8
20.8
8.6
12.4
16.8
10.7
24.3
7.1
29.9
29.9
19.3
13.4
6.4
42
44
46.5
35
14.3
37.6
29
18.5
25.1
11.5
12.1
23.3
18.4
2.1
13
7.4
10.1
3.7
16.6
3.3
1.6
8.4
28.5
28.5
14.3
20.8
22.1
28
15.4
9.5
11.8
13.9
24.2
7.4
5.1
3.3
7.9
6.6
18.8
43.1
8.9
3.5
13
8.5
9.7
14.6
14.6
10.6
29.4
3.1
11.6
15.6
13.1
43.6
14.9
6.4
1.1
21.2
15.4
12.8
3.3
2.5
13.3
22
2.6
18.2
7.8
10.5
10.5
7.2
12.9
6.1
35.8
21.4
8
4
4
5.4
21.5
11.8
12.9
15.8
26.5
28
10.3
15.7
9.3
8.8
6.4
9.6
16.4
8.2
7.2
26.1
4.6
13
17.1
13.3
17.5
11.6
6.7
6.6
26.5
2.1
7.5
10.1
16.2
34
26.3
8.5
11.2
7.4
48.9
7.2
8.7
14
19.2
13.3
17.7
1.9
14.4
6.5
23.3
47.9
14.3
36.1
27.9
17
30.9
5.6
5.4
4.8
37.6
25.9
4.1
10.5
15.1
3.6
3.6
8
18.9
19.2
30.5
15.8
9.3
21
12
37.2
36.8
1.9
16.6
10.8
8.2
8.2
11.7
9.9
9.9
28
28
4.3
10.7
2.3
28.9
3.2
7.4
17.7
17.7
29.4
23.7
2.4
11
10.7
44.1
13.4
12.8
13.4
13.9
7.9
24.9
8.9
2.9
2.9
5.5
31.2
13.8
21.6
25.4
25.3
31
13.9
45.3
39.3
36.4
3.3
8.8
21.7
22.2
18.5
20
10.4
10.1
18.1
30.8
7.7
5.6
32.8
42.6
33.9
29.2
7.5
3.3
8.3
45.3
6.3
6.9
8.7
6.7
5.3
43
14
19.6
19.4
30.8
14.2
5.5
8.2
6.3
2.5
36.6
11.3
9.7
9.7
11.4
46.8
12.4
16
13.5
24.1
25.1
4.7
6.9
6.8
22.3
22.3
13.7
35
35
46.8
46.8
7.4
7.6
4.9
31.3
33.8
6.5
8.6
0.9
12.4
6.9
2.9
12.1
4
22.2
31.3
5.5
16.8
16.8
18.1
16.4
12.5
7.8
5.3
3.5
17.2
11.2
16.5
11.2
28.7
34.7
8.3
16
21.7
31.8
31.8
12.1
16.9
12.8
11.1
23.8
9.7
16.6
16.6
25.5
29.8
9.6
6.6
16.5
10.2
32.4
11.3
27.2
34.7
24.8
30.9
9.1
20.1
15.6
32
32
10.6
30.4
30.4
43.1
20
11.4
13.6
7.4
22.8
4.1
8.4
3.5
12.1
20.6
12.6
7.6
17.7
7.2
8.1
28.7
13.9
6.5
7.4
37.3
2.9
14
3.9
3.2
24
40
14.4
10.2
10.2
33
21.7
15
4.6
10.5
14.4
39.9
8.3
12.5
12.5
41.7
22.9
8.5
15.6
15.9
31.8
11.4
13.7
20.9
20.9
16.4
16.4
11
16.1
14.6
9
22.8
6.9
3.1
8.2
2
21.6
6.4
16.8
11.3
48.6
13.4
11.2
22.8
20.4
16.4
19.7
10
13.9
11.7
25
26.8
8.9
12.8
21.4
13.6
16.6
8.6
16.9
8.9
14.7
14.7
1.2
33.1
34.4
39.5
22.3
7.1
14.1
8.3
10.3
48.3
1.7
0.8
11.2
22.3
42.4
6.3
8.6
16.3
27.6
27.6
9.4
13.2
2.4
32.9
22.7
12.6
17
17
29.5
13.4
11.8
21.2
20.8
10.9
35.7
8.9
20
15.5
32.2
32.2
3.4
14.5
11.7
6.2
6.2
7.9
13.7
19
30.8
30.8
10.6
10.7
5.1
31.4
8.1
5.2
20.5
10.6
39.1
29.4
10.3
15.4
17.6
7.3
26.1
11.5
6.2
6.2
27.1
1.6
3.3
13
8.7
5
49.6
36.7
9.2
9.2
25.1
5
7.7
9.1
10.6
22.1
8.2
4
15.1
16.1
31.5
12.1
9.7
23.6
23.6
20.1
15.7
8.7
6.3
6.3
12.4
47.8
2.1
3.3
3.3
30
6.1
10.9
8.7
11.4
15.9
11.3
9.1
20.5
22
18.8
16.7
6.9
6.4
15.3
9.7
9.7
12.5
19.9
18.6
23.2
25.1
8.7
15.2
10.6
29.3
28.4
3.8
29.9
19
23.3
20.7
5.3
35.4
28.3
30.2
30.2
7.6
8.7
8.7
37.1
17.3
12
17.3
17.3
34.3
25.1
8.2
17.5
17.5
30.4
44.9
12.2
20.6
18.4
30.5
24.5
8.5
16.5
13
31.9
25.8
9.4
12.9
12.3
31.2
25.2
3.7
9.4
7.6
6.3
37.1
14.8
30.8
32.7
16.5
16.5
14.6
22.8
22.8
20.1
20.1
8.1
39.5
39.5
45.8
45.8
9.3
4.4
6.5
36.2
8.4
9.8
15.6
11.7
24.5
8.8
9.5
11.3
10.3
6.9
9.2
14
44.9
54.8
39.8
46.4
2.4
27.2
33.9
25.3
41.4
5.1
12.9
18.2
45.8
32.5
2.2
10.7
18.9
14.7
34.7
10.7
33
35.8
28.3
40.5
14.6
16.1
18.2
10.8
9.5
7.3
7.6
7.6
24.6
4.3
13.9
21
21
24.9
24.9
14.9
32.9
42.6
26.1
26.1
14.8
10.9
15.4
22
7.9
3.1
10.2
8.3
41.5
7.7
11.8
4.1
4.1
45.9
8.8
11.8
28.7
28.7
22.9
41.7
8.2
9
3.7
21
6.6
10.3
27.8
29.8
14.6
26.2
14.8
14.8
14.8
10
31.3
9.4
7.3
16.1
11.1
20.1
10.2
25.1
25.1
41.6
21.8
3.4
26.9
27.1
23.4
23.4
7.7
8.2
6.5
10
9.3
4.2
14.4
16.3
31.1
25.7
8.6
10.3
13.3
21.9
7.3
11.2
19.4
13.8
26.4
9.6
8.7
2
6.5
5.1
39.3
6.4
3.1
6.7
34.2
1.5
10.9
31.2
28.6
15.2
15.2
8.9
8.6
11.1
23
5.1
9.3
32.9
23.1
30.9
47.2
13
27.1
26
24.4
22.1
6.5
7
5.7
20.2
5.6
4.2
19.1
26.8
20.2
42.8
5.8
12.5
5.8
11.2
25.3
12.7
8.2
8.2
2.3
2.3
5.9
10.7
7.7
2.4
7.3
12.5
4
10.8
5.4
5.4
3.2
7.8
10.1
18.5
7
13.3
6.2
3.4
2.1
18.3
11.8
8.2
8.5
26.7
26.7
7.9
10.1
19.2
8.7
32.6
6.5
1.9
10.5
34.4
28.7
9.8
22.2
17
31.8
20.1
3.7
16.5
9.8
10.1
12.6
6.9
7.6
6.7
4.7
25.3
12.2
19.3
25.5
17.4
17.4
10.3
28.8
25.9
17.5
22
11.3
17.9
17.9
13.5
13.5
13.7
52.2
54.7
44.1
42.3
8.8
10.7
17.9
8.9
26.5
14.1
36
36
17.4
44.6
15.5
42.1
40.7
34.3
34.3
18.4
54.7
52
42.1
42.1
13.4
7.2
9.5
3.1
4.7
3.2
18.2
12.3
42.8
21
6.8
9.9
8.3
34.6
34.6
8.9
11
12
5.8
8.8
8.5
17.7
17.7
7.4
30.1
9.3
15.5
15.5
12.6
12.6
7.6
5.8
5.8
8.1
11.4
8.5
16.5
14
7.8
7.4
7.4
3.8
12.2
12.4
6.8
12.9
16.3
13.7
17.2
1.9
3.6
2.6
7
27.4
38.1
12.8
3.3
10.7
5.9
5.9
6.5
10.9
3.8
12.9
6.3
9.7
16.7
27
21.3
16
11
30.5
33.3
30.9
30.9
8.2
9.6
11.5
6.3
29
4
5.9
8.6
17
38.8
3.6
2.1
9.2
27.5
18
15.3
33.1
27.4
20.9
20.9
5.2
19.6
19.6
16.1
16.3
13.4
9.2
17
23.5
14.9
7
9.1
7.5
8.4
27.9
8.6
29.2
29.2
46.2
46.2
4.9
29.2
29.2
30.1
30.1
6.9
2.8
3.5
6.3
12.8
11.3
40.8
38.1
47.2
28.2
5.5
2.7
9.1
3
3
2.6
6.2
6.2
46.1
6.5
10.1
18.9
11.3
36.5
13.8
5.1
7.6
3.6
21.5
4.2
8.2
16
7.6
8.2
8.2
13.8
27.3
25.5
20.6
27.6
11.7
17.2
16.4
27.5
27.5
4.8
12
9.2
39.2
5.4
3.5
14.5
20.4
11.4
20.6
12.7
15.3
15.3
9.8
9.8
8.8
19.8
19.8
16.7
12.3
12.4
29.3
28.9
24
34.8
6
6.9
11.4
8
11.9
10.9
19.6
8.4
49.4
19
13.7
55.1
51.9
42
42
3.1
10.1
8
7.1
7.1
7.1
10.1
8
20.1
34.4
10.7
50.8
50
31.6
42.4
13.4
41.2
38.4
31.5
43.5
14.8
33.9
33.9
27.3
26.3
15.7
26.2
27.9
22.5
20.1
7
20.7
22.4
23.1
23.1
11.4
14.1
14.5
27.3
9.7
12.6
7.7
10.1
7.1
7.1
2.2
10.3
1.8
3.1
41.4
1.9
1.8
1.8
42.6
11.9
13.5
22.1
24.4
29.3
25.5
2.2
22
30.7
37.7
24.3
13.4
24.4
15.7
13.7
18.6
14.9
4.7
5
19.4
1.2
8.3
35.1
39
25.4
25.4
9.7
16.8
11.1
30.3
30.3
3.7
2
9.1
4.6
19.7
14.2
13.2
11.7
10.9
15.1
1.7
14
14
10.2
7
9.4
47.5
47.5
40.4
30.3
12.7
46.1
46.1
40.5
45
7.3
16.7
11
14.9
14.9
13.6
28.3
36.9
24.8
29.7
8.2
16.1
16.1
28.8
28.8
11.9
21.7
18.5
23.5
13.6
7.5
4.5
11.1
42.2
3.7
1.7
33.3
32.6
22.6
22.6
9.8
14.6
12.6
11.5
20.7
4.4
8.7
2.6
3.5
8.1
5.6
11.3
3.3
21.5
7.5
3.5
7.6
7.6
6.9
36.6
11.1
14.4
14.4
12.6
27.9
12.6
8
1.4
38.4
16
6.1
23
29.9
40.1
21
43.2
27.8
26.4
41.4
26.5
12.2
14.8
9.5
31.2
5.5
8
39.7
36.4
31.7
37.6
7.5
31.2
34.1
35
35
14
36.1
40.3
33
31
11.7
24.5
24.5
31.5
16.8
14.8
6.2
4
2.5
25.2
3.3
7.2
7.2
3.9
3.9
12.9
4
7.6
2
16.6
10.6
18.5
14.6
25.6
26.4
4.6
6.3
11.2
8.6
8.6
13.6
49.8
40.9
35.5
40
7
7.2
7.5
37
3.1
9.8
16.9
15.6
28.6
11.4
9.1
5.6
10.3
36.1
5
5.3
6.9
8.2
27.7
6.4
10.2
11.4
11.4
10.2
7.6
10.7
30.9
28
44.4
37.6
10.8
14.6
12.3
15.6
19.3
10.9
13
4.4
7.1
7.1
9.9
24
30.7
47.3
38.9
6.2
16.5
25
37.8
37.8
8.4
7.7
5.1
46.7
11.3
5.5
22.2
21.4
47.2
17.4
7.2
8.8
3.3
18.2
18.2
8
12.7
12.7
32.6
33.3
11.1
18.6
13
18.1
7
7.4
20.3
17.4
18.8
18.8
12
14.8
9.7
32.3
32.3
5.8
11.6
11.6
20.1
8.2
8.4
19.9
23.2
24.5
12.6
12.2
9.9
9.9
11.4
7.8
6.9
3.7
11.7
42.4
3.3
11.2
8.4
3
4.7
19.1
14.1
20
15.7
34
16.1
11.3
8
9.3
31.1
4.7
7.8
19.5
14.2
30.4
13.7
10.8
47.7
47.7
36.2
40.4
14.8
15
16.3
26.1
30.7
6
15.6
28.2
22
31.4
9.1
11.2
11
6.6
6.6
10.8
24.3
34.3
26.3
10.5
11.8
6.6
12.3
26.3
3.2
35.2
21.9
18.8
12.9
30.6
2.3
10.3
4.3
30.9
30.9
3.8
1.7
4.8
39
1.8
11.7
13
10.2
17.3
29
6.2
4.2
9.7
18.9
23.7
9.4
11.1
11.7
27.7
7.1
11.8
7.7
15.8
46.5
8.3
3.7
1.9
11.4
20.1
36
8.3
11
11
26.8
13.5
12.1
24.1
16.7
17.3
1.6
12.7
14.4
15.4
27
12.6
7.5
18.5
19.3
30.4
15.9
14
8.1
9.6
22.8
5
9
16.5
11.7
27
27.3
10
8.1
8.1
7.2
7.2
2
7.9
10
47.1
11.7
10.9
22.4
20
22.5
13.7
9
11
11.7
9
28.1
7.2
8.5
1.8
13.5
21.1
9.6
4.6
7.7
45.3
19.2
7
17.4
16.5
26.5
26.5
13.4
42.1
29.5
28.8
28.8
11.4
10.5
3.5
4.8
28.5
11.3
12.1
20.8
17.2
2.4
12.5
3.3
8.3
41.4
21.5
8.8
13.6
10.9
8.4
10.8
8.6
19.5
19.5
15.3
15.3
11.5
10
12
35.6
35.6
6.3
10
2.1
6.7
20.4
5.8
37.7
40.7
45.9
45.9
11.5
9.1
9.1
5.9
1.3
8.2
1.9
11.6
41.3
6.2
10.9
15.4
15.4
22.2
22.2
6.4
19.5
19.5
14.9
14.9
10.4
30.6
23.5
26.7
12.9
9.5
12.1
6.3
5.2
36.2
7.7
13.6
8.5
7.9
26.6
10.2
30.4
33.2
21.4
26.1
6.7
16.4
29.1
44
30.3
7.4
11.9
6.8
8.2
8.2
17.3
48.7
53.6
3.8
3.8
13.1
16.6
19.6
25.8
12.8
7.9
10
10
32.8
26
13.1
17.7
19.3
27.1
27.8
4.6
21.8
13.3
28.1
30.3
6.9
22.6
28.1
23.7
23.7
3.2
9.2
16.7
8.9
13.4
13.7
1.7
9.5
6
13.6
7.9
15.4
5.6
13.4
34.7
8.7
31.1
31.1
32.4
32.4
10.3
15.7
15.7
7.7
23.4
6
19.3
23.1
2.9
15.4
11.8
15.6
13.3
26.4
8.5
10.8
11.4
11.4
8.5
8.5
12
26.5
26.5
30.9
42.9
5
17.7
17.7
12.8
14.5
6.6
13
11.2
33.5
13.7
8.8
45.5
47.9
49
33
6.9
15.5
15.5
14.2
39.7
3.9
14.5
13.5
13.1
46.7
4.3
3.3
5.4
4
26.1
10.3
13.3
12.6
25.2
31.1
12.8
9.5
9.9
24.9
21.3
12.2
4.7
4.7
42.6
3.3
8
16.8
16.3
12
49.5
11.6
20.5
14.5
13.2
29
13.9
9.8
5.6
3.3
3.4
15.8
48.3
58.3
42.2
49.2
10.3
3.1
11.4
3.2
21.6
14.8
33.8
26.1
33.5
26.1
6.2
9.6
12.2
8.7
5
11.4
21
19.7
12.6
38.3
6.5
19.6
7
14.8
10.4
9.9
30.5
32.3
26
26
8.1
3.9
3.9
20.2
20.2
5.6
3.3
6
20.1
26.7
13.8
42.2
42.2
45.8
45.8
8.8
43.6
43.5
37
35.2
3.2
8.5
4.2
38.3
18.8
13.8
7.9
8.8
23.2
19.9
9.3
30.5
19.4
24.5
24.5
2.3
13.3
6.8
10.8
25.5
7.2
20.4
19.6
19.3
46.2
12
6.2
4.2
19.9
34.5
10.1
19.2
16.3
12.3
26.9
13.3
8.5
10.4
20.6
4.2
12.2
23.2
27.8
24
38.5
12.2
9.5
3.5
11.9
3.3
11.1
13.8
13.8
25.6
25.6
11.4
19.3
17.9
34.9
34.9
11.4
13
15
26.3
13.3
8.8
10.3
10
9.1
5.7
13
24.7
27.3
49.2
42
7.8
4.5
2.4
39.9
39.9
8.1
3
2.4
6.4
2.2
5.6
2.4
2.4
43.6
43.6
11.7
33.5
33.5
16.4
26
6.8
24.5
12.6
32
26.2
9.8
11.8
11.8
10.5
27.2
13.7
10.1
6.7
23
4.4
14.4
6.6
5.5
1.1
3.8
10.4
12.5
12.5
29.6
26
5.4
4.1
7.2
40.1
15.5
11.4
7
4.9
44.6
44.6
10.8
13.1
14.7
9.3
13
11.6
14.7
20.5
10.9
10.9
10.5
12.1
9.7
5.5
8.7
10.3
11.2
6.9
12.8
12.8
11.9
13
3.7
20.6
7.9
8.2
6.1
4.1
6
45.5
10.8
10.7
12.9
16.1
3.4
6.4
17.2
7.4
32.4
26.4
8.3
8.2
9.3
22.6
26.4
5.9
10.5
4.1
30.5
30.5
3.4
15.2
15.2
12.2
49.8
4.6
17
16.6
28.2
14
6.2
23.2
15.6
14.4
17
14.5
31.5
31.5
14.8
29.4
10
12.5
13
11.6
28.2
8.8
27.5
24.9
13.6
13.6
13.5
14.2
12.9
11.4
29.7
11.5
22.1
27.3
16.8
16.8
9.5
9
9
24.1
32.5
6.2
19.6
18.7
16.2
15.1
7.7
33.2
31
28.2
47.1
14.5
23
26.6
21.6
39.3
8.2
19.1
21.8
3.4
15.9
5.2
10.7
8
2.1
16.6
12.7
29.3
26.9
22.9
27.8
14.4
10.5
8.4
32.8
21.9
6.9
16.4
22.2
28.7
18
7.4
8.4
12.4
26
3.4
2.5
7.8
13.5
23.8
48.8
5.8
8.7
4.8
26.5
26.5
2.4
15.1
6.9
9.5
9.8
7.9
21.8
23.3
20.1
19.2
9.5
9.3
11.6
8.9
19.1
9.8
9.7
16.9
29.6
10.9
7.1
2.1
9.3
3.2
3.2
4.3
14.5
16.5
31
25.9
7.4
10
7.2
40.5
14
2.8
5.9
7.9
40.5
5.6
13.3
12.8
14
10.2
8.1
8.3
10.8
11
35.1
35.1
5.5
6.6
15
24.8
24.8
10.6
12
7.9
7
34.8
9.6
5
4.7
9.9
5.4
7
6.7
9.2
26.2
1.8
7.3
3.4
9.6
4.9
4.9
8.4
19.9
32.6
29.2
25.5
11.8
13.7
13.7
12.1
27.9
7.2
7
6.8
3.5
22.3
6.4
4
5.5
1.6
26.9
6.5
8.5
7.3
42.9
24.3
7.8
16.5
9.3
6.4
6.4
6.5
10.6
7.7
2.6
7.2
5.5
7.3
3.2
3.2
16.1
9.2
23.9
17.9
23.5
20.7
5.7
8.2
5.6
26.1
5.2
14.6
17.3
27.2
34.5
44.7
8.6
18.1
17.5
16.6
11.9
11
15
13.6
10
11.9
10.1
28.5
22.9
28.5
26.4
7.1
24.8
27.3
15.6
1.8
8.8
21
19.6
10.3
10.3
7.4
26.6
26.6
37.7
34.1
13.2
8.5
8.5
3.8
35.8
8.3
15.9
14.6
32.9
7.2
7.5
21
15.8
35
35
8.5
10.6
13.1
11.8
27.9
10.3
16.6
10
9.9
29.8
12.9
11.5
10.1
4.7
36.5
11.2
22.4
17.2
34.1
37.4
13.1
6.3
3.4
1.8
3.7
5.5
3.4
3.9
6.2
6.2
8.7
9.9
3.7
2.5
20.2
6.6
7
7.5
24.1
4.9
10.3
12.3
12.3
31.7
23.7
8.8
16.7
25.3
23.5
21.9
10.5
15.4
13.8
10
31.9
6.5
35.4
32.7
31.2
48.2
4.8
2.3
6.3
37.8
37.8
12.1
17.4
20.1
30.1
15.6
4.2
19.4
12.9
3.5
3.5
12.8
24.7
19.1
24.5
13.7
8.3
21.6
17.3
10.8
15.3
14.9
5.8
11.6
35.9
5.8
12.2
14.2
25.1
19.5
19
11.2
15.9
18.3
10.9
24.6
3.5
25.5
26
40.5
33.4
14.9
22.5
21.4
25.7
29.2
9
12.4
12.4
31.7
25
5.1
8.2
4.8
5.2
26.4
8.4
18.1
18.1
35.9
34.9
5.9
19.8
20.5
17.9
29.6
8.6
17.7
27
41.5
16
6
7.9
7.9
35.1
19.4
12.1
5.6
5.6
21.1
9
9.6
27.8
19.3
19.9
34.6
8.5
10.3
8.5
17.4
37
7.6
4.5
0.9
3.9
22.1
11.1
16.9
15.3
30.1
30.1
7.3
44.8
43.6
37.1
27.7
14.4
7.2
7.3
5.5
13.6
5.1
6.2
4.3
35
35
1.2
10.1
6.8
19.3
19.3
8.6
17.7
17.7
8.3
15.7
8.8
13.8
21.2
35.3
35.3
4.7
18.6
26.4
41
42.1
10.9
48.9
48.8
36.1
37
8.1
18
18
30.8
12
3.2
18.5
13.5
24.1
11.8
9.8
15.1
16.6
13.5
36.6
9.6
8.8
16
46.1
10.8
11.8
23.3
15.3
23.6
43.6
10.1
12.9
18.9
28.5
28.5
14.1
31.8
30.7
28.2
28.2
10.1
51.1
41.3
41
41
14.7
33
32.5
36.3
36.3
13.8
15.6
15.6
13.4
13.4
8
13.4
10.6
9.7
5.6
7.1
11.5
11.5
21.2
2.1
2.8
35
35
24.4
40.8
12.1
17.2
16.6
14.5
28.8
7.1
8.4
11.3
3.4
22.4
5.5
8.2
6.7
41
25.6
5.9
6.1
3.5
18.4
2.6
8.1
24.2
19.8
22.6
17.3
1.7
11.2
14.2
19.2
32
8.6
14.5
11.9
23.3
5.1
13.4
9.4
5.7
41.8
41.8
3.9
2.2
9.3
28.4
3.3
4.7
8.4
8.4
15.2
8.8
5.1
1.6
7.2
18.2
26.8
11.5
10.5
2.5
19.8
11.6
8.8
28
25
21.7
34.7
6
9.2
9.2
11.4
44.7
7.4
7.7
14
44.9
44.9
10.6
21.7
21.7
16.1
16.1
8.5
19.4
20.1
12.4
16.4
11.4
16.8
15.8
29.2
29.2
5.2
35.8
28.7
30.5
32.2
11.2
10.2
2.9
4
28.3
14.7
15.1
13.7
30.7
25.4
11
1.5
8.2
4.2
19.5
14.4
20.1
20.1
17.5
25.5
11
16.4
17.8
11.8
10.9
8
17
18.5
8.1
31.6
11.6
16
23
27.2
17.6
9.2
9.9
14.8
5.7
5.7
9.7
32.9
34.9
9.4
33.8
5.6
3.2
5.9
3.5
2.1
6.3
10.4
6.5
26.9
20.6
8.1
7.9
10.2
23.6
19.9
10.1
20.9
26.5
35.2
35.2
14.2
6.9
15.7
24.8
12.6
2.5
11
17.6
33.2
12.5
5.5
11.5
6.1
30.9
31.1
8.1
12.3
4.1
11.5
43
8.9
11.9
10.6
25.3
31.6
13.2
34.8
34.8
48.1
29.3
9.9
16
18.6
28.8
31.7
8.6
5.5
5.5
36.5
3.8
3.4
33.6
30.6
40
19.8
9.5
7.8
7.8
7.3
49.1
6.8
10.8
8
5.3
20
2.8
20.2
20.2
37.3
17.1
8.2
36.6
39.5
26.7
26.7
13.8
30.5
31
17.5
17.5
9
24.8
24.8
18
18
14.4
6.3
8
40.3
4.3
4.1
7.7
10
37.2
6.9
10
14.9
12.6
13.2
26.4
10.1
17.6
17.6
32
32
1.7
30.2
31.3
21.4
37.8
8.7
12.6
12.6
14.5
33.3
8.2
14.6
9.6
25.4
6.5
8.9
19.9
11.8
9.2
13.5
11.7
16.1
14.7
9.5
9.5
14.2
18.5
18
14.1
28
12.6
17.5
21
30.1
26.5
9.4
12.9
6.5
19.5
21.7
13.8
12.9
11.1
31.3
31.3
12.3
29.5
29.5
17.9
17.9
5.6
5.6
2.1
5.5
5.5
11.4
36.7
27.9
42
42
8.7
45.4
45.4
35.7
46.1
11.5
11.6
17.2
29.1
28
7.3
10.1
7.8
35
5.7
5.4
17.8
15.4
29.6
8.8
10.9
15.5
22.2
3.1
16.4
17.8
56.1
46.6
43.5
43.5
12
21.1
27.9
34
21.1
10.7
25.9
25
14.6
14.6
12.9
7.9
2
2.8
41.4
4.2
14.5
9.1
44.9
11.6
4.7
6.8
6.8
25.4
38.7
6.9
2.5
10.3
4.8
14.4
12.8
17.9
16.7
13.9
11.2
13.1
24.8
21.9
14.2
19.1
9.3
30.1
40.4
31
48.3
7.8
8.6
6.4
19.4
35
14
26.8
32.3
24.9
9.6
12.8
51
41.1
40.5
43.2
3.2
9.3
8
5.7
18.2
11.4
32.3
31.2
25.5
25.5
9.6
11.7
9.3
6.4
29.4
2.9
15.4
14.1
11.3
9.5
14.3
44.4
42.5
30.8
46.9
10.5
21.7
19.2
33.1
31.5
8.6
5.9
5.9
3.6
3.6
8.5
47.5
47.5
39.1
26.7
5.8
39.6
32.6
33.7
33.7
3.6
1.3
10.2
39.8
21.9
1.9
32
31
28
38.4
12.8
44.9
40.7
5.1
38.2
6.4
20.9
12.1
15.9
36.3
14.8
35.2
33.8
22.1
26.6
8
8.9
8.9
4.7
4.7
12.6
2.9
4.3
4.2
4.2
6.3
43.8
43
20.8
47.9
39.4
10.9
8.6
37.3
5
14.2
4.1
2.5
4.2
4.2
1.1
12.8
7.2
33
8.9
1.7
30.2
30.2
19.2
37.3
13.2
19.5
25
25
29.9
10.4
12.4
12.4
9.9
23.1
5.6
6.9
4
35.8
18.6
12.7
29.1
29.1
33.8
25.9
13.3
39.6
36.8
26.7
26.7
13.1
26.1
18.4
14.5
14.5
10.6
9.3
2.5
8.3
29.6
8.9
10.6
14.3
12.8
27.3
6.9
5.4
7.7
13.1
6.5
8
18.2
9.4
49.7
12.1
12.6
1.7
1.7
8.6
10.8
10.9
29.4
26.4
20.8
22.9
11
10.8
8.5
14.2
49.6
8.9
17.2
28.4
17.6
42.8
4.3
19.3
12.2
18.1
11.4
11.5
22.6
22.6
30
30
4.1
7.8
7.8
6.5
40.9
6
28.2
25.9
17.7
31.5
13.1
19.6
21.6
13.6
26.7
9.9
8.2
9.6
7.3
21
14.3
25.9
20.6
17.9
17.9
14.5
20
20.8
32.2
32.4
11.3
15
12.3
26.4
31
14.5
22.6
23.9
16.4
23.6
12.9
7.1
4.5
39.7
4.6
11.4
27
27
42.1
31.9
8.1
15.3
15.3
8.9
49.9
3.8
10.5
1
30.7
22.2
12.6
8.2
3
13.3
13.3
13.5
15.3
10.1
8.3
22.4
11.4
10.7
15.5
22.8
8.6
6.6
18.9
15.5
33.4
15.5
14.5
24.6
22.6
19.5
32.2
5.7
6.7
6.1
3.4
37.2
6.8
45.6
42.7
48.7
25
32.4
17.3
19.7
15.3
29.7
7.4
14.3
26.9
41.9
20.8
8.8
23.1
18.5
11.9
30.9
12.2
15.1
15.1
28
26.9
12.8
36.1
36.1
26.2
26.2
12.3
17.8
17.3
29.9
12.5
9
5.7
12.1
6.1
11
7.8
32.2
44.8
31.5
33.4
6.3
14.1
14.1
12.7
12.7
9.4
23
21.5
13.1
20
12.9
19.1
19.2
13.7
13.7
3.2
11.9
9.5
5.4
19.9
7.3
2.7
6.8
36.7
17.7
14.3
42.9
42.9
39.8
35.4
10
14.6
12.9
32.4
23
7.9
22.4
18.3
18.2
48.5
13.4
24.1
15.9
22.7
35.9
11.3
22.1
24.1
15
4.4
9.9
22
19.9
11.9
18.9
11.3
47
37.6
40.3
39.1
7.2
36.3
46.1
28.7
36.8
10.9
15.2
11
8.8
13.5
10.1
23.2
22.3
14
20.4
7.6
16.7
14.8
48.3
24
5.3
13.4
11.4
11.6
28.9
11
11.7
16.3
22.4
22.2
10
11.7
11.9
29.6
10.5
6.1
43.9
34.2
30.3
30.3
12.9
9.2
10.3
15.7
7.8
11.4
15.9
19.9
26.4
29
6.9
16.9
8.1
32.3
26.8
1.4
11
5.9
7.3
4.5
13.6
39
41.6
48
6.8
14.1
28.3
29.4
15.2
24.1
5.4
6.2
3.7
5.9
36.1
12
18.3
16.8
13.4
13.4
5.5
18.3
13
16.6
15.3
4.8
1.7
2.9
38.6
1.5
7.7
8.1
14.5
8.7
30.9
13.9
25.9
16.1
14.8
17.6
1.4
29.6
30.8
37.4
25.5
6.2
10.5
4.6
41.8
3.5
10.5
14.1
11.8
18.8
11
3.4
5.1
5.1
6.3
5.8
9.3
19.9
23.8
21
21
10.4
15.3
13.2
28.7
28.7
9.5
7.8
14.5
11
38.1
10.7
18.8
14.5
27.5
8.4
16.2
30.9
26.1
21.4
34.9
7.9
18.1
18.6
15.2
12.2
8.7
12.5
18.8
25.7
9.7
8
4.4
6.5
22.8
4.2
11
12.8
12.8
8.9
27.2
13
26.5
28.8
35.4
37
5.3
14.4
14.3
10.3
5.4
7.9
19.7
14.4
16.5
13.8
7.7
4.5
4.5
12.7
34
7.4
13.2
12.1
13.2
24.4
11.1
15.8
18.4
27.9
29.1
4.8
4.8
7.9
5
5
8.6
19.8
17
23.3
16.9
7.5
10.6
8.1
5.9
6
7.8
10.3
10.3
27
12.4
5.7
16.7
16.7
35.4
47.6
8.3
11.8
9.6
25.3
29.2
3.3
23.4
26.2
28.5
39.8
14.5
5.6
5.6
19.1
21.9
8.5
15.8
15.8
26.3
34.5
4.7
4.1
4.1
18.2
38
7.4
33.1
33.1
44.2
34.5
9.6
10.2
10.2
7.5
26.5
9.2
17.1
17.8
15.4
36.6
6.4
11.2
19.9
27.3
28.8
12.6
17.2
17.2
25.2
12.2
11.9
19.3
16.4
29.4
29.4
15.8
50.1
44.1
36.2
40.4
8.2
10.2
10.2
16.2
1.6
5.9
16.6
12.7
48.2
15.4
15.2
27.4
33
20.7
20.8
7.8
23.7
23.3
14.3
31.7
6.1
26.4
29.1
22.7
32.9
14.8
16.4
19.2
2.5
19.8
1.8
14.7
5.8
7.8
36.9
7.5
13.9
1.6
31.2
22.8
8.6
0.9
0.9
41.1
1.4
9.4
36.3
36.3
37.4
46.2
8.7
15.3
21.3
31
23.4
1.5
27.3
20.9
23.6
36.7
14.4
32.4
25.4
27.9
27.9
2.6
2.8
2.8
2.2
6.1
3.9
11.3
11.3
17.7
17.7
6.9
7.6
7.6
11.5
28.9
3.9
4.9
4.9
3.2
38.6
8
7.2
12.2
6.5
25.5
9.9
23.8
14.4
17.2
22.9
12.2
12.3
12.3
26.4
30.8
1.6
16.5
16.5
35.8
23
11.8
35.8
25.7
30.4
40.9
4.3
8.1
3.9
26
6.1
6.7
9.2
9.2
32.4
26.7
5.8
18.8
22.7
19
20.1
13.8
25.7
21.9
16.7
16.7
13.5
22.9
24.4
22.4
32.2
15.7
24.6
29.4
30.4
24.8
3.9
18
29
17.6
21.2
13.9
31.3
31.3
46.2
35.1
9.4
18.7
12.6
9.9
32.1
13.7
1.6
1.6
6.2
6.2
8.2
10.7
13.3
22.2
22.2
6.1
9.3
9.3
32.7
11.2
9.6
19.2
27.7
21.5
35.5
6.2
5.9
5.9
27
21.1
11.4
12.9
20
13.3
13.3
14.1
34.4
44.2
29.9
35.3
7.9
7.8
7.8
4
4
2
6.4
3.4
19.9
10.3
12.6
22.4
16.9
25.7
14.8
4.9
15.8
19.2
16
33.8
7.1
17.9
18.6
13.3
33.2
6.1
34.4
34.4
22.4
22.4
6.4
8.3
10.9
22.4
22.4
9.2
17.7
17.7
19.2
19.2
7.3
6.5
6.5
44.9
44.9
8.4
4.6
4.6
1
1
8.9
17.6
8.9
12.1
36.1
8.7
10.2
12
18
4
13.3
12.2
1.8
11.3
8.7
11.2
16.1
20.4
12.7
12.7
5.6
41.7
41.7
44.3
28.6
8.5
7.3
12.7
9.4
44.9
3
5.1
5.6
15.3
25.4
6.6
14.2
19.5
33.3
18
12
22.3
32.9
25.9
25.9
13.3
29.5
30.2
35
26.3
7.4
16.1
20.8
18.9
18.9
9.6
16.8
21.7
14.7
14.7
10.5
18.4
17.1
17.7
25.4
9.7
15.9
12.3
32.8
9
1.1
7.1
6.9
5.6
7.9
13.4
5.3
6.7
2.8
17.1
13.4
16.3
16.3
27.3
27.3
5.7
8.8
8.9
5.8
5.8
6.9
12.2
12.2
2.8
21.9
3.7
9.2
4.7
1.5
15.2
7.7
14
14
29.6
24.8
14.6
7.7
5.6
22.8
22.8
7.3
14.7
8.5
26.5
26.5
12.2
2.2
1.4
5.7
1.1
8.1
20.4
21.4
15.2
19
14
16.1
14.7
32.6
9
4
3.2
4.2
16.4
16.4
14.9
17.1
19.8
28.4
14.8
12.5
8.5
7.1
6.5
9.6
6.4
19.5
14.6
24
31.4
8.5
24.6
19
17
17
3.4
26.6
29.6
43
18.2
4.9
14.2
16.6
13.1
2.5
7.4
10.5
8.3
32.4
22
13.4
20.6
27
21.9
33.2
11.8
28.1
35.6
31.1
25.7
4.7
8.7
4.7
17.6
5.9
5
11.1
10.7
34.1
21.1
8.5
15.2
10.5
29.4
7.2
4.8
17
17
14.4
11.6
9.5
21.8
21.5
19.7
29.4
12.8
33.1
30.8
30.8
26.8
4.7
5.2
2.3
4.7
26.1
12.9
30.1
24.6
35.6
20.1
12.5
20.2
20.2
17.6
7.8
10.6
21
21
31
28.7
14
22.3
14.9
19.4
22.7
10.3
17.7
10.6
11.5
11.5
12.9
10.5
12.7
5.8
5.5
14.9
14.6
13.2
10
8.3
8.2
2.4
12
1.9
38.9
1.3
13.2
4.9
4.7
10.7
11
4.1
3.9
29.8
19.3
0.3
12.8
2.4
6.4
10.8
0.8
6.9
8.5
6.4
10.1
9.3
10.2
15.9
25.7
31
13
3.9
6.7
35.6
4.1
6.1
5.7
11.9
31.4
31.4
8
27.2
18.7
10.7
10.7
3.4
19.6
17
48.9
13.7
9.6
20.4
23.2
37.3
29.8
8.4
14.5
4
10.7
10.7
4.5
5.1
7.2
25.8
4.8
3.7
14.3
20
34.8
14.7
14
15.6
14.2
25
9.3
4.9
3.4
11.8
3.4
3.4
9.8
28.8
17.9
22.3
22.3
14
22.4
27.9
6.4
23.6
11.1
11.1
11.1
37.2
37.2
7.6
2
9.7
2.4
7
13.5
6.2
5.8
3.4
23.5
14.3
13.7
17
23.6
8.6
3.5
15.2
9.4
12.3
48.8
8.4
17.4
14.8
9.4
9.4
9.9
11
11
22.7
22.7
9.9
7.9
12.8
21.8
19.7
8
20.9
20
10.3
25.5
13.9
13.4
11.8
22.7
23
7.4
17.8
17.8
31.8
27.9
7.9
14.7
14.7
32.8
32.8
11.5
20.4
14.6
10.7
12.8
4.1
16.9
15.9
13.4
13.4
12.5
13.4
10.4
32.4
7.4
11.6
18.6
24.4
28.8
19.8
2.3
13.9
13.9
33.6
34.2
13.1
11.7
11.7
5.7
22.3
11.6
23.8
34.6
40.5
27.6
9.6
11.8
10.1
22
34
7.6
25.2
26.7
9.6
9.6
10.3
23.8
23.8
47.4
21.7
10.4
16.4
24.2
17.1
34.8
9.6
27.6
29.7
30.9
13.3
8.7
39.3
36.5
28.8
28.8
13.8
28
39.7
31.5
26.6
14.5
40.3
43.2
46.8
29.6
8.7
1.6
9.8
12.4
42
2.2
8.8
8.8
19.8
19.8
5.9
6.4
2.8
25
5.2
13.5
43.4
39.7
3.1
3.1
2.9
15.5
5.1
13.2
33.7
14.6
23.7
24
23.3
17.6
5.6
15.1
15.1
29.3
34.2
9
9.9
9.9
5.9
5.9
14.8
36.5
29
28.9
43.4
10.8
6.7
6.7
47.8
8.2
8.6
24.7
27.6
21.4
21.4
8.8
11.8
14.9
8.1
8.1
8
11.8
16.6
25.4
29.3
4.5
3.2
1.3
27.8
27.8
11.5
23.1
11.9
12.6
18.1
14.1
22.2
25.3
34.4
16.7
13
23.6
14.9
13.1
24.8
10
14.8
7.1
5
5
5.6
24
22
15.3
5.3
9.3
16.5
7.5
9.2
36.8
9.8
17.7
10.2
28
7.4
8
10.1
8
4.3
35.7
14.9
33.8
25.8
32.8
15.6
12.4
19.6
22.1
15.8
3.5
11.3
15.6
15.6
10.3
28.1
9.4
29.9
21.4
21.2
7.3
3.1
30.1
30.1
43.6
21.5
11.7
25.1
26.6
21.4
21.4
5.2
10.5
23.2
13.5
29.2
12.7
13.6
13.6
31.8
11.7
6
9.2
12.2
31.8
22.9
13.7
4.7
1.8
16.5
23.5
5.7
6.3
3.6
25.1
3.9
12.9
31.4
23.2
16.6
18.9
6.1
14
14.8
46.3
46.3
5.8
2.4
4.8
5.4
14
14.8
13.2
15.3
12.7
25.9
12.1
14.9
13.4
10.3
30.7
7.8
9
12.1
31
4.8
14.5
9.6
15
12.4
15.3
6.2
6.7
7
13.7
24.5
7.8
21
15.4
18.5
17.3
14.3
18
15.2
2.3
15.4
10.2
15.7
10.5
28
26.3
11.4
16.8
16.4
30
29.3
3
6.7
7.3
15.5
15.5
9
12.7
11.1
28.3
29.2
9.3
15.6
10.9
31.2
23.4
9.6
13.6
13.6
27.6
26.8
13.9
9.4
1.8
13.6
6.2
8
9.6
4.6
11.1
11.1
13.9
8.9
10.7
32.8
32.8
1.8
18.5
16
34.6
10.1
12.2
16.3
20.6
11.3
11.3
9.3
11.9
11.9
29.9
27
7.8
1
9.6
20.2
41.4
3.9
7.6
11.4
9.6
29
5.7
13.7
18.9
17.5
17.5
13.2
12.1
14
25.8
24.9
6.1
6.8
17.1
29.1
29.1
6.9
10.1
7.2
21.9
14.8
3
9
9
8.1
36.1
5
11.3
14.1
39.5
29.8
12.8
27.2
21.4
24.8
30.5
11.4
11.2
11.2
31.8
27
8.3
28.9
26.2
26.8
26.8
6.6
20.4
20.4
16.9
31.8
10.3
34.7
27.6
29.6
29.6
3.7
14.1
14.3
14.5
14.5
12.5
1
7.9
21.4
0.1
10.5
20.5
25.7
36.5
19.9
3.7
25
27.9
16.8
21.7
9.9
17.9
17.9
27.1
10.9
12
28.2
28.2
30
30
1
10.8
2.1
21.4
21.4
11.5
18.6
23.3
16.6
38.1
11.9
22
28.8
20.2
20.2
2.4
6.8
6.8
12.7
19.4
4.2
6.7
16.2
9.6
14.3
12.8
21
17.1
25.9
18.3
13.3
11.6
8.2
5.5
21.9
3.7
11.3
9.3
7.9
4.6
9.6
19.1
19.1
12.3
16.8
4.9
5.8
1.5
26.8
36.8
9
28.9
25.1
21.5
24.2
13.8
4.5
1.6
2.2
2.2
8.6
3.4
6.9
21.2
21.2
9
11.3
4.8
17.3
21.4
7.9
20.6
20.3
14.5
16.9
6.4
15.6
18.6
17.7
46.8
10.2
9.8
9.1
27.9
12.1
3.9
8.7
1.6
7.4
19.4
10.7
13.5
16
29.3
12
11.5
11.1
11.1
28
7.2
12.2
18.7
18.7
31.5
24.6
9.7
22
22.6
15.1
32.3
3.5
19.2
9.7
12.3
7.8
4.9
4.8
6.9
6
39.9
4.6
8.5
7.8
25.5
25.5
14.7
20.8
26.6
31.4
26.3
12.3
19.7
19.7
12
25.2
3.7
7.4
1.2
4.7
14.6
6.1
17
11.2
32.7
32.7
5.8
6.1
6.1
19.9
1.2
28.5
42.8
40.4
29.6
29.6
14.1
20.9
17.1
24.9
14.7
1.4
31.4
34.3
39.8
39.8
13.1
15.2
16.5
27.6
10.8
7.6
10.1
20
34
26.4
10.1
10.4
7.1
30.5
24
12.5
16.6
18.6
25.7
25.7
9.7
14.4
20.3
36.6
13.1
11.2
50
50
42.2
35.9
14.9
23.2
21.4
32.3
16
2.8
2.8
4.8
3.1
5
4.6
8.5
18.1
28.4
37
11.2
14
12.3
8.9
30.4
14.5
10.9
12.6
6.4
22.3
10.3
18.7
14.1
14.2
10.3
5.8
1.3
5.2
13.2
3.9
12.3
23.8
31.5
16.9
43.3
13
18.7
21.2
25.1
29.8
6.5
13.5
12.7
33.6
13.7
13.6
6.2
4.9
39.7
24.8
12.5
24.8
27.6
38.9
36.2
11.5
12.3
12.3
27.3
30.8
8.1
15.2
15.8
49.1
27.1
13.4
5.4
10.6
6.9
4.3
9.8
19.8
23.2
39
17.9
6.8
0.2
10.1
5
30.4
5.7
2.7
5
35.4
26.4
5.6
20.3
15.8
17
34.8
8.9
26.2
24.3
18.8
14.5
2.9
14.7
17.4
8.7
33
11.8
13.2
18.7
15.8
15.8
1.3
0.7
0.7
4.4
4.4
10.6
12.6
15.2
29.4
26
5.6
8.1
2.2
7.4
7.4
6.8
11
7.3
6.4
9.5
1.4
3.7
14.2
7.4
36.9
7.5
5.9
3.4
23.1
2.9
13.6
9.7
13
5.2
5.2
2.1
9.5
3.3
19.7
37.4
11.3
25.9
33.3
15.7
24.3
2.2
10
11.6
42
32.3
5.5
10.1
4
37.5
30
3.9
4.2
4.2
33.3
33.3
8.8
12
6.7
4.8
4.8
8.2
29.8
32.5
46.8
21.1
8.8
20.2
24.8
23.7
30.3
14.3
22.5
29.4
30.6
13.1
11.1
19.3
19.7
18.6
33.2
6.5
7.8
4.3
24.1
24.1
1.2
10.9
9.8
9.4
9.4
9.8
19.7
19.7
10.8
15
12.4
19.7
17.7
25.6
29.7
11.2
6
11.5
19.5
10.6
7.9
3.7
14.9
6.2
6.2
13
14
10.1
7.7
7.7
10.6
15
17.4
10
29.3
14
13.7
14.7
15.2
33.2
2.3
17.4
17.4
10.2
7.9
4.8
6.7
6.3
16.5
3.5
4.6
36.6
26.8
24.6
39.6
8.9
11.4
11.4
8.1
27.5
3.3
29.6
29.6
20.6
20.6
14.4
16.1
14.5
13.2
10.8
10.3
24.1
24.2
31.5
21.8
14.2
23
23
16.7
19.9
3.5
13.5
18.1
14
34.6
13.3
27.9
27.2
31.2
31.2
11.9
39
34.1
43.9
25.1
5.7
32.3
32.3
20.6
20.6
11.2
10.5
8.5
28.5
4.2
8.6
17.2
17.2
15.2
15.2
1.6
30.7
24.3
22.8
26.4
12
24.2
13
13.4
1.3
12.5
7.9
10.7
21.4
21.4
9.3
14.4
11.5
25.4
29.1
4.8
34.4
36.9
29.2
39.7
6.6
0.7
10.2
21.3
5.3
11.9
2.5
2.5
8
8
7.9
9.6
11.6
3.5
24.3
5
18.2
22.7
23.4
18
9.4
10.5
16.2
23.8
25.5
10.3
11
11
11.7
29.7
6.7
21.1
21.3
17.5
17.5
9.5
13.5
16.3
26.7
7.7
5.6
3.1
3.7
6.1
6.1
6.6
5
3.7
23.8
14.7
10.3
15.2
11.6
28.4
26
6.7
11.3
13.3
12.3
20.4
7.3
23.2
20.3
18.5
23.3
9.9
27.7
27.8
16.6
35.3
7.4
1.3
2.2
1.1
13.6
13.2
8.2
5
6.8
6.8
13.5
19.9
15.3
26.5
28.9
7.4
18.5
17.1
30.7
15.2
11
29
31.8
31.5
24.9
5.7
7.5
3.9
18.7
35.7
12.4
31.7
26.4
37.2
37.2
13.9
44
44
30.6
35.1
12.6
27.5
22.4
19.4
35.2
12
19.7
21.4
14.9
29.6
5.3
8
7.3
15
6.2
14.9
9.8
8.7
21.5
6.9
6.7
7.5
16.8
11.8
11.8
6.3
6.9
4.9
26.8
2.1
17
46.8
54.9
42.7
42.7
7.4
16.8
14.3
40.5
40.5
6.4
14
19
47.6
32.8
14.1
20.2
14.4
11.7
11.7
9.5
18.7
17.1
32
9.6
6.7
13.7
7.6
26.5
12.3
9.1
11.2
14.3
30.8
24
11.8
36.2
36.2
25.6
29.3
4.9
25.4
25.4
25.2
18.3
8.7
12.1
9.5
27.1
5.1
12.3
8.6
3.4
21.4
21.4
13
29.6
29.2
34.6
23.9
14.5
14.5
19.1
31.4
11.5
1.8
14.2
10.1
20.2
7.7
2.6
27.4
27.4
29.7
19.1
3
13.8
25
15.4
20.1
6.6
0.6
11.1
30.5
30.5
11.4
44.1
41.3
43.4
33.5
10.4
14.6
6.5
8.3
31.8
9.7
15.4
14.1
12.4
35.5
4.7
27.6
37.5
27.4
43.4
8.2
18.7
18.2
17.1
15.2
3.2
10.7
18.6
8.1
47.7
12.6
26.6
27.4
22.5
24.5
12.4
18.3
18.3
24.3
35
12.8
32.4
32.4
39
24.5
2.4
20.1
17.7
15.3
11.8
9
50.1
47.2
40.1
40.1
5.3
5.4
4.6
36.2
36.2
8.2
1.6
1.6
5.3
5.3
5.9
10
18.7
35
14.3
11.2
20.3
13.5
4.6
4.6
6.9
19.8
24.6
15.7
15.7
12.8
19.7
21.7
30.9
14.1
14.5
9.3
10.7
43.6
8.3
2.1
15
17.2
24.3
12.9
11.6
7.4
7.5
18.4
6.7
5.9
9.2
5.6
34.1
20.4
5.2
27.3
27.3
34.9
34.9
8.7
16.1
12.5
28.8
9.7
7.7
17.4
9
28.2
11.7
5.8
18.8
18.9
17
30.5
3.9
5.2
3.9
38.4
38.4
13.5
8.3
6.2
5.8
23.5
9.5
16.7
11.8
27.3
29.7
11.6
19.4
22.3
17
17
14.4
13.7
11.3
45.7
16.3
11.8
21
21.2
36
36
6.5
12.4
12.4
28.7
23.3
7.6
26.2
18.1
42.8
21.6
7.3
27.5
17.7
42.5
16.8
2.9
9.6
10.9
18.7
12
9.7
15.9
12.1
30.8
25.9
9.2
15.7
20.1
15.8
12.5
14.9
14.8
14.2
10.6
26
13.2
9
11.4
7.6
4.7
9.5
26.9
26.2
22.1
34.9
10.7
25.6
18.6
36.4
41.2
13.3
10
8
21.7
21.7
2.4
19.6
11.3
21.9
41.4
3.4
20.3
26.2
21.7
36.4
11.9
15.3
17.9
26.2
26.2
9.8
8.7
7.5
20.2
6.6
13.9
11.9
15.3
7.1
8.5
10.9
49.7
40
39.9
38.2
13.8
17.7
21.6
13.8
13.8
8.5
3.2
8.7
38.2
16.4
10.5
9
16.1
6.1
25
9.6
12.8
12.8
12.7
9
9.9
12.9
15.2
26
27.8
8.1
12.6
20.8
10.1
33.7
9.9
25.6
16
0.4
14.7
12.2
8.3
3.2
9.9
21.3
10
39
51.6
36.8
41.4
4.3
8.9
3.7
36.3
18.9
13.2
10.9
16.5
8.5
9.6
6.5
42
35.4
31
48.4
6.8
5.7
8.5
5.6
34.1
9.1
19.1
11.1
13.7
13.7
7.7
12.7
24.4
25.9
39.2
12.9
6.2
6.2
1
22
14.4
45
32.3
35.9
47.1
1.1
7.9
7.9
9.5
12.9
7.6
19.6
19.6
12.3
17.2
7.9
10.3
16
33.6
26.3
12.7
39.3
28.7
29.1
32.9
8.1
17.3
11.4
25.3
29.7
13.8
34.3
34.3
31.3
30.7
7
12.5
15.3
31.8
14.4
2.6
14
12.5
20.5
13.3
9
20
15
18.5
18.5
10.4
12.5
15.1
28.2
7.9
9.4
9.7
7.1
18.1
46.9
9.7
15.1
15.1
30.3
6.5
6.5
28.7
25.9
21
28.8
5.4
16.6
24
1.7
1.7
12.1
10.5
5.8
34.1
34.1
6.4
7.5
12
32.8
7.1
7.2
13.8
13.8
21.7
10
12.3
55.3
45.5
35.7
40
13.4
38.4
37.3
23.3
30.4
34.3
34.6
41.4
35.7
31.3
8.3
10.3
19.5
28.2
26.5
13.2
23.4
23.4
21
25.7
12
27.8
27.9
22.8
15.6
8
9
11.7
42.2
3
4.4
5.6
8.4
16.5
16.5
8.4
19.7
11.8
26.3
9.1
11.1
19
13.3
25
18.3
14.1
16.8
13.6
9.3
24.7
14.7
12.6
14.4
31.6
23.6
8.7
15.1
2.5
11.1
32.4
13.2
14
10
23.2
6.9
12
15.3
12.9
10
24.1
15.9
31.7
26.6
32.4
35.8
13.5
24.5
24.5
28.9
25.5
6.6
10.6
7.9
5.9
41.3
9.1
21.1
21.1
17.3
18.8
15.5
42.4
43.9
36.1
0.7
8.4
33.2
43.1
32.3
36.6
6.6
20.3
27.4
32.5
34.1
11
10.4
10.7
6.9
29
8.6
39.5
36.7
27.1
27.1
2.3
16.6
19.1
35.7
14.5
11.1
30.5
24.8
27.1
27.1
9.4
20.1
18.2
31.9
26
6
11.8
11.8
37.9
7.7
14
27.7
15.2
42.5
16.8
7.2
10.5
5.3
34.1
34.1
13.1
5.7
6.1
38
38
7.6
12.5
6.8
33.5
24.3
13.3
19.4
18.2
15.7
29.4
13.7
39.6
38.1
36.7
36.7
6.9
40
42.6
34
34
8.1
8.4
11.8
5.8
5.8
12.4
22.3
22.3
43.8
43.8
11.7
26.4
20.5
42.2
43.4
10
17.5
17.3
28.8
7.9
0.9
1.1
1.1
11.8
11.8
6.1
4.3
7
18.6
4.2
4.5
8.6
7.3
16
40.8
12.1
10
4
43.2
21
11.8
13.6
11
8.6
8.6
1.6
14.7
5.1
23.9
34.2
7.6
8.1
8.5
36.5
17.8
9.7
11.8
11.8
25.4
25.4
7.5
8.8
8.8
28.1
28.1
14.6
20.2
26.5
15.1
32.4
8.6
7.1
7.1
15.9
4.5
7.8
11.2
11.2
33.3
26.6
11.8
21.4
24.5
26.4
17.3
12.1
14
13.7
28
26.7
9
12.7
10.7
4.6
4.6
12.4
14.7
16.9
19.8
19.8
10.3
13.2
16
11.8
26.5
6.4
26
13.6
15.4
32.6
11.8
4.4
7.9
28.1
28.1
13.5
5.1
5.4
6.4
41.8
6.8
3.2
7.8
40.8
5.5
14.6
33.7
33.7
41.3
13.5
2.4
2.2
10.2
5.8
30.3
12.1
14.8
19.2
11.6
25.3
13.9
18
18
13.7
13.7
7.6
1.6
7.7
1.8
38
13.1
6.2
9
11.9
22.7
2.2
12.4
12.4
42.6
7.3
13.1
9.6
13
30.1
10.4
3.1
12.7
17.7
13.9
43.5
7.9
0.2
0.2
3.9
3.9
9.6
23.7
15.5
16.5
16.5
8.5
8.1
6.8
6.8
31.8
6.4
6.3
5.6
43.2
7.8
12
1.8
1.8
14.4
20.5
8.1
14.7
15
35.6
13.1
12.1
31.5
28.5
18
18
3.1
1
2.2
29.7
20.7
11.7
20.1
22.3
19.1
19.1
8.1
13.4
16.6
26
26
12.1
17.5
17.8
11.6
13.7
6.5
10.2
7.8
19.7
5
9
10.6
10.6
26.7
9
9
18.4
13.4
27.7
29.7
6.7
20.9
20.1
17.4
17.4
13.5
2.1
6.9
2.6
23.6
8.5
18.4
10.6
30.7
30.7
8.3
17.7
10.4
30.3
30.3
11.6
40
35.3
44.9
9.5
1.6
6.4
6.4
8.4
39.7
13.8
21.3
24.9
33.9
16.3
8.1
31.2
39.8
26.2
48.9
10.4
16.5
9.9
7.8
27.6
12.1
17.8
17.7
13
24.1
9.5
12.4
12.4
7.7
27.7
14.6
40
34.9
30.1
30.1
9.7
16.8
11.8
24.5
7.4
9.2
12.3
12.3
4.8
23.4
3.4
15
12.2
10.9
10.9
10.3
30.9
32.4
11.8
23.5
9
7.2
5.3
5.3
9.5
12.7
5.5
3.4
16.8
5.3
5
3.7
3
7.7
18.7
8.5
7.8
16.3
25.7
6.7
8.3
11.8
11.8
2.2
2.2
13.3
7.9
5.7
17.8
3.2
8
6.2
15
45.4
45.4
7.9
8.7
8.7
23.3
23.3
6.4
10.5
11.4
7.9
21
3.3
9.1
11.7
19.5
5.6
7.7
4
8.4
2.9
21.8
11.7
5.6
8.6
35.3
0.9
9.6
17.3
14.3
7.4
27.7
6.6
14.2
13.8
46.6
11.3
10.6
23.4
23.4
16.3
33.5
12.2
15.3
17
22.4
22.4
12.3
32.7
32.7
23.2
22.6
12
27.8
27.8
19.6
19.6
5.8
20.2
16.3
47.1
47.1
3.8
36.5
33.6
42.3
28.8
9.5
11.2
10
9
23.4
10.6
12.7
12.7
11.4
29.4
14.1
26.9
26.6
18.6
31.8
9.8
16
21.2
30.7
30.7
4.6
11.6
12.4
15.4
43.1
13.8
20.7
20.6
24.2
24.2
6.8
7.8
7.8
16.2
2.5
8.1
21.6
18
19.3
34.3
0.9
11.1
19.8
24.2
24.2
11.2
16
11.1
9.6
9
2.8
2.8
10.5
22
40.8
1.9
33.5
23.7
28.2
47.4
2
30.4
27.6
23.7
28.6
11.5
14.7
13
11.2
19.1
13.8
25
21
32.9
32.9
14.6
27.8
27.8
29.7
29.7
11.5
30.6
33.8
11
11
11.8
8.1
4.5
28.2
18
9.8
7.4
11.1
30
30
8.9
21.1
22.1
17.8
30.8
4.7
4
6.5
38
16.4
11.9
8.7
6.1
18.3
27
10
13.8
7.6
7.6
7.6
7.3
28.3
31.1
42.4
16.2
13.1
0.7
5.1
6.8
12.6
2.2
8.4
4.1
6.5
6.5
4.7
6.7
15.4
11.7
11.7
10.3
14.8
9.5
27.2
8.3
7.1
12.9
14.5
4.8
9.2
4.5
16.6
16.1
16.2
13.4
10.9
15.5
19.8
19.2
19.2
14.2
12.2
12.5
34.6
6.1
1.5
9.8
9.8
32.8
19.9
10.8
10.7
10.7
7.6
33.5
21.3
27
37
33.6
31.5
14.5
3.9
6
3.4
3.4
9.9
8.6
14.5
29.8
29.3
7.1
19.3
19.3
17
30.9
10.8
24.1
30.1
33.8
16.2
6
25.1
25.1
18.3
13.7
6.3
18.7
23.3
15.6
34.7
8.8
14.8
11.1
23
31.5
14.2
23.5
23.5
34.3
34.7
14.5
18.7
14.3
10.1
24.7
6.9
9
8.2
34.2
34.2
2.4
6.7
6.7
6.5
5.6
12.6
38
30.9
41.8
32.4
4.8
16.5
15.5
47.5
14.4
2.5
9.1
6.6
11.3
18.5
2.7
9.7
5.9
7.3
7.3
13.7
14
10.1
23.8
7.4
11.4
14.5
14.5
29.8
25.8
8.8
12.5
21.3
10.4
25
7.6
8.4
5.7
25.2
34.1
14.2
42.7
32.7
46.7
46.7
4.5
3
11.7
4.1
4.1
10.1
13.9
21.2
14.3
31.4
14.8
39.9
31.3
45.5
45.5
2.7
17.5
26.1
17.7
17.7
8.5
17.3
26
23.4
33.9
8.2
22
22
22.1
17.5
12.3
30.9
30.9
23.1
41.1
10.8
12.5
14.1
12.5
12.5
5.7
9.3
9.3
1.3
19.8
9.5
13.2
15.8
45.8
19.6
5.9
13.4
5.4
31.6
23.2
16
37.4
44.8
32.3
30.4
12.8
37.7
29.5
27.9
27.9
12
16.7
18.7
28.8
11.9
12.3
19
25.2
23.6
14
9.6
26.3
16.2
20.3
20.3
11.9
11.7
16.8
24.9
10.2
8.5
11.5
19.1
33.1
14.4
13.6
20.6
22.5
6.6
18.1
2.5
35.4
24.2
23.7
41.5
6.1
17.7
17.7
45.9
14.6
6.8
1.2
4.6
21.7
11.6
12.7
12.5
11.9
26.5
8
10.3
18.7
12.9
11.7
27.9
5
20.9
14.2
18.9
13.1
14.7
5.2
10.5
19.9
34.5
12.1
20
13.2
2.9
12.2
6.3
22.3
23.6
39.1
39.1
2.4
33.6
33.6
28.8
28.8
8.3
39
32.6
37
42.9
9.4
4
6.4
9.1
4.5
3.1
12.8
12.8
36.3
22.5
4.5
8.3
2
5.5
26.7
5.4
18.4
13.4
32.2
32.2
15.2
5.4
5.4
38.5
4.9
10.1
25.4
32.5
32.9
32.9
14.9
11.3
11.3
36.1
5.6
5.4
21.5
12.8
1.8
13
13.8
21.6
27.3
16
16
13.4
16.6
19
1.9
15.1
3.6
7.2
9.6
18.5
37.2
3.4
9.1
2.5
1.3
1.3
6
27.8
27.8
26.9
44.9
9.7
13.2
14.5
6.7
6.7
12.3
31.8
35
17
17
13.6
47.9
47.9
46.5
42.1
2.1
14
15.8
10.1
12.6
5.9
5.8
4.7
26.7
19.6
11.7
15.7
15.7
30.6
10.4
8.8
17.8
22.3
17.6
17.6
10.3
17.8
17.8
26.5
9.3
3.9
7
7.3
28.8
28.8
11.6
39.1
26.5
31
28.8
8.6
10
8
23.8
5.4
5.4
17.2
13.1
17.9
13.6
13.1
12.5
14.8
23
13
2.4
4.4
9.8
20.5
20.5
13.6
42.1
35
31.6
36.2
10.5
11.1
10.4
20.8
34.6
3.7
13.2
10.2
15.3
10.4
11.2
26.3
34.9
25.7
25.7
4
17.4
18.3
32.9
18.2
13.8
50.9
49.4
4.3
39.7
13.1
15.4
15.4
26.3
9.7
13.2
27.3
25.9
25.1
22.2
5.7
8.6
5.6
1.3
5.5
14
6.4
8.6
41.2
14.4
12.2
14.1
9.1
8.4
47.4
10.3
20.2
31.3
46.5
46.5
4.4
6.3
3.6
6
2.4
5.3
3
5.9
17.9
3.3
13.6
3.8
7.6
39.7
39.7
3.9
5.8
4
3.5
15.7
8.2
15.5
10.1
11.4
7.2
11.6
14.9
17.1
27.6
27.6
18.2
56.6
51.8
39.9
4.9
14.5
43.7
43.5
32.8
37.1
9.6
14.1
11.8
10.6
10.6
5.3
5.9
5.9
24
3.3
5.7
31.8
38.8
44.1
33.1
10
22.7
28.4
26.1
29.1
12.1
15
10.9
11.7
16.1
10.3
39.6
49.4
37.3
42
5.8
4.5
10.6
26.1
13.4
12.4
1.3
6.5
21
39.1
12.9
31.9
39.8
29.2
44.6
13.8
24.9
19.8
25.1
30.8
12.6
10.5
10.5
15.3
0.6
2.7
5.2
6.3
5.4
14.7
14
18.8
16
33.6
43.6
12.9
18.6
21.8
26.3
28.7
15.7
27.1
28.4
36.5
36.5
10.8
18.9
13.3
26.1
28.7
6.9
7.7
10
5.2
23.7
9.1
16.6
11.4
10.2
6.7
1.8
3.8
1.8
21.3
21.3
5.1
12.9
12.9
13.3
27.6
8.4
11.6
4.5
20.8
5.8
5.4
7.6
1
28.3
36.5
14.1
6
1.3
38.8
15.6
12.3
4.2
9.9
37.4
20.7
8.1
16.8
9.5
10.9
25.6
8.4
2.4
3
2.6
20.8
12.9
24.6
27.2
23.9
28.8
4.4
6
1.9
23.6
23.6
1.2
10.6
6.6
4.9
8.6
9
16.5
19
13.3
9.7
9.2
13.1
20.6
24.3
11.2
6.2
6.5
7.9
35
35
7.6
19.3
16.4
28.8
14.5
10.4
37.2
34.9
30.7
25
8
9.3
12.1
20.2
20.2
1.5
18.8
9.7
12.2
12.2
8.4
5.5
9
36.7
21.6
10.2
13.2
13.2
26.7
26.7
14
25.2
21.7
34.3
33.9
7.2
11.7
11.7
8.1
28.9
7.5
0.6
4.8
7.1
7.1
6.8
11.7
11.7
3.6
21.7
5.8
3.5
6.3
18.4
2.1
35.9
6.2
12.8
23.5
9.1
5.6
3.9
7.3
2.1
20.8
5.1
10.5
10.5
39
17.7
13.8
10.9
8.3
19.4
3.9
9.2
50.4
38
40.4
40.4
10.2
26.3
34.2
23.9
40.1
8.7
4
4
31.5
11.5
5.9
19.3
19.9
16.2
16.2
7
8.3
9.4
4.2
4.2
7.9
7.3
7.3
6.9
21.6
5.5
16.1
14.4
14
29.7
5.8
26.8
24.8
20.8
20.8
7.1
13.1
6.6
38
12.6
8.9
10.7
11.2
6.4
9.6
11.1
24.3
21.5
16.6
23.4
7.4
45.3
37.1
32.6
49.4
9.7
47.3
44.4
33.7
47.5
7.8
2.2
1.4
15.2
5
4.2
7.8
6.7
14.8
6.5
10.7
45.7
36.4
38.6
39.2
11.7
2.9
4.2
8.5
8.5
9.5
17.2
15.4
8.1
24.3
10.9
7.5
0.9
20.1
35.2
2.2
4.9
4.9
10.4
21.5
1.8
8.5
4.3
12.5
42.2
8.7
3.2
3.2
11.3
7.9
13.3
22.2
29.9
38.3
17.9
4.3
3.9
3.9
41.3
6.7
10.6
13.1
12.3
8.2
26.4
5.1
1.9
7.6
22.6
2.3
12.5
6.9
9.1
33.1
6.5
4
21.2
21.2
25.1
19.3
7.7
7.4
14.4
12.8
33.1
7.6
13.5
10.8
20.4
4.3
7.2
16.8
22.5
31.3
20.1
11.7
17.3
12
25.4
29.1
5.6
7.2
7.2
15.3
5
12.3
37.7
35.6
24.4
29.9
7.8
9.7
3.2
1.8
20.1
12.5
6.1
7.7
26.2
5
14.1
17
17.8
13.3
28.7
6.9
8.2
6.3
5.6
34
7.4
17
15.3
48.7
11.2
8.8
47.4
36.6
34.8
29.6
11.5
17.7
13.3
10.4
10.4
7.8
8.5
8.5
35.1
19.2
2.4
13.3
13.3
48.4
48.4
3.1
18.8
16.2
12.9
48.1
10.7
24
15.1
18
13.8
8.6
43.3
46
46.9
34.8
12.8
39
39
29
29
5.8
39.3
38.9
44
31
9
38.5
38.5
28.2
10.5
2.5
8.9
8.9
12.6
21.1
8.5
13.6
13.6
26.8
26.8
9.4
42.5
43.5
10
33.9
2.2
2.9
6.7
5.6
14.1
7.7
13.6
13.6
20.2
20.2
8.5
17.9
17.9
23.6
25.5
8.5
11.1
12.1
28.6
33
13.7
26.7
29.4
43.5
26
14.3
18.1
12.7
17.3
14.4
14.8
7.4
3.7
36.6
3.9
2.7
17.7
12.7
37.5
37.5
10.8
8.2
5.8
6.1
45.7
2.8
10.1
7.5
8.7
40.9
6.3
41.2
44.2
35.2
48.1
6.2
42.6
29.9
31.1
33.9
11.4
3.2
8.1
18.8
7.2
5
34.8
26.4
27.4
41.4
9.8
16.8
16.8
7.2
7.2
7.7
7.6
17.1
13.5
24
2.7
11.1
11.5
43
43
6.8
21.3
27.5
38.2
42
9.6
4.5
7.4
19.1
5.4
5.6
8.6
4.3
8
22.2
13.9
5.6
8.6
34.7
19.8
13.5
24.2
18.7
25.4
29.6
7.7
15.9
15.9
29
29.6
8.1
11.1
11.1
24.5
32.8
5
20.6
16
19.1
19.1
2.3
4.2
16.5
34.7
12.3
16.3
59.2
52.1
43.1
43.1
3.6
3.2
8.1
6.7
27.9
6
5.2
5.2
5.5
5.5
10.4
31.3
32.3
28.4
38.5
3.1
32.1
25.1
29.8
27.5
5.7
22.2
22.1
36.9
11.9
7.7
20
22.5
19
18.3
10.8
30.1
24.3
27.2
32.1
7.6
22.6
17.4
19
14.4
7
21.7
16.1
14.7
31.7
1.6
5.8
12.5
20
45.3
9.6
12
18.2
36.3
36.3
9.4
20.9
27.6
36.4
36.4
11
13.6
21.9
15.1
5.2
11.3
14.3
19.4
16.9
15.6
6.2
19.5
10.8
28.4
9.7
9.6
21.8
27.5
22.4
25.2
9.5
11.9
9.6
4.9
35.7
14.6
44.3
53.5
41.8
43.1
9.6
17.7
13.2
26.5
7.4
10.1
11.6
13
9
23.3
11.1
23.3
23.3
35.4
13.7
11
29.2
31.7
33.7
24.8
3.4
6.9
9.5
17.8
27.7
5.2
11.9
3.7
3.1
3.1
2.1
13.9
13.9
8.1
34.6
11.9
14.9
14.9
30.4
30.4
12.7
11.5
15.9
12.5
18.7
8.9
18.9
13.9
22.1
14.4
5.2
7
8.7
41.8
41.8
13.7
30
37.3
31.8
36.5
8.7
23.1
20.2
47
21.2
10.7
23.4
18.5
33.1
33.1
8.1
14
16.7
24.1
8.3
12.3
35.4
35.7
23.3
41.7
1.2
4.3
12.3
11.3
11.3
5.7
37.5
40.2
32
42.5
9.6
11.6
13.5
10
27.2
9
38.1
31
30.2
47.7
11.6
21.3
12.3
27.1
27.1
9.1
7.7
12
5.3
38.8
9.4
19
8
49.3
49.3
5
23.3
23.3
17.9
39.2
10.6
30.6
21.8
43.5
43.5
6.5
9.1
9.1
26.5
34.1
7.5
16.8
7.7
9.4
22.7
12.7
18.7
14.2
12.6
12.6
5.5
21.7
19.7
18.9
23.1
9.4
24.5
24.5
21.2
21.2
12.8
20.4
20.4
13.4
29.2
10.5
5.3
11.7
3.3
28.5
4.9
9.7
3.6
6.6
6.6
8.2
9.7
9.9
11.3
26
3.1
23.5
23.5
28.5
29
9.3
14.3
6
30.4
9.9
9.5
32.1
32.1
21.4
21.4
12.4
31.7
34.7
21
22.4
9.6
4.8
4.8
45.7
10.5
13.7
20.8
16.9
24.1
13.5
9.7
8.1
8.1
6.2
25.3
11
14.9
13.1
26.4
11.5
15
2
7.3
5.8
12.7
8.5
11.7
16.6
10.5
25.3
6.9
4.4
4.4
19.2
19.2
9.2
25.6
28.4
35.8
22.2
3
7.3
7.9
5.7
5.7
13.5
4.7
5.1
16.3
24
13.7
19.3
16.9
15.3
19.6
6.5
2.8
2.8
36.3
36.3
10.5
14
18.4
12.5
8
8.5
9
9.2
24.2
6.5
10.8
19.1
19.1
9
28
9.3
14.1
14.1
5.5
5.5
6.5
24.2
31.8
16.5
24.9
9.1
5.5
11.3
4.5
35.1
8.2
17.5
18.7
14.8
11
13.7
26.5
31.2
29.4
24.5
9.4
28.6
28.6
12.3
12.3
7.5
10.1
18.3
32.2
27.2
7.7
16.4
9.3
26.7
26.7
12.1
7.9
2.3
36.1
5
2.4
13.1
15.8
5.6
10.1
6.6
7.3
7.3
3.7
24.8
11.8
2.9
4.7
3.2
20.4
12.1
11.7
3.1
20.9
1.9
14.9
24.4
24.4
35.1
35.1
13.7
17.1
15.1
8
12.1
7.8
13.2
8.2
32
3.8
3.1
11.1
15.6
19.2
12
8.9
16.5
4.9
8.9
12.3
8.8
4.7
8.3
4.2
2.9
1.6
6.4
6.4
44.1
10.4
1.2
5.4
5.4
21.1
5.2
8.9
17.1
11.2
10.7
7.1
11.1
15.9
15.9
24.8
25.6
11.2
19.3
19.3
28.7
34.3
3.9
26.3
36.1
28
28
8.6
15.3
17
11.1
11.1
14.3
17.5
15.8
9.3
11.6
6.2
21
21
29.7
16.6
8.5
15.2
9.5
8
15.1
8
9.1
8.6
26.8
23.1
9.2
17.7
12.3
49.1
10.8
8.2
10.3
6.2
5.9
4.1
4.3
11.4
11.4
32.6
23.4
10.2
11.1
11.1
24
8.3
14.2
43.3
53.2
44.9
42.9
4.8
20.6
12.4
17.6
14.1
10
32.2
25.3
23
27.6
1.1
7.2
16.2
36.6
33.3
9.7
12.4
15.8
28.7
28.7
13.2
19
18
26.1
13.1
7.8
47.4
35.2
37.9
33
3.8
11
15.8
17.7
43.3
7
26
15.8
22.1
17.8
10.7
11.1
15.9
24.8
29.6
13.4
29.5
35.8
18.9
36.1
1.1
1.4
6.4
4.8
21.8
8.2
27.1
22
17.5
22.1
10.9
23.2
23.2
13.6
17.9
10.4
12.2
12.2
7.3
9.2
7.8
8.4
9.9
30.7
9
6.9
2.6
2.6
33.9
21.7
8.9
6.7
15.6
11.3
32.1
10.4
35.8
45.7
48.1
34.7
14.9
17.4
27.3
29.4
29.4
3.1
12.6
20.1
17.1
14.8
10.8
13.5
15.9
12
26.6
1.5
11.7
17.8
35.8
23.1
10.7
18.6
12.2
27.7
27.7
8.6
34.4
34.4
27.2
41.8
12.2
17.2
20
24.6
24.6
8.4
17.8
17.8
25.4
25.4
5.5
8.6
8.4
5.6
26.8
14.3
18.8
13
11.3
11.3
8.4
10.9
13.8
8
12.3
9.6
17.4
15.7
26.2
26.2
7.8
8.3
11.1
21.2
16
11.8
32.3
34.8
27.7
42.4
6.1
5.1
5.1
28.3
28.3
7.6
10.2
7.4
7
7
10.9
8.5
8.5
12.4
12.4
8.9
32.4
32.3
27.7
37.9
7
17.7
17.3
16
32
8.8
17.7
12.8
26.2
26.2
9.3
18.3
15.5
23.6
30.8
12.2
4.5
8.6
39.3
17.4
14.5
14.7
14.7
24.6
25.1
7.6
3.1
7.9
45.1
6
11.6
4.7
12.6
1.9
41.4
2.8
3.2
2.5
21.5
5.8
11.1
19.4
15.6
29
9.8
10.7
19.1
27.7
39.9
20.6
14.9
45.2
45.2
32.3
36.3
7.5
10.9
10.9
11.1
24.4
14.2
33.2
34.7
43.4
43.4
6.3
7.9
10.8
2.8
32.6
8.8
15
15
23.7
5.3
13.1
18.6
24.3
22.7
27.4
14.2
18.2
18.2
15
18
6.7
6.6
9.6
33.6
26.7
8.2
17.6
12.9
7.3
26.2
3.3
11.7
4.3
20.7
1.3
8.8
15.6
15.6
17.5
13.5
11.8
29.8
32.6
38.2
38.2
8.7
3
3.5
43.3
11.1
13
40
42.7
34.1
32.3
8.6
22.1
16.4
14.7
19.5
13.6
27.3
28.7
26.1
23.4
12.7
1.9
8.4
41.8
6.5
12.6
16.5
16.5
25
25
11
22.9
31.6
19.8
19.8
8.3
7.3
12.8
32
6.6
14.5
13.6
18
9.4
24.3
9.8
25.5
28.5
17.2
22.1
9.4
9.4
16.1
28.8
10.2
9.5
17.6
17.6
30.5
34.1
31
31.5
41.3
35.2
35.2
5.6
9.2
6.3
34.3
1.2
13
3.1
7.1
1.5
17.7
3.9
10.5
18.4
23.6
12.8
3.2
33.4
32.7
29.7
23.5
13.3
15.9
14.8
26.4
26.4
9.8
11
11
24.5
10
11.3
2.1
8.1
7.2
37.3
7.4
25.2
16.9
2.9
20.3
14.6
15.5
14
23.5
9.9
8.2
29.8
32.5
42.9
42.9
4.9
19.6
22.6
37.8
12.3
6.1
9.5
5.9
34.8
26
8.9
15.2
18.8
14.9
1.3
15
7.7
5.6
5.5
5.5
2
10.1
12.8
43
3.6
12.4
9.3
9.3
4.7
35.1
13.1
40.2
38.1
43.1
29.6
10
27.3
29.6
25.4
32.8
15.7
30
30
31
34.3
3.1
7.7
10.2
4.8
18.4
13.9
26.9
26.9
43.8
22
8.1
8.6
9.7
31.5
4.8
4.7
1.9
9.3
18.2
6.3
12.9
15.4
20.6
23.8
11.9
13.5
20.7
25.3
21.7
21.7
8.6
10.7
11.8
8.6
23.2
8.8
39.4
47.3
34.8
34.8
11.4
13
18.3
8.1
8.1
13.6
7.3
5.2
13.3
13.3
12.6
27.6
27.6
26
43.9
4
10.4
17.4
30.9
27.6
3.8
18.5
10.2
22.1
22.1
6.4
6.9
11.4
2.2
26.7
8.8
17.3
19.2
31.2
26.8
13.9
12.4
13.9
7.5
32.1
5.6
4.1
6.3
19.8
35.1
10.9
14.8
14.8
33.1
33.1
9.7
10.3
10.3
27.8
30.7
10.1
28.1
27.6
35.6
35.6
5.7
8.9
8.9
12.3
12.3
4.2
7
5.6
16.7
3.1
10.4
18.1
12.3
27.9
11.3
14.6
48.3
55.4
46.8
42.6
6.8
6.6
6.6
35.3
19.1
9.2
12.4
10.7
6.5
9.2
11.9
7.6
7.6
21.8
26.3
7.6
20.4
20.2
14.7
16.8
6.9
18.6
19.6
15.1
49.9
11.6
42.2
34.1
27.8
27.8
8.3
1.7
11.2
0.6
40.2
12.4
11.6
20.3
2.5
2.5
14
31.5
30
27.4
25.4
8.8
6.8
3.1
5
5
11.2
13.4
14.6
26.9
27.6
8.1
8.9
11.4
4.4
4.4
7.6
32.9
24.5
42.6
15.7
9.2
12.8
13.5
10.3
29.3
11.9
7.1
5.4
9.2
9.2
12.6
32.2
34.7
23.8
27.6
8.8
4.9
8.9
37
37
14.2
18.5
15.8
19.1
19.1
9.6
10.1
10.1
25
29.3
14.3
32.9
35.7
32.1
46.7
8.8
30
30
19.2
19.2
10.8
3.5
3.5
19.6
6.2
13
3.3
7.6
3.6
3.6
7.2
33.9
41
32.8
34.9
9.4
15.9
13.4
11.9
26.6
10.2
15.5
10.4
28.2
7.3
6.2
20.4
23.3
13
27.3
7
46.1
44.7
49.1
49.1
13.2
20.1
25.6
23.8
31.7
10.6
8
3.5
2.6
33.3
13.2
55.6
52.7
39.6
44.8
13
25.6
18.2
19.3
22.3
11.8
4.6
3.5
7.3
28.3
9.1
12.7
19.8
10.6
10.6
7.2
6
9
43.1
7.7
9.4
30.3
28.8
11.2
22
7
9.6
11.9
19.2
22.1
7.4
10.6
8.6
20.4
5.9
9.8
20.6
20.6
16.8
16.8
11.3
15.9
14.1
28.9
26.5
6.5
1.2
3.7
3.7
38.9
5.6
14.4
13.3
14.7
45.7
14.4
55.9
45.9
40
40
11.8
11.9
12.1
27.7
7.4
9.5
32.8
24.9
27.3
28.7
11.8
5.4
7.2
5.3
9.9
14.9
13.1
11.5
6.6
6.5
8.5
10.4
17.3
7.1
25.9
5.3
9.7
3
36.4
3.2
10.3
16
22.7
10.5
10.5
6.9
8.9
6.9
23.7
5.6
6.7
8.1
14.1
21.1
33
10.2
13.8
14.5
24.7
24.7
11
23.1
21.6
16.3
16.3
4.6
18.7
14.8
46.2
46.2
11.3
40.4
40.4
36.2
49.4
11.1
7.2
4
21.1
6.8
9.5
21
16.1
18.3
37.6
13
22.6
30.8
17.1
17.1
11.8
37.4
25.1
29.5
42.2
10.7
13.9
12.1
8.8
10.6
4.1
29.1
18.3
17.8
17.8
9
15.9
21.2
14.7
14.7
8.5
14.2
12.4
12.1
16.9
7.4
17.2
8.9
27.6
27.2
14.6
26.9
25
21.9
18.8
5.4
10.8
10.8
30.3
30.3
8.8
16.4
16.4
22
49.4
4
15.4
12.7
31
16.6
6
7.1
17.8
13.2
33.7
7.1
13.3
25.7
40.8
27.9
7.3
13.4
11.2
47
9.1
11.1
3.1
3.1
5.5
37.5
12.7
23.3
25.8
47.3
22.5
7.4
22.9
27.3
21.9
18.4
6.9
6.6
12.7
11
4.9
10.3
15.3
13.6
12.1
6.5
7.7
38.3
41.8
28
28
8.9
31.7
28.8
20.1
24.8
5.6
16.4
18.9
35.2
28.6
12.2
12.8
17.1
23.5
31.7
8.5
22.5
30.2
17.9
17.9
6.5
6.6
6.6
18
36.6
11.7
17.2
13.6
20.8
20.8
14.5
43.3
33.7
29.7
34.5
14.9
19.5
28.2
4.2
4.2
4.4
13.2
10.9
32.6
9.9
5
5.7
14.5
26.1
26.1
15
22.8
23.5
25.9
18.4
11.8
12.1
16.7
41.7
13
11.1
15.4
9.6
8.4
21.8
6.5
20.7
20.7
18.3
31.3
4.1
7.9
2.1
2.9
16.3
9.3
18.5
5.9
13.9
9.5
5.6
20.8
16.3
16.4
46.8
11.6
20.7
22.1
37.1
43
8.5
14.5
6.3
5
30.8
10.6
30.1
29.8
19.9
37.3
11.2
15.3
21.2
10.7
10.7
3.3
25.9
28.8
22.4
22.2
10.2
34.2
26.3
23.9
28.5
8
16.3
19.8
36
36
6.2
23.5
25.7
21.2
15.9
48
10.4
13.1
15.9
4.3
8.4
24.1
33.9
42.7
14.6
13.3
27
17.2
1
1
10.1
32.5
32.5
20.8
46.5
12.8
3.4
6.5
17.5
25.1
4.8
19.9
24.6
17.6
39.1
12.6
18
22.9
12.6
27.9
14.6
1
4.9
1
1
4.9
3.9
6.4
1.7
35.1
6.9
20.7
21.6
19.1
17.7
9.5
13.8
12.2
27
8.5
7.5
8.7
8.7
31.6
31.6
5.5
1.9
6.5
1.7
1.7
7.1
16.9
19.1
33.4
35.7
6.7
2.6
6.4
40.6
23.5
8.3
6.7
4.9
6.5
30.6
4.7
22
17.5
6
17.4
7.8
11.1
9.4
12.2
22.7
8.5
36.6
39.5
49.3
45.8
11
13.1
14.5
15.1
23.3
9.2
11.2
11.2
6.1
22.6
12.6
8.4
8.4
41.5
6.2
4.3
3.4
14.4
24.2
11.1
14.5
11
14.1
7.4
37.2
9.1
19.8
19.8
37.6
37.6
9.6
29.3
26.9
22.1
17.6
11.9
22.2
19.8
33.2
22.6
12.5
19.9
16.3
12.2
25.7
8.1
40.1
39.6
34
34
12.5
9.6
9.6
8.2
22.8
9.8
25.4
17.6
36
17.2
6.7
21
21.5
17.7
13
14.7
12.1
12.1
22.3
21.2
13.1
8.1
10.8
6.6
6.6
10.8
40.6
40.6
40.4
40.4
5.5
6
6
23.9
3.4
10.5
15.7
18.2
26.5
7.9
2.6
18.2
20.5
15
17.5
9.2
14.2
4.1
8.5
8.5
9.6
13.9
19.3
10.5
14.6
12.5
27.3
21.5
22.3
22.3
13.3
25.6
30
25
29.1
13.5
43.9
41.4
45.5
35.1
7.8
9
9
25.7
30.1
8.7
10.6
17.5
28.3
28.3
2.1
8.1
17.2
7.8
19.5
11.4
6.2
13.6
39.5
19
2.8
31.5
31.8
20
44.9
5.5
5.1
9.2
0.7
19.6
7.5
3
3
1.8
20.6
10
6.4
6.4
12.4
6.9
14.1
12.5
8.9
6.6
34.2
7.1
21.6
22.4
30.5
13.2
12.5
15.3
15.9
27.9
13.5
9.2
9.9
15.6
6.7
25.2
7.5
42.8
42.8
31.6
36.5
3.6
18.3
12.3
15.3
4.5
3.3
12.8
20.6
17.6
15.1
13.7
44.2
44.2
30.8
32.8
5.2
5.1
7.8
26
18.5
10.2
17.9
17.9
32.4
12.6
7.3
28.6
28.6
31.3
31.3
10.8
21.2
18.8
21.4
18.5
7.7
8.4
4.4
2.2
35.2
9.3
9.9
15.8
30.2
6.1
8.6
31
31
35.5
45.5
11.1
15.8
15.8
23.1
23.1
9.4
12
13.2
11.8
8.1
8.5
10.5
16.6
28.5
28.5
8.7
11.1
16.2
6.3
6.3
8.4
17.6
18.7
12.1
28.5
10.3
23.2
25.3
22
26.6
13.3
8.8
8.9
6.2
24
8.1
7.8
11
32.8
5
9.2
18.3
18.3
8.9
48.4
10.4
42.4
50.1
32.2
37.3
6.4
7.5
7.5
3.5
3
10.1
18.5
26.5
22.9
22.9
12.1
27.6
27.7
22.6
24.7
8
20.2
19.1
17.8
17.8
14.7
1.5
4.8
13.4
6.1
7.8
12.7
14.8
9.1
23.8
11.2
5.4
16.6
7.8
19.7
8.9
11.2
11.2
27.7
11.9
6.4
0.4
5.2
11.9
22.4
11.3
26.6
26.6
20.6
39
8.4
3.8
3.8
12.4
20.7
13.1
8.6
13.4
6.5
21.8
20.2
12.7
8.3
20.1
20.1
9.1
17.9
25.6
35.4
18.2
3
26.1
16.5
15.3
20.1
6.1
11.5
20.2
15.3
10.5
10.4
14.6
10.4
6.9
23.1
9.9
15
22
16.9
19
9.3
20.1
20.1
16.4
16.4
10.2
20
14.7
9.6
9.6
15
18.9
27.2
16.2
16.2
13.2
16.8
17.8
12.1
24.8
10
12.8
15
8.4
11.4
9.6
12.6
15.9
8.4
8.4
7.8
12.8
9.8
32.9
32.9
9.1
19.4
17
17.1
22.8
10.2
23.9
23.3
21.4
19.9
14.6
27
18.3
41.3
34.7
13.4
20.6
20.6
33.1
32.8
5.8
17.5
17.5
15.2
15.1
11.5
20.3
14.5
30.4
24.7
13.7
9.4
5.9
42.5
8.7
13.9
25
19.4
14
14
8.6
6.8
10
4.1
33.4
8.1
7.9
16.5
35.2
11.2
7.1
9.9
11.5
21.3
21.3
6.1
7.1
5.8
4.2
25.5
13.1
11.6
11.6
4.7
22
13.2
24.4
27.9
33.5
19.3
13.7
8.6
14.4
7.6
33.7
9.1
17.8
13.8
23.1
6.9
11.2
3.1
6.9
6.5
6.5
9.8
12.8
0.2
9.1
34.8
8.8
21.9
22.6
32.5
20.2
7
21.2
12
31.4
31.4
3.5
29.2
19.4
22.8
22.8
14.3
11.4
9.4
23.9
21.2
12.6
9.6
12.1
4.1
18
14.3
25.5
28.8
39.5
41.2
7.1
7.2
2.6
17.5
24
7.8
1.7
6.1
8.7
22.2
1.9
14.3
9.7
32
41.5
11.6
8.1
13
3.6
22.3
13.3
25.5
33.8
26.6
21.8
7
25
27.8
39.9
18.8
7
9.8
12.6
28.5
30.8
6.5
5
4.5
42.4
3.8
9.4
16.6
14.9
34.3
13.2
9.2
25.4
28.2
35.7
17.1
8.7
17.4
13.5
31.8
28.3
8.6
14.5
10.1
8
32.1
8.7
13.4
6.7
7.3
7.3
4.1
7.9
4.8
17.6
17.6
13.9
8.1
2.9
3.9
39.3
14.3
7.4
7.4
24
19.5
6.6
12.2
21.8
36.1
11.8
7.2
8.8
5.1
35.7
24.5
14
27.7
30.3
18.1
17.9
13.5
17.5
24.2
13
13
6.3
41.7
35
35.5
35.5
9.3
13.6
9.4
28.3
28.3
6.5
40.2
37.1
15.2
15.2
14.5
23.9
31.2
37.6
17.6
7.1
10.5
10.5
5.2
30.8
2.6
21.1
31
38
29.2
12.9
12.3
12.3
22.3
43.7
7.6
12.1
14.6
9.4
21.2
13
24.3
19.2
30.4
29.7
10.2
11.7
14.7
7.7
11
9.8
6.4
17.2
8.1
33.6
13.5
10.6
11.7
12.5
17.1
9.2
11.8
11.9
6.2
25.4
14.3
19.9
19.2
33.7
16
12.4
36.4
25.6
28
41.9
5.2
5.1
3.8
3.6
3.6
10.9
10.7
2.1
36.2
7.4
14.5
9.3
1.9
13.8
4.2
2.4
18.9
12.6
13.4
16.2
11
14.1
16.8
9.6
23
3.6
20
27.4
23.3
19
9.8
16.9
16.9
7.1
7.1
8.5
15.2
19.6
18.3
14.8
8.7
24.2
31.3
31.4
31.4
10.3
42.1
33.2
29
48.2
13.9
20.5
21.9
31.4
14.9
14.9
10.4
6.6
12.8
5.3
2.9
30.1
30.1
18.7
43.7
8.5
10.5
10.5
7.2
25.9
3.6
24
26.5
27.9
47.8
14.5
26.7
26.7
12.7
25.8
13.8
23.1
22.3
17.7
36.8
6.2
33.6
43.4
46
46
8.6
10.4
12.1
7
7
11.6
23.5
12.4
17.4
17.4
5.3
3.5
9.1
38.3
16.2
8.1
31.2
20.4
18.5
5
7.9
19.9
22.7
35.7
17.4
13.7
28.5
25.9
24.9
20.4
9.9
24.8
24.8
16.6
35.3
7.8
3.9
7.9
37.8
21.5
9.7
12.9
4.4
9
4.2
13.4
22.8
20.7
23.2
31.7
14.3
32.6
32.6
24.1
24.1
11.4
36.9
36.9
30.3
30.3
8.5
12.6
7.1
4
33.1
5.8
1
6
23.5
4.7
14.3
16.5
14.9
23.2
23.6
1.9
2.6
7.4
7.1
42
11.3
13.2
8.6
10.6
45.9
14
28.8
27.3
30.2
30.2
9.4
12.8
12.8
8.6
32.9
11.3
10
20
28.8
9.7
12.3
34
34
29.7
16.8
1.5
8.2
7.3
5.9
24.5
11
9.3
10.5
8.1
22.1
14.2
25.7
20.6
32.6
18
2.1
11.4
14.7
13
48.2
8.3
12.2
2.7
20.9
7
10.2
13.6
15.9
8.8
12
9.1
15.5
9.7
22.7
8.7
11.5
14.3
6.9
20.1
20.1
14.3
12.1
10.1
33.1
21.7
11
10.1
2.5
6.8
27.8
11.6
15
19.4
12
26
13.7
4.8
7.9
38.5
38.5
13.5
25.3
21.4
34.4
19.8
7.1
10.6
7.8
26.8
32.4
10.3
14.9
17.1
11.1
24.6
5.5
8.4
8.6
26.8
35.8
3.2
11.8
11.4
31.9
43.7
14.2
41.1
41.1
33.2
28.9
9.5
3.8
9.4
5.4
5.4
10.6
14.6
14.4
29.4
29.4
12.9
30.6
28.4
26.7
12.1
12.4
38.2
40.7
38.4
41.1
5.4
8.7
1.7
4.3
19.3
8.7
16.8
16.8
46.9
46.9
4.5
37.5
38.5
43.7
43.7
7
26.1
27.4
16.5
21.2
11.8
9.9
12.8
9.4
24.1
8.5
13.1
11.7
10.8
15.5
12.2
4.6
4.6
4.3
43.5
12.1
4.7
1.9
26.4
5.1
5.5
2.3
10.7
36.4
28.7
12.7
19.1
14.4
22.8
11.5
12.7
2.2
6.6
9.2
10.4
11.6
15.8
15.5
10.4
29.4
8.5
25
25
24.6
24.6
9.9
13.3
16.9
9.2
29.6
4.9
34.9
34.7
29.6
25.3
7.2
8.3
17.3
26.4
26.4
13.6
46.8
56.6
41.4
41.4
2
11.5
13.9
6.7
31.4
11
17.1
17.1
11.8
11.8
8.6
9.6
5.3
7.8
7.8
0.6
15.1
15.1
38.2
33.9
3.8
2
8.4
7.6
28.9
10
30
20.8
23.8
38.7
4.4
10.3
9
19.7
7
7.3
8
4.5
7.5
32.8
8.3
4.4
14.2
34.6
12.5
14.4
18.1
17.5
2.9
21.2
14.7
19.4
14.7
33.6
16.1
13
10.9
7.3
44.8
10.5
8.5
19.5
19.8
30.5
17.7
10.1
11.8
16.8
17.9
13.5
4.6
0.9
9.5
3.3
3.3
''')

array_string_1 = data_sting_1.split('\n')
array_cords_time = time_based_constraint.split('\n')
origin_log_time = []
origin_logtime = []
for i in range(len(array_cords_time)-1):
  if(array_cords_time[i] != ''):
     origin_log_time.append(float(array_cords_time[i]))
array_string_1.pop(0)
for i in range(len(origin_log_time)-1):
  if(array_string_1[i] == 'Work'):
    origin_logtime.append(origin_log_time[i])

print(array_string_1[0])
print(origin_logtime[3])
print(len(origin_logtime))

In [ ]:
array_string_auto = data_string_2.split('\n')
origin_log_auto = []
origin_logauto = []

array_string_auto.pop(0)
for i in range(len(origin_log)-1):
  if(array_string_auto[i] == 'Individual Auto' or array_string_auto[i] == 'Share Auto'):
    origin_logauto.append(origin_log[i])

array_string_auto_1 = data_string_2.split('\n')
origin_lat_auto = []
origin_latauto = []

array_string_auto_1.pop(0)
for i in range(len(origin_lat)-1):
  if(array_string_auto_1[i] == 'Individual Auto' or array_string_auto_1[i] == 'Share Auto'):
    origin_latauto.append(origin_lat[i])
print(len(origin_logauto))
print(len(origin_latauto))

In [ ]:
array_string_auto_shop = data_string_shop_mode.split('\n')
# origin_log_auto_shop = []
origin_logauto_shop = []

array_string_auto_shop.pop(0)
for i in range(len(origin_log)-1):
  if(array_string_auto_shop[i] == 'Individual Auto' or array_string_auto_shop[i] == 'Share Auto'):
    origin_logauto_shop.append(origin_log[i])

array_string_auto_1_shop = data_string_shop_mode.split('\n')
# origin_lat_auto_shop = []
origin_latauto_shop = []

array_string_auto_1_shop.pop(0)
for i in range(len(origin_lat)-1):
  if(array_string_auto_1_shop[i] == 'Individual Auto' or array_string_auto_1_shop[i] == 'Share Auto'):
    origin_latauto_shop.append(origin_lat[i])
print(len(origin_logauto_shop))
print(len(origin_latauto))

In [ ]:
array_string_auto_free = data_string_free_mode.split('\n')
# origin_log_auto_shop = []
origin_logauto_free = []

array_string_auto_free.pop(0)
for i in range(len(origin_log)-1):
  if(array_string_auto_free[i] == 'Individual Auto' or array_string_auto_free[i] == 'Share Auto'):
    origin_logauto_free.append(origin_log[i])

array_string_auto_1_free = data_string_free_mode.split('\n')
# origin_lat_auto_shop = []
origin_latauto_free = []

array_string_auto_1_free.pop(0)
for i in range(len(origin_lat)-1):
  if(array_string_auto_1_free[i] == 'Individual Auto' or array_string_auto_1_free[i] == 'Share Auto'):
    origin_latauto_free.append(origin_lat[i])
print(len(origin_logauto_free))
print(len(origin_latauto))

In [ ]:
array_string_2 = data_sting_1.split('\n')
array_cords_3 = data_cords_2.split('\n')
origin_log_2 = []
origin_log2 = []
for i in range(len(array_cords_3)-1):
  if(array_cords_3[i] != ''):
     origin_log_2.append(float(array_cords_3[i]))
array_string_2.pop(0)
for i in range(len(origin_log_2)-1):
  if(array_string_2[i] == 'Shopping'):
    origin_log2.append(origin_log_2[i])

array_string_3 = data_sting_1.split('\n')
array_cords_4 = data_cords_1.split('\n')
origin_lat_2 = []
origin_lat2 = []
for i in range(len(array_cords_4)-1):
  if(array_cords_4[i] != ''):
     origin_lat_2.append(float(array_cords_4[i]))
array_string_3.pop(0)
for i in range(len(origin_lat_2)-1):
  if(array_string_3[i] == 'Shopping'):
    origin_lat2.append(origin_lat_2[i])

In [ ]:
data_2 = {'lat' : origin_latauto_shop, 'lon' : origin_logauto_shop}
df_2 = pd.DataFrame(data_2)
kmeans= KMeans(n_clusters= 35, random_state = 0).fit(df_2[['lat', 'lon']])
df_2['cluster'] = kmeans.labels_
cluster_means_2 = df_2.groupby('cluster')[['lat', 'lon']].mean().reset_index()
sns.scatterplot(data=df_2, x='lon', y='lat', hue='cluster', palette='viridis')
plt.title("Spatial Groups")
plt.show()

In [ ]:
data_3 = {'lat' : origin_latauto_shop, 'lon' : origin_logauto_shop}
df_3 = pd.DataFrame(data_2)
kmeans= KMeans(n_clusters= 69, random_state = 0).fit(df_3[['lat', 'lon']])
df_3['cluster'] = kmeans.labels_
cluster_means_3 = df_3.groupby('cluster')[['lat', 'lon']].mean().reset_index()
sns.scatterplot(data=df_3, x='lon', y='lat', hue='cluster', palette='viridis')
plt.title("Spatial Groups")
plt.show()

In [ ]:
work_log = list(set(origin_log1))
work_lat = list(set(origin_lat1))
shop_log = list(set(origin_log2))
shop_lat = list(set(origin_lat2))

for b in work_log:
  cnt = 0
  for a in origin_log1:
    if(a == b):
      cnt = cnt + 1
  print(b,cnt)

In [ ]:
or_node = ox.distance.nearest_nodes(G, X=cluster_means_3['lon'], Y=cluster_means_3['lat'])
des_node = ox.distance.nearest_nodes(G, X=work_log, Y=work_lat)

routes = []

missing_paths = 0

for o in or_node:
    for d in des_node:
        try:
            path = nx.shortest_path(G, source=o, target=d, weight='length')
            routes.append(path)
        except nx.NetworkXNoPath:
            missing_paths += 1

print(missing_paths)

route_lines = []

for path in routes:
    route_coords = [(G.nodes[n]['x'], G.nodes[n]['y']) for n in path]
    route_lines.append(LineString(route_coords))

gdf_routes = gpd.GeoDataFrame(geometry=route_lines, crs="EPSG:4326")

orig_points = [Point(lon, lat) for lon, lat in zip(cluster_means_3['lon'], cluster_means_3['lat'])]
dest_points = [Point(lon, lat) for lon, lat in zip(work_log, work_lat)]

gdf_orig = gpd.GeoDataFrame(geometry=orig_points, crs="EPSG:4326")
gdf_dest = gpd.GeoDataFrame(geometry=dest_points, crs="EPSG:4326")
station_point = (82.98599053490672 ,25.3275846445095)
gdf_station = gpd.GeoDataFrame(geometry=[Point(station_point)], crs="EPSG:4326")

m = gdf_routes.explore(
    color="blue",
    weight=3,
    opacity=0.6,
    name="Vehicle Routes"
)

m = gdf_orig.explore(
    m=m,
    color="red",
    marker_kwds={"radius": 6, "fill": True},
    name="Origins"
)

m = gdf_dest.explore(
    m=m,
    color="green",
    marker_kwds={"radius": 6, "fill": True},
    name="Destinations"
)

m = gdf_station.explore(
    m=m,
    color="black",
    marker_kwds={"radius": 6, "fill": True},
    name = "station"
)


folium.LayerControl().add_to(m)

m

In [ ]:
mp = {}
for path in routes:
  for ind in path:
    node_data = G.nodes[ind]
    mp[(node_data['x'], node_data['y'])] = 0

path_demand = [[]]

for lat,lon in zip(cluster_means_3['lat'], cluster_means_3['lon']):
  mp[(lon,lat)] = 1

for path in routes:
  for i in range(len(path)):
    node_data = G.nodes[path[i]]
    if(i != 0):
      mp[(node_data['x'], node_data['y'])] = mp[(node_data['x'], node_data['y'])] + 1

In [ ]:
edge_data = []

for path in routes:
  for i in range(len(path) - 1):
    u = path[i]
    v = path[i+1]

    v_lon = G.nodes[v]['x']
    v_lat = G.nodes[v]['y']

    demand = mp[(v_lon, v_lat)]

    if 'geometry' in G[u][v][0]:
      geom = G[u][v][0]['geometry']
    else:
      u_lon = G.nodes[u]['x']
      u_lat = G.nodes[u]['y']
      geom = LineString([(u_lon, u_lat), (v_lon, v_lat)])

    edge_data.append(
        {
            'geometry': geom,
            'forward demand': demand
        }
    )

gdf_edges = gpd.GeoDataFrame(edge_data, crs="EPSG:4326")

m = gdf_edges.explore(
    column = 'forward demand',
    cmap = 'plasma',
    tooltip = 'forward demand',
    style_kwds={'weight': 4, 'opacity': 0.8},
    legend=True,
    name="Vehicle Routes"
)
m

In [ ]:
import statistics
sum = 0
cnt = 0
arr_x = []
for path in routes:
  # print(len(path))
  cnt += len(path)
  for i in range(len(path)):
    node_data = G.nodes[path[i]]
    if(i != 0):
      sum  += mp[(node_data['x'], node_data['y'])]
      arr_x.append(mp[(node_data['x'], node_data['y'])])
      # print((node_data['x']),(node_data['y']) )
print(cnt)
print(len(routes))
print(sum/cnt)
print(statistics.mean(arr_x))
print(statistics.median(arr_x))
max(arr_x)